# Counterfactual Dataset – Manufacturer Swap (Setup)

Prepare two hospital datasets (Philips vs GE) in a fixed window and apply a counterfactual manufacturer swap at a chosen timepoint.
No analysis or exports yet.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import ks_2samp
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.proportion import proportions_ztest

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)


In [ ]:
# Runtime Profiler Setup (run once before Run All)
import time
import hashlib
from datetime import datetime
from IPython import get_ipython

ip = get_ipython()

if '_NB_RUNTIME_LOG' not in globals():
    _NB_RUNTIME_LOG = []

# Remove old hooks if re-running this cell
if '_NB_RT_PRE' in globals() and '_NB_RT_POST' in globals():
    try:
        ip.events.unregister('pre_run_cell', _NB_RT_PRE)
    except Exception:
        pass
    try:
        ip.events.unregister('post_run_cell', _NB_RT_POST)
    except Exception:
        pass


def _NB_RT_PRE(info):
    globals()['_NB_RT_T0'] = time.perf_counter()
    raw = getattr(info, 'raw_cell', '') or ''
    globals()['_NB_RT_SRC'] = raw


def _NB_RT_POST(result):
    t0 = globals().get('_NB_RT_T0', None)
    if t0 is None:
        return
    dt = float(time.perf_counter() - t0)
    src = globals().get('_NB_RT_SRC', '') or ''
    src_hash = hashlib.md5(src.encode('utf-8')).hexdigest()[:12]

    # first non-empty line for quick context
    first = ''
    for line in src.splitlines():
        s = line.strip()
        if s:
            first = s[:160]
            break

    _NB_RUNTIME_LOG.append({
        'timestamp': datetime.now().isoformat(timespec='seconds'),
        'exec_count': int(getattr(ip, 'execution_count', -1)),
        'cell_hash': src_hash,
        'elapsed_sec': dt,
        'first_line': first,
    })


# Register hooks
_NB_RT_PRE = _NB_RT_PRE
_NB_RT_POST = _NB_RT_POST
ip.events.register('pre_run_cell', _NB_RT_PRE)
ip.events.register('post_run_cell', _NB_RT_POST)

print('Runtime profiler enabled.')
print('Logged entries so far:', len(_NB_RUNTIME_LOG))
print('Tip: Run this once, then Run All. Use the report cell at the end for Top 10.')


## Configuration


In [ ]:
DATA_PATH = "../../data_vaib/project_starting_dataset.csv"
START_DATE = "2017-01-01"
END_DATE = "2017-05-31"

DATE_COL = "study_date"
MANUFACTURER_COL = "manufacturer"
HOSPITAL_COL = "imp_performing_clinic"
CANCER_LABEL_COL = "CANCER_36_months_within_exam"
AI_COLUMNS = ["lunit", "therapixel", "vara"]

# Option A hospitals (clean single-manufacturer sites)
HOSPITAL_PHILIPS = "Linkoping University Hospital"
HOSPITAL_GE = "Vastmanland Hospital Vasteras"

# Transition will be GE -> Philips after concatenation
SWAP_FROM = "GE"
SWAP_TO = "Philips"

# Ratio target: cancers/controls = 1/100
TARGET_RATIO = 1 / 100

# AI flagging rule
AI_FLAG_METHOD = "quantile"  # "quantile" or "fixed"
AI_FLAG_QUANTILE = 0.90
AI_FLAG_FIXED_THRESHOLD = 0.50

# Radiologist flagging mode
RADIOL_MODE = "combined"  # "combined" = r1 OR r2, "single" = r1 only

# Screen-detected definition
SCREEN_THRESHOLD_DAYS = 90

# Reproducibility
RANDOM_SEED = 44

# Sliding window parameters (Level 1/2)
REF_SIZE = 1500
GAP_SIZE = 500
CURR_SIZE = 500
STEP_SIZE = 250
N_PERMUTATIONS = 1000


## Study Population Reporting (Figure 1 / Table 1)

Run this block on the full longitudinal dataset to generate the Figure 1 data selection flow and the Table 1 cohort summary before the filtered exploratory analysis below.


In [ ]:
from pathlib import Path
import importlib
import os
import sys

from IPython.display import Image, display

CWD = Path.cwd()
if (CWD / "study_population_reporting.py").exists():
    NOTEBOOK_DIR = CWD
elif (CWD / "code_files" / "active_notebooks" / "study_population_reporting.py").exists():
    NOTEBOOK_DIR = CWD / "code_files" / "active_notebooks"
else:
    raise FileNotFoundError("Could not locate study_population_reporting.py from the current working directory.")

MPLCONFIGDIR = NOTEBOOK_DIR / "outputs" / ".mplconfig"
MPLCONFIGDIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(MPLCONFIGDIR))

if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

import study_population_reporting as study_population_reporting
study_population_reporting = importlib.reload(study_population_reporting)
generate_study_population_reporting_bundle = study_population_reporting.generate_study_population_reporting_bundle

data_path = Path(DATA_PATH)
if not data_path.is_absolute():
    data_path = (NOTEBOOK_DIR / data_path).resolve()

FIG1_TABLE1_OUTPUT_DIR = NOTEBOOK_DIR / "outputs" / "2026_04_13_study_population_reporting"

study_population_bundle = generate_study_population_reporting_bundle(
    data_path=data_path,
    output_dir=FIG1_TABLE1_OUTPUT_DIR,
    hospital_col=HOSPITAL_COL,
    manufacturer_col=MANUFACTURER_COL,
    date_col=DATE_COL,
    cancer_label_col=CANCER_LABEL_COL,
    ratio=int(round(1 / TARGET_RATIO)),
    random_state=RANDOM_SEED,
    min_cancers=100,
    ref_size=20000,
    gap_size=2000,
    curr_size=20000,
    step_size=500,
)

print("Figure 1 asset:", study_population_bundle["figure_path"])
print("Manuscript Table 1 CSV:", study_population_bundle["manuscript_table1_path"])
print("Full stage-comparison Table CSV:", study_population_bundle["table1_path"])
print("Pair status CSV:", study_population_bundle["pair_status_path"])
print("Flow data CSV:", study_population_bundle["flow_path"])
print("Exclusion summary CSV:", study_population_bundle["exclusion_summary_path"])

display(Image(filename=str(study_population_bundle["figure_path"])))
display(study_population_bundle["manuscript_table1_df"])
display(study_population_bundle["flow_df"])
display(study_population_bundle["table1_df"])
display(study_population_bundle["pair_status_df"])
display(study_population_bundle["exclusion_summary_df"])


## Load and basic preprocessing


In [ ]:
df = pd.read_csv(DATA_PATH, low_memory=False)

# Parse dates and filter range
df[DATE_COL] = pd.to_datetime(df[DATE_COL], format='mixed')
df = df[(df[DATE_COL] >= START_DATE) & (df[DATE_COL] <= END_DATE)].copy()

# Outcome label for ratio balancing
if 'has_cancer' not in df.columns:
    df['has_cancer'] = df[CANCER_LABEL_COL]

print(f"Filtered dataset: {len(df):,} exams")
print(f"Date range: {df[DATE_COL].min().date()} to {df[DATE_COL].max().date()}")


In [ ]:
df

In [ ]:
# Sanity-check hospital labels for the selected sites
selected_hospitals = [HOSPITAL_PHILIPS, HOSPITAL_GE]
print('Selected hospitals (imp_performing_clinic):', selected_hospitals)

for hosp in selected_hospitals:
    sub = df[df[HOSPITAL_COL] == hosp]
    print(f'{hosp}')
    print('  n exams:', len(sub))
    print('  manufacturers:', sub[MANUFACTURER_COL].value_counts().to_dict())
    print('  date range:', sub[DATE_COL].min().date(), 'to', sub[DATE_COL].max().date())

# Filter to the two hospitals only
df_sites = df[df[HOSPITAL_COL].isin(selected_hospitals)].copy()
print('Filtered to selected hospitals:', len(df_sites))


## Counterfactual dataset creation (reshuffle + ratio balancing)
We reshuffle GE and Philips separately, balance cancer/control ratios by upscaling controls in the higher-cancer-ratio group, then concatenate GE → Philips to create the counterfactual switch.


In [ ]:
def balance_controls_to_ratio(df_group, target_ratio, label_col='has_cancer', random_state=RANDOM_SEED):
    # Upscale controls (with replacement) so cancers/controls matches target_ratio.
    rng = np.random.default_rng(random_state)
    cancers = df_group[df_group[label_col] == 1].copy()
    controls = df_group[df_group[label_col] == 0].copy()

    n_cancers = len(cancers)
    n_controls = len(controls)
    if n_controls == 0:
        raise ValueError('No controls found in group; cannot balance ratio.')

    current_ratio = n_cancers / n_controls
    if current_ratio <= target_ratio:
        return df_group.copy(), 0

    needed_controls = int(np.ceil(n_cancers / target_ratio))
    n_to_add = max(0, needed_controls - n_controls)
    if n_to_add == 0:
        return df_group.copy(), 0

    add_idx = rng.choice(controls.index.to_numpy(), size=n_to_add, replace=True)
    controls_extra = controls.loc[add_idx]
    balanced = pd.concat([cancers, controls, controls_extra], axis=0)
    return balanced, n_to_add

# Split hospitals and ensure single-manufacturer subsets
df_ge = df_sites[df_sites[MANUFACTURER_COL] == 'GE'].copy()
df_ph = df_sites[df_sites[MANUFACTURER_COL] == 'Philips'].copy()

# Compute cancer/control ratios
ge_cancers = (df_ge['has_cancer'] == 1).sum()
ge_controls = (df_ge['has_cancer'] == 0).sum()
ph_cancers = (df_ph['has_cancer'] == 1).sum()
ph_controls = (df_ph['has_cancer'] == 0).sum()

ge_ratio = ge_cancers / ge_controls if ge_controls > 0 else np.nan
ph_ratio = ph_cancers / ph_controls if ph_controls > 0 else np.nan

# Target ratio: fixed cancers/controls = 1/100
target_ratio = TARGET_RATIO

print('GE ratio (cancers/controls):', ge_ratio)
print('Philips ratio (cancers/controls):', ph_ratio)
print('Target ratio:', target_ratio)

# Balance controls in the higher-ratio group
balanced_ge, ge_added = balance_controls_to_ratio(df_ge, target_ratio)
balanced_ph, ph_added = balance_controls_to_ratio(df_ph, target_ratio)

print('Controls added - GE:', ge_added)
print('Controls added - Philips:', ph_added)

# Reshuffle within each manufacturer
balanced_ge = balanced_ge.sample(frac=1.0, random_state=RANDOM_SEED).reset_index(drop=True)
balanced_ph = balanced_ph.sample(frac=1.0, random_state=RANDOM_SEED).reset_index(drop=True)

# Concatenate to simulate GE -> Philips transition (no mixing)
combined_df = pd.concat([balanced_ge, balanced_ph], axis=0).reset_index(drop=True)
transition_point = len(balanced_ge)

print('Combined dataset size:', len(combined_df))
print('Transition point (index):', transition_point)
print('Manufacturers (post-reshuffle counts):')
print(combined_df[MANUFACTURER_COL].value_counts().to_dict())


In [ ]:
# Display cancer:control ratios as 1:x (controls per cancer)

def ratio_1_to_x(cancers, controls):
    if cancers == 0:
        return '1:inf'
    return f"1:{controls / cancers:.0f}"

print('Cancer:Control ratios (1:x)')
print(f"GE before:  {ratio_1_to_x(ge_cancers, ge_controls)}")
print(f"Philips before:  {ratio_1_to_x(ph_cancers, ph_controls)}")
print(f"GE after:   {ratio_1_to_x(balanced_ge['has_cancer'].sum(), (balanced_ge['has_cancer'] == 0).sum())}")
print(f"Philips after:   {ratio_1_to_x(balanced_ph['has_cancer'].sum(), (balanced_ph['has_cancer'] == 0).sum())}")


In [ ]:
# Define AI thresholds on GE (pre-transition) and derive flags
def compute_ai_thresholds(ref_df, ai_columns=AI_COLUMNS, method=AI_FLAG_METHOD,
                          quantile=AI_FLAG_QUANTILE, fixed=AI_FLAG_FIXED_THRESHOLD):
    thresholds = {}
    for ai in ai_columns:
        if method == 'quantile':
            thresholds[ai] = ref_df[ai].quantile(quantile)
        else:
            thresholds[ai] = fixed
    return thresholds

ai_thresholds = compute_ai_thresholds(balanced_ge)
print('AI thresholds (GE reference):', ai_thresholds)

combined_df = combined_df.copy()

# Radiologist flagging
if RADIOL_MODE == 'single':
    combined_df['rad_flagged'] = (combined_df['r1'] == 'DISCUSSION').astype(int)
else:
    combined_df['rad_flagged'] = ((combined_df['r1'] == 'DISCUSSION') | (combined_df['r2'] == 'DISCUSSION')).astype(int)

# AI flags
for ai in AI_COLUMNS:
    combined_df[f'{ai}_flagged'] = (combined_df[ai] >= ai_thresholds[ai]).astype(int)

# Screen-detected cancer definition (time-based ≤ 90 days)
if 'days_to_diagnosis' not in combined_df.columns:
    raise ValueError('Missing days_to_diagnosis column')

def extract_min_positive_days(val):
    if pd.isna(val):
        return np.nan
    if isinstance(val, (int, float)):
        return val if val > 0 else np.nan
    s = str(val).strip()
    if s in ('[]', ''):
        return np.nan
    try:
        import ast
        parsed = ast.literal_eval(s)
    except Exception:
        return np.nan
    if isinstance(parsed, (list, tuple)):
        nums = [float(x) for x in parsed if pd.notna(x)]
    else:
        nums = [float(parsed)]
    pos = [x for x in nums if x > 0]
    return min(pos) if pos else np.nan

combined_df['days_to_diagnosis_pos'] = combined_df['days_to_diagnosis'].apply(extract_min_positive_days)
combined_df['time_based_screen'] = ((combined_df['has_cancer'] == 1) & \
    (combined_df['days_to_diagnosis_pos'].notna()) & \
    (combined_df['days_to_diagnosis_pos'] <= SCREEN_THRESHOLD_DAYS))

# Subset helpers
combined_df['rad_not_flagged'] = combined_df['rad_flagged'] == 0
combined_df['screen_rad_flagged'] = combined_df['time_based_screen'] & (combined_df['rad_flagged'] == 1)
combined_df['screen_rad_not_flagged'] = combined_df['time_based_screen'] & (combined_df['rad_flagged'] == 0)
combined_df['cancer_rad_flagged'] = (combined_df['has_cancer'] == 1) & (combined_df['rad_flagged'] == 1)
combined_df['cancer_rad_not_flagged'] = (combined_df['has_cancer'] == 1) & (combined_df['rad_flagged'] == 0)


print('Screen-detected cancers:', int(combined_df['time_based_screen'].sum()))


## Level 1 monitoring (AI score shifts)


In [ ]:
# Combined dataset is already constructed as GE -> Philips
print('Combined dataset size:', len(combined_df))
print('Transition point (index):', transition_point)
print('Manufacturers:', combined_df[MANUFACTURER_COL].value_counts().to_dict())


In [ ]:
# ============================================================================
# LEVEL 1 MONITORING: SLIDING WINDOW ANALYSIS (Median, P90, KS)
# ============================================================================

def sliding_window_monitoring(combined_df, ai_columns=AI_COLUMNS,
                             ref_size=REF_SIZE, gap_size=GAP_SIZE, curr_size=CURR_SIZE, step_size=STEP_SIZE,
                             n_permutations=N_PERMUTATIONS, random_state=42):
    """
    Monitor AI score distributions using sliding windows with permutation testing.
    
    Metrics: median, 90th percentile, KS statistic.
    Permutation tests for median and P90; KS uses ks_2samp p-value.
    FDR correction is applied per AI system and metric.
    """
    np.random.seed(random_state)
    total_samples = len(combined_df)
    results = []

    current_start = ref_size + gap_size
    while current_start + curr_size <= total_samples:
        ref_start = current_start - ref_size - gap_size
        ref_end = current_start - gap_size
        ref_window = combined_df.iloc[ref_start:ref_end]
        curr_window = combined_df.iloc[current_start:current_start + curr_size]

        window_center = current_start + curr_size // 2

        for ai in ai_columns:
            ref_scores = ref_window[ai].dropna().values
            curr_scores = curr_window[ai].dropna().values

            if len(ref_scores) < 30 or len(curr_scores) < 30:
                continue

            pooled = np.concatenate([ref_scores, curr_scores])
            n_ref = len(ref_scores)

            result = {'window_center': window_center, 'ai_system': ai}

            # Median monitoring
            ref_median = np.median(ref_scores)
            curr_median = np.median(curr_scores)
            median_diff = curr_median - ref_median

            perm_median_diffs = np.empty(n_permutations)
            for i in range(n_permutations):
                perm_shuffled = np.random.permutation(pooled)
                perm_ref = perm_shuffled[:n_ref]
                perm_curr = perm_shuffled[n_ref:]
                perm_median_diffs[i] = np.median(perm_curr) - np.median(perm_ref)

            median_se = np.std(perm_median_diffs)
            median_z = median_diff / median_se if median_se > 0 else 0
            median_p = np.mean(np.abs(perm_median_diffs) >= np.abs(median_diff))
            if median_p == 0:
                median_p = 1.0 / n_permutations

            result.update({
                'ref_median': ref_median,
                'curr_median': curr_median,
                'median_diff': median_diff,
                'median_z': median_z,
                'median_p': median_p
            })

            # 90th percentile monitoring
            ref_p90 = np.percentile(ref_scores, 90)
            curr_p90 = np.percentile(curr_scores, 90)
            p90_diff = curr_p90 - ref_p90

            perm_p90_diffs = np.empty(n_permutations)
            for i in range(n_permutations):
                perm_shuffled = np.random.permutation(pooled)
                perm_ref = perm_shuffled[:n_ref]
                perm_curr = perm_shuffled[n_ref:]
                perm_p90_diffs[i] = np.percentile(perm_curr, 90) - np.percentile(perm_ref, 90)

            p90_se = np.std(perm_p90_diffs)
            p90_z = p90_diff / p90_se if p90_se > 0 else 0
            p90_p = np.mean(np.abs(perm_p90_diffs) >= np.abs(p90_diff))
            if p90_p == 0:
                p90_p = 1.0 / n_permutations

            result.update({
                'ref_p90': ref_p90,
                'curr_p90': curr_p90,
                'p90_diff': p90_diff,
                'p90_z': p90_z,
                'p90_p': p90_p
            })

            # KS test
            ks_stat, ks_p = ks_2samp(ref_scores, curr_scores)
            result.update({'ks_stat': ks_stat, 'ks_p': ks_p})

            results.append(result)

        current_start += step_size

    results_df = pd.DataFrame(results)

    if results_df.empty or 'ai_system' not in results_df.columns:
        return pd.DataFrame(columns=['window_center', 'ai_system', 'curr_median', 'curr_p90', 'ks_stat', 'median_p', 'p90_p', 'ks_p', 'median_p_fdr', 'p90_p_fdr', 'ks_p_fdr'])

    # FDR correction per AI system and metric
    for ai in ai_columns:
        ai_mask = results_df['ai_system'] == ai
        for metric in ['median', 'p90', 'ks']:
            p_col = f'{metric}_p'
            if p_col in results_df.columns:
                p_vals = results_df.loc[ai_mask, p_col].values
                if len(p_vals) == 0:
                    continue
                _, p_corrected, _, _ = multipletests(p_vals, method='fdr_bh')
                results_df.loc[ai_mask, f'{metric}_p_fdr'] = p_corrected

    return results_df


In [ ]:
# ============================================================================
# VISUALIZATION: LEVEL 1 MONITORING RESULTS
# ============================================================================

def _standardize_series(series, mean, std):
    if pd.isna(std) or std <= 0:
        return pd.Series(np.zeros(len(series)), index=series.index)
    return (series - mean) / std


def summarize_monitoring_results(results_df, transition_point, alpha=0.05):
    '''
    Summarize pre/post signal counts and pre/post shifts for each metric and AI system.
    '''
    rows = []
    for ai in AI_COLUMNS:
        ai_data = results_df[results_df['ai_system'] == ai].copy()
        for metric in ['median', 'p90', 'ks']:
            if metric == 'ks':
                p_col = 'ks_p_fdr'
                value_col = 'ks_stat'
            else:
                p_col = f'{metric}_p_fdr'
                value_col = f'curr_{metric}'

            if p_col not in ai_data.columns:
                continue

            sig_mask = ai_data[p_col] < alpha
            tp_count = int(((ai_data['window_center'] >= transition_point) & sig_mask).sum())
            fp_count = int(((ai_data['window_center'] < transition_point) & sig_mask).sum())

            before_vals = ai_data.loc[ai_data['window_center'] < transition_point, value_col]
            after_vals = ai_data.loc[ai_data['window_center'] >= transition_point, value_col]
            before_mean = before_vals.mean() if len(before_vals) else float('nan')
            after_mean = after_vals.mean() if len(after_vals) else float('nan')

            if pd.notna(before_mean) and pd.notna(after_mean):
                change = after_mean - before_mean
                pct_change = (change / before_mean * 100) if before_mean != 0 else float('nan')
            else:
                change = float('nan')
                pct_change = float('nan')

            rows.append({
                'ai_system': ai,
                'metric': metric,
                'tp_count': tp_count,
                'fp_count': fp_count,
                'before_mean': before_mean,
                'after_mean': after_mean,
                'abs_change': change,
                'pct_change': pct_change
            })

    return pd.DataFrame(rows)


def plot_monitoring_results(results_df, transition_point, metric='median', alpha=0.05, title=None,
                            figsize=(14, 16), plot_units='absolute', z_ylim='data'):
    '''
    Visualize Level 1 monitoring results for a specific metric.

    plot_units: 'absolute' or 'zscore'
    z_ylim: 'data' for global min/max across all panels, tuple for fixed, or None
    '''
    ai_systems = AI_COLUMNS
    colors = {'lunit': '#ff7f0e', 'therapixel': '#2ca02c', 'vara': '#d62728'}

    metric_labels = {
        'median': 'Median Score',
        'p90': '90th Percentile Score',
        'ks': 'KS Statistic'
    }

    use_z = plot_units == 'zscore'
    title_metric = metric_labels[metric]
    if use_z:
        title_metric = f'{title_metric} (Z Score)'

    fig, axes = plt.subplots(len(ai_systems), 1, figsize=figsize, sharex=True)
    if len(ai_systems) == 1:
        axes = [axes]

    all_values = []

    for idx, ai in enumerate(ai_systems):
        ax = axes[idx]
        ai_data = results_df[results_df['ai_system'] == ai].copy()
        color = colors.get(ai, '#333333')

        before_transition = ai_data[ai_data['window_center'] < transition_point]
        after_transition = ai_data[ai_data['window_center'] >= transition_point]

        # Add shaded regions for GE and Philips
        x_min = ai_data['window_center'].min() if len(ai_data) > 0 else 0
        x_max = ai_data['window_center'].max() if len(ai_data) > 0 else transition_point * 2
        ax.axvspan(x_min, transition_point, alpha=0.1, color='blue', label='GE')
        ax.axvspan(transition_point, x_max, alpha=0.1, color='orange', label='Philips')

        if metric == 'ks':
            value_series = ai_data['ks_stat']
            if use_z:
                baseline_vals = before_transition['ks_stat'] if len(before_transition) else value_series
                baseline_mean = baseline_vals.mean()
                baseline_std = baseline_vals.std()
                value_series = _standardize_series(value_series, baseline_mean, baseline_std)

            ax.plot(ai_data['window_center'], value_series,
                   linewidth=2.5, alpha=0.9, color=color, label='KS Statistic')
            sig_mask = ai_data['ks_p_fdr'] < alpha
            y_vals = value_series
            all_values.append(value_series)
        else:
            ref_col = f'ref_{metric}'
            curr_col = f'curr_{metric}'

            ref_series = ai_data[ref_col]
            value_series = ai_data[curr_col]

            if use_z:
                baseline_vals = before_transition[curr_col] if len(before_transition) else value_series
                baseline_mean = baseline_vals.mean()
                baseline_std = baseline_vals.std()
                ref_series = _standardize_series(ref_series, baseline_mean, baseline_std)
                value_series = _standardize_series(value_series, baseline_mean, baseline_std)

            ax.plot(ai_data['window_center'], ref_series,
                   linewidth=2, alpha=0.6, linestyle='--', color=color,
                   label='Prior Window')

            ax.plot(ai_data['window_center'], value_series,
                   linewidth=2.5, alpha=0.9, color=color,
                   label='Current Window')

            sig_mask = ai_data[f'{metric}_p_fdr'] < alpha
            y_vals = value_series
            all_values.extend([ref_series, value_series])

        change_label = None
        if len(before_transition) > 0 and len(after_transition) > 0:
            before_mean = y_vals[ai_data['window_center'] < transition_point].mean()
            after_mean = y_vals[ai_data['window_center'] >= transition_point].mean()
            change = after_mean - before_mean

            if use_z:
                change_label = f'Delta (before→after): {change:+.2f} z'
            else:
                if abs(before_mean) > 1e-12:
                    pct_change = (change / before_mean * 100)
                    rel_str = f'{pct_change:+.1f}%'
                else:
                    rel_str = 'n/a'
                change_label = (
                    f'Change (before→after transition):'
                    f'Absolute: {change:+.4f} | Relative: {rel_str}'
                )

        if sig_mask.any():
            sig_data = ai_data[sig_mask].copy()
            tp_mask = sig_data['window_center'] >= transition_point
            if tp_mask.any():
                ax.scatter(sig_data.loc[tp_mask, 'window_center'],
                           y_vals[sig_mask][tp_mask],
                           color='green', marker='^', s=50, alpha=0.7,
                           label='True Positive', zorder=5)
            fp_mask = sig_data['window_center'] < transition_point
            if fp_mask.any():
                ax.scatter(sig_data.loc[fp_mask, 'window_center'],
                           y_vals[sig_mask][fp_mask],
                           color='red', marker='x', s=100, linewidths=2,
                           label='False Positive', zorder=5)

        ax.axvline(transition_point, color='red', linestyle='--',
                  linewidth=2, label='Transition', zorder=4)

        if change_label:
            ax.text(0.98, 0.02,
                    change_label,
                    transform=ax.transAxes, fontsize=9,
                    verticalalignment='bottom', horizontalalignment='right',
                    bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.7))

        if use_z:
            ax.axhline(0, color='black', linewidth=1, alpha=0.2)

        ax.set_ylabel('Z Score' if use_z else metric_labels[metric], fontsize=11)
        ax.set_title(f'{ai.upper()}', fontsize=12, fontweight='bold')
        ax.legend(fontsize=9, loc='upper left')
        ax.grid(True, alpha=0.3)

        if idx == len(ai_systems) - 1:
            ax.set_xlabel('Current Window Center (Exam Index)', fontsize=11)

    if use_z:
        z_ylim_final = None
        if z_ylim == 'data':
            data = []
            for series in all_values:
                if len(series) == 0:
                    continue
                vals = np.asarray(series, dtype=float)
                vals = vals[np.isfinite(vals)]
                if len(vals):
                    data.append(vals)
            if data:
                stacked = np.concatenate(data)
                y_min = float(np.min(stacked))
                y_max = float(np.max(stacked))
                y_min = min(y_min, 0.0)
                y_max = max(y_max, 0.0)
                pad = 0.1 * (y_max - y_min) if y_max > y_min else 1.0
                z_ylim_final = (y_min - pad, y_max + pad)
        elif z_ylim is not None:
            z_ylim_final = z_ylim

        if z_ylim_final is not None:
            for ax in axes:
                ax.set_ylim(z_ylim_final)

    default_title = (
        f'Level 1 Monitoring: {title_metric}'
        f'Sliding Window (Ref: {REF_SIZE}, Curr: {CURR_SIZE}, Gap: {GAP_SIZE}, Step: {STEP_SIZE})'
    )
    plt.suptitle(
        title or default_title,
        fontsize=15, fontweight='bold', y=0.995
    )
    plt.tight_layout()
    plt.show()



In [ ]:
# ============================================================================
# RUN LEVEL 1 MONITORING
# ============================================================================

print('Running Level 1 Monitoring (Median, P90, KS)...')
monitoring_results = sliding_window_monitoring(combined_df)
print('Total windows analyzed:', len(monitoring_results) // len(AI_COLUMNS))

# Plot key metrics
plot_monitoring_results(monitoring_results, transition_point, metric='median')
plot_monitoring_results(monitoring_results, transition_point, metric='p90')
plot_monitoring_results(monitoring_results, transition_point, metric='ks')


In [ ]:
# ============================================================================
# COMPARISON: Z-SCORE PLOTS + SUMMARY TABLE
# ============================================================================

from pathlib import Path

output_dir = Path('outputs/2025_12_29_counterfactual_manufacturer_swap_setup')
output_dir.mkdir(parents=True, exist_ok=True)

plot_monitoring_results(monitoring_results, transition_point, metric='median', plot_units='zscore')
plot_monitoring_results(monitoring_results, transition_point, metric='p90', plot_units='zscore')
plot_monitoring_results(monitoring_results, transition_point, metric='ks', plot_units='zscore')

summary_df = summarize_monitoring_results(monitoring_results, transition_point, alpha=0.05)
summary_df

# Render summary table as an image with two-decimal formatting and styled header
display_df = summary_df.copy()
int_cols = ['tp_count', 'fp_count']
for col in int_cols:
    display_df[col] = display_df[col].apply(lambda x: '' if pd.isna(x) else str(int(x)))

for col in display_df.columns:
    if col in ['ai_system', 'metric'] + int_cols:
        continue
    display_df[col] = pd.to_numeric(display_df[col], errors='coerce')
    display_df[col] = display_df[col].map(lambda x: f"{x:.2f}" if pd.notna(x) else '')

fig_height = 1.6 + 0.5 * len(display_df)
fig, ax = plt.subplots(figsize=(14, fig_height))
ax.axis('off')

table = ax.table(
    cellText=display_df.values,
    colLabels=display_df.columns,
    cellLoc='center',
    loc='center',
    bbox=[0, 0, 1, 1]
)

col_widths = [0.14, 0.12, 0.09, 0.09, 0.14, 0.14, 0.14, 0.14]
for col_idx, width in enumerate(col_widths):
    for (row, col), cell in table.get_celld().items():
        if col == col_idx:
            cell.set_width(width)

table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 1.8)

header_color = '#4472C4'
alt_row_color = '#E7E6E6'

# Style header
for i in range(len(display_df.columns)):
    cell = table[(0, i)]
    cell.set_facecolor(header_color)
    cell.set_text_props(weight='bold', color='white')

# Alternate row colors
for i in range(1, len(display_df) + 1):
    for j in range(len(display_df.columns)):
        cell = table[(i, j)]
        if i % 2 == 0:
            cell.set_facecolor(alt_row_color)
        else:
            cell.set_facecolor('white')

# Borders
for cell in table.get_celld().values():
    cell.set_edgecolor('black')
    cell.set_linewidth(1.2)

plt.title('Level 1 Monitoring Summary Table', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
table_path = output_dir / 'summary_table.png'
plt.savefig(table_path, dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
table_path


## Level 2 monitoring and parameter plots


In [ ]:
# Helper: sliding window monitoring for score distributions within a subset
def sliding_window_monitoring_subset(combined_df, subset_col, subset_value=True, subset_label=None,
                                    ai_columns=AI_COLUMNS, ref_size=REF_SIZE, gap_size=GAP_SIZE,
                                    curr_size=CURR_SIZE, step_size=STEP_SIZE, n_permutations=N_PERMUTATIONS,
                                    random_state=42, min_samples=30):
    np.random.seed(random_state)
    total_samples = len(combined_df)
    results = []
    current_start = ref_size + gap_size

    while current_start + curr_size <= total_samples:
        ref_start = current_start - ref_size - gap_size
        ref_end = current_start - gap_size
        ref_window = combined_df.iloc[ref_start:ref_end]
        curr_window = combined_df.iloc[current_start:current_start + curr_size]

        ref_window = ref_window[ref_window[subset_col] == subset_value]
        curr_window = curr_window[curr_window[subset_col] == subset_value]

        window_center = current_start + curr_size // 2

        for ai in ai_columns:
            ref_scores = ref_window[ai].dropna().values
            curr_scores = curr_window[ai].dropna().values

            if len(ref_scores) < min_samples or len(curr_scores) < min_samples:
                continue

            pooled = np.concatenate([ref_scores, curr_scores])
            n_ref = len(ref_scores)

            result = {
                'window_center': window_center,
                'ai_system': ai,
                'subset': subset_label or subset_col
            }

            # Median monitoring
            ref_median = np.median(ref_scores)
            curr_median = np.median(curr_scores)
            median_diff = curr_median - ref_median

            perm_median_diffs = np.empty(n_permutations)
            for i in range(n_permutations):
                perm_shuffled = np.random.permutation(pooled)
                perm_ref = perm_shuffled[:n_ref]
                perm_curr = perm_shuffled[n_ref:]
                perm_median_diffs[i] = np.median(perm_curr) - np.median(perm_ref)

            median_se = np.std(perm_median_diffs)
            median_z = median_diff / median_se if median_se > 0 else 0
            median_p = np.mean(np.abs(perm_median_diffs) >= np.abs(median_diff))
            if median_p == 0:
                median_p = 1.0 / n_permutations

            result.update({
                'ref_median': ref_median,
                'curr_median': curr_median,
                'median_diff': median_diff,
                'median_z': median_z,
                'median_p': median_p
            })

            # 90th percentile monitoring
            ref_p90 = np.percentile(ref_scores, 90)
            curr_p90 = np.percentile(curr_scores, 90)
            p90_diff = curr_p90 - ref_p90

            perm_p90_diffs = np.empty(n_permutations)
            for i in range(n_permutations):
                perm_shuffled = np.random.permutation(pooled)
                perm_ref = perm_shuffled[:n_ref]
                perm_curr = perm_shuffled[n_ref:]
                perm_p90_diffs[i] = np.percentile(perm_curr, 90) - np.percentile(perm_ref, 90)

            p90_se = np.std(perm_p90_diffs)
            p90_z = p90_diff / p90_se if p90_se > 0 else 0
            p90_p = np.mean(np.abs(perm_p90_diffs) >= np.abs(p90_diff))
            if p90_p == 0:
                p90_p = 1.0 / n_permutations

            result.update({
                'ref_p90': ref_p90,
                'curr_p90': curr_p90,
                'p90_diff': p90_diff,
                'p90_z': p90_z,
                'p90_p': p90_p
            })

            # KS test
            ks_stat, ks_p = ks_2samp(ref_scores, curr_scores)
            result.update({'ks_stat': ks_stat, 'ks_p': ks_p})

            results.append(result)

        current_start += step_size

    results_df = pd.DataFrame(results)
    if results_df.empty:
        return results_df

    for ai in ai_columns:
        ai_mask = results_df['ai_system'] == ai
        for metric in ['median', 'p90', 'ks']:
            p_col = f'{metric}_p'
            if p_col in results_df.columns:
                p_vals = results_df.loc[ai_mask, p_col].values
                if len(p_vals) == 0:
                    continue
                _, p_corrected, _, _ = multipletests(p_vals, method='fdr_bh')
                results_df.loc[ai_mask, f'{metric}_p_fdr'] = p_corrected

    return results_df


def compute_flagging_rates(df, subset_col=None, subset_value=True,
                           ref_size=REF_SIZE, gap_size=GAP_SIZE, curr_size=CURR_SIZE, step_size=STEP_SIZE):
    if 'rad_flagged' not in df.columns:
        df = df.copy()
        df['rad_flagged'] = compute_radiologist_flag(df, mode='combined')
    results = {
        'radiologist': [],
        'lunit': [],
        'therapixel': [],
        'vara': []
    }

    total_span = ref_size + gap_size + curr_size
    for ref_start in range(0, len(df) - total_span + 1, step_size):
        ref_end = ref_start + ref_size
        curr_start = ref_end + gap_size
        curr_end = curr_start + curr_size

        ref_window = df.iloc[ref_start:ref_end]
        curr_window = df.iloc[curr_start:curr_end]

        if subset_col is not None:
            ref_window = ref_window[ref_window[subset_col] == subset_value]
            curr_window = curr_window[curr_window[subset_col] == subset_value]

        curr_center = (curr_start + curr_end) / 2

        # Radiologist flagging rates
        ref_n = len(ref_window)
        curr_n = len(curr_window)

        if ref_n > 0:
            ref_rad_flagged = ref_window['rad_flagged'].sum()
            ref_rad_pct = (ref_rad_flagged / ref_n * 100)
        else:
            ref_rad_flagged = 0
            ref_rad_pct = np.nan

        if curr_n > 0:
            curr_rad_flagged = curr_window['rad_flagged'].sum()
            curr_rad_pct = (curr_rad_flagged / curr_n * 100)
        else:
            curr_rad_flagged = 0
            curr_rad_pct = np.nan

        if ref_n > 0 and curr_n > 0:
            count = np.array([curr_rad_flagged, ref_rad_flagged])
            nobs = np.array([curr_n, ref_n])
            _, p_val_rad = proportions_ztest(count, nobs, alternative='two-sided')
        else:
            p_val_rad = 1.0

        results['radiologist'].append({
            'curr_center': curr_center,
            'ref_pct': ref_rad_pct,
            'curr_pct': curr_rad_pct,
            'p_value': p_val_rad
        })

        for ai_name in AI_COLUMNS:
            flag_col = f'{ai_name}_flagged'

            if ref_n > 0:
                ref_ai_flagged = ref_window[flag_col].sum()
                ref_ai_pct = (ref_ai_flagged / ref_n * 100)
            else:
                ref_ai_flagged = 0
                ref_ai_pct = np.nan

            if curr_n > 0:
                curr_ai_flagged = curr_window[flag_col].sum()
                curr_ai_pct = (curr_ai_flagged / curr_n * 100)
            else:
                curr_ai_flagged = 0
                curr_ai_pct = np.nan

            if ref_n > 0 and curr_n > 0:
                count = np.array([curr_ai_flagged, ref_ai_flagged])
                nobs = np.array([curr_n, ref_n])
                _, p_val_ai = proportions_ztest(count, nobs, alternative='two-sided')
            else:
                p_val_ai = 1.0

            results[ai_name].append({
                'curr_center': curr_center,
                'ref_pct': ref_ai_pct,
                'curr_pct': curr_ai_pct,
                'p_value': p_val_ai
            })

    # Apply FDR correction
    for system in results.keys():
        results[system] = pd.DataFrame(results[system])
        if len(results[system]) > 0:
            _, p_corrected, _, _ = multipletests(results[system]['p_value'], method='fdr_bh')
            results[system]['p_value_corrected'] = p_corrected
            results[system]['is_significant'] = p_corrected < 0.05

    return results

def compute_relative_flagging_rates(flagging_results):
    if flagging_results is None:
        return {}
    rad_data = flagging_results.get('radiologist')
    if rad_data is None or len(rad_data) == 0:
        return {}
    results = {}
    for ai in AI_COLUMNS:
        ai_data = flagging_results.get(ai)
        if ai_data is None or len(ai_data) == 0:
            continue
        merged = ai_data[['curr_center', 'curr_pct', 'p_value_corrected']].merge(
            rad_data[['curr_center', 'curr_pct']], on='curr_center', how='left', suffixes=('', '_rad')
        )
        merged['curr_rel_rate'] = merged['curr_pct'] / merged['curr_pct_rad']
        merged.loc[merged['curr_pct_rad'] <= 0, 'curr_rel_rate'] = np.nan
        results[ai] = merged
    return results

def plot_flagging_rates(flagging_results, transition_point, title='Flagging Rate (AI / Radiologist)',
                        alpha=0.05, figsize=(14, 12), normalize_by_radiologist=False,
                        rad_baseline=None, uniform_y=True):
    fig, axes = plt.subplots(3, 1, figsize=figsize, sharex=True)
    colors = {'lunit': '#ff7f0e', 'therapixel': '#2ca02c', 'vara': '#d62728'}

    rad_data = flagging_results.get('radiologist') if flagging_results is not None else None

    y_min = None
    y_max = None
    if uniform_y and flagging_results is not None:
        for ai in AI_COLUMNS:
            ai_data = flagging_results.get(ai)
            if ai_data is None or len(ai_data) == 0:
                continue

            plot_vals = ai_data['curr_pct']
            if normalize_by_radiologist and rad_data is not None and len(rad_data) > 0:
                merged = ai_data[['curr_center', 'curr_pct', 'p_value_corrected']].merge(
                    rad_data[['curr_center', 'curr_pct']], on='curr_center', how='left', suffixes=('', '_rad')
                )
                denom = merged['curr_pct_rad']
                valid = denom > 0
                if valid.any():
                    plot_vals = merged.loc[valid, 'curr_pct'] / denom[valid]
                else:
                    continue

            if len(plot_vals) > 0:
                y_min = plot_vals.min() if y_min is None else min(y_min, plot_vals.min())
                y_max = plot_vals.max() if y_max is None else max(y_max, plot_vals.max())

        if not normalize_by_radiologist and rad_data is not None and len(rad_data) > 0:
            y_min = rad_data['curr_pct'].min() if y_min is None else min(y_min, rad_data['curr_pct'].min())
            y_max = rad_data['curr_pct'].max() if y_max is None else max(y_max, rad_data['curr_pct'].max())

    for idx, ai in enumerate(AI_COLUMNS):
        ax = axes[idx]
        ai_data = flagging_results.get(ai) if flagging_results is not None else None

        if ai_data is None or len(ai_data) == 0:
            ax.axvspan(0, transition_point, alpha=0.1, color='blue', label='GE')
            ax.axvspan(transition_point, transition_point * 2, alpha=0.1, color='orange', label='Philips')
            ax.text(0.5, 0.5, 'No data', ha='center', va='center')
            ax.axvline(transition_point, color='red', linestyle='--', linewidth=2, label='Transition')
            ax.set_ylabel('Flagging rate (%)')
            ax.set_title(ai.upper())
            ax.grid(True, alpha=0.3)
            ax.legend(fontsize=9, loc='upper left')
            continue

        x_min = ai_data['curr_center'].min() if len(ai_data) > 0 else 0
        x_max = ai_data['curr_center'].max() if len(ai_data) > 0 else transition_point * 2
        ax.axvspan(x_min, transition_point, alpha=0.1, color='blue', label='GE')
        ax.axvspan(transition_point, x_max, alpha=0.1, color='orange', label='Philips')

        plot_vals = ai_data['curr_pct']
        rad_plot_vals = None
        label_suffix = ''
        normalization_note = False

        if normalize_by_radiologist and rad_data is not None and len(rad_data) > 0:
            merged = ai_data[['curr_center', 'curr_pct', 'p_value_corrected']].merge(
                rad_data[['curr_center', 'curr_pct']], on='curr_center', how='left', suffixes=('', '_rad')
            )

            denom = merged['curr_pct_rad']
            valid = denom > 0
            if valid.any():
                plot_vals = merged['curr_pct'] / denom
                rad_plot_vals = denom / denom
                label_suffix = ' (x Rad baseline)'
            else:
                normalization_note = True

        ax.plot(ai_data['curr_center'], plot_vals, color=colors.get(ai, '#333333'), linewidth=2, label=ai.upper())
        if rad_data is not None and len(rad_data) > 0:
            if rad_plot_vals is None:
                ax.plot(rad_data['curr_center'], rad_data['curr_pct'], color='gray', linestyle='--', linewidth=1.5, label='Radiologist')
            else:
                ax.plot(rad_data['curr_center'], rad_plot_vals, color='gray', linestyle='--', linewidth=1.5, label='Radiologist')

        sig_mask = ai_data['p_value_corrected'] < alpha if 'p_value_corrected' in ai_data.columns else pd.Series([False] * len(ai_data))
        if sig_mask.any():
            ax.scatter(ai_data.loc[sig_mask, 'curr_center'],
                       plot_vals[sig_mask], color='black', s=40, label='Signal')

        if normalization_note:
            ax.text(0.98, 0.98, 'Normalization unavailable; showing raw %',
                    transform=ax.transAxes, fontsize=8, color='gray',
                    verticalalignment='top', horizontalalignment='right')

        ax.axvline(transition_point, color='red', linestyle='--', linewidth=2, label='Transition')
        ax.set_ylabel('Flagging rate (%)' + label_suffix)
        ax.set_title(ai.upper())
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=9, loc='upper left')

    if uniform_y and y_min is not None and y_max is not None:
        span = max(1.0, y_max - y_min)
        pad = 0.05 * span
        for ax in axes:
            ax.set_ylim(y_min - pad, y_max + pad)

    axes[-1].set_xlabel('Current Window Center (Exam Index)')
    plt.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()


def compute_conditional_overlap(df, ai_columns=AI_COLUMNS, subset_col=None, subset_value=True,
                                ref_size=REF_SIZE, gap_size=GAP_SIZE, curr_size=CURR_SIZE, step_size=STEP_SIZE):
    overlap_results = {}
    total_span = ref_size + gap_size + curr_size

    for ai_name in ai_columns:
        rows = []
        for ref_start in range(0, len(df) - total_span + 1, step_size):
            ref_end = ref_start + ref_size
            curr_start = ref_end + gap_size
            curr_end = curr_start + curr_size

            ref_window = df.iloc[ref_start:ref_end]
            curr_window = df.iloc[curr_start:curr_end]

            if subset_col is not None:
                ref_window = ref_window[ref_window[subset_col] == subset_value]
                curr_window = curr_window[curr_window[subset_col] == subset_value]

            curr_center = (curr_start + curr_end) / 2

            ref_ai_flagged = ref_window[f'{ai_name}_flagged'].sum()
            ref_rad_flagged = ref_window['rad_flagged'].sum()
            ref_both = (ref_window[f'{ai_name}_flagged'] & ref_window['rad_flagged']).sum()

            curr_ai_flagged = curr_window[f'{ai_name}_flagged'].sum()
            curr_rad_flagged = curr_window['rad_flagged'].sum()
            curr_both = (curr_window[f'{ai_name}_flagged'] & curr_window['rad_flagged']).sum()

            ref_p_ai_given_rad = (ref_both / ref_rad_flagged * 100) if ref_rad_flagged > 0 else np.nan
            curr_p_ai_given_rad = (curr_both / curr_rad_flagged * 100) if curr_rad_flagged > 0 else np.nan

            ref_p_rad_given_ai = (ref_both / ref_ai_flagged * 100) if ref_ai_flagged > 0 else np.nan
            curr_p_rad_given_ai = (curr_both / curr_ai_flagged * 100) if curr_ai_flagged > 0 else np.nan

            if ref_rad_flagged > 0 and curr_rad_flagged > 0:
                count = np.array([curr_both, ref_both])
                nobs = np.array([curr_rad_flagged, ref_rad_flagged])
                _, p_val_ai_given_rad = proportions_ztest(count, nobs, alternative='two-sided')
            else:
                p_val_ai_given_rad = 1.0

            if ref_ai_flagged > 0 and curr_ai_flagged > 0:
                count = np.array([curr_both, ref_both])
                nobs = np.array([curr_ai_flagged, ref_ai_flagged])
                _, p_val_rad_given_ai = proportions_ztest(count, nobs, alternative='two-sided')
            else:
                p_val_rad_given_ai = 1.0

            rows.append({
                'curr_center': curr_center,
                'ref_p_ai_given_rad': ref_p_ai_given_rad,
                'curr_p_ai_given_rad': curr_p_ai_given_rad,
                'ref_p_rad_given_ai': ref_p_rad_given_ai,
                'curr_p_rad_given_ai': curr_p_rad_given_ai,
                'p_val_ai_given_rad': p_val_ai_given_rad,
                'p_val_rad_given_ai': p_val_rad_given_ai
            })

        df_results = pd.DataFrame(rows)
        if len(df_results) > 0:
            _, p_corr_ai_rad, _, _ = multipletests(df_results['p_val_ai_given_rad'], method='fdr_bh')
            _, p_corr_rad_ai, _, _ = multipletests(df_results['p_val_rad_given_ai'], method='fdr_bh')
            df_results['p_val_ai_given_rad_corrected'] = p_corr_ai_rad
            df_results['p_val_rad_given_ai_corrected'] = p_corr_rad_ai
            df_results['sig_ai_given_rad'] = p_corr_ai_rad < 0.05
            df_results['sig_rad_given_ai'] = p_corr_rad_ai < 0.05

        overlap_results[ai_name] = df_results

    return overlap_results




def compute_sensitivity_windows(df, ai_columns=AI_COLUMNS,
                               ref_size=REF_SIZE, gap_size=GAP_SIZE, curr_size=CURR_SIZE, step_size=STEP_SIZE,
                               subset_col=None, subset_value=True):
    results = []
    total_span = ref_size + gap_size + curr_size

    for ref_start in range(0, len(df) - total_span + 1, step_size):
        ref_end = ref_start + ref_size
        curr_start = ref_end + gap_size
        curr_end = curr_start + curr_size

        ref_window = df.iloc[ref_start:ref_end]
        curr_window = df.iloc[curr_start:curr_end]

        if subset_col is not None:
            ref_window = ref_window[ref_window[subset_col] == subset_value]
            curr_window = curr_window[curr_window[subset_col] == subset_value]

        curr_center = (curr_start + curr_end) / 2

        ref_cancers = ref_window[ref_window['has_cancer'] == 1]
        curr_cancers = curr_window[curr_window['has_cancer'] == 1]

        for ai in ai_columns:
            ref_n = len(ref_cancers)
            curr_n = len(curr_cancers)

            ref_flagged = (ref_cancers[f'{ai}_flagged'] == 1).sum() if ref_n > 0 else 0
            curr_flagged = (curr_cancers[f'{ai}_flagged'] == 1).sum() if curr_n > 0 else 0

            ref_sens = (ref_flagged / ref_n * 100) if ref_n > 0 else np.nan
            curr_sens = (curr_flagged / curr_n * 100) if curr_n > 0 else np.nan

            if ref_n > 0 and curr_n > 0:
                count = np.array([curr_flagged, ref_flagged])
                nobs = np.array([curr_n, ref_n])
                _, p_val = proportions_ztest(count, nobs, alternative='two-sided')
            else:
                p_val = 1.0

            results.append({
                'curr_center': curr_center,
                'ai_system': ai,
                'ref_sensitivity': ref_sens,
                'curr_sensitivity': curr_sens,
                'p_value': p_val
            })

    results_df = pd.DataFrame(results)
    if len(results_df) == 0:
        return results_df



def compute_specificity_windows(df, ai_columns=AI_COLUMNS,
                               ref_size=REF_SIZE, gap_size=GAP_SIZE, curr_size=CURR_SIZE, step_size=STEP_SIZE,
                               subset_col=None, subset_value=True):
    results = []
    total_span = ref_size + gap_size + curr_size

    for ref_start in range(0, len(df) - total_span + 1, step_size):
        ref_end = ref_start + ref_size
        curr_start = ref_end + gap_size
        curr_end = curr_start + curr_size

        ref_window = df.iloc[ref_start:ref_end]
        curr_window = df.iloc[curr_start:curr_end]

        if subset_col is not None:
            ref_window = ref_window[ref_window[subset_col] == subset_value]
            curr_window = curr_window[curr_window[subset_col] == subset_value]

        curr_center = (curr_start + curr_end) / 2

        ref_controls = ref_window[ref_window['has_cancer'] == 0]
        curr_controls = curr_window[curr_window['has_cancer'] == 0]

        for ai in ai_columns:
            ref_n = len(ref_controls)
            curr_n = len(curr_controls)

            ref_correct = (ref_controls[f'{ai}_flagged'] == 0).sum() if ref_n > 0 else 0
            curr_correct = (curr_controls[f'{ai}_flagged'] == 0).sum() if curr_n > 0 else 0

            ref_spec = (ref_correct / ref_n * 100) if ref_n > 0 else np.nan
            curr_spec = (curr_correct / curr_n * 100) if curr_n > 0 else np.nan

            if ref_n > 0 and curr_n > 0:
                count = np.array([curr_correct, ref_correct])
                nobs = np.array([curr_n, ref_n])
                _, p_val = proportions_ztest(count, nobs, alternative='two-sided')
            else:
                p_val = 1.0

            results.append({
                'curr_center': curr_center,
                'ai_system': ai,
                'ref_specificity': ref_spec,
                'curr_specificity': curr_spec,
                'p_value': p_val
            })

    results_df = pd.DataFrame(results)
    if len(results_df) == 0:
        return results_df

    for ai in ai_columns:
        ai_mask = results_df['ai_system'] == ai
        p_vals = results_df.loc[ai_mask, 'p_value'].values
        if len(p_vals) == 0:
            continue
        _, p_corrected, _, _ = multipletests(p_vals, method='fdr_bh')
        results_df.loc[ai_mask, 'p_value_corrected'] = p_corrected

    return results_df


def plot_sensitivity_results(sensitivity_df, transition_point, title='Sensitivity (AI)', alpha=0.05, figsize=(14, 12)):
    fig, axes = plt.subplots(3, 1, figsize=figsize, sharex=True)
    colors = {'lunit': '#ff7f0e', 'therapixel': '#2ca02c', 'vara': '#d62728'}

    if sensitivity_df is None or len(sensitivity_df) == 0 or 'ai_system' not in sensitivity_df.columns:
        for ax in axes:
            ax.axvspan(0, transition_point, alpha=0.1, color='blue', label='GE')
            ax.axvspan(transition_point, transition_point * 2, alpha=0.1, color='orange', label='Philips')
            ax.text(0.5, 0.5, 'No data', ha='center', va='center')
            ax.axvline(transition_point, color='red', linestyle='--', linewidth=2, label='Transition')
            ax.set_ylabel('Sensitivity (%)')
            ax.grid(True, alpha=0.3)
            ax.legend(fontsize=9, loc='upper left')
        axes[-1].set_xlabel('Current Window Center (Exam Index)')
        plt.suptitle(title, fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.show()
        return

    for idx, ai in enumerate(AI_COLUMNS):
        ax = axes[idx]
        data = sensitivity_df[sensitivity_df['ai_system'] == ai]

        if data is None or len(data) == 0:
            ax.axvspan(0, transition_point, alpha=0.1, color='blue', label='GE')
            ax.axvspan(transition_point, transition_point * 2, alpha=0.1, color='orange', label='Philips')
            ax.text(0.5, 0.5, 'No data', ha='center', va='center')
            ax.axvline(transition_point, color='red', linestyle='--', linewidth=2, label='Transition')
            ax.set_ylabel('Sensitivity (%)')
            ax.set_title(ai.upper())
            ax.grid(True, alpha=0.3)
            ax.legend(fontsize=9, loc='upper left')
            continue

        # Add shaded regions for GE and Philips
        x_min = data['curr_center'].min() if len(data) > 0 else 0
        x_max = data['curr_center'].max() if len(data) > 0 else transition_point * 2
        ax.axvspan(x_min, transition_point, alpha=0.1, color='blue', label='GE')
        ax.axvspan(transition_point, x_max, alpha=0.1, color='orange', label='Philips')

        ax.plot(data['curr_center'], data['curr_sensitivity'], color=colors[ai], linewidth=2, label=ai.upper())

        sig_mask = data['p_value_corrected'] < alpha if 'p_value_corrected' in data.columns else pd.Series([False]*len(data))
        if sig_mask.any():
            sig_data = data[sig_mask]
            ax.scatter(sig_data['curr_center'], sig_data['curr_sensitivity'], color='black', s=40, label='Signal')

        if data['curr_sensitivity'].isna().any():
            ax.text(0.98, 0.98, 'Gaps = no cancers in window',
                    transform=ax.transAxes, fontsize=8, color='gray',
                    verticalalignment='top', horizontalalignment='right')

        ax.axvline(transition_point, color='red', linestyle='--', linewidth=2, label='Transition')
        ax.set_ylabel('Sensitivity (%)')
        ax.set_title(ai.upper())
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=9, loc='upper left')

    axes[-1].set_xlabel('Current Window Center (Exam Index)')
    plt.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()


def plot_conditional_overlap(overlap_results, transition_point, metric='ai_given_rad', title=None, alpha=0.05, figsize=(14, 12), uniform_y=True):
    fig, axes = plt.subplots(3, 1, figsize=figsize, sharex=True)
    colors = {'lunit': '#ff7f0e', 'therapixel': '#2ca02c', 'vara': '#d62728'}

    y_col = 'curr_p_ai_given_rad' if metric == 'ai_given_rad' else 'curr_p_rad_given_ai'
    ylabel = 'P(AI | Rad) (%)' if metric == 'ai_given_rad' else 'P(Rad | AI) (%)'
    p_col = 'p_val_ai_given_rad_corrected' if metric == 'ai_given_rad' else 'p_val_rad_given_ai_corrected'

    y_min = None
    y_max = None
    if uniform_y:
        for ai in AI_COLUMNS:
            data = overlap_results.get(ai)
            if data is None or len(data) == 0 or y_col not in data.columns:
                continue
            vals = data[y_col].dropna()
            if len(vals) == 0:
                continue
            y_min = vals.min() if y_min is None else min(y_min, vals.min())
            y_max = vals.max() if y_max is None else max(y_max, vals.max())

    for idx, ai in enumerate(AI_COLUMNS):
        ax = axes[idx]
        data = overlap_results.get(ai)

        # Add shaded regions for GE and Philips
        if data is not None and len(data) > 0:
            x_min = data['curr_center'].min()
            x_max = data['curr_center'].max()
        else:
            x_min = 0
            x_max = transition_point * 2
        ax.axvspan(x_min, transition_point, alpha=0.1, color='blue', label='GE')
        ax.axvspan(transition_point, x_max, alpha=0.1, color='orange', label='Philips')

        if data is None or len(data) == 0:
            ax.text(0.5, 0.5, 'No data', ha='center', va='center')
            ax.axvline(transition_point, color='red', linestyle='--', linewidth=2, label='Transition', zorder=4)
            ax.set_ylabel('Rate (%)')
            ax.set_title(ai.upper())
            ax.grid(True, alpha=0.3)
            continue

        ax.plot(data['curr_center'], data[y_col], color=colors[ai], linewidth=2, label=ai.upper())
        sig_mask = data[p_col] < alpha
        if sig_mask.any():
            sig_data = data[sig_mask]
            ax.scatter(sig_data['curr_center'], sig_data[y_col], color='black', s=40, label='Signal')

        ax.axvline(transition_point, color='red', linestyle='--', linewidth=2, label='Transition', zorder=4)
        ax.set_ylabel(ylabel)
        ax.set_title(ai.upper())
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=9, loc='upper left')

    if uniform_y and y_min is not None and y_max is not None:
        span = max(1.0, y_max - y_min)
        pad = 0.05 * span
        for ax in axes:
            ax.set_ylim(y_min - pad, y_max + pad)

    axes[-1].set_xlabel('Current Window Center (Exam Index)')
    plt.suptitle(title or 'Conditional Overlap', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()


def plot_subset_distribution(subset_a, subset_b, transition_point, metric='median',
                             labels=('Subset A', 'Subset B'), title=None, alpha=0.05, figsize=(14, 12), uniform_y=True):
    fig, axes = plt.subplots(3, 1, figsize=figsize, sharex=True)
    colors = {'a': '#1f77b4', 'b': '#ff7f0e'}
    
    # Check if DataFrames are empty or missing required columns
    if (subset_a is None or subset_b is None or 
        len(subset_a) == 0 or len(subset_b) == 0 or
        'ai_system' not in subset_a.columns or 'ai_system' not in subset_b.columns):
        for ax in axes:
            ax.axvspan(0, transition_point, alpha=0.1, color='blue', label='GE')
            ax.axvspan(transition_point, transition_point * 2, alpha=0.1, color='orange', label='Philips')
            ax.text(0.5, 0.5, 'Insufficient data for screen-detected subset', ha='center', va='center')
            ax.axvline(transition_point, color='red', linestyle='--', linewidth=2, label='Transition', zorder=4)
            ax.set_ylabel(f'{metric} score')
            ax.grid(True, alpha=0.3)
        axes[0].set_title('LUNIT')
        axes[1].set_title('THERAPIXEL')
        axes[2].set_title('VARA')
        axes[-1].set_xlabel('Current Window Center (Exam Index)')
        plt.suptitle(title or 'Subset Distribution', fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.show()
        return

    y_min = None
    y_max = None
    if uniform_y:
        for ai in AI_COLUMNS:
            data_a = subset_a[subset_a['ai_system'] == ai]
            data_b = subset_b[subset_b['ai_system'] == ai]
            for data in (data_a, data_b):
                if data is None or len(data) == 0:
                    continue
                vals = data[f'curr_{metric}'].dropna()
                if len(vals) == 0:
                    continue
                y_min = vals.min() if y_min is None else min(y_min, vals.min())
                y_max = vals.max() if y_max is None else max(y_max, vals.max())

    for idx, ai in enumerate(AI_COLUMNS):
        ax = axes[idx]
        data_a = subset_a[subset_a['ai_system'] == ai]
        data_b = subset_b[subset_b['ai_system'] == ai]

        # Add shaded regions for GE and Philips
        if len(data_a) > 0:
            x_min = data_a['window_center'].min()
            x_max = data_a['window_center'].max()
        elif len(data_b) > 0:
            x_min = data_b['window_center'].min()
            x_max = data_b['window_center'].max()
        else:
            x_min = 0
            x_max = transition_point * 2
        ax.axvspan(x_min, transition_point, alpha=0.1, color='blue', label='GE')
        ax.axvspan(transition_point, x_max, alpha=0.1, color='orange', label='Philips')

        if data_a is None or data_b is None or len(data_a) == 0 or len(data_b) == 0:
            ax.text(0.5, 0.5, 'No data', ha='center', va='center')
            ax.axvline(transition_point, color='red', linestyle='--', linewidth=2, label='Transition', zorder=4)
            ax.set_ylabel(f'{metric} score')
            ax.set_title(ai.upper())
            ax.grid(True, alpha=0.3)
            continue

        ax.plot(data_a['window_center'], data_a[f'curr_{metric}'], color=colors['a'], linewidth=2, label=labels[0])
        ax.plot(data_b['window_center'], data_b[f'curr_{metric}'], color=colors['b'], linewidth=2, label=labels[1])

        sig_a = data_a[f'{metric}_p_fdr'] < alpha if f'{metric}_p_fdr' in data_a.columns else pd.Series([False]*len(data_a))
        sig_b = data_b[f'{metric}_p_fdr'] < alpha if f'{metric}_p_fdr' in data_b.columns else pd.Series([False]*len(data_b))
        if sig_a.any():
            ax.scatter(data_a.loc[sig_a, 'window_center'], data_a.loc[sig_a, f'curr_{metric}'],
                       color=colors['a'], marker='^', s=40, label=f'{labels[0]} signal')
        if sig_b.any():
            ax.scatter(data_b.loc[sig_b, 'window_center'], data_b.loc[sig_b, f'curr_{metric}'],
                       color=colors['b'], marker='s', s=40, label=f'{labels[1]} signal')

        ax.axvline(transition_point, color='red', linestyle='--', linewidth=2, label='Transition', zorder=4)
        ax.set_ylabel(f'{metric} score')
        ax.set_title(ai.upper())
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=8, loc='upper left')

    if uniform_y and y_min is not None and y_max is not None:
        span = max(1.0, y_max - y_min)
        pad = 0.05 * span
        for ax in axes:
            ax.set_ylim(y_min - pad, y_max + pad)

    axes[-1].set_xlabel('Current Window Center (Exam Index)')
    plt.suptitle(title or 'Subset Distribution', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()


def plot_single_subset_distribution(subset_data, transition_point, metric='median', label='Subset',
                                    title=None, alpha=0.05, figsize=(14, 12), uniform_y=True):
    fig, axes = plt.subplots(3, 1, figsize=figsize, sharex=True)
    colors = {'lunit': '#ff7f0e', 'therapixel': '#2ca02c', 'vara': '#d62728'}

    if subset_data is None or len(subset_data) == 0 or 'ai_system' not in subset_data.columns:
        for ax in axes:
            ax.axvspan(0, transition_point, alpha=0.1, color='blue', label='GE')
            ax.axvspan(transition_point, transition_point * 2, alpha=0.1, color='orange', label='Philips')
            ax.text(0.5, 0.5, 'No data', ha='center', va='center')
            ax.axvline(transition_point, color='red', linestyle='--', linewidth=2, label='Transition', zorder=4)
            ax.set_ylabel(f'{metric} score')
            ax.grid(True, alpha=0.3)
            ax.legend(fontsize=9, loc='upper left')
        axes[-1].set_xlabel('Current Window Center (Exam Index)')
        plt.suptitle(title or f'{label} distribution', fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.show()
        return

    y_min = None
    y_max = None
    if uniform_y:
        for ai in AI_COLUMNS:
            data = subset_data[subset_data['ai_system'] == ai]
            if data is None or len(data) == 0:
                continue
            vals = data[f'curr_{metric}'].dropna()
            if len(vals) == 0:
                continue
            y_min = vals.min() if y_min is None else min(y_min, vals.min())
            y_max = vals.max() if y_max is None else max(y_max, vals.max())

    for idx, ai in enumerate(AI_COLUMNS):
        ax = axes[idx]
        data = subset_data[subset_data['ai_system'] == ai]

        # Add shaded regions for GE and Philips
        if data is not None and len(data) > 0:
            x_min = data['window_center'].min()
            x_max = data['window_center'].max()
        else:
            x_min = 0
            x_max = transition_point * 2
        ax.axvspan(x_min, transition_point, alpha=0.1, color='blue', label='GE')
        ax.axvspan(transition_point, x_max, alpha=0.1, color='orange', label='Philips')

        if data is None or len(data) == 0:
            ax.text(0.5, 0.5, 'No data', ha='center', va='center')
            ax.axvline(transition_point, color='red', linestyle='--', linewidth=2, label='Transition', zorder=4)
            ax.set_ylabel(f'{metric} score')
            ax.set_title(ai.upper())
            ax.grid(True, alpha=0.3)
            ax.legend(fontsize=9, loc='upper left')
            continue

        ax.plot(data['window_center'], data[f'curr_{metric}'], color=colors.get(ai, '#333333'),
                linewidth=2, label=label)

        sig_mask = data[f'{metric}_p_fdr'] < alpha if f'{metric}_p_fdr' in data.columns else pd.Series([False]*len(data))
        if sig_mask.any():
            ax.scatter(data.loc[sig_mask, 'window_center'], data.loc[sig_mask, f'curr_{metric}'],
                       color='black', s=40, label='Signal')

        ax.axvline(transition_point, color='red', linestyle='--', linewidth=2, label='Transition', zorder=4)
        ax.set_ylabel(f'{metric} score')
        ax.set_title(ai.upper())
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=9, loc='upper left')

    if uniform_y and y_min is not None and y_max is not None:
        span = max(1.0, y_max - y_min)
        pad = 0.05 * span
        for ax in axes:
            ax.set_ylim(y_min - pad, y_max + pad)

    axes[-1].set_xlabel('Current Window Center (Exam Index)')
    plt.suptitle(title or f'{label} distribution', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()


In [ ]:
# Compute monitoring parameters beyond Level 1
flagging_overall = compute_flagging_rates(combined_df)
relative_flagging_overall = compute_relative_flagging_rates(flagging_overall)
flagging_screen_ref5000 = compute_flagging_rates(
    combined_df, subset_col='time_based_screen',
    ref_size=5000, curr_size=5000, step_size=500
)

flagging_screen_ref10000 = compute_flagging_rates(
    combined_df, subset_col='time_based_screen',
    ref_size=10000, curr_size=5000, step_size=500
)

# Default screen metrics for tables
flagging_screen = flagging_screen_ref5000
relative_flagging_screen = compute_relative_flagging_rates(flagging_screen)

overlap_overall = compute_conditional_overlap(combined_df)
overlap_screen_ref5000 = compute_conditional_overlap(
    combined_df, subset_col='time_based_screen',
    ref_size=5000, curr_size=5000, step_size=500
)

overlap_screen_ref10000 = compute_conditional_overlap(
    combined_df, subset_col='time_based_screen',
    ref_size=10000, curr_size=5000, step_size=500
)

# Default screen metrics for tables
overlap_screen = overlap_screen_ref5000

flagging_cancer_ref5000 = compute_flagging_rates(
    combined_df, subset_col='has_cancer', subset_value=1,
    ref_size=5000, curr_size=5000, step_size=500
)

flagging_cancer_ref10000 = compute_flagging_rates(
    combined_df, subset_col='has_cancer', subset_value=1,
    ref_size=10000, curr_size=5000, step_size=500
)

relative_flagging_cancer_ref5000 = compute_relative_flagging_rates(flagging_cancer_ref5000)
relative_flagging_cancer_ref10000 = compute_relative_flagging_rates(flagging_cancer_ref10000)

# Default cancer metrics for tables
flagging_cancer = flagging_cancer_ref5000
relative_flagging_cancer = compute_relative_flagging_rates(flagging_cancer)

overlap_cancer_ref5000 = compute_conditional_overlap(
    combined_df, subset_col='has_cancer', subset_value=1,
    ref_size=5000, curr_size=5000, step_size=500
)

overlap_cancer_ref10000 = compute_conditional_overlap(
    combined_df, subset_col='has_cancer', subset_value=1,
    ref_size=10000, curr_size=5000, step_size=500
)

# Default cancer metrics for tables
overlap_cancer = overlap_cancer_ref5000

flagging_cancer_ref2500_curr1000_step500 = compute_flagging_rates(
    combined_df, subset_col='has_cancer', subset_value=1,
    ref_size=2500, curr_size=1000, step_size=500
)

flagging_cancer_ref2500_curr1000_step250 = compute_flagging_rates(
    combined_df, subset_col='has_cancer', subset_value=1,
    ref_size=2500, curr_size=1000, step_size=250
)

relative_flagging_cancer_ref2500_curr1000_step500 = compute_relative_flagging_rates(flagging_cancer_ref2500_curr1000_step500)
relative_flagging_cancer_ref2500_curr1000_step250 = compute_relative_flagging_rates(flagging_cancer_ref2500_curr1000_step250)

overlap_cancer_ref2500_curr1000_step500 = compute_conditional_overlap(
    combined_df, subset_col='has_cancer', subset_value=1,
    ref_size=2500, curr_size=1000, step_size=500
)

overlap_cancer_ref2500_curr1000_step250 = compute_conditional_overlap(
    combined_df, subset_col='has_cancer', subset_value=1,
    ref_size=2500, curr_size=1000, step_size=250
)

cancer_rad_flagged_results = sliding_window_monitoring_subset(
    combined_df, 'cancer_rad_flagged', True, subset_label='cancer_rad_flagged',
    ref_size=5000, curr_size=5000, step_size=500, min_samples=3
)
cancer_rad_not_flagged_results = sliding_window_monitoring_subset(
    combined_df, 'cancer_rad_not_flagged', True, subset_label='cancer_rad_not_flagged',
    ref_size=5000, curr_size=5000, step_size=500, min_samples=3
)

rad_flagged_results = sliding_window_monitoring_subset(combined_df, 'rad_flagged', True, subset_label='rad_flagged', min_samples=10)
rad_not_flagged_results = sliding_window_monitoring_subset(combined_df, 'rad_not_flagged', True, subset_label='rad_not_flagged', min_samples=10)

screen_rad_flagged_results = sliding_window_monitoring_subset(
    combined_df, 'screen_rad_flagged', True, subset_label='screen_rad_flagged',
    ref_size=5000, curr_size=5000, step_size=500, min_samples=3
)
screen_rad_not_flagged_results = sliding_window_monitoring_subset(
    combined_df, 'screen_rad_not_flagged', True, subset_label='screen_rad_not_flagged',
    ref_size=5000, curr_size=5000, step_size=500, min_samples=3
)

print('Computed flagging/overlap and subset distribution results.')


In [ ]:
# Sensitivity stability (AI sensitivity among cancers)
sensitivity_results = compute_sensitivity_windows(combined_df)

# All-cancer sensitivity (36-month follow-up) with larger windows
sensitivity_cancer_ref5000 = compute_sensitivity_windows(
    combined_df,
    ref_size=5000, curr_size=5000, step_size=500
)

sensitivity_cancer_ref10000 = compute_sensitivity_windows(
    combined_df,
    ref_size=10000, curr_size=5000, step_size=500
)

# Default all-cancer sensitivity for summary tables
sensitivity_cancer = sensitivity_cancer_ref5000

# Radiologist baseline (pre-transition) for all cancers
pre_cancer = combined_df.iloc[:transition_point]
pre_cancer = pre_cancer[pre_cancer['has_cancer'] == 1]
rad_cancer_sens_pre = (
    pre_cancer['rad_flagged'].sum() / len(pre_cancer) * 100
    if len(pre_cancer) > 0 else np.nan
)


def summarize_sensitivity_before_after(sensitivity_df, transition_point):
    if sensitivity_df is None or len(sensitivity_df) == 0 or 'ai_system' not in sensitivity_df.columns:
        return pd.DataFrame(columns=['ai_system', 'before_mean', 'after_mean', 'abs_change'])
    rows = []
    for ai in AI_COLUMNS:
        data = sensitivity_df[sensitivity_df['ai_system'] == ai]
        before = data[data['curr_center'] < transition_point]['curr_sensitivity'].mean()
        after = data[data['curr_center'] >= transition_point]['curr_sensitivity'].mean()
        change = after - before if pd.notna(before) and pd.notna(after) else np.nan
        rows.append({
            'ai_system': ai,
            'before_mean': before,
            'after_mean': after,
            'abs_change': change
        })
    return pd.DataFrame(rows)

cancer_sensitivity_summary = summarize_sensitivity_before_after(sensitivity_cancer, transition_point)

print('Computed sensitivity results.')
cancer_sensitivity_summary


def interpret_sensitivity_summary(summary_df):
    lines = []
    lines.append('All-cancer sensitivity change (counterfactual GE -> Philips swap, 36-month follow-up):')
    for _, row in summary_df.iterrows():
        ai = str(row['ai_system']).upper()
        before = row['before_mean']
        after = row['after_mean']
        change = row['abs_change']
        if pd.isna(before) or pd.isna(after):
            lines.append(f'  - {ai}: insufficient data')
            continue
        direction = 'increase' if change >= 0 else 'decrease'
        lines.append(f'  - {ai}: {before:.2f}% -> {after:.2f}% ({change:+.2f} pp, {direction})')
    return ''.join(lines)

print(interpret_sensitivity_summary(cancer_sensitivity_summary))


def plot_sensitivity_before_after(summary_df, title, rad_baseline=None, rad_label='Radiologist R1+R2 (pre-swap)'):
    if summary_df is None or len(summary_df) == 0:
        print('No sensitivity summary to plot.')
        return
    labels = summary_df['ai_system'].str.upper()
    before = summary_df['before_mean']
    after = summary_df['after_mean']

    x = np.arange(len(labels))
    width = 0.35

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(x - width / 2, before, width, label='Pre-swap', color='#4C72B0')
    ax.bar(x + width / 2, after, width, label='Post-swap', color='#DD8452')

    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.set_ylabel('Sensitivity (%)')
    ax.set_title(title)
    ax.grid(True, axis='y', alpha=0.3)
    if pd.notna(rad_baseline):
        ax.axhline(rad_baseline, linestyle='--', color='#444444', linewidth=1.5, label=rad_label)
    ax.legend()
    plt.tight_layout()
    plt.show()

plot_sensitivity_before_after(
    cancer_sensitivity_summary,
    title='All-cancer Sensitivity (Pre-swap vs Post-swap, 36-month follow-up)',
    rad_baseline=rad_cancer_sens_pre
)

# Styled table image for all-cancer sensitivity summary
from pathlib import Path

output_dir = Path('outputs/2025_12_29_counterfactual_manufacturer_swap_setup')
output_dir.mkdir(parents=True, exist_ok=True)

if len(cancer_sensitivity_summary) > 0:
    display_df = cancer_sensitivity_summary.copy()
    display_df['change_pct'] = (
        (display_df['after_mean'] - display_df['before_mean'])
        / display_df['before_mean'] * 100
    )

    display_df = display_df.rename(columns={
        'ai_system': 'AI system',
        'before_mean': 'Pre-swap mean (%)',
        'after_mean': 'Post-swap mean (%)',
        'change_pct': 'Change (%)'
    })
    display_df = display_df[['AI system', 'Pre-swap mean (%)', 'Post-swap mean (%)', 'Change (%)']]

    for col in ['Pre-swap mean (%)', 'Post-swap mean (%)']:
        display_df[col] = pd.to_numeric(display_df[col], errors='coerce')
        display_df[col] = display_df[col].map(lambda x: f"{x:.2f}" if pd.notna(x) else '')

    display_df['Change (%)'] = pd.to_numeric(display_df['Change (%)'], errors='coerce')
    display_df['Change (%)'] = display_df['Change (%)'].map(lambda x: f"{x:+.2f}%" if pd.notna(x) else '')

    fig_height = 1.6 + 0.5 * len(display_df)
    fig, ax = plt.subplots(figsize=(11, fig_height))
    ax.axis('off')

    table = ax.table(
        cellText=display_df.values,
        colLabels=display_df.columns,
        cellLoc='center',
        loc='center',
        bbox=[0, 0, 1, 1]
    )

    col_widths = [0.20, 0.27, 0.27, 0.26]
    for col_idx, width in enumerate(col_widths):
        for (row, col), cell in table.get_celld().items():
            if col == col_idx:
                cell.set_width(width)

    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1, 1.7)

    header_color = '#4472C4'
    alt_row_color = '#E7E6E6'

    for i in range(len(display_df.columns)):
        cell = table[(0, i)]
        cell.set_facecolor(header_color)
        cell.set_text_props(weight='bold', color='white')

    for i in range(1, len(display_df) + 1):
        for j in range(len(display_df.columns)):
            cell = table[(i, j)]
            if i % 2 == 0:
                cell.set_facecolor(alt_row_color)
            else:
                cell.set_facecolor('white')

    for cell in table.get_celld().values():
        cell.set_edgecolor('black')
        cell.set_linewidth(1.2)

    plt.title('All-cancer Sensitivity Summary (Pre-swap vs Post-swap, 36-month follow-up)',
              fontsize=14, fontweight='bold', pad=20)
    plt.tight_layout()
    table_path = output_dir / 'cancer_sensitivity_summary_table.png'
    plt.savefig(table_path, dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()
    table_path

# All-cancer specificity (36-month follow-up) with larger windows
specificity_cancer_ref5000 = compute_specificity_windows(
    combined_df,
    ref_size=5000, curr_size=5000, step_size=500
)

specificity_cancer_ref10000 = compute_specificity_windows(
    combined_df,
    ref_size=10000, curr_size=5000, step_size=500
)

# Default all-cancer specificity for summary tables
specificity_cancer = specificity_cancer_ref5000

# Radiologist baseline (pre-transition) for controls
pre_controls = combined_df.iloc[:transition_point]
pre_controls = pre_controls[pre_controls['has_cancer'] == 0]
rad_cancer_spec_pre = (
    (1 - pre_controls['rad_flagged'].mean()) * 100
    if len(pre_controls) > 0 else np.nan
)


def summarize_specificity_before_after(specificity_df, transition_point):
    if specificity_df is None or len(specificity_df) == 0 or 'ai_system' not in specificity_df.columns:
        return pd.DataFrame(columns=['ai_system', 'before_mean', 'after_mean', 'abs_change'])
    rows = []
    for ai in AI_COLUMNS:
        data = specificity_df[specificity_df['ai_system'] == ai]
        before = data[data['curr_center'] < transition_point]['curr_specificity'].mean()
        after = data[data['curr_center'] >= transition_point]['curr_specificity'].mean()
        change = after - before if pd.notna(before) and pd.notna(after) else np.nan
        rows.append({
            'ai_system': ai,
            'before_mean': before,
            'after_mean': after,
            'abs_change': change
        })
    return pd.DataFrame(rows)

cancer_specificity_summary = summarize_specificity_before_after(specificity_cancer, transition_point)

print('Computed specificity results.')
cancer_specificity_summary


def interpret_specificity_summary(summary_df):
    lines = []
    lines.append('All-cancer specificity change (counterfactual GE -> Philips swap, 36-month follow-up):')
    for _, row in summary_df.iterrows():
        ai = str(row['ai_system']).upper()
        before = row['before_mean']
        after = row['after_mean']
        change = row['abs_change']
        if pd.isna(before) or pd.isna(after):
            lines.append(f'  - {ai}: insufficient data')
            continue
        direction = 'increase' if change >= 0 else 'decrease'
        lines.append(f'  - {ai}: {before:.2f}% -> {after:.2f}% ({change:+.2f} pp, {direction})')
    return ''.join(lines)

print(interpret_specificity_summary(cancer_specificity_summary))


def plot_specificity_before_after(summary_df, title, rad_baseline=None, rad_label='Radiologist R1+R2 (pre-swap)'):
    if summary_df is None or len(summary_df) == 0:
        print('No specificity summary to plot.')
        return
    labels = summary_df['ai_system'].str.upper()
    before = summary_df['before_mean']
    after = summary_df['after_mean']

    x = np.arange(len(labels))
    width = 0.35

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(x - width / 2, before, width, label='Pre-swap', color='#4C72B0')
    ax.bar(x + width / 2, after, width, label='Post-swap', color='#DD8452')

    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.set_ylabel('Specificity (%)')
    ax.set_title(title)
    ax.grid(True, axis='y', alpha=0.3)
    if pd.notna(rad_baseline):
        ax.axhline(rad_baseline, linestyle='--', color='#444444', linewidth=1.5, label=rad_label)
    ax.legend()
    plt.tight_layout()
    plt.show()

plot_specificity_before_after(
    cancer_specificity_summary,
    title='All-cancer Specificity (Pre-swap vs Post-swap, 36-month follow-up)',
    rad_baseline=rad_cancer_spec_pre
)

# Styled table image for all-cancer specificity summary
if len(cancer_specificity_summary) > 0:
    display_df = cancer_specificity_summary.copy()
    display_df['change_pct'] = (
        (display_df['after_mean'] - display_df['before_mean'])
        / display_df['before_mean'] * 100
    )

    display_df = display_df.rename(columns={
        'ai_system': 'AI system',
        'before_mean': 'Pre-swap mean (%)',
        'after_mean': 'Post-swap mean (%)',
        'change_pct': 'Change (%)'
    })
    display_df = display_df[['AI system', 'Pre-swap mean (%)', 'Post-swap mean (%)', 'Change (%)']]

    for col in ['Pre-swap mean (%)', 'Post-swap mean (%)']:
        display_df[col] = pd.to_numeric(display_df[col], errors='coerce')
        display_df[col] = display_df[col].map(lambda x: f"{x:.2f}" if pd.notna(x) else '')

    display_df['Change (%)'] = pd.to_numeric(display_df['Change (%)'], errors='coerce')
    display_df['Change (%)'] = display_df['Change (%)'].map(lambda x: f"{x:+.2f}%" if pd.notna(x) else '')

    fig_height = 1.6 + 0.5 * len(display_df)
    fig, ax = plt.subplots(figsize=(11, fig_height))
    ax.axis('off')

    table = ax.table(
        cellText=display_df.values,
        colLabels=display_df.columns,
        cellLoc='center',
        loc='center',
        bbox=[0, 0, 1, 1]
    )

    col_widths = [0.20, 0.27, 0.27, 0.26]
    for col_idx, width in enumerate(col_widths):
        for (row, col), cell in table.get_celld().items():
            if col == col_idx:
                cell.set_width(width)

    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1, 1.7)

    header_color = '#4472C4'
    alt_row_color = '#E7E6E6'

    for i in range(len(display_df.columns)):
        cell = table[(0, i)]
        cell.set_facecolor(header_color)
        cell.set_text_props(weight='bold', color='white')

    for i in range(1, len(display_df) + 1):
        for j in range(len(display_df.columns)):
            cell = table[(i, j)]
            if i % 2 == 0:
                cell.set_facecolor(alt_row_color)
            else:
                cell.set_facecolor('white')

    for cell in table.get_celld().values():
        cell.set_edgecolor('black')
        cell.set_linewidth(1.2)

    plt.title('All-cancer Specificity Summary (Pre-swap vs Post-swap, 36-month follow-up)',
              fontsize=14, fontweight='bold', pad=20)
    plt.tight_layout()
    table_path = output_dir / 'cancer_specificity_summary_table.png'
    plt.savefig(table_path, dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()
    table_path


In [ ]:
# Counterfactual dataset: nine parameter plots (median for score distributions)
# 1) AI score distribution
plot_monitoring_results(monitoring_results, transition_point, metric='median',
                        title='AI Score Distribution: Median')

# 2) Flagging rate: AI / Radiologist
plot_flagging_rates(flagging_overall, transition_point,
                    title='Flagging Rate: AI / Radiologist',
                    normalize_by_radiologist=True,
                    uniform_y=True)

# 3) P(AI | Radiologist)
plot_conditional_overlap(overlap_overall, transition_point, metric='ai_given_rad',
                         title='Conditional Overlap: P(AI | Radiologist)')

# 4) P(Radiologist | AI)
plot_conditional_overlap(overlap_overall, transition_point, metric='rad_given_ai',
                         title='Conditional Overlap: P(Radiologist | AI)')

# 5) AI score distribution among Rad-flagged vs Not
plot_subset_distribution(rad_flagged_results, rad_not_flagged_results, transition_point,
                         metric='median', labels=('Rad flagged', 'Not rad flagged'),
                         title='AI Score Distribution: Rad-flagged vs Not',
                         uniform_y=True)

# 6) All-cancer flagging rate: AI / Radiologist
plot_flagging_rates(flagging_cancer, transition_point,
                    title='All-cancer Flagging Rate: AI / Radiologist',
                    normalize_by_radiologist=True,
                    uniform_y=True)

# 7) All-cancer P(AI | Radiologist)
plot_conditional_overlap(overlap_cancer, transition_point, metric='ai_given_rad',
                         title='All-cancer P(AI | Radiologist)')

# 8) All-cancer P(Radiologist | AI)
plot_conditional_overlap(overlap_cancer, transition_point, metric='rad_given_ai',
                         title='All-cancer P(Radiologist | AI)')

# 9) All-cancer AI score distribution among Rad-flagged vs Not
plot_subset_distribution(cancer_rad_flagged_results, cancer_rad_not_flagged_results, transition_point,
                         metric='median', labels=('Cancer rad flagged', 'Cancer not rad flagged'),
                         title='All-cancer AI Score Distribution: Rad-flagged vs Not',
                         uniform_y=True)

# 10) All-cancer window-size comparison (ref=2500, curr=1000, step=500)
plot_flagging_rates(flagging_cancer_ref2500_curr1000_step500, transition_point,
                    title='All-cancer Flagging Rate (ref=2500, curr=1000, step=500)',
                    normalize_by_radiologist=True,
                    uniform_y=True)

plot_conditional_overlap(overlap_cancer_ref2500_curr1000_step500, transition_point, metric='ai_given_rad',
                         title='All-cancer P(AI | Radiologist) (ref=2500, curr=1000, step=500)')

plot_conditional_overlap(overlap_cancer_ref2500_curr1000_step500, transition_point, metric='rad_given_ai',
                         title='All-cancer P(Radiologist | AI) (ref=2500, curr=1000, step=500)')

# 11) All-cancer window-size comparison (ref=2500, curr=1000, step=250)
plot_flagging_rates(flagging_cancer_ref2500_curr1000_step250, transition_point,
                    title='All-cancer Flagging Rate (ref=2500, curr=1000, step=250)',
                    normalize_by_radiologist=True,
                    uniform_y=True)

plot_conditional_overlap(overlap_cancer_ref2500_curr1000_step250, transition_point, metric='ai_given_rad',
                         title='All-cancer P(AI | Radiologist) (ref=2500, curr=1000, step=250)')

plot_conditional_overlap(overlap_cancer_ref2500_curr1000_step250, transition_point, metric='rad_given_ai',
                         title='All-cancer P(Radiologist | AI) (ref=2500, curr=1000, step=250)')

# Window-size ablation: cancer-only pre/post signal counts

def _tp_fp_time(results_df, p_col, center_col, transition_point, alpha=0.05):
    if results_df is None or len(results_df) == 0 or p_col not in results_df.columns:
        return 0, 0, np.nan
    sig = results_df[p_col] < alpha
    before = results_df[center_col] < transition_point
    after = results_df[center_col] >= transition_point
    fp = int((sig & before).sum())
    tp = int((sig & after).sum())
    after_sig = results_df[sig & after]
    if len(after_sig) == 0:
        return tp, fp, np.nan
    first_center = after_sig[center_col].min()
    return tp, fp, first_center - transition_point

window_configs = [
    {
        'label': 'ref5000_curr5000_step500',
        'flagging': flagging_cancer_ref5000,
        'relative': relative_flagging_cancer_ref5000,
        'overlap': overlap_cancer_ref5000
    },
    {
        'label': 'ref10000_curr5000_step500',
        'flagging': flagging_cancer_ref10000,
        'relative': relative_flagging_cancer_ref10000,
        'overlap': overlap_cancer_ref10000
    },
    {
        'label': 'ref2500_curr1000_step500',
        'flagging': flagging_cancer_ref2500_curr1000_step500,
        'relative': relative_flagging_cancer_ref2500_curr1000_step500,
        'overlap': overlap_cancer_ref2500_curr1000_step500
    },
    {
        'label': 'ref2500_curr1000_step250',
        'flagging': flagging_cancer_ref2500_curr1000_step250,
        'relative': relative_flagging_cancer_ref2500_curr1000_step250,
        'overlap': overlap_cancer_ref2500_curr1000_step250
    }
]

detail_rows = []
for cfg in window_configs:
    label = cfg['label']
    # Flagging rate (cancers)
    for ai in AI_COLUMNS:
        data = cfg['flagging'].get(ai) if cfg.get('flagging') is not None else None
        tp, fp, t_first = _tp_fp_time(data, 'p_value_corrected', 'curr_center', transition_point)
        detail_rows.append({
            'window': label,
            'metric': 'Flagging rate (cancers)',
            'ai_system': ai,
            'signals_post_start': tp,
            'signals_pre_start': fp,
            'first_signal_delay': t_first
        })
    # Relative flagging rate (cancers)
    for ai in AI_COLUMNS:
        data = cfg['relative'].get(ai) if cfg.get('relative') is not None else None
        tp, fp, t_first = _tp_fp_time(data, 'p_value_corrected', 'curr_center', transition_point)
        detail_rows.append({
            'window': label,
            'metric': 'Relative flagging rate (cancers)',
            'ai_system': ai,
            'signals_post_start': tp,
            'signals_pre_start': fp,
            'first_signal_delay': t_first
        })
    # P(AI | Radiologist) (cancers)
    for ai in AI_COLUMNS:
        data = cfg['overlap'].get(ai) if cfg.get('overlap') is not None else None
        tp, fp, t_first = _tp_fp_time(data, 'p_val_ai_given_rad_corrected', 'curr_center', transition_point)
        detail_rows.append({
            'window': label,
            'metric': 'P(AI | Radiologist) (cancers)',
            'ai_system': ai,
            'signals_post_start': tp,
            'signals_pre_start': fp,
            'first_signal_delay': t_first
        })
    # P(Radiologist | AI) (cancers)
    for ai in AI_COLUMNS:
        data = cfg['overlap'].get(ai) if cfg.get('overlap') is not None else None
        tp, fp, t_first = _tp_fp_time(data, 'p_val_rad_given_ai_corrected', 'curr_center', transition_point)
        detail_rows.append({
            'window': label,
            'metric': 'P(Radiologist | AI) (cancers)',
            'ai_system': ai,
            'signals_post_start': tp,
            'signals_pre_start': fp,
            'first_signal_delay': t_first
        })

detail_df = pd.DataFrame(detail_rows)
if len(detail_df) > 0:
    summary_df = (
        detail_df
        .groupby('window', as_index=False)
        .agg(
            total_signals_post_start=('signals_post_start', 'sum'),
            total_signals_pre_start=('signals_pre_start', 'sum'),
            avg_time_to_first_signal=('first_signal_delay', 'mean')
        )
    )
    summary_df['tp_minus_fp'] = summary_df['total_signals_post_start'] - summary_df['total_signals_pre_start']
    summary_df = summary_df.sort_values(['tp_minus_fp', 'total_signals_post_start'], ascending=False)

    print('Window-size ablation summary (cancers)')
    display(summary_df)

    metric_summary_df = (
        detail_df
        .groupby(['window', 'metric'], as_index=False)
        .agg(
            total_signals_post_start=('signals_post_start', 'sum'),
            total_signals_pre_start=('signals_pre_start', 'sum'),
            avg_time_to_first_signal=('first_signal_delay', 'mean')
        )
    )
    metric_summary_df['tp_minus_fp'] = metric_summary_df['total_signals_post_start'] - metric_summary_df['total_signals_pre_start']
    metric_summary_df = metric_summary_df.sort_values(['window', 'tp_minus_fp', 'total_signals_post_start'], ascending=[True, False, False])

    print('Window-size ablation by metric (cancers)')
    display(metric_summary_df)

    from pathlib import Path
    output_dir = Path('outputs/2025_12_29_counterfactual_manufacturer_swap_setup')
    output_dir.mkdir(parents=True, exist_ok=True)

    def _format_table(df):
        formatted = df.copy()
        int_cols = [
            'total_signals_post_start',
            'total_signals_pre_start',
            'tp_minus_fp',
            'avg_time_to_first_signal'
        ]
        for col in int_cols:
            if col in formatted.columns:
                formatted[col] = formatted[col].apply(lambda x: '' if pd.isna(x) else str(int(x)))
        formatted.columns = [col.replace('_', ' ') for col in formatted.columns]
        return formatted

    def _render_table_image(df, title, path, col_widths=None):
        if df is None or len(df) == 0:
            return
        fig_height = 1.8 + 0.45 * len(df)
        fig, ax = plt.subplots(figsize=(14, fig_height))
        ax.axis('off')
        table = ax.table(
            cellText=df.values,
            colLabels=df.columns,
            cellLoc='center',
            loc='center',
            bbox=[0, 0, 1, 1]
        )
        if col_widths:
            for col_idx, width in enumerate(col_widths):
                for (row, col), cell in table.get_celld().items():
                    if col == col_idx:
                        cell.set_width(width)
        table.auto_set_font_size(False)
        table.set_fontsize(9)
        table.scale(1, 1.6)

        header_color = '#4472C4'
        alt_row_color = '#E7E6E6'
        for i in range(len(df.columns)):
            cell = table[(0, i)]
            cell.set_facecolor(header_color)
            cell.set_text_props(weight='bold', color='white')
        for i in range(1, len(df) + 1):
            for j in range(len(df.columns)):
                cell = table[(i, j)]
                cell.set_facecolor(alt_row_color if i % 2 == 0 else 'white')
        for cell in table.get_celld().values():
            cell.set_edgecolor('black')
            cell.set_linewidth(1.0)
        plt.title(title, fontsize=14, fontweight='bold', pad=16)
        plt.tight_layout()
        plt.savefig(path, dpi=300, bbox_inches='tight', facecolor='white')
        plt.show()

    summary_disp = _format_table(summary_df)
    metric_disp = _format_table(metric_summary_df)

    summary_path = output_dir / 'window_ablation_summary_table.png'
    metric_path = output_dir / 'window_ablation_metric_table.png'

    _render_table_image(
        summary_disp,
        'Window-size ablation summary (cancers)',
        summary_path,
        col_widths=[0.36, 0.16, 0.16, 0.20, 0.12]
    )
    _render_table_image(
        metric_disp,
        'Window-size ablation by metric (cancers)',
        metric_path,
        col_widths=[0.26, 0.34, 0.14, 0.14, 0.22]
    )

    print('Window-size ablation detail (cancers)')
    display(detail_df)



In [ ]:
# Parameter-specific tables: Signals pre-start / missed post-start

def fp_fn_counts(results_df, p_col, center_col, transition_point, alpha=0.05):
    if results_df is None or len(results_df) == 0 or p_col not in results_df.columns:
        return None
    sig = results_df[p_col] < alpha
    before = results_df[center_col] < transition_point
    after = results_df[center_col] >= transition_point
    fp = int((sig & before).sum())
    tp = int((sig & after).sum())
    total_after = int(after.sum())
    fn = int(total_after - tp)
    return fp, fn, tp, total_after


def print_table(title, rows):
    if not rows:
        print(f"{title}(no data)")
        return
    df = pd.DataFrame(rows)
    print(f"{title}")
    print(df)

# 1) AI score distribution (overall)
rows = []
for ai in AI_COLUMNS:
    ai_data = monitoring_results[monitoring_results['ai_system'] == ai] if 'monitoring_results' in globals() else None
    for test in ['median', 'p90', 'ks']:
        p_col = f'{test}_p_fdr'
        res = fp_fn_counts(ai_data, p_col, 'window_center', transition_point) if ai_data is not None else None
        if res is None:
            continue
        fp, fn, tp, total_after = res
        rows.append({
            'ai_system': ai,
            'test': test,
            'signals_pre_start': fp,
            'signals_missed_post_start': fn
        })
print_table('AI score distribution (overall)', rows)

# 2) Flagging rate: AI
rows = []
for ai in AI_COLUMNS:
    data = flagging_overall.get(ai) if 'flagging_overall' in globals() else None
    res = fp_fn_counts(data, 'p_value_corrected', 'curr_center', transition_point) if data is not None else None
    if res is None:
        continue
    fp, fn, tp, total_after = res
    rows.append({
        'ai_system': ai,
        'test': 'z-test',
        'signals_pre_start': fp,
        'signals_missed_post_start': fn
    })
print_table('Flagging rate (AI)', rows)

# 3) Flagging rate: Radiologist
rows = []
rad_data = flagging_overall.get('radiologist') if 'flagging_overall' in globals() else None
res = fp_fn_counts(rad_data, 'p_value_corrected', 'curr_center', transition_point) if rad_data is not None else None
if res is not None:
    fp, fn, tp, total_after = res
    rows.append({
        'ai_system': 'radiologist',
        'test': 'z-test',
        'signals_pre_start': fp,
        'signals_missed_post_start': fn
    })
print_table('Flagging rate (Radiologist)', rows)

# 4) P(AI | Radiologist)
rows = []
for ai in AI_COLUMNS:
    data = overlap_overall.get(ai) if 'overlap_overall' in globals() else None
    res = fp_fn_counts(data, 'p_val_ai_given_rad_corrected', 'curr_center', transition_point) if data is not None else None
    if res is None:
        continue
    fp, fn, tp, total_after = res
    rows.append({
        'ai_system': ai,
        'test': 'z-test',
        'signals_pre_start': fp,
        'signals_missed_post_start': fn
    })
print_table('P(AI | Radiologist)', rows)

# 5) P(Radiologist | AI)
rows = []
for ai in AI_COLUMNS:
    data = overlap_overall.get(ai) if 'overlap_overall' in globals() else None
    res = fp_fn_counts(data, 'p_val_rad_given_ai_corrected', 'curr_center', transition_point) if data is not None else None
    if res is None:
        continue
    fp, fn, tp, total_after = res
    rows.append({
        'ai_system': ai,
        'test': 'z-test',
        'signals_pre_start': fp,
        'signals_missed_post_start': fn
    })
print_table('P(Radiologist | AI)', rows)

# 6) AI score distribution by radiologist flag (overall)
rows = []
for ai in AI_COLUMNS:
    for subset_label, subset_data in [('Rad flagged', rad_flagged_results), ('Not rad flagged', rad_not_flagged_results)]:
        if subset_data is None or len(subset_data) == 0 or 'ai_system' not in subset_data.columns:
            continue
        data = subset_data[subset_data['ai_system'] == ai]
        for test in ['median', 'p90', 'ks']:
            res = fp_fn_counts(data, f'{test}_p_fdr', 'window_center', transition_point)
            if res is None:
                continue
            fp, fn, tp, total_after = res
            rows.append({
                'ai_system': ai,
                'subset': subset_label,
                'test': test,
                'signals_pre_start': fp,
                'signals_missed_post_start': fn
            })
print_table('AI score distribution by radiologist flag (overall)', rows)

# 7) Screen-detected flagging rate: AI
rows = []
for ai in AI_COLUMNS:
    data = flagging_screen.get(ai) if 'flagging_screen' in globals() else None
    res = fp_fn_counts(data, 'p_value_corrected', 'curr_center', transition_point) if data is not None else None
    if res is None:
        continue
    fp, fn, tp, total_after = res
    rows.append({
        'ai_system': ai,
        'test': 'z-test',
        'signals_pre_start': fp,
        'signals_missed_post_start': fn
    })
print_table('Screen-detected flagging rate (AI)', rows)

# 8) Screen-detected flagging rate: Radiologist
rows = []
rad_data = flagging_screen.get('radiologist') if 'flagging_screen' in globals() else None
res = fp_fn_counts(rad_data, 'p_value_corrected', 'curr_center', transition_point) if rad_data is not None else None
if res is not None:
    fp, fn, tp, total_after = res
    rows.append({
        'ai_system': 'radiologist',
        'test': 'z-test',
        'signals_pre_start': fp,
        'signals_missed_post_start': fn
    })
print_table('Screen-detected flagging rate (Radiologist)', rows)

# 9) Screen-detected P(AI | Radiologist)
rows = []
for ai in AI_COLUMNS:
    data = overlap_screen.get(ai) if 'overlap_screen' in globals() else None
    res = fp_fn_counts(data, 'p_val_ai_given_rad_corrected', 'curr_center', transition_point) if data is not None else None
    if res is None:
        continue
    fp, fn, tp, total_after = res
    rows.append({
        'ai_system': ai,
        'test': 'z-test',
        'signals_pre_start': fp,
        'signals_missed_post_start': fn
    })
print_table('Screen-detected P(AI | Radiologist)', rows)

# 10) Screen-detected P(Radiologist | AI)
rows = []
for ai in AI_COLUMNS:
    data = overlap_screen.get(ai) if 'overlap_screen' in globals() else None
    res = fp_fn_counts(data, 'p_val_rad_given_ai_corrected', 'curr_center', transition_point) if data is not None else None
    if res is None:
        continue
    fp, fn, tp, total_after = res
    rows.append({
        'ai_system': ai,
        'test': 'z-test',
        'signals_pre_start': fp,
        'signals_missed_post_start': fn
    })
print_table('Screen-detected P(Radiologist | AI)', rows)

# 11) Screen-detected AI score distribution by radiologist flag
rows = []
for ai in AI_COLUMNS:
    for subset_label, subset_data in [('Screen rad flagged', screen_rad_flagged_results), ('Screen not rad flagged', screen_rad_not_flagged_results)]:
        if subset_data is None or len(subset_data) == 0 or 'ai_system' not in subset_data.columns:
            continue
        data = subset_data[subset_data['ai_system'] == ai]
        for test in ['median', 'p90', 'ks']:
            res = fp_fn_counts(data, f'{test}_p_fdr', 'window_center', transition_point)
            if res is None:
                continue
            fp, fn, tp, total_after = res
            rows.append({
                'ai_system': ai,
                'subset': subset_label,
                'test': test,
                'signals_pre_start': fp,
                'signals_missed_post_start': fn
            })
print_table('Screen-detected AI score distribution by radiologist flag', rows)

# Sensitivity stability
rows = []
if 'sensitivity_results' in globals():
    for ai in AI_COLUMNS:
        if sensitivity_results is None or len(sensitivity_results) == 0 or 'ai_system' not in sensitivity_results.columns:
            continue
        data = sensitivity_results[sensitivity_results['ai_system'] == ai]
        res = fp_fn_counts(data, 'p_value_corrected', 'curr_center', transition_point)
        if res is None:
            continue
        fp, fn, tp, total_after = res
        rows.append({
            'ai_system': ai,
            'test': 'z-test',
            'signals_pre_start': fp,
            'signals_missed_post_start': fn
        })
print_table('Sensitivity (AI)', rows)

# Summary table across Level 1 + Part 2 metrics: pre/post signal counts + grade

def _grade_from_fp_tp(fp, tp):
    if pd.isna(fp) or pd.isna(tp):
        return 'Not Available'
    if fp == 0 and tp == 0:
        return 'Not applicable'
    if tp <= 0:
        return 'Not Recommended'
    if fp == 0:
        return 'Highly Recommended'
    if fp <= 2:
        return 'Recommended'
    return 'Not Recommended'


def _first_signal_offset(results_df, p_col, center_col, transition_point, alpha=0.05):
    if results_df is None or len(results_df) == 0 or p_col not in results_df.columns:
        return np.nan
    sig = results_df[p_col] < alpha
    after = results_df[center_col] >= transition_point
    after_sig = results_df[sig & after]
    if len(after_sig) == 0:
        return np.nan
    first_center = after_sig[center_col].min()
    return first_center - transition_point


def build_drift_detection_summary_table(transition_point, alpha=0.05, include_cancers=True):
    rows = []

    def _add_row(level, plot_label, ai_label, res, time_to_first_signal):
        if res is None:
            return
        fp, fn, tp, total_after = res
        rows.append({
            'level': level,
            'plot': plot_label,
            'ai_system': ai_label,
            'signals_post_start': tp,
            'signals_pre_start': fp,
            'first_signal_delay': time_to_first_signal,
            'grade': _grade_from_fp_tp(fp, tp)
        })

    # Level 1: Median and P90 score tests
    if 'monitoring_results' in globals():
        for ai in AI_COLUMNS:
            ai_data = monitoring_results[monitoring_results['ai_system'] == ai]
            res = fp_fn_counts(ai_data, 'median_p_fdr', 'window_center', transition_point, alpha)
            t_first = _first_signal_offset(ai_data, 'median_p_fdr', 'window_center', transition_point, alpha)
            _add_row('1', 'Median score', ai, res, t_first)

            res = fp_fn_counts(ai_data, 'p90_p_fdr', 'window_center', transition_point, alpha)
            t_first = _first_signal_offset(ai_data, 'p90_p_fdr', 'window_center', transition_point, alpha)
            _add_row('1', 'P90 score', ai, res, t_first)

    # Level 2: Relative flagging rate (AI / Radiologist)
    for ai in AI_COLUMNS:
        data = relative_flagging_overall.get(ai) if 'relative_flagging_overall' in globals() else None
        res = fp_fn_counts(data, 'p_value_corrected', 'curr_center', transition_point, alpha) if data is not None else None
        t_first = _first_signal_offset(data, 'p_value_corrected', 'curr_center', transition_point, alpha) if data is not None else np.nan
        _add_row('2', 'Relative flagging rate', ai, res, t_first)
    for ai in AI_COLUMNS:
        data = flagging_overall.get(ai) if 'flagging_overall' in globals() else None
        res = fp_fn_counts(data, 'p_value_corrected', 'curr_center', transition_point, alpha) if data is not None else None
        t_first = _first_signal_offset(data, 'p_value_corrected', 'curr_center', transition_point, alpha) if data is not None else np.nan
        _add_row('2', 'Flagging rate', ai, res, t_first)

    rad_data = flagging_overall.get('radiologist') if 'flagging_overall' in globals() else None
    res = fp_fn_counts(rad_data, 'p_value_corrected', 'curr_center', transition_point, alpha) if rad_data is not None else None
    t_first = _first_signal_offset(rad_data, 'p_value_corrected', 'curr_center', transition_point, alpha) if rad_data is not None else np.nan
    _add_row('2', 'Flagging rate', 'radiologist', res, t_first)

    # Level 2: P(AI | Radiologist)
    for ai in AI_COLUMNS:
        data = overlap_overall.get(ai) if 'overlap_overall' in globals() else None
        res = fp_fn_counts(data, 'p_val_ai_given_rad_corrected', 'curr_center', transition_point, alpha) if data is not None else None
        t_first = _first_signal_offset(data, 'p_val_ai_given_rad_corrected', 'curr_center', transition_point, alpha) if data is not None else np.nan
        _add_row('2', 'P(AI | Radiologist)', ai, res, t_first)

    # Level 2: P(Radiologist | AI)
    for ai in AI_COLUMNS:
        data = overlap_overall.get(ai) if 'overlap_overall' in globals() else None
        res = fp_fn_counts(data, 'p_val_rad_given_ai_corrected', 'curr_center', transition_point, alpha) if data is not None else None
        t_first = _first_signal_offset(data, 'p_val_rad_given_ai_corrected', 'curr_center', transition_point, alpha) if data is not None else np.nan
        _add_row('2', 'P(Radiologist | AI)', ai, res, t_first)

    # Cancers metrics (optional)
    if include_cancers:
        for ai in AI_COLUMNS:
            data = flagging_cancer.get(ai) if 'flagging_cancer' in globals() else None
            res = fp_fn_counts(data, 'p_value_corrected', 'curr_center', transition_point, alpha) if data is not None else None
            t_first = _first_signal_offset(data, 'p_value_corrected', 'curr_center', transition_point, alpha) if data is not None else np.nan
            _add_row('2', 'Flagging rate (cancers)', ai, res, t_first)

        for ai in AI_COLUMNS:
            data = relative_flagging_cancer.get(ai) if 'relative_flagging_cancer' in globals() else None
            res = fp_fn_counts(data, 'p_value_corrected', 'curr_center', transition_point, alpha) if data is not None else None
            t_first = _first_signal_offset(data, 'p_value_corrected', 'curr_center', transition_point, alpha) if data is not None else np.nan
            _add_row('2', 'Relative flagging rate (cancers)', ai, res, t_first)

        for ai in AI_COLUMNS:
            data = overlap_cancer.get(ai) if 'overlap_cancer' in globals() else None
            res = fp_fn_counts(data, 'p_val_ai_given_rad_corrected', 'curr_center', transition_point, alpha) if data is not None else None
            t_first = _first_signal_offset(data, 'p_val_ai_given_rad_corrected', 'curr_center', transition_point, alpha) if data is not None else np.nan
            _add_row('2', 'P(AI | Radiologist) (cancers)', ai, res, t_first)

        for ai in AI_COLUMNS:
            data = overlap_cancer.get(ai) if 'overlap_cancer' in globals() else None
            res = fp_fn_counts(data, 'p_val_rad_given_ai_corrected', 'curr_center', transition_point, alpha) if data is not None else None
            t_first = _first_signal_offset(data, 'p_val_rad_given_ai_corrected', 'curr_center', transition_point, alpha) if data is not None else np.nan
            _add_row('2', 'P(Radiologist | AI) (cancers)', ai, res, t_first)

    return pd.DataFrame(rows)

part2_summary_df = build_drift_detection_summary_table(transition_point, alpha=0.05, include_cancers=True)
part2_summary_df

# Styled table image (same style as previous tables)
from pathlib import Path

output_dir = Path('outputs/2025_12_29_counterfactual_manufacturer_swap_setup')
output_dir.mkdir(parents=True, exist_ok=True)

if len(part2_summary_df) > 0:
    display_df = part2_summary_df[[
        'level', 'plot', 'ai_system', 'signals_post_start', 'signals_pre_start', 'first_signal_delay', 'grade'
    ]].copy()

    int_cols = ['signals_post_start', 'signals_pre_start', 'first_signal_delay']
    for col in int_cols:
        display_df[col] = display_df[col].apply(lambda x: '' if pd.isna(x) else str(int(x)))

    display_df.columns = [col.replace('_', ' ') for col in display_df.columns]

    fig_height = 1.8 + 0.45 * len(display_df)
    fig, ax = plt.subplots(figsize=(16, fig_height))
    ax.axis('off')

    table = ax.table(
        cellText=display_df.values,
        colLabels=display_df.columns,
        cellLoc='center',
        loc='center',
        bbox=[0, 0, 1, 1]
    )

    col_widths = [0.06, 0.28, 0.14, 0.10, 0.10, 0.14, 0.18]
    for col_idx, width in enumerate(col_widths):
        for (row, col), cell in table.get_celld().items():
            if col == col_idx:
                cell.set_width(width)

    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1, 1.7)

    header_color = '#4472C4'
    alt_row_color = '#E7E6E6'

    for i in range(len(display_df.columns)):
        cell = table[(0, i)]
        cell.set_facecolor(header_color)
        cell.set_text_props(weight='bold', color='white')

    for i in range(1, len(display_df) + 1):
        for j in range(len(display_df.columns)):
            cell = table[(i, j)]
            if i % 2 == 0:
                cell.set_facecolor(alt_row_color)
            else:
                cell.set_facecolor('white')

    for cell in table.get_celld().values():
        cell.set_edgecolor('black')
        cell.set_linewidth(1.2)

    plt.title('Drift Detection Summary (pre/post signal + Grade)',
              fontsize=16, fontweight='bold', pad=20)
    plt.tight_layout()
    table_path = output_dir / 'part2_drift_detection_summary_table.png'
    plt.savefig(table_path, dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()
    table_path



# PART 3: Simulation Environment (Hospital-Level Monitoring)

This section implements the hospital-level simulation plan: define a pre-start period, compute radiologist sensitivities, set AI thresholds to match those sensitivities, and track sensitivity/specificity over time.


In [ ]:
# Part 3 parameters
PART3_START_DATE = START_DATE
PART3_END_DATE = END_DATE
PART3_PRESTART_MIN_CANCERS = 100  # per hospital+manufacturer; exclude if fewer
PART3_SIGNAL_MIN_RUN = 3  # consecutive significant windows required to flag a signal
PART3_RECAL_USE_SCORE_SIGNALS = False  # skip score-based signals in recalibration for speed
PART3_CALIBRATION_STRATEGY = 'rad_sensitivity'
PART3_RAD_MODE = 'combined'  # radiologist flagged if R1 OR R2 marked DISCUSSION
PART3_TARGET_MODE = 'combined'  # calibration target reader flag: R1 OR R2 marked DISCUSSION
PART3_SYSTEM_MODE = 'ai_only'  # ai_only or ai_plus_r1
PART3_ORDERING = 'date'  # 'date' or 'month_random' (shuffle within month)

PART3_SCENARIOS = [
    {
        'key': 'single_reader',
        'label': 'single reader',
        'rad_mode': 'r1',
        'target_mode': 'r1',
        'system_mode': 'ai_only'
    },
    {
        'key': 'double_reader',
        'label': 'double reader',
        'rad_mode': 'r1',
        'target_mode': 'combined',
        'system_mode': 'ai_plus_r1'
    }
]


PART3_REF_SIZE = 5000
PART3_GAP_SIZE = 500
PART3_CURR_SIZE = 6000
PART3_STEP_SIZE = 2000

part3_start = pd.to_datetime(PART3_START_DATE)
part3_end = pd.to_datetime(PART3_END_DATE)

# Load a wider time range for the pre-start period
df_part3 = pd.read_csv(DATA_PATH, low_memory=False)
df_part3[DATE_COL] = pd.to_datetime(df_part3[DATE_COL], format='mixed')
if 'has_cancer' not in df_part3.columns:
    df_part3['has_cancer'] = df_part3[CANCER_LABEL_COL]

# Use full available data for Part 3; start/end define calibration window only


if 'time_based_screen' not in df_part3.columns:
    if 'days_to_diagnosis_pos' not in df_part3.columns:
        if 'days_to_diagnosis' in df_part3.columns:
            df_part3['days_to_diagnosis_pos'] = df_part3['days_to_diagnosis'].apply(extract_min_positive_days)
        else:
            df_part3['days_to_diagnosis_pos'] = np.nan
    df_part3['time_based_screen'] = (
        (df_part3['has_cancer'] == 1)
        & (df_part3['days_to_diagnosis_pos'].notna())
        & (df_part3['days_to_diagnosis_pos'] <= SCREEN_THRESHOLD_DAYS)
    )

# Balance controls to 1:100 per hospital/manufacturer/month (cases fixed)
df_part3['month_key'] = df_part3[DATE_COL].dt.to_period('M').astype(str)

def balance_controls_to_ratio_monthly(df, ratio=100, label_col='has_cancer', group_cols=None, random_state=RANDOM_SEED):
    if group_cols is None:
        group_cols = [HOSPITAL_COL, MANUFACTURER_COL, 'month_key']
    balanced = []
    groups_total = 0
    groups_kept = 0
    skipped_no_cases = 0
    skipped_no_controls = 0
    for _, group in df.groupby(group_cols):
        groups_total += 1
        cases = group[group[label_col] == 1]
        controls = group[group[label_col] == 0]
        n_cases = len(cases)
        if n_cases == 0:
            skipped_no_cases += 1
            continue
        target_controls = n_cases * ratio
        if len(controls) == 0:
            skipped_no_controls += 1
            balanced.append(cases)
            groups_kept += 1
            continue
        replace = len(controls) < target_controls
        controls_sample = controls.sample(n=target_controls, replace=replace, random_state=random_state)
        balanced.append(pd.concat([cases, controls_sample], ignore_index=True))
        groups_kept += 1
    if len(balanced) == 0:
        return df.iloc[0:0].copy(), {
            'groups_total': groups_total,
            'groups_kept': groups_kept,
            'skipped_no_cases': skipped_no_cases,
            'skipped_no_controls': skipped_no_controls
        }
    balanced_df = pd.concat(balanced, ignore_index=True)
    return balanced_df, {
        'groups_total': groups_total,
        'groups_kept': groups_kept,
        'skipped_no_cases': skipped_no_cases,
        'skipped_no_controls': skipped_no_controls
    }

df_part3_raw = df_part3.copy()
df_part3, balance_stats = balance_controls_to_ratio_monthly(df_part3, ratio=100)
print('Balance stats:', balance_stats)

hospital_summary = (
    df_part3.groupby(HOSPITAL_COL)['has_cancer']
    .agg(total_exams='size', cancers='sum')
    .reset_index()
)
hospital_summary['cancers_per_1000'] = (hospital_summary['cancers'] / hospital_summary['total_exams'] * 1000)
hospital_summary['controls_per_1000'] = (
    (hospital_summary['total_exams'] - hospital_summary['cancers']) / hospital_summary['total_exams'] * 1000
)
hospital_summary


In [ ]:
# Hospital counts across all available data (by hospital)
print(f"Full data window: {df_part3[DATE_COL].min().date()} to {df_part3[DATE_COL].max().date()}")

hospital_counts = (
    df_part3.groupby(HOSPITAL_COL)
    .agg(
        total_exams=('has_cancer', 'size'),
        cancers=('has_cancer', 'sum'),
        start_date=(DATE_COL, 'min'),
        end_date=(DATE_COL, 'max')
    )
    .reset_index()
)

hospital_counts['controls'] = hospital_counts['total_exams'] - hospital_counts['cancers']
hospital_counts['period_days'] = (hospital_counts['end_date'] - hospital_counts['start_date']).dt.days
hospital_counts['start_date'] = hospital_counts['start_date'].dt.date
hospital_counts['end_date'] = hospital_counts['end_date'].dt.date
hospital_counts


In [ ]:
# Helper functions for Part 3
def compute_radiologist_flag(df, mode='combined'):
    mode = str(mode).lower()
    if mode in ('r1', 'reader1', 'reader_1'):
        return (df['r1'].astype(str) == 'DISCUSSION').astype(int)
    if mode in ('r2', 'reader2', 'reader_2'):
        return (df['r2'].astype(str) == 'DISCUSSION').astype(int)
    if mode in ('combined', 'double', 'any', 'r1_or_r2'):
        return ((df['r1'].astype(str) == 'DISCUSSION') | (df['r2'].astype(str) == 'DISCUSSION')).astype(int)
    raise ValueError("Radiologist mode must be one of: 'r1', 'r2', 'combined'.")

def compute_radiologist_sensitivity(df, mode='combined'):
    flagged = compute_radiologist_flag(df, mode=mode)
    cancers = df['has_cancer'] == 1
    if cancers.sum() == 0:
        return np.nan
    return (flagged[cancers].sum() / cancers.sum()) * 100


def compute_radiologist_flagging_rate(df, mode='combined'):
    flagged = compute_radiologist_flag(df, mode=mode)
    if len(df) == 0:
        return np.nan
    return flagged.mean() * 100


def compute_radiologist_specificity(df, mode='combined'):
    flagged = compute_radiologist_flag(df, mode=mode)
    controls = df['has_cancer'] == 0
    if controls.sum() == 0:
        return np.nan
    return (1 - (flagged[controls].mean())) * 100


def resolve_part3_modes(rad_mode=None, calib_target_mode=None, system_mode=None):
    if rad_mode is None:
        rad_mode = globals().get('PART3_RAD_MODE', 'r1')
    if calib_target_mode is None:
        calib_target_mode = globals().get('PART3_TARGET_MODE', rad_mode)
    if system_mode is None:
        system_mode = globals().get('PART3_SYSTEM_MODE', 'ai_only')
    return rad_mode, calib_target_mode, system_mode

def order_part3_df(df, ordering=None, random_state=None):
    if ordering is None:
        ordering = globals().get('PART3_ORDERING', 'date')
    if random_state is None:
        random_state = globals().get('RANDOM_SEED', 42)
    df_local = df.copy()
    if ordering == 'date':
        return df_local.sort_values(DATE_COL, kind='mergesort').reset_index(drop=True)
    if ordering == 'month_random':
        if 'month_key' not in df_local.columns:
            df_local['month_key'] = pd.to_datetime(df_local[DATE_COL], format='mixed').dt.to_period('M').astype(str)
        rng = np.random.RandomState(random_state)
        df_local['_rand'] = rng.rand(len(df_local))
        df_local = df_local.sort_values(['month_key', '_rand']).drop(columns=['_rand']).reset_index(drop=True)
        return df_local
    raise ValueError("PART3_ORDERING must be 'date' or 'month_random'")



def ensure_union_flags(df, base_suffix, r1_flag, out_suffix=None):
    union_suffix = out_suffix if out_suffix is not None else f"{base_suffix}_r1_union"
    for ai in AI_COLUMNS:
        base_col = f'{ai}_flagged_{base_suffix}'
        if base_col not in df.columns:
            df[base_col] = np.nan
        df[f'{ai}_flagged_{union_suffix}'] = (
            (df[base_col].fillna(0).astype(int) == 1) | r1_flag
        ).astype(int)
    return union_suffix

def compute_part3_transition_by_cancer_count(df_sorted, start_date, min_cancers,
                                            ref_size, gap_size, curr_size, step_size):
    # start_date kept for call-site compatibility; no date-based fallback.
    if len(df_sorted) == 0:
        return None, None, 'excluded_no_data'

    if min_cancers is None or min_cancers <= 0:
        return None, None, 'excluded_invalid_min_cancers'

    cancers = df_sorted['has_cancer'].fillna(0).astype(int)
    if cancers.sum() < min_cancers:
        return None, None, 'excluded_insufficient_cancers'

    cum_cancers = cancers.cumsum()
    idx_target = int(cum_cancers[cum_cancers >= min_cancers].index[0])

    total_span = ref_size + gap_size + curr_size
    start_idx = None
    for ref_start in range(0, len(df_sorted) - total_span + 1, step_size):
        curr_start = ref_start + ref_size + gap_size
        if curr_start > idx_target:
            start_idx = curr_start
            break

    if start_idx is None:
        return None, None, 'excluded_insufficient_exams'

    start_date = df_sorted.loc[start_idx, DATE_COL]
    return start_idx, start_date, f'cancer_count_{min_cancers}'


def _transition_center(transition_idx, curr_size):
    if transition_idx is None or curr_size is None:
        return None
    return transition_idx + (curr_size / 2)


def _confirmed_signal_centers(df, sig_mask, center_col, min_run=3):
    if df is None or len(df) == 0:
        return []
    if center_col not in df.columns:
        return []
    df_local = df.copy()
    df_local = df_local[[center_col]].reset_index(drop=True)
    centers = df_local[center_col].to_numpy()
    sig = sig_mask.reset_index(drop=True).to_numpy()

    confirmed = []
    run = 0
    for idx, is_sig in enumerate(sig):
        if is_sig:
            run += 1
            if run == min_run:
                confirmed.append(float(centers[idx]))
        else:
            run = 0
    return confirmed


def compute_ai_threshold_for_target_rate(df_pre, ai_col, target_rate):
    scores = df_pre[ai_col].dropna()
    if len(scores) == 0 or pd.isna(target_rate):
        return np.nan
    quantile = 1 - (target_rate / 100)
    if quantile < 0:
        quantile = 0
    if quantile > 1:
        quantile = 1
    return scores.quantile(quantile)

def compute_ai_threshold_for_target_sensitivity(df_pre, ai_col, target_sens):
    cancer_scores = df_pre[df_pre['has_cancer'] == 1][ai_col].dropna()
    if len(cancer_scores) == 0 or pd.isna(target_sens):
        return np.nan
    quantile = 1 - (target_sens / 100)
    return cancer_scores.quantile(quantile)

def compute_sensitivity_windows_by_flag(df, flag_suffix, ai_columns=AI_COLUMNS,
                                        ref_size=REF_SIZE, gap_size=GAP_SIZE,
                                        curr_size=CURR_SIZE, step_size=STEP_SIZE):
    results = []
    total_span = ref_size + gap_size + curr_size
    for ref_start in range(0, len(df) - total_span + 1, step_size):
        ref_end = ref_start + ref_size
        curr_start = ref_end + gap_size
        curr_end = curr_start + curr_size
        curr_center = (curr_start + curr_end) / 2
        curr_window = df.iloc[curr_start:curr_end]
        curr_cancers = curr_window[curr_window['has_cancer'] == 1]
        for ai in ai_columns:
            flag_col = f'{ai}_flagged_{flag_suffix}'
            if flag_col not in curr_window.columns:
                continue
            curr_n = len(curr_cancers)
            curr_flagged = (curr_cancers[flag_col] == 1).sum() if curr_n > 0 else 0
            curr_sens = (curr_flagged / curr_n * 100) if curr_n > 0 else np.nan
            results.append({
                'curr_center': curr_center,
                'ai_system': ai,
                'curr_sensitivity': curr_sens
            })
    return pd.DataFrame(results)

def compute_specificity_windows_by_flag(df, flag_suffix, ai_columns=AI_COLUMNS,
                                        ref_size=REF_SIZE, gap_size=GAP_SIZE,
                                        curr_size=CURR_SIZE, step_size=STEP_SIZE):
    results = []
    total_span = ref_size + gap_size + curr_size
    for ref_start in range(0, len(df) - total_span + 1, step_size):
        ref_end = ref_start + ref_size
        curr_start = ref_end + gap_size
        curr_end = curr_start + curr_size
        curr_center = (curr_start + curr_end) / 2
        curr_window = df.iloc[curr_start:curr_end]
        curr_controls = curr_window[curr_window['has_cancer'] == 0]
        for ai in ai_columns:
            flag_col = f'{ai}_flagged_{flag_suffix}'
            if flag_col not in curr_window.columns:
                continue
            curr_n = len(curr_controls)
            curr_flagged = (curr_controls[flag_col] == 1).sum() if curr_n > 0 else 0
            curr_spec = (1 - (curr_flagged / curr_n)) * 100 if curr_n > 0 else np.nan
            results.append({
                'curr_center': curr_center,
                'ai_system': ai,
                'curr_specificity': curr_spec
            })
    return pd.DataFrame(results)

def run_part3_hospital_pipeline(df_part3, hospital_name, start_date, calibration_strategy='rad_flagging_rate',
                               min_cancers=PART3_PRESTART_MIN_CANCERS, rad_mode=None, target_mode=None):
    rad_mode, calib_target_mode, _ = resolve_part3_modes(rad_mode, target_mode, None)

    df_h = df_part3[df_part3[HOSPITAL_COL] == hospital_name].copy()
    if len(df_h) == 0:
        print(f'No data for hospital: {hospital_name}')
        return None

    df_h = order_part3_df(df_h)
    if 'rad_flagged' not in df_h.columns:
        df_h['rad_flagged'] = compute_radiologist_flag(df_h, mode=rad_mode)
    transition_idx, start_date_used, start_method = compute_part3_transition_by_cancer_count(
        df_h, start_date, min_cancers,
        ref_size=PART3_REF_SIZE, gap_size=PART3_GAP_SIZE,
        curr_size=PART3_CURR_SIZE, step_size=PART3_STEP_SIZE
    )
    if transition_idx is None:
        print(f'Excluded hospital: {hospital_name} ({start_method})')
        return None

    pre_df = df_h.iloc[:transition_idx].copy()
    cancers = df_h['has_cancer'].fillna(0).astype(int)
    cum_cancers = cancers.cumsum()
    idx_target = int(cum_cancers[cum_cancers >= min_cancers].index[0])
    pre_df_cancer = df_h.iloc[:idx_target + 1].copy()
    prestart_exam_count = len(pre_df_cancer)
    prestart_cancer_count = int(cancers.iloc[:idx_target + 1].sum())
    prestart_control_count = int((cancers.iloc[:idx_target + 1] == 0).sum())

    rad_sens = {
        'r1': compute_radiologist_sensitivity(pre_df_cancer, mode='r1'),
        'r2': compute_radiologist_sensitivity(pre_df_cancer, mode='r2'),
        'combined': compute_radiologist_sensitivity(pre_df_cancer, mode='combined')
    }
    rad_spec = {
        'r1': compute_radiologist_specificity(pre_df_cancer, mode='r1'),
        'r2': compute_radiologist_specificity(pre_df_cancer, mode='r2'),
        'combined': compute_radiologist_specificity(pre_df_cancer, mode='combined')
    }
    rad_flag_rate = compute_radiologist_flagging_rate(pre_df_cancer, mode=calib_target_mode)

    thresholds = {}
    for ai in AI_COLUMNS:
        if calibration_strategy == 'rad_sensitivity':
            thresholds[ai] = compute_ai_threshold_for_target_sensitivity(
                pre_df_cancer, ai, rad_sens[calib_target_mode]
            )
        elif calibration_strategy == 'rad_flagging_rate':
            thresholds[ai] = compute_ai_threshold_for_target_rate(
                pre_df_cancer, ai, rad_flag_rate
            )
        else:
            raise ValueError('calibration_strategy must be "rad_sensitivity" or "rad_flagging_rate"')

    for ai in AI_COLUMNS:
        thr = thresholds[ai]
        df_h[f'{ai}_flagged_part3'] = (df_h[ai] >= thr).astype(int) if pd.notna(thr) else np.nan

    sens_ts = compute_sensitivity_windows_by_flag(
        df_h, flag_suffix='part3',
        ref_size=PART3_REF_SIZE, gap_size=PART3_GAP_SIZE,
        curr_size=PART3_CURR_SIZE, step_size=PART3_STEP_SIZE
    )
    spec_ts = compute_specificity_windows_by_flag(
        df_h, flag_suffix='part3',
        ref_size=PART3_REF_SIZE, gap_size=PART3_GAP_SIZE,
        curr_size=PART3_CURR_SIZE, step_size=PART3_STEP_SIZE
    )

    return {
        'hospital': hospital_name,
        'transition_index': transition_idx,
        'df_h': df_h,
        'radiologist_sensitivity_pre': rad_sens,
        'radiologist_flagging_rate_pre': rad_flag_rate,
        'radiologist_specificity_pre': rad_spec,
        'rad_mode_used': rad_mode,
        'calibration_target_mode': calib_target_mode,
        'prestart_exam_count': prestart_exam_count,
        'prestart_cancer_count': prestart_cancer_count,
        'prestart_control_count': prestart_control_count,
        'ai_thresholds': thresholds,
        'calibration_strategy': calibration_strategy,
        'start_date': start_date_used,
        'start_method': start_method,
        'sensitivity_ts': sens_ts,
        'specificity_ts': spec_ts
    }


## Part 3A: Pre-start radiologist sensitivity + AI thresholds


In [ ]:
# Summaries for the pre-start period (min cancers per hospital+manufacturer)
def build_part3_prestart_summary(df_part3, start_date, min_cancers=PART3_PRESTART_MIN_CANCERS):
    sensitivity_rows = []
    threshold_rows = []
    excluded_rows = []
    hospitals = sorted(df_part3[HOSPITAL_COL].dropna().unique())
    for hospital in hospitals:
        df_hospital = df_part3[df_part3[HOSPITAL_COL] == hospital].copy()
        if MANUFACTURER_COL in df_hospital.columns:
            manufacturers = sorted(df_hospital[MANUFACTURER_COL].dropna().unique())
            if len(manufacturers) == 0:
                manufacturers = ['Unknown']
        else:
            manufacturers = ['Unknown']

        for manufacturer in manufacturers:
            if manufacturer == 'Unknown':
                df_subset = df_hospital.copy()
            else:
                df_subset = df_hospital[df_hospital[MANUFACTURER_COL] == manufacturer].copy()
            if len(df_subset) == 0:
                continue

            df_h = order_part3_df(df_subset)
            transition_idx, start_date_used, start_method = compute_part3_transition_by_cancer_count(
                df_h, start_date, min_cancers,
                ref_size=PART3_REF_SIZE, gap_size=PART3_GAP_SIZE,
                curr_size=PART3_CURR_SIZE, step_size=PART3_STEP_SIZE
            )
            if transition_idx is None:
                total_exams = len(df_h)
                cancer_exams = int(df_h['has_cancer'].fillna(0).sum())
                excluded_rows.append({
                    'hospital': hospital,
                    'manufacturer': manufacturer,
                    'total_exams': total_exams,
                    'cancer_exams': cancer_exams,
                    'control_exams': total_exams - cancer_exams,
                    'min_cancers_required': min_cancers,
                    'exclusion_reason': start_method
                })
                continue

            pre_df = df_h.iloc[:transition_idx].copy()
            if len(pre_df) == 0:
                continue

            start_exams = f"{transition_idx} / {len(df_h)}"

            rad_sens = {
                'r1': compute_radiologist_sensitivity(pre_df, mode='r1'),
                'r2': compute_radiologist_sensitivity(pre_df, mode='r2'),
                'combined': compute_radiologist_sensitivity(pre_df, mode='combined')
            }
            sensitivity_rows.append({
                'hospital': hospital,
                'manufacturer': manufacturer,
                'start_exams': start_exams,
                'r1_sensitivity': rad_sens['r1'],
                'r2_sensitivity': rad_sens['r2'],
                'combined_sensitivity': rad_sens['combined']
            })
            for ai in AI_COLUMNS:
                thr = compute_ai_threshold_for_target_sensitivity(
                    pre_df, ai, rad_sens['combined']
                )
                threshold_rows.append({
                    'hospital': hospital,
                    'manufacturer': manufacturer,
                    'start_exams': start_exams,
                    'ai_system': ai,
                    'ai_threshold': thr
                })
    return pd.DataFrame(sensitivity_rows), pd.DataFrame(threshold_rows), pd.DataFrame(excluded_rows)

part3_radiologist_sensitivity_df, part3_ai_thresholds_df, part3_prestart_exclusions_df = build_part3_prestart_summary(
    df_part3, part3_start
)
display(part3_radiologist_sensitivity_df)
display(part3_ai_thresholds_df)
if part3_prestart_exclusions_df is not None and len(part3_prestart_exclusions_df) > 0:
    display(part3_prestart_exclusions_df)
    part3_exclusions_summary = part3_prestart_exclusions_df[['total_exams', 'cancer_exams', 'control_exams']].sum().to_frame().T
    part3_exclusions_summary['groups_excluded'] = len(part3_prestart_exclusions_df)
    display(part3_exclusions_summary)


In [ ]:
# Styled table image for radiologist sensitivity (r1/r2/combined)
from pathlib import Path

output_dir = Path('outputs/2025_12_29_counterfactual_manufacturer_swap_setup')
output_dir.mkdir(parents=True, exist_ok=True)

if part3_radiologist_sensitivity_df is not None and len(part3_radiologist_sensitivity_df) > 0:
    display_df = part3_radiologist_sensitivity_df.copy()
    display_df = display_df[
        ['hospital', 'manufacturer', 'start_exams', 'r1_sensitivity', 'r2_sensitivity', 'combined_sensitivity']
    ]
    display_df = display_df.rename(columns={
        'hospital': 'Hospital',
        'manufacturer': 'Manufacturer',
        'start_exams': 'Start exams (pre/total)',
        'r1_sensitivity': 'R1 sens (%)',
        'r2_sensitivity': 'R2 sens (%)',
        'combined_sensitivity': 'R1+R2 sens (%)'
    })
    for col in ['R1 sens (%)', 'R2 sens (%)', 'R1+R2 sens (%)']:
        if col in display_df.columns:
            display_df[col] = display_df[col].map(lambda x: '' if pd.isna(x) else f"{x:.2f}")

    fig_height = 1.8 + 0.45 * len(display_df)
    fig, ax = plt.subplots(figsize=(16, fig_height))
    ax.axis('off')

    table = ax.table(
        cellText=display_df.values,
        colLabels=display_df.columns,
        cellLoc='center',
        loc='center',
        bbox=[0, 0, 1, 1]
    )

    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1, 1.6)

    header_color = '#4472C4'
    alt_row_color = '#E7E6E6'

    for i in range(len(display_df.columns)):
        cell = table[(0, i)]
        cell.set_facecolor(header_color)
        cell.set_text_props(weight='bold', color='white')

    for i in range(1, len(display_df) + 1):
        for j in range(len(display_df.columns)):
            cell = table[(i, j)]
            cell.set_facecolor(alt_row_color if i % 2 == 0 else 'white')

    for cell in table.get_celld().values():
        cell.set_edgecolor('black')
        cell.set_linewidth(1.2)

    plt.title('Radiologist Sensitivity (Pre-start, min cancers, 36-month outcome)', fontsize=16, fontweight='bold', pad=20)
    plt.tight_layout()
    table_path = output_dir / 'part3_radiologist_sensitivity_table.png'
    plt.savefig(table_path, dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()
    table_path


In [ ]:
# Scenario calibration: single-reader vs double-read replacement

def _find_threshold_for_union_target(pre_df, ai_col, r1_flag, target_sens):
    scores = pre_df[ai_col].dropna()
    if len(scores) == 0 or pd.isna(target_sens):
        return np.nan
    cancers = pre_df['has_cancer'] == 1
    if cancers.sum() == 0:
        return np.nan

    quantiles = np.linspace(0.01, 0.99, 99)
    candidates = scores.quantile(quantiles).unique()
    candidates = np.unique(np.concatenate([candidates, [scores.max() + 1e-6]]))

    best_thr = candidates[0]
    best_diff = np.inf
    for thr in candidates:
        ai_flag = pre_df[ai_col] >= thr
        union_sens = (r1_flag | ai_flag)[cancers].mean() * 100
        diff = abs(union_sens - target_sens)
        if diff < best_diff:
            best_diff = diff
            best_thr = thr

    return best_thr


def _ai_metrics(pre_df, ai_col, threshold, r1_flag):
    if pd.isna(threshold):
        return np.nan, np.nan, np.nan, np.nan
    ai_flag = pre_df[ai_col] >= threshold
    cancers = pre_df['has_cancer'] == 1
    controls = pre_df['has_cancer'] == 0

    ai_sens = ai_flag[cancers].mean() * 100 if cancers.sum() > 0 else np.nan
    ai_spec = (1 - ai_flag[controls].mean()) * 100 if controls.sum() > 0 else np.nan
    ai_flag_rate = ai_flag.mean() * 100 if len(pre_df) > 0 else np.nan
    union_sens = (r1_flag | ai_flag)[cancers].mean() * 100 if cancers.sum() > 0 else np.nan

    return ai_sens, ai_spec, ai_flag_rate, union_sens


scenario_rows = []

hospitals = sorted(df_part3[HOSPITAL_COL].dropna().unique())
for hospital in hospitals:
    df_hospital = df_part3[df_part3[HOSPITAL_COL] == hospital].copy()
    if MANUFACTURER_COL in df_hospital.columns:
        manufacturers = sorted(df_hospital[MANUFACTURER_COL].dropna().unique())
        if len(manufacturers) == 0:
            manufacturers = ['Unknown']
    else:
        manufacturers = ['Unknown']

    for manufacturer in manufacturers:
        if manufacturer == 'Unknown':
            df_subset = df_hospital.copy()
        else:
            df_subset = df_hospital[df_hospital[MANUFACTURER_COL] == manufacturer].copy()
        if len(df_subset) == 0:
            continue

        df_h = order_part3_df(df_subset)
        transition_idx, start_date_used, start_method = compute_part3_transition_by_cancer_count(
            df_h, part3_start, PART3_PRESTART_MIN_CANCERS,
            ref_size=PART3_REF_SIZE, gap_size=PART3_GAP_SIZE,
            curr_size=PART3_CURR_SIZE, step_size=PART3_STEP_SIZE
        )
        if transition_idx is None:
            continue
        pre_df = df_h.iloc[:transition_idx].copy()
        if len(pre_df) == 0:
            continue

        r1_flag = pre_df['r1'] == 'DISCUSSION'
        r2_flag = pre_df['r2'] == 'DISCUSSION'
        cancers = pre_df['has_cancer'] == 1
        if cancers.sum() == 0:
            continue

        r1_sens = r1_flag[cancers].mean() * 100
        combined_sens = (r1_flag | r2_flag)[cancers].mean() * 100

        for ai in AI_COLUMNS:
            # Scenario A: single-reader replacement (match R1 sensitivity)
            thr_single = compute_ai_threshold_for_target_sensitivity(pre_df, ai, r1_sens)
            ai_sens, ai_spec, ai_rate, union_sens = _ai_metrics(pre_df, ai, thr_single, r1_flag)
            scenario_rows.append({
                'hospital': hospital,
                'manufacturer': manufacturer,
                'ai_system': ai,
                'scenario': 'Single reader (match R1)',
                'target_sensitivity': r1_sens,
                'ai_threshold': thr_single,
                'ai_sensitivity': ai_sens,
                'union_sensitivity': union_sens,
                'ai_specificity': ai_spec,
                'ai_flagging_rate': ai_rate
            })

            # Scenario B: double-read replacement (match R1+R2 combined)
            thr_union = _find_threshold_for_union_target(pre_df, ai, r1_flag, combined_sens)
            ai_sens_u, ai_spec_u, ai_rate_u, union_sens_u = _ai_metrics(pre_df, ai, thr_union, r1_flag)
            scenario_rows.append({
                'hospital': hospital,
                'manufacturer': manufacturer,
                'ai_system': ai,
                'scenario': 'Double read (match R1+R2)',
                'target_sensitivity': combined_sens,
                'ai_threshold': thr_union,
                'ai_sensitivity': ai_sens_u,
                'union_sensitivity': union_sens_u,
                'ai_specificity': ai_spec_u,
                'ai_flagging_rate': ai_rate_u
            })

scenario_df = pd.DataFrame(scenario_rows)

if len(scenario_df) > 0:
    display_df = scenario_df.copy()
    pct_cols = ['target_sensitivity', 'ai_sensitivity', 'union_sensitivity', 'ai_specificity', 'ai_flagging_rate']
    for col in pct_cols:
        display_df[col] = display_df[col].map(lambda x: '' if pd.isna(x) else f"{x:.2f}")
    display_df['ai_threshold'] = display_df['ai_threshold'].map(lambda x: '' if pd.isna(x) else f"{x:.4f}")

    display_df = display_df.rename(columns={
        'hospital': 'Hospital',
        'manufacturer': 'Manufacturer',
        'ai_system': 'AI system',
        'scenario': 'Scenario',
        'target_sensitivity': 'Target sens (%)',
        'ai_threshold': 'AI threshold',
        'ai_sensitivity': 'AI sens (%)',
        'union_sensitivity': 'R1+AI sens (%)',
        'ai_specificity': 'AI spec (%)',
        'ai_flagging_rate': 'AI flag rate (%)'
    })

    display_df = display_df[[
        'Hospital', 'Manufacturer', 'AI system', 'Scenario',
        'Target sens (%)', 'AI threshold', 'AI sens (%)',
        'R1+AI sens (%)', 'AI spec (%)', 'AI flag rate (%)'
    ]]

    display(display_df)

    output_dir = Path('outputs/2025_12_29_counterfactual_manufacturer_swap_setup')
    output_dir.mkdir(parents=True, exist_ok=True)
    table_path = output_dir / 'part3_ai_replacement_scenarios_table.png'

    fig_height = 1.8 + 0.45 * len(display_df)
    fig, ax = plt.subplots(figsize=(18, fig_height))
    ax.axis('off')

    table = ax.table(
        cellText=display_df.values,
        colLabels=display_df.columns,
        cellLoc='center',
        loc='center',
        bbox=[0, 0, 1, 1]
    )

    col_widths = [0.18, 0.12, 0.10, 0.18, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10]
    for col_idx, width in enumerate(col_widths):
        for (row, col), cell in table.get_celld().items():
            if col == col_idx:
                cell.set_width(width)

    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.scale(1, 1.6)

    header_color = '#4472C4'
    alt_row_color = '#E7E6E6'

    for i in range(len(display_df.columns)):
        cell = table[(0, i)]
        cell.set_facecolor(header_color)
        cell.set_text_props(weight='bold', color='white')

    for i in range(1, len(display_df) + 1):
        for j in range(len(display_df.columns)):
            cell = table[(i, j)]
            cell.set_facecolor(alt_row_color if i % 2 == 0 else 'white')

    for cell in table.get_celld().values():
        cell.set_edgecolor('black')
        cell.set_linewidth(1.0)

    plt.title('AI Replacement Scenarios (Pre-start, min cancers)', fontsize=14, fontweight='bold', pad=16)
    plt.tight_layout()
    plt.savefig(table_path, dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()
    table_path


In [ ]:
# Pre-start (min cancers) sensitivity/specificity: radiologists and AI (Scenario A/B)
def build_part3_prestart_perf_table(df_part3, start_date, min_cancers=PART3_PRESTART_MIN_CANCERS):
    rows = []
    hospitals = sorted(df_part3[HOSPITAL_COL].dropna().unique())
    for hospital in hospitals:
        df_hospital = df_part3[df_part3[HOSPITAL_COL] == hospital].copy()
        if MANUFACTURER_COL in df_hospital.columns:
            manufacturers = sorted(df_hospital[MANUFACTURER_COL].dropna().unique())
            if len(manufacturers) == 0:
                manufacturers = ['Unknown']
        else:
            manufacturers = ['Unknown']

        for manufacturer in manufacturers:
            if manufacturer == 'Unknown':
                df_subset = df_hospital.copy()
            else:
                df_subset = df_hospital[df_hospital[MANUFACTURER_COL] == manufacturer].copy()
            if len(df_subset) == 0:
                continue

            df_h = order_part3_df(df_subset)
            transition_idx, start_date_used, start_method = compute_part3_transition_by_cancer_count(
                df_h, start_date, min_cancers,
                ref_size=PART3_REF_SIZE, gap_size=PART3_GAP_SIZE,
                curr_size=PART3_CURR_SIZE, step_size=PART3_STEP_SIZE
            )
            if transition_idx is None:
                continue
            pre_df = df_h.iloc[:transition_idx].copy()
            if len(pre_df) == 0:
                continue

            start_exams = f"{transition_idx} / {len(df_h)}"

            # Radiologists (reference)
            for rad_label in ['r1', 'combined']:
                sens = compute_radiologist_sensitivity(pre_df, mode=rad_label)
                spec = compute_radiologist_specificity(pre_df, mode=rad_label)
                rows.append({
                    'Hospital': hospital,
                    'Manufacturer': manufacturer,
                    'Start exams': start_exams,
                    'Scenario': 'Reference',
                    'System': 'Radiologist R1' if rad_label == 'r1' else 'Radiologist R1+R2',
                    'Sensitivity (%)': sens,
                    'Specificity (%)': spec
                })

            r1_sens = compute_radiologist_sensitivity(pre_df, mode='r1')
            combined_sens = compute_radiologist_sensitivity(pre_df, mode='combined')
            r1_flag = compute_radiologist_flag(pre_df, mode='r1').astype(bool)

            # AI systems (Scenario A/B calibrations)
            for ai in AI_COLUMNS:
                # Scenario A: match R1 sensitivity
                thr_a = compute_ai_threshold_for_target_sensitivity(pre_df, ai, r1_sens)
                ai_sens_a, ai_spec_a, _, _ = _ai_metrics(pre_df, ai, thr_a, r1_flag)
                rows.append({
                    'Hospital': hospital,
                    'Manufacturer': manufacturer,
                    'Start exams': start_exams,
                    'Scenario': 'Scenario A (match R1)',
                    'System': ai,
                    'Sensitivity (%)': ai_sens_a,
                    'Specificity (%)': ai_spec_a
                })

                # Scenario B: match R1+R2 sensitivity
                thr_b = _find_threshold_for_union_target(pre_df, ai, r1_flag, combined_sens)
                ai_sens_b, ai_spec_b, _, _ = _ai_metrics(pre_df, ai, thr_b, r1_flag)
                rows.append({
                    'Hospital': hospital,
                    'Manufacturer': manufacturer,
                    'Start exams': start_exams,
                    'Scenario': 'Scenario B (match R1+R2)',
                    'System': ai,
                    'Sensitivity (%)': ai_sens_b,
                    'Specificity (%)': ai_spec_b
                })

    df = pd.DataFrame(rows)
    if len(df) > 0:
        df = df[['Hospital', 'Manufacturer', 'Start exams', 'Scenario', 'System', 'Sensitivity (%)', 'Specificity (%)']]
    return df

part3_prestart_perf = build_part3_prestart_perf_table(df_part3, part3_start)
part3_prestart_perf

# Styled table image
output_dir = Path('outputs/2025_12_29_counterfactual_manufacturer_swap_setup')
output_dir.mkdir(parents=True, exist_ok=True)

if part3_prestart_perf is not None and len(part3_prestart_perf) > 0:
    display_df = part3_prestart_perf.copy()
    display_df = display_df[['Hospital', 'Manufacturer', 'Start exams', 'Scenario', 'System', 'Sensitivity (%)', 'Specificity (%)']]
    for col in ['Sensitivity (%)', 'Specificity (%)']:
        if col in display_df.columns:
            display_df[col] = display_df[col].map(lambda x: '' if pd.isna(x) else f"{x:.2f}")

    fig_height = 1.8 + 0.45 * len(display_df)
    fig, ax = plt.subplots(figsize=(18, fig_height))
    ax.axis('off')

    table = ax.table(
        cellText=display_df.values,
        colLabels=display_df.columns,
        cellLoc='center',
        loc='center',
        bbox=[0, 0, 1, 1]
    )

    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1, 1.6)

    header_color = '#4472C4'
    alt_row_color = '#E7E6E6'

    for i in range(len(display_df.columns)):
        cell = table[(0, i)]
        cell.set_facecolor(header_color)
        cell.set_text_props(weight='bold', color='white')

    for i in range(1, len(display_df) + 1):
        for j in range(len(display_df.columns)):
            cell = table[(i, j)]
            cell.set_facecolor(alt_row_color if i % 2 == 0 else 'white')

    for cell in table.get_celld().values():
        cell.set_edgecolor('black')
        cell.set_linewidth(1.2)

    plt.title('Pre-start (min cancers) Sensitivity/Specificity: Radiologists + AI (Scenario A/B)', fontsize=16, fontweight='bold', pad=20)
    plt.tight_layout()
    table_path = output_dir / 'part3_prestart_sens_spec_table.png'
    plt.savefig(table_path, dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()
    table_path


In [ ]:
# Time-based windows (6-month cumulative) for screen-detected monitoring in Part 3
def _iter_time_windows(df_sorted, ref_months, gap_months, curr_months, step_months):
    if len(df_sorted) == 0:
        return
    start_date = df_sorted[DATE_COL].min()
    end_date = df_sorted[DATE_COL].max()
    current_start = start_date + pd.DateOffset(months=ref_months + gap_months)
    while current_start + pd.DateOffset(months=curr_months) <= end_date:
        ref_start = current_start - pd.DateOffset(months=ref_months + gap_months)
        ref_end = current_start - pd.DateOffset(months=gap_months)
        curr_end = current_start + pd.DateOffset(months=curr_months)
        yield ref_start, ref_end, current_start, curr_end
        current_start = current_start + pd.DateOffset(months=step_months)

def compute_flagging_rates_time(df, subset_col=None, subset_value=True,
                                ref_months=6, gap_months=0, curr_months=6, step_months=6):
    df_sorted = df.sort_values(DATE_COL).reset_index(drop=True)
    results = {
        'radiologist': [],
        'lunit': [],
        'therapixel': [],
        'vara': []
    }
    for ref_start, ref_end, curr_start, curr_end in _iter_time_windows(
        df_sorted, ref_months, gap_months, curr_months, step_months
    ):
        ref_window = df_sorted[(df_sorted[DATE_COL] >= ref_start) & (df_sorted[DATE_COL] < ref_end)]
        curr_window = df_sorted[(df_sorted[DATE_COL] >= curr_start) & (df_sorted[DATE_COL] < curr_end)]
        if subset_col is not None:
            ref_window = ref_window[ref_window[subset_col] == subset_value]
            curr_window = curr_window[curr_window[subset_col] == subset_value]
        if len(curr_window) > 0:
            curr_center = float(curr_window.index.to_numpy().mean())
        else:
            curr_center = np.nan

        ref_n = len(ref_window)
        curr_n = len(curr_window)

        if ref_n > 0:
            ref_rad_flagged = ref_window['rad_flagged'].sum()
            ref_rad_pct = (ref_rad_flagged / ref_n * 100)
        else:
            ref_rad_flagged = 0
            ref_rad_pct = np.nan

        if curr_n > 0:
            curr_rad_flagged = curr_window['rad_flagged'].sum()
            curr_rad_pct = (curr_rad_flagged / curr_n * 100)
        else:
            curr_rad_flagged = 0
            curr_rad_pct = np.nan

        if ref_n > 0 and curr_n > 0:
            count = np.array([curr_rad_flagged, ref_rad_flagged])
            nobs = np.array([curr_n, ref_n])
            _, p_val_rad = proportions_ztest(count, nobs, alternative='two-sided')
        else:
            p_val_rad = 1.0

        results['radiologist'].append({
            'curr_center': curr_center,
            'ref_pct': ref_rad_pct,
            'curr_pct': curr_rad_pct,
            'p_value': p_val_rad
        })

        for ai_name in AI_COLUMNS:
            flag_col = f'{ai_name}_flagged'
            if ref_n > 0:
                ref_ai_flagged = ref_window[flag_col].sum()
                ref_ai_pct = (ref_ai_flagged / ref_n * 100)
            else:
                ref_ai_flagged = 0
                ref_ai_pct = np.nan
            if curr_n > 0:
                curr_ai_flagged = curr_window[flag_col].sum()
                curr_ai_pct = (curr_ai_flagged / curr_n * 100)
            else:
                curr_ai_flagged = 0
                curr_ai_pct = np.nan
            if ref_n > 0 and curr_n > 0:
                count = np.array([curr_ai_flagged, ref_ai_flagged])
                nobs = np.array([curr_n, ref_n])
                _, p_val_ai = proportions_ztest(count, nobs, alternative='two-sided')
            else:
                p_val_ai = 1.0
            results[ai_name].append({
                'curr_center': curr_center,
                'ref_pct': ref_ai_pct,
                'curr_pct': curr_ai_pct,
                'p_value': p_val_ai
            })

    for system in results.keys():
        results[system] = pd.DataFrame(results[system])
        if len(results[system]) > 0:
            _, p_corrected, _, _ = multipletests(results[system]['p_value'], method='fdr_bh')
            results[system]['p_value_corrected'] = p_corrected
            results[system]['is_significant'] = p_corrected < 0.05
    return results

def compute_conditional_overlap_time(df, subset_col=None, subset_value=True,
                                     ref_months=6, gap_months=0, curr_months=6, step_months=6):
    df_sorted = df.sort_values(DATE_COL).reset_index(drop=True)
    overlap_results = {}
    for ai_name in AI_COLUMNS:
        rows = []
        for ref_start, ref_end, curr_start, curr_end in _iter_time_windows(
            df_sorted, ref_months, gap_months, curr_months, step_months
        ):
            ref_window = df_sorted[(df_sorted[DATE_COL] >= ref_start) & (df_sorted[DATE_COL] < ref_end)]
            curr_window = df_sorted[(df_sorted[DATE_COL] >= curr_start) & (df_sorted[DATE_COL] < curr_end)]
            if subset_col is not None:
                ref_window = ref_window[ref_window[subset_col] == subset_value]
                curr_window = curr_window[curr_window[subset_col] == subset_value]
            if len(curr_window) > 0:
                curr_center = float(curr_window.index.to_numpy().mean())
            else:
                curr_center = np.nan
            ref_ai_flagged = ref_window[f'{ai_name}_flagged'].sum()
            ref_rad_flagged = ref_window['rad_flagged'].sum()
            ref_both = (ref_window[f'{ai_name}_flagged'] & ref_window['rad_flagged']).sum()
            curr_ai_flagged = curr_window[f'{ai_name}_flagged'].sum()
            curr_rad_flagged = curr_window['rad_flagged'].sum()
            curr_both = (curr_window[f'{ai_name}_flagged'] & curr_window['rad_flagged']).sum()
            ref_p_ai_given_rad = (ref_both / ref_rad_flagged * 100) if ref_rad_flagged > 0 else np.nan
            curr_p_ai_given_rad = (curr_both / curr_rad_flagged * 100) if curr_rad_flagged > 0 else np.nan
            ref_p_rad_given_ai = (ref_both / ref_ai_flagged * 100) if ref_ai_flagged > 0 else np.nan
            curr_p_rad_given_ai = (curr_both / curr_ai_flagged * 100) if curr_ai_flagged > 0 else np.nan
            if ref_rad_flagged > 0 and curr_rad_flagged > 0:
                count = np.array([curr_both, ref_both])
                nobs = np.array([curr_rad_flagged, ref_rad_flagged])
                _, p_val_ai_given_rad = proportions_ztest(count, nobs, alternative='two-sided')
            else:
                p_val_ai_given_rad = 1.0
            if ref_ai_flagged > 0 and curr_ai_flagged > 0:
                count = np.array([curr_both, ref_both])
                nobs = np.array([curr_ai_flagged, ref_ai_flagged])
                _, p_val_rad_given_ai = proportions_ztest(count, nobs, alternative='two-sided')
            else:
                p_val_rad_given_ai = 1.0
            rows.append({
                'curr_center': curr_center,
                'ref_p_ai_given_rad': ref_p_ai_given_rad,
                'curr_p_ai_given_rad': curr_p_ai_given_rad,
                'ref_p_rad_given_ai': ref_p_rad_given_ai,
                'curr_p_rad_given_ai': curr_p_rad_given_ai,
                'p_val_ai_given_rad': p_val_ai_given_rad,
                'p_val_rad_given_ai': p_val_rad_given_ai
            })
        df_results = pd.DataFrame(rows)
        if len(df_results) > 0:
            _, p_corr_ai_rad, _, _ = multipletests(df_results['p_val_ai_given_rad'], method='fdr_bh')
            _, p_corr_rad_ai, _, _ = multipletests(df_results['p_val_rad_given_ai'], method='fdr_bh')
            df_results['p_val_ai_given_rad_corrected'] = p_corr_ai_rad
            df_results['p_val_rad_given_ai_corrected'] = p_corr_rad_ai
            df_results['sig_ai_given_rad'] = p_corr_ai_rad < 0.05
            df_results['sig_rad_given_ai'] = p_corr_rad_ai < 0.05
        overlap_results[ai_name] = df_results
    return overlap_results


## Part 3B: Sensitivity and specificity over time (example hospital)


In [ ]:
# Part 3: per-hospital detection summary (one test per input type)
if '_confirmed_signal_centers' not in globals():
    def _confirmed_signal_centers(df, sig_mask, center_col, min_run=3):
        if df is None or len(df) == 0:
            return []
        if center_col not in df.columns:
            return []
        df_local = df.copy()
        df_local = df_local[[center_col]].reset_index(drop=True)
        centers = df_local[center_col].to_numpy()
        sig = sig_mask.reset_index(drop=True).to_numpy()

        confirmed = []
        run = 0
        for idx, is_sig in enumerate(sig):
            if is_sig:
                run += 1
                if run == min_run:
                    confirmed.append(float(centers[idx]))
            else:
                run = 0
        return confirmed


def _count_signals(df, p_col, alpha=0.05, center_col='curr_center', min_run=None):
    if df is None or len(df) == 0 or p_col not in df.columns:
        return 0
    if min_run is None:
        min_run = globals().get('PART3_SIGNAL_MIN_RUN', 3)
    sig = df[p_col] < alpha
    centers = _confirmed_signal_centers(df, sig, center_col, min_run=min_run)
    return int(len(centers))

def build_part3_detection_summary(df_part3, alpha=0.05,
                                  screen_months=6, screen_step_months=6):
    rows = []
    hospitals = sorted(df_part3[HOSPITAL_COL].dropna().unique())
    for hospital in hospitals:
        df_hospital = df_part3[df_part3[HOSPITAL_COL] == hospital].copy()
        if len(df_hospital) == 0:
            continue
        if MANUFACTURER_COL in df_hospital.columns:
            manufacturers = sorted(df_hospital[MANUFACTURER_COL].dropna().unique())
            if len(manufacturers) == 0:
                manufacturers = ['Unknown']
        else:
            manufacturers = ['Unknown']

        for manufacturer in manufacturers:
            if manufacturer == 'Unknown':
                df_subset = df_hospital.copy()
            else:
                df_subset = df_hospital[df_hospital[MANUFACTURER_COL] == manufacturer].copy()
            if len(df_subset) == 0:
                continue

            result = run_part3_hospital_pipeline(
                df_subset, hospital, part3_start, calibration_strategy=PART3_CALIBRATION_STRATEGY
            )
            if result is None:
                continue
            df_subset = result['df_h']

            if 'rad_flagged' not in df_subset.columns:
                df_subset['rad_flagged'] = compute_radiologist_flag(df_subset, mode=PART3_RAD_MODE)

            for ai in AI_COLUMNS:
                base_flag = f'{ai}_flagged'
                part3_flag = f'{ai}_flagged_part3'
                if base_flag not in df_subset.columns and part3_flag in df_subset.columns:
                    df_subset[base_flag] = df_subset[part3_flag]

            score_results = sliding_window_monitoring(
                df_subset, ai_columns=AI_COLUMNS,
                ref_size=PART3_REF_SIZE, gap_size=PART3_GAP_SIZE,
                curr_size=PART3_CURR_SIZE, step_size=PART3_STEP_SIZE
            )
            flagging_results = compute_flagging_rates(
                df_subset, ref_size=PART3_REF_SIZE, gap_size=PART3_GAP_SIZE,
                curr_size=PART3_CURR_SIZE, step_size=PART3_STEP_SIZE
            )
            relative_flagging_results = compute_relative_flagging_rates(flagging_results)
            overlap_results = compute_conditional_overlap(
                df_subset, ref_size=PART3_REF_SIZE, gap_size=PART3_GAP_SIZE,
                curr_size=PART3_CURR_SIZE, step_size=PART3_STEP_SIZE
            )

            screen_flagging = compute_flagging_rates_time(
                df_subset, subset_col='time_based_screen', subset_value=True,
                ref_months=screen_months, gap_months=0,
                curr_months=screen_months, step_months=screen_step_months
            )
            screen_overlap = compute_conditional_overlap_time(
                df_subset, subset_col='time_based_screen', subset_value=True,
                ref_months=screen_months, gap_months=0,
                curr_months=screen_months, step_months=screen_step_months
            )

            for ai in AI_COLUMNS:
                ai_score = score_results[score_results['ai_system'] == ai] if score_results is not None else pd.DataFrame()
                ai_flag = flagging_results.get(ai) if flagging_results is not None else None
                ai_rel = relative_flagging_results.get(ai) if relative_flagging_results is not None else None
                ai_overlap = overlap_results.get(ai) if overlap_results is not None else None
                ai_screen_flag = screen_flagging.get(ai) if screen_flagging is not None else None
                ai_screen_overlap = screen_overlap.get(ai) if screen_overlap is not None else None

                rows.append({
                    'hospital': hospital,
                    'manufacturer': manufacturer,
                    'ai_system': ai,
                    'parameter': 'AI score (median)',
                    'test': 'median',
                    'signal_count': _count_signals(ai_score, 'median_p_fdr', alpha, center_col='window_center')
                })
                rows.append({
                    'hospital': hospital,
                    'manufacturer': manufacturer,
                    'ai_system': ai,
                    'parameter': 'Flagging rate',
                    'test': 'z-test',
                    'signal_count': _count_signals(ai_flag, 'p_value_corrected', alpha) if ai_flag is not None else 0
                })
                rows.append({
                    'hospital': hospital,
                    'manufacturer': manufacturer,
                    'ai_system': ai,
                    'parameter': 'Relative flagging rate',
                    'test': 'ratio',
                    'signal_count': _count_signals(ai_rel, 'p_value_corrected', alpha) if ai_rel is not None else 0
                })
                rows.append({
                    'hospital': hospital,
                    'manufacturer': manufacturer,
                    'ai_system': ai,
                    'parameter': 'P(AI | Radiologist)',
                    'test': 'z-test',
                    'signal_count': _count_signals(ai_overlap, 'p_val_ai_given_rad_corrected', alpha) if ai_overlap is not None else 0
                })
                rows.append({
                    'hospital': hospital,
                    'manufacturer': manufacturer,
                    'ai_system': ai,
                    'parameter': 'P(Radiologist | AI)',
                    'test': 'z-test',
                    'signal_count': _count_signals(ai_overlap, 'p_val_rad_given_ai_corrected', alpha) if ai_overlap is not None else 0
                })
                rows.append({
                    'hospital': hospital,
                    'manufacturer': manufacturer,
                    'ai_system': ai,
                    'parameter': 'Screen-detected flagging rate (6-mo)',
                    'test': 'z-test',
                    'signal_count': _count_signals(ai_screen_flag, 'p_value_corrected', alpha) if ai_screen_flag is not None else 0
                })
                rows.append({
                    'hospital': hospital,
                    'manufacturer': manufacturer,
                    'ai_system': ai,
                    'parameter': 'Screen-detected P(AI | Radiologist) (6-mo)',
                    'test': 'z-test',
                    'signal_count': _count_signals(ai_screen_overlap, 'p_val_ai_given_rad_corrected', alpha) if ai_screen_overlap is not None else 0
                })
                rows.append({
                    'hospital': hospital,
                    'manufacturer': manufacturer,
                    'ai_system': ai,
                    'parameter': 'Screen-detected P(Radiologist | AI) (6-mo)',
                    'test': 'z-test',
                    'signal_count': _count_signals(ai_screen_overlap, 'p_val_rad_given_ai_corrected', alpha) if ai_screen_overlap is not None else 0
                })

    return pd.DataFrame(rows)

RUN_PART3_DETECTION_SUMMARY = False  # set True to run the slow detection summary

if RUN_PART3_DETECTION_SUMMARY:
    part3_detection_summary = build_part3_detection_summary(df_part3, alpha=0.05)
    part3_detection_summary
else:
    print('Skipped Part 3 detection summary (set RUN_PART3_DETECTION_SUMMARY=True to run).')


In [ ]:
if '_confirmed_signal_centers' not in globals():
    def _confirmed_signal_centers(df, sig_mask, center_col, min_run=3):
        if df is None or len(df) == 0:
            return []
        if center_col not in df.columns:
            return []
        df_local = df.copy()
        df_local = df_local[[center_col]].reset_index(drop=True)
        centers = df_local[center_col].to_numpy()
        sig = sig_mask.reset_index(drop=True).to_numpy()

        confirmed = []
        run = 0
        for idx, is_sig in enumerate(sig):
            if is_sig:
                run += 1
                if run == min_run:
                    confirmed.append(float(centers[idx]))
            else:
                run = 0
        return confirmed

def compute_rate_windows_by_flag(df, flag_suffix, rate_kind, ai_columns=AI_COLUMNS,
                                ref_size=REF_SIZE, gap_size=GAP_SIZE,
                                curr_size=CURR_SIZE, step_size=STEP_SIZE):
    if rate_kind not in {'sensitivity', 'specificity'}:
        raise ValueError("rate_kind must be 'sensitivity' or 'specificity'")
    results = []
    total_span = ref_size + gap_size + curr_size

    for ref_start in range(0, len(df) - total_span + 1, step_size):
        ref_end = ref_start + ref_size
        curr_start = ref_end + gap_size
        curr_end = curr_start + curr_size
        curr_center = (curr_start + curr_end) / 2

        ref_window = df.iloc[ref_start:ref_end]
        curr_window = df.iloc[curr_start:curr_end]

        ref_subset = ref_window[ref_window['has_cancer'] == (1 if rate_kind == 'sensitivity' else 0)]
        curr_subset = curr_window[curr_window['has_cancer'] == (1 if rate_kind == 'sensitivity' else 0)]

        for ai in ai_columns:
            flag_col = f'{ai}_flagged_{flag_suffix}'
            if flag_col not in curr_window.columns:
                continue

            ref_n = len(ref_subset)
            curr_n = len(curr_subset)
            if rate_kind == 'sensitivity':
                ref_success = (ref_subset[flag_col] == 1).sum() if ref_n > 0 else 0
                curr_success = (curr_subset[flag_col] == 1).sum() if curr_n > 0 else 0
            else:
                ref_success = (ref_subset[flag_col] == 0).sum() if ref_n > 0 else 0
                curr_success = (curr_subset[flag_col] == 0).sum() if curr_n > 0 else 0

            ref_rate = (ref_success / ref_n * 100) if ref_n > 0 else np.nan
            curr_rate = (curr_success / curr_n * 100) if curr_n > 0 else np.nan

            if ref_n > 0 and curr_n > 0:
                count = np.array([curr_success, ref_success])
                nobs = np.array([curr_n, ref_n])
                _, p_val = proportions_ztest(count, nobs, alternative='two-sided')
            else:
                p_val = 1.0

            results.append({
                'curr_center': curr_center,
                'ai_system': ai,
                'ref_rate': ref_rate,
                'curr_rate': curr_rate,
                'p_value': p_val
            })

    results_df = pd.DataFrame(results)
    if len(results_df) == 0:
        return pd.DataFrame(columns=['curr_center', 'ai_system', 'ref_rate', 'curr_rate', 'p_value', 'p_value_corrected'])
    for ai in ai_columns:
        ai_mask = results_df['ai_system'] == ai
        p_vals = results_df.loc[ai_mask, 'p_value'].values
        if len(p_vals) == 0:
            continue
        _, p_corrected, _, _ = multipletests(p_vals, method='fdr_bh')
        results_df.loc[ai_mask, 'p_value_corrected'] = p_corrected
    return results_df

def plot_part3_rate_results(rate_df, transition_point, title, ylabel, gap_label, alpha=0.05, figsize=(14, 12), auto_width=True):
    min_run = globals().get('PART3_SIGNAL_MIN_RUN', 3)
    fig_w, fig_h = figsize
    if auto_width and rate_df is not None and len(rate_df) > 0:
        n_points = rate_df['curr_center'].nunique() if 'curr_center' in rate_df.columns else len(rate_df)
        fig_w = min(24, max(fig_w, 10 + n_points / 50))
    fig, axes = plt.subplots(3, 1, figsize=(fig_w, fig_h), sharex=True)
    colors = {'lunit': '#ff7f0e', 'therapixel': '#2ca02c', 'vara': '#d62728'}

    for idx, ai in enumerate(AI_COLUMNS):
        ax = axes[idx]
        data = rate_df[rate_df['ai_system'] == ai] if rate_df is not None else None

        if data is None or len(data) == 0:
            ax.axvspan(0, transition_point, alpha=0.1, color='blue', label='Pre-start')
            ax.axvspan(transition_point, transition_point * 2, alpha=0.1, color='orange', label='Post-start')
            ax.text(0.5, 0.5, 'No data', ha='center', va='center')
            ax.axvline(transition_point, color='red', linestyle='--', linewidth=2, label='Start')
            ax.set_ylabel(ylabel)
            ax.set_title(ai.upper())
            ax.grid(True, alpha=0.3)
            ax.legend(fontsize=9, loc='upper left')
            continue

        x_min = data['curr_center'].min() if len(data) > 0 else 0
        x_max = data['curr_center'].max() if len(data) > 0 else transition_point * 2
        ax.axvspan(x_min, transition_point, alpha=0.1, color='blue', label='Pre-start')
        ax.axvspan(transition_point, x_max, alpha=0.1, color='orange', label='Post-start')

        ax.plot(data['curr_center'], data['curr_rate'], color=colors[ai], linewidth=2, label=ai.upper())

        sig_mask = data['p_value_corrected'] < alpha if 'p_value_corrected' in data.columns else pd.Series([False] * len(data))
        confirmed_centers = _confirmed_signal_centers(data, sig_mask, 'curr_center', min_run=min_run)
        if confirmed_centers:
            sig_data = data[data['curr_center'].isin(confirmed_centers)]
            ax.scatter(sig_data['curr_center'], sig_data['curr_rate'], color='black', s=40, label='Signal')

        if data['curr_rate'].isna().any():
            ax.text(0.98, 0.98, gap_label,
                    transform=ax.transAxes, fontsize=8, color='gray',
                    verticalalignment='top', horizontalalignment='right')

        ax.axvline(transition_point, color='red', linestyle='--', linewidth=2, label='Start')
        ax.set_ylabel(ylabel)
        ax.set_title(ai.upper())
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=9, loc='upper left')

    axes[-1].set_xlabel('Current Window Center (Exam Index)')
    plt.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

# Example hospital run (adjust as needed)
PART3_EXAMPLE_HOSPITAL = HOSPITAL_PHILIPS
PART3_EXAMPLE_MANUFACTURER = None  # set to a manufacturer name to pin

df_h = df_part3[df_part3[HOSPITAL_COL] == PART3_EXAMPLE_HOSPITAL].copy()
if len(df_h) == 0:
    print(f'No data for hospital: {PART3_EXAMPLE_HOSPITAL}')
else:
    if MANUFACTURER_COL in df_h.columns:
        if PART3_EXAMPLE_MANUFACTURER is None:
            manufacturers = sorted(df_h[MANUFACTURER_COL].dropna().unique())
            PART3_EXAMPLE_MANUFACTURER = manufacturers[0] if len(manufacturers) > 0 else None
        if PART3_EXAMPLE_MANUFACTURER is not None:
            df_h = df_h[df_h[MANUFACTURER_COL] == PART3_EXAMPLE_MANUFACTURER].copy()

    if len(df_h) == 0:
        print(f'No data for hospital/manufacturer: {PART3_EXAMPLE_HOSPITAL} / {PART3_EXAMPLE_MANUFACTURER}')
    else:
        df_h = order_part3_df(df_h)
        if 'rad_flagged' not in df_h.columns:
            df_h['rad_flagged'] = compute_radiologist_flag(df_h, mode=PART3_RAD_MODE)

        transition_idx, start_date_used, start_method = compute_part3_transition_by_cancer_count(
            df_h, part3_start, PART3_PRESTART_MIN_CANCERS,
            ref_size=PART3_REF_SIZE, gap_size=PART3_GAP_SIZE,
            curr_size=PART3_CURR_SIZE, step_size=PART3_STEP_SIZE
        )
        if transition_idx is None:
            print(f'Excluded hospital/manufacturer: {PART3_EXAMPLE_HOSPITAL} / {PART3_EXAMPLE_MANUFACTURER} ({start_method})')

        pre_df = df_h.iloc[:transition_idx].copy() if transition_idx is not None else df_h.iloc[:0].copy()
        r1_flag = pre_df['r1'] == 'DISCUSSION'
        r2_flag = pre_df['r2'] == 'DISCUSSION'
        cancers = pre_df['has_cancer'] == 1

        if cancers.sum() == 0:
            print('No cancers in pre-start window; cannot calibrate scenarios.')
        else:
            r1_sens = r1_flag[cancers].mean() * 100
            combined_sens = (r1_flag | r2_flag)[cancers].mean() * 100

            def _find_threshold_for_union_target(pre_df, ai_col, r1_flag, target_sens):
                scores = pre_df[ai_col].dropna()
                if len(scores) == 0 or pd.isna(target_sens):
                    return np.nan
                cancers = pre_df['has_cancer'] == 1
                if cancers.sum() == 0:
                    return np.nan

                quantiles = np.linspace(0.01, 0.99, 99)
                candidates = scores.quantile(quantiles).unique()
                candidates = np.unique(np.concatenate([candidates, [scores.max() + 1e-6]]))

                best_thr = candidates[0]
                best_diff = np.inf
                for thr in candidates:
                    ai_flag = pre_df[ai_col] >= thr
                    union_sens = (r1_flag | ai_flag)[cancers].mean() * 100
                    diff = abs(union_sens - target_sens)
                    if diff < best_diff:
                        best_diff = diff
                        best_thr = thr

                return best_thr

            for ai in AI_COLUMNS:
                # Scenario A: match R1 sensitivity
                thr_r1 = compute_ai_threshold_for_target_sensitivity(pre_df, ai, r1_sens)
                df_h[f'{ai}_flagged_part3_r1'] = (df_h[ai] >= thr_r1).astype(int) if pd.notna(thr_r1) else np.nan

                # Scenario B: match R1+R2 combined sensitivity using R1+AI union
                thr_union = _find_threshold_for_union_target(pre_df, ai, r1_flag, combined_sens)
                df_h[f'{ai}_flagged_part3_union'] = (df_h[ai] >= thr_union).astype(int) if pd.notna(thr_union) else np.nan

            sens_r1 = compute_rate_windows_by_flag(
                df_h, flag_suffix='part3_r1', rate_kind='sensitivity',
                ref_size=PART3_REF_SIZE, gap_size=PART3_GAP_SIZE,
                curr_size=PART3_CURR_SIZE, step_size=PART3_STEP_SIZE
            )
            spec_r1 = compute_rate_windows_by_flag(
                df_h, flag_suffix='part3_r1', rate_kind='specificity',
                ref_size=PART3_REF_SIZE, gap_size=PART3_GAP_SIZE,
                curr_size=PART3_CURR_SIZE, step_size=PART3_STEP_SIZE
            )

            sens_union = compute_rate_windows_by_flag(
                df_h, flag_suffix='part3_union', rate_kind='sensitivity',
                ref_size=PART3_REF_SIZE, gap_size=PART3_GAP_SIZE,
                curr_size=PART3_CURR_SIZE, step_size=PART3_STEP_SIZE
            )
            spec_union = compute_rate_windows_by_flag(
                df_h, flag_suffix='part3_union', rate_kind='specificity',
                ref_size=PART3_REF_SIZE, gap_size=PART3_GAP_SIZE,
                curr_size=PART3_CURR_SIZE, step_size=PART3_STEP_SIZE
            )

            data_range = f"{df_h[DATE_COL].min().date()} to {df_h[DATE_COL].max().date()}"
            manufacturer_label = PART3_EXAMPLE_MANUFACTURER or 'All manufacturers'
            start_label = str(start_method).replace('_', ' ')
            base_title = f"{PART3_EXAMPLE_HOSPITAL} | {manufacturer_label} | Data: {data_range} | Start: {start_date_used.date()} ({start_label})"

            plot_part3_rate_results(
                sens_r1, transition_idx,
                title=f"Part 3 Sensitivity - Scenario A (match R1) | {base_title}",
                ylabel='Sensitivity (%)',
                gap_label='Gaps = no cancers in window'
            )
            plot_part3_rate_results(
                spec_r1, transition_idx,
                title=f"Part 3 Specificity - Scenario A (match R1) | {base_title}",
                ylabel='Specificity (%)',
                gap_label='Gaps = no controls in window'
            )

            plot_part3_rate_results(
                sens_union, transition_idx,
                title=f"Part 3 Sensitivity - Scenario B (match R1+R2) | {base_title}",
                ylabel='Sensitivity (%)',
                gap_label='Gaps = no cancers in window'
            )
            plot_part3_rate_results(
                spec_union, transition_idx,
                title=f"Part 3 Specificity - Scenario B (match R1+R2) | {base_title}",
                ylabel='Specificity (%)',
                gap_label='Gaps = no controls in window'
            )


## Part 3C: All-hospital summary + Part 2 signal overlays


In [ ]:
# Build a consolidated summary table across all hospitals (Scenario A/B)
def build_part3_all_hospitals_summary(df_part3, start_date, min_cancers=PART3_PRESTART_MIN_CANCERS):
    rows = []
    hospitals = sorted(df_part3[HOSPITAL_COL].dropna().unique())
    for hospital in hospitals:
        df_hospital = df_part3[df_part3[HOSPITAL_COL] == hospital].copy()
        if MANUFACTURER_COL in df_hospital.columns:
            manufacturers = sorted(df_hospital[MANUFACTURER_COL].dropna().unique())
            if len(manufacturers) == 0:
                manufacturers = ['Unknown']
        else:
            manufacturers = ['Unknown']

        for manufacturer in manufacturers:
            if manufacturer == 'Unknown':
                df_subset = df_hospital.copy()
            else:
                df_subset = df_hospital[df_hospital[MANUFACTURER_COL] == manufacturer].copy()
            if len(df_subset) == 0:
                continue

            df_h = order_part3_df(df_subset)
            transition_idx, start_date_used, start_method = compute_part3_transition_by_cancer_count(
                df_h, start_date, min_cancers,
                ref_size=PART3_REF_SIZE, gap_size=PART3_GAP_SIZE,
                curr_size=PART3_CURR_SIZE, step_size=PART3_STEP_SIZE
            )
            if transition_idx is None:
                continue
            pre_df = df_h.iloc[:transition_idx].copy()
            transition_center = _transition_center(transition_idx, PART3_CURR_SIZE)
            if len(pre_df) == 0:
                continue

            r1_sens = compute_radiologist_sensitivity(pre_df, mode='r1')
            combined_sens = compute_radiologist_sensitivity(pre_df, mode='combined')
            r1_flag = compute_radiologist_flag(pre_df, mode='r1').astype(bool)

            scenarios = [
                {
                    'key': 'part3_a',
                    'label': 'Scenario A (match R1)',
                    'target_sens': r1_sens,
                    'thresholds': {
                        ai: compute_ai_threshold_for_target_sensitivity(pre_df, ai, r1_sens)
                        for ai in AI_COLUMNS
                    }
                },
                {
                    'key': 'part3_b',
                    'label': 'Scenario B (match R1+R2)',
                    'target_sens': combined_sens,
                    'thresholds': {
                        ai: _find_threshold_for_union_target(pre_df, ai, r1_flag, combined_sens)
                        for ai in AI_COLUMNS
                    }
                }
            ]

            for scenario in scenarios:
                for ai in AI_COLUMNS:
                    thr = scenario['thresholds'].get(ai, np.nan)
                    flag_col = f"{ai}_flagged_{scenario['key']}"
                    df_h[flag_col] = (df_h[ai] >= thr).astype(int) if pd.notna(thr) else np.nan

                sens_rates = compute_rate_windows_by_flag(
                    df_h, flag_suffix=scenario['key'], rate_kind='sensitivity',
                    ref_size=PART3_REF_SIZE, gap_size=PART3_GAP_SIZE,
                    curr_size=PART3_CURR_SIZE, step_size=PART3_STEP_SIZE
                )
                spec_rates = compute_rate_windows_by_flag(
                    df_h, flag_suffix=scenario['key'], rate_kind='specificity',
                    ref_size=PART3_REF_SIZE, gap_size=PART3_GAP_SIZE,
                    curr_size=PART3_CURR_SIZE, step_size=PART3_STEP_SIZE
                )

                for ai in AI_COLUMNS:
                    if sens_rates is None or 'ai_system' not in sens_rates.columns:
                        sens_ai = pd.DataFrame()
                    else:
                        sens_ai = sens_rates[sens_rates['ai_system'] == ai]
                    if spec_rates is None or 'ai_system' not in spec_rates.columns:
                        spec_ai = pd.DataFrame()
                    else:
                        spec_ai = spec_rates[spec_rates['ai_system'] == ai]

                    thr = scenario['thresholds'].get(ai, np.nan)
                    pre_calib_sens = np.nan
                    pre_calib_spec = np.nan
                    if pd.notna(thr):
                        pre_flags = (pre_df[ai] >= thr)
                        cancers = pre_df['has_cancer'] == 1
                        controls = pre_df['has_cancer'] == 0
                        if cancers.sum() > 0:
                            pre_calib_sens = pre_flags[cancers].mean() * 100
                        if controls.sum() > 0:
                            pre_calib_spec = (1 - pre_flags[controls].mean()) * 100

                    sens_post = sens_ai[sens_ai['curr_center'] >= transition_center]['curr_rate'].mean() if len(sens_ai) > 0 else np.nan
                    spec_post = spec_ai[spec_ai['curr_center'] >= transition_center]['curr_rate'].mean() if len(spec_ai) > 0 else np.nan

                    rows.append({
                        'hospital': hospital,
                        'manufacturer': manufacturer,
                        'scenario': scenario['label'],
                        'ai_system': ai,
                        'target_sensitivity': scenario['target_sens'],
                        'ai_threshold': scenario['thresholds'].get(ai, np.nan),
                        'sens_pre_mean': pre_calib_sens,
                        'sens_post_mean': sens_post,
                        'spec_pre_mean': pre_calib_spec,
                        'spec_post_mean': spec_post
                    })

    return pd.DataFrame(rows)
part3_all_hospitals_summary = build_part3_all_hospitals_summary(df_part3, part3_start)
part3_all_hospitals_summary


In [ ]:
# Styled table image for Part 3 all-hospital summary
from pathlib import Path

output_dir = Path('outputs/2025_12_29_counterfactual_manufacturer_swap_setup')
output_dir.mkdir(parents=True, exist_ok=True)

if part3_all_hospitals_summary is not None and len(part3_all_hospitals_summary) > 0:
    display_df = part3_all_hospitals_summary.copy()
    display_df = display_df[
        ['hospital', 'manufacturer', 'scenario', 'ai_system', 'target_sensitivity', 'ai_threshold',
         'sens_pre_mean', 'sens_post_mean', 'spec_pre_mean', 'spec_post_mean']
    ]
    display_df = display_df.rename(columns={
        'hospital': 'Hospital',
        'manufacturer': 'Manufacturer',
        'scenario': 'Scenario',
        'ai_system': 'AI system',
        'target_sensitivity': 'Target sens (%)',
        'ai_threshold': 'AI threshold',
        'sens_pre_mean': 'Sens pre (calib) (%)',
        'sens_post_mean': 'Sens post mean (%)',
        'spec_pre_mean': 'Spec pre (calib) (%)',
        'spec_post_mean': 'Spec post mean (%)'
    })
    # Format numeric columns
    pct_cols = ['Target sens (%)', 'Sens pre (calib) (%)', 'Sens post mean (%)', 'Spec pre (calib) (%)', 'Spec post mean (%)']
    for col in pct_cols:
        if col in display_df.columns:
            display_df[col] = display_df[col].map(lambda x: '' if pd.isna(x) else f"{x:.2f}")
    if 'AI threshold' in display_df.columns:
        display_df['AI threshold'] = display_df['AI threshold'].map(lambda x: '' if pd.isna(x) else f"{x:.4f}")

    fig_height = 1.8 + 0.45 * len(display_df)
    fig, ax = plt.subplots(figsize=(20, fig_height))
    ax.axis('off')

    table = ax.table(
        cellText=display_df.values,
        colLabels=display_df.columns,
        cellLoc='center',
        loc='center',
        bbox=[0, 0, 1, 1]
    )

    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1, 1.6)

    header_color = '#4472C4'
    alt_row_color = '#E7E6E6'

    for i in range(len(display_df.columns)):
        cell = table[(0, i)]
        cell.set_facecolor(header_color)
        cell.set_text_props(weight='bold', color='white')

    for i in range(1, len(display_df) + 1):
        for j in range(len(display_df.columns)):
            cell = table[(i, j)]
            cell.set_facecolor(alt_row_color if i % 2 == 0 else 'white')

    for cell in table.get_celld().values():
        cell.set_edgecolor('black')
        cell.set_linewidth(1.2)

    plt.title('Part 3 Summary: All Hospitals (Sensitivity/Specificity, Scenario A/B)', fontsize=16, fontweight='bold', pad=20)
    plt.tight_layout()
    table_path = output_dir / 'part3_all_hospitals_summary_table.png'
    plt.savefig(table_path, dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()
    table_path


In [ ]:
'''
# Overlay Part 2 deviation signals on Part 3 monitoring plots
if '_confirmed_signal_centers' not in globals():
    def _confirmed_signal_centers(df, sig_mask, center_col, min_run=3):
        if df is None or len(df) == 0:
            return []
        if center_col not in df.columns:
            return []
        df_local = df.copy()
        df_local = df_local[[center_col]].reset_index(drop=True)
        centers = df_local[center_col].to_numpy()
        sig = sig_mask.reset_index(drop=True).to_numpy()

        confirmed = []
        run = 0
        for idx, is_sig in enumerate(sig):
            if is_sig:
                run += 1
                if run == min_run:
                    confirmed.append(float(centers[idx]))
            else:
                run = 0
        return confirmed


def collect_part2_signal_centers(df_h, alpha=0.05,
                                ref_size=REF_SIZE, gap_size=GAP_SIZE,
                                curr_size=CURR_SIZE, step_size=STEP_SIZE,
                                use_score_signals=True, score_n_permutations=None):
    signal_centers = {ai: {} for ai in AI_COLUMNS}
    min_run = globals().get('PART3_SIGNAL_MIN_RUN', 3)
    if 'rad_flagged' not in df_h.columns:
        df_h = df_h.copy()
        df_h['rad_flagged'] = compute_radiologist_flag(df_h, mode=PART3_RAD_MODE)
    # Ensure generic AI flag columns exist for Part 2-style signals
    for ai in AI_COLUMNS:
        base_flag = f'{ai}_flagged'
        part3_flag = f'{ai}_flagged_part3'
        if base_flag not in df_h.columns and part3_flag in df_h.columns:
            df_h[base_flag] = df_h[part3_flag]

    score_results = None
    if use_score_signals:
        if score_n_permutations is None:
            score_n_permutations = N_PERMUTATIONS
        score_results = sliding_window_monitoring(
            df_h, ai_columns=AI_COLUMNS,
            ref_size=ref_size, gap_size=gap_size, curr_size=curr_size, step_size=step_size,
            n_permutations=score_n_permutations
        )
    if score_results is not None and len(score_results) > 0:
        for ai in AI_COLUMNS:
            ai_data = score_results[score_results['ai_system'] == ai]
            for metric in ['median', 'p90', 'ks']:
                p_col = f'{metric}_p_fdr'
                if p_col in ai_data.columns:
                    sig = ai_data[p_col] < alpha
                    centers = _confirmed_signal_centers(ai_data, sig, 'window_center', min_run=min_run)
                    signal_centers[ai][f'score_{metric}'] = centers

    flagging = compute_flagging_rates(
        df_h, ref_size=ref_size, gap_size=gap_size, curr_size=curr_size, step_size=step_size
    )
    if flagging is not None:
        for ai in AI_COLUMNS:
            data = flagging.get(ai)
            if data is None or len(data) == 0:
                continue
            sig = data['p_value_corrected'] < alpha if 'p_value_corrected' in data.columns else pd.Series([False] * len(data))
            signal_centers[ai]['flagging_rate'] = _confirmed_signal_centers(data, sig, 'curr_center', min_run=min_run)

    overlap = compute_conditional_overlap(
        df_h, ref_size=ref_size, gap_size=gap_size, curr_size=curr_size, step_size=step_size
    )
    if overlap is not None:
        for ai in AI_COLUMNS:
            data = overlap.get(ai)
            if data is None or len(data) == 0:
                continue
            if 'p_val_ai_given_rad_corrected' in data.columns:
                sig = data['p_val_ai_given_rad_corrected'] < alpha
                signal_centers[ai]['p_ai_given_rad'] = _confirmed_signal_centers(data, sig, 'curr_center', min_run=min_run)
            if 'p_val_rad_given_ai_corrected' in data.columns:
                sig = data['p_val_rad_given_ai_corrected'] < alpha
                signal_centers[ai]['p_rad_given_ai'] = _confirmed_signal_centers(data, sig, 'curr_center', min_run=min_run)

    return signal_centers

def plot_part3_combined_sens_spec(df_h, sens_rates, spec_rates, transition_point, signal_centers,
                                 title, target_sensitivity=None, target_specificity=None,
                                 alpha=0.05, figsize=(14, 12)):
    fig, axes = plt.subplots(3, 1, figsize=figsize, sharex=True)
    colors = {'lunit': '#ff7f0e', 'therapixel': '#2ca02c', 'vara': '#d62728'}
    signal_styles = {
        'score_median': {'color': '#1f77b4', 'marker': '^', 'label': 'Median signal'},
        'score_p90': {'color': '#9467bd', 'marker': 's', 'label': 'P90 signal'},
        'score_ks': {'color': '#8c564b', 'marker': 'D', 'label': 'KS signal'},
        'flagging_rate': {'color': '#7f7f7f', 'marker': 'o', 'label': 'Flagging signal'},
        'p_ai_given_rad': {'color': '#2ca02c', 'marker': 'v', 'label': 'P(AI|Rad) signal'},
        'p_rad_given_ai': {'color': '#d62728', 'marker': 'x', 'label': 'P(Rad|AI) signal'}
    }

    for idx, ai in enumerate(AI_COLUMNS):
        ax = axes[idx]
        sens_ai = sens_rates[sens_rates['ai_system'] == ai] if sens_rates is not None else pd.DataFrame()
        spec_ai = spec_rates[spec_rates['ai_system'] == ai] if spec_rates is not None else pd.DataFrame()

        if len(sens_ai) == 0 and len(spec_ai) == 0:
            ax.text(0.5, 0.5, 'No data', ha='center', va='center')
            ax.set_title(ai.upper())
            ax.grid(True, alpha=0.3)
            continue

        x_min = min(sens_ai['curr_center'].min() if len(sens_ai) > 0 else transition_point,
                   spec_ai['curr_center'].min() if len(spec_ai) > 0 else transition_point)
        x_max = max(sens_ai['curr_center'].max() if len(sens_ai) > 0 else transition_point,
                   spec_ai['curr_center'].max() if len(spec_ai) > 0 else transition_point)
        ax.axvspan(x_min, transition_point, alpha=0.1, color='blue', label='Pre-start')
        ax.axvspan(transition_point, x_max, alpha=0.1, color='orange', label='Post-start')

        if len(sens_ai) > 0:
            ax.plot(sens_ai['curr_center'], sens_ai['curr_rate'], color=colors[ai], linewidth=2, label='Sensitivity')
        if len(spec_ai) > 0:
            ax.plot(spec_ai['curr_center'], spec_ai['curr_rate'], color=colors[ai], linewidth=2, linestyle='--', label='Specificity')

        if target_sensitivity is not None and not pd.isna(target_sensitivity):
            ax.axhline(target_sensitivity, color=colors[ai], linestyle=':', linewidth=1.5, label='Target sensitivity')
        if target_specificity is not None and not pd.isna(target_specificity):
            ax.axhline(target_specificity, color=colors[ai], linestyle='-.', linewidth=1.5, label='Target specificity')

        y_levels = [102, 100, 98, 96, 94, 92]
        method_keys = list(signal_styles.keys())
        for method_idx, method in enumerate(method_keys):
            centers = signal_centers.get(ai, {}).get(method, [])
            if not centers:
                continue
            style = signal_styles[method]
            y = y_levels[method_idx % len(y_levels)]
            ax.scatter(centers, [y] * len(centers),
                       color=style['color'], marker=style['marker'], s=35, label=style['label'])

        ax.axvline(transition_point, color='red', linestyle='--', linewidth=2, label='Start')
        ax.set_ylabel('Rate (%)')
        ax.set_ylim(0, 105)
        ax.set_title(ai.upper())
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=8, loc='upper left')

    axes[-1].set_xlabel('Current Window Center (Exam Index)')
    plt.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

# Example hospital plot with Part 2 signal overlays
PART3_EXAMPLE_HOSPITAL = HOSPITAL_PHILIPS
PART3_EXAMPLE_MANUFACTURER = None  # set to a manufacturer name to pin

df_example = df_part3[df_part3[HOSPITAL_COL] == PART3_EXAMPLE_HOSPITAL].copy()
if MANUFACTURER_COL in df_example.columns:
    if PART3_EXAMPLE_MANUFACTURER is None:
        manufacturers = sorted(df_example[MANUFACTURER_COL].dropna().unique())
        PART3_EXAMPLE_MANUFACTURER = manufacturers[0] if len(manufacturers) > 0 else None
    if PART3_EXAMPLE_MANUFACTURER is not None:
        df_example = df_example[df_example[MANUFACTURER_COL] == PART3_EXAMPLE_MANUFACTURER].copy()

part3_example_results = run_part3_hospital_pipeline(
    df_example, PART3_EXAMPLE_HOSPITAL, part3_start, calibration_strategy=PART3_CALIBRATION_STRATEGY
)
if part3_example_results is not None:
    df_h = part3_example_results['df_h']
    transition_idx = part3_example_results['transition_index']
    transition_center = _transition_center(transition_idx, PART3_CURR_SIZE)
    start_date_used = part3_example_results.get('start_date')
    start_method = part3_example_results.get('start_method', '')

    sens_rates = compute_rate_windows_by_flag(
        df_h, flag_suffix='part3', rate_kind='sensitivity',
        ref_size=PART3_REF_SIZE, gap_size=PART3_GAP_SIZE,
        curr_size=PART3_CURR_SIZE, step_size=PART3_STEP_SIZE
    )
    spec_rates = compute_rate_windows_by_flag(
        df_h, flag_suffix='part3', rate_kind='specificity',
        ref_size=PART3_REF_SIZE, gap_size=PART3_GAP_SIZE,
        curr_size=PART3_CURR_SIZE, step_size=PART3_STEP_SIZE
    )

    signal_centers = collect_part2_signal_centers(
        df_h, alpha=0.05,
        ref_size=PART3_REF_SIZE, gap_size=PART3_GAP_SIZE,
        curr_size=PART3_CURR_SIZE, step_size=PART3_STEP_SIZE
    )

    manufacturer_label = PART3_EXAMPLE_MANUFACTURER or 'All manufacturers'
    start_label = str(start_method).replace('_', ' ')
    title = (
        f"Part 3 Simulation Environment: Sensitivity + Specificity with Signal Overlays | "
        f"{PART3_EXAMPLE_HOSPITAL} — {manufacturer_label} | "
        f"Start: {start_date_used.date()} ({start_label})"
    )

    plot_part3_combined_sens_spec(
        df_h, sens_rates, spec_rates, transition_idx, signal_centers,
        title=title,
        target_sensitivity=sens_rates[sens_rates['curr_center'] < transition_center]['curr_rate'].mean() if len(sens_rates) > 0 else None,
        target_specificity=spec_rates[spec_rates['curr_center'] < transition_center]['curr_rate'].mean() if len(spec_rates) > 0 else None
    )
'''

In [ ]:

from IPython.display import display, Markdown
'''
# Run Part 3 plots for all hospitals; one scenario at a time
hospitals = sorted(df_part3[HOSPITAL_COL].dropna().unique())

for scenario in PART3_SCENARIOS:
    rad_mode, calib_target_mode, system_mode = resolve_part3_modes(
        scenario.get('rad_mode'), scenario.get('target_mode'), scenario.get('system_mode')
    )
    display(Markdown(f"### {scenario['label']}"))
    enough_data = []
    insufficient_data = []

    for hospital in hospitals:
        df_hospital = df_part3[df_part3[HOSPITAL_COL] == hospital].copy()
        if MANUFACTURER_COL in df_hospital.columns:
            manufacturers = sorted(df_hospital[MANUFACTURER_COL].dropna().unique())
            if len(manufacturers) == 0:
                manufacturers = ['Unknown']
        else:
            manufacturers = ['Unknown']

        for manufacturer in manufacturers:
            if manufacturer == 'Unknown':
                df_subset = df_hospital.copy()
            else:
                df_subset = df_hospital[df_hospital[MANUFACTURER_COL] == manufacturer].copy()
            if len(df_subset) == 0:
                insufficient_data.append(f"{hospital} — {manufacturer}")
                continue

            result = run_part3_hospital_pipeline(
                df_subset, hospital, part3_start,
                calibration_strategy=PART3_CALIBRATION_STRATEGY,
                rad_mode=rad_mode, target_mode=calib_target_mode
            )
            if result is None:
                insufficient_data.append(f"{hospital} — {manufacturer}")
                continue
            df_h = result['df_h']
            transition_idx = result['transition_index']
            transition_center = _transition_center(transition_idx, PART3_CURR_SIZE)

            base_suffix = 'part3'
            if system_mode == 'ai_plus_r1':
                r1_flag = compute_radiologist_flag(df_h, mode='r1').astype(bool)
                base_suffix = ensure_union_flags(df_h, 'part3', r1_flag)

            sens_rates = compute_rate_windows_by_flag(
                df_h, flag_suffix=base_suffix, rate_kind='sensitivity',
                ref_size=PART3_REF_SIZE, gap_size=PART3_GAP_SIZE,
                curr_size=PART3_CURR_SIZE, step_size=PART3_STEP_SIZE
            )
            spec_rates = compute_rate_windows_by_flag(
                df_h, flag_suffix=base_suffix, rate_kind='specificity',
                ref_size=PART3_REF_SIZE, gap_size=PART3_GAP_SIZE,
                curr_size=PART3_CURR_SIZE, step_size=PART3_STEP_SIZE
            )

            has_data = False
            if sens_rates is not None and 'ai_system' in sens_rates.columns:
                for ai in AI_COLUMNS:
                    if len(sens_rates[sens_rates['ai_system'] == ai]) > 0:
                        has_data = True
                        break
            if not has_data and spec_rates is not None and 'ai_system' in spec_rates.columns:
                for ai in AI_COLUMNS:
                    if len(spec_rates[spec_rates['ai_system'] == ai]) > 0:
                        has_data = True
                        break

            if not has_data:
                insufficient_data.append(f"{hospital} — {manufacturer}")
                continue

            signal_centers = collect_part2_signal_centers(
                df_h, alpha=0.05,
                ref_size=PART3_REF_SIZE, gap_size=PART3_GAP_SIZE,
                curr_size=PART3_CURR_SIZE, step_size=PART3_STEP_SIZE
            )

            plot_part3_combined_sens_spec(
                df_h, sens_rates, spec_rates, transition_idx, signal_centers,
                title=f"Part 3 Simulation ({scenario['label']}): {hospital} — {manufacturer}",
                target_sensitivity=sens_rates[sens_rates['curr_center'] < transition_center]['curr_rate'].mean() if len(sens_rates) > 0 else None,
                target_specificity=spec_rates[spec_rates['curr_center'] < transition_center]['curr_rate'].mean() if len(spec_rates) > 0 else None
            )
            enough_data.append(f"{hospital} — {manufacturer}")

    print("Enough data for plots:")
    print('- ' + '- '.join(enough_data) if enough_data else '- None')
    print("Insufficient data for plots:")
    print('- ' + '- '.join(insufficient_data) if insufficient_data else '- None')

'''

## Part 3D: Drift detection + recalibration experiment
Compare average post-start sensitivity/specificity with and without drift-triggered recalibration.


In [ ]:
# Drift detection + recalibration (what-if)
from IPython.display import display, Markdown
import inspect


def _legacy_ai_columns_default():
    if 'P3C_COLS' in globals() and hasattr(P3C_COLS, 'ai_columns'):
        return list(P3C_COLS.ai_columns)
    if 'AI_COLUMNS' in globals():
        return list(AI_COLUMNS)
    return []

def _legacy_settings_default():
    if 'P3C_SETTINGS' in globals():
        return P3C_SETTINGS
    from types import SimpleNamespace
    return SimpleNamespace(
        default_rad_mode=globals().get('PART3_RAD_MODE', 'all'),
        signal_min_run=int(globals().get('PART3_SIGNAL_MIN_RUN', 3)),
        recal_use_score_signals=bool(globals().get('PART3_RECAL_USE_SCORE_SIGNALS', False)),
    )



# Backward-compat shim: this cell expects legacy signatures.
# If embedded helper signatures are active, wrap them locally.
if 'collect_part2_signal_centers' in globals():
    try:
        _collect_needs_bridge = False
        _sig_collect = inspect.signature(collect_part2_signal_centers)
        if 'ai_columns' in _sig_collect.parameters and 'settings' in _sig_collect.parameters:
            _collect_needs_bridge = True

        # Also bridge if a prior legacy wrapper exists but its captured core needs settings.
        if (not _collect_needs_bridge) and ('_collect_part2_signal_centers_core' in globals()):
            try:
                _sig_core = inspect.signature(_collect_part2_signal_centers_core)
                if 'settings' in _sig_core.parameters:
                    _collect_needs_bridge = True
            except Exception:
                pass

        if _collect_needs_bridge:
            _collect_part2_signal_centers_core = globals().get('_collect_part2_signal_centers_core', collect_part2_signal_centers)

            def collect_part2_signal_centers(df_h, alpha=0.05,
                                            ref_size=REF_SIZE, gap_size=GAP_SIZE,
                                            curr_size=CURR_SIZE, step_size=STEP_SIZE,
                                            use_score_signals=True, score_n_permutations=None):
                kwargs = {
                    'alpha': alpha,
                    'ref_size': ref_size,
                    'gap_size': gap_size,
                    'curr_size': curr_size,
                    'step_size': step_size,
                    'use_score_signals': use_score_signals,
                    'ai_columns': _legacy_ai_columns_default(),
                    'settings': _legacy_settings_default(),
                }
                return _collect_part2_signal_centers_core(df_h, **kwargs)
    except Exception:
        pass

if 'compute_flagging_rates' in globals():
    try:
        _sig_flag = inspect.signature(compute_flagging_rates)
        if 'ai_columns' in _sig_flag.parameters:
            _compute_flagging_rates_core = compute_flagging_rates

            def compute_flagging_rates(df, *args, **kwargs):
                k = dict(kwargs)
                if 'ai_columns' not in k:
                    k['ai_columns'] = _legacy_ai_columns_default()
                return _compute_flagging_rates_core(df, *args, **k)
    except Exception:
        pass

if 'compute_conditional_overlap' in globals():
    try:
        _sig_ov = inspect.signature(compute_conditional_overlap)
        if 'ai_columns' in _sig_ov.parameters:
            _compute_conditional_overlap_core = compute_conditional_overlap

            def compute_conditional_overlap(df, *args, **kwargs):
                k = dict(kwargs)
                if 'ai_columns' not in k:
                    k['ai_columns'] = _legacy_ai_columns_default()
                return _compute_conditional_overlap_core(df, *args, **k)
    except Exception:
        pass

if '_get_target_metrics' in globals():
    try:
        _sig_targets = inspect.signature(_get_target_metrics)
        if 'ai_columns' in _sig_targets.parameters:
            _get_target_metrics_core = _get_target_metrics

            def _get_target_metrics(df_h, transition_idx, ref_size=REF_SIZE, gap_size=GAP_SIZE,
                                    curr_size=CURR_SIZE, step_size=STEP_SIZE):
                kwargs = {
                    'transition_idx': transition_idx,
                    'ref_size': ref_size,
                    'gap_size': gap_size,
                    'curr_size': curr_size,
                    'step_size': step_size,
                    'ai_columns': _legacy_ai_columns_default(),
                }
                return _get_target_metrics_core(df_h, **kwargs)
    except Exception:
        pass

if 'compute_rate_windows_by_flag' in globals():
    try:
        _sig_rates = inspect.signature(compute_rate_windows_by_flag)
        if 'ai_columns' in _sig_rates.parameters:
            _compute_rate_windows_by_flag_core = compute_rate_windows_by_flag

            def compute_rate_windows_by_flag(df, flag_suffix, rate_kind,
                                            ref_size=REF_SIZE, gap_size=GAP_SIZE,
                                            curr_size=CURR_SIZE, step_size=STEP_SIZE):
                kwargs = {
                    'ai_columns': _legacy_ai_columns_default(),
                    'flag_suffix': flag_suffix,
                    'rate_kind': rate_kind,
                    'ref_size': ref_size,
                    'gap_size': gap_size,
                    'curr_size': curr_size,
                    'step_size': step_size,
                }
                return _compute_rate_windows_by_flag_core(df, **kwargs)
    except Exception:
        pass

if 'ensure_union_flags' in globals():
    try:
        _sig_union = inspect.signature(ensure_union_flags)
        if 'ai_columns' in _sig_union.parameters:
            _ensure_union_flags_core = ensure_union_flags

            def ensure_union_flags(df, base_suffix, r1_flag, out_suffix=None):
                kwargs = {
                    'ai_columns': _legacy_ai_columns_default(),
                    'base_suffix': base_suffix,
                    'r1_flag': r1_flag,
                    'out_suffix': out_suffix,
                }
                return _ensure_union_flags_core(df, **kwargs)
    except Exception:
        pass
if 'collect_part2_signal_centers' not in globals():
    def collect_part2_signal_centers(df_h, alpha=0.05,
                                    ref_size=REF_SIZE, gap_size=GAP_SIZE,
                                    curr_size=CURR_SIZE, step_size=STEP_SIZE,
                                    use_score_signals=True, score_n_permutations=None):
        signal_centers = {ai: {} for ai in AI_COLUMNS}
        min_run = globals().get('PART3_SIGNAL_MIN_RUN', 3)
        if 'rad_flagged' not in df_h.columns:
            df_h = df_h.copy()
            df_h['rad_flagged'] = compute_radiologist_flag(df_h, mode=PART3_RAD_MODE)
        for ai in AI_COLUMNS:
            base_flag = f'{ai}_flagged'
            part3_flag = f'{ai}_flagged_part3'
            if base_flag not in df_h.columns and part3_flag in df_h.columns:
                df_h[base_flag] = df_h[part3_flag]

        score_results = None
        if use_score_signals:
            if score_n_permutations is None:
                score_n_permutations = N_PERMUTATIONS
            score_results = sliding_window_monitoring(
                df_h, ai_columns=AI_COLUMNS,
                ref_size=ref_size, gap_size=gap_size, curr_size=curr_size, step_size=step_size,
                n_permutations=score_n_permutations
            )
        if score_results is not None and len(score_results) > 0:
            for ai in AI_COLUMNS:
                ai_data = score_results[score_results['ai_system'] == ai]
                for metric in ['median', 'p90', 'ks']:
                    p_col = f'{metric}_p_fdr'
                    if p_col in ai_data.columns:
                        sig = ai_data[p_col] < alpha
                        centers = _confirmed_signal_centers(ai_data, sig, 'window_center', min_run=min_run)
                        signal_centers[ai][f'score_{metric}'] = centers

        flagging = compute_flagging_rates(
            df_h, ref_size=ref_size, gap_size=gap_size, curr_size=curr_size, step_size=step_size
        )
        if flagging is not None:
            for ai in AI_COLUMNS:
                data = flagging.get(ai)
                if data is None or len(data) == 0:
                    continue
                sig = data['p_value_corrected'] < alpha if 'p_value_corrected' in data.columns else pd.Series([False] * len(data))
                signal_centers[ai]['flagging_rate'] = _confirmed_signal_centers(data, sig, 'curr_center', min_run=min_run)

        overlap = compute_conditional_overlap(
            df_h, ref_size=ref_size, gap_size=gap_size, curr_size=curr_size, step_size=step_size
        )
        if overlap is not None:
            for ai in AI_COLUMNS:
                data = overlap.get(ai)
                if data is None or len(data) == 0:
                    continue
                if 'p_val_ai_given_rad_corrected' in data.columns:
                    sig = data['p_val_ai_given_rad_corrected'] < alpha
                    signal_centers[ai]['p_ai_given_rad'] = _confirmed_signal_centers(data, sig, 'curr_center', min_run=min_run)
                if 'p_val_rad_given_ai_corrected' in data.columns:
                    sig = data['p_val_rad_given_ai_corrected'] < alpha
                    signal_centers[ai]['p_rad_given_ai'] = _confirmed_signal_centers(data, sig, 'curr_center', min_run=min_run)

        return signal_centers

if '_signal_indices_after' not in globals():
    def _signal_indices_after(signal_centers, min_center):
        if not signal_centers:
            return []
        centers = [c for c in signal_centers if c is not None and c >= min_center]
        centers = sorted(set(centers))
        return [int(round(c)) for c in centers]

if '_transition_center' not in globals():
    def _transition_center(transition_idx, curr_size):
        if transition_idx is None or curr_size is None:
            return None
        return transition_idx + (curr_size / 2)


PART3_RECAL_WINDOW_SIZE = PART3_REF_SIZE * 2  # larger past-only window to reduce NaN thresholds
PART3_RECAL_ALPHA = 0.05
PART3_RECAL_METHODS = ['A']  # A: use current window
PART3_RECAL_METRICS = [
    'flagging_rate',
    'relative_flagging_rate',
    'p_ai_given_rad',
    'p_rad_given_ai'
]
PART3_RECAL_PLOT_METRIC = 'flagging_rate'
PART3_RECAL_PLOT_METHOD = 'A'
RUN_PART3_RECAL_PLOTS = False  # set True to generate recalibration plots


def _mean_metric(df, ai, col, center_min=None, center_max=None):
    if df is None or len(df) == 0:
        return np.nan
    df_ai = df if ai is None else df[df['ai_system'] == ai]
    if len(df_ai) == 0:
        return np.nan
    if center_min is not None:
        df_ai = df_ai[df_ai['curr_center'] >= center_min]
    if center_max is not None:
        df_ai = df_ai[df_ai['curr_center'] < center_max]
    return df_ai[col].mean() if len(df_ai) > 0 else np.nan


def _signal_indices_after(signal_centers, min_center):
    if not signal_centers:
        return []
    centers = [c for c in signal_centers if c is not None and c >= min_center]
    centers = sorted(set(centers))
    return [int(round(c)) for c in centers]


def _first_signal_after(signal_centers, min_center):
    if not signal_centers:
        return None
    centers = [c for c in signal_centers if c is not None and c >= min_center]
    return int(round(min(centers))) if centers else None


def _weighted_quantile(values, weights, quantile):
    if len(values) == 0:
        return np.nan
    sorter = np.argsort(values)
    values = values[sorter]
    weights = weights[sorter]
    cum_weights = np.cumsum(weights)
    if cum_weights[-1] == 0:
        return np.nan
    cum = cum_weights / cum_weights[-1]
    return np.interp(quantile, cum, values)


def _histogram_weights(ref_scores, curr_scores, bins=20):
    if len(ref_scores) == 0 or len(curr_scores) == 0:
        return None
    hist_ref, edges = np.histogram(ref_scores, bins=bins)
    hist_curr, _ = np.histogram(curr_scores, bins=edges)
    ratio = np.zeros_like(hist_ref, dtype=float)
    np.divide(hist_curr, hist_ref, out=ratio, where=hist_ref > 0)
    bin_idx = np.searchsorted(edges, ref_scores, side='right') - 1
    bin_idx = np.clip(bin_idx, 0, len(ratio) - 1)
    return ratio[bin_idx]


def _histogram_ratio(ref_scores, curr_scores, bins=20):
    if len(ref_scores) == 0 or len(curr_scores) == 0:
        return None, None
    hist_ref, edges = np.histogram(ref_scores, bins=bins)
    hist_curr, _ = np.histogram(curr_scores, bins=edges)
    ratio = np.zeros_like(hist_ref, dtype=float)
    np.divide(hist_curr, hist_ref, out=ratio, where=hist_ref > 0)
    return edges, ratio


def _weights_for_scores(scores, edges, ratio):
    if edges is None or ratio is None or len(scores) == 0:
        return None
    bin_idx = np.searchsorted(edges, scores, side='right') - 1
    bin_idx = np.clip(bin_idx, 0, len(ratio) - 1)
    return ratio[bin_idx]


    if len(ref_scores) == 0 or len(curr_scores) == 0:
        return None
    hist_ref, edges = np.histogram(ref_scores, bins=bins)
    hist_curr, _ = np.histogram(curr_scores, bins=edges)
    ratio = np.zeros_like(hist_ref, dtype=float)
    np.divide(hist_curr, hist_ref, out=ratio, where=hist_ref > 0)
    bin_idx = np.searchsorted(edges, ref_scores, side='right') - 1
    bin_idx = np.clip(bin_idx, 0, len(ratio) - 1)
    return ratio[bin_idx]


def _threshold_for_flagging_rate(scores, target_rate, weights=None):
    if len(scores) == 0 or pd.isna(target_rate):
        return np.nan
    q = 1 - (target_rate / 100)
    q = min(max(q, 0), 1)
    if weights is None:
        return np.quantile(scores, q)
    return _weighted_quantile(scores, weights, q)


def _p_rad_given_ai(scores, rad_flagged, threshold, weights=None):
    ai_flagged = scores >= threshold
    if weights is None:
        denom = ai_flagged.sum()
        numer = (ai_flagged & rad_flagged).sum()
    else:
        denom = weights[ai_flagged].sum()
        numer = weights[ai_flagged & rad_flagged].sum()
    return (numer / denom * 100) if denom > 0 else np.nan


def _threshold_for_p_rad_given_ai(scores, rad_flagged, target_rate, weights=None):
    if len(scores) == 0 or pd.isna(target_rate):
        return np.nan
    qs = np.linspace(0.05, 0.95, 19)
    if weights is None:
        thresholds = np.quantile(scores, qs)
    else:
        thresholds = np.array([_weighted_quantile(scores, weights, q) for q in qs])
    thresholds = np.unique(thresholds[~np.isnan(thresholds)])
    if len(thresholds) == 0:
        return np.nan
    best_thr = thresholds[0]
    best_diff = np.inf
    for thr in thresholds:
        val = _p_rad_given_ai(scores, rad_flagged, thr, weights=weights)
        diff = abs(val - target_rate) if not pd.isna(val) else np.inf
        if diff < best_diff:
            best_diff = diff
            best_thr = thr
    return best_thr


def _get_scores_and_rad(df, ai):
    if df is None or len(df) == 0:
        return np.array([]), np.array([], dtype=bool)
    df_ai = df[[ai, 'rad_flagged']].dropna(subset=[ai]).copy()
    return df_ai[ai].to_numpy(), df_ai['rad_flagged'].to_numpy().astype(bool)


def _get_target_metrics(df_h, transition_idx, ref_size, gap_size, curr_size, step_size):
    targets = {ai: {} for ai in AI_COLUMNS}
    transition_center = _transition_center(transition_idx, curr_size)

    flagging = compute_flagging_rates(
        df_h, ref_size=ref_size, gap_size=gap_size, curr_size=curr_size, step_size=step_size
    )
    overlap = compute_conditional_overlap(
        df_h, ref_size=ref_size, gap_size=gap_size, curr_size=curr_size, step_size=step_size
    )
    rel_flagging = compute_relative_flagging_rates(flagging)

    for ai in AI_COLUMNS:
        flag_df = flagging.get(ai)
        if flag_df is not None and len(flag_df) > 0:
            targets[ai]['flagging_rate'] = flag_df[flag_df['curr_center'] < transition_center]['curr_pct'].mean()
        rel_df = rel_flagging.get(ai)
        if rel_df is not None and len(rel_df) > 0:
            targets[ai]['relative_flagging_rate'] = rel_df[rel_df['curr_center'] < transition_center]['curr_rel_rate'].mean()
        ov_df = overlap.get(ai)
        if ov_df is not None and len(ov_df) > 0:
            targets[ai]['p_ai_given_rad'] = ov_df[ov_df['curr_center'] < transition_center]['curr_p_ai_given_rad'].mean()
            targets[ai]['p_rad_given_ai'] = ov_df[ov_df['curr_center'] < transition_center]['curr_p_rad_given_ai'].mean()

    return targets


def _compute_recal_threshold(ai, metric_key, method, target_value,
                             recal_window, ref_window, threshold_metric=None):
    if threshold_metric is None:
        threshold_metric = metric_key

    if threshold_metric in ['score_median', 'score_p90', 'score_ks']:
        return np.nan

    if threshold_metric == 'sensitivity':
        if method == 'A':
            cancer_scores = recal_window[recal_window['has_cancer'] == 1][ai].dropna().values
            return _threshold_for_flagging_rate(cancer_scores, target_value, weights=None)
        ref_scores_all = ref_window[ai].dropna().values if ref_window is not None else np.array([])
        curr_scores_all = recal_window[ai].dropna().values
        if len(ref_scores_all) == 0 or len(curr_scores_all) == 0:
            return np.nan
        edges, ratio = _histogram_ratio(ref_scores_all, curr_scores_all)
        ref_cancer_scores = (
            ref_window[ref_window['has_cancer'] == 1][ai].dropna().values
            if ref_window is not None else np.array([])
        )
        if len(ref_cancer_scores) == 0:
            return np.nan
        weights = _weights_for_scores(ref_cancer_scores, edges, ratio)
        return _threshold_for_flagging_rate(ref_cancer_scores, target_value, weights=weights)

    if threshold_metric == 'relative_flagging_rate':
        rad_rate = recal_window['rad_flagged'].mean() * 100 if len(recal_window) > 0 else np.nan
        if pd.isna(rad_rate) or pd.isna(target_value):
            return np.nan
        target_value = target_value * rad_rate

    if method == 'A':
        base_scores, base_rad = _get_scores_and_rad(recal_window, ai)
        weights = None
    else:
        ref_scores, ref_rad = _get_scores_and_rad(ref_window, ai)
        curr_scores, _ = _get_scores_and_rad(recal_window, ai)
        if len(ref_scores) == 0 or len(curr_scores) == 0:
            return np.nan
        weights = _histogram_weights(ref_scores, curr_scores)
        base_scores, base_rad = ref_scores, ref_rad

    if threshold_metric in ['flagging_rate', 'relative_flagging_rate']:
        return _threshold_for_flagging_rate(base_scores, target_value, weights=weights)

    if threshold_metric == 'p_ai_given_rad':
        scores_rad = base_scores[base_rad]
        if len(scores_rad) == 0:
            return np.nan
        if weights is None:
            return _threshold_for_flagging_rate(scores_rad, target_value, weights=None)
        weights_rad = weights[base_rad]
        return _threshold_for_flagging_rate(scores_rad, target_value, weights=weights_rad)

    if threshold_metric == 'p_rad_given_ai':
        return _threshold_for_p_rad_given_ai(base_scores, base_rad, target_value, weights=weights)

    return np.nan

def build_part3_recalibration_summary(df_part3, start_date, calibration_strategy=None, metrics=None, target_mode='metric',
                                    rad_mode=None, calib_target_mode=None, system_mode=None, scenario_label=None, curr_size_by_hospital=None, eval_curr_size=None):
    if calibration_strategy is None:
        calibration_strategy = globals().get('PART3_CALIBRATION_STRATEGY', 'rad_flagging_rate')

    if metrics is None:
        metrics = PART3_RECAL_METRICS

    rad_mode, calib_target_mode, system_mode = resolve_part3_modes(rad_mode, calib_target_mode, system_mode)

    rows = []
    hospitals = sorted(df_part3[HOSPITAL_COL].dropna().unique())
    for hospital in hospitals:
        curr_size = PART3_CURR_SIZE
        if isinstance(curr_size_by_hospital, dict) and hospital in curr_size_by_hospital:
            curr_size = curr_size_by_hospital[hospital]
        eval_size = eval_curr_size
        if eval_size is None:
            eval_size = globals().get('PART3_EVAL_CURR_SIZE', None)
        if eval_size is None:
            eval_size = curr_size
        curr_size = PART3_CURR_SIZE
        if isinstance(curr_size_by_hospital, dict) and hospital in curr_size_by_hospital:
            curr_size = curr_size_by_hospital[hospital]
        df_hospital = df_part3[df_part3[HOSPITAL_COL] == hospital].copy()
        if MANUFACTURER_COL in df_hospital.columns:
            manufacturers = sorted(df_hospital[MANUFACTURER_COL].dropna().unique())
            if len(manufacturers) == 0:
                manufacturers = ['Unknown']
        else:
            manufacturers = ['Unknown']

        for manufacturer in manufacturers:
            if manufacturer == 'Unknown':
                df_subset = df_hospital.copy()
            else:
                df_subset = df_hospital[df_hospital[MANUFACTURER_COL] == manufacturer].copy()
            if len(df_subset) == 0:
                continue

            result = run_part3_hospital_pipeline(
                df_subset, hospital, start_date, calibration_strategy=calibration_strategy,
                rad_mode=rad_mode, target_mode=calib_target_mode
            )
            if result is None:
                continue

            df_h = result['df_h']
            transition_idx = result['transition_index']
            transition_center = _transition_center(transition_idx, curr_size)
            eval_transition_center = _transition_center(transition_idx, eval_size)

            cancers_h = df_h['has_cancer'].fillna(0).astype(int)
            if cancers_h.sum() >= PART3_PRESTART_MIN_CANCERS:
                cum_cancers_h = cancers_h.cumsum()
                idx_target_h = int(cum_cancers_h[cum_cancers_h >= PART3_PRESTART_MIN_CANCERS].index[0])
                prestart_exam_count = int(idx_target_h + 1)
                prestart_cancer_count = int(cancers_h.iloc[:idx_target_h + 1].sum())
                prestart_control_count = int((cancers_h.iloc[:idx_target_h + 1] == 0).sum())
            else:
                prestart_exam_count = np.nan
                prestart_cancer_count = np.nan
                prestart_control_count = np.nan

            if 'rad_flagged' not in df_h.columns:
                df_h = df_h.copy()
                df_h['rad_flagged'] = compute_radiologist_flag(df_h, mode=rad_mode)

            signal_centers = collect_part2_signal_centers(
                df_h, alpha=PART3_RECAL_ALPHA,
                ref_size=PART3_REF_SIZE, gap_size=PART3_GAP_SIZE,
                curr_size=eval_size, step_size=PART3_STEP_SIZE,
                use_score_signals=PART3_RECAL_USE_SCORE_SIGNALS
            )

            targets = _get_target_metrics(
                df_h, transition_idx,
                ref_size=PART3_REF_SIZE, gap_size=PART3_GAP_SIZE,
                curr_size=eval_size, step_size=PART3_STEP_SIZE
            )

            base_flag_suffix = 'part3'
            r1_flag = None
            if system_mode == 'ai_plus_r1':
                r1_flag = compute_radiologist_flag(df_h, mode='r1').astype(bool)
                base_flag_suffix = ensure_union_flags(df_h, 'part3', r1_flag)

            sens_base = compute_rate_windows_by_flag(
                df_h, flag_suffix=base_flag_suffix, rate_kind='sensitivity',
                ref_size=PART3_REF_SIZE, gap_size=PART3_GAP_SIZE,
                curr_size=eval_size, step_size=PART3_STEP_SIZE
            )
            spec_base = compute_rate_windows_by_flag(
                df_h, flag_suffix=base_flag_suffix, rate_kind='specificity',
                ref_size=PART3_REF_SIZE, gap_size=PART3_GAP_SIZE,
                curr_size=eval_size, step_size=PART3_STEP_SIZE
            )

            target_sens_pre = result.get('radiologist_sensitivity_pre', {}).get(calib_target_mode, np.nan)
            target_spec_pre = result.get('radiologist_specificity_pre', {}).get(calib_target_mode, np.nan)

            for ai in AI_COLUMNS:
                target_sens = target_sens_pre
                target_spec = target_spec_pre

                post_sens_base = _mean_metric(sens_base, ai, 'curr_rate', center_min=eval_transition_center)
                post_spec_base = _mean_metric(spec_base, ai, 'curr_rate', center_min=eval_transition_center)

                for metric_key in metrics:
                    signal_key = 'flagging_rate' if metric_key == 'relative_flagging_rate' else metric_key
                    signal_idxs = _signal_indices_after(signal_centers.get(ai, {}).get(signal_key, []), transition_center)
                    first_signal_idx = signal_idxs[0] if signal_idxs else None
                    signal_count = len(signal_idxs)

                    if first_signal_idx is not None:
                        first_start_idx = max(0, first_signal_idx - PART3_RECAL_WINDOW_SIZE)
                        recal_window_first = df_h.iloc[first_start_idx:first_signal_idx]
                        recal_window_n = len(recal_window_first)
                        recal_window_cancers = recal_window_first['has_cancer'].fillna(0).astype(int).sum()
                        recal_window_radpos = (recal_window_first['rad_flagged'] == 1).sum() if 'rad_flagged' in recal_window_first.columns else np.nan
                        recal_window_scores = recal_window_first[ai].notna().sum()
                    else:
                        recal_window_n = np.nan
                        recal_window_cancers = np.nan
                        recal_window_radpos = np.nan
                        recal_window_scores = np.nan

                    for method in PART3_RECAL_METHODS:
                        recal_applied = False
                        recal_flag_suffix = f'recal_{metric_key}_{method}'
                        base_flag_col = f'{ai}_flagged_part3'
                        recal_flag_col = f'{ai}_flagged_{recal_flag_suffix}'

                        df_h[recal_flag_col] = df_h[base_flag_col]
                        recal_flag_suffix_use = recal_flag_suffix

                        recal_points = []
                        for signal_idx in signal_idxs:
                            start_idx = max(0, signal_idx - PART3_RECAL_WINDOW_SIZE)
                            recal_window = df_h.iloc[start_idx:signal_idx]
                            ref_start_idx = max(0, transition_idx - PART3_RECAL_WINDOW_SIZE)
                            ref_window = df_h.iloc[ref_start_idx:transition_idx]
                            target_value = targets.get(ai, {}).get(metric_key)
                            threshold_metric = metric_key
                            if target_mode == 'sensitivity':
                                target_value = result['radiologist_sensitivity_pre'][calib_target_mode]
                                threshold_metric = 'sensitivity'

                            recal_thr = _compute_recal_threshold(
                                ai, metric_key, method, target_value,
                                recal_window=recal_window,
                                ref_window=ref_window,
                                threshold_metric=threshold_metric
                            )

                            if pd.notna(recal_thr):
                                df_h.loc[signal_idx:, recal_flag_col] = (df_h.loc[signal_idx:, ai] >= recal_thr).astype(int)
                                recal_applied = True
                                recal_points.append(signal_idx)

                        if system_mode == 'ai_plus_r1':
                            if r1_flag is None:
                                r1_flag = compute_radiologist_flag(df_h, mode='r1').astype(bool)
                            union_suffix = f"{recal_flag_suffix}_r1_union"
                            df_h[f'{ai}_flagged_{union_suffix}'] = (
                                (df_h[recal_flag_col].fillna(0).astype(int) == 1) | r1_flag
                            ).astype(int)
                            recal_flag_suffix_use = union_suffix

                        sens_recal = compute_rate_windows_by_flag(
                            df_h, flag_suffix=recal_flag_suffix_use, rate_kind='sensitivity',
                            ref_size=PART3_REF_SIZE, gap_size=PART3_GAP_SIZE,
                            curr_size=eval_size, step_size=PART3_STEP_SIZE
                        )
                        spec_recal = compute_rate_windows_by_flag(
                            df_h, flag_suffix=recal_flag_suffix_use, rate_kind='specificity',
                            ref_size=PART3_REF_SIZE, gap_size=PART3_GAP_SIZE,
                            curr_size=eval_size, step_size=PART3_STEP_SIZE
                        )

                        post_sens_recal = _mean_metric(sens_recal, ai, 'curr_rate', center_min=eval_transition_center)
                        post_spec_recal = _mean_metric(spec_recal, ai, 'curr_rate', center_min=eval_transition_center)

                        def _abs_delta(value, target):
                            if pd.isna(value) or pd.isna(target):
                                return np.nan
                            return abs(value - target)

                        if signal_count == 0:
                            recal_status = 'no_signal'
                        elif len(recal_points) == 0:
                            recal_status = 'no_threshold'
                        else:
                            recal_status = 'recal_applied'

                        rows.append({
                            'scenario': scenario_label,
                            'hospital': hospital,
                            'manufacturer': manufacturer,
                            'ai_system': ai,
                            'signal_metric': metric_key,
                            'recal_method': method,
                            'calibration_strategy': calibration_strategy,
                            'target_mode': target_mode,
                            'first_signal_center': first_signal_idx,
                            'prestart_exam_count': prestart_exam_count,
                            'prestart_cancer_count': prestart_cancer_count,
                            'prestart_control_count': prestart_control_count,
                            'signal_count': signal_count,
                            'recal_point_count': len(recal_points),
                            'recal_status': recal_status,
                            'recal_window_n': recal_window_n,
                            'recal_window_cancers': recal_window_cancers,
                            'recal_window_radpos': recal_window_radpos,
                            'recal_window_scores': recal_window_scores,
                            'target_sensitivity': target_sens,
                            'target_specificity': target_spec,
                            'post_sens_base': post_sens_base,
                            'post_spec_base': post_spec_base,
                            'post_sens_recal': post_sens_recal,
                            'post_spec_recal': post_spec_recal,
                            'abs_delta_sens_base': _abs_delta(post_sens_base, target_sens),
                            'abs_delta_sens_recal': _abs_delta(post_sens_recal, target_sens),
                            'abs_delta_spec_base': _abs_delta(post_spec_base, target_spec),
                            'abs_delta_spec_recal': _abs_delta(post_spec_recal, target_spec),
                            'sens_improvement': _abs_delta(post_sens_base, target_sens) - _abs_delta(post_sens_recal, target_sens),
                            'spec_improvement': _abs_delta(post_spec_base, target_spec) - _abs_delta(post_spec_recal, target_spec),
                            'overall_improvement': (
                                _abs_delta(post_sens_base, target_sens) - _abs_delta(post_sens_recal, target_sens)
                            ) + (
                                _abs_delta(post_spec_base, target_spec) - _abs_delta(post_spec_recal, target_spec)
                            ),
                            'recal_applied': recal_applied
                        })

    return pd.DataFrame(rows)


def _interesting_rows(df):
    if df is None or len(df) == 0:
        return df
    # No filtering: return all rows to inspect recalibration behavior across metrics.
    return df.copy()


def plot_part3_recalibration_effects(df_part3, metric_key=PART3_RECAL_PLOT_METRIC,
                                     method=PART3_RECAL_PLOT_METHOD, alpha=0.05,
                                     rad_mode=None, calib_target_mode=None, system_mode=None, scenario_label=None):
    plotted = []
    skipped = []

    rad_mode, calib_target_mode, system_mode = resolve_part3_modes(rad_mode, calib_target_mode, system_mode)

    hospitals = sorted(df_part3[HOSPITAL_COL].dropna().unique())
    for hospital in hospitals:
        df_hospital = df_part3[df_part3[HOSPITAL_COL] == hospital].copy()
        if MANUFACTURER_COL in df_hospital.columns:
            manufacturers = sorted(df_hospital[MANUFACTURER_COL].dropna().unique())
            if len(manufacturers) == 0:
                manufacturers = ['Unknown']
        else:
            manufacturers = ['Unknown']

        for manufacturer in manufacturers:
            if manufacturer == 'Unknown':
                df_subset = df_hospital.copy()
            else:
                df_subset = df_hospital[df_hospital[MANUFACTURER_COL] == manufacturer].copy()
            if len(df_subset) == 0:
                continue

            result = run_part3_hospital_pipeline(
                df_subset, hospital, part3_start, calibration_strategy=PART3_CALIBRATION_STRATEGY,
                rad_mode=rad_mode, target_mode=calib_target_mode
            )
            if result is None:
                continue

            df_h = result['df_h']
            transition_idx = result['transition_index']
            transition_center = _transition_center(transition_idx, PART3_CURR_SIZE)
            start_method = result.get('start_method', '')

            if 'rad_flagged' not in df_h.columns:
                df_h = df_h.copy()
                df_h['rad_flagged'] = compute_radiologist_flag(df_h, mode=rad_mode)

            signal_centers = collect_part2_signal_centers(
                df_h, alpha=alpha,
                ref_size=PART3_REF_SIZE, gap_size=PART3_GAP_SIZE,
                curr_size=PART3_CURR_SIZE, step_size=PART3_STEP_SIZE,
                use_score_signals=PART3_RECAL_USE_SCORE_SIGNALS
            )

            targets = _get_target_metrics(
                df_h, transition_idx,
                ref_size=PART3_REF_SIZE, gap_size=PART3_GAP_SIZE,
                curr_size=PART3_CURR_SIZE, step_size=PART3_STEP_SIZE
            )

            base_flag_suffix = 'part3'
            r1_flag = None
            if system_mode == 'ai_plus_r1':
                r1_flag = compute_radiologist_flag(df_h, mode='r1').astype(bool)
                base_flag_suffix = ensure_union_flags(df_h, 'part3', r1_flag)

            sens_base = compute_rate_windows_by_flag(
                df_h, flag_suffix=base_flag_suffix, rate_kind='sensitivity',
                ref_size=PART3_REF_SIZE, gap_size=PART3_GAP_SIZE,
                curr_size=PART3_CURR_SIZE, step_size=PART3_STEP_SIZE
            )
            spec_base = compute_rate_windows_by_flag(
                df_h, flag_suffix=base_flag_suffix, rate_kind='specificity',
                ref_size=PART3_REF_SIZE, gap_size=PART3_GAP_SIZE,
                curr_size=PART3_CURR_SIZE, step_size=PART3_STEP_SIZE
            )

            recal_flags_ready = {}
            recal_points_map = {}
            for ai in AI_COLUMNS:
                signal_key = metric_key if metric_key != 'relative_flagging_rate' else 'flagging_rate'
                signal_idxs = _signal_indices_after(signal_centers.get(ai, {}).get(signal_key, []), transition_center)
                if not signal_idxs:
                    skipped.append((hospital, manufacturer, ai, 'no_signal'))
                    continue

                recal_flag_col = f'{ai}_flagged_recal_{metric_key}_{method}'
                df_h[recal_flag_col] = df_h[f'{ai}_flagged_part3']
                recal_points = []
                recal_flag_suffix_use = f'recal_{metric_key}_{method}'

                for signal_idx in signal_idxs:
                    start_idx = max(0, signal_idx - PART3_RECAL_WINDOW_SIZE)
                    recal_window = df_h.iloc[start_idx:signal_idx]
                    ref_start_idx = max(0, transition_idx - PART3_RECAL_WINDOW_SIZE)
                    ref_window = df_h.iloc[ref_start_idx:transition_idx]
                    target_value = targets.get(ai, {}).get(metric_key)

                    recal_thr = _compute_recal_threshold(
                        ai, metric_key, method, target_value,
                        recal_window=recal_window,
                        ref_window=ref_window
                    )
                    if pd.isna(recal_thr):
                        continue

                    df_h.loc[signal_idx:, recal_flag_col] = (df_h.loc[signal_idx:, ai] >= recal_thr).astype(int)
                    recal_points.append(signal_idx)

                if system_mode == 'ai_plus_r1':
                    if r1_flag is None:
                        r1_flag = compute_radiologist_flag(df_h, mode='r1').astype(bool)
                    union_suffix = f"recal_{metric_key}_{method}_r1_union"
                    df_h[f'{ai}_flagged_{union_suffix}'] = (
                        (df_h[recal_flag_col].fillna(0).astype(int) == 1) | r1_flag
                    ).astype(int)
                    recal_flag_suffix_use = union_suffix

                if not recal_points:
                    skipped.append((hospital, manufacturer, ai, 'no_threshold'))
                    continue

                recal_flags_ready[ai] = f"{ai}_flagged_{recal_flag_suffix_use}"
                recal_points_map[ai] = recal_points

            if len(recal_flags_ready) == 0:
                continue

            fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)
            colors = {'lunit': '#ff7f0e', 'therapixel': '#2ca02c', 'vara': '#d62728'}

            for idx, ai in enumerate(AI_COLUMNS):
                ax = axes[idx]
                if ai not in recal_flags_ready:
                    ax.text(0.5, 0.5, 'No recalibration', ha='center', va='center')
                    ax.set_title(ai.upper())
                    ax.grid(True, alpha=0.3)
                    continue

                recal_suffix = recal_flags_ready[ai].split(f'{ai}_flagged_')[1]
                sens_recal = compute_rate_windows_by_flag(
                    df_h, flag_suffix=recal_suffix, rate_kind='sensitivity',
                    ref_size=PART3_REF_SIZE, gap_size=PART3_GAP_SIZE,
                    curr_size=PART3_CURR_SIZE, step_size=PART3_STEP_SIZE
                )
                spec_recal = compute_rate_windows_by_flag(
                    df_h, flag_suffix=recal_suffix, rate_kind='specificity',
                    ref_size=PART3_REF_SIZE, gap_size=PART3_GAP_SIZE,
                    curr_size=PART3_CURR_SIZE, step_size=PART3_STEP_SIZE
                )

                sens_ai = sens_base[sens_base['ai_system'] == ai]
                spec_ai = spec_base[spec_base['ai_system'] == ai]
                sens_re_ai = sens_recal[sens_recal['ai_system'] == ai]
                spec_re_ai = spec_recal[spec_recal['ai_system'] == ai]

                x_min = min(sens_ai['curr_center'].min(), spec_ai['curr_center'].min())
                x_max = max(sens_ai['curr_center'].max(), spec_ai['curr_center'].max())
                ax.axvspan(x_min, transition_center, alpha=0.1, color='blue', label='Pre-start')
                ax.axvspan(transition_center, x_max, alpha=0.1, color='orange', label='Post-start')

                ax.plot(sens_ai['curr_center'], sens_ai['curr_rate'], color=colors[ai], label='Sensitivity (base)')
                ax.plot(spec_ai['curr_center'], spec_ai['curr_rate'], color=colors[ai], linestyle='--', label='Specificity (base)')
                ax.plot(sens_re_ai['curr_center'], sens_re_ai['curr_rate'], color=colors[ai], linewidth=2.5, label='Sensitivity (recal)')
                ax.plot(spec_re_ai['curr_center'], spec_re_ai['curr_rate'], color=colors[ai], linestyle='-.', label='Specificity (recal)')

                target_sens = _mean_metric(sens_ai, None, 'curr_rate', center_max=transition_center)
                target_spec = _mean_metric(spec_ai, None, 'curr_rate', center_max=transition_center)
                if pd.notna(target_sens):
                    ax.axhline(target_sens, color=colors[ai], linestyle=':', linewidth=1.3)
                if pd.notna(target_spec):
                    ax.axhline(target_spec, color=colors[ai], linestyle='-.', linewidth=1.1)

                recal_points = recal_points_map.get(ai, [])
                for j, rp in enumerate(recal_points):
                    ax.axvline(rp, color='black', linestyle=':', linewidth=1.2, label='Recalibration' if j == 0 else None)

                ax.axvline(transition_center, color='red', linestyle='--', linewidth=2, label='Start')
                ax.set_ylabel('Rate (%)')
                ax.set_title(ai.upper())
                ax.grid(True, alpha=0.3)
                ax.legend(fontsize=8, loc='upper left')

                plotted.append((hospital, manufacturer, ai))

            axes[-1].set_xlabel('Current Window Center (Exam Index)')
            start_label = str(start_method).replace('_', ' ')
            plt.suptitle(f'Part 3 Recalibration Effect: {hospital} — {manufacturer} ({metric_key}, {method}) | start: {start_label}',
                         fontsize=14, fontweight='bold')
            plt.tight_layout()
            plt.show()

    print('Plotted:', plotted)
    print('Skipped:', skipped)

part3_scenario_summaries = []
part3_display_tables = []

for scenario in PART3_SCENARIOS:
    label = scenario.get('label')
    label_lower = str(label).lower()
    short_label = 'single reader' if 'single' in label_lower else ('double reader' if 'double' in label_lower else str(label))
    display(Markdown(f"### {label}: Recalibration Summary"))
    scenario_summary = build_part3_recalibration_summary(
        df_part3, part3_start,
        calibration_strategy=PART3_CALIBRATION_STRATEGY,
        rad_mode=scenario.get('rad_mode'),
        calib_target_mode=scenario.get('target_mode'),
        system_mode=scenario.get('system_mode'),
        scenario_label=scenario.get('label')
    )
    if scenario_summary is None or len(scenario_summary) == 0:
        print('No rows returned for this scenario.')
        part3_display_tables.append({'key': scenario['key'], 'label': scenario['label'], 'df': None})
        continue

    part3_scenario_summaries.append(scenario_summary)

    scenario_interesting = _interesting_rows(scenario_summary)
    if scenario_interesting is None or len(scenario_interesting) == 0:
        print('No recalibration rows to display.')
        part3_display_tables.append({'key': scenario['key'], 'label': scenario['label'], 'df': None})
        continue

    display_df = scenario_interesting.copy()
    rename_map = {
        'scenario': 'Scenario',
        'hospital': 'Hospital',
        'manufacturer': 'Manufacturer',
        'ai_system': 'AI',
        'signal_metric': 'Signal Metric',
        'first_signal_center': 'First Signal',
        'signal_count': 'Signal count',
        'prestart_exam_count': 'Prestart exams',
        'prestart_cancer_count': 'Prestart cancers',
        'prestart_control_count': 'Prestart controls',
        'target_sensitivity': 'Sens pre (%)',
        'target_specificity': 'Spec pre (%)',
        'post_sens_base': 'Sens (no recal)',
        'post_spec_base': 'Spec (no recal)',
        'post_sens_recal': 'Sens (recal)',
        'post_spec_recal': 'Spec (recal)'
    }
    display_df = display_df.rename(columns=rename_map)
    display_df['Scenario'] = short_label

    # Deltas vs prestart targets
    display_df['Δ Sens (no recal)'] = display_df['Sens (no recal)'] - display_df['Sens pre (%)']
    display_df['Δ Sens (recal)'] = display_df['Sens (recal)'] - display_df['Sens pre (%)']
    display_df['Δ Spec (no recal)'] = display_df['Spec (no recal)'] - display_df['Spec pre (%)']
    display_df['Δ Spec (recal)'] = display_df['Spec (recal)'] - display_df['Spec pre (%)']

    ordered_cols = [
        'Scenario', 'Hospital', 'Manufacturer', 'AI', 'Signal Metric',
        'First Signal', 'Signal count', 'Prestart exams',
        'Sens pre (%)', 'Spec pre (%)',
        'Sens (no recal)', 'Spec (no recal)',
        'Sens (recal)', 'Spec (recal)',
        'Δ Sens (no recal)', 'Δ Sens (recal)',
        'Δ Spec (no recal)', 'Δ Spec (recal)'
    ]
    display_df = display_df[ordered_cols]

    num_cols = [
        'First Signal', 'Signal count', 'Prestart exams',
        'Sens pre (%)', 'Spec pre (%)',
        'Sens (no recal)', 'Spec (no recal)',
        'Sens (recal)', 'Spec (recal)',
        'Δ Sens (no recal)', 'Δ Sens (recal)',
        'Δ Spec (no recal)', 'Δ Spec (recal)'
    ]
    for col in num_cols:
        if col.startswith('Δ '):
            display_df[col] = display_df[col].map(lambda x: '' if pd.isna(x) else f"{x:+.2f}")
        elif col == 'Signal count':
            display_df[col] = display_df[col].map(lambda x: '' if pd.isna(x) else f"{x:.0f}")
        else:
            display_df[col] = display_df[col].map(lambda x: '' if pd.isna(x) else f"{x:.2f}")

    display(display_df)
    part3_display_tables.append({'key': scenario['key'], 'label': scenario['label'], 'df': display_df})

    # Styled table image (readable columns)
    from pathlib import Path
    output_dir = Path('outputs/2025_12_29_counterfactual_manufacturer_swap_setup')
    output_dir.mkdir(parents=True, exist_ok=True)

    fig_height = 2.0 + 0.5 * len(display_df)
    fig_width = max(22, 1.2 * len(display_df.columns) + 8)
    fig, ax = plt.subplots(figsize=(fig_width, fig_height))
    ax.axis('off')

    table = ax.table(
        cellText=display_df.values,
        colLabels=display_df.columns,
        cellLoc='center',
        loc='center',
        bbox=[0, 0, 1, 1]
    )

    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1, 1.7)

    header_color = '#4472C4'
    alt_row_color = '#E7E6E6'

    for i in range(len(display_df.columns)):
        cell = table[(0, i)]
        cell.set_facecolor(header_color)
        cell.set_text_props(weight='bold', color='white')

    for i in range(1, len(display_df) + 1):
        for j in range(len(display_df.columns)):
            cell = table[(i, j)]
            cell.set_facecolor(alt_row_color if i % 2 == 0 else 'white')

    col_weights = []
    for col in display_df.columns:
        if col in ['Scenario', 'Hospital', 'Manufacturer', 'Signal Metric']:
            col_weights.append(1.8)
        elif col == 'AI':
            col_weights.append(1.0)
        else:
            col_weights.append(1.1)
    total = sum(col_weights)
    col_widths = [w / total for w in col_weights]

    for col_idx, width in enumerate(col_widths):
        for row in range(len(display_df) + 1):
            table[(row, col_idx)].set_width(width)

    for cell in table.get_celld().values():
        cell.set_edgecolor('black')
        cell.set_linewidth(1.1)

    plt.title(f"Part 3 Recalibration Summary (All Rows) — {scenario['label']}",
              fontsize=16, fontweight='bold', pad=20)
    plt.tight_layout()
    table_path = output_dir / f"part3_recalibration_summary_{scenario['key']}.png"
    plt.savefig(table_path, dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()
    table_path

    if RUN_PART3_RECAL_PLOTS:
        plot_part3_recalibration_effects(
            df_part3, metric_key=PART3_RECAL_PLOT_METRIC,
            method=PART3_RECAL_PLOT_METHOD, alpha=PART3_RECAL_ALPHA,
            rad_mode=scenario.get('rad_mode'),
            calib_target_mode=scenario.get('target_mode'),
            system_mode=scenario.get('system_mode'),
            scenario_label=scenario.get('label')
        )

part3_recalibration_summary = pd.concat(part3_scenario_summaries, ignore_index=True) if part3_scenario_summaries else pd.DataFrame()



In [ ]:
# Copy-paste tables (TSV) per scenario
if 'part3_display_tables' in globals() and part3_display_tables:
    for entry in part3_display_tables:
        label = entry.get('label')
        df = entry.get('df')
        print(f"\n--- {label} ---")
        if df is None or len(df) == 0:
            print('No table to copy (display_df is empty).')
            continue
        tsv_text = df.to_csv(index=False, sep='\t')
        print(tsv_text)
else:
    print('No tables to copy.')


In [ ]:
# Stability vs target (sensitivity/specificity) per scenario
if 'part3_display_tables' in globals() and part3_display_tables:
    for entry in part3_display_tables:
        label = entry.get('label')
        df = entry.get('df')
        print(f"\n=== Stability Summary: {label} ===")
        if df is None or len(df) == 0:
            print('No table to summarize (display_df is empty).')
            continue

        stability_df = df.copy()
        num_cols = ['Sens pre (%)', 'Spec pre (%)', 'Sens (no recal)', 'Spec (no recal)', 'Sens (recal)', 'Spec (recal)']
        for col in num_cols:
            stability_df[col] = pd.to_numeric(stability_df[col], errors='coerce')

        stability_df['sens_drift_base'] = (stability_df['Sens (no recal)'] - stability_df['Sens pre (%)']).abs()
        stability_df['sens_drift_recal'] = (stability_df['Sens (recal)'] - stability_df['Sens pre (%)']).abs()
        stability_df['spec_drift_base'] = (stability_df['Spec (no recal)'] - stability_df['Spec pre (%)']).abs()
        stability_df['spec_drift_recal'] = (stability_df['Spec (recal)'] - stability_df['Spec pre (%)']).abs()

        stability_df['sens_improved'] = stability_df['sens_drift_recal'] < stability_df['sens_drift_base']
        stability_df['spec_improved'] = stability_df['spec_drift_recal'] < stability_df['spec_drift_base']

        summary_overall = pd.DataFrame({
            'metric': ['sensitivity', 'specificity'],
            'improved_rows': [stability_df['sens_improved'].sum(), stability_df['spec_improved'].sum()],
            'total_rows': [len(stability_df), len(stability_df)],
            'improved_pct': [
                100 * stability_df['sens_improved'].mean() if len(stability_df) > 0 else 0,
                100 * stability_df['spec_improved'].mean() if len(stability_df) > 0 else 0
            ],
            'median_drift_base': [stability_df['sens_drift_base'].median(), stability_df['spec_drift_base'].median()],
            'median_drift_recal': [stability_df['sens_drift_recal'].median(), stability_df['spec_drift_recal'].median()]
        })
        summary_overall['improved_pct'] = summary_overall['improved_pct'].round(1)
        summary_overall['median_drift_base'] = summary_overall['median_drift_base'].round(2)
        summary_overall['median_drift_recal'] = summary_overall['median_drift_recal'].round(2)
        display(summary_overall)

        summary_by_metric = stability_df.groupby('Signal Metric').agg(
            sens_improved_pct=('sens_improved', lambda s: 100 * s.mean()),
            spec_improved_pct=('spec_improved', lambda s: 100 * s.mean()),
            sens_median_base=('sens_drift_base', 'median'),
            sens_median_recal=('sens_drift_recal', 'median'),
            spec_median_base=('spec_drift_base', 'median'),
            spec_median_recal=('spec_drift_recal', 'median')
        ).reset_index()

        summary_by_metric['sens_improved_pct'] = summary_by_metric['sens_improved_pct'].round(1)
        summary_by_metric['spec_improved_pct'] = summary_by_metric['spec_improved_pct'].round(1)
        summary_by_metric['sens_median_base'] = summary_by_metric['sens_median_base'].round(2)
        summary_by_metric['sens_median_recal'] = summary_by_metric['sens_median_recal'].round(2)
        summary_by_metric['spec_median_base'] = summary_by_metric['spec_median_base'].round(2)
        summary_by_metric['spec_median_recal'] = summary_by_metric['spec_median_recal'].round(2)

        display(summary_by_metric)
else:
    print('No tables to summarize.')


In [ ]:
from IPython.display import display, Markdown

# Compact tables for publication (per scenario)
if 'part3_scenario_summaries' not in globals() or not part3_scenario_summaries:
    print('Run the Part 3 recalibration summary first.')
else:
    compact_tables = []
    for scenario in PART3_SCENARIOS:
        label = scenario.get('label')
        label_lower = str(label).lower()
        short_label = 'single reader' if 'single' in label_lower else ('double reader' if 'double' in label_lower else str(label))
        title_label = 'single reader (AI replaces R1)' if 'single' in label_lower else ('double reader (AI replaces R2; AI+R1)' if 'double' in label_lower else str(label))
        display(Markdown(f"### Compact Table — {title_label}"))
        df = None
        for s in part3_scenario_summaries:
            if len(s) == 0:
                continue
            if 'scenario' in s.columns and s['scenario'].iloc[0] == label:
                df = s.copy()
                break
        if df is None or len(df) == 0:
            print('No rows for this scenario.')
            continue

        compact_df = df.copy()
        compact_df['scenario'] = short_label
        compact_df['Δ Sens (no recal)'] = compact_df['post_sens_base'] - compact_df['target_sensitivity']
        compact_df['Δ Sens (recal)'] = compact_df['post_sens_recal'] - compact_df['target_sensitivity']
        compact_df['Δ Spec (no recal)'] = compact_df['post_spec_base'] - compact_df['target_specificity']
        compact_df['Δ Spec (recal)'] = compact_df['post_spec_recal'] - compact_df['target_specificity']

        compact_df = compact_df.rename(columns={
            'scenario': 'Scenario',
            'hospital': 'Hospital',
            'manufacturer': 'Manufacturer',
            'ai_system': 'AI',
            'signal_metric': 'Signal Metric',
            'signal_count': 'Signal count'
        })

        ordered_cols = [
            'Scenario', 'Hospital', 'Manufacturer', 'AI', 'Signal Metric',
            'Signal count',
            'Δ Sens (no recal)', 'Δ Sens (recal)',
            'Δ Spec (no recal)', 'Δ Spec (recal)'
        ]
        compact_df = compact_df[ordered_cols]

        for col in ['Signal count']:
            compact_df[col] = compact_df[col].map(lambda x: '' if pd.isna(x) else f"{x:.0f}")
        for col in ['Δ Sens (no recal)', 'Δ Sens (recal)', 'Δ Spec (no recal)', 'Δ Spec (recal)']:
            compact_df[col] = compact_df[col].map(lambda x: '' if pd.isna(x) else f"{x:+.2f}")

        display(compact_df)

        # Save compact table as image
        from pathlib import Path
        output_dir = Path('outputs/2025_12_29_counterfactual_manufacturer_swap_setup')
        output_dir.mkdir(parents=True, exist_ok=True)

        fig_height = 1.8 + 0.4 * len(compact_df)
        fig_width = max(18, 1.1 * len(compact_df.columns) + 6)
        fig, ax = plt.subplots(figsize=(fig_width, fig_height))
        ax.axis('off')

        table = ax.table(
            cellText=compact_df.values,
            colLabels=compact_df.columns,
            cellLoc='center',
            loc='center',
            bbox=[0, 0, 1, 1]
        )

        table.auto_set_font_size(False)
        table.set_fontsize(10)
        table.scale(1, 1.5)

        header_color = '#4472C4'
        alt_row_color = '#E7E6E6'

        for i in range(len(compact_df.columns)):
            cell_h = table[(0, i)]
            cell_h.set_facecolor(header_color)
            cell_h.set_text_props(weight='bold', color='white')

        for i in range(1, len(compact_df) + 1):
            for j in range(len(compact_df.columns)):
                cell_c = table[(i, j)]
                cell_c.set_facecolor(alt_row_color if i % 2 == 0 else 'white')

        col_weights = []
        for col in compact_df.columns:
            if col in ['Scenario', 'Hospital', 'Manufacturer', 'Signal Metric']:
                col_weights.append(1.6)
            elif col == 'AI':
                col_weights.append(1.0)
            else:
                col_weights.append(1.2)
        total = sum(col_weights)
        col_widths = [w / total for w in col_weights]

        for col_idx, width in enumerate(col_widths):
            for row in range(len(compact_df) + 1):
                table[(row, col_idx)].set_width(width)

        for cell_b in table.get_celld().values():
            cell_b.set_edgecolor('black')
            cell_b.set_linewidth(1.0)

        plt.title(f"Part 3 Compact Table - {title_label}", fontsize=14, fontweight='bold', pad=16)
        plt.tight_layout()
        safe_label = short_label.replace(' ', '_').replace(':', '').replace(';', '').replace(',', '')
        table_path = output_dir / f"part3_compact_table_{safe_label}.png"
        plt.savefig(table_path, dpi=300, bbox_inches='tight', facecolor='white')
        plt.show()

        compact_tables.append({'label': short_label, 'df': compact_df, 'image': str(table_path)})

    compact_tables


In [ ]:
from IPython.display import display, Markdown

# Summary by scenario × AI × Signal Metric (compact)
if 'part3_scenario_summaries' not in globals() or not part3_scenario_summaries:
    print('Run the Part 3 recalibration summary first.')
else:
    summary_tables = []
    for scenario in PART3_SCENARIOS:
        label = scenario.get('label')
        label_lower = str(label).lower()
        short_label = 'single reader' if 'single' in label_lower else ('double reader' if 'double' in label_lower else str(label))
        display(Markdown(f"### Compact Summary — {short_label}"))
        df = None
        for s in part3_scenario_summaries:
            if len(s) == 0:
                continue
            if 'scenario' in s.columns and s['scenario'].iloc[0] == label:
                df = s.copy()
                break
        if df is None or len(df) == 0:
            print('No rows for this scenario.')
            continue

        df['Δ Sens (no recal)'] = df['post_sens_base'] - df['target_sensitivity']
        df['Δ Sens (recal)'] = df['post_sens_recal'] - df['target_sensitivity']
        df['Δ Spec (no recal)'] = df['post_spec_base'] - df['target_specificity']
        df['Δ Spec (recal)'] = df['post_spec_recal'] - df['target_specificity']

        summary = df.groupby(['ai_system', 'signal_metric']).agg(
            n_rows=('ai_system', 'size'),
            sens_median_no_recal=('Δ Sens (no recal)', 'median'),
            sens_median_recal=('Δ Sens (recal)', 'median'),
            spec_median_no_recal=('Δ Spec (no recal)', 'median'),
            spec_median_recal=('Δ Spec (recal)', 'median'),
            sens_improved_pct=('Δ Sens (recal)', lambda s: 100 * (s.abs() < df.loc[s.index, 'Δ Sens (no recal)'].abs()).mean()),
            spec_improved_pct=('Δ Spec (recal)', lambda s: 100 * (s.abs() < df.loc[s.index, 'Δ Spec (no recal)'].abs()).mean())
        ).reset_index()

        summary = summary.rename(columns={
            'ai_system': 'AI',
            'signal_metric': 'Signal Metric',
            'n_rows': 'N rows'
        })
        summary['sens_median_no_recal'] = summary['sens_median_no_recal'].round(2)
        summary['sens_median_recal'] = summary['sens_median_recal'].round(2)
        summary['spec_median_no_recal'] = summary['spec_median_no_recal'].round(2)
        summary['spec_median_recal'] = summary['spec_median_recal'].round(2)
        summary['sens_improved_pct'] = summary['sens_improved_pct'].round(1)
        summary['spec_improved_pct'] = summary['spec_improved_pct'].round(1)

        display(summary)
        summary_tables.append({'label': short_label, 'df': summary})

    summary_tables


In [ ]:
# Part 3: Publication tables (one table per scenario)
from IPython.display import display, Markdown

if 'part3_scenario_summaries' not in globals() or not part3_scenario_summaries:
    print('Run the Part 3 recalibration summary first.')
elif 'df_part3' not in globals():
    print('df_part3 not found. Run the Part 3 data load cell first.')
else:
    from pathlib import Path
    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd

    output_dir = Path('outputs/2025_12_29_counterfactual_manufacturer_swap_setup')
    output_dir.mkdir(parents=True, exist_ok=True)

    def _format_date(d):
        if pd.isna(d):
            return ''
        return d.strftime('%b %Y')

    def _format_time_period(df_subset):
        if df_subset is None or len(df_subset) == 0 or DATE_COL not in df_subset.columns:
            return ''
        start = df_subset[DATE_COL].min()
        end = df_subset[DATE_COL].max()
        return f"{_format_date(start)} - {_format_date(end)}"

    def _scenario_title(label):
        label_lower = str(label).lower()
        if 'single' in label_lower:
            return 'Single-reader (AI replaces R1)'
        if 'double' in label_lower:
            return 'Double-reader (AI replaces R2; AI+R1)'
        return str(label)

    def _scenario_display_label(label):
        label_lower = str(label).lower()
        if 'single' in label_lower:
            return 'Single-reader'
        if 'double' in label_lower:
            return 'Double-reader'
        return str(label)

    def _ai_label(ai):
        return ai.capitalize()

    metrics = [
        ('flagging_rate', 'Flagging\nRate'),
        ('relative_flagging_rate', 'Rel Flag\nRate'),
        ('p_ai_given_rad', 'P(AI|Rad)'),
        ('p_rad_given_ai', 'P(Rad|AI)')
    ]

    num_cols = 6 + 2 * len(metrics)

    for scenario in PART3_SCENARIOS:
        label = scenario.get('label')
        display_label = _scenario_display_label(label)
        title_label = _scenario_title(label)
        scenario_df = None
        for s in part3_scenario_summaries:
            if len(s) == 0:
                continue
            if 'scenario' in s.columns and s['scenario'].iloc[0] == label:
                scenario_df = s.copy()
                scenario_df = scenario_df[~scenario_df['hospital'].str.contains('Danderyd', case=False, na=False)]
                break
        if scenario_df is None or len(scenario_df) == 0:
            print(f'No rows for scenario: {label}')
            continue

        rows = []
        for (hospital, manufacturer), group in scenario_df.groupby(['hospital', 'manufacturer']):
            if 'danderyd' in str(hospital).lower():
                continue
            df_subset = df_part3[(df_part3[HOSPITAL_COL] == hospital) & (df_part3[MANUFACTURER_COL] == manufacturer)]
            time_period = _format_time_period(df_subset)
            rows.append([f"{hospital} ({manufacturer})", '', '', time_period, '', ''] + [''] * (num_cols - 6))
            for ai in AI_COLUMNS:
                ai_group = group[group['ai_system'] == ai]
                if len(ai_group) == 0:
                    continue
                sens = ai_group['target_sensitivity'].dropna().iloc[0] if ai_group['target_sensitivity'].notna().any() else np.nan
                spec = ai_group['target_specificity'].dropna().iloc[0] if ai_group['target_specificity'].notna().any() else np.nan
                sens_txt = '' if pd.isna(sens) else f"{sens:.2f}"
                spec_txt = '' if pd.isna(spec) else f"{spec:.2f}"
                obs_sens = ''
                obs_spec = ''
                if len(ai_group) > 0:
                    base_row = ai_group.iloc[0]
                    ds_no = base_row['post_sens_base'] - base_row['target_sensitivity']
                    dp_no = base_row['post_spec_base'] - base_row['target_specificity']
                    obs_sens = '' if pd.isna(ds_no) else f"{ds_no:+.2f}"
                    obs_spec = '' if pd.isna(dp_no) else f"{dp_no:+.2f}"

                recal_vals = []
                for metric_key, _metric_label in metrics:
                    mrow = ai_group[ai_group['signal_metric'] == metric_key]
                    if len(mrow) == 0:
                        recal_vals.extend(['', ''])
                        continue
                    mrow = mrow.iloc[0]
                    ds_re = mrow['post_sens_recal'] - mrow['target_sensitivity']
                    dp_re = mrow['post_spec_recal'] - mrow['target_specificity']
                    sens_re = '' if pd.isna(ds_re) else f"{ds_re:+.2f}"
                    spec_re = '' if pd.isna(dp_re) else f"{dp_re:+.2f}"
                    recal_vals.extend([sens_re, spec_re])

                row = [f"  {_ai_label(ai)}", sens_txt, spec_txt, '', obs_sens, obs_spec] + recal_vals
                rows.append(row)

        metrics_start = 6
        metrics_span = 2 * len(metrics)

        header_0 = ['', 'Target\nMetrics', 'Target\nMetrics', 'Time period', 'Observed\nDeltas*', '']
        header_0 += ['Recalibrated using...'] + [''] * (metrics_span - 1)

        header_1 = ['AI Model', 'Sensitivity', 'Specificity', ''] + ['', '']
        for _key, metric_label in metrics:
            header_1.extend([metric_label, ''])

        header_2 = ['', '', '', '', '', '']
        for _key, _metric_label in metrics:
            header_2.extend(['Recalib\nDeltas*', ''])

        header_3 = ['', '', '', ''] + ['Sensitivity', 'Specificity']
        header_3 += ['Sensitivity', 'Specificity'] * len(metrics)

        table_text = [header_0, header_1, header_2, header_3] + rows

        fig_height = 2.7 + 0.33 * len(rows)
        fig_width = 24
        fig, ax = plt.subplots(figsize=(fig_width, fig_height))
        ax.axis('off')

        base_widths = [0.12, 0.05, 0.05, 0.09, 0.05, 0.05]
        metric_width = (1.0 - sum(base_widths)) / (2 * len(metrics))
        col_widths = base_widths + [metric_width] * (4 * len(metrics))

        table = ax.table(
            colWidths=col_widths,
            cellText=table_text,
            cellLoc='center',
            loc='center',
            bbox=[0, 0, 1, 1]
        )
        table.auto_set_font_size(False)
        table.set_fontsize(8)
        table.scale(1, 1.25)

        fig.canvas.draw()

        # Clear borders; we'll add only bottom lines under header labels
        for cell_obj in table.get_celld().values():
            cell_obj.set_linewidth(0.0)
            cell_obj.visible_edges = ''

        def _merge_row_cells(row, start, span):
            label = table[(row, start)].get_text().get_text()
            if label == '':
                return
            x0 = table[(row, start)].get_x()
            y0 = table[(row, start)].get_y()
            width = sum(table[(row, i)].get_width() for i in range(start, start + span))
            height = table[(row, start)].get_height()
            for i in range(start, start + span):
                cell = table[(row, i)]
                cell.set_linewidth(0.8)
                cell.visible_edges = 'B'
                cell.get_text().set_text('')
            ax.text(x0 + width / 2, y0 + height / 2, label, ha='center', va='center',
                    fontsize=table[(row, start)].get_text().get_fontsize(),
                    fontweight='bold', transform=ax.transAxes)

        # Merge Target Metrics header (row 0, cols 1-2)
        _merge_row_cells(0, 1, 2)
        _merge_row_cells(0, 4, 2)

        # Merge Recalibrated using... header across all metric columns (row 0)
        metrics_start = 6
        _merge_row_cells(0, metrics_start, metrics_span)

        # Merge metric headers on row 1
        metric_start = metrics_start
        for _ in metrics:
            _merge_row_cells(1, metric_start, 2)
            metric_start += 2

        # Merge observed/recal deltas on row 2
        metric_start = metrics_start
        for _ in metrics:
            _merge_row_cells(2, metric_start, 2)
            metric_start += 2

        # Bold header rows
        for j in range(len(header_3)):
            table[(0, j)].set_text_props(weight='bold')
            table[(1, j)].set_text_props(weight='bold')
            table[(2, j)].set_text_props(weight='bold')
            table[(3, j)].set_text_props(weight='bold')

        # Left-align key columns
        for i in range(len(table_text)):
            for j in (0, 3):
                if (i, j) in table.get_celld():
                    table[(i, j)].get_text().set_ha('left')

        # Add bottom borders for header cells with text
        for r in range(4):
            for c in range(len(header_3)):
                cell = table[(r, c)]
                if cell.get_text().get_text().strip():
                    cell.set_linewidth(0.8)
                    cell.visible_edges = 'B'
        plt.title(f"Table X. {title_label} scenario", fontsize=16, pad=18)
        plt.tight_layout()
        out_path = output_dir / f"part3_pub_table_{display_label.lower().replace(' ', '_')}"
        plt.savefig(out_path.with_suffix('.png'), dpi=300, bbox_inches='tight', facecolor='white')
        plt.show()

        print('* Deltas were calculated as (post - target) in percentage points. Observed = no recalibration; Recalibrated = with recalibration.')


## Part 3E: Recalibration plots for recommended metrics
Plots sensitivity/specificity before vs after recalibration, per hospital/manufacturer,
for the recommended recalibration metrics using method A.


In [ ]:
# It continued for 86 minutes (!), it takes too long
'''
# Recalibration plots for recommended metrics (separate single vs double reader)
if 'PART3_RECAL_ALPHA' not in globals():
    PART3_RECAL_ALPHA = 0.05
if 'PART3_RECAL_USE_SCORE_SIGNALS' not in globals():
    PART3_RECAL_USE_SCORE_SIGNALS = False
if 'PART3_RECAL_WINDOW_SIZE' not in globals() and 'PART3_REF_SIZE' in globals():
    PART3_RECAL_WINDOW_SIZE = PART3_REF_SIZE * 2

if '_transition_center' not in globals():
    def _transition_center(transition_idx, curr_size):
        if transition_idx is None or curr_size is None:
            return None
        return transition_idx + (curr_size / 2)



if '_get_target_metrics' not in globals():
    def _get_target_metrics(df_h, transition_idx, ref_size, gap_size, curr_size, step_size):
        targets = {ai: {} for ai in AI_COLUMNS}
        transition_center = _transition_center(transition_idx, curr_size)

        flagging = compute_flagging_rates(
            df_h, ref_size=ref_size, gap_size=gap_size, curr_size=curr_size, step_size=step_size
        )
        overlap = compute_conditional_overlap(
            df_h, ref_size=ref_size, gap_size=gap_size, curr_size=curr_size, step_size=step_size
        )
        rel_flagging = compute_relative_flagging_rates(flagging)

        for ai in AI_COLUMNS:
            flag_df = flagging.get(ai)
            if flag_df is not None and len(flag_df) > 0:
                targets[ai]['flagging_rate'] = flag_df[flag_df['curr_center'] < transition_center]['curr_pct'].mean()
            rel_df = rel_flagging.get(ai)
            if rel_df is not None and len(rel_df) > 0:
                targets[ai]['relative_flagging_rate'] = rel_df[rel_df['curr_center'] < transition_center]['curr_rel_rate'].mean()
            ov_df = overlap.get(ai)
            if ov_df is not None and len(ov_df) > 0:
                targets[ai]['p_ai_given_rad'] = ov_df[ov_df['curr_center'] < transition_center]['curr_p_ai_given_rad'].mean()
                targets[ai]['p_rad_given_ai'] = ov_df[ov_df['curr_center'] < transition_center]['curr_p_rad_given_ai'].mean()

        return targets

if '_signal_indices_after' not in globals():
    def _signal_indices_after(signal_centers, min_center):
        if not signal_centers:
            return []
        centers = [c for c in signal_centers if c is not None and c >= min_center]
        centers = sorted(set(centers))
        return [int(round(c)) for c in centers]


RECAL_METRICS_RECOMMENDED = ['flagging_rate', 'relative_flagging_rate', 'p_ai_given_rad', 'p_rad_given_ai']
RECAL_METHODS_RECOMMENDED = ['A']

recal_plot_dir = Path('outputs/2025_12_29_counterfactual_manufacturer_swap_setup/recalibration_plots')
recal_plot_dir.mkdir(parents=True, exist_ok=True)


def _sanitize_name(value):
    return ''.join(ch if ch.isalnum() or ch in ['_', '-'] else '_' for ch in str(value))


def _scenario_title(label):
    label_lower = str(label).lower()
    if 'single' in label_lower:
        return 'Single-reader (AI replaces R1)'
    if 'double' in label_lower:
        return 'Double-reader (AI replaces R2; AI+R1)'
    return str(label)


def plot_recalibration_effects_for_metric(df_part3, metric_key, method, scenario=None, alpha=PART3_RECAL_ALPHA):
    plotted = []
    skipped = []

    rad_mode = None
    system_mode = None
    scenario_label = ''
    if scenario is not None:
        rad_mode = scenario.get('rad_mode')
        system_mode = scenario.get('system_mode')
        scenario_label = _scenario_title(scenario.get('label', ''))

    hospitals = sorted(df_part3[HOSPITAL_COL].dropna().unique())
    for hospital in hospitals:
        if 'danderyd' in str(hospital).lower():
            continue
        df_hospital = df_part3[df_part3[HOSPITAL_COL] == hospital].copy()
        if MANUFACTURER_COL in df_hospital.columns:
            manufacturers = sorted(df_hospital[MANUFACTURER_COL].dropna().unique())
            if len(manufacturers) == 0:
                manufacturers = ['Unknown']
        else:
            manufacturers = ['Unknown']

        for manufacturer in manufacturers:
            if manufacturer == 'Unknown':
                df_subset = df_hospital.copy()
            else:
                df_subset = df_hospital[df_hospital[MANUFACTURER_COL] == manufacturer].copy()
            if len(df_subset) == 0:
                continue

            result = run_part3_hospital_pipeline(
                df_subset, hospital, part3_start, calibration_strategy=PART3_CALIBRATION_STRATEGY,
                rad_mode=rad_mode
            )
            if result is None:
                continue

            df_h = result['df_h']
            transition_idx = result['transition_index']
            transition_center = _transition_center(transition_idx, PART3_CURR_SIZE)
            start_method = result.get('start_method', '')

            if 'rad_flagged' not in df_h.columns:
                df_h = df_h.copy()
                df_h['rad_flagged'] = compute_radiologist_flag(df_h, mode=PART3_RAD_MODE)

            # Ensure consistent ordering before windowing
            df_h = order_part3_df(df_h)

            signal_centers = collect_part2_signal_centers(
                df_h, alpha=alpha,
                ref_size=PART3_REF_SIZE, gap_size=PART3_GAP_SIZE,
                curr_size=PART3_CURR_SIZE, step_size=PART3_STEP_SIZE
            )

            targets = _get_target_metrics(
                df_h, transition_idx,
                ref_size=PART3_REF_SIZE, gap_size=PART3_GAP_SIZE,
                curr_size=PART3_CURR_SIZE, step_size=PART3_STEP_SIZE
            )

            base_flag_suffix = 'part3'
            r1_flag = None
            if system_mode == 'ai_plus_r1':
                r1_flag = compute_radiologist_flag(df_h, mode='r1').astype(bool)
                base_flag_suffix = ensure_union_flags(df_h, 'part3', r1_flag)

            sens_base = compute_rate_windows_by_flag(
                df_h, flag_suffix=base_flag_suffix, rate_kind='sensitivity',
                ref_size=PART3_REF_SIZE, gap_size=PART3_GAP_SIZE,
                curr_size=PART3_CURR_SIZE, step_size=PART3_STEP_SIZE
            )
            spec_base = compute_rate_windows_by_flag(
                df_h, flag_suffix=base_flag_suffix, rate_kind='specificity',
                ref_size=PART3_REF_SIZE, gap_size=PART3_GAP_SIZE,
                curr_size=PART3_CURR_SIZE, step_size=PART3_STEP_SIZE
            )

            if sens_base is None or len(sens_base) == 0 or spec_base is None or len(spec_base) == 0:
                skipped.append((scenario_label, hospital, manufacturer, metric_key, method, 'no_base'))
                continue

            recal_flags_ready = {}
            recal_points_map = {}
            for ai in AI_COLUMNS:
                signal_key = metric_key if metric_key != 'relative_flagging_rate' else 'flagging_rate'
                signal_idxs = _signal_indices_after(signal_centers.get(ai, {}).get(signal_key, []), transition_center)
                if not signal_idxs:
                    skipped.append((scenario_label, hospital, manufacturer, ai, metric_key, method, 'no_signal'))
                    continue

                recal_flag_col = f'{ai}_flagged_recal_{metric_key}_{method}'
                df_h[recal_flag_col] = df_h[f'{ai}_flagged_part3']
                recal_points = []

                for signal_idx in signal_idxs:
                    start_idx = max(0, signal_idx - PART3_RECAL_WINDOW_SIZE)
                    recal_window = df_h.iloc[start_idx:signal_idx]
                    ref_start_idx = max(0, transition_idx - PART3_RECAL_WINDOW_SIZE)
                    ref_window = df_h.iloc[ref_start_idx:transition_idx]
                    target_value = targets.get(ai, {}).get(metric_key)

                    recal_thr = _compute_recal_threshold(
                        ai, metric_key, method, target_value,
                        recal_window=recal_window,
                        ref_window=ref_window
                    )
                    if pd.isna(recal_thr):
                        continue

                    df_h.loc[signal_idx:, recal_flag_col] = (df_h.loc[signal_idx:, ai] >= recal_thr).astype(int)
                    recal_points.append(signal_idx)

                if not recal_points:
                    skipped.append((scenario_label, hospital, manufacturer, ai, metric_key, method, 'no_threshold'))
                    continue

                recal_flags_ready[ai] = recal_flag_col
                recal_points_map[ai] = recal_points

            if len(recal_flags_ready) == 0:
                skipped.append((scenario_label, hospital, manufacturer, metric_key, method, 'no_recal'))
                continue

            def _plot_rate(rate_kind, ylabel, base_ls, recal_ls, title_suffix):
                fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)
                colors = {'lunit': '#ff7f0e', 'therapixel': '#2ca02c', 'vara': '#d62728'}

                for idx, ai in enumerate(AI_COLUMNS):
                    ax = axes[idx]
                    base_df = sens_base if rate_kind == 'sensitivity' else spec_base
                    base_ai = base_df[base_df['ai_system'] == ai]

                    if len(base_ai) == 0:
                        ax.text(0.5, 0.5, 'No data', ha='center', va='center')
                        ax.set_title(ai.upper())
                        ax.grid(True, alpha=0.3)
                        continue

                    x_min = base_ai['curr_center'].min()
                    x_max = base_ai['curr_center'].max()
                    ax.axvspan(x_min, transition_center, alpha=0.1, color='blue', label='Pre-start')
                    ax.axvspan(transition_center, x_max, alpha=0.1, color='orange', label='Post-start')

                    ax.plot(base_ai['curr_center'], base_ai['curr_rate'], color=colors[ai], linestyle=base_ls, label=f"{ylabel} (base)")

                    if ai in recal_flags_ready:
                        recal_suffix = recal_flags_ready[ai].split(f'{ai}_flagged_')[1]
                        recal_df = compute_rate_windows_by_flag(
                            df_h, flag_suffix=recal_suffix, rate_kind=rate_kind,
                            ref_size=PART3_REF_SIZE, gap_size=PART3_GAP_SIZE,
                            curr_size=PART3_CURR_SIZE, step_size=PART3_STEP_SIZE
                        )
                        recal_ai = recal_df[recal_df['ai_system'] == ai]
                        ax.plot(recal_ai['curr_center'], recal_ai['curr_rate'], color=colors[ai], linestyle=recal_ls,
                                linewidth=2.5, label=f"{ylabel} (recal)")

                        recal_points = recal_points_map.get(ai, [])
                        for j, rp in enumerate(recal_points):
                            ax.axvline(rp, color='black', linestyle=':', linewidth=1.2, label='Recalibration' if j == 0 else None)
                    else:
                        ax.text(0.98, 0.92, 'No recalibration', transform=ax.transAxes,
                                fontsize=8, color='gray', ha='right')

                    target_val = _mean_metric(base_ai, None, 'curr_rate', center_max=transition_center)
                    if pd.notna(target_val):
                        ax.axhline(target_val, color=colors[ai], linestyle=':', linewidth=1.2)

                    ax.axvline(transition_center, color='red', linestyle='--', linewidth=2, label='Start')
                    ax.set_ylabel('Rate (%)')
                    ax.set_ylim(0, 100)
                    ax.set_title(ai.upper())
                    ax.grid(True, alpha=0.3)
                    ax.legend(fontsize=8, loc='upper left')

                axes[-1].set_xlabel('Current Window Center (Exam Index)')
                start_label = str(start_method).replace('_', ' ')
                title = f"Part 3 Recalibration Effect ({title_suffix}): {hospital} — {manufacturer} | {scenario_label} | {metric_key} | start: {start_label}"
                plt.suptitle(title, fontsize=14, fontweight='bold')
                plt.tight_layout()

                scenario_slug = _sanitize_name(scenario_label) if scenario_label else 'default'
                out_dir = recal_plot_dir / scenario_slug
                out_dir.mkdir(parents=True, exist_ok=True)
                filename = f"recal_{_sanitize_name(hospital)}_{_sanitize_name(manufacturer)}_{metric_key}_{rate_kind}.png"
                plt.savefig(out_dir / filename, dpi=300, bbox_inches='tight', facecolor='white')
                plt.show()

            _plot_rate('sensitivity', 'Sensitivity', base_ls='-', recal_ls='-', title_suffix='Sensitivity')
            _plot_rate('specificity', 'Specificity', base_ls='--', recal_ls='-.', title_suffix='Specificity')

            plotted.append((scenario_label, hospital, manufacturer, metric_key, method))

    return plotted, skipped

all_plotted = []
all_skipped = []
for scenario in PART3_SCENARIOS:
    for metric in RECAL_METRICS_RECOMMENDED:
        for method in RECAL_METHODS_RECOMMENDED:
            plotted, skipped = plot_recalibration_effects_for_metric(
                df_part3, metric, method, scenario=scenario
            )
            all_plotted.extend(plotted)
            all_skipped.extend(skipped)

print('Plotted:', all_plotted)
print('Skipped:', all_skipped)

'''


In [ ]:
# Danderyds radiologist sensitivity audit
dand = df_part3[df_part3[HOSPITAL_COL] == 'Danderyds Hospital'].copy()
cancers = dand[dand['has_cancer'] == 1]

print("Cancer rows:", len(cancers))
print("Missing r1:", cancers['r1'].isna().mean(), "Missing r2:", cancers['r2'].isna().mean())

print("\nTop r1 values (cancers):")
print(cancers['r1'].value_counts(dropna=False).head(10))

print("\nTop r2 values (cancers):")
print(cancers['r2'].value_counts(dropna=False).head(10))

# Sensitivity under current rule
curr_sens_r1 = ((cancers['r1'] == 'DISCUSSION')).mean() * 100
curr_sens_comb = (((cancers['r1'] == 'DISCUSSION') | (cancers['r2'] == 'DISCUSSION'))).mean() * 100
print("\nCurrent rule sensitivity:")
print("R1:", curr_sens_r1, "Combined:", curr_sens_comb)

# Sensitivity for screen-detected only (if you want to compare)
if 'time_based_screen' in cancers.columns:
    screen_cancers = cancers[cancers['time_based_screen'] == True]
    print("\nScreen-detected cancers:", len(screen_cancers))
    print("R1 DISCUSSION sensitivity (screen only):", (screen_cancers['r1']=='DISCUSSION').mean()*100)


In [ ]:
# Part 3: Average calendar time to accumulate N exams per hospital/manufacturer
# Uses date counts (after upscaling) and computes span between start/end dates for windows reaching N exams.

WINDOW_SIZES = [1500, 2000, 3000, 4000, 5000, 6000]

if 'df_part3' not in globals():
    print('df_part3 not found. Run the Part 3 data load cell first.')
else:
    import numpy as np
    import pandas as pd

    group_cols = [HOSPITAL_COL]
    if MANUFACTURER_COL in df_part3.columns:
        group_cols.append(MANUFACTURER_COL)

    def _window_spans_by_date_counts(df_group, window_size):
        # Aggregate counts per date (date-level, not time)
        if DATE_COL not in df_group.columns or len(df_group) == 0:
            return []
        dates = pd.to_datetime(df_group[DATE_COL]).dt.normalize()
        counts = dates.value_counts().sort_index()
        if counts.sum() < window_size:
            return []
        cumsum = counts.cumsum().values
        date_index = counts.index.to_numpy()

        spans = []
        # For each end date, find earliest start date to reach window_size
        for end_i, end_total in enumerate(cumsum):
            target = end_total - window_size
            if target < 0:
                continue
            start_i = np.searchsorted(cumsum, target, side='right')
            span_days = (date_index[end_i] - date_index[start_i]).astype('timedelta64[D]').astype(int)
            spans.append(span_days)
        return spans

    rows = []
    for keys, df_group in df_part3.groupby(group_cols):
        # Ensure deterministic ordering, but we use date counts anyway
        df_group = df_group.copy()
        df_group = order_part3_df(df_group)

        if isinstance(keys, tuple):
            hospital = keys[0]
            manufacturer = keys[1] if len(keys) > 1 else 'All'
        else:
            hospital = keys
            manufacturer = 'All'

        for w in WINDOW_SIZES:
            spans = _window_spans_by_date_counts(df_group, w)
            if len(spans) == 0:
                rows.append({
                    'Hospital': hospital,
                    'Manufacturer': manufacturer,
                    'Window size (exams)': w,
                    'Avg days': np.nan,
                    'Median days': np.nan,
                    'Avg months': np.nan,
                    'N windows': 0
                })
                continue
            avg_days = float(np.mean(spans))
            med_days = float(np.median(spans))
            rows.append({
                'Hospital': hospital,
                'Manufacturer': manufacturer,
                'Window size (exams)': w,
                'Avg days': avg_days,
                'Median days': med_days,
                'Avg months': avg_days / 30.44,
                'N windows': len(spans)
            })

    window_time_df = pd.DataFrame(rows)
    window_time_df = window_time_df.sort_values(['Hospital', 'Manufacturer', 'Window size (exams)'])

    def _fmt(x):
        return '' if pd.isna(x) else f"{x:.1f}"

    display_df = window_time_df.copy()
    for col in ['Avg days', 'Median days', 'Avg months']:
        display_df[col] = display_df[col].map(_fmt)

    display(display_df)


In [ ]:
# Part 3: Max deviation from target (bars per AI, per scenario)
# New summary focusing on worst deviations after start.

if 'PART3_RECAL_ALPHA' not in globals():
    PART3_RECAL_ALPHA = 0.05
if 'PART3_RECAL_USE_SCORE_SIGNALS' not in globals():
    PART3_RECAL_USE_SCORE_SIGNALS = False
if 'PART3_RECAL_WINDOW_SIZE' not in globals() and 'PART3_REF_SIZE' in globals():
    PART3_RECAL_WINDOW_SIZE = PART3_REF_SIZE * 2
if '_signal_indices_after' not in globals():
    def _signal_indices_after(signal_centers, min_center):
        if not signal_centers:
            return []
        centers = [c for c in signal_centers if c is not None and c >= min_center]
        centers = sorted(set(centers))
        return [int(round(c)) for c in centers]

EXCLUDE_DANDERYDS = True
DEVIATION_METRICS = ['flagging_rate', 'relative_flagging_rate', 'p_ai_given_rad', 'p_rad_given_ai']
DEVIATION_METHOD = 'A'  # only one method in use

if 'df_part3' not in globals():
    print('df_part3 not found. Run the Part 3 data load cell first.')
else:
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt

    if '_get_target_metrics' not in globals():
        def _get_target_metrics(df_h, transition_idx, ref_size, gap_size, curr_size, step_size):
            targets = {ai: {} for ai in AI_COLUMNS}
            transition_center = _transition_center(transition_idx, curr_size)
    
            flagging = compute_flagging_rates(
                df_h, ref_size=ref_size, gap_size=gap_size, curr_size=curr_size, step_size=step_size
            )
            overlap = compute_conditional_overlap(
                df_h, ref_size=ref_size, gap_size=gap_size, curr_size=curr_size, step_size=step_size
            )
            rel_flagging = compute_relative_flagging_rates(flagging)
    
            for ai in AI_COLUMNS:
                flag_df = flagging.get(ai)
                if flag_df is not None and len(flag_df) > 0:
                    targets[ai]['flagging_rate'] = flag_df[flag_df['curr_center'] < transition_center]['curr_pct'].mean()
                rel_df = rel_flagging.get(ai)
                if rel_df is not None and len(rel_df) > 0:
                    targets[ai]['relative_flagging_rate'] = rel_df[rel_df['curr_center'] < transition_center]['curr_rel_rate'].mean()
                ov_df = overlap.get(ai)
                if ov_df is not None and len(ov_df) > 0:
                    targets[ai]['p_ai_given_rad'] = ov_df[ov_df['curr_center'] < transition_center]['curr_p_ai_given_rad'].mean()
                    targets[ai]['p_rad_given_ai'] = ov_df[ov_df['curr_center'] < transition_center]['curr_p_rad_given_ai'].mean()
    
            return targets


    def _max_signed_devs(series, target, center_min=None):
        if series is None or len(series) == 0 or pd.isna(target):
            return np.nan, np.nan
        df = series
        if center_min is not None and 'curr_center' in df.columns:
            df = df[df['curr_center'] >= center_min]
        diffs = (df['curr_rate'] - target).dropna()
        if len(diffs) == 0:
            return np.nan, np.nan
        return float(diffs.max()), float(diffs.min())

    def _compute_recal_flags(df_h, ai, metric_key, signal_idxs, targets, transition_idx):
        recal_flag_col = f'{ai}_flagged_recal_{metric_key}_{DEVIATION_METHOD}'
        df_h[recal_flag_col] = df_h[f'{ai}_flagged_part3']
        recal_points = []
        for signal_idx in signal_idxs:
            start_idx = max(0, signal_idx - PART3_RECAL_WINDOW_SIZE)
            recal_window = df_h.iloc[start_idx:signal_idx]
            ref_start_idx = max(0, transition_idx - PART3_RECAL_WINDOW_SIZE)
            ref_window = df_h.iloc[ref_start_idx:transition_idx]
            target_value = targets.get(ai, {}).get(metric_key)

            recal_thr = _compute_recal_threshold(
                ai, metric_key, DEVIATION_METHOD, target_value,
                recal_window=recal_window,
                ref_window=ref_window
            )
            if pd.isna(recal_thr):
                continue

            df_h.loc[signal_idx:, recal_flag_col] = (df_h.loc[signal_idx:, ai] >= recal_thr).astype(int)
            recal_points.append(signal_idx)
        return recal_flag_col, recal_points

    def _scenario_label(label):
        label_lower = str(label).lower()
        if 'single' in label_lower:
            return 'Single-reader'
        if 'double' in label_lower:
            return 'Double-reader'
        return str(label)

    rows = []
    for scenario in PART3_SCENARIOS:
        rad_mode = scenario.get('rad_mode')
        system_mode = scenario.get('system_mode')
        scenario_label = _scenario_label(scenario.get('label', ''))

        hospitals = sorted(df_part3[HOSPITAL_COL].dropna().unique())
        for hospital in hospitals:
            if EXCLUDE_DANDERYDS and 'danderyd' in str(hospital).lower():
                continue
            df_hospital = df_part3[df_part3[HOSPITAL_COL] == hospital].copy()
            if MANUFACTURER_COL in df_hospital.columns:
                manufacturers = sorted(df_hospital[MANUFACTURER_COL].dropna().unique())
                if len(manufacturers) == 0:
                    manufacturers = ['Unknown']
            else:
                manufacturers = ['Unknown']

            for manufacturer in manufacturers:
                if manufacturer == 'Unknown':
                    df_subset = df_hospital.copy()
                else:
                    df_subset = df_hospital[df_hospital[MANUFACTURER_COL] == manufacturer].copy()
                if len(df_subset) == 0:
                    continue

                result = run_part3_hospital_pipeline(
                    df_subset, hospital, part3_start, calibration_strategy=PART3_CALIBRATION_STRATEGY,
                    rad_mode=rad_mode
                )
                if result is None:
                    continue

                rad_mode_eff, calib_target_mode, system_mode_eff = resolve_part3_modes(rad_mode, None, system_mode)

                target_sens_pre = result.get('radiologist_sensitivity_pre', {}).get(calib_target_mode, np.nan)
                target_spec_pre = result.get('radiologist_specificity_pre', {}).get(calib_target_mode, np.nan)

                df_h = result['df_h']
                df_h = order_part3_df(df_h)
                transition_idx = result['transition_index']
                transition_center = _transition_center(transition_idx, PART3_CURR_SIZE)

                if 'rad_flagged' not in df_h.columns:
                    df_h = df_h.copy()
                    df_h['rad_flagged'] = compute_radiologist_flag(df_h, mode=PART3_RAD_MODE)

                # Ensure base AI flag columns exist for target metrics
                for ai in AI_COLUMNS:
                    base_flag = f'{ai}_flagged'
                    part3_flag = f'{ai}_flagged_part3'
                    if base_flag not in df_h.columns and part3_flag in df_h.columns:
                        df_h[base_flag] = df_h[part3_flag]

                targets = _get_target_metrics(
                    df_h, transition_idx,
                    ref_size=PART3_REF_SIZE, gap_size=PART3_GAP_SIZE,
                    curr_size=PART3_CURR_SIZE, step_size=PART3_STEP_SIZE
                )

                base_flag_suffix = 'part3'
                r1_flag = None
                if system_mode_eff == 'ai_plus_r1':
                    r1_flag = compute_radiologist_flag(df_h, mode='r1').astype(bool)
                    base_flag_suffix = ensure_union_flags(df_h, 'part3', r1_flag)

                sens_base = compute_rate_windows_by_flag(
                    df_h, flag_suffix=base_flag_suffix, rate_kind='sensitivity',
                    ref_size=PART3_REF_SIZE, gap_size=PART3_GAP_SIZE,
                    curr_size=PART3_CURR_SIZE, step_size=PART3_STEP_SIZE
                )
                spec_base = compute_rate_windows_by_flag(
                    df_h, flag_suffix=base_flag_suffix, rate_kind='specificity',
                    ref_size=PART3_REF_SIZE, gap_size=PART3_GAP_SIZE,
                    curr_size=PART3_CURR_SIZE, step_size=PART3_STEP_SIZE
                )

                signal_centers = collect_part2_signal_centers(
                    df_h, alpha=PART3_RECAL_ALPHA,
                    ref_size=PART3_REF_SIZE, gap_size=PART3_GAP_SIZE,
                    curr_size=PART3_CURR_SIZE, step_size=PART3_STEP_SIZE,
                    use_score_signals=PART3_RECAL_USE_SCORE_SIGNALS
                )

                for ai in AI_COLUMNS:
                    target_sens = target_sens_pre
                    target_spec = target_spec_pre

                    # Base (uncalibrated) max deviations
                    sens_base_ai = sens_base[sens_base['ai_system'] == ai]
                    spec_base_ai = spec_base[spec_base['ai_system'] == ai]
                    sens_base_max, sens_base_min = _max_signed_devs(sens_base_ai, target_sens, center_min=transition_center)
                    spec_base_max, spec_base_min = _max_signed_devs(spec_base_ai, target_spec, center_min=transition_center)

                    rows.append({
                        'Scenario': scenario_label,
                        'Hospital': hospital,
                        'Manufacturer': manufacturer,
                        'AI': ai,
                        'Metric': 'uncalibrated',
                        'Max dev sens': sens_base_max,
                        'Min dev sens': sens_base_min,
                        'Max dev spec': spec_base_max,
                        'Min dev spec': spec_base_min
                    })

                    # Recalibrated by metrics
                    for metric_key in DEVIATION_METRICS:
                        signal_key = metric_key if metric_key != 'relative_flagging_rate' else 'flagging_rate'
                        signal_idxs = _signal_indices_after(signal_centers.get(ai, {}).get(signal_key, []), transition_center)
                        if not signal_idxs:
                            rows.append({
                                'Scenario': scenario_label,
                                'Hospital': hospital,
                                'Manufacturer': manufacturer,
                                'AI': ai,
                                'Metric': metric_key,
                                'Max dev sens': np.nan,
                                'Max dev spec': np.nan
                            })
                            continue

                        recal_flag_col, _recal_points = _compute_recal_flags(df_h, ai, metric_key, signal_idxs, targets, transition_idx)
                        recal_suffix = recal_flag_col.split(f'{ai}_flagged_')[1]
                        if system_mode_eff == 'ai_plus_r1':
                            if r1_flag is None:
                                r1_flag = compute_radiologist_flag(df_h, mode='r1').astype(bool)
                            union_suffix = f"{recal_suffix}_r1_union"
                            df_h[f'{ai}_flagged_{union_suffix}'] = (
                                (df_h[recal_flag_col].fillna(0).astype(int) == 1) | r1_flag
                            ).astype(int)
                            recal_suffix = union_suffix

                        sens_recal = compute_rate_windows_by_flag(
                            df_h, flag_suffix=recal_suffix, rate_kind='sensitivity',
                            ref_size=PART3_REF_SIZE, gap_size=PART3_GAP_SIZE,
                            curr_size=PART3_CURR_SIZE, step_size=PART3_STEP_SIZE
                        )
                        spec_recal = compute_rate_windows_by_flag(
                            df_h, flag_suffix=recal_suffix, rate_kind='specificity',
                            ref_size=PART3_REF_SIZE, gap_size=PART3_GAP_SIZE,
                            curr_size=PART3_CURR_SIZE, step_size=PART3_STEP_SIZE
                        )

                        sens_re_ai = sens_recal[sens_recal['ai_system'] == ai]
                        spec_re_ai = spec_recal[spec_recal['ai_system'] == ai]
                        sens_re_max, sens_re_min = _max_signed_devs(sens_re_ai, target_sens, center_min=transition_center)
                        spec_re_max, spec_re_min = _max_signed_devs(spec_re_ai, target_spec, center_min=transition_center)

                        rows.append({
                            'Scenario': scenario_label,
                            'Hospital': hospital,
                            'Manufacturer': manufacturer,
                            'AI': ai,
                            'Metric': metric_key,
                            'Max dev sens': sens_re_max,
                            'Min dev sens': sens_re_min,
                            'Max dev spec': spec_re_max,
                            'Min dev spec': spec_re_min
                        })

    dev_df = pd.DataFrame(rows)

    # Plot bars: one bar per AI, colors = metric (uncalibrated + recal metrics)
    metric_order = ['uncalibrated'] + DEVIATION_METRICS
    metric_labels = {
        'uncalibrated': 'Uncalibrated',
        'flagging_rate': 'Flagging rate',
        'relative_flagging_rate': 'Rel flag rate',
        'p_ai_given_rad': 'P(AI|Rad)',
        'p_rad_given_ai': 'P(Rad|AI)'
    }
    metric_colors = {
        'uncalibrated': '#1b6a8a',
        'flagging_rate': '#5cb85c',
        'relative_flagging_rate': '#c77dd4',
        'p_ai_given_rad': '#d07a28',
        'p_rad_given_ai': '#e43d3d'
    }

    for scenario_label in dev_df['Scenario'].dropna().unique():
        df_s = dev_df[dev_df['Scenario'] == scenario_label]
        for (hospital, manufacturer), df_h in df_s.groupby(['Hospital', 'Manufacturer']):
            # Build per-AI bar positions
            ais = [ai for ai in AI_COLUMNS if ai in df_h['AI'].unique()]
            if len(ais) == 0:
                continue

            fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
            axes[0].set_title('Sensitivity')
            axes[1].set_title('Specificity')

            x = np.arange(len(ais))
            bar_w = 0.12
            offsets = np.linspace(-(len(metric_order)-1)/2, (len(metric_order)-1)/2, len(metric_order)) * bar_w

            for j, metric_key in enumerate(metric_order):
                df_m = df_h[df_h['Metric'] == metric_key]
                sens_vals = []
                sens_min_vals = []
                spec_vals = []
                spec_min_vals = []
                for ai in ais:
                    row = df_m[df_m['AI'] == ai]
                    sens_vals.append(row['Max dev sens'].iloc[0] if len(row) > 0 else np.nan)
                    sens_min_vals.append(row['Min dev sens'].iloc[0] if len(row) > 0 else np.nan)
                    spec_vals.append(row['Max dev spec'].iloc[0] if len(row) > 0 else np.nan)
                    spec_min_vals.append(row['Min dev spec'].iloc[0] if len(row) > 0 else np.nan)

                axes[0].bar(x + offsets[j], sens_vals, width=bar_w, color=metric_colors.get(metric_key, '#999999'), label=metric_labels.get(metric_key, metric_key))
                axes[0].bar(x + offsets[j], sens_min_vals, width=bar_w, color=metric_colors.get(metric_key, '#999999'))
                axes[1].bar(x + offsets[j], spec_vals, width=bar_w, color=metric_colors.get(metric_key, '#999999'))
                axes[1].bar(x + offsets[j], spec_min_vals, width=bar_w, color=metric_colors.get(metric_key, '#999999'))

            for ax in axes:
                ax.axhline(0, color='#333333', linewidth=1)
                ax.set_xticks(x)
                ax.set_xticklabels([ai.upper() for ai in ais])
                ax.set_ylabel('Deviation from target (pp)')
                ax.grid(True, axis='y', alpha=0.3)

            axes[0].legend(loc='upper right', fontsize=8, frameon=False)
            fig.suptitle(f'Max deviation from target | {hospital} — {manufacturer} | {scenario_label}', fontsize=12, fontweight='bold')
            plt.tight_layout()
            plt.show()


In [ ]:
# Part 3: Noise vs current window size (uncalibrated only)
# Explore how max deviation changes as CURR_SIZE increases.

# Choose hospital/manufacturer and scenario
NOISE_HOSPITAL = 'Karolinska University Hospital'
NOISE_MANUFACTURER = 'Hologic'  # set to a specific manufacturer name or keep None for all
NOISE_SCENARIO_LABEL = 'single'  # 'single' or 'double'

# Window sizes to test: stepwise up to half of available exams
MIN_CURR = 1500
STEP_CURR = 500
MAX_FRAC = 0.5

if '_transition_center' not in globals():
    def _transition_center(transition_idx, curr_size):
        if transition_idx is None or curr_size is None:
            return None
        return transition_idx + (curr_size / 2)

if 'df_part3' not in globals():
    print('df_part3 not found. Run the Part 3 data load cell first.')
else:
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt

    # Pick scenario
    scenario = None
    for s in PART3_SCENARIOS:
        label = str(s.get('label','')).lower()
        if NOISE_SCENARIO_LABEL in label:
            scenario = s
            break
    if scenario is None:
        print('Scenario not found; check NOISE_SCENARIO_LABEL.')
    else:
        rad_mode = scenario.get('rad_mode')
        system_mode = scenario.get('system_mode')

        df_h = df_part3[df_part3[HOSPITAL_COL] == NOISE_HOSPITAL].copy()
        if len(df_h) == 0:
            print(f'No data for hospital: {NOISE_HOSPITAL}')
        else:
            if MANUFACTURER_COL in df_h.columns:
                if NOISE_MANUFACTURER is None:
                    manufacturers = sorted(df_h[MANUFACTURER_COL].dropna().unique())
                    NOISE_MANUFACTURER = manufacturers[0] if len(manufacturers) > 0 else None
                if NOISE_MANUFACTURER is not None:
                    df_h = df_h[df_h[MANUFACTURER_COL] == NOISE_MANUFACTURER].copy()

            if len(df_h) == 0:
                print(f'No data for hospital/manufacturer: {NOISE_HOSPITAL} / {NOISE_MANUFACTURER}')
            else:
                result = run_part3_hospital_pipeline(
                    df_h, NOISE_HOSPITAL, part3_start, calibration_strategy=PART3_CALIBRATION_STRATEGY,
                    rad_mode=rad_mode
                )
                if result is None:
                    print('Pipeline failed for selected hospital/manufacturer.')
                else:
                    df_h = order_part3_df(result['df_h'])
                    transition_idx = result['transition_index']
                    total_exams = len(df_h)
                    max_curr = max(int(total_exams * MAX_FRAC), MIN_CURR)

                    # Build window sizes
                    curr_sizes = list(range(MIN_CURR, max_curr + 1, STEP_CURR))

                    # Ensure base flags exist
                    for ai in AI_COLUMNS:
                        base_flag = f'{ai}_flagged'
                        part3_flag = f'{ai}_flagged_part3'
                        if base_flag not in df_h.columns and part3_flag in df_h.columns:
                            df_h[base_flag] = df_h[part3_flag]

                    # Targets from pre-start (radiologist)
                    rad_mode_eff, calib_target_mode, system_mode_eff = resolve_part3_modes(rad_mode, None, system_mode)
                    target_sens_pre = result.get('radiologist_sensitivity_pre', {}).get(calib_target_mode, np.nan)
                    target_spec_pre = result.get('radiologist_specificity_pre', {}).get(calib_target_mode, np.nan)

                    def _cumsum(arr):
                        return np.concatenate(([0.0], np.cumsum(arr)))

                    def _window_sum(csum, starts, ends):
                        return csum[ends] - csum[starts]

                    def _window_rate(num_csum, den_csum, starts, ends, scale=100.0):
                        num = _window_sum(num_csum, starts, ends)
                        den = _window_sum(den_csum, starts, ends)
                        out = np.full_like(num, np.nan, dtype=float)
                        mask = den > 0
                        out[mask] = (num[mask] / den[mask]) * scale
                        return out

                    def _max_min(diffs):
                        diffs = np.array(diffs, dtype=float)
                        diffs = diffs[~np.isnan(diffs)]
                        if len(diffs) == 0:
                            return np.nan, np.nan
                        return float(diffs.max()), float(diffs.min())

                    # Precompute window indices per curr_size
                    window_index = {}
                    for curr in curr_sizes:
                        total_span = PART3_REF_SIZE + PART3_GAP_SIZE + curr
                        centers = []
                        starts = []
                        ends = []
                        for ref_start in range(0, len(df_h) - total_span + 1, PART3_STEP_SIZE):
                            ref_end = ref_start + PART3_REF_SIZE
                            curr_start = ref_end + PART3_GAP_SIZE
                            curr_end = curr_start + curr
                            curr_center = (curr_start + curr_end) / 2
                            centers.append(curr_center)
                            starts.append(curr_start)
                            ends.append(curr_end)
                        window_index[curr] = (
                            np.array(centers, dtype=float),
                            np.array(starts, dtype=int),
                            np.array(ends, dtype=int)
                        )

                    has_cancer = df_h['has_cancer'].fillna(0).astype(int).to_numpy()
                    has_control = (has_cancer == 0).astype(int)

                    rows = []
                    for ai in AI_COLUMNS:
                        flag_col = f'{ai}_flagged_part3'
                        if flag_col not in df_h.columns:
                            continue
                        flags = df_h[flag_col].fillna(0).astype(int).to_numpy()

                        c_flag_cancer = _cumsum(flags * has_cancer)
                        c_cancer = _cumsum(has_cancer)
                        c_flag_control = _cumsum(flags * has_control)
                        c_control = _cumsum(has_control)

                        for curr in curr_sizes:
                            centers, starts, ends = window_index[curr]
                            sens = _window_rate(c_flag_cancer, c_cancer, starts, ends)
                            flagged_rate = _window_rate(c_flag_control, c_control, starts, ends, scale=1.0)
                            spec = (1.0 - flagged_rate) * 100.0

                            transition_center = _transition_center(transition_idx, curr)
                            if transition_center is not None:
                                mask = centers >= transition_center
                                sens = sens[mask]
                                spec = spec[mask]

                            s_max, s_min = _max_min(sens - target_sens_pre)
                            p_max, p_min = _max_min(spec - target_spec_pre)

                            rows.append({
                                'AI': ai,
                                'Curr size': curr,
                                'Sens max': s_max,
                                'Sens min': s_min,
                                'Spec max': p_max,
                                'Spec min': p_min
                            })

                    df_plot = pd.DataFrame(rows)

                    fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
                    axes[0].set_title('Sensitivity (uncalibrated)')
                    axes[1].set_title('Specificity (uncalibrated)')

                    for ai in AI_COLUMNS:
                        df_ai = df_plot[df_plot['AI'] == ai]
                        if len(df_ai) == 0:
                            continue
                        axes[0].plot(df_ai['Curr size'], df_ai['Sens max'], label=f'{ai.upper()} max')
                        axes[0].plot(df_ai['Curr size'], df_ai['Sens min'], linestyle='--', label=f'{ai.upper()} min')
                        axes[1].plot(df_ai['Curr size'], df_ai['Spec max'], label=f'{ai.upper()} max')
                        axes[1].plot(df_ai['Curr size'], df_ai['Spec min'], linestyle='--', label=f'{ai.upper()} min')

                    for ax in axes:
                        ax.axhline(0, color='#333333', linewidth=1)
                        ax.set_xlabel('Current window size (exams)')
                        ax.set_ylabel('Deviation from target (pp)')
                        ax.grid(True, alpha=0.3)
                        ax.legend(fontsize=8)

                    manufacturer_label = NOISE_MANUFACTURER or 'All manufacturers'
                    scen_label = str(scenario.get('label',''))
                    fig.suptitle(f'Noise vs window size | {NOISE_HOSPITAL} — {manufacturer_label} | {scen_label}', fontsize=12, fontweight='bold')
                    plt.tight_layout()
                    plt.show()



In [ ]:
# Part 3: Root mean squared error deviation vs current window size (uncalibrated only)
# Uses same setup as the previous noise-vs-window-size cell.

if '_transition_center' not in globals():
    def _transition_center(transition_idx, curr_size):
        if transition_idx is None or curr_size is None:
            return None
        return transition_idx + (curr_size / 2)

if 'df_part3' not in globals():
    print('df_part3 not found. Run the Part 3 data load cell first.')
else:
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt

    # Reuse the same settings from the previous cell if present
    try:
        _ = NOISE_HOSPITAL
    except NameError:
        NOISE_HOSPITAL = 'Karolinska University Hospital'
    try:
        _ = NOISE_MANUFACTURER
    except NameError:
        NOISE_MANUFACTURER = None
    try:
        _ = NOISE_SCENARIO_LABEL
    except NameError:
        NOISE_SCENARIO_LABEL = 'single'
    try:
        _ = MIN_CURR
        _ = STEP_CURR
        _ = MAX_FRAC
    except NameError:
        MIN_CURR = 1500
        STEP_CURR = 500
        MAX_FRAC = 0.5

    # Pick scenario
    scenario = None
    for s in PART3_SCENARIOS:
        label = str(s.get('label','')).lower()
        if NOISE_SCENARIO_LABEL in label:
            scenario = s
            break
    if scenario is None:
        print('Scenario not found; check NOISE_SCENARIO_LABEL.')
    else:
        rad_mode = scenario.get('rad_mode')
        system_mode = scenario.get('system_mode')

        df_h = df_part3[df_part3[HOSPITAL_COL] == NOISE_HOSPITAL].copy()
        if len(df_h) == 0:
            print(f'No data for hospital: {NOISE_HOSPITAL}')
        else:
            if MANUFACTURER_COL in df_h.columns:
                if NOISE_MANUFACTURER is None:
                    manufacturers = sorted(df_h[MANUFACTURER_COL].dropna().unique())
                    NOISE_MANUFACTURER = manufacturers[0] if len(manufacturers) > 0 else None
                if NOISE_MANUFACTURER is not None:
                    df_h = df_h[df_h[MANUFACTURER_COL] == NOISE_MANUFACTURER].copy()

            if len(df_h) == 0:
                print(f'No data for hospital/manufacturer: {NOISE_HOSPITAL} / {NOISE_MANUFACTURER}')
            else:
                result = run_part3_hospital_pipeline(
                    df_h, NOISE_HOSPITAL, part3_start, calibration_strategy=PART3_CALIBRATION_STRATEGY,
                    rad_mode=rad_mode
                )
                if result is None:
                    print('Pipeline failed for selected hospital/manufacturer.')
                else:
                    df_h = order_part3_df(result['df_h'])
                    transition_idx = result['transition_index']
                    total_exams = len(df_h)
                    max_curr = max(int(total_exams * MAX_FRAC), MIN_CURR)

                    curr_sizes = list(range(MIN_CURR, max_curr + 1, STEP_CURR))

                    for ai in AI_COLUMNS:
                        base_flag = f'{ai}_flagged'
                        part3_flag = f'{ai}_flagged_part3'
                        if base_flag not in df_h.columns and part3_flag in df_h.columns:
                            df_h[base_flag] = df_h[part3_flag]

                    rad_mode_eff, calib_target_mode, system_mode_eff = resolve_part3_modes(rad_mode, None, system_mode)
                    target_sens_pre = result.get('radiologist_sensitivity_pre', {}).get(calib_target_mode, np.nan)
                    target_spec_pre = result.get('radiologist_specificity_pre', {}).get(calib_target_mode, np.nan)

                    def _cumsum(arr):
                        return np.concatenate(([0.0], np.cumsum(arr)))

                    def _window_sum(csum, starts, ends):
                        return csum[ends] - csum[starts]

                    def _window_rate(num_csum, den_csum, starts, ends, scale=100.0):
                        num = _window_sum(num_csum, starts, ends)
                        den = _window_sum(den_csum, starts, ends)
                        out = np.full_like(num, np.nan, dtype=float)
                        mask = den > 0
                        out[mask] = (num[mask] / den[mask]) * scale
                        return out

                    def _rms(values, target):
                        if target is None or pd.isna(target):
                            return np.nan
                        diffs = np.array(values, dtype=float) - target
                        diffs = diffs[~np.isnan(diffs)]
                        if len(diffs) == 0:
                            return np.nan
                        return float(np.sqrt(np.mean(diffs ** 2)))

                    # Precompute window indices per curr_size
                    window_index = {}
                    for curr in curr_sizes:
                        total_span = PART3_REF_SIZE + PART3_GAP_SIZE + curr
                        centers = []
                        starts = []
                        ends = []
                        for ref_start in range(0, len(df_h) - total_span + 1, PART3_STEP_SIZE):
                            ref_end = ref_start + PART3_REF_SIZE
                            curr_start = ref_end + PART3_GAP_SIZE
                            curr_end = curr_start + curr
                            curr_center = (curr_start + curr_end) / 2
                            centers.append(curr_center)
                            starts.append(curr_start)
                            ends.append(curr_end)
                        window_index[curr] = (
                            np.array(centers, dtype=float),
                            np.array(starts, dtype=int),
                            np.array(ends, dtype=int)
                        )

                    has_cancer = df_h['has_cancer'].fillna(0).astype(int).to_numpy()
                    has_control = (has_cancer == 0).astype(int)

                    rows = []
                    for ai in AI_COLUMNS:
                        flag_col = f'{ai}_flagged_part3'
                        if flag_col not in df_h.columns:
                            continue
                        flags = df_h[flag_col].fillna(0).astype(int).to_numpy()

                        c_flag_cancer = _cumsum(flags * has_cancer)
                        c_cancer = _cumsum(has_cancer)
                        c_flag_control = _cumsum(flags * has_control)
                        c_control = _cumsum(has_control)

                        for curr in curr_sizes:
                            centers, starts, ends = window_index[curr]
                            sens = _window_rate(c_flag_cancer, c_cancer, starts, ends)
                            flagged_rate = _window_rate(c_flag_control, c_control, starts, ends, scale=1.0)
                            spec = (1.0 - flagged_rate) * 100.0

                            transition_center = _transition_center(transition_idx, curr)
                            if transition_center is not None:
                                mask = centers >= transition_center
                                sens = sens[mask]
                                spec = spec[mask]

                            s_rms = _rms(sens, target_sens_pre)
                            p_rms = _rms(spec, target_spec_pre)

                            rows.append({
                                'AI': ai,
                                'Curr size': curr,
                                'Sens Root mean squared error': s_rms,
                                'Spec Root mean squared error': p_rms
                            })

                    df_plot = pd.DataFrame(rows)

                    fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
                    axes[0].set_title('Sensitivity Root mean squared error (uncalibrated)')
                    axes[1].set_title('Specificity Root mean squared error (uncalibrated)')

                    for ai in AI_COLUMNS:
                        df_ai = df_plot[df_plot['AI'] == ai]
                        if len(df_ai) == 0:
                            continue
                        axes[0].plot(df_ai['Curr size'], df_ai['Sens Root mean squared error'], label=f'{ai.upper()}')
                        axes[1].plot(df_ai['Curr size'], df_ai['Spec Root mean squared error'], label=f'{ai.upper()}')

                    for ax in axes:
                        ax.set_xlabel('Current window size (exams)')
                        ax.set_ylabel('Root mean squared error deviation from target (pp)')
                        ax.grid(True, alpha=0.3)
                        ax.legend(fontsize=8)

                    manufacturer_label = NOISE_MANUFACTURER or 'All manufacturers'
                    scen_label = str(scenario.get('label',''))
                    fig.suptitle(f'Root mean squared error vs window size | {NOISE_HOSPITAL} — {manufacturer_label} | {scen_label}', fontsize=12, fontweight='bold')
                    plt.tight_layout()
                    plt.show()


## Part 3B: RMS vs window size for additional signals

RMS vs window size for multiple detection metrics and AI score signals, using a large-window ground truth.


**What the RMS axis means**\nFor each current window size, we compute many windowed estimates of the metric.\nGround truth is defined as the mean metric value from large windows (20k–40k exams).\nThe y-axis shows the root-mean-square (RMS) deviation of those windowed estimates from that ground truth, in percentage points (pp). Lower is less noise.


In [ ]:
# Part 3B: RMS vs window size for multiple signals (flags + scores)
# Uses the same Part 3 pipeline and selection logic as the RMS plot above.

RMS_SIGNAL_HOSPITAL = NOISE_HOSPITAL
RMS_SIGNAL_MANUFACTURER = NOISE_MANUFACTURER
RMS_SIGNAL_SCENARIO_LABEL = NOISE_SCENARIO_LABEL

# Finer window grid (more windows)
RMS_SIGNAL_MIN_CURR = MIN_CURR
RMS_SIGNAL_STEP_CURR = 250  # smaller step = more windows (slower)
RMS_SIGNAL_MAX_FRAC = MAX_FRAC

GROUND_TRUTH_MIN = 20000
GROUND_TRUTH_MAX = 40000
INCLUDE_SCORE_METRIC = False
INCLUDE_OVERLAP_METRICS = True

# Normalize AI score to compare across models
SCORE_NORMALIZE = 'zscore'  # 'zscore' or None
SCORE_NORM_SCOPE = 'prestart'  # 'prestart' or 'full'

if '_transition_center' not in globals():
    def _transition_center(transition_idx, curr_size):
        if transition_idx is None or curr_size is None:
            return None
        return transition_idx + (curr_size / 2)

if 'df_part3' not in globals():
    print('df_part3 not found. Run the Part 3 data load cell first.')
else:
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import re
    from IPython.display import display

    # Pick scenario
    scenario = None
    for s in PART3_SCENARIOS:
        label = str(s.get('label','')).lower()
        if RMS_SIGNAL_SCENARIO_LABEL in label:
            scenario = s
            break

    if scenario is None:
        print('Scenario not found; check RMS_SIGNAL_SCENARIO_LABEL.')
    else:
        rad_mode = scenario.get('rad_mode')
        system_mode = scenario.get('system_mode')

        df_h = df_part3[df_part3[HOSPITAL_COL] == RMS_SIGNAL_HOSPITAL].copy()
        if len(df_h) == 0:
            print(f'No data for hospital: {RMS_SIGNAL_HOSPITAL}')
        else:
            if MANUFACTURER_COL in df_h.columns:
                if RMS_SIGNAL_MANUFACTURER is None:
                    manufacturers = sorted(df_h[MANUFACTURER_COL].dropna().unique())
                    RMS_SIGNAL_MANUFACTURER = manufacturers[0] if len(manufacturers) > 0 else None
                if RMS_SIGNAL_MANUFACTURER is not None:
                    df_h = df_h[df_h[MANUFACTURER_COL] == RMS_SIGNAL_MANUFACTURER].copy()

            if len(df_h) == 0:
                print(f'No data for hospital/manufacturer: {RMS_SIGNAL_HOSPITAL} / {RMS_SIGNAL_MANUFACTURER}')
            else:
                result = run_part3_hospital_pipeline(
                    df_h, RMS_SIGNAL_HOSPITAL, part3_start,
                    calibration_strategy=PART3_CALIBRATION_STRATEGY,
                    rad_mode=rad_mode
                )
                if result is None:
                    print('Pipeline failed for selected hospital/manufacturer.')
                else:
                    df_h = order_part3_df(result['df_h'])
                    transition_idx = result['transition_index']

                    total_exams = len(df_h)
                    max_curr = max(int(total_exams * RMS_SIGNAL_MAX_FRAC), RMS_SIGNAL_MIN_CURR)
                    curr_sizes = list(range(RMS_SIGNAL_MIN_CURR, max_curr + 1, RMS_SIGNAL_STEP_CURR))

                    # Ensure flagged columns exist
                    for ai in AI_COLUMNS:
                        base_flag = f'{ai}_flagged'
                        part3_flag = f'{ai}_flagged_part3'
                        if part3_flag not in df_h.columns and base_flag in df_h.columns:
                            df_h[part3_flag] = df_h[base_flag]

                    # Ensure radiologist flags exist (needed for overlap metrics)
                    if 'rad_flagged' not in df_h.columns:
                        if 'compute_radiologist_flag' in globals():
                            df_h['rad_flagged'] = compute_radiologist_flag(df_h, mode=rad_mode)
                        else:
                            df_h['rad_flagged'] = np.nan

                    # Optional score normalization (z-score per AI)
                    score_norm_cols = {}
                    if INCLUDE_SCORE_METRIC and SCORE_NORMALIZE == 'zscore':
                        if SCORE_NORM_SCOPE == 'prestart' and transition_idx is not None:
                            df_base = df_h.iloc[:transition_idx]
                        else:
                            df_base = df_h
                        for ai in AI_COLUMNS:
                            if ai not in df_h.columns:
                                continue
                            mu = df_base[ai].mean()
                            sigma = df_base[ai].std()
                            norm_col = f'{ai}_score_norm'
                            if pd.isna(sigma) or sigma == 0:
                                df_h[norm_col] = np.nan
                            else:
                                df_h[norm_col] = (df_h[ai] - mu) / sigma
                            score_norm_cols[ai] = norm_col

                    # Precompute window indices per curr_size
                    window_index = {}
                    for curr in curr_sizes:
                        total_span = PART3_REF_SIZE + PART3_GAP_SIZE + curr
                        centers = []
                        starts = []
                        ends = []
                        for ref_start in range(0, len(df_h) - total_span + 1, PART3_STEP_SIZE):
                            ref_end = ref_start + PART3_REF_SIZE
                            curr_start = ref_end + PART3_GAP_SIZE
                            curr_end = curr_start + curr
                            curr_center = (curr_start + curr_end) / 2
                            centers.append(curr_center)
                            starts.append(curr_start)
                            ends.append(curr_end)
                        window_index[curr] = (
                            np.array(centers, dtype=float),
                            np.array(starts, dtype=int),
                            np.array(ends, dtype=int)
                        )

                    def _cumsum(arr):
                        return np.concatenate(([0.0], np.cumsum(arr)))

                    def _window_sum(csum, starts, ends):
                        return csum[ends] - csum[starts]

                    def _window_rate(num_csum, den_csum, starts, ends, scale=100.0):
                        num = _window_sum(num_csum, starts, ends)
                        den = _window_sum(den_csum, starts, ends)
                        out = np.full_like(num, np.nan, dtype=float)
                        mask = den > 0
                        out[mask] = (num[mask] / den[mask]) * scale
                        return out

                    def _filtered(values, centers, center_min=None):
                        if center_min is None:
                            return values
                        return values[centers >= center_min]

                    def _rms(values, target):
                        if target is None or pd.isna(target):
                            return np.nan
                        diffs = np.array(values, dtype=float) - target
                        diffs = diffs[~np.isnan(diffs)]
                        if len(diffs) == 0:
                            return np.nan
                        return float(np.sqrt(np.mean(diffs ** 2)))

                    score_unit_label = 'score units'
                    if INCLUDE_SCORE_METRIC and SCORE_NORMALIZE == 'zscore':
                        score_unit_label = 'z-score units'

                    # Define metric list based on available columns
                    metric_specs = [
                        ('sensitivity', 'Sensitivity', 'pp'),
                        ('specificity', 'Specificity', 'pp'),
                        ('flagging_rate', 'Flagging rate', 'pp')
                    ]
                    if 'time_based_screen' in df_h.columns:
                        metric_specs.append(('screen_flagging_rate', 'Screen-detected flagging rate', 'pp'))
                    if INCLUDE_OVERLAP_METRICS:
                        metric_specs.extend([
                            ('p_ai_given_rad', 'P(AI flagged | Rad flagged)', 'pp'),
                            ('p_rad_given_ai', 'P(Rad flagged | AI flagged)', 'pp'),
                            ('relative_flagging_rate', 'Relative flagging rate (AI/Rad)', 'ratio')
                        ])
                    if INCLUDE_SCORE_METRIC:
                        metric_specs.append(('score_mean', 'Mean AI score', score_unit_label))

                    # Ensure score metric is excluded
                    metric_specs = [m for m in metric_specs if m[0] != "score_mean"]

                    gt_sizes = [s for s in curr_sizes if GROUND_TRUTH_MIN <= s <= GROUND_TRUTH_MAX]
                    if len(gt_sizes) == 0:
                        gt_sizes = [max(curr_sizes)]
                        print('No curr_sizes in ground-truth range; using max curr size:', gt_sizes[0])

                    rows = []
                    targets = {}

                    # Reset cache to avoid stale metrics from prior runs
                    _part3b_values_cache = {}

                    # Vectorized per-AI precomputation
                    has_cancer = df_h['has_cancer'].fillna(0).astype(int).to_numpy()
                    has_control = (has_cancer == 0).astype(int)
                    has_screen = None
                    if 'time_based_screen' in df_h.columns:
                        has_screen = df_h['time_based_screen'].fillna(0).astype(int).to_numpy()

                    rad_flags = None
                    if 'rad_flagged' in df_h.columns:
                        rad_flags = df_h['rad_flagged'].fillna(0).astype(int).to_numpy()
                        c_rad = _cumsum(rad_flags)
                    else:
                        c_rad = None

                    for ai in AI_COLUMNS:
                        flag_col = f'{ai}_flagged_part3'
                        if flag_col not in df_h.columns:
                            continue
                        flags = df_h[flag_col].fillna(0).astype(int).to_numpy()

                        # Cumsums for rates
                        c_flag_cancer = _cumsum(flags * has_cancer)
                        c_cancer = _cumsum(has_cancer)
                        c_flag_control = _cumsum(flags * has_control)
                        c_control = _cumsum(has_control)
                        c_flag_all = _cumsum(flags)
                        c_all = _cumsum(np.ones_like(flags, dtype=float))

                        if has_screen is not None:
                            c_flag_screen = _cumsum(flags * has_screen)
                            c_screen = _cumsum(has_screen)
                        else:
                            c_flag_screen = None
                            c_screen = None

                        if rad_flags is not None:
                            c_both = _cumsum(flags * rad_flags)
                        else:
                            c_both = None

                        # Score cumsum
                        if INCLUDE_SCORE_METRIC:
                            if SCORE_NORMALIZE == 'zscore':
                                norm_col = score_norm_cols.get(ai)
                                if norm_col and norm_col in df_h.columns:
                                    score_arr = df_h[norm_col].to_numpy(dtype=float)
                                else:
                                    score_arr = None
                            else:
                                score_arr = df_h[ai].to_numpy(dtype=float) if ai in df_h.columns else None
                            c_score = _cumsum(score_arr) if score_arr is not None else None

                        for metric, _, _ in metric_specs:
                            for curr in curr_sizes:
                                centers, starts, ends = window_index[curr]
                                if metric == 'sensitivity':
                                    vals = _window_rate(c_flag_cancer, c_cancer, starts, ends)
                                elif metric == 'specificity':
                                    # specificity = 1 - flagged rate among controls
                                    flagged_rate = _window_rate(c_flag_control, c_control, starts, ends, scale=1.0)
                                    vals = (1.0 - flagged_rate) * 100.0
                                elif metric == 'flagging_rate':
                                    vals = _window_rate(c_flag_all, c_all, starts, ends)
                                elif metric == 'screen_flagging_rate':
                                    if c_flag_screen is None:
                                        vals = np.full_like(centers, np.nan, dtype=float)
                                    else:
                                        vals = _window_rate(c_flag_screen, c_screen, starts, ends)
                                elif metric == 'p_ai_given_rad':
                                    if c_both is None or c_rad is None:
                                        vals = np.full_like(centers, np.nan, dtype=float)
                                    else:
                                        vals = _window_rate(c_both, c_rad, starts, ends)
                                elif metric == 'p_rad_given_ai':
                                    if c_both is None:
                                        vals = np.full_like(centers, np.nan, dtype=float)
                                    else:
                                        vals = _window_rate(c_both, c_flag_all, starts, ends)
                                elif metric == 'relative_flagging_rate':
                                    if c_rad is None:
                                        vals = np.full_like(centers, np.nan, dtype=float)
                                    else:
                                        ai_rate = _window_rate(c_flag_all, c_all, starts, ends, scale=1.0)
                                        rad_rate = _window_rate(c_rad, c_all, starts, ends, scale=1.0)
                                        vals = np.full_like(ai_rate, np.nan, dtype=float)
                                        mask = rad_rate > 0
                                        vals[mask] = ai_rate[mask] / rad_rate[mask]
                                elif metric == 'score_mean':
                                    if c_score is None:
                                        vals = np.full_like(centers, np.nan, dtype=float)
                                    else:
                                        sums = _window_sum(c_score, starts, ends)
                                        counts = _window_sum(c_all, starts, ends)
                                        vals = np.full_like(sums, np.nan, dtype=float)
                                        mask = counts > 0
                                        vals[mask] = sums[mask] / counts[mask]
                                else:
                                    vals = np.full_like(centers, np.nan, dtype=float)

                                # Cache values per metric/ai/curr for GT + RMS use
                                key = (metric, ai, curr)
                                _part3b_values_cache[key] = (centers, vals)

                    # Compute RMS vs window size for each metric
                    for metric, _, _ in metric_specs:
                        for ai in AI_COLUMNS:
                            # Ground truth = mean of large-window estimates
                            gt_vals = []
                            for gsize in gt_sizes:
                                centers, vals = _part3b_values_cache.get((metric, ai, gsize), (None, None))
                                if centers is None:
                                    continue
                                gt_center_min = _transition_center(transition_idx, gsize)
                                gt_vals.extend([v for v in _filtered(vals, centers, gt_center_min) if not pd.isna(v)])
                            target = float(np.mean(gt_vals)) if len(gt_vals) > 0 else np.nan
                            targets[(metric, ai)] = target

                            for curr in curr_sizes:
                                centers, vals = _part3b_values_cache.get((metric, ai, curr), (None, None))
                                if centers is None:
                                    continue
                                curr_center_min = _transition_center(transition_idx, curr)
                                vals_f = _filtered(vals, centers, curr_center_min)
                                rms_val = _rms(vals_f, target)
                                rows.append({
                                    'metric': metric,
                                    'ai_system': ai,
                                    'curr_size': curr,
                                    'rms': rms_val
                                })

                    df_rms = pd.DataFrame(rows)

                    # Plot (one figure per metric)
                    for metric, label, unit_label in metric_specs:
                        label_clean = re.sub(r"\s*\(.*\)$", "", label)
                        fig, ax = plt.subplots(1, 1, figsize=(7, 4.5))
                        for ai in AI_COLUMNS:
                            df_ai = df_rms[(df_rms['metric'] == metric) & (df_rms['ai_system'] == ai)]
                            if len(df_ai) == 0:
                                continue
                            ax.plot(df_ai['curr_size'], df_ai['rms'], label=f'{ai.upper()}')
                        ax.set_title(label_clean)
                        ax.set_xlabel('Current window size (exams)')
                        if unit_label:
                            ax.set_ylabel(f'RMS deviation from ground truth ({unit_label})')
                        else:
                            ax.set_ylabel('RMS deviation from ground truth')
                        ax.grid(True, alpha=0.3)
                        if GROUND_TRUTH_MIN is not None and GROUND_TRUTH_MAX is not None:
                            ax.axvspan(GROUND_TRUTH_MIN, min(GROUND_TRUTH_MAX, max_curr), color='gray', alpha=0.1)
                        ax.legend(fontsize=8)
                        manufacturer_label = RMS_SIGNAL_MANUFACTURER or 'All manufacturers'
                        scen_label = str(scenario.get('label',''))
                        fig.suptitle(
                            f'RMS vs window size | {RMS_SIGNAL_HOSPITAL} - {manufacturer_label} | {scen_label}',
                            fontsize=12, fontweight='bold'
                        )
                        plt.tight_layout()
                        plt.show()
                    # Display ground-truth targets
                    gt_rows = [
                        {'metric': m, 'ai_system': a, 'ground_truth': t}
                        for (m, a), t in targets.items()
                    ]
                    display(pd.DataFrame(gt_rows))


In [ ]:
# Part 3B: Knee detection on RMS curves (Karolinska plots)
# Requires df_rms and metric_specs from the RMS-vs-window cell above.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

if 'df_rms' not in globals() or 'metric_specs' not in globals():
    print('df_rms or metric_specs not found. Run the RMS-vs-window cell first.')
else:
    def _knee_point(x, y):
        x = np.asarray(x, dtype=float)
        y = np.asarray(y, dtype=float)
        mask = (~np.isnan(x)) & (~np.isnan(y))
        x = x[mask]
        y = y[mask]
        if len(x) < 3:
            return np.nan, np.nan, np.nan
        order = np.argsort(x)
        x = x[order]
        y = y[order]

        x_min, x_max = x.min(), x.max()
        y_min, y_max = y.min(), y.max()
        x_rng = x_max - x_min
        y_rng = y_max - y_min
        if x_rng == 0:
            return np.nan, np.nan, np.nan
        # Normalize to [0,1] for scale invariance
        x_n = (x - x_min) / x_rng
        if y_rng == 0:
            y_n = np.zeros_like(y)
        else:
            y_n = (y - y_min) / y_rng

        # Line from first to last point
        x0, y0 = x_n[0], y_n[0]
        x1, y1 = x_n[-1], y_n[-1]
        denom = np.hypot(y1 - y0, x1 - x0)
        if denom == 0:
            return np.nan, np.nan, np.nan

        # Perpendicular distance from each point to the line
        # Distance formula for line through (x0,y0)-(x1,y1)
        dist = np.abs((y1 - y0) * x_n - (x1 - x0) * y_n + x1*y0 - y1*x0) / denom
        idx = int(np.argmax(dist))
        return x[idx], y[idx], dist[idx]

    # Compute knee per metric/AI
    rows = []
    knee_lookup = {}
    for item in metric_specs:
        if len(item) == 3:
            metric, label, _unit = item
        else:
            metric, label = item
            _unit = None
        for ai in AI_COLUMNS:
            df_ai = df_rms[(df_rms['metric'] == metric) & (df_rms['ai_system'] == ai)]
            if len(df_ai) == 0:
                continue
            kx, ky, kd = _knee_point(df_ai['curr_size'].values, df_ai['rms'].values)
            rows.append({
                'metric': metric,
                'ai_system': ai,
                'knee_curr_size': kx,
                'knee_rms': ky,
                'knee_distance': kd
            })
            knee_lookup[(metric, ai)] = (kx, ky)

    knee_df = pd.DataFrame(rows)
    display(knee_df)

    # Plot RMS curves with knee markers (Karolinska scope from df_rms)
    for item in metric_specs:
        if len(item) == 3:
            metric, label, _unit = item
        else:
            metric, label = item
            _unit = None
        fig, ax = plt.subplots(1, 1, figsize=(7, 4.5))
        for ai in AI_COLUMNS:
            df_ai = df_rms[(df_rms['metric'] == metric) & (df_rms['ai_system'] == ai)]
            if len(df_ai) == 0:
                continue
            ax.plot(df_ai['curr_size'], df_ai['rms'], label=f'{ai.upper()}')
            kx, ky = knee_lookup.get((metric, ai), (np.nan, np.nan))
            if pd.notna(kx) and pd.notna(ky):
                ax.scatter([kx], [ky], s=35)
                ax.axvline(kx, linestyle='--', alpha=0.25)

        label_clean = label
        ax.set_title(label_clean)
        ax.set_xlabel('Current window size (exams)')
        ax.set_ylabel('RMS deviation from ground truth (pp)')
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=8)
        plt.tight_layout()
        plt.show()



In [ ]:
# Part 3: Uncalibrated sensitivity over time for multiple current window sizes
# Plots for all hospitals and each manufacturer, one figure per AI and window size.
'''
COMPARE_SCENARIO_LABEL = 'single'  # 'single' or 'double'
COMPARE_CURR_SIZES = [5000, 10000, 20000]
COMPARE_STEP_SIZE = 500
EXCLUDE_DANDERYDS = True

if 'df_part3' not in globals():
    print('df_part3 not found. Run the Part 3 data load cell first.')
else:
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt

    # Pick scenario
    scenario = None
    for s in PART3_SCENARIOS:
        label = str(s.get('label','')).lower()
        if COMPARE_SCENARIO_LABEL in label:
            scenario = s
            break
    if scenario is None:
        print('Scenario not found; check COMPARE_SCENARIO_LABEL.')
    else:
        rad_mode = scenario.get('rad_mode')
        system_mode = scenario.get('system_mode')

        colors = {'lunit': '#ff7f0e', 'therapixel': '#2ca02c', 'vara': '#d62728'}

        hospitals = sorted(df_part3[HOSPITAL_COL].dropna().unique())
        for hospital in hospitals:
            if EXCLUDE_DANDERYDS and 'danderyd' in str(hospital).lower():
                continue
            df_hospital = df_part3[df_part3[HOSPITAL_COL] == hospital].copy()
            if MANUFACTURER_COL in df_hospital.columns:
                manufacturers = sorted(df_hospital[MANUFACTURER_COL].dropna().unique())
                if len(manufacturers) == 0:
                    manufacturers = ['Unknown']
            else:
                manufacturers = ['Unknown']

            for manufacturer in manufacturers:
                if manufacturer == 'Unknown':
                    df_subset = df_hospital.copy()
                else:
                    df_subset = df_hospital[df_hospital[MANUFACTURER_COL] == manufacturer].copy()
                if len(df_subset) == 0:
                    continue

                result = run_part3_hospital_pipeline(
                    df_subset, hospital, part3_start, calibration_strategy=PART3_CALIBRATION_STRATEGY,
                    rad_mode=rad_mode
                )
                if result is None:
                    continue

                df_h = order_part3_df(result['df_h'])
                transition_idx = result['transition_index']

                # Ensure base flags exist
                for ai in AI_COLUMNS:
                    base_flag = f'{ai}_flagged'
                    part3_flag = f'{ai}_flagged_part3'
                    if base_flag not in df_h.columns and part3_flag in df_h.columns:
                        df_h[base_flag] = df_h[part3_flag]

                for ai in AI_COLUMNS:
                    for curr in COMPARE_CURR_SIZES:
                        sens_base = compute_rate_windows_by_flag(
                            df_h, flag_suffix='part3', rate_kind='sensitivity',
                            ref_size=PART3_REF_SIZE, gap_size=PART3_GAP_SIZE,
                            curr_size=curr, step_size=COMPARE_STEP_SIZE
                        )
                        sens_ai = sens_base[sens_base['ai_system'] == ai]
                        if len(sens_ai) == 0:
                            continue
                        transition_center = _transition_center(transition_idx, curr)

                        fig, ax = plt.subplots(figsize=(12, 4))
                        ax.plot(sens_ai['curr_center'], sens_ai['curr_rate'], color=colors[ai])

                        x_min = sens_ai['curr_center'].min()
                        x_max = sens_ai['curr_center'].max()
                        ax.axvspan(x_min, transition_center, alpha=0.1, color='blue', label='Pre-start')
                        ax.axvspan(transition_center, x_max, alpha=0.1, color='orange', label='Post-start')
                        ax.axvline(transition_center, color='red', linestyle='--', linewidth=2, label='Start')

                        ax.set_ylabel('Sensitivity (%)')
                        ax.set_xlabel('Current Window Center (Exam Index)')
                        ax.grid(True, alpha=0.3)
                        ax.legend(fontsize=8, loc='upper left')

                        scen_label = str(scenario.get('label',''))
                        plt.title(f'Uncalibrated sensitivity | {hospital} — {manufacturer} | {ai.upper()} | curr={curr} | step={COMPARE_STEP_SIZE}\n{scen_label}', fontsize=11, fontweight='bold')
                        plt.tight_layout()
                        plt.show()
'''


In [ ]:
# Part 3: Grid search plots (current size x step size) for uncalibrated sensitivity
# One plot per hospital/manufacturer/AI for each (curr, step) combination.

GRID_CURR_SIZES = [5000, 10000, 20000]
GRID_STEP_SIZES = [500, 1000, 2000]
GRID_SCENARIO_LABEL = 'single'  # 'single' or 'double'
EXCLUDE_DANDERYDS = True

if 'df_part3' not in globals():
    print('df_part3 not found. Run the Part 3 data load cell first.')
else:
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt

    # Pick scenario
    scenario = None
    for s in PART3_SCENARIOS:
        label = str(s.get('label','')).lower()
        if GRID_SCENARIO_LABEL in label:
            scenario = s
            break
    if scenario is None:
        print('Scenario not found; check GRID_SCENARIO_LABEL.')
    else:
        rad_mode = scenario.get('rad_mode')
        system_mode = scenario.get('system_mode')

        colors = {'lunit': '#ff7f0e', 'therapixel': '#2ca02c', 'vara': '#d62728'}

        hospitals = sorted(df_part3[HOSPITAL_COL].dropna().unique())
        for hospital in hospitals:
            if EXCLUDE_DANDERYDS and 'danderyd' in str(hospital).lower():
                continue
            df_hospital = df_part3[df_part3[HOSPITAL_COL] == hospital].copy()
            if MANUFACTURER_COL in df_hospital.columns:
                manufacturers = sorted(df_hospital[MANUFACTURER_COL].dropna().unique())
                if len(manufacturers) == 0:
                    manufacturers = ['Unknown']
            else:
                manufacturers = ['Unknown']

            for manufacturer in manufacturers:
                if manufacturer == 'Unknown':
                    df_subset = df_hospital.copy()
                else:
                    df_subset = df_hospital[df_hospital[MANUFACTURER_COL] == manufacturer].copy()
                if len(df_subset) == 0:
                    continue

                result = run_part3_hospital_pipeline(
                    df_subset, hospital, part3_start, calibration_strategy=PART3_CALIBRATION_STRATEGY,
                    rad_mode=rad_mode
                )
                if result is None:
                    continue

                df_h = order_part3_df(result['df_h'])
                transition_idx = result['transition_index']

                # Ensure base flags exist
                for ai in AI_COLUMNS:
                    base_flag = f'{ai}_flagged'
                    part3_flag = f'{ai}_flagged_part3'
                    if base_flag not in df_h.columns and part3_flag in df_h.columns:
                        df_h[base_flag] = df_h[part3_flag]

                for ai in AI_COLUMNS:
                    for curr in GRID_CURR_SIZES:
                        for step in GRID_STEP_SIZES:
                            sens_base = compute_rate_windows_by_flag(
                                df_h, flag_suffix='part3', rate_kind='sensitivity',
                                ref_size=PART3_REF_SIZE, gap_size=PART3_GAP_SIZE,
                                curr_size=curr, step_size=step
                            )
                            sens_ai = sens_base[sens_base['ai_system'] == ai]
                            if len(sens_ai) == 0:
                                continue
                            transition_center = _transition_center(transition_idx, curr)

                            fig, ax = plt.subplots(figsize=(12, 4))
                            ax.plot(sens_ai['curr_center'], sens_ai['curr_rate'], color=colors[ai])

                            x_min = sens_ai['curr_center'].min()
                            x_max = sens_ai['curr_center'].max()
                            ax.axvspan(x_min, transition_center, alpha=0.1, color='blue', label='Pre-start')
                            ax.axvspan(transition_center, x_max, alpha=0.1, color='orange', label='Post-start')
                            ax.axvline(transition_center, color='red', linestyle='--', linewidth=2, label='Start')

                            ax.set_ylabel('Sensitivity (%)')
                            ax.set_xlabel('Current Window Center (Exam Index)')
                            ax.grid(True, alpha=0.3)
                            ax.legend(fontsize=8, loc='upper left')

                            scen_label = str(scenario.get('label',''))
                            plt.title(f'Grid search | {hospital} — {manufacturer} | {ai.upper()} | curr={curr} | step={step}\n{scen_label}', fontsize=11, fontweight='bold')
                            plt.tight_layout()
                            plt.show()


In [ ]:
# Part 3: Relative delay vs baseline signal (per metric)
# Baseline = first detected signal for each metric at BASE_CURR_SIZE/BASE_STEP_SIZE.
# For each (curr, step), compute delay to first signal and count signals before baseline.

if 'PART3_RECAL_ALPHA' not in globals():
    PART3_RECAL_ALPHA = 0.05
if 'PART3_RECAL_USE_SCORE_SIGNALS' not in globals():
    PART3_RECAL_USE_SCORE_SIGNALS = False
if 'PART3_RECAL_WINDOW_SIZE' not in globals() and 'PART3_REF_SIZE' in globals():
    PART3_RECAL_WINDOW_SIZE = PART3_REF_SIZE * 2

BASE_CURR_SIZE = 5000
BASE_STEP_SIZE = 500
GRID_CURR_SIZES = [5000, 10000, 20000]
GRID_STEP_SIZES = [500, 1000, 2000]
GRID_METRICS = ['flagging_rate', 'relative_flagging_rate', 'p_ai_given_rad', 'p_rad_given_ai']
GRID_SCENARIO_LABEL = 'single'  # 'single' or 'double'
EXCLUDE_DANDERYDS = True

if 'df_part3' not in globals():
    print('df_part3 not found. Run the Part 3 data load cell first.')
else:
    import numpy as np
    import pandas as pd

    # Pick scenario
    scenario = None
    for s in PART3_SCENARIOS:
        label = str(s.get('label','')).lower()
        if GRID_SCENARIO_LABEL in label:
            scenario = s
            break
    if scenario is None:
        print('Scenario not found; check GRID_SCENARIO_LABEL.')
    else:
        rad_mode = scenario.get('rad_mode')

        rows = []
        hospitals = sorted(df_part3[HOSPITAL_COL].dropna().unique())
        for hospital in hospitals:
            if EXCLUDE_DANDERYDS and 'danderyd' in str(hospital).lower():
                continue
            df_hospital = df_part3[df_part3[HOSPITAL_COL] == hospital].copy()
            if MANUFACTURER_COL in df_hospital.columns:
                manufacturers = sorted(df_hospital[MANUFACTURER_COL].dropna().unique())
                if len(manufacturers) == 0:
                    manufacturers = ['Unknown']
            else:
                manufacturers = ['Unknown']

            for manufacturer in manufacturers:
                if manufacturer == 'Unknown':
                    df_subset = df_hospital.copy()
                else:
                    df_subset = df_hospital[df_hospital[MANUFACTURER_COL] == manufacturer].copy()
                if len(df_subset) == 0:
                    continue

                result = run_part3_hospital_pipeline(
                    df_subset, hospital, part3_start, calibration_strategy=PART3_CALIBRATION_STRATEGY,
                    rad_mode=rad_mode
                )
                if result is None:
                    continue

                df_h = order_part3_df(result['df_h'])
                transition_idx = result['transition_index']

                # Baseline signals with base params
                baseline_centers = collect_part2_signal_centers(
                    df_h, alpha=PART3_RECAL_ALPHA,
                    ref_size=PART3_REF_SIZE, gap_size=PART3_GAP_SIZE,
                    curr_size=BASE_CURR_SIZE, step_size=BASE_STEP_SIZE,
                    use_score_signals=PART3_RECAL_USE_SCORE_SIGNALS
                )

                for ai in AI_COLUMNS:
                    for metric_key in GRID_METRICS:
                        signal_key = metric_key if metric_key != 'relative_flagging_rate' else 'flagging_rate'
                        base_centers = baseline_centers.get(ai, {}).get(signal_key, [])
                        if not base_centers:
                            base_first = np.nan
                        else:
                            base_first = min([c for c in base_centers if c is not None])

                        for curr in GRID_CURR_SIZES:
                            for step in GRID_STEP_SIZES:
                                centers = collect_part2_signal_centers(
                                    df_h, alpha=PART3_RECAL_ALPHA,
                                    ref_size=PART3_REF_SIZE, gap_size=PART3_GAP_SIZE,
                                    curr_size=curr, step_size=step,
                                    use_score_signals=PART3_RECAL_USE_SCORE_SIGNALS
                                )
                                sig_centers = centers.get(ai, {}).get(signal_key, [])
                                sig_centers = [c for c in sig_centers if c is not None]
                                if len(sig_centers) == 0:
                                    first_sig = np.nan
                                    n_before = 0
                                else:
                                    first_sig = min(sig_centers)
                                    if pd.isna(base_first):
                                        n_before = 0
                                    else:
                                        n_before = sum(1 for c in sig_centers if c < base_first)

                                delay = np.nan
                                if pd.notna(first_sig) and pd.notna(base_first):
                                    delay = float(first_sig - base_first)

                                rows.append({
                                    'Scenario': str(scenario.get('label','')),
                                    'Hospital': hospital,
                                    'Manufacturer': manufacturer,
                                    'AI': ai,
                                    'Metric': metric_key,
                                    'Curr size': curr,
                                    'Step size': step,
                                    'Baseline first signal': base_first,
                                    'First signal': first_sig,
                                    'Delay (exams)': delay,
                                    'Signals before baseline': n_before
                                })

        delay_df = pd.DataFrame(rows)
        if len(delay_df) == 0:
            print('No results. Check data or filters.')
        else:
            display(delay_df)
            summary = (
                delay_df.groupby(['Curr size', 'Step size', 'Metric'])
                .agg(
                    mean_delay=('Delay (exams)', 'mean'),
                    median_delay=('Delay (exams)', 'median'),
                    mean_false_pos=('Signals before baseline', 'mean')
                )
                .reset_index()
            )
            display(summary)


In [ ]:
# Part 3: Image of delay table (split by hospital/manufacturer)
from pathlib import Path

output_dir = Path('outputs/2025_12_29_counterfactual_manufacturer_swap_setup')
output_dir.mkdir(parents=True, exist_ok=True)

if 'delay_df' not in globals() or delay_df is None or len(delay_df) == 0:
    print('delay_df not found or empty. Run the delay grid cell first.')
else:
    def _format_df(df):
        df = df.copy()
        for col in ['Baseline first signal', 'First signal', 'Delay (exams)']:
            if col in df.columns:
                df[col] = df[col].map(lambda x: '' if pd.isna(x) else f"{x:.0f}")
        if 'Signals before baseline' in df.columns:
            df['Signals before baseline'] = df['Signals before baseline'].map(lambda x: '' if pd.isna(x) else f"{int(x)}")
        return df

    for (hospital, manufacturer), df_sub in delay_df.groupby(['Hospital', 'Manufacturer']):
        display_df = _format_df(df_sub)
        fig_height = 2.0 + 0.35 * len(display_df)
        fig_width = max(18, 1.2 * len(display_df.columns) + 6)
        fig, ax = plt.subplots(figsize=(fig_width, fig_height))
        ax.axis('off')

        table = ax.table(
            cellText=display_df.values,
            colLabels=display_df.columns,
            cellLoc='center',
            loc='center',
            bbox=[0, 0, 1, 1]
        )

        table.auto_set_font_size(False)
        table.set_fontsize(9)
        table.scale(1, 1.4)

        header_color = '#4472C4'
        alt_row_color = '#E7E6E6'

        for i in range(len(display_df.columns)):
            cell = table[(0, i)]
            cell.set_facecolor(header_color)
            cell.set_text_props(weight='bold', color='white')

        for i in range(1, len(display_df) + 1):
            for j in range(len(display_df.columns)):
                cell = table[(i, j)]
                cell.set_facecolor(alt_row_color if i % 2 == 0 else 'white')

        for cell in table.get_celld().values():
            cell.set_edgecolor('black')
            cell.set_linewidth(1.0)

        plt.title(f'Signal delay vs baseline | {hospital} — {manufacturer}', fontsize=13, fontweight='bold', pad=16)
        plt.tight_layout()
        filename = f"part3_signal_delay_{hospital}_{manufacturer}".replace(' ', '_').replace('/', '_') + '.png'
        table_path = output_dir / filename
        plt.savefig(table_path, dpi=300, bbox_inches='tight', facecolor='white')
        plt.show()
        print(table_path)


In [ ]:
# Part 3: Delay plots for Karolinska (from delay_df)
# Plots delay vs step size, grouped by metric, with lines for current window sizes.

DELAY_HOSPITAL = 'Karolinska University Hospital'
DELAY_MANUFACTURER = None  # set to a manufacturer or None for all

if 'delay_df' not in globals() or delay_df is None or len(delay_df) == 0:
    print('delay_df not found or empty. Run the delay grid cell first.')
else:
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from pathlib import Path

    df = delay_df.copy()
    df = df[df['Hospital'] == DELAY_HOSPITAL]
    if DELAY_MANUFACTURER is not None:
        df = df[df['Manufacturer'] == DELAY_MANUFACTURER]

    if len(df) == 0:
        print('No rows for the selected hospital/manufacturer.')
    else:
        out_dir = Path('outputs/2025_12_29_counterfactual_manufacturer_swap_setup/delay_plots')
        out_dir.mkdir(parents=True, exist_ok=True)

        metrics = ['flagging_rate', 'relative_flagging_rate', 'p_ai_given_rad', 'p_rad_given_ai']
        metric_labels = {
            'flagging_rate': 'Flagging rate',
            'relative_flagging_rate': 'Relative flag rate',
            'p_ai_given_rad': 'P(AI|Rad)',
            'p_rad_given_ai': 'P(Rad|AI)'
        }

        for metric in metrics:
            df_m = df[df['Metric'] == metric]
            if len(df_m) == 0:
                continue

            fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
            for ai_idx, ai in enumerate(AI_COLUMNS):
                ax = axes[ai_idx]
                df_ai = df_m[df_m['AI'] == ai]
                if len(df_ai) == 0:
                    ax.set_title(ai.upper())
                    ax.text(0.5, 0.5, 'No data', ha='center', va='center')
                    ax.axis('off')
                    continue

                # Plot delay vs step size, one line per curr size
                for curr in sorted(df_ai['Curr size'].unique()):
                    df_c = df_ai[df_ai['Curr size'] == curr]
                    x = df_c['Step size'].values
                    y = df_c['Delay (exams)'].values
                    ax.plot(x, y, marker='o', label=f'curr={curr}')

                ax.axhline(0, color='#333333', linewidth=1)
                ax.set_title(ai.upper())
                ax.set_xlabel('Step size (exams)')
                if ai_idx == 0:
                    ax.set_ylabel('Delay vs baseline (exams)')
                ax.grid(True, alpha=0.3)

            handles, labels = axes[0].get_legend_handles_labels()
            axes[0].legend(handles, labels, fontsize=8, loc='upper left')

            m_label = metric_labels.get(metric, metric)
            man_label = DELAY_MANUFACTURER or 'All manufacturers'
            title = f'Delay vs step size | {DELAY_HOSPITAL} — {man_label} | {m_label}'
            fig.suptitle(title, fontsize=12, fontweight='bold')
            plt.tight_layout()

            fname = f"delay_{DELAY_HOSPITAL}_{man_label}_{metric}".replace(' ', '_').replace('/', '_') + '.png'
            out_path = out_dir / fname
            plt.savefig(out_path, dpi=300, bbox_inches='tight', facecolor='white')
            plt.show()
            print(out_path)


# Ground truth definition

Ground truth was defined using large current windows to reduce random fluctuations. For each metric, we computed windowed estimates using current window sizes between **20,000–40,000 exams** (shadow band) and defined ground truth as the **mean** of those estimates. Ground truth is used only as a stable reference and **not** during monitoring, which uses smaller windows for detection.


In [ ]:
# Ground truth definition (large current windows only)
# Uses 20k–40k exam windows to define a stable reference for each metric.

GT_HOSPITAL = 'Karolinska University Hospital'
GT_MANUFACTURER = NOISE_MANUFACTURER  # None for all manufacturers
GT_SCENARIO_LABEL = NOISE_SCENARIO_LABEL

GT_MIN_CURR = 20000
GT_MAX_CURR = 40000
GT_STEP_CURR = 2000  # coarse step within shadow band
print(f'Ground truth scope: {GT_HOSPITAL} / {GT_MANUFACTURER}')

if '_transition_center' not in globals():
    def _transition_center(transition_idx, curr_size):
        if transition_idx is None or curr_size is None:
            return None
        return transition_idx + (curr_size / 2)

if 'df_part3' not in globals():
    print('df_part3 not found. Run the Part 3 data load cell first.')
else:
    import numpy as np
    import pandas as pd

    # Pick scenario
    scenario = None
    for s in PART3_SCENARIOS:
        label = str(s.get('label','')).lower()
        if GT_SCENARIO_LABEL in label:
            scenario = s
            break

    if scenario is None:
        print('Scenario not found; check GT_SCENARIO_LABEL.')
    else:
        rad_mode = scenario.get('rad_mode')
        system_mode = scenario.get('system_mode')

        df_h = df_part3[df_part3[HOSPITAL_COL] == GT_HOSPITAL].copy()
        if len(df_h) == 0:
            print(f'No data for hospital: {GT_HOSPITAL}')
        else:
            if MANUFACTURER_COL in df_h.columns:
                if GT_MANUFACTURER is None:
                    manufacturers = sorted(df_h[MANUFACTURER_COL].dropna().unique())
                if GT_MANUFACTURER is not None:
                    df_h = df_h[df_h[MANUFACTURER_COL] == GT_MANUFACTURER].copy()

            if len(df_h) == 0:
                print(f'No data for hospital/manufacturer: {GT_HOSPITAL} / {GT_MANUFACTURER}')
            else:
                result = run_part3_hospital_pipeline(
                    df_h, GT_HOSPITAL, part3_start,
                    calibration_strategy=PART3_CALIBRATION_STRATEGY,
                    rad_mode=rad_mode
                )
                if result is None:
                    print('Pipeline failed for selected hospital/manufacturer.')
                else:
                    df_h = order_part3_df(result['df_h'])
                    transition_idx = result['transition_index']

                    # Build shadow window sizes
                    gt_sizes = list(range(GT_MIN_CURR, GT_MAX_CURR + 1, GT_STEP_CURR))

                    # Ensure flags exist
                    for ai in AI_COLUMNS:
                        base_flag = f'{ai}_flagged'
                        part3_flag = f'{ai}_flagged_part3'
                        if base_flag not in df_h.columns and part3_flag in df_h.columns:
                            df_h[base_flag] = df_h[part3_flag]

                    # Optional score normalization (z-score per AI)
                    score_norm_cols = {}
                    if 'SCORE_NORMALIZE' in globals() and SCORE_NORMALIZE == 'zscore':
                        if 'SCORE_NORM_SCOPE' in globals() and SCORE_NORM_SCOPE == 'prestart' and transition_idx is not None:
                            df_base = df_h.iloc[:transition_idx]
                        else:
                            df_base = df_h
                        for ai in AI_COLUMNS:
                            if ai not in df_h.columns:
                                continue
                            mu = df_base[ai].mean()
                            sigma = df_base[ai].std()
                            norm_col = f'{ai}_score_norm'
                            if pd.isna(sigma) or sigma == 0:
                                df_h[norm_col] = np.nan
                            else:
                                df_h[norm_col] = (df_h[ai] - mu) / sigma
                            score_norm_cols[ai] = norm_col

                    def _cumsum(arr):
                        return np.concatenate(([0.0], np.cumsum(arr)))

                    def _window_sum(csum, starts, ends):
                        return csum[ends] - csum[starts]

                    def _window_rate(num_csum, den_csum, starts, ends, scale=100.0):
                        num = _window_sum(num_csum, starts, ends)
                        den = _window_sum(den_csum, starts, ends)
                        out = np.full_like(num, np.nan, dtype=float)
                        mask = den > 0
                        out[mask] = (num[mask] / den[mask]) * scale
                        return out

                    # Precompute window indices
                    window_index = {}
                    for curr in gt_sizes:
                        total_span = PART3_REF_SIZE + PART3_GAP_SIZE + curr
                        centers = []
                        starts = []
                        ends = []
                        for ref_start in range(0, len(df_h) - total_span + 1, PART3_STEP_SIZE):
                            ref_end = ref_start + PART3_REF_SIZE
                            curr_start = ref_end + PART3_GAP_SIZE
                            curr_end = curr_start + curr
                            curr_center = (curr_start + curr_end) / 2
                            centers.append(curr_center)
                            starts.append(curr_start)
                            ends.append(curr_end)
                        window_index[curr] = (
                            np.array(centers, dtype=float),
                            np.array(starts, dtype=int),
                            np.array(ends, dtype=int)
                        )

                    has_cancer = df_h['has_cancer'].fillna(0).astype(int).to_numpy()
                    has_control = (has_cancer == 0).astype(int)
                    has_screen = None
                    if 'time_based_screen' in df_h.columns:
                        has_screen = df_h['time_based_screen'].fillna(0).astype(int).to_numpy()

                    metric_specs = [
                        ('sensitivity', 'Sensitivity (flagged | cancer)'),
                        ('specificity', 'Specificity (not flagged | control)'),
                        ('flagging_rate', 'Flagging rate (flagged | all exams)')
                    ]
                    if 'time_based_screen' in df_h.columns:
                        metric_specs.append(('screen_flagging_rate', 'Screen-detected flagging rate'))
                    if 'INCLUDE_SCORE_METRIC' in globals() and INCLUDE_SCORE_METRIC:
                        metric_specs.append(('score_mean', 'Mean AI score'))

                    rows = []
                    for ai in AI_COLUMNS:
                        flag_col = f'{ai}_flagged_part3'
                        if flag_col not in df_h.columns:
                            continue
                        flags = df_h[flag_col].fillna(0).astype(int).to_numpy()

                        c_flag_cancer = _cumsum(flags * has_cancer)
                        c_cancer = _cumsum(has_cancer)
                        c_flag_control = _cumsum(flags * has_control)
                        c_control = _cumsum(has_control)
                        c_flag_all = _cumsum(flags)
                        c_all = _cumsum(np.ones_like(flags, dtype=float))

                        if has_screen is not None:
                            c_flag_screen = _cumsum(flags * has_screen)
                            c_screen = _cumsum(has_screen)
                        else:
                            c_flag_screen = None
                            c_screen = None

                        score_arr = None
                        if 'INCLUDE_SCORE_METRIC' in globals() and INCLUDE_SCORE_METRIC:
                            if 'SCORE_NORMALIZE' in globals() and SCORE_NORMALIZE == 'zscore':
                                norm_col = score_norm_cols.get(ai)
                                if norm_col and norm_col in df_h.columns:
                                    score_arr = df_h[norm_col].to_numpy(dtype=float)
                            else:
                                if ai in df_h.columns:
                                    score_arr = df_h[ai].to_numpy(dtype=float)
                        c_score = _cumsum(score_arr) if score_arr is not None else None

                        for metric_key, metric_label in metric_specs:
                            all_vals = []
                            for curr in gt_sizes:
                                centers, starts, ends = window_index[curr]
                                center_min = _transition_center(transition_idx, curr)
                                if metric_key == 'sensitivity':
                                    vals = _window_rate(c_flag_cancer, c_cancer, starts, ends)
                                elif metric_key == 'specificity':
                                    flagged_rate = _window_rate(c_flag_control, c_control, starts, ends, scale=1.0)
                                    vals = (1.0 - flagged_rate) * 100.0
                                elif metric_key == 'flagging_rate':
                                    vals = _window_rate(c_flag_all, c_all, starts, ends)
                                elif metric_key == 'screen_flagging_rate':
                                    if c_flag_screen is None:
                                        vals = np.full_like(centers, np.nan, dtype=float)
                                    else:
                                        vals = _window_rate(c_flag_screen, c_screen, starts, ends)
                                elif metric_key == 'score_mean':
                                    if c_score is None:
                                        vals = np.full_like(centers, np.nan, dtype=float)
                                    else:
                                        sums = _window_sum(c_score, starts, ends)
                                        counts = _window_sum(c_all, starts, ends)
                                        vals = np.full_like(sums, np.nan, dtype=float)
                                        mask = counts > 0
                                        vals[mask] = sums[mask] / counts[mask]
                                else:
                                    vals = np.full_like(centers, np.nan, dtype=float)

                                if center_min is not None:
                                    vals = vals[centers >= center_min]
                                all_vals.extend([v for v in vals if not pd.isna(v)])

                            gt_value = float(np.mean(all_vals)) if len(all_vals) > 0 else np.nan
                            rows.append({
                                'ai_system': ai,
                                'metric': metric_label,
                                'ground_truth': gt_value,
                                'window_range': f'{GT_MIN_CURR}-{GT_MAX_CURR}',
                                'step': GT_STEP_CURR
                            })

                    gt_df = pd.DataFrame(rows)
                    display(gt_df)

                    # Render table as a styled image
                    import matplotlib.pyplot as plt
                    from pathlib import Path

                    output_dir = Path('outputs/2025_12_29_counterfactual_manufacturer_swap_setup')
                    output_dir.mkdir(parents=True, exist_ok=True)

                    gt_display = gt_df.copy()
                    gt_display = gt_display.rename(columns={
                        'ai_system': 'AI system',
                        'metric': 'Metric',
                        'ground_truth': 'Ground truth',
                        'window_range': 'Window range',
                        'step': 'Step'
                    })
                    # Clean metric labels and drop score rows
                    gt_display['Metric'] = gt_display['Metric'].str.replace(r'\s*\(.*\)$', '', regex=True)
                    gt_display = gt_display[gt_display['Metric'].str.lower() != 'mean ai score']

                    gt_display['Ground truth'] = gt_display.apply(
                        lambda r: '' if pd.isna(r['Ground truth']) else (f"{r['Ground truth']:.3f}" if 'score' in str(r['Metric']).lower() else f"{r['Ground truth']:.3f}%"),
                        axis=1
                    )

                    fig_height = 1.8 + 0.45 * len(gt_display)
                    fig, ax = plt.subplots(figsize=(14, fig_height))
                    ax.axis('off')

                    table = ax.table(
                        cellText=gt_display.values,
                        colLabels=gt_display.columns,
                        cellLoc='center',
                        loc='center',
                        bbox=[0, 0, 1, 1]
                    )

                    table.auto_set_font_size(False)
                    table.set_fontsize(10)
                    table.scale(1, 1.6)

                    header_color = '#4472C4'
                    alt_row_color = '#E7E6E6'

                    for i in range(len(gt_display.columns)):
                        cell = table[(0, i)]
                        cell.set_facecolor(header_color)
                        cell.set_text_props(weight='bold', color='white')

                    for i in range(1, len(gt_display) + 1):
                        for j in range(len(gt_display.columns)):
                            cell = table[(i, j)]
                            cell.set_facecolor(alt_row_color if i % 2 == 0 else 'white')

                    for cell in table.get_celld().values():
                        cell.set_edgecolor('black')
                        cell.set_linewidth(1.2)

                    plt.title(f'Ground Truth (20k–40k window band) | {GT_HOSPITAL} - {GT_MANUFACTURER}', fontsize=16, fontweight='bold', pad=20)
                    plt.tight_layout()
                    table_path = output_dir / 'ground_truth_table.png'
                    plt.savefig(table_path, dpi=300, bbox_inches='tight', facecolor='white')
                    plt.show()
                    table_path














In [ ]:
# Part 3B: Save RMS-vs-window plots (one image per metric)
# Requires df_rms and metric_specs from the Part 3B cell.

from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import re

if 'df_rms' not in globals() or 'metric_specs' not in globals():
    print('df_rms or metric_specs not found. Run the Part 3B cell first.')
else:
    def _knee_point(x, y):
        x = np.asarray(x, dtype=float)
        y = np.asarray(y, dtype=float)
        mask = (~np.isnan(x)) & (~np.isnan(y))
        x = x[mask]
        y = y[mask]
        if len(x) < 3:
            return np.nan, np.nan, np.nan
        order = np.argsort(x)
        x = x[order]
        y = y[order]

        x_min, x_max = x.min(), x.max()
        y_min, y_max = y.min(), y.max()
        x_rng = x_max - x_min
        y_rng = y_max - y_min
        if x_rng == 0:
            return np.nan, np.nan, np.nan
        x_n = (x - x_min) / x_rng
        y_n = np.zeros_like(y) if y_rng == 0 else (y - y_min) / y_rng

        x0, y0 = x_n[0], y_n[0]
        x1, y1 = x_n[-1], y_n[-1]
        denom = np.hypot(y1 - y0, x1 - x0)
        if denom == 0:
            return np.nan, np.nan, np.nan

        dist = np.abs((y1 - y0) * x_n - (x1 - x0) * y_n + x1*y0 - y1*x0) / denom
        idx = int(np.argmax(dist))
        return x[idx], y[idx], dist[idx]

    # Precompute knee points for each metric/AI
    knee_lookup = {}
    for item in metric_specs:
        if len(item) == 3:
            metric, _label, _unit = item
        else:
            metric, _label = item
        for ai in AI_COLUMNS:
            df_ai = df_rms[(df_rms['metric'] == metric) & (df_rms['ai_system'] == ai)]
            if len(df_ai) == 0:
                continue
            kx, ky, _kd = _knee_point(df_ai['curr_size'].values, df_ai['rms'].values)
            knee_lookup[(metric, ai)] = (kx, ky)

    output_dir = Path('outputs/2025_12_29_counterfactual_manufacturer_swap_setup/part3b_rms_plots')
    output_dir.mkdir(parents=True, exist_ok=True)

    manufacturer_label = RMS_SIGNAL_MANUFACTURER or 'All manufacturers'
    scen_label = str(scenario.get('label','')) if 'scenario' in globals() else ''

    for item in metric_specs:
        if len(item) == 3:
            metric, label, unit_label = item
        else:
            metric, label = item
            unit_label = ''
        label_clean = re.sub(r"\s*\(.*\)$", "", label)
        if "score" in metric.lower():
            continue

        fig, ax = plt.subplots(1, 1, figsize=(7, 4.5))
        for ai in AI_COLUMNS:
            df_ai = df_rms[(df_rms['metric'] == metric) & (df_rms['ai_system'] == ai)]
            if len(df_ai) == 0:
                continue
            ax.plot(df_ai['curr_size'], df_ai['rms'], label=f'{ai.upper()}')
            kx, ky = knee_lookup.get((metric, ai), (np.nan, np.nan))
            if np.isfinite(kx) and np.isfinite(ky):
                ax.scatter([kx], [ky], s=35)
                ax.axvline(kx, linestyle='--', alpha=0.25)
        ax.set_title(label_clean)
        ax.set_xlabel('Current window size (exams)')
        ax.set_ylabel('RMS deviation from ground truth (pp)')
        ax.grid(True, alpha=0.3)
        if GROUND_TRUTH_MIN is not None and GROUND_TRUTH_MAX is not None:
            ax.axvspan(GROUND_TRUTH_MIN, min(GROUND_TRUTH_MAX, df_rms['curr_size'].max()), color='gray', alpha=0.1)
        ax.legend(fontsize=8)
        fig.suptitle(
            f'RMS vs window size | {RMS_SIGNAL_HOSPITAL} - {manufacturer_label} | {scen_label}',
            fontsize=12, fontweight='bold'
        )
        plt.tight_layout()

        safe_metric = metric.replace(' ', '_')
        out_path = output_dir / f'rms_window_{safe_metric}.png'
        fig.savefig(out_path, dpi=300, bbox_inches='tight', facecolor='white')
        plt.show()

    print(f'Saved plots to: {output_dir}')



In [ ]:
# Part 3B: Minority-class counts in 20k-exam current windows (REWRITTEN - Clean Version v2)
# Same output as the prior cell, but avoids duplicate display/overlap issues.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Configuration
CURR_SIZE = 20000
COUNT_HOSPITAL = RMS_SIGNAL_HOSPITAL
COUNT_MANUFACTURER = RMS_SIGNAL_MANUFACTURER
COUNT_SCENARIO_LABEL = RMS_SIGNAL_SCENARIO_LABEL
SHOW_TABLE = False
SHOW_IMAGE = True

# Validation
if 'df_part3' not in globals():
    print('df_part3 not found. Run the Part 3 data load cell first.')
else:
    # Find scenario
    scenario = next(
        (s for s in PART3_SCENARIOS if COUNT_SCENARIO_LABEL in str(s.get('label', '')).lower()),
        None
    )

    if scenario is None:
        print('Scenario not found; check COUNT_SCENARIO_LABEL.')
    else:
        # Filter data
        df_h = df_part3[df_part3[HOSPITAL_COL] == COUNT_HOSPITAL].copy()
        if MANUFACTURER_COL in df_h.columns and COUNT_MANUFACTURER is not None:
            df_h = df_h[df_h[MANUFACTURER_COL] == COUNT_MANUFACTURER].copy()

        if len(df_h) == 0:
            print(f'No data for hospital/manufacturer: {COUNT_HOSPITAL} / {COUNT_MANUFACTURER}')
        else:
            # Run pipeline
            result = run_part3_hospital_pipeline(
                df_h, COUNT_HOSPITAL, part3_start,
                calibration_strategy=PART3_CALIBRATION_STRATEGY,
                rad_mode=scenario.get('rad_mode')
            )
            if result is None:
                print('Pipeline failed for selected hospital/manufacturer.')
            else:
                df_h = order_part3_df(result['df_h'])
                transition_idx = result['transition_index']

                # Ensure flags exist
                for ai in AI_COLUMNS:
                    base_flag = f'{ai}_flagged'
                    part3_flag = f'{ai}_flagged_part3'
                    if part3_flag not in df_h.columns and base_flag in df_h.columns:
                        df_h[part3_flag] = df_h[base_flag]

                # Window positions
                total_span = PART3_REF_SIZE + PART3_GAP_SIZE + CURR_SIZE
                windows = []
                for ref_start in range(0, len(df_h) - total_span + 1, PART3_STEP_SIZE):
                    curr_start = ref_start + PART3_REF_SIZE + PART3_GAP_SIZE
                    curr_end = curr_start + CURR_SIZE
                    curr_center = (curr_start + curr_end) / 2
                    windows.append((curr_start, curr_end, curr_center))

                if len(windows) == 0:
                    print('No valid windows for this configuration.')
                else:
                    starts = np.array([w[0] for w in windows], dtype=int)
                    ends = np.array([w[1] for w in windows], dtype=int)
                    centers = np.array([w[2] for w in windows], dtype=float)

                    # Helper functions
                    def cumsum_array(arr):
                        return np.concatenate(([0.0], np.cumsum(arr)))

                    def window_sum(csum, s, e):
                        return csum[e] - csum[s]

                    def compute_stats(arr):
                        arr = np.asarray(arr, dtype=float)
                        valid = arr[~np.isnan(arr)]
                        if len(valid) == 0:
                            return {k: np.nan for k in ['n_windows', 'mean', 'median', 'p10', 'p90', 'min', 'max']}
                        return {
                            'n_windows': int(len(valid)),
                            'mean': float(np.mean(valid)),
                            'median': float(np.median(valid)),
                            'p10': float(np.percentile(valid, 10)),
                            'p90': float(np.percentile(valid, 90)),
                            'min': float(np.min(valid)),
                            'max': float(np.max(valid))
                        }

                    # Data arrays
                    has_cancer = df_h['has_cancer'].fillna(0).astype(int).to_numpy()
                    has_control = (has_cancer == 0).astype(int)

                    # Cancer counts
                    c_cancer = cumsum_array(has_cancer)
                    cancer_counts = window_sum(c_cancer, starts, ends)

                    # Post-transition filter
                    transition_center = _transition_center(transition_idx, CURR_SIZE)
                    if transition_center is not None:
                        mask = centers >= transition_center
                    else:
                        mask = np.ones(len(centers), dtype=bool)

                    cancer_counts = cancer_counts[mask]

                    # Build summary
                    summary_rows = []
                    stats = compute_stats(cancer_counts)
                    stats.update({'metric': 'cancers', 'ai_system': 'all'})
                    summary_rows.append(stats)

                    for ai in AI_COLUMNS:
                        flag_col = f'{ai}_flagged_part3'
                        if flag_col not in df_h.columns:
                            continue
                        flags = df_h[flag_col].fillna(0).astype(int).to_numpy()

                        c_flag = cumsum_array(flags)
                        flag_counts = window_sum(c_flag, starts, ends)[mask]
                        stats = compute_stats(flag_counts)
                        stats.update({'metric': 'flagged_all', 'ai_system': ai})
                        summary_rows.append(stats)

                        c_flag_control = cumsum_array(flags * has_control)
                        flag_control_counts = window_sum(c_flag_control, starts, ends)[mask]
                        stats = compute_stats(flag_control_counts)
                        stats.update({'metric': 'flagged_controls', 'ai_system': ai})
                        summary_rows.append(stats)

                    summary_df = pd.DataFrame(summary_rows)
                    col_order = ['metric', 'ai_system', 'n_windows', 'mean', 'median', 'p10', 'p90', 'min', 'max']
                    summary_df = summary_df[[c for c in col_order if c in summary_df.columns]]

                    if SHOW_TABLE:
                        display(summary_df)

                    if SHOW_IMAGE:
                        # Format values with 2 decimals
                        table_df = summary_df.copy()
                        num_cols = [c for c in table_df.columns if c not in ['metric', 'ai_system']]
                        for c in num_cols:
                            table_df[c] = table_df[c].map(lambda v: '' if pd.isna(v) else f'{v:.2f}')

                        fig_h = 1.6 + 0.45 * len(table_df)
                        fig_w = max(12, 1.6 * len(table_df.columns))
                        fig, ax = plt.subplots(figsize=(fig_w, fig_h))
                        ax.axis('off')

                        table = ax.table(
                            cellText=table_df.values,
                            colLabels=table_df.columns,
                            cellLoc='center',
                            loc='center',
                            bbox=[0, 0, 1, 1]
                        )

                        table.auto_set_font_size(False)
                        table.set_fontsize(9)
                        table.scale(1.2, 1.6)
                        try:
                            table.auto_set_column_width(col=list(range(len(table_df.columns))))
                        except Exception:
                            pass

                        header_color = '#4472C4'
                        alt_row_color = '#E7E6E6'

                        for j in range(len(table_df.columns)):
                            cell = table[(0, j)]
                            cell.set_facecolor(header_color)
                            cell.set_text_props(weight='bold', color='white')

                        for r in range(1, len(table_df) + 1):
                            for j in range(len(table_df.columns)):
                                cell = table[(r, j)]
                                cell.set_facecolor(alt_row_color if r % 2 == 0 else 'white')

                        for cell in table.get_celld().values():
                            cell.set_edgecolor('black')
                            cell.set_linewidth(1.1)

                        plt.title(
                            f'Minority-class counts per {CURR_SIZE} exams | {COUNT_HOSPITAL} - {COUNT_MANUFACTURER}',
                            fontsize=14, fontweight='bold', pad=16
                        )
                        plt.tight_layout()

                        out_dir = Path('outputs/2025_12_29_counterfactual_manufacturer_swap_setup')
                        out_dir.mkdir(parents=True, exist_ok=True)
                        out_path = out_dir / 'minority_class_counts_20k.png'
                        plt.savefig(out_path, dpi=300, bbox_inches='tight', facecolor='white')
                        plt.show()
                        plt.close(fig)

                        print('Notes:')
                        print('- "cancers" is the minority class for sensitivity.')
                        print('- "flagged_all" and "flagged_controls" are the minority classes for flagging rate and specificity.')
                        print(f'Saved table image to: {out_path}')



In [ ]:
# Part 3B: Knee summary across all metrics (from RMS curves)
# Requires df_rms (from RMS-vs-window cell). Includes score if present in df_rms.

import numpy as np
import pandas as pd

if 'df_rms' not in globals():
    print('df_rms not found. Run the RMS-vs-window cell first.')
else:
    def _knee_point(x, y):
        x = np.asarray(x, dtype=float)
        y = np.asarray(y, dtype=float)
        mask = (~np.isnan(x)) & (~np.isnan(y))
        x = x[mask]
        y = y[mask]
        if len(x) < 3:
            return np.nan, np.nan, np.nan
        order = np.argsort(x)
        x = x[order]
        y = y[order]

        x_min, x_max = x.min(), x.max()
        y_min, y_max = y.min(), y.max()
        x_rng = x_max - x_min
        y_rng = y_max - y_min
        if x_rng == 0:
            return np.nan, np.nan, np.nan
        x_n = (x - x_min) / x_rng
        y_n = np.zeros_like(y) if y_rng == 0 else (y - y_min) / y_rng

        x0, y0 = x_n[0], y_n[0]
        x1, y1 = x_n[-1], y_n[-1]
        denom = np.hypot(y1 - y0, x1 - x0)
        if denom == 0:
            return np.nan, np.nan, np.nan

        dist = np.abs((y1 - y0) * x_n - (x1 - x0) * y_n + x1*y0 - y1*x0) / denom
        idx = int(np.argmax(dist))
        return x[idx], y[idx], dist[idx]

    rows = []
    for metric in sorted(df_rms['metric'].dropna().unique()):
        for ai in AI_COLUMNS:
            df_ai = df_rms[(df_rms['metric'] == metric) & (df_rms['ai_system'] == ai)]
            if len(df_ai) == 0:
                continue
            kx, ky, kd = _knee_point(df_ai['curr_size'].values, df_ai['rms'].values)
            rows.append({
                'metric': metric,
                'ai_system': ai,
                'knee_curr_size': kx,
                'knee_rms': ky,
                'knee_distance': kd
            })

    knee_df = pd.DataFrame(rows)
    if len(knee_df) == 0:
        print('No knee points computed (df_rms empty?)')
    else:
        # Small summary table per AI (one row per metric/AI)
        display(knee_df.sort_values(['metric', 'ai_system']))

        # Optional pivot for quick view (knee sizes only)
        knee_pivot = knee_df.pivot(index='metric', columns='ai_system', values='knee_curr_size')
        display(knee_pivot)



In [ ]:
# Part 3B: Step-size justification (pre/post signal counts + delay)
# Grid over (curr, step) and compute pre/post signal counts and delay vs transition.

import numpy as np
import pandas as pd

if 'PART3_RECAL_ALPHA' not in globals():
    PART3_RECAL_ALPHA = 0.05
if 'PART3_RECAL_USE_SCORE_SIGNALS' not in globals():
    PART3_RECAL_USE_SCORE_SIGNALS = False
if 'PART3_RECAL_WINDOW_SIZE' not in globals() and 'PART3_REF_SIZE' in globals():
    PART3_RECAL_WINDOW_SIZE = PART3_REF_SIZE * 2

GRID_CURR_SIZES = [5000, 10000, 20000]
GRID_STEP_SIZES = [500, 1000, 2000]
GRID_METRICS = ['flagging_rate', 'relative_flagging_rate', 'p_ai_given_rad', 'p_rad_given_ai']
GRID_SCENARIO_LABEL = 'single'  # 'single' or 'double'
EXCLUDE_DANDERYDS = True

if 'df_part3' not in globals():
    print('df_part3 not found. Run the Part 3 data load cell first.')
else:
    # Pick scenario
    scenario = None
    for s in PART3_SCENARIOS:
        label = str(s.get('label','')).lower()
        if GRID_SCENARIO_LABEL in label:
            scenario = s
            break
    if scenario is None:
        print('Scenario not found; check GRID_SCENARIO_LABEL.')
    else:
        rad_mode = scenario.get('rad_mode')
        rows = []

        hospitals = sorted(df_part3[HOSPITAL_COL].dropna().unique())
        for hospital in hospitals:
            if EXCLUDE_DANDERYDS and 'danderyd' in str(hospital).lower():
                continue
            df_hospital = df_part3[df_part3[HOSPITAL_COL] == hospital].copy()
            if MANUFACTURER_COL in df_hospital.columns:
                manufacturers = sorted(df_hospital[MANUFACTURER_COL].dropna().unique())
                if len(manufacturers) == 0:
                    manufacturers = ['Unknown']
            else:
                manufacturers = ['Unknown']

            for manufacturer in manufacturers:
                if manufacturer == 'Unknown':
                    df_subset = df_hospital.copy()
                else:
                    df_subset = df_hospital[df_hospital[MANUFACTURER_COL] == manufacturer].copy()
                if len(df_subset) == 0:
                    continue

                result = run_part3_hospital_pipeline(
                    df_subset, hospital, part3_start,
                    calibration_strategy=PART3_CALIBRATION_STRATEGY,
                    rad_mode=rad_mode
                )
                if result is None:
                    continue

                df_h = order_part3_df(result['df_h'])
                transition_idx = result['transition_index']

                for curr in GRID_CURR_SIZES:
                    for step in GRID_STEP_SIZES:
                        # Build centers for window count
                        total_span = PART3_REF_SIZE + PART3_GAP_SIZE + curr
                        centers = []
                        for ref_start in range(0, len(df_h) - total_span + 1, step):
                            ref_end = ref_start + PART3_REF_SIZE
                            curr_start = ref_end + PART3_GAP_SIZE
                            curr_end = curr_start + curr
                            centers.append((curr_start + curr_end) / 2)
                        centers = np.array(centers, dtype=float)
                        transition_center = _transition_center(transition_idx, curr)
                        if transition_center is None:
                            continue
                        total_after = int(np.sum(centers >= transition_center))

                        # Collect signals
                        centers_dict = collect_part2_signal_centers(
                            df_h, alpha=PART3_RECAL_ALPHA,
                            ref_size=PART3_REF_SIZE, gap_size=PART3_GAP_SIZE,
                            curr_size=curr, step_size=step,
                            use_score_signals=PART3_RECAL_USE_SCORE_SIGNALS
                        )

                        for ai in AI_COLUMNS:
                            for metric_key in GRID_METRICS:
                                signal_key = metric_key if metric_key != 'relative_flagging_rate' else 'flagging_rate'
                                sig_centers = centers_dict.get(ai, {}).get(signal_key, [])
                                sig_centers = [c for c in sig_centers if c is not None]

                                fp = sum(1 for c in sig_centers if c < transition_center)
                                tp = sum(1 for c in sig_centers if c >= transition_center)

                                # Delay: first post-start signal
                                post_centers = [c for c in sig_centers if c >= transition_center]
                                if len(post_centers) == 0:
                                    delay = np.nan
                                else:
                                    delay = float(min(post_centers) - transition_center)

                                rows.append({
                                    'Scenario': str(scenario.get('label','')),
                                    'Hospital': hospital,
                                    'Manufacturer': manufacturer,
                                    'AI': ai,
                                    'Metric': metric_key,
                                    'Curr size': curr,
                                    'Step size': step,
                                    'Total windows after start': total_after,
                                    'signals_post_start': tp,
                                    'signals_pre_start': fp,
                                    'first_signal_delay': delay
                                })

        step_df = pd.DataFrame(rows)
        if len(step_df) == 0:
            print('No results. Check data or filters.')
        else:
            display(step_df)
            summary = (
                step_df.groupby(['Curr size', 'Step size', 'Metric'])
                .agg(
                    mean_delay=('first_signal_delay', 'mean'),
                    median_delay=('first_signal_delay', 'median'),
                    mean_signals_pre=('signals_pre_start', 'mean'),
                    mean_signals_post=('signals_post_start', 'mean'),
                )
                .reset_index()
            )
            display(summary)





In [ ]:
# Part 3B: Save knee summary tables as images
# Requires knee_df and knee_pivot from the knee summary cell above.

import matplotlib.pyplot as plt
from pathlib import Path
import pandas as pd

if 'knee_df' not in globals() or 'knee_pivot' not in globals():
    print('knee_df or knee_pivot not found. Run the knee summary cell first.')
else:
    out_dir = Path('outputs/2025_12_29_counterfactual_manufacturer_swap_setup')
    out_dir.mkdir(parents=True, exist_ok=True)

    def _save_table_image(df, title, out_path, fmt='%.2f'):
        df_disp = df.copy()
        # format numeric columns
        for c in df_disp.columns:
            if pd.api.types.is_numeric_dtype(df_disp[c]):
                df_disp[c] = df_disp[c].map(lambda v: '' if pd.isna(v) else (fmt % v))
        fig_h = 1.6 + 0.45 * len(df_disp)
        fig_w = max(12, 1.6 * len(df_disp.columns))
        fig, ax = plt.subplots(figsize=(fig_w, fig_h))
        ax.axis('off')
        table = ax.table(
            cellText=df_disp.values,
            colLabels=df_disp.columns,
            cellLoc='center',
            loc='center',
            bbox=[0, 0, 1, 1]
        )
        table.auto_set_font_size(False)
        table.set_fontsize(9)
        table.scale(1.2, 1.6)
        try:
            table.auto_set_column_width(col=list(range(len(df_disp.columns))))
        except Exception:
            pass
        header_color = '#4472C4'
        alt_row_color = '#E7E6E6'
        for j in range(len(df_disp.columns)):
            cell = table[(0, j)]
            cell.set_facecolor(header_color)
            cell.set_text_props(weight='bold', color='white')
        for r in range(1, len(df_disp) + 1):
            for j in range(len(df_disp.columns)):
                cell = table[(r, j)]
                cell.set_facecolor(alt_row_color if r % 2 == 0 else 'white')
        for cell in table.get_celld().values():
            cell.set_edgecolor('black')
            cell.set_linewidth(1.1)
        plt.title(title, fontsize=14, fontweight='bold', pad=16)
        plt.tight_layout()
        plt.savefig(out_path, dpi=300, bbox_inches='tight', facecolor='white')
        plt.show()
        plt.close(fig)
        print(f'Saved: {out_path}')

    # Full knee table
    _save_table_image(
        knee_df.sort_values(['metric', 'ai_system']).reset_index(drop=True),
        'Knee Summary (All Metrics / AI)',
        out_dir / 'knee_summary_table.png'
    )

    # Pivot table (knee sizes only)
    pivot_df = knee_pivot.reset_index()
    _save_table_image(
        pivot_df,
        'Knee Window Size by Metric and AI',
        out_dir / 'knee_summary_pivot.png'
    )



In [ ]:
# Part 3B: Save step-size tables as images
# Requires step_df and summary from the step-size justification cell.

import matplotlib.pyplot as plt
from pathlib import Path
import pandas as pd

if 'step_df' not in globals() or 'summary' not in globals():
    print('step_df or summary not found. Run the step-size justification cell first.')
else:
    out_dir = Path('outputs/2025_12_29_counterfactual_manufacturer_swap_setup')
    out_dir.mkdir(parents=True, exist_ok=True)

    def _save_table_image(df, title, out_path, fmt='%.2f', max_rows=None):
        df_disp = df.copy()
        if max_rows is not None and len(df_disp) > max_rows:
            df_disp = df_disp.head(max_rows)
        for c in df_disp.columns:
            if pd.api.types.is_numeric_dtype(df_disp[c]):
                df_disp[c] = df_disp[c].map(lambda v: '' if pd.isna(v) else (fmt % v))
        fig_h = 1.6 + 0.45 * len(df_disp)
        fig_w = max(12, 1.6 * len(df_disp.columns))
        fig, ax = plt.subplots(figsize=(fig_w, fig_h))
        ax.axis('off')
        table = ax.table(
            cellText=df_disp.values,
            colLabels=df_disp.columns,
            cellLoc='center',
            loc='center',
            bbox=[0, 0, 1, 1]
        )
        table.auto_set_font_size(False)
        table.set_fontsize(9)
        table.scale(1.2, 1.6)
        try:
            table.auto_set_column_width(col=list(range(len(df_disp.columns))))
        except Exception:
            pass
        header_color = '#4472C4'
        alt_row_color = '#E7E6E6'
        for j in range(len(df_disp.columns)):
            cell = table[(0, j)]
            cell.set_facecolor(header_color)
            cell.set_text_props(weight='bold', color='white')
        for r in range(1, len(df_disp) + 1):
            for j in range(len(df_disp.columns)):
                cell = table[(r, j)]
                cell.set_facecolor(alt_row_color if r % 2 == 0 else 'white')
        for cell in table.get_celld().values():
            cell.set_edgecolor('black')
            cell.set_linewidth(1.1)
        plt.title(title, fontsize=14, fontweight='bold', pad=16)
        plt.tight_layout()
        plt.savefig(out_path, dpi=300, bbox_inches='tight', facecolor='white')
        plt.show()
        plt.close(fig)
        print(f'Saved: {out_path}')

    # Summary table (compact)
    _save_table_image(
        summary,
        'Step-size Summary (All Hospitals)',
        out_dir / 'step_size_summary.png'
    )

    # Karolinska-only detailed table
    karolinska_df = step_df[step_df['Hospital'] == 'Karolinska University Hospital']
    if len(karolinska_df) == 0:
        print('No Karolinska rows found in step_df.')
    else:
        _save_table_image(
            karolinska_df,
            'Step-size Grid (Karolinska University Hospital)',
            out_dir / 'step_size_karolinska.png'
        )



In [ ]:
# Part 3B: Time to reach N exams (months) per hospital/manufacturer
# Uses monthly exam counts to estimate how long it takes to reach N exams.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

N_EXAMS_LIST = [10000, 20000, 40000, 100000]
INCLUDE_MANUFACTURER = True
EXCLUDE_DANDERYDS = True

if 'df_part3' not in globals():
    print('df_part3 not found. Run the Part 3 data load cell first.')
else:
    # Ensure date + month_key exist
    if DATE_COL not in df_part3.columns:
        print(f'{DATE_COL} not in df_part3.')
    else:
        df_tmp = df_part3.copy()
        df_tmp[DATE_COL] = pd.to_datetime(df_tmp[DATE_COL], format='mixed')
        df_tmp['month_key'] = df_tmp[DATE_COL].dt.to_period('M').astype(str)

        group_cols = [HOSPITAL_COL]
        if INCLUDE_MANUFACTURER and MANUFACTURER_COL in df_tmp.columns:
            group_cols.append(MANUFACTURER_COL)

        rows = []
        for keys, df_g in df_tmp.groupby(group_cols):
            # Filter out Danderyds if requested
            hospital = keys[0] if isinstance(keys, tuple) else keys
            if EXCLUDE_DANDERYDS and 'danderyd' in str(hospital).lower():
                continue

            # Monthly counts
            monthly = df_g.groupby('month_key').size().sort_index()
            if len(monthly) == 0:
                continue

            cumsum = monthly.cumsum()
            months = monthly.index.to_list()

            for N in N_EXAMS_LIST:
                reached = cumsum[cumsum >= N]
                if len(reached) == 0:
                    months_to = np.nan
                else:
                    first_month = months[0]
                    reach_month = reached.index[0]
                    # month difference
                    m0 = pd.Period(first_month, freq='M')
                    m1 = pd.Period(reach_month, freq='M')
                    months_to = int(m1.ordinal - m0.ordinal)

                if isinstance(keys, tuple):
                    row = {
                        'Hospital': keys[0],
                        'Manufacturer': keys[1],
                        'N_exams': N,
                        'Months_to_reach': months_to
                    }
                else:
                    row = {
                        'Hospital': keys,
                        'N_exams': N,
                        'Months_to_reach': months_to
                    }
                rows.append(row)

        time_df = pd.DataFrame(rows)
        display(time_df)

        # Summary plot (by hospital, for each N)
        if len(time_df) > 0:
            out_dir = Path('outputs/2025_12_29_counterfactual_manufacturer_swap_setup')
            out_dir.mkdir(parents=True, exist_ok=True)

            for N in N_EXAMS_LIST:
                df_n = time_df[time_df['N_exams'] == N].copy()
                if len(df_n) == 0:
                    continue
                # Aggregate if multiple manufacturers
                if 'Manufacturer' in df_n.columns:
                    df_plot = df_n.groupby('Hospital')['Months_to_reach'].mean().reset_index()
                else:
                    df_plot = df_n

                df_plot = df_plot.sort_values('Months_to_reach', ascending=True)
                fig_h = 0.35 * len(df_plot) + 2
                fig, ax = plt.subplots(figsize=(10, fig_h))
                ax.barh(df_plot['Hospital'], df_plot['Months_to_reach'], color='#4C72B0')
                ax.set_xlabel('Months to reach N exams')
                ax.set_title(f'Time to reach {N:,} exams (months)')
                ax.grid(True, axis='x', alpha=0.3)
                plt.tight_layout()

                out_path = out_dir / f'time_to_{N}_exams_months.png'
                plt.savefig(out_path, dpi=300, bbox_inches='tight', facecolor='white')
                plt.show()
                plt.close(fig)

            print('Saved plots to:', out_dir)



In [ ]:
# Part 3B: Recommended parameters (curr=20000, step=1000)
# Quick summary using the step-size grid logic for the chosen parameters.

import numpy as np
import pandas as pd

if 'PART3_RECAL_ALPHA' not in globals():
    PART3_RECAL_ALPHA = 0.05
if 'PART3_RECAL_USE_SCORE_SIGNALS' not in globals():
    PART3_RECAL_USE_SCORE_SIGNALS = False
if 'PART3_RECAL_WINDOW_SIZE' not in globals() and 'PART3_REF_SIZE' in globals():
    PART3_RECAL_WINDOW_SIZE = PART3_REF_SIZE * 2

REC_CURR_SIZE = 20000
REC_STEP_SIZE = 1000
REC_METRICS = ['flagging_rate', 'relative_flagging_rate', 'p_ai_given_rad', 'p_rad_given_ai']
REC_SCENARIO_LABEL = 'single'  # 'single' or 'double'
EXCLUDE_DANDERYDS = True

if 'df_part3' not in globals():
    print('df_part3 not found. Run the Part 3 data load cell first.')
else:
    # Pick scenario
    scenario = None
    for s in PART3_SCENARIOS:
        label = str(s.get('label','')).lower()
        if REC_SCENARIO_LABEL in label:
            scenario = s
            break
    if scenario is None:
        print('Scenario not found; check REC_SCENARIO_LABEL.')
    else:
        rad_mode = scenario.get('rad_mode')
        rows = []

        hospitals = sorted(df_part3[HOSPITAL_COL].dropna().unique())
        for hospital in hospitals:
            if EXCLUDE_DANDERYDS and 'danderyd' in str(hospital).lower():
                continue
            df_hospital = df_part3[df_part3[HOSPITAL_COL] == hospital].copy()
            if MANUFACTURER_COL in df_hospital.columns:
                manufacturers = sorted(df_hospital[MANUFACTURER_COL].dropna().unique())
                if len(manufacturers) == 0:
                    manufacturers = ['Unknown']
            else:
                manufacturers = ['Unknown']

            for manufacturer in manufacturers:
                if manufacturer == 'Unknown':
                    df_subset = df_hospital.copy()
                else:
                    df_subset = df_hospital[df_hospital[MANUFACTURER_COL] == manufacturer].copy()
                if len(df_subset) == 0:
                    continue

                result = run_part3_hospital_pipeline(
                    df_subset, hospital, part3_start,
                    calibration_strategy=PART3_CALIBRATION_STRATEGY,
                    rad_mode=rad_mode
                )
                if result is None:
                    continue

                df_h = order_part3_df(result['df_h'])
                transition_idx = result['transition_index']

                # Window centers for counting total windows after start
                total_span = PART3_REF_SIZE + PART3_GAP_SIZE + REC_CURR_SIZE
                centers = []
                for ref_start in range(0, len(df_h) - total_span + 1, REC_STEP_SIZE):
                    ref_end = ref_start + PART3_REF_SIZE
                    curr_start = ref_end + PART3_GAP_SIZE
                    curr_end = curr_start + REC_CURR_SIZE
                    centers.append((curr_start + curr_end) / 2)
                centers = np.array(centers, dtype=float)
                transition_center = _transition_center(transition_idx, REC_CURR_SIZE)
                if transition_center is None:
                    continue
                total_after = int(np.sum(centers >= transition_center))

                centers_dict = collect_part2_signal_centers(
                    df_h, alpha=PART3_RECAL_ALPHA,
                    ref_size=PART3_REF_SIZE, gap_size=PART3_GAP_SIZE,
                    curr_size=REC_CURR_SIZE, step_size=REC_STEP_SIZE,
                    use_score_signals=PART3_RECAL_USE_SCORE_SIGNALS
                )

                for ai in AI_COLUMNS:
                    for metric_key in REC_METRICS:
                        signal_key = metric_key if metric_key != 'relative_flagging_rate' else 'flagging_rate'
                        sig_centers = centers_dict.get(ai, {}).get(signal_key, [])
                        sig_centers = [c for c in sig_centers if c is not None]

                        fp = sum(1 for c in sig_centers if c < transition_center)
                        tp = sum(1 for c in sig_centers if c >= transition_center)

                        post_centers = [c for c in sig_centers if c >= transition_center]
                        if len(post_centers) == 0:
                            delay = np.nan
                        else:
                            delay = float(min(post_centers) - transition_center)

                        rows.append({
                            'Scenario': str(scenario.get('label','')),
                            'Hospital': hospital,
                            'Manufacturer': manufacturer,
                            'AI': ai,
                            'Metric': metric_key,
                            'Curr size': REC_CURR_SIZE,
                            'Step size': REC_STEP_SIZE,
                            'Total windows after start': total_after,
                            'signals_post_start': tp,
                            'signals_pre_start': fp,
                            'First signal delay (exams)': delay
                        })

        rec_df = pd.DataFrame(rows)
        if len(rec_df) == 0:
            print('No results. Check data or filters.')
        else:
            # Drop columns not needed for reporting
            rec_df = rec_df.drop(columns=['Curr size', 'Step size', 'signals_pre_start'], errors='ignore')
            summary = (
                rec_df.groupby(['Metric'])
                .agg(
                    mean_delay=('First signal delay (exams)', 'mean'),
                    median_delay=('First signal delay (exams)', 'median'),
                    mean_signals_post=('signals_post_start', 'mean')
                )
                .reset_index()
            )
            display(summary)

            # Save tables as images
            import matplotlib.pyplot as plt
            from pathlib import Path

            out_dir = Path('outputs/2025_12_29_counterfactual_manufacturer_swap_setup')
            out_dir.mkdir(parents=True, exist_ok=True)

            def _save_table_image(df, title, out_path, fmt='%.2f'):
                df_disp = df.copy()
                for c in df_disp.columns:
                    if pd.api.types.is_numeric_dtype(df_disp[c]):
                        df_disp[c] = df_disp[c].map(lambda v: '' if pd.isna(v) else (fmt % v))
                fig_h = 1.6 + 0.45 * len(df_disp)
                fig_w = max(12, 1.6 * len(df_disp.columns))
                fig, ax = plt.subplots(figsize=(fig_w, fig_h))
                ax.axis('off')
                table = ax.table(
                    cellText=df_disp.values,
                    colLabels=df_disp.columns,
                    cellLoc='center',
                    loc='center',
                    bbox=[0, 0, 1, 1]
                )
                table.auto_set_font_size(False)
                table.set_fontsize(9)
                table.scale(1.2, 1.6)
                try:
                    table.auto_set_column_width(col=list(range(len(df_disp.columns))))
                except Exception:
                    pass
                header_color = '#4472C4'
                alt_row_color = '#E7E6E6'
                for j in range(len(df_disp.columns)):
                    cell = table[(0, j)]
                    cell.set_facecolor(header_color)
                    cell.set_text_props(weight='bold', color='white')
                for r in range(1, len(df_disp) + 1):
                    for j in range(len(df_disp.columns)):
                        cell = table[(r, j)]
                        cell.set_facecolor(alt_row_color if r % 2 == 0 else 'white')
                for cell in table.get_celld().values():
                    cell.set_edgecolor('black')
                    cell.set_linewidth(1.1)
                plt.title(title, fontsize=14, fontweight='bold', pad=16)
                plt.tight_layout()
                plt.savefig(out_path, dpi=300, bbox_inches='tight', facecolor='white')
                plt.show()
                plt.close(fig)
                print(f'Saved: {out_path}')

            _save_table_image(
                summary,
                'Recommended Params (curr=20000, step=1000) - Summary',
                out_dir / 'recommended_params_summary.png'
            )

            # Save one image per hospital
            out_dir_h = out_dir / 'recommended_params_by_hospital'
            out_dir_h.mkdir(parents=True, exist_ok=True)

            for hospital, df_h in rec_df.groupby('Hospital'):
                title = f'Recommended Params (curr=20000, step=1000) | {hospital}'
                safe_name = str(hospital).replace('/', '_').replace(' ', '_')
                out_path = out_dir_h / f'recommended_params_{safe_name}.png'
                _save_table_image(df_h, title, out_path)
        


### Ground Truth Definition

To reduce random fluctuation, we defined a stable reference ("ground truth") for each metric using large current-window sizes between **20,000** and **40,000** exams. For each AI system and metric, ground truth was computed as the mean of windowed estimates within this 20k-40k band.

This ground-truth reference was used only to quantify noise and support parameter selection (e.g., RMS-vs-window and knee analysis). It was **not** used for online monitoring or alert generation. Online detection used the operational monitoring windows (**current window = 20,000 exams; step size = 1,000 exams**).

We selected **20,000 exams** as the minimum operational current window based on RMS/knee stability across metrics.



In [ ]:
# Part 3B: Recalibration evaluation - Method A (knee-based window)
# One window size per metric (rounded up to nearest 1,000), same for all AIs/hospitals.

import numpy as np
import pandas as pd

RECAL_STEP_SIZE = 1000
EXCLUDE_DANDERYDS = True
FAST_MODE = True
MAX_HOSPITALS = 5 if FAST_MODE else None  # set to None for full run
MANUFACTURER_WHITELIST = None  # e.g., ['Hologic']

if 'df_part3' not in globals() or 'build_part3_recalibration_summary' not in globals():
    print('Missing df_part3 or build_part3_recalibration_summary. Run Part 3 setup cells first.')
else:
    # Metrics used for recalibration summary
    if 'PART3_RECAL_METRICS' in globals():
        RECAL_METRICS = PART3_RECAL_METRICS
    else:
        RECAL_METRICS = ['flagging_rate', 'relative_flagging_rate', 'p_ai_given_rad', 'p_rad_given_ai']

    # Helper: round up to nearest 1,000
    def _round_up_1000(x):
        if x is None or pd.isna(x):
            return np.nan
        return int(np.ceil(float(x) / 1000.0) * 1000)

    # Knee-based window per metric
    knee_sizes = {}
    if 'knee_df' in globals() and knee_df is not None and len(knee_df) > 0:
        for m in RECAL_METRICS:
            if m in knee_df['metric'].unique():
                base = knee_df[knee_df['metric'] == m]['knee_curr_size'].median()
            else:
                base = knee_df[knee_df['metric'] == 'flagging_rate']['knee_curr_size'].median() if 'flagging_rate' in knee_df['metric'].unique() else 20000
            knee_sizes[m] = _round_up_1000(base)
    else:
        knee_sizes = {m: 20000 for m in RECAL_METRICS}

    # Subset for speed
    df_subset = df_part3.copy()
    if EXCLUDE_DANDERYDS:
        df_subset = df_subset[~df_subset[HOSPITAL_COL].str.contains('danderyd', case=False, na=False)]
    if MANUFACTURER_WHITELIST and MANUFACTURER_COL in df_subset.columns:
        df_subset = df_subset[df_subset[MANUFACTURER_COL].isin(MANUFACTURER_WHITELIST)]
    if MAX_HOSPITALS is not None:
        keep = sorted(df_subset[HOSPITAL_COL].dropna().unique())[:MAX_HOSPITALS]
        df_subset = df_subset[df_subset[HOSPITAL_COL].isin(keep)]

    # Save original globals to restore
    orig_curr = globals().get('PART3_CURR_SIZE', None)
    orig_step = globals().get('PART3_STEP_SIZE', None)
    orig_use_score = globals().get('PART3_RECAL_USE_SCORE_SIGNALS', None)
    orig_eval = globals().get('PART3_EVAL_CURR_SIZE', None)
    PART3_EVAL_CURR_SIZE = 20000


    PART3_RECAL_USE_SCORE_SIGNALS = False  # speed: skip score permutations

    all_rows = []
    for scenario in PART3_SCENARIOS:
        for metric in RECAL_METRICS:
            PART3_CURR_SIZE = knee_sizes.get(metric, 20000)
            PART3_STEP_SIZE = RECAL_STEP_SIZE

            df_summary = build_part3_recalibration_summary(
                df_subset, part3_start,
                calibration_strategy=PART3_CALIBRATION_STRATEGY,
                metrics=[metric],
                rad_mode=scenario.get('rad_mode'),
                calib_target_mode=scenario.get('target_mode'),
                system_mode=scenario.get('system_mode'),
                scenario_label=scenario.get('label')
            )
            if df_summary is None or len(df_summary) == 0:
                continue
            df_summary = df_summary.copy()
            df_summary['window_method'] = 'knee'
            df_summary['metric_window_size'] = PART3_CURR_SIZE
            all_rows.append(df_summary)

    # Restore globals
    if orig_curr is not None:
        PART3_CURR_SIZE = orig_curr
    if orig_step is not None:
        PART3_STEP_SIZE = orig_step
    if orig_use_score is not None:
        PART3_RECAL_USE_SCORE_SIGNALS = orig_use_score
    if orig_eval is not None:
        PART3_EVAL_CURR_SIZE = orig_eval

    if len(all_rows) == 0:
        print('No recalibration summaries produced. Check inputs.')
    else:
        recalib_knee = pd.concat(all_rows, ignore_index=True)
        display(recalib_knee)

        if 'metric' in recalib_knee.columns:
            recalib_knee_summary = (
                recalib_knee
                .groupby(['metric'])
                .mean(numeric_only=True)
                .reset_index()
            )
            display(recalib_knee_summary)



In [ ]:
# Part 3B: Save Method A (knee) recalibration tables per hospital
# Requires recalib_knee from Method A cell.

import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

if 'recalib_knee' not in globals() or recalib_knee is None or len(recalib_knee) == 0:
    print('recalib_knee not found. Run Method A cell first.')
else:
    recalib_df = recalib_knee.copy()
    # Keep only key columns for presentation
    keep_cols = [
        'scenario', 'hospital', 'manufacturer', 'ai_system', 'signal_metric',
        'window_method', 'metric_window_size', 'recal_status', 'first_signal_center', 'signal_count',
        'post_sens_base', 'post_sens_recal', 'sens_improvement',
        'post_spec_base', 'post_spec_recal', 'spec_improvement', 'overall_improvement'
    ]
    keep_cols = [c for c in keep_cols if c in recalib_df.columns]
    if len(keep_cols) > 0:
        recalib_df = recalib_df[keep_cols].copy()

    out_dir = Path('outputs/2025_12_29_counterfactual_manufacturer_swap_setup/recalib_by_hospital_knee')
    out_dir.mkdir(parents=True, exist_ok=True)

    def _save_table_image(df, title, out_path, fmt='%.2f'):
        df_disp = df.copy()
        for c in df_disp.columns:
            if pd.api.types.is_numeric_dtype(df_disp[c]):
                df_disp[c] = df_disp[c].map(lambda v: '' if pd.isna(v) else (fmt % v))
        fig_h = 1.6 + 0.45 * len(df_disp)
        fig_w = max(12, 1.6 * len(df_disp.columns))
        fig, ax = plt.subplots(figsize=(fig_w, fig_h))
        ax.axis('off')
        table = ax.table(
            cellText=df_disp.values,
            colLabels=df_disp.columns,
            cellLoc='center',
            loc='center',
            bbox=[0, 0, 1, 1]
        )
        table.auto_set_font_size(False)
        table.set_fontsize(8)
        table.scale(1.1, 1.4)
        try:
            table.auto_set_column_width(col=list(range(len(df_disp.columns))))
        except Exception:
            pass
        header_color = '#4472C4'
        alt_row_color = '#E7E6E6'
        for j in range(len(df_disp.columns)):
            cell = table[(0, j)]
            cell.set_facecolor(header_color)
            cell.set_text_props(weight='bold', color='white')
        for r in range(1, len(df_disp) + 1):
            for j in range(len(df_disp.columns)):
                cell = table[(r, j)]
                cell.set_facecolor(alt_row_color if r % 2 == 0 else 'white')
        for cell in table.get_celld().values():
            cell.set_edgecolor('black')
            cell.set_linewidth(1.0)
        plt.title(title, fontsize=12, fontweight='bold', pad=12)
        plt.tight_layout()
        plt.savefig(out_path, dpi=300, bbox_inches='tight', facecolor='white')
        plt.show()
        plt.close(fig)
        print(f'Saved: {out_path}')

    group_col = 'hospital' if 'hospital' in recalib_df.columns else 'Hospital'
    for hospital, df_h in recalib_df.groupby(group_col):
        safe_name = str(hospital).replace('/', '_').replace(' ', '_')
        title = f'Recalibration Summary (Method A - knee) | {hospital}'
        out_path = out_dir / f'recalib_knee_{safe_name}.png'
        _save_table_image(df_h, title, out_path)



In [ ]:
# Part 3B: Recalibration evaluation - Method B (3-month window)
# One window size per hospital: average monthly volume * 3.

import numpy as np
import pandas as pd
import inspect

RECAL_STEP_SIZE = 1000
EXCLUDE_DANDERYDS = True
FAST_MODE = True
MAX_HOSPITALS = 5 if FAST_MODE else None  # set to None for full run
MANUFACTURER_WHITELIST = None  # e.g., ['Hologic']
ROUND_WINDOW_TO_1000 = True

if 'df_part3' not in globals() or 'build_part3_recalibration_summary' not in globals():
    print('Missing df_part3 or build_part3_recalibration_summary. Run Part 3 setup cells first.')
else:
    if 'PART3_RECAL_METRICS' in globals():
        RECAL_METRICS = PART3_RECAL_METRICS
    else:
        RECAL_METRICS = ['flagging_rate', 'relative_flagging_rate', 'p_ai_given_rad', 'p_rad_given_ai']

    def _round_up_1000(x):
        if x is None or pd.isna(x):
            return np.nan
        return int(np.ceil(float(x) / 1000.0) * 1000)

    # Subset for speed
    df_subset = df_part3.copy()
    if EXCLUDE_DANDERYDS:
        df_subset = df_subset[~df_subset[HOSPITAL_COL].str.contains('danderyd', case=False, na=False)]
    if MANUFACTURER_WHITELIST and MANUFACTURER_COL in df_subset.columns:
        df_subset = df_subset[df_subset[MANUFACTURER_COL].isin(MANUFACTURER_WHITELIST)]
    if MAX_HOSPITALS is not None:
        keep = sorted(df_subset[HOSPITAL_COL].dropna().unique())[:MAX_HOSPITALS]
        df_subset = df_subset[df_subset[HOSPITAL_COL].isin(keep)]

    # Compute 3-month window per hospital (avg monthly exams * 3)
    df_tmp = df_subset.copy()
    df_tmp[DATE_COL] = pd.to_datetime(df_tmp[DATE_COL], format='mixed')
    df_tmp['month_key'] = df_tmp[DATE_COL].dt.to_period('M').astype(str)

    three_month_by_hospital = {}
    for hospital, df_h in df_tmp.groupby(HOSPITAL_COL):
        monthly_counts = df_h.groupby('month_key').size()
        if len(monthly_counts) == 0:
            continue
        avg_monthly = monthly_counts.mean()
        win = avg_monthly * 3
        if ROUND_WINDOW_TO_1000:
            win = _round_up_1000(win)
        three_month_by_hospital[hospital] = int(win) if not pd.isna(win) else np.nan

    # Fallback if something missing
    if len(three_month_by_hospital) == 0:
        three_month_by_hospital = {h: 20000 for h in df_subset[HOSPITAL_COL].dropna().unique()}

    # Save original globals to restore
    orig_curr = globals().get('PART3_CURR_SIZE', None)
    orig_step = globals().get('PART3_STEP_SIZE', None)
    orig_use_score = globals().get('PART3_RECAL_USE_SCORE_SIGNALS', None)
    orig_eval = globals().get('PART3_EVAL_CURR_SIZE', None)
    PART3_EVAL_CURR_SIZE = 20000


    PART3_RECAL_USE_SCORE_SIGNALS = False  # speed: skip score permutations
    PART3_STEP_SIZE = RECAL_STEP_SIZE

    all_rows = []

    # Check whether the function supports per-hospital window mapping
    supports_mapping = False
    try:
        sig = inspect.signature(build_part3_recalibration_summary)
        supports_mapping = 'curr_size_by_hospital' in sig.parameters
    except Exception:
        supports_mapping = False

    for scenario in PART3_SCENARIOS:
        if supports_mapping:
            df_summary = build_part3_recalibration_summary(
                df_subset, part3_start,
                calibration_strategy=PART3_CALIBRATION_STRATEGY,
                metrics=RECAL_METRICS,
                rad_mode=scenario.get('rad_mode'),
                calib_target_mode=scenario.get('target_mode'),
                system_mode=scenario.get('system_mode'),
                scenario_label=scenario.get('label'),
                curr_size_by_hospital=three_month_by_hospital
            )
            if df_summary is None or len(df_summary) == 0:
                continue
            df_summary = df_summary.copy()
            df_summary['window_method'] = '3_months'
            df_summary['metric_window_size'] = df_summary['hospital'].map(three_month_by_hospital)
            all_rows.append(df_summary)
        else:
            # Fallback: run per hospital with PART3_CURR_SIZE set per hospital
            for hospital, win in three_month_by_hospital.items():
                PART3_CURR_SIZE = win
                df_hosp = df_subset[df_subset[HOSPITAL_COL] == hospital].copy()
                if len(df_hosp) == 0:
                    continue
                df_summary = build_part3_recalibration_summary(
                    df_hosp, part3_start,
                    calibration_strategy=PART3_CALIBRATION_STRATEGY,
                    metrics=RECAL_METRICS,
                    rad_mode=scenario.get('rad_mode'),
                    calib_target_mode=scenario.get('target_mode'),
                    system_mode=scenario.get('system_mode'),
                    scenario_label=scenario.get('label')
                )
                if df_summary is None or len(df_summary) == 0:
                    continue
                df_summary = df_summary.copy()
                df_summary['window_method'] = '3_months'
                df_summary['metric_window_size'] = win
                all_rows.append(df_summary)

    # Restore globals
    if orig_curr is not None:
        PART3_CURR_SIZE = orig_curr
    if orig_step is not None:
        PART3_STEP_SIZE = orig_step
    if orig_use_score is not None:
        PART3_RECAL_USE_SCORE_SIGNALS = orig_use_score
    if orig_eval is not None:
        PART3_EVAL_CURR_SIZE = orig_eval

    if len(all_rows) == 0:
        print('No recalibration summaries produced. Check inputs.')
    else:
        recalib_3m = pd.concat(all_rows, ignore_index=True)
        display(recalib_3m)

        if 'metric' in recalib_3m.columns:
            recalib_3m_summary = (
                recalib_3m
                .groupby(['metric'])
                .mean(numeric_only=True)
                .reset_index()
            )
            display(recalib_3m_summary)


In [ ]:
# Part 3B: 3-month window size per hospital (avg monthly exams * 3)
# Requires three_month_by_hospital from Method B cell.

import pandas as pd
from IPython.display import display

if 'three_month_by_hospital' not in globals() or len(three_month_by_hospital) == 0:
    print('three_month_by_hospital not found. Run the Method B cell first.')
else:
    win_df = pd.DataFrame({
        'Hospital': list(three_month_by_hospital.keys()),
        '3-month window (exams)': list(three_month_by_hospital.values())
    }).sort_values('Hospital')
    display(win_df)


In [ ]:
# Part 3B: Save recalibration summary tables per hospital (one image each)
# Uses recalib_3m or recalib_knee if available.

import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Select the latest recalibration table
recalib_df = None
if 'recalib_3m' in globals():
    recalib_df = recalib_3m
elif 'recalib_knee' in globals():
    recalib_df = recalib_knee
elif 'recalib_compare' in globals():
    recalib_df = recalib_compare

if recalib_df is None or len(recalib_df) == 0:
    print('No recalibration table found (recalib_3m / recalib_knee / recalib_compare). Run the recalibration cell first.')
else:
    # Keep only key columns for presentation
    keep_cols = [
        'scenario', 'hospital', 'manufacturer', 'ai_system', 'signal_metric',
        'window_method', 'metric_window_size', 'recal_status', 'first_signal_center', 'signal_count',
        'post_sens_base', 'post_sens_recal', 'sens_improvement',
        'post_spec_base', 'post_spec_recal', 'spec_improvement', 'overall_improvement'
    ]
    keep_cols = [c for c in keep_cols if c in recalib_df.columns]
    if len(keep_cols) > 0:
        recalib_df = recalib_df[keep_cols].copy()

    out_dir = Path('outputs/2025_12_29_counterfactual_manufacturer_swap_setup/recalib_by_hospital')
    out_dir.mkdir(parents=True, exist_ok=True)

    def _save_table_image(df, title, out_path, fmt='%.2f'):
        df_disp = df.copy()
        for c in df_disp.columns:
            if pd.api.types.is_numeric_dtype(df_disp[c]):
                df_disp[c] = df_disp[c].map(lambda v: '' if pd.isna(v) else (fmt % v))
        fig_h = 1.6 + 0.45 * len(df_disp)
        fig_w = max(12, 1.6 * len(df_disp.columns))
        fig, ax = plt.subplots(figsize=(fig_w, fig_h))
        ax.axis('off')
        table = ax.table(
            cellText=df_disp.values,
            colLabels=df_disp.columns,
            cellLoc='center',
            loc='center',
            bbox=[0, 0, 1, 1]
        )
        table.auto_set_font_size(False)
        table.set_fontsize(8)
        table.scale(1.1, 1.4)
        try:
            table.auto_set_column_width(col=list(range(len(df_disp.columns))))
        except Exception:
            pass
        header_color = '#4472C4'
        alt_row_color = '#E7E6E6'
        for j in range(len(df_disp.columns)):
            cell = table[(0, j)]
            cell.set_facecolor(header_color)
            cell.set_text_props(weight='bold', color='white')
        for r in range(1, len(df_disp) + 1):
            for j in range(len(df_disp.columns)):
                cell = table[(r, j)]
                cell.set_facecolor(alt_row_color if r % 2 == 0 else 'white')
        for cell in table.get_celld().values():
            cell.set_edgecolor('black')
            cell.set_linewidth(1.0)
        plt.title(title, fontsize=12, fontweight='bold', pad=12)
        plt.tight_layout()
        plt.savefig(out_path, dpi=300, bbox_inches='tight', facecolor='white')
        plt.show()
        plt.close(fig)
        print(f'Saved: {out_path}')

    group_col = 'hospital' if 'hospital' in recalib_df.columns else 'Hospital'
    for hospital, df_h in recalib_df.groupby(group_col):
        safe_name = str(hospital).replace('/', '_').replace(' ', '_')
        title = f'Recalibration Summary | {hospital}'
        out_path = out_dir / f'recalib_summary_{safe_name}.png'
        _save_table_image(df_h, title, out_path)



In [ ]:
# Part 3B: Recalibration comparison summary tables (Method A vs Method B)
# Computes per-hospital deltas vs target, then summarizes across hospitals.
# Requires recalib_knee and/or recalib_3m from prior cells.

import pandas as pd

summaries = []

# Determine metric column name (metric vs signal_metric)
def _metric_col(df):
    if 'metric' in df.columns:
        return 'metric'
    if 'signal_metric' in df.columns:
        return 'signal_metric'
    return None

# Add per-row deltas vs target (base and recal)
def _add_target_deltas(df):
    df = df.copy()
    required = [
        'target_sensitivity', 'target_specificity',
        'post_sens_base', 'post_sens_recal',
        'post_spec_base', 'post_spec_recal'
    ]
    missing = [c for c in required if c not in df.columns]
    if missing:
        print('Missing target or post columns for delta computation:', missing)
        return df
    df['delta_sens_base'] = df['post_sens_base'] - df['target_sensitivity']
    df['delta_sens_recal'] = df['post_sens_recal'] - df['target_sensitivity']
    df['delta_spec_base'] = df['post_spec_base'] - df['target_specificity']
    df['delta_spec_recal'] = df['post_spec_recal'] - df['target_specificity']
    return df

# Column display names
COL_MAP = {
    'metric': 'Metric',
    'window_method': 'Window method',
    'metric_window_size_mean': 'Window size (exams) mean',
    'post_sens_base_mean': 'Sens (base) mean',
    'post_sens_recal_mean': 'Sens (recal) mean',
    'post_spec_base_mean': 'Spec (base) mean',
    'post_spec_recal_mean': 'Spec (recal) mean',
    'recal_point_count_mean': 'Avg threshold changes',
    'delta_sens_base_min': 'Δ sens (base) min',
    'delta_sens_base_max': 'Δ sens (base) max',
    'delta_sens_recal_min': 'Δ sens (recal) min',
    'delta_sens_recal_max': 'Δ sens (recal) max',
    'delta_spec_base_min': 'Δ spec (base) min',
    'delta_spec_base_max': 'Δ spec (base) max',
    'delta_spec_recal_min': 'Δ spec (recal) min',
    'delta_spec_recal_max': 'Δ spec (recal) max'
}

# Format metric labels (remove underscores)
def _format_metric_labels(df):
    df = df.copy()
    if 'Metric' in df.columns:
        df['Metric'] = df['Metric'].astype(str).str.replace('_', ' ', regex=False)
    return df

# Build summary with mean for base/recal and min/max for deltas
def _summarize(df, method_label, mcol):
    df = _add_target_deltas(df)
    mean_cols = ['metric_window_size', 'post_sens_base', 'post_sens_recal', 'post_spec_base', 'post_spec_recal']
    delta_cols = ['delta_sens_base', 'delta_sens_recal', 'delta_spec_base', 'delta_spec_recal']

    agg_kwargs = {}
    for c in mean_cols:
        if c in df.columns:
            agg_kwargs[f'{c}_mean'] = (c, 'mean')
    for c in delta_cols:
        if c in df.columns:
            agg_kwargs[f'{c}_min'] = (c, 'min')
            agg_kwargs[f'{c}_max'] = (c, 'max')

    summary = (
        df.groupby([mcol])
          .agg(**agg_kwargs)
          .reset_index()
    )
    summary['window_method'] = method_label
    if mcol != 'metric':
        summary = summary.rename(columns={mcol: 'metric'})

    # Reorder columns
    ordered_cols = [
        'metric', 'window_method',
        'metric_window_size_mean',
        'post_sens_base_mean', 'post_sens_recal_mean',
        'post_spec_base_mean', 'post_spec_recal_mean',
        'delta_sens_base_min', 'delta_sens_base_max',
        'delta_sens_recal_min', 'delta_sens_recal_max',
        'delta_spec_base_min', 'delta_spec_base_max',
        'delta_spec_recal_min', 'delta_spec_recal_max'
    ]
    summary = summary[[c for c in ordered_cols if c in summary.columns]]
    summary = summary.rename(columns=COL_MAP)
    summary = _format_metric_labels(summary)
    return summary

if 'recalib_knee' in globals() and recalib_knee is not None and len(recalib_knee) > 0:
    mcol = _metric_col(recalib_knee)
    if mcol is None:
        print('recalib_knee has no metric column')
    else:
        knee_summary = _summarize(recalib_knee, 'knee', mcol)
        summaries.append(knee_summary)
        display(knee_summary)
else:
    print('recalib_knee not available')

if 'recalib_1w' in globals() and recalib_1w is not None and len(recalib_1w) > 0:
    mcol = _metric_col(recalib_1w)
    if mcol is None:
        print('recalib_1w has no metric column')
    else:
        w1_summary = _summarize(recalib_1w, '1_week', mcol)
        summaries.append(w1_summary)
        display(w1_summary)
else:
    print('recalib_1w not available')

if 'recalib_3m' in globals() and recalib_3m is not None and len(recalib_3m) > 0:
    mcol = _metric_col(recalib_3m)
    if mcol is None:
        print('recalib_3m has no metric column')
    else:
        m3_summary = _summarize(recalib_3m, '3_months', mcol)
        summaries.append(m3_summary)
        display(m3_summary)
else:
    print('recalib_3m not available')

if len(summaries) > 0:
    compare_summary = pd.concat(summaries, ignore_index=True)
    display(compare_summary)


In [ ]:
# Part 3B: Compact interval table for deltas (min–max as intervals)
# Requires compare_summary from the summary cell.

import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

if 'compare_summary' not in globals() or compare_summary is None or len(compare_summary) == 0:
    print('compare_summary not found. Run the summary cell first.')
else:
    df = compare_summary.copy()

    # Helper to format signed numbers
    def _fmt(v):
        return '' if pd.isna(v) else f'{v:+.2f}'

    # Build interval columns from min/max
    def _interval(min_col, max_col):
        if min_col not in df.columns or max_col not in df.columns:
            return None
        return df[min_col].map(_fmt) + ' to ' + df[max_col].map(_fmt)

    df['Δ sens (base)'] = _interval('Δ sens (base) min', 'Δ sens (base) max')
    df['Δ sens (recal)'] = _interval('Δ sens (recal) min', 'Δ sens (recal) max')
    df['Δ spec (base)'] = _interval('Δ spec (base) min', 'Δ spec (base) max')
    df['Δ spec (recal)'] = _interval('Δ spec (recal) min', 'Δ spec (recal) max')

    # Keep compact columns
    keep = [
        'Metric', 'Window method', 'Window size (exams) mean',
        'Sens (base) mean', 'Sens (recal) mean',
        'Spec (base) mean', 'Spec (recal) mean',
        'Avg threshold changes',
        'Δ sens (base)', 'Δ sens (recal)',
        'Δ spec (base)', 'Δ spec (recal)'
    ]
    keep = [c for c in keep if c in df.columns]
    compact_df = df[keep].copy()
    # Format numeric columns to 2 decimals
    for col in compact_df.columns:
        if pd.api.types.is_numeric_dtype(compact_df[col]):
            compact_df[col] = compact_df[col].map(lambda v: '' if pd.isna(v) else f'{v:.2f}')


    display(compact_df)

    # Save image
    out_dir = Path('outputs/2025_12_29_counterfactual_manufacturer_swap_setup/recalib_summary_tables')
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / 'recalib_summary_intervals.png'

    fig_h = 1.6 + 0.45 * len(compact_df)
    fig_w = max(12, 1.6 * len(compact_df.columns))
    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    ax.axis('off')

    table = ax.table(
        cellText=compact_df.values,
        colLabels=compact_df.columns,
        cellLoc='center',
        loc='center',
        bbox=[0, 0, 1, 1]
    )
    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.scale(1.2, 1.6)

    header_color = '#4472C4'
    alt_row_color = '#E7E6E6'
    for j in range(len(compact_df.columns)):
        cell = table[(0, j)]
        cell.set_facecolor(header_color)
        cell.set_text_props(weight='bold', color='white')
    for r in range(1, len(compact_df) + 1):
        for j in range(len(compact_df.columns)):
            cell = table[(r, j)]
            cell.set_facecolor(alt_row_color if r % 2 == 0 else 'white')
    for cell in table.get_celld().values():
        cell.set_edgecolor('black')
        cell.set_linewidth(1.1)

    plt.title('Recalibration Summary (A vs B) – Interval Deltas', fontsize=14, fontweight='bold', pad=16)
    plt.tight_layout()
    plt.savefig(out_path, dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()
    plt.close(fig)
    print(f'Saved interval table image to: {out_path}')


In [ ]:
# Part 3B: Save recalibration comparison summaries as images
# Requires knee_summary / m3_summary / compare_summary from the summary cell.

import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

out_dir = Path('outputs/2025_12_29_counterfactual_manufacturer_swap_setup/recalib_summary_tables')
out_dir.mkdir(parents=True, exist_ok=True)


def _save_table_image(df, title, out_path, fmt='%.2f'):
    df_disp = df.copy()
    # Add + sign for delta columns
    delta_cols = [c for c in df_disp.columns if ('Delta' in c) or ('Δ' in c)]
    for c in delta_cols:
        df_disp[c] = df_disp[c].map(lambda v: '' if pd.isna(v) else (f'{v:+.2f}'))
    for col in df_disp.columns:
        if pd.api.types.is_numeric_dtype(df_disp[col]):
            if col in delta_cols:
                continue
            df_disp[col] = df_disp[col].map(lambda v: '' if pd.isna(v) else (fmt % v))
    fig_h = 1.6 + 0.45 * len(df_disp)
    fig_w = max(12, 1.6 * len(df_disp.columns))
    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    ax.axis('off')
    table = ax.table(
        cellText=df_disp.values,
        colLabels=df_disp.columns,
        cellLoc='center',
        loc='center',
        bbox=[0, 0, 1, 1]
    )
    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.scale(1.2, 1.6)
    try:
        table.auto_set_column_width(col=list(range(len(df_disp.columns))))
    except Exception:
        pass
    header_color = '#4472C4'
    alt_row_color = '#E7E6E6'
    for j in range(len(df_disp.columns)):
        cell = table[(0, j)]
        cell.set_facecolor(header_color)
        cell.set_text_props(weight='bold', color='white')
    for r in range(1, len(df_disp) + 1):
        for j in range(len(df_disp.columns)):
            cell = table[(r, j)]
            cell.set_facecolor(alt_row_color if r % 2 == 0 else 'white')
    for cell in table.get_celld().values():
        cell.set_edgecolor('black')
        cell.set_linewidth(1.0)
    plt.title(title, fontsize=12, fontweight='bold', pad=12)
    plt.tight_layout()
    plt.savefig(out_path, dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()
    plt.close(fig)
    print(f'Saved: {out_path}')

if 'knee_summary' in globals():
    knee_summary = knee_summary.rename(columns={'metric': 'Metric', 'window_method': 'Window method', 'metric_window_size': 'Window size (exams)', 'post_sens_base': 'Post sens (base)', 'post_sens_recal': 'Post sens (recal)', 'sens_improvement': 'Sens improvement', 'post_spec_base': 'Post spec (base)', 'post_spec_recal': 'Post spec (recal)', 'spec_improvement': 'Spec improvement', 'overall_improvement': 'Overall improvement'})
    _save_table_image(knee_summary, 'Recalibration Summary (Method A - knee)', out_dir / 'recalib_summary_knee.png')
else:
    print('knee_summary not found')

if 'm3_summary' in globals():
    m3_summary = m3_summary.rename(columns={'metric': 'Metric', 'window_method': 'Window method', 'metric_window_size': 'Window size (exams)', 'post_sens_base': 'Post sens (base)', 'post_sens_recal': 'Post sens (recal)', 'sens_improvement': 'Sens improvement', 'post_spec_base': 'Post spec (base)', 'post_spec_recal': 'Post spec (recal)', 'spec_improvement': 'Spec improvement', 'overall_improvement': 'Overall improvement'})
    _save_table_image(m3_summary, 'Recalibration Summary (Method B - 3 months)', out_dir / 'recalib_summary_3months.png')
else:
    print('m3_summary not found')

if 'compare_summary' in globals():
    compare_summary = compare_summary.rename(columns={'metric': 'Metric', 'window_method': 'Window method', 'metric_window_size': 'Window size (exams)', 'post_sens_base': 'Post sens (base)', 'post_sens_recal': 'Post sens (recal)', 'sens_improvement': 'Sens improvement', 'post_spec_base': 'Post spec (base)', 'post_spec_recal': 'Post spec (recal)', 'spec_improvement': 'Spec improvement', 'overall_improvement': 'Overall improvement'})
    _save_table_image(compare_summary, 'Recalibration Summary (A vs B)', out_dir / 'recalib_summary_compare.png')
else:
    print('compare_summary not found')





In [ ]:
# Part 3B: Recalibration evaluation - Method C (1-week window)
# One window size per hospital: average weekly volume * 1 (no rounding).

import numpy as np
import pandas as pd
import inspect

RECAL_STEP_SIZE = 1000
EXCLUDE_DANDERYDS = True
FAST_MODE = True
MAX_HOSPITALS = 5 if FAST_MODE else None  # set to None for full run
MANUFACTURER_WHITELIST = None  # e.g., ['Hologic']

if 'df_part3' not in globals() or 'build_part3_recalibration_summary' not in globals():
    print('Missing df_part3 or build_part3_recalibration_summary. Run Part 3 setup cells first.')
else:
    if 'PART3_RECAL_METRICS' in globals():
        RECAL_METRICS = PART3_RECAL_METRICS
    else:
        RECAL_METRICS = ['flagging_rate', 'relative_flagging_rate', 'p_ai_given_rad', 'p_rad_given_ai']

    # Subset for speed
    df_subset = df_part3.copy()
    if EXCLUDE_DANDERYDS:
        df_subset = df_subset[~df_subset[HOSPITAL_COL].str.contains('danderyd', case=False, na=False)]
    if MANUFACTURER_WHITELIST and MANUFACTURER_COL in df_subset.columns:
        df_subset = df_subset[df_subset[MANUFACTURER_COL].isin(MANUFACTURER_WHITELIST)]
    if MAX_HOSPITALS is not None:
        keep = sorted(df_subset[HOSPITAL_COL].dropna().unique())[:MAX_HOSPITALS]
        df_subset = df_subset[df_subset[HOSPITAL_COL].isin(keep)]

    # Compute 1-week window per hospital (avg weekly exams)
    df_tmp = df_subset.copy()
    df_tmp[DATE_COL] = pd.to_datetime(df_tmp[DATE_COL], format='mixed')
    df_tmp['week_key'] = df_tmp[DATE_COL].dt.to_period('W').astype(str)

    one_week_by_hospital = {}
    for hospital, df_h in df_tmp.groupby(HOSPITAL_COL):
        weekly_counts = df_h.groupby('week_key').size()
        if len(weekly_counts) == 0:
            continue
        avg_weekly = weekly_counts.mean()
        one_week_by_hospital[hospital] = float(avg_weekly)

    if len(one_week_by_hospital) == 0:
        one_week_by_hospital = {h: 2000 for h in df_subset[HOSPITAL_COL].dropna().unique()}

    # Save original globals to restore
    orig_curr = globals().get('PART3_CURR_SIZE', None)
    orig_step = globals().get('PART3_STEP_SIZE', None)
    orig_use_score = globals().get('PART3_RECAL_USE_SCORE_SIGNALS', None)
    orig_eval = globals().get('PART3_EVAL_CURR_SIZE', None)

    PART3_RECAL_USE_SCORE_SIGNALS = False
    PART3_STEP_SIZE = RECAL_STEP_SIZE
    PART3_EVAL_CURR_SIZE = 20000

    all_rows = []

    # Check whether the function supports per-hospital window mapping
    supports_mapping = False
    try:
        sig = inspect.signature(build_part3_recalibration_summary)
        supports_mapping = 'curr_size_by_hospital' in sig.parameters
    except Exception:
        supports_mapping = False

    for scenario in PART3_SCENARIOS:
        if supports_mapping:
            df_summary = build_part3_recalibration_summary(
                df_subset, part3_start,
                calibration_strategy=PART3_CALIBRATION_STRATEGY,
                metrics=RECAL_METRICS,
                rad_mode=scenario.get('rad_mode'),
                calib_target_mode=scenario.get('target_mode'),
                system_mode=scenario.get('system_mode'),
                scenario_label=scenario.get('label'),
                curr_size_by_hospital=one_week_by_hospital
            )
            if df_summary is None or len(df_summary) == 0:
                continue
            df_summary = df_summary.copy()
            df_summary['window_method'] = '1_week'
            df_summary['metric_window_size'] = df_summary['hospital'].map(one_week_by_hospital)
            all_rows.append(df_summary)
        else:
            for hospital, win in one_week_by_hospital.items():
                PART3_CURR_SIZE = win
                df_hosp = df_subset[df_subset[HOSPITAL_COL] == hospital].copy()
                if len(df_hosp) == 0:
                    continue
                df_summary = build_part3_recalibration_summary(
                    df_hosp, part3_start,
                    calibration_strategy=PART3_CALIBRATION_STRATEGY,
                    metrics=RECAL_METRICS,
                    rad_mode=scenario.get('rad_mode'),
                    calib_target_mode=scenario.get('target_mode'),
                    system_mode=scenario.get('system_mode'),
                    scenario_label=scenario.get('label')
                )
                if df_summary is None or len(df_summary) == 0:
                    continue
                df_summary = df_summary.copy()
                df_summary['window_method'] = '1_week'
                df_summary['metric_window_size'] = win
                all_rows.append(df_summary)

    # Restore globals
    if orig_curr is not None:
        PART3_CURR_SIZE = orig_curr
    if orig_step is not None:
        PART3_STEP_SIZE = orig_step
    if orig_use_score is not None:
        PART3_RECAL_USE_SCORE_SIGNALS = orig_use_score
    if orig_eval is not None:
        PART3_EVAL_CURR_SIZE = orig_eval

    if len(all_rows) == 0:
        print('No recalibration summaries produced. Check inputs.')
    else:
        recalib_1w = pd.concat(all_rows, ignore_index=True)
        display(recalib_1w)


In [ ]:
# Debug: radiologist name columns per hospital (r1_name/r2_name)
import pandas as pd

if 'df_part3' not in globals():
    print('df_part3 not found')
else:
    cols = [c for c in ['r1_name', 'r2_name', 'r1', 'r2'] if c in df_part3.columns]
    if len(cols) == 0:
        print('No r1_name/r2_name/r1/r2 columns found in df_part3')
    else:
        rows = []
        for hospital, df_h in df_part3.groupby(HOSPITAL_COL):
            row = {'Hospital': hospital}
            for c in cols:
                vals = df_h[c].dropna().astype(str).str.strip()
                vals = vals[~vals.str.lower().isin(['', 'nan', 'none', 'unknown', 'discussion'])]
                row[f'unique_{c}'] = vals.nunique()
            rows.append(row)
        dbg = pd.DataFrame(rows).sort_values('Hospital')
        display(dbg)


In [ ]:
# Part 3B: Hospital descriptive table

import pandas as pd
from IPython.display import display

if 'df_part3' not in globals():
    print('df_part3 not found.')
else:
    df_tmp = df_part3.copy()
    df_tmp[DATE_COL] = pd.to_datetime(df_tmp[DATE_COL], format='mixed')

    def _count_radiologists(df_h):
        # If name columns exist, use them exclusively
        name_cols = [c for c in ['r1_name', 'r2_name'] if c in df_h.columns]
        if len(name_cols) > 0:
            vals = []
            for c in name_cols:
                series = df_h[c].dropna().astype(str).str.strip()
                vals.extend(series.tolist())
            bad = {'', 'nan', 'none', 'unknown', 'discussion'}
            cleaned = [v.strip() for v in vals if v.strip().lower() not in bad]
            uniq = sorted(set(cleaned))
            return len(uniq) if len(uniq) > 0 else None

        # Fallbacks: any radiologist/reader name/id columns
        cand_cols = []
        for c in df_h.columns:
            cl = c.lower()
            if 'rad_flagged' in cl:
                continue
            if any(tok in cl for tok in ['radiologist', 'reader', 'rad']) and any(tok in cl for tok in ['name', 'id']):
                cand_cols.append(c)

        # Always include r1/r2 if present (even if numeric codes)
        if len(cand_cols) == 0:
            for c in ['r1', 'r2']:
                if c in df_h.columns:
                    cand_cols.append(c)

        if len(cand_cols) == 0:
            return None

        vals = []
        for c in cand_cols:
            series = df_h[c].dropna().astype(str).str.strip()
            vals.extend(series.tolist())

        bad = {'', 'nan', 'none', 'unknown', 'discussion'}
        cleaned = [v.strip() for v in vals if v.strip().lower() not in bad]
        uniq = sorted(set(cleaned))
        return len(uniq) if len(uniq) > 0 else None

    rows = []
    for hospital, df_h in df_tmp.groupby(HOSPITAL_COL):
        start_date = df_h[DATE_COL].min()
        end_date = df_h[DATE_COL].max()
        total_exams = len(df_h)
        if MANUFACTURER_COL in df_h.columns:
            manufacturers = sorted(df_h[MANUFACTURER_COL].dropna().unique())
            manufacturer = ', '.join([str(m) for m in manufacturers]) if manufacturers else 'Unknown'
        else:
            manufacturer = 'Unknown'
        n_rads = _count_radiologists(df_h)
        rows.append({
            'Hospital': hospital,
            'Start date': start_date.date() if pd.notna(start_date) else None,
            'End date': end_date.date() if pd.notna(end_date) else None,
            'Mammography equipment': manufacturer,
            'Total exams': total_exams,
            'Number of radiologists': n_rads
        })

    desc_df = pd.DataFrame(rows).sort_values('Hospital')
    display(desc_df)


In [ ]:
# Debug: radiologist name columns per hospital (r1_name/r2_name)
import pandas as pd

if 'df_part3' not in globals():
    print('df_part3 not found')
else:
    cols = [c for c in ['r1_name', 'r2_name', 'r1', 'r2'] if c in df_part3.columns]
    if len(cols) == 0:
        print('No r1_name/r2_name/r1/r2 columns found in df_part3')
    else:
        rows = []
        for hospital, df_h in df_part3.groupby(HOSPITAL_COL):
            row = {'Hospital': hospital}
            for c in cols:
                vals = df_h[c].dropna().astype(str).str.strip()
                vals = vals[~vals.str.lower().isin(['', 'nan', 'none', 'unknown', 'discussion'])]
                row[f'unique_{c}'] = vals.nunique()
            rows.append(row)
        dbg = pd.DataFrame(rows).sort_values('Hospital')
        display(dbg)


In [ ]:
# Part 3B: Save Method C (1-week) per-hospital tables + hospital descriptive table as images

import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

out_dir = Path('outputs/2025_12_29_counterfactual_manufacturer_swap_setup/recalib_1w_by_hospital')
out_dir.mkdir(parents=True, exist_ok=True)

# Helper to save a dataframe as a styled image

def _save_table_image(df, title, out_path, fmt='%.2f'):
    df_disp = df.copy()
    # Round metric window size to integer if present
    for c in df_disp.columns:
        if 'metric window size' in str(c).lower():
            df_disp[c] = pd.to_numeric(df_disp[c], errors='coerce').round(0)

    for col in df_disp.columns:
        if pd.api.types.is_numeric_dtype(df_disp[col]):
            df_disp[col] = df_disp[col].map(lambda v: '' if pd.isna(v) else (fmt % v))
    # Add + sign for delta columns
    delta_cols = [c for c in df_disp.columns if 'delta' in str(c).lower() or 'Δ' in str(c)]
    for c in delta_cols:
        if c in df_disp.columns:
            df_disp[c] = df_disp[c].map(lambda v: '' if pd.isna(v) else f'{float(v):+0.2f}')

    fig_h = 1.6 + 0.45 * len(df_disp)
    fig_w = max(12, 1.6 * len(df_disp.columns))
    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    ax.axis('off')
    table = ax.table(
        cellText=df_disp.values,
        colLabels=df_disp.columns,
        cellLoc='center',
        loc='center',
        bbox=[0, 0, 1, 1]
    )
    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.scale(1.2, 1.6)
    try:
        table.auto_set_column_width(col=list(range(len(df_disp.columns))))
    except Exception:
        pass
    header_color = '#4472C4'
    alt_row_color = '#E7E6E6'
    for j in range(len(df_disp.columns)):
        cell = table[(0, j)]
        cell.set_facecolor(header_color)
        cell.set_text_props(weight='bold', color='white')
    for r in range(1, len(df_disp) + 1):
        for j in range(len(df_disp.columns)):
            cell = table[(r, j)]
            cell.set_facecolor(alt_row_color if r % 2 == 0 else 'white')
    for cell in table.get_celld().values():
        cell.set_edgecolor('black')
        cell.set_linewidth(1.1)
    plt.title(title, fontsize=14, fontweight='bold', pad=16)
    plt.tight_layout()
    plt.savefig(out_path, dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()
    plt.close(fig)


# Replace underscores in column headers and common string fields

def _format_no_underscores(df):
    df = df.copy()
    df.columns = [str(c).replace('_', ' ') for c in df.columns]
# Replace 'post' prefix in column names
    df.columns = [c.replace('post sens base', 'sens base').replace('post spec base', 'spec base') for c in df.columns]
    # Replace 'delta' prefix in column names with Greek Delta
    df.columns = [c.replace('delta ', 'Δ ') for c in df.columns]

    # Replace underscores in common categorical columns if present
    for col in df.columns:
        if df[col].dtype == object:
            df[col] = df[col].astype(str).str.replace('_', ' ', regex=False)
    return df

# 1-week tables per hospital
if 'recalib_1w' in globals() and recalib_1w is not None and len(recalib_1w) > 0:
    # Keep key columns for readability
    keep_cols = [
        'scenario', 'hospital', 'manufacturer', 'ai_system', 'signal_metric',
        'window_method', 'metric_window_size', 'recal_status', 'recal_point_count',
        'post_sens_base', 'post_sens_recal', 'post_spec_base', 'post_spec_recal'
    ]
    keep_cols = [c for c in keep_cols if c in recalib_1w.columns]
    df_small = recalib_1w.copy()
    # Add delta columns if targets are present
    if 'target_sensitivity' in df_small.columns and 'target_specificity' in df_small.columns:
        if 'post_sens_base' in df_small.columns:
            df_small['delta_sens_base'] = df_small['post_sens_base'] - df_small['target_sensitivity']
        if 'post_sens_recal' in df_small.columns:
            df_small['delta_sens_recal'] = df_small['post_sens_recal'] - df_small['target_sensitivity']
        if 'post_spec_base' in df_small.columns:
            df_small['delta_spec_base'] = df_small['post_spec_base'] - df_small['target_specificity']
        if 'post_spec_recal' in df_small.columns:
            df_small['delta_spec_recal'] = df_small['post_spec_recal'] - df_small['target_specificity']

    keep_cols = [
        'scenario', 'hospital', 'ai_system', 'signal_metric',
        'metric_window_size', 'recal_point_count',
        'post_sens_base', 'post_spec_base',
        'delta_sens_base', 'delta_sens_recal', 'delta_spec_base', 'delta_spec_recal'
    ]
    keep_cols = [c for c in keep_cols if c in df_small.columns]
    df_small = df_small[keep_cols].copy()
    df_small = _format_no_underscores(df_small)

    for hospital, df_h in df_small.groupby('hospital'):
        out_path = out_dir / f'recalib_1w_{str(hospital).replace(" ", "_")}.png'
        _save_table_image(
            _format_no_underscores(df_h),
            f'Recalibration (1-week) – {hospital}',
            out_path
        )
else:
    print('recalib_1w not found. Run Method C cell first.')

# Hospital descriptive table image
if 'desc_df' in globals() and desc_df is not None and len(desc_df) > 0:
    desc_out = Path('outputs/2025_12_29_counterfactual_manufacturer_swap_setup/hospital_descriptive_table.png')
    _save_table_image(
        _format_no_underscores(desc_df),
        'Hospital Descriptive Table',
        desc_out
    )
else:
    print('desc_df not found. Run the hospital descriptive table cell first.')


In [ ]:
# Part 3B: Per-hospital window sizes (knee / 3-month / 1-week)
# Requires knee_sizes (method A), three_month_by_hospital (method B), one_week_by_hospital (method C).

import pandas as pd
from IPython.display import display

if 'three_month_by_hospital' not in globals() or 'one_week_by_hospital' not in globals():
    print('Run Method B and Method C cells first.')
else:
    # Knee sizes are per metric, not per hospital
    if 'knee_sizes' in globals() and isinstance(knee_sizes, dict):
        knee_str = ', '.join([f"{k}: {v}" for k, v in knee_sizes.items()])
    else:
        knee_str = 'not available'

    rows = []
    for hospital in sorted(set(list(three_month_by_hospital.keys()) + list(one_week_by_hospital.keys()))):
        rows.append({
            'Hospital': hospital,
            'Knee window (per metric)': knee_str,
            '3-month window (exams)': three_month_by_hospital.get(hospital),
            '1-week window (exams)': one_week_by_hospital.get(hospital)
        })

    win_all = pd.DataFrame(rows).sort_values('Hospital')
    display(win_all)


In [ ]:
# Part 3B: Per-hospital recalibration tables (all methods: knee/3-month/1-week)
# Uses the same columns as the 1-week per-hospital tables, but includes all methods.

import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

out_dir = Path('outputs/2025_12_29_counterfactual_manufacturer_swap_setup/recalib_all_methods_by_hospital')
out_dir.mkdir(parents=True, exist_ok=True)

# Helper to save a dataframe as a styled image

def _save_table_image(df, title, out_path, fmt='%.2f'):
    df_disp = df.copy()
    # Round metric window size to integer
    for c in df_disp.columns:
        if 'metric window size' in str(c).lower():
            df_disp[c] = pd.to_numeric(df_disp[c], errors='coerce').round(0)

    # Add + sign for delta columns
    delta_cols = [c for c in df_disp.columns if 'Δ' in str(c) or 'delta' in str(c).lower()]
    for c in delta_cols:
        if c in df_disp.columns:
            df_disp[c] = df_disp[c].map(lambda v: '' if pd.isna(v) else f'{float(v):+0.2f}')

    for col in df_disp.columns:
        if pd.api.types.is_numeric_dtype(df_disp[col]):
            if col in delta_cols:
                continue
            df_disp[col] = df_disp[col].map(lambda v: '' if pd.isna(v) else (fmt % v))

    fig_h = 1.6 + 0.45 * len(df_disp)
    fig_w = max(12, 1.6 * len(df_disp.columns))
    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    ax.axis('off')
    table = ax.table(
        cellText=df_disp.values,
        colLabels=df_disp.columns,
        cellLoc='center',
        loc='center',
        bbox=[0, 0, 1, 1]
    )
    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.scale(1.2, 1.6)
    try:
        table.auto_set_column_width(col=list(range(len(df_disp.columns))))
    except Exception:
        pass
    header_color = '#4472C4'
    alt_row_color = '#E7E6E6'
    for j in range(len(df_disp.columns)):
        cell = table[(0, j)]
        cell.set_facecolor(header_color)
        cell.set_text_props(weight='bold', color='white')
    for r in range(1, len(df_disp) + 1):
        for j in range(len(df_disp.columns)):
            cell = table[(r, j)]
            cell.set_facecolor(alt_row_color if r % 2 == 0 else 'white')
    for cell in table.get_celld().values():
        cell.set_edgecolor('black')
        cell.set_linewidth(1.1)
    plt.title(title, fontsize=14, fontweight='bold', pad=16)
    plt.tight_layout()
    plt.savefig(out_path, dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()
    plt.close(fig)

# Merge all methods if available
frames = []
for name in ['recalib_knee', 'recalib_3m', 'recalib_1w']:
    if name in globals() and globals()[name] is not None and len(globals()[name]) > 0:
        frames.append(globals()[name])

if len(frames) == 0:
    print('No recalibration tables found. Run Method A/B/C first.')
else:
    df_all = pd.concat(frames, ignore_index=True)

    # Add delta columns if targets present
    if 'target_sensitivity' in df_all.columns and 'target_specificity' in df_all.columns:
        if 'post_sens_base' in df_all.columns:
            df_all['delta_sens_base'] = df_all['post_sens_base'] - df_all['target_sensitivity']
        if 'post_sens_recal' in df_all.columns:
            df_all['delta_sens_recal'] = df_all['post_sens_recal'] - df_all['target_sensitivity']
        if 'post_spec_base' in df_all.columns:
            df_all['delta_spec_base'] = df_all['post_spec_base'] - df_all['target_specificity']
        if 'post_spec_recal' in df_all.columns:
            df_all['delta_spec_recal'] = df_all['post_spec_recal'] - df_all['target_specificity']

    # Keep columns similar to the 1-week tables
    keep_cols = [
        'scenario', 'hospital', 'ai_system', 'signal_metric',
        'window_method', 'metric_window_size', 'recal_point_count',
        'post_sens_base', 'post_spec_base',
        'delta_sens_base', 'delta_sens_recal',
        'delta_spec_base', 'delta_spec_recal'
    ]
    keep_cols = [c for c in keep_cols if c in df_all.columns]
    df_all = df_all[keep_cols].copy()

    # Clean labels (remove underscores, use Δ)
    df_all.columns = [str(c).replace('_', ' ') for c in df_all.columns]
    df_all.columns = [c.replace('post sens base', 'sens base').replace('post spec base', 'spec base') for c in df_all.columns]
    df_all.columns = [c.replace('delta ', 'Δ ') for c in df_all.columns]
    for col in df_all.columns:
        if df_all[col].dtype == object:
            df_all[col] = df_all[col].astype(str).str.replace('_', ' ', regex=False)

    for hospital, df_h in df_all.groupby('hospital'):
        for scenario, df_s in df_h.groupby('scenario'):
            out_path = out_dir / f'recalib_all_methods_{str(hospital).replace(" ", "_")}_{str(scenario).replace(" ", "_")}.png'
            _save_table_image(df_s, f'Recalibration (all methods) – {hospital} ({scenario})', out_path)


## Part 3C: Next Iteration (Professor Alignment, Revised)

This revised block uses embedded Part 3C helpers and keeps execution fully inside this notebook (no external `.py` dependency).

Key updates:
- shared transition anchoring (`T0`) across compared methods,
- fixed evaluation reference standard (`20,000`, 36-month outcome),
- Stage A pilot gap selection (`0, 500, 1000, 2000`) with reproducible summaries,
- Stage B method comparison (`knee`, `3_months`, `1_week`, `fixed_20_000`, practical fixed windows),
- recalibration window policy locked to `equal_current` (recalibration window matches detection window),
- fixed calibration basis size reused across comparable scenarios,
- delta metrics include mean delta, mean absolute delta, worst negative, max absolute,
- detection false negatives are tracked (`false_negative_detected`).


### Rationale

- `Step size` controls how often we test drift and allow recalibration opportunities.
- `Gap size` is evidence-selected (Stage A), not assumed.
- Detection uses `prior/current` windows; evaluation uses a fixed `20,000` reference standard to keep methods comparable.
- Recalibration is locked to detection window size (`equal_current`) within each test.
- Compared methods share the same transition anchor (`T0`) computed from the largest tested detection window.
- A fixed calibration basis size (`P3C_CALIBRATION_BASIS_SIZE`, default `17,000`) is reused across comparable scenarios.


In [ ]:
# Part 3C: Embedded helper library (self-contained)
import types
_p3c_before = set(globals().keys())

# --- Begin embedded code from part3c_professor_alignment.py ---
"""Focused helpers for the Part 3C professor-alignment workflow.

This module extracts the minimum logic needed from the large
`2026_2_4_hospital_level_simulations.ipynb` notebook and makes it
reproducible in a fresh notebook.

Key fix versus prior implementation:
- Transition/start anchoring is computed with the *scenario-specific*
  prior/current/gap/step values passed to the pipeline.
"""

import ast
import math
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import numpy as np
import pandas as pd


@dataclass(frozen=True)
class Part3Columns:
    date_col: str = "study_date"
    manufacturer_col: str = "manufacturer"
    hospital_col: str = "imp_performing_clinic"
    cancer_label_col: str = "CANCER_36_months_within_exam"
    ai_columns: tuple[str, ...] = ("lunit", "therapixel", "vara")


@dataclass(frozen=True)
class Part3Settings:
    random_seed: int = 44
    screen_threshold_days: int = 90
    prestart_min_cancers: int = 100
    signal_min_run: int = 3
    recal_alpha: float = 0.05
    recal_methods: tuple[str, ...] = ("A",)
    recal_use_score_signals: bool = False
    ordering: str = "date"

    # Defaults used when parameters are not explicitly overridden.
    default_calibration_strategy: str = "rad_sensitivity"
    default_rad_mode: str = "r1"
    default_target_mode: str = "r1"
    default_system_mode: str = "ai_only"


DEFAULT_SCENARIOS: list[dict[str, str]] = [
    {
        "key": "single_reader",
        "label": "single reader",
        "rad_mode": "r1",
        "target_mode": "r1",
        "system_mode": "ai_only",
    },
    {
        "key": "double_reader",
        "label": "double reader",
        "rad_mode": "r1",
        "target_mode": "combined",
        "system_mode": "ai_plus_r1",
    },
]


def proportions_ztest_two_sided(count: np.ndarray, nobs: np.ndarray) -> float:
    """Approximate two-sided z-test p-value for two proportions."""
    x1, x2 = float(count[0]), float(count[1])
    n1, n2 = float(nobs[0]), float(nobs[1])
    if n1 <= 0 or n2 <= 0:
        return 1.0
    p1 = x1 / n1
    p2 = x2 / n2
    p_pool = (x1 + x2) / (n1 + n2)
    denom = p_pool * (1.0 - p_pool) * ((1.0 / n1) + (1.0 / n2))
    if denom <= 0:
        return 1.0
    z = (p1 - p2) / math.sqrt(denom)
    # 2 * (1 - Phi(|z|)) == erfc(|z| / sqrt(2))
    p = math.erfc(abs(z) / math.sqrt(2.0))
    return float(min(max(p, 0.0), 1.0))


def fdr_bh(p_vals: np.ndarray | pd.Series) -> np.ndarray:
    """Benjamini-Hochberg FDR-adjusted p-values."""
    p = np.asarray(p_vals, dtype=float)
    m = len(p)
    if m == 0:
        return p
    order = np.argsort(p)
    p_sorted = p[order]
    q_sorted = np.empty(m, dtype=float)
    prev = 1.0
    for i in range(m - 1, -1, -1):
        rank = i + 1
        q = p_sorted[i] * m / rank
        prev = min(prev, q)
        q_sorted[i] = prev
    q = np.empty(m, dtype=float)
    q[order] = np.clip(q_sorted, 0.0, 1.0)
    return q


def extract_min_positive_days(val: Any) -> float:
    if pd.isna(val):
        return np.nan
    if isinstance(val, (int, float)):
        return float(val) if val > 0 else np.nan

    s = str(val).strip()
    if s in ("[]", ""):
        return np.nan

    try:
        parsed = ast.literal_eval(s)
    except Exception:
        return np.nan

    if isinstance(parsed, (list, tuple)):
        nums = [float(x) for x in parsed if pd.notna(x)]
    else:
        nums = [float(parsed)]
    pos = [x for x in nums if x > 0]
    return min(pos) if pos else np.nan


def load_part3_dataset(
    data_path: str | Path,
    cols: Part3Columns = Part3Columns(),
    settings: Part3Settings = Part3Settings(),
) -> pd.DataFrame:
    df = pd.read_csv(data_path, low_memory=False)
    df[cols.date_col] = pd.to_datetime(df[cols.date_col], format="mixed")
    if "has_cancer" not in df.columns:
        df["has_cancer"] = df[cols.cancer_label_col]

    if "days_to_diagnosis_pos" not in df.columns:
        if "days_to_diagnosis" in df.columns:
            df["days_to_diagnosis_pos"] = df["days_to_diagnosis"].apply(
                extract_min_positive_days
            )
        else:
            df["days_to_diagnosis_pos"] = np.nan

    if "time_based_screen" not in df.columns:
        df["time_based_screen"] = (
            (df["has_cancer"] == 1)
            & (df["days_to_diagnosis_pos"].notna())
            & (df["days_to_diagnosis_pos"] <= settings.screen_threshold_days)
        )
    return df


def compute_radiologist_flag(df: pd.DataFrame, mode: str = "combined") -> pd.Series:
    if mode == "r1":
        return (df["r1"] == "DISCUSSION").astype(int)
    if mode == "r2":
        return (df["r2"] == "DISCUSSION").astype(int)
    return ((df["r1"] == "DISCUSSION") | (df["r2"] == "DISCUSSION")).astype(int)


def compute_radiologist_sensitivity(df: pd.DataFrame, mode: str = "combined") -> float:
    flagged = compute_radiologist_flag(df, mode=mode)
    cancers = df["has_cancer"] == 1
    if cancers.sum() == 0:
        return np.nan
    return float((flagged[cancers].sum() / cancers.sum()) * 100)


def compute_radiologist_flagging_rate(df: pd.DataFrame, mode: str = "combined") -> float:
    flagged = compute_radiologist_flag(df, mode=mode)
    if len(df) == 0:
        return np.nan
    return float(flagged.mean() * 100)


def compute_radiologist_specificity(df: pd.DataFrame, mode: str = "combined") -> float:
    flagged = compute_radiologist_flag(df, mode=mode)
    controls = df["has_cancer"] == 0
    if controls.sum() == 0:
        return np.nan
    return float((1 - flagged[controls].mean()) * 100)


def resolve_part3_modes(
    settings: Part3Settings,
    rad_mode: str | None = None,
    calib_target_mode: str | None = None,
    system_mode: str | None = None,
) -> tuple[str, str, str]:
    rad = rad_mode if rad_mode is not None else settings.default_rad_mode
    target = calib_target_mode if calib_target_mode is not None else settings.default_target_mode
    system = system_mode if system_mode is not None else settings.default_system_mode
    return rad, target, system


def order_part3_df(
    df: pd.DataFrame,
    cols: Part3Columns,
    settings: Part3Settings,
    ordering: str | None = None,
    random_state: int | None = None,
) -> pd.DataFrame:
    mode = ordering if ordering is not None else settings.ordering
    seed = random_state if random_state is not None else settings.random_seed
    d = df.copy()
    if mode == "date":
        return d.sort_values(cols.date_col, kind="mergesort").reset_index(drop=True)
    if mode == "month_random":
        if "month_key" not in d.columns:
            d["month_key"] = (
                pd.to_datetime(d[cols.date_col], format="mixed")
                .dt.to_period("M")
                .astype(str)
            )
        rng = np.random.RandomState(seed)
        d["_rand"] = rng.rand(len(d))
        return d.sort_values(["month_key", "_rand"]).drop(columns=["_rand"]).reset_index(
            drop=True
        )
    raise ValueError("ordering must be 'date' or 'month_random'")


def ensure_union_flags(
    df: pd.DataFrame,
    ai_columns: list[str] | tuple[str, ...],
    base_suffix: str,
    r1_flag: pd.Series,
    out_suffix: str | None = None,
) -> str:
    union_suffix = out_suffix if out_suffix is not None else f"{base_suffix}_r1_union"
    for ai in ai_columns:
        base_col = f"{ai}_flagged_{base_suffix}"
        if base_col not in df.columns:
            df[base_col] = np.nan
        df[f"{ai}_flagged_{union_suffix}"] = (
            (df[base_col].fillna(0).astype(int) == 1) | r1_flag
        ).astype(int)
    return union_suffix


def compute_part3_transition_by_cancer_count(
    df_sorted: pd.DataFrame,
    cols: Part3Columns,
    start_date: str | pd.Timestamp | None,
    min_cancers: int,
    ref_size: int,
    gap_size: int,
    curr_size: int,
    step_size: int,
) -> tuple[int | None, pd.Timestamp | None, str]:
    # start_date kept for compatibility with legacy call sites.
    _ = start_date
    if len(df_sorted) == 0:
        return None, None, "excluded_no_data"
    if min_cancers is None or min_cancers <= 0:
        return None, None, "excluded_invalid_min_cancers"

    cancers = df_sorted["has_cancer"].fillna(0).astype(int)
    if cancers.sum() < min_cancers:
        return None, None, "excluded_insufficient_cancers"

    cum_cancers = cancers.cumsum()
    idx_target = int(cum_cancers[cum_cancers >= min_cancers].index[0])

    total_span = ref_size + gap_size + curr_size
    start_idx = None
    for ref_start in range(0, len(df_sorted) - total_span + 1, step_size):
        curr_start = ref_start + ref_size + gap_size
        if curr_start > idx_target:
            start_idx = curr_start
            break

    if start_idx is None:
        return None, None, "excluded_insufficient_exams"

    return start_idx, df_sorted.loc[start_idx, cols.date_col], f"cancer_count_{min_cancers}"


def transition_center(transition_idx: int | None, curr_size: int | float | None) -> float | None:
    if transition_idx is None or curr_size is None:
        return None
    return float(transition_idx + (curr_size / 2))


def compute_transition_anchor_by_pair(
    df_part3: pd.DataFrame,
    cols: Part3Columns,
    settings: Part3Settings,
    start_date: str | pd.Timestamp | None,
    ref_size: int,
    gap_size: int,
    curr_size: int,
    step_size: int,
    pilot_hospital: str | None = None,
    pilot_manufacturer: str | None = None,
) -> dict[tuple[str, str], int]:
    """Compute a shared transition anchor per (hospital, manufacturer) pair.

    Use this with the largest window configuration and pass the returned map
    to `build_part3_recalibration_summary_v3(..., transition_idx_by_pair=...)`
    so all compared methods use the same start point.
    """
    out: dict[tuple[str, str], int] = {}
    hospitals = sorted(df_part3[cols.hospital_col].dropna().unique())
    if pilot_hospital is not None:
        hospitals = [h for h in hospitals if h == pilot_hospital]

    for hospital in hospitals:
        d_h = df_part3[df_part3[cols.hospital_col] == hospital].copy()
        if cols.manufacturer_col in d_h.columns:
            mans = sorted(d_h[cols.manufacturer_col].dropna().unique())
            if len(mans) == 0:
                mans = ["Unknown"]
        else:
            mans = ["Unknown"]
        if pilot_manufacturer is not None:
            mans = [m for m in mans if m == pilot_manufacturer]

        for manufacturer in mans:
            d = d_h if manufacturer == "Unknown" else d_h[d_h[cols.manufacturer_col] == manufacturer].copy()
            if len(d) == 0:
                continue
            d = order_part3_df(d, cols=cols, settings=settings)
            t_idx, _, _ = compute_part3_transition_by_cancer_count(
                d,
                cols=cols,
                start_date=start_date,
                min_cancers=settings.prestart_min_cancers,
                ref_size=ref_size,
                gap_size=gap_size,
                curr_size=curr_size,
                step_size=step_size,
            )
            if t_idx is None:
                continue
            out[(hospital, manufacturer)] = int(t_idx)
    return out


def _confirmed_signal_centers(
    df: pd.DataFrame | None,
    sig_mask: pd.Series,
    center_col: str,
    min_run: int = 3,
) -> list[float]:
    if df is None or len(df) == 0 or center_col not in df.columns:
        return []
    centers = df[[center_col]].reset_index(drop=True)[center_col].to_numpy()
    sig = sig_mask.reset_index(drop=True).to_numpy()
    confirmed: list[float] = []
    run = 0
    for idx, is_sig in enumerate(sig):
        if is_sig:
            run += 1
            if run == min_run:
                confirmed.append(float(centers[idx]))
        else:
            run = 0
    return confirmed


def compute_ai_threshold_for_target_rate(
    df_pre: pd.DataFrame,
    ai_col: str,
    target_rate: float,
) -> float:
    scores = df_pre[ai_col].dropna()
    if len(scores) == 0 or pd.isna(target_rate):
        return np.nan
    quantile = np.clip(1 - (target_rate / 100), 0, 1)
    return float(scores.quantile(quantile))


def compute_ai_threshold_for_target_sensitivity(
    df_pre: pd.DataFrame,
    ai_col: str,
    target_sens: float,
) -> float:
    cancer_scores = df_pre[df_pre["has_cancer"] == 1][ai_col].dropna()
    if len(cancer_scores) == 0 or pd.isna(target_sens):
        return np.nan
    quantile = np.clip(1 - (target_sens / 100), 0, 1)
    return float(cancer_scores.quantile(quantile))


def compute_flagging_rates(
    df: pd.DataFrame,
    ai_columns: list[str] | tuple[str, ...],
    subset_col: str | None = None,
    subset_value: Any = True,
    ref_size: int = 5000,
    gap_size: int = 500,
    curr_size: int = 5000,
    step_size: int = 500,
) -> dict[str, pd.DataFrame]:
    d = df.copy()
    if "rad_flagged" not in d.columns:
        d["rad_flagged"] = compute_radiologist_flag(d, mode="combined")

    results: dict[str, list[dict[str, float]]] = {"radiologist": []}
    for ai in ai_columns:
        results[ai] = []

    total_span = ref_size + gap_size + curr_size
    for ref_start in range(0, len(d) - total_span + 1, step_size):
        ref_end = ref_start + ref_size
        curr_start = ref_end + gap_size
        curr_end = curr_start + curr_size

        ref_window = d.iloc[ref_start:ref_end]
        curr_window = d.iloc[curr_start:curr_end]
        if subset_col is not None:
            ref_window = ref_window[ref_window[subset_col] == subset_value]
            curr_window = curr_window[curr_window[subset_col] == subset_value]

        center = (curr_start + curr_end) / 2
        ref_n = len(ref_window)
        curr_n = len(curr_window)

        ref_rad = ref_window["rad_flagged"].sum() if ref_n > 0 else 0
        curr_rad = curr_window["rad_flagged"].sum() if curr_n > 0 else 0
        ref_rad_pct = (ref_rad / ref_n * 100) if ref_n > 0 else np.nan
        curr_rad_pct = (curr_rad / curr_n * 100) if curr_n > 0 else np.nan
        if ref_n > 0 and curr_n > 0:
            p_rad = proportions_ztest_two_sided(
                np.array([curr_rad, ref_rad], dtype=float),
                np.array([curr_n, ref_n], dtype=float),
            )
        else:
            p_rad = 1.0
        results["radiologist"].append(
            {"curr_center": center, "ref_pct": ref_rad_pct, "curr_pct": curr_rad_pct, "p_value": p_rad}
        )

        for ai in ai_columns:
            flag_col = f"{ai}_flagged"
            ref_ai = ref_window[flag_col].sum() if ref_n > 0 else 0
            curr_ai = curr_window[flag_col].sum() if curr_n > 0 else 0
            ref_ai_pct = (ref_ai / ref_n * 100) if ref_n > 0 else np.nan
            curr_ai_pct = (curr_ai / curr_n * 100) if curr_n > 0 else np.nan
            if ref_n > 0 and curr_n > 0:
                p_ai = proportions_ztest_two_sided(
                    np.array([curr_ai, ref_ai], dtype=float),
                    np.array([curr_n, ref_n], dtype=float),
                )
            else:
                p_ai = 1.0
            results[ai].append(
                {"curr_center": center, "ref_pct": ref_ai_pct, "curr_pct": curr_ai_pct, "p_value": p_ai}
            )

    out: dict[str, pd.DataFrame] = {}
    for key, rows in results.items():
        df_res = pd.DataFrame(rows)
        if len(df_res) > 0:
            df_res["p_value_corrected"] = fdr_bh(df_res["p_value"].values)
            df_res["is_significant"] = df_res["p_value_corrected"] < 0.05
        out[key] = df_res
    return out


def compute_relative_flagging_rates(
    flagging_results: dict[str, pd.DataFrame] | None,
    ai_columns: list[str] | tuple[str, ...],
) -> dict[str, pd.DataFrame]:
    if flagging_results is None:
        return {}
    rad = flagging_results.get("radiologist")
    if rad is None or len(rad) == 0:
        return {}
    out: dict[str, pd.DataFrame] = {}
    for ai in ai_columns:
        ai_df = flagging_results.get(ai)
        if ai_df is None or len(ai_df) == 0:
            continue
        merged = ai_df[["curr_center", "curr_pct", "p_value_corrected"]].merge(
            rad[["curr_center", "curr_pct"]],
            on="curr_center",
            how="left",
            suffixes=("", "_rad"),
        )
        merged["curr_rel_rate"] = merged["curr_pct"] / merged["curr_pct_rad"]
        merged.loc[merged["curr_pct_rad"] <= 0, "curr_rel_rate"] = np.nan
        out[ai] = merged
    return out


def compute_conditional_overlap(
    df: pd.DataFrame,
    ai_columns: list[str] | tuple[str, ...],
    subset_col: str | None = None,
    subset_value: Any = True,
    ref_size: int = 5000,
    gap_size: int = 500,
    curr_size: int = 5000,
    step_size: int = 500,
) -> dict[str, pd.DataFrame]:
    out: dict[str, pd.DataFrame] = {}
    total_span = ref_size + gap_size + curr_size
    for ai in ai_columns:
        rows: list[dict[str, float]] = []
        for ref_start in range(0, len(df) - total_span + 1, step_size):
            ref_end = ref_start + ref_size
            curr_start = ref_end + gap_size
            curr_end = curr_start + curr_size
            ref_window = df.iloc[ref_start:ref_end]
            curr_window = df.iloc[curr_start:curr_end]
            if subset_col is not None:
                ref_window = ref_window[ref_window[subset_col] == subset_value]
                curr_window = curr_window[curr_window[subset_col] == subset_value]

            center = (curr_start + curr_end) / 2
            ref_ai = ref_window[f"{ai}_flagged"].sum()
            ref_rad = ref_window["rad_flagged"].sum()
            ref_both = (ref_window[f"{ai}_flagged"] & ref_window["rad_flagged"]).sum()
            curr_ai = curr_window[f"{ai}_flagged"].sum()
            curr_rad = curr_window["rad_flagged"].sum()
            curr_both = (curr_window[f"{ai}_flagged"] & curr_window["rad_flagged"]).sum()

            ref_par = (ref_both / ref_rad * 100) if ref_rad > 0 else np.nan
            curr_par = (curr_both / curr_rad * 100) if curr_rad > 0 else np.nan
            ref_pra = (ref_both / ref_ai * 100) if ref_ai > 0 else np.nan
            curr_pra = (curr_both / curr_ai * 100) if curr_ai > 0 else np.nan

            if ref_rad > 0 and curr_rad > 0:
                p_par = proportions_ztest_two_sided(
                    np.array([curr_both, ref_both], dtype=float),
                    np.array([curr_rad, ref_rad], dtype=float),
                )
            else:
                p_par = 1.0
            if ref_ai > 0 and curr_ai > 0:
                p_pra = proportions_ztest_two_sided(
                    np.array([curr_both, ref_both], dtype=float),
                    np.array([curr_ai, ref_ai], dtype=float),
                )
            else:
                p_pra = 1.0

            rows.append(
                {
                    "curr_center": center,
                    "ref_p_ai_given_rad": ref_par,
                    "curr_p_ai_given_rad": curr_par,
                    "ref_p_rad_given_ai": ref_pra,
                    "curr_p_rad_given_ai": curr_pra,
                    "p_val_ai_given_rad": p_par,
                    "p_val_rad_given_ai": p_pra,
                }
            )

        df_res = pd.DataFrame(rows)
        if len(df_res) > 0:
            df_res["p_val_ai_given_rad_corrected"] = fdr_bh(df_res["p_val_ai_given_rad"].values)
            df_res["p_val_rad_given_ai_corrected"] = fdr_bh(df_res["p_val_rad_given_ai"].values)
            df_res["sig_ai_given_rad"] = df_res["p_val_ai_given_rad_corrected"] < 0.05
            df_res["sig_rad_given_ai"] = df_res["p_val_rad_given_ai_corrected"] < 0.05
        out[ai] = df_res
    return out


def compute_rate_windows_by_flag(
    df: pd.DataFrame,
    ai_columns: list[str] | tuple[str, ...],
    flag_suffix: str,
    rate_kind: str,
    ref_size: int,
    gap_size: int,
    curr_size: int,
    step_size: int,
) -> pd.DataFrame:
    if rate_kind not in {"sensitivity", "specificity"}:
        raise ValueError("rate_kind must be 'sensitivity' or 'specificity'")

    rows: list[dict[str, float]] = []
    total_span = ref_size + gap_size + curr_size
    for ref_start in range(0, len(df) - total_span + 1, step_size):
        ref_end = ref_start + ref_size
        curr_start = ref_end + gap_size
        curr_end = curr_start + curr_size
        center = (curr_start + curr_end) / 2

        ref_window = df.iloc[ref_start:ref_end]
        curr_window = df.iloc[curr_start:curr_end]

        ref_subset = ref_window[ref_window["has_cancer"] == (1 if rate_kind == "sensitivity" else 0)]
        curr_subset = curr_window[curr_window["has_cancer"] == (1 if rate_kind == "sensitivity" else 0)]

        for ai in ai_columns:
            flag_col = f"{ai}_flagged_{flag_suffix}"
            if flag_col not in curr_window.columns:
                continue
            ref_n = len(ref_subset)
            curr_n = len(curr_subset)
            if rate_kind == "sensitivity":
                ref_success = (ref_subset[flag_col] == 1).sum() if ref_n > 0 else 0
                curr_success = (curr_subset[flag_col] == 1).sum() if curr_n > 0 else 0
            else:
                ref_success = (ref_subset[flag_col] == 0).sum() if ref_n > 0 else 0
                curr_success = (curr_subset[flag_col] == 0).sum() if curr_n > 0 else 0
            ref_rate = (ref_success / ref_n * 100) if ref_n > 0 else np.nan
            curr_rate = (curr_success / curr_n * 100) if curr_n > 0 else np.nan

            if ref_n > 0 and curr_n > 0:
                p_val = proportions_ztest_two_sided(
                    np.array([curr_success, ref_success], dtype=float),
                    np.array([curr_n, ref_n], dtype=float),
                )
            else:
                p_val = 1.0

            rows.append(
                {
                    "curr_center": center,
                    "ai_system": ai,
                    "ref_rate": ref_rate,
                    "curr_rate": curr_rate,
                    "p_value": p_val,
                }
            )

    out = pd.DataFrame(rows)
    if len(out) == 0:
        return pd.DataFrame(
            columns=["curr_center", "ai_system", "ref_rate", "curr_rate", "p_value", "p_value_corrected"]
        )
    for ai in ai_columns:
        mask = out["ai_system"] == ai
        p_vals = out.loc[mask, "p_value"].values
        if len(p_vals) == 0:
            continue
        out.loc[mask, "p_value_corrected"] = fdr_bh(p_vals)
    return out


def collect_part2_signal_centers(
    df_h: pd.DataFrame,
    ai_columns: list[str] | tuple[str, ...] | None = None,
    settings: Part3Settings | None = None,
    alpha: float = 0.05,
    ref_size: int = 20000,
    gap_size: int = 0,
    curr_size: int = 20000,
    step_size: int = 500,
    use_score_signals: bool = False,
) -> dict[str, dict[str, list[float]]]:
    """Backwards-compatible signal collector.

    Supports both APIs:
    1) New: collect_part2_signal_centers(df_h, ai_columns=..., settings=..., ...)
    2) Legacy: collect_part2_signal_centers(df_h, alpha=..., ref_size=..., ...)
    """
    if ai_columns is None:
        if "P3C_COLS" in globals() and hasattr(P3C_COLS, "ai_columns"):
            ai_columns = list(P3C_COLS.ai_columns)
        elif "AI_COLUMNS" in globals():
            ai_columns = list(AI_COLUMNS)
        else:
            ai_columns = []

    if settings is None:
        if "P3C_SETTINGS" in globals():
            settings = P3C_SETTINGS

    default_rad_mode = getattr(settings, "default_rad_mode", globals().get("PART3_RAD_MODE", "all"))
    signal_min_run = int(getattr(settings, "signal_min_run", globals().get("PART3_SIGNAL_MIN_RUN", 3)))

    # In this notebook workflow we rely on flagging/overlap signals.
    # Keep compatibility with old cells that pass use_score_signals=True.
    _ = use_score_signals

    d = df_h.copy()
    if "rad_flagged" not in d.columns:
        d["rad_flagged"] = compute_radiologist_flag(d, mode=default_rad_mode)

    signal_centers: dict[str, dict[str, list[float]]] = {ai: {} for ai in ai_columns}
    for ai in ai_columns:
        base_flag = f"{ai}_flagged"
        part3_flag = f"{ai}_flagged_part3"
        if base_flag not in d.columns and part3_flag in d.columns:
            d[base_flag] = d[part3_flag]

    flagging = compute_flagging_rates(
        d,
        ai_columns=ai_columns,
        ref_size=ref_size,
        gap_size=gap_size,
        curr_size=curr_size,
        step_size=step_size,
    )
    for ai in ai_columns:
        ai_data = flagging.get(ai)
        if ai_data is None or len(ai_data) == 0:
            continue
        sig = (
            ai_data["p_value_corrected"] < alpha
            if "p_value_corrected" in ai_data.columns
            else pd.Series([False] * len(ai_data))
        )
        signal_centers[ai]["flagging_rate"] = _confirmed_signal_centers(
            ai_data, sig, "curr_center", min_run=signal_min_run
        )

    overlap = compute_conditional_overlap(
        d,
        ai_columns=ai_columns,
        ref_size=ref_size,
        gap_size=gap_size,
        curr_size=curr_size,
        step_size=step_size,
    )
    for ai in ai_columns:
        ov = overlap.get(ai)
        if ov is None or len(ov) == 0:
            continue
        if "p_val_ai_given_rad_corrected" in ov.columns:
            sig = ov["p_val_ai_given_rad_corrected"] < alpha
            signal_centers[ai]["p_ai_given_rad"] = _confirmed_signal_centers(
                ov, sig, "curr_center", min_run=signal_min_run
            )
        if "p_val_rad_given_ai_corrected" in ov.columns:
            sig = ov["p_val_rad_given_ai_corrected"] < alpha
            signal_centers[ai]["p_rad_given_ai"] = _confirmed_signal_centers(
                ov, sig, "curr_center", min_run=signal_min_run
            )
    return signal_centers

def _weighted_quantile(values: np.ndarray, weights: np.ndarray, quantile: float) -> float:
    if len(values) == 0:
        return np.nan
    sorter = np.argsort(values)
    v = values[sorter]
    w = weights[sorter]
    cum_w = np.cumsum(w)
    if cum_w[-1] == 0:
        return np.nan
    cum = cum_w / cum_w[-1]
    return float(np.interp(quantile, cum, v))


def _histogram_weights(ref_scores: np.ndarray, curr_scores: np.ndarray, bins: int = 20) -> np.ndarray | None:
    if len(ref_scores) == 0 or len(curr_scores) == 0:
        return None
    hist_ref, edges = np.histogram(ref_scores, bins=bins)
    hist_curr, _ = np.histogram(curr_scores, bins=edges)
    ratio = np.zeros_like(hist_ref, dtype=float)
    np.divide(hist_curr, hist_ref, out=ratio, where=hist_ref > 0)
    bin_idx = np.searchsorted(edges, ref_scores, side="right") - 1
    bin_idx = np.clip(bin_idx, 0, len(ratio) - 1)
    return ratio[bin_idx]


def _histogram_ratio(ref_scores: np.ndarray, curr_scores: np.ndarray, bins: int = 20) -> tuple[np.ndarray | None, np.ndarray | None]:
    if len(ref_scores) == 0 or len(curr_scores) == 0:
        return None, None
    hist_ref, edges = np.histogram(ref_scores, bins=bins)
    hist_curr, _ = np.histogram(curr_scores, bins=edges)
    ratio = np.zeros_like(hist_ref, dtype=float)
    np.divide(hist_curr, hist_ref, out=ratio, where=hist_ref > 0)
    return edges, ratio


def _weights_for_scores(scores: np.ndarray, edges: np.ndarray | None, ratio: np.ndarray | None) -> np.ndarray | None:
    if edges is None or ratio is None or len(scores) == 0:
        return None
    bin_idx = np.searchsorted(edges, scores, side="right") - 1
    bin_idx = np.clip(bin_idx, 0, len(ratio) - 1)
    return ratio[bin_idx]


def _threshold_for_flagging_rate(scores: np.ndarray, target_rate: float, weights: np.ndarray | None = None) -> float:
    if len(scores) == 0 or pd.isna(target_rate):
        return np.nan
    q = float(np.clip(1 - (target_rate / 100), 0, 1))
    if weights is None:
        return float(np.quantile(scores, q))
    return float(_weighted_quantile(scores, weights, q))


def _p_rad_given_ai(scores: np.ndarray, rad_flagged: np.ndarray, threshold: float, weights: np.ndarray | None = None) -> float:
    ai_flagged = scores >= threshold
    if weights is None:
        denom = ai_flagged.sum()
        numer = (ai_flagged & rad_flagged).sum()
    else:
        denom = weights[ai_flagged].sum()
        numer = weights[ai_flagged & rad_flagged].sum()
    return float((numer / denom * 100) if denom > 0 else np.nan)


def _threshold_for_p_rad_given_ai(
    scores: np.ndarray,
    rad_flagged: np.ndarray,
    target_rate: float,
    weights: np.ndarray | None = None,
) -> float:
    if len(scores) == 0 or pd.isna(target_rate):
        return np.nan
    qs = np.linspace(0.05, 0.95, 19)
    if weights is None:
        thresholds = np.quantile(scores, qs)
    else:
        thresholds = np.array([_weighted_quantile(scores, weights, q) for q in qs])
    thresholds = np.unique(thresholds[~np.isnan(thresholds)])
    if len(thresholds) == 0:
        return np.nan
    best_thr = thresholds[0]
    best_diff = np.inf
    for thr in thresholds:
        val = _p_rad_given_ai(scores, rad_flagged, float(thr), weights=weights)
        diff = abs(val - target_rate) if not pd.isna(val) else np.inf
        if diff < best_diff:
            best_diff = diff
            best_thr = float(thr)
    return float(best_thr)


def _get_scores_and_rad(df: pd.DataFrame, ai: str) -> tuple[np.ndarray, np.ndarray]:
    if df is None or len(df) == 0:
        return np.array([]), np.array([], dtype=bool)
    d = df[[ai, "rad_flagged"]].dropna(subset=[ai]).copy()
    return d[ai].to_numpy(), d["rad_flagged"].to_numpy().astype(bool)


def _get_target_metrics(
    df_h: pd.DataFrame,
    ai_columns: list[str] | tuple[str, ...],
    transition_idx: int,
    ref_size: int,
    gap_size: int,
    curr_size: int,
    step_size: int,
) -> dict[str, dict[str, float]]:
    targets: dict[str, dict[str, float]] = {ai: {} for ai in ai_columns}
    t_center = transition_center(transition_idx, curr_size)
    flagging = compute_flagging_rates(
        df_h,
        ai_columns=ai_columns,
        ref_size=ref_size,
        gap_size=gap_size,
        curr_size=curr_size,
        step_size=step_size,
    )
    overlap = compute_conditional_overlap(
        df_h,
        ai_columns=ai_columns,
        ref_size=ref_size,
        gap_size=gap_size,
        curr_size=curr_size,
        step_size=step_size,
    )
    rel_flag = compute_relative_flagging_rates(flagging, ai_columns=ai_columns)

    for ai in ai_columns:
        flag_df = flagging.get(ai)
        if flag_df is not None and len(flag_df) > 0:
            targets[ai]["flagging_rate"] = flag_df[flag_df["curr_center"] < t_center]["curr_pct"].mean()
        rel_df = rel_flag.get(ai)
        if rel_df is not None and len(rel_df) > 0:
            targets[ai]["relative_flagging_rate"] = rel_df[rel_df["curr_center"] < t_center]["curr_rel_rate"].mean()
        ov_df = overlap.get(ai)
        if ov_df is not None and len(ov_df) > 0:
            targets[ai]["p_ai_given_rad"] = ov_df[ov_df["curr_center"] < t_center]["curr_p_ai_given_rad"].mean()
            targets[ai]["p_rad_given_ai"] = ov_df[ov_df["curr_center"] < t_center]["curr_p_rad_given_ai"].mean()
    return targets


def _compute_recal_threshold(
    ai: str,
    metric_key: str,
    method: str,
    target_value: float,
    recal_window: pd.DataFrame,
    ref_window: pd.DataFrame | None,
    threshold_metric: str | None = None,
) -> float:
    metric = threshold_metric if threshold_metric is not None else metric_key
    if metric in ["score_median", "score_p90", "score_ks"]:
        return np.nan

    if metric == "sensitivity":
        if method == "A":
            cancer_scores = recal_window[recal_window["has_cancer"] == 1][ai].dropna().values
            return _threshold_for_flagging_rate(cancer_scores, target_value, weights=None)
        ref_scores_all = ref_window[ai].dropna().values if ref_window is not None else np.array([])
        curr_scores_all = recal_window[ai].dropna().values
        if len(ref_scores_all) == 0 or len(curr_scores_all) == 0:
            return np.nan
        edges, ratio = _histogram_ratio(ref_scores_all, curr_scores_all)
        ref_cancer_scores = (
            ref_window[ref_window["has_cancer"] == 1][ai].dropna().values
            if ref_window is not None
            else np.array([])
        )
        if len(ref_cancer_scores) == 0:
            return np.nan
        weights = _weights_for_scores(ref_cancer_scores, edges, ratio)
        return _threshold_for_flagging_rate(ref_cancer_scores, target_value, weights=weights)

    if metric == "relative_flagging_rate":
        rad_rate = recal_window["rad_flagged"].mean() * 100 if len(recal_window) > 0 else np.nan
        if pd.isna(rad_rate) or pd.isna(target_value):
            return np.nan
        target_value = target_value * rad_rate

    if method == "A":
        base_scores, base_rad = _get_scores_and_rad(recal_window, ai)
        weights = None
    else:
        ref_scores, ref_rad = _get_scores_and_rad(ref_window, ai)
        curr_scores, _ = _get_scores_and_rad(recal_window, ai)
        if len(ref_scores) == 0 or len(curr_scores) == 0:
            return np.nan
        weights = _histogram_weights(ref_scores, curr_scores)
        base_scores, base_rad = ref_scores, ref_rad

    if metric in ["flagging_rate", "relative_flagging_rate"]:
        return _threshold_for_flagging_rate(base_scores, target_value, weights=weights)
    if metric == "p_ai_given_rad":
        scores_rad = base_scores[base_rad]
        if len(scores_rad) == 0:
            return np.nan
        if weights is None:
            return _threshold_for_flagging_rate(scores_rad, target_value, weights=None)
        return _threshold_for_flagging_rate(scores_rad, target_value, weights=weights[base_rad])
    if metric == "p_rad_given_ai":
        return _threshold_for_p_rad_given_ai(base_scores, base_rad, target_value, weights=weights)
    return np.nan


def _safe_unique_int(values: list[float] | np.ndarray) -> np.ndarray:
    vals = pd.Series(values).dropna()
    if len(vals) == 0:
        return np.array([], dtype=int)
    vals = np.unique(np.round(vals.astype(float)).astype(int))
    return np.sort(vals)


def _delta_stats(
    rate_df: pd.DataFrame | None,
    ai: str,
    target: float,
    center_min: float | None = None,
) -> dict[str, float]:
    if rate_df is None or len(rate_df) == 0 or pd.isna(target):
        return {
            "mean_delta": np.nan,
            "mean_abs_delta": np.nan,
            "worst_negative_delta": np.nan,
            "max_abs_delta": np.nan,
            "negative_dev_count": np.nan,
            "negative_dev_rate": np.nan,
            "post_mean_rate": np.nan,
        }

    d = rate_df[rate_df["ai_system"] == ai].copy()
    if center_min is not None:
        d = d[d["curr_center"] >= center_min]
    if len(d) == 0:
        return {
            "mean_delta": np.nan,
            "mean_abs_delta": np.nan,
            "worst_negative_delta": np.nan,
            "max_abs_delta": np.nan,
            "negative_dev_count": np.nan,
            "negative_dev_rate": np.nan,
            "post_mean_rate": np.nan,
        }

    deltas = (d["curr_rate"] - target).dropna().to_numpy(dtype=float)
    post_mean = float(d["curr_rate"].mean()) if len(d) > 0 else np.nan
    if len(deltas) == 0:
        return {
            "mean_delta": np.nan,
            "mean_abs_delta": np.nan,
            "worst_negative_delta": np.nan,
            "max_abs_delta": np.nan,
            "negative_dev_count": np.nan,
            "negative_dev_rate": np.nan,
            "post_mean_rate": post_mean,
        }

    neg_count = int(np.sum(deltas < 0))
    return {
        "mean_delta": float(np.nanmean(deltas)),
        "mean_abs_delta": float(np.nanmean(np.abs(deltas))),
        "worst_negative_delta": float(np.nanmin(deltas)),
        "max_abs_delta": float(np.nanmax(np.abs(deltas))),
        "negative_dev_count": float(neg_count),
        "negative_dev_rate": float(neg_count / len(deltas)),
        "post_mean_rate": post_mean,
    }


def run_part3_hospital_pipeline_v2(
    df_subset: pd.DataFrame,
    hospital_name: str,
    cols: Part3Columns,
    settings: Part3Settings,
    start_date: str | pd.Timestamp | None,
    calibration_strategy: str,
    rad_mode: str,
    target_mode: str,
    ref_size: int,
    gap_size: int,
    curr_size: int,
    step_size: int,
    transition_index_override: int | None = None,
    calibration_basis_size: int | None = None,
) -> dict[str, Any] | None:
    if len(df_subset) == 0:
        return None

    d = order_part3_df(df_subset, cols=cols, settings=settings)
    if "rad_flagged" not in d.columns:
        d["rad_flagged"] = compute_radiologist_flag(d, mode=rad_mode)

    if transition_index_override is not None:
        if transition_index_override < 0 or transition_index_override >= len(d):
            return None
        t_idx = int(transition_index_override)
        t_date = d.loc[t_idx, cols.date_col]
        t_method = "override_shared_anchor"
    else:
        t_idx, t_date, t_method = compute_part3_transition_by_cancer_count(
            d,
            cols=cols,
            start_date=start_date,
            min_cancers=settings.prestart_min_cancers,
            ref_size=ref_size,
            gap_size=gap_size,
            curr_size=curr_size,
            step_size=step_size,
        )
        if t_idx is None:
            return None

    cancers = d["has_cancer"].fillna(0).astype(int)
    cum = cancers.cumsum()
    idx_target = int(cum[cum >= settings.prestart_min_cancers].index[0])
    pre_df = d.iloc[: idx_target + 1].copy()
    pre_df_by_t0 = d.iloc[:t_idx].copy()
    if calibration_basis_size is not None and calibration_basis_size > 0:
        calib_start = max(0, t_idx - int(calibration_basis_size))
        pre_df_calib = d.iloc[calib_start:t_idx].copy()
    else:
        pre_df_calib = pre_df_by_t0

    rad_sens = {
        "r1": compute_radiologist_sensitivity(pre_df_calib, mode="r1"),
        "r2": compute_radiologist_sensitivity(pre_df_calib, mode="r2"),
        "combined": compute_radiologist_sensitivity(pre_df_calib, mode="combined"),
    }
    rad_spec = {
        "r1": compute_radiologist_specificity(pre_df_calib, mode="r1"),
        "r2": compute_radiologist_specificity(pre_df_calib, mode="r2"),
        "combined": compute_radiologist_specificity(pre_df_calib, mode="combined"),
    }
    rad_flag = compute_radiologist_flagging_rate(pre_df_calib, mode=target_mode)

    thresholds: dict[str, float] = {}
    for ai in cols.ai_columns:
        if calibration_strategy == "rad_sensitivity":
            sens_df = pre_df_calib if int((pre_df_calib["has_cancer"] == 1).sum()) > 0 else pre_df
            thresholds[ai] = compute_ai_threshold_for_target_sensitivity(sens_df, ai, rad_sens[target_mode])
        elif calibration_strategy == "rad_flagging_rate":
            thresholds[ai] = compute_ai_threshold_for_target_rate(pre_df_calib, ai, rad_flag)
        else:
            raise ValueError("calibration_strategy must be 'rad_sensitivity' or 'rad_flagging_rate'")
        thr = thresholds[ai]
        d[f"{ai}_flagged_part3"] = (d[ai] >= thr).astype(int) if pd.notna(thr) else np.nan
        # Keep generic Part-2-style flag columns available for overlap/target helpers.
        if f"{ai}_flagged" not in d.columns:
            d[f"{ai}_flagged"] = d[f"{ai}_flagged_part3"]

    return {
        "hospital": hospital_name,
        "transition_index": int(t_idx),
        "start_date": t_date,
        "start_method": t_method,
        "df_h": d,
        "radiologist_sensitivity_pre": rad_sens,
        "radiologist_specificity_pre": rad_spec,
        "radiologist_flagging_rate_pre": rad_flag,
        "ai_thresholds": thresholds,
        "calibration_basis_size": calibration_basis_size,
    }


def build_part3_recalibration_summary_v3(
    df_part3: pd.DataFrame,
    cols: Part3Columns,
    settings: Part3Settings,
    start_date: str | pd.Timestamp | None,
    calibration_strategy: str | None = None,
    metrics: list[str] | None = None,
    target_mode: str = "metric",
    rad_mode: str | None = None,
    calib_target_mode: str | None = None,
    system_mode: str | None = None,
    scenario_label: str | None = None,
    curr_size_by_hospital: dict[str, float] | None = None,
    prior_size_by_hospital: dict[str, float] | None = None,
    prior_size: int | None = None,
    current_size: int | None = None,
    gap_size: int | None = None,
    step_size: int | None = None,
    recal_window_policy: str = "equal_current",
    recal_window_value: int | None = None,
    calibration_basis_size: int | None = None,
    transition_idx_by_pair: dict[tuple[str, str], int] | None = None,
    eval_reference_size: int = 20000,
    eval_horizon_months: int = 36,
    pilot_hospital: str | None = None,
    pilot_manufacturer: str | None = None,
) -> pd.DataFrame:
    strategy = calibration_strategy or settings.default_calibration_strategy
    metrics_use = metrics or ["flagging_rate", "relative_flagging_rate", "p_ai_given_rad", "p_rad_given_ai"]
    rad_mode_use, calib_target_use, system_mode_use = resolve_part3_modes(
        settings, rad_mode, calib_target_mode, system_mode
    )

    curr_default = int(current_size if current_size is not None else 10000)
    gap_use = int(gap_size if gap_size is not None else 0)
    step_use = int(step_size if step_size is not None else 500)
    recal_window_default = int(recal_window_value if recal_window_value is not None else 5000)

    hospitals = sorted(df_part3[cols.hospital_col].dropna().unique())
    if pilot_hospital is not None:
        hospitals = [h for h in hospitals if h == pilot_hospital]

    rows: list[dict[str, Any]] = []
    for hospital in hospitals:
        curr_h = curr_default
        if isinstance(curr_size_by_hospital, dict) and hospital in curr_size_by_hospital:
            curr_h = int(round(float(curr_size_by_hospital[hospital])))

        if isinstance(prior_size_by_hospital, dict) and hospital in prior_size_by_hospital:
            prior_h = int(round(float(prior_size_by_hospital[hospital])))
        elif prior_size is None:
            prior_h = curr_h
        else:
            prior_h = int(round(float(prior_size)))

        recal_window_h = curr_h if recal_window_policy == "equal_current" else int(round(float(recal_window_default)))

        d_hospital = df_part3[df_part3[cols.hospital_col] == hospital].copy()
        if cols.manufacturer_col in d_hospital.columns:
            manufacturers = sorted(d_hospital[cols.manufacturer_col].dropna().unique())
            if len(manufacturers) == 0:
                manufacturers = ["Unknown"]
        else:
            manufacturers = ["Unknown"]
        if pilot_manufacturer is not None:
            manufacturers = [m for m in manufacturers if m == pilot_manufacturer]

        for manufacturer in manufacturers:
            subset = d_hospital if manufacturer == "Unknown" else d_hospital[d_hospital[cols.manufacturer_col] == manufacturer].copy()
            if len(subset) == 0:
                continue

            result = run_part3_hospital_pipeline_v2(
                subset,
                hospital_name=hospital,
                cols=cols,
                settings=settings,
                start_date=start_date,
                calibration_strategy=strategy,
                rad_mode=rad_mode_use,
                target_mode=calib_target_use,
                ref_size=prior_h,
                gap_size=gap_use,
                curr_size=curr_h,
                step_size=step_use,
                transition_index_override=(
                    transition_idx_by_pair.get((hospital, manufacturer))
                    if isinstance(transition_idx_by_pair, dict)
                    else None
                ),
                calibration_basis_size=calibration_basis_size,
            )
            if result is None:
                continue

            d = result["df_h"]
            t_idx = int(result["transition_index"])
            t_center = transition_center(t_idx, curr_h)
            eval_t_center = transition_center(t_idx, eval_reference_size)
            if t_center is None or eval_t_center is None:
                continue

            if "rad_flagged" not in d.columns:
                d["rad_flagged"] = compute_radiologist_flag(d, mode=rad_mode_use)

            signals = collect_part2_signal_centers(
                d,
                ai_columns=cols.ai_columns,
                settings=settings,
                alpha=settings.recal_alpha,
                ref_size=prior_h,
                gap_size=gap_use,
                curr_size=curr_h,
                step_size=step_use,
                use_score_signals=settings.recal_use_score_signals,
            )
            targets = _get_target_metrics(
                d,
                ai_columns=cols.ai_columns,
                transition_idx=t_idx,
                ref_size=prior_h,
                gap_size=gap_use,
                curr_size=curr_h,
                step_size=step_use,
            )

            base_flag_suffix = "part3"
            r1_flag = None
            if system_mode_use == "ai_plus_r1":
                r1_flag = compute_radiologist_flag(d, mode="r1").astype(bool)
                base_flag_suffix = ensure_union_flags(d, cols.ai_columns, "part3", r1_flag)

            sens_base = compute_rate_windows_by_flag(
                d,
                ai_columns=cols.ai_columns,
                flag_suffix=base_flag_suffix,
                rate_kind="sensitivity",
                ref_size=eval_reference_size,
                gap_size=gap_use,
                curr_size=eval_reference_size,
                step_size=step_use,
            )
            spec_base = compute_rate_windows_by_flag(
                d,
                ai_columns=cols.ai_columns,
                flag_suffix=base_flag_suffix,
                rate_kind="specificity",
                ref_size=eval_reference_size,
                gap_size=gap_use,
                curr_size=eval_reference_size,
                step_size=step_use,
            )

            target_sens = result.get("radiologist_sensitivity_pre", {}).get(calib_target_use, np.nan)
            target_spec = result.get("radiologist_specificity_pre", {}).get(calib_target_use, np.nan)

            for ai in cols.ai_columns:
                base_sens = _delta_stats(sens_base, ai, target_sens, center_min=eval_t_center)
                base_spec = _delta_stats(spec_base, ai, target_spec, center_min=eval_t_center)

                for metric_key in metrics_use:
                    signal_key = "flagging_rate" if metric_key == "relative_flagging_rate" else metric_key
                    centers = signals.get(ai, {}).get(signal_key, [])
                    sig_all = _safe_unique_int(centers)
                    sig_post = _safe_unique_int([c for c in centers if c is not None and c >= t_center])

                    signal_count_post = int(len(sig_post))
                    signal_count_pre = int(max(0, len(sig_all) - signal_count_post))
                    first_signal = int(sig_post[0]) if signal_count_post > 0 else None
                    first_delay = (first_signal - t_center) if first_signal is not None else np.nan
                    false_negative = int(signal_count_post == 0)
                    true_positive = int(signal_count_post > 0)

                    if first_signal is not None:
                        first_start = max(0, first_signal - recal_window_h)
                        first_window = d.iloc[first_start:first_signal]
                        recal_n = len(first_window)
                        recal_cancers = int(first_window["has_cancer"].fillna(0).astype(int).sum())
                        recal_radpos = int((first_window["rad_flagged"] == 1).sum())
                        recal_scores = int(first_window[ai].notna().sum())
                    else:
                        recal_n = np.nan
                        recal_cancers = np.nan
                        recal_radpos = np.nan
                        recal_scores = np.nan

                    for method in settings.recal_methods:
                        recal_flag_suffix = f"recal_{metric_key}_{method}"
                        base_flag_col = f"{ai}_flagged_part3"
                        recal_flag_col = f"{ai}_flagged_{recal_flag_suffix}"
                        d[recal_flag_col] = d[base_flag_col]
                        recal_flag_suffix_use = recal_flag_suffix
                        recal_applied = False
                        recal_points: list[int] = []

                        for sig_idx in sig_post:
                            start_idx = max(0, int(sig_idx) - recal_window_h)
                            recal_window = d.iloc[start_idx : int(sig_idx)]
                            ref_start_idx = max(0, int(t_idx) - recal_window_h)
                            ref_window = d.iloc[ref_start_idx:int(t_idx)]

                            target_val = targets.get(ai, {}).get(metric_key)
                            threshold_metric = metric_key
                            if target_mode == "sensitivity":
                                target_val = result["radiologist_sensitivity_pre"][calib_target_use]
                                threshold_metric = "sensitivity"

                            recal_thr = _compute_recal_threshold(
                                ai=ai,
                                metric_key=metric_key,
                                method=method,
                                target_value=target_val,
                                recal_window=recal_window,
                                ref_window=ref_window,
                                threshold_metric=threshold_metric,
                            )
                            if pd.notna(recal_thr):
                                d.loc[int(sig_idx):, recal_flag_col] = (
                                    d.loc[int(sig_idx):, ai] >= recal_thr
                                ).astype(int)
                                recal_applied = True
                                recal_points.append(int(sig_idx))

                        if system_mode_use == "ai_plus_r1":
                            if r1_flag is None:
                                r1_flag = compute_radiologist_flag(d, mode="r1").astype(bool)
                            recal_flag_suffix_use = ensure_union_flags(
                                d,
                                cols.ai_columns,
                                recal_flag_suffix,
                                r1_flag,
                            )

                        sens_recal = compute_rate_windows_by_flag(
                            d,
                            ai_columns=cols.ai_columns,
                            flag_suffix=recal_flag_suffix_use,
                            rate_kind="sensitivity",
                            ref_size=eval_reference_size,
                            gap_size=gap_use,
                            curr_size=eval_reference_size,
                            step_size=step_use,
                        )
                        spec_recal = compute_rate_windows_by_flag(
                            d,
                            ai_columns=cols.ai_columns,
                            flag_suffix=recal_flag_suffix_use,
                            rate_kind="specificity",
                            ref_size=eval_reference_size,
                            gap_size=gap_use,
                            curr_size=eval_reference_size,
                            step_size=step_use,
                        )
                        recal_sens = _delta_stats(sens_recal, ai, target_sens, center_min=eval_t_center)
                        recal_spec = _delta_stats(spec_recal, ai, target_spec, center_min=eval_t_center)

                        if signal_count_post == 0:
                            status = "no_signal"
                        elif len(recal_points) == 0:
                            status = "no_threshold"
                        else:
                            status = "recal_applied"

                        rows.append(
                            {
                                "scenario": scenario_label,
                                "hospital": hospital,
                                "manufacturer": manufacturer,
                                "ai_system": ai,
                                "signal_metric": metric_key,
                                "recal_method": method,
                                "calibration_strategy": strategy,
                                "target_mode": target_mode,
                                "first_signal_center": first_signal,
                                "first_signal_delay": first_delay,
                                "true_positive_detected": true_positive,
                                "false_negative_detected": false_negative,
                                "signal_count": int(len(sig_all)),
                                "signal_count_pre": signal_count_pre,
                                "signal_count_post": signal_count_post,
                                "recal_point_count": int(len(recal_points)),
                                "avg_threshold_changes": float(len(recal_points)),
                                "recal_status": status,
                                "recal_window_policy": recal_window_policy,
                                "recal_window_size": recal_window_h,
                                "prior_window_size": prior_h,
                                "current_window_size": curr_h,
                                "gap_size": gap_use,
                                "step_size": step_use,
                                "eval_reference_size": eval_reference_size,
                                "eval_horizon_months": eval_horizon_months,
                                "recal_window_n": recal_n,
                                "recal_window_cancers": recal_cancers,
                                "recal_window_radpos": recal_radpos,
                                "recal_window_scores": recal_scores,
                                "target_sensitivity": target_sens,
                                "target_specificity": target_spec,
                                "sens_base": base_sens["post_mean_rate"],
                                "spec_base": base_spec["post_mean_rate"],
                                "sens_recal": recal_sens["post_mean_rate"],
                                "spec_recal": recal_spec["post_mean_rate"],
                                "delta_sens_base": (
                                    base_sens["post_mean_rate"] - target_sens
                                    if pd.notna(base_sens["post_mean_rate"]) and pd.notna(target_sens)
                                    else np.nan
                                ),
                                "delta_sens_recal": (
                                    recal_sens["post_mean_rate"] - target_sens
                                    if pd.notna(recal_sens["post_mean_rate"]) and pd.notna(target_sens)
                                    else np.nan
                                ),
                                "delta_spec_base": (
                                    base_spec["post_mean_rate"] - target_spec
                                    if pd.notna(base_spec["post_mean_rate"]) and pd.notna(target_spec)
                                    else np.nan
                                ),
                                "delta_spec_recal": (
                                    recal_spec["post_mean_rate"] - target_spec
                                    if pd.notna(recal_spec["post_mean_rate"]) and pd.notna(target_spec)
                                    else np.nan
                                ),
                                "mean_delta_sens_base": base_sens["mean_delta"],
                                "mean_delta_sens_recal": recal_sens["mean_delta"],
                                "mean_delta_spec_base": base_spec["mean_delta"],
                                "mean_delta_spec_recal": recal_spec["mean_delta"],
                                "mean_abs_delta_sens_base": base_sens["mean_abs_delta"],
                                "mean_abs_delta_sens_recal": recal_sens["mean_abs_delta"],
                                "mean_abs_delta_spec_base": base_spec["mean_abs_delta"],
                                "mean_abs_delta_spec_recal": recal_spec["mean_abs_delta"],
                                "worst_negative_delta_sens_base": base_sens["worst_negative_delta"],
                                "worst_negative_delta_sens_recal": recal_sens["worst_negative_delta"],
                                "worst_negative_delta_spec_base": base_spec["worst_negative_delta"],
                                "worst_negative_delta_spec_recal": recal_spec["worst_negative_delta"],
                                "max_abs_delta_sens_base": base_sens["max_abs_delta"],
                                "max_abs_delta_sens_recal": recal_sens["max_abs_delta"],
                                "max_abs_delta_spec_base": base_spec["max_abs_delta"],
                                "max_abs_delta_spec_recal": recal_spec["max_abs_delta"],
                                "negative_dev_rate_sens_base": base_sens["negative_dev_rate"],
                                "negative_dev_rate_sens_recal": recal_sens["negative_dev_rate"],
                                "negative_dev_rate_spec_base": base_spec["negative_dev_rate"],
                                "negative_dev_rate_spec_recal": recal_spec["negative_dev_rate"],
                                "recal_applied": recal_applied,
                            }
                        )

    return pd.DataFrame(rows)


def summarize_stage_a_gap(stage_a_df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, int]:
    if stage_a_df is None or len(stage_a_df) == 0:
        return pd.DataFrame(), pd.DataFrame(), 0

    gap_summary = (
        stage_a_df.groupby(["pilot_gap_size", "recal_window_policy", "signal_metric"], as_index=False)
        .agg(
            mean_signals_pre=("signal_count_pre", "mean"),
            mean_signals_post=("signal_count_post", "mean"),
            mean_first_delay=("first_signal_delay", "mean"),
            mean_delta_sens=("mean_delta_sens_recal", "mean"),
            mean_delta_spec=("mean_delta_spec_recal", "mean"),
            mean_abs_delta_sens=("mean_abs_delta_sens_recal", "mean"),
            mean_abs_delta_spec=("mean_abs_delta_spec_recal", "mean"),
            worst_negative_sens=("worst_negative_delta_sens_recal", "min"),
            worst_negative_spec=("worst_negative_delta_spec_recal", "min"),
            max_abs_sens=("max_abs_delta_sens_recal", "max"),
            max_abs_spec=("max_abs_delta_spec_recal", "max"),
            mean_false_negative=("false_negative_detected", "mean"),
            avg_threshold_changes=("avg_threshold_changes", "mean"),
        )
    )

    src = stage_a_df[stage_a_df["recal_window_policy"] == "equal_current"].copy()
    if len(src) == 0:
        return gap_summary, pd.DataFrame(), 0

    choice = (
        src.groupby(["pilot_gap_size"], as_index=False)
        .agg(
            worst_negative_sens=("worst_negative_delta_sens_recal", "min"),
            worst_negative_spec=("worst_negative_delta_spec_recal", "min"),
            max_abs_sens=("max_abs_delta_sens_recal", "max"),
            max_abs_spec=("max_abs_delta_spec_recal", "max"),
            mean_abs_sens=("mean_abs_delta_sens_recal", "mean"),
            mean_abs_spec=("mean_abs_delta_spec_recal", "mean"),
            mean_first_delay=("first_signal_delay", "mean"),
            mean_signals_pre=("signal_count_pre", "mean"),
            mean_signals_post=("signal_count_post", "mean"),
            mean_false_negative=("false_negative_detected", "mean"),
        )
    )
    choice["worst_case_penalty"] = (
        choice["worst_negative_sens"].clip(upper=0).abs()
        + choice["worst_negative_spec"].clip(upper=0).abs()
        + choice["max_abs_sens"]
        + choice["max_abs_spec"]
        + choice["mean_abs_sens"]
        + choice["mean_abs_spec"]
    )

    best_penalty = choice["worst_case_penalty"].min()
    tied = choice[(choice["worst_case_penalty"] - best_penalty).abs() <= 1e-9].copy()
    selected_gap = int(tied["pilot_gap_size"].min()) if len(tied) > 0 else 0
    if len(tied) > 1 and 0 in tied["pilot_gap_size"].values:
        selected_gap = 0
    return gap_summary, choice, selected_gap


def derive_knee_size_map(knee_df: pd.DataFrame | None, metrics: list[str]) -> dict[str, int]:
    if knee_df is None or len(knee_df) == 0 or "metric" not in knee_df.columns or "knee_curr_size" not in knee_df.columns:
        return {m: 20000 for m in metrics}
    out: dict[str, int] = {}
    for m in metrics:
        d = knee_df[knee_df["metric"] == m]
        if len(d) == 0:
            out[m] = 20000
            continue
        k = float(np.nanmedian(d["knee_curr_size"]))
        out[m] = int(np.ceil(k / 1000.0) * 1000)
    return out


def build_stage_b_summary(recalib_df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    if recalib_df is None or len(recalib_df) == 0:
        return pd.DataFrame(), pd.DataFrame()

    summary = (
        recalib_df.groupby(["window_method", "signal_metric"], as_index=False)
        .agg(
            metric_window_size_mean=("metric_window_size", "mean"),
            sens_base_mean=("sens_base", "mean"),
            sens_recal_mean=("sens_recal", "mean"),
            spec_base_mean=("spec_base", "mean"),
            spec_recal_mean=("spec_recal", "mean"),
            mean_delta_sens_recal=("mean_delta_sens_recal", "mean"),
            mean_delta_spec_recal=("mean_delta_spec_recal", "mean"),
            mean_abs_delta_sens_recal=("mean_abs_delta_sens_recal", "mean"),
            mean_abs_delta_spec_recal=("mean_abs_delta_spec_recal", "mean"),
            worst_negative_sens_recal=("worst_negative_delta_sens_recal", "min"),
            worst_negative_spec_recal=("worst_negative_delta_spec_recal", "min"),
            max_abs_sens_recal=("max_abs_delta_sens_recal", "max"),
            max_abs_spec_recal=("max_abs_delta_spec_recal", "max"),
            mean_false_negative=("false_negative_detected", "mean"),
            avg_threshold_changes=("avg_threshold_changes", "mean"),
        )
    )

    interval = (
        recalib_df.groupby(["window_method", "signal_metric"], as_index=False)
        .agg(
            metric_window_size_mean=("metric_window_size", "mean"),
            sens_base_mean=("sens_base", "mean"),
            sens_recal_mean=("sens_recal", "mean"),
            spec_base_mean=("spec_base", "mean"),
            spec_recal_mean=("spec_recal", "mean"),
            delta_sens_recal_min=("delta_sens_recal", "min"),
            delta_sens_recal_max=("delta_sens_recal", "max"),
            delta_spec_recal_min=("delta_spec_recal", "min"),
            delta_spec_recal_max=("delta_spec_recal", "max"),
            avg_threshold_changes=("avg_threshold_changes", "mean"),
        )
    )
    interval["Delta Sens (recal) interval"] = interval.apply(
        lambda r: f"{r['delta_sens_recal_min']:+.2f} to {r['delta_sens_recal_max']:+.2f}",
        axis=1,
    )
    interval["Delta Spec (recal) interval"] = interval.apply(
        lambda r: f"{r['delta_spec_recal_min']:+.2f} to {r['delta_spec_recal_max']:+.2f}",
        axis=1,
    )
    return summary, interval


def save_table_img(df: pd.DataFrame, title: str, out_path: str | Path) -> None:
    d = df.copy()
    d.columns = [str(c).replace("_", " ") for c in d.columns]
    for c in d.columns:
        if d[c].dtype == object:
            d[c] = d[c].astype(str).str.replace("_", " ", regex=False)

    delta_cols = [c for c in d.columns if "delta" in str(c).lower()]
    for c in d.columns:
        if pd.api.types.is_numeric_dtype(d[c]):
            if c in delta_cols:
                d[c] = d[c].map(lambda v: "" if pd.isna(v) else f"{float(v):+0.2f}")
            elif "window size" in c.lower():
                d[c] = d[c].map(lambda v: "" if pd.isna(v) else f"{int(round(float(v)))}")
            else:
                d[c] = d[c].map(lambda v: "" if pd.isna(v) else f"{float(v):0.2f}")

    fig_h = 1.6 + 0.45 * len(d)
    fig_w = max(12, 1.6 * len(d.columns))
    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    ax.axis("off")
    table = ax.table(
        cellText=d.values,
        colLabels=d.columns,
        cellLoc="center",
        loc="center",
        bbox=[0, 0, 1, 1],
    )
    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.scale(1.2, 1.6)

    header_color = "#4472C4"
    alt_row_color = "#E7E6E6"
    for j in range(len(d.columns)):
        cell = table[(0, j)]
        cell.set_facecolor(header_color)
        cell.set_text_props(weight="bold", color="white")
    for r in range(1, len(d) + 1):
        for j in range(len(d.columns)):
            cell = table[(r, j)]
            cell.set_facecolor(alt_row_color if r % 2 == 0 else "white")
    for cell in table.get_celld().values():
        cell.set_edgecolor("black")
        cell.set_linewidth(1.1)

    plt.title(title, fontsize=14, fontweight="bold", pad=16)
    plt.tight_layout()
    out = Path(out_path)
    out.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(out, dpi=300, bbox_inches="tight", facecolor="white")
    plt.close(fig)


def create_window_schematic(out_path: str | Path) -> None:
    from pathlib import Path as _Path
    import importlib as _importlib
    import sys as _sys

    helper_dir = _Path.cwd()
    if not (helper_dir / "part3c_professor_alignment.py").exists():
        helper_dir = _Path("code_files/active_notebooks")
    if str(helper_dir) not in _sys.path:
        _sys.path.insert(0, str(helper_dir))

    import part3c_professor_alignment as _p3c_source

    _importlib.reload(_p3c_source)
    _p3c_source.create_window_schematic(out_path)

# --- End embedded code ---

_p3c_after = {
    k: v for k, v in globals().items()
    if k not in _p3c_before and not k.startswith("__")
}
p3c = types.SimpleNamespace(**_p3c_after)
print(f"Embedded Part 3C helpers loaded into local `p3c`: {len(_p3c_after)} symbols")
del _p3c_before, _p3c_after


In [ ]:
# Part 3C: API compatibility bridge (legacy <-> embedded helpers)
import numpy as np
import pandas as pd


def _p3c_default_ai_columns():
    if 'P3C_COLS' in globals() and hasattr(P3C_COLS, 'ai_columns'):
        return list(P3C_COLS.ai_columns)
    if 'AI_COLUMNS' in globals():
        return list(AI_COLUMNS)
    return []


def _p3c_default_settings():
    if 'P3C_SETTINGS' in globals():
        return P3C_SETTINGS
    from types import SimpleNamespace
    return SimpleNamespace(
        default_rad_mode=globals().get('PART3_RAD_MODE', 'all'),
        signal_min_run=int(globals().get('PART3_SIGNAL_MIN_RUN', 3)),
        recal_use_score_signals=bool(globals().get('PART3_RECAL_USE_SCORE_SIGNALS', False)),
    )


if '_P3C_CORE_FUNCS' not in globals():
    _P3C_CORE_FUNCS = {
        'collect_part2_signal_centers': collect_part2_signal_centers,
        'compute_flagging_rates': compute_flagging_rates,
        'compute_conditional_overlap': compute_conditional_overlap,
        'compute_rate_windows_by_flag': compute_rate_windows_by_flag,
        '_get_target_metrics': _get_target_metrics,
        'ensure_union_flags': ensure_union_flags,
    }



def collect_part2_signal_centers(df_h, *args, **kwargs):
    core = _P3C_CORE_FUNCS['collect_part2_signal_centers']
    k = dict(kwargs)
    if 'ai_columns' not in k and not (len(args) >= 1 and isinstance(args[0], (list, tuple))):
        k['ai_columns'] = _p3c_default_ai_columns()
    if 'settings' not in k:
        k['settings'] = _p3c_default_settings()
    return core(df_h, *args, **k)



def compute_flagging_rates(df, *args, **kwargs):
    core = _P3C_CORE_FUNCS['compute_flagging_rates']
    k = dict(kwargs)
    if 'ai_columns' not in k and not (len(args) >= 1 and isinstance(args[0], (list, tuple))):
        k['ai_columns'] = _p3c_default_ai_columns()
    return core(df, *args, **k)



def compute_conditional_overlap(df, *args, **kwargs):
    core = _P3C_CORE_FUNCS['compute_conditional_overlap']
    k = dict(kwargs)
    if 'ai_columns' not in k and not (len(args) >= 1 and isinstance(args[0], (list, tuple))):
        k['ai_columns'] = _p3c_default_ai_columns()
    return core(df, *args, **k)



def compute_rate_windows_by_flag(df, *args, **kwargs):
    core = _P3C_CORE_FUNCS['compute_rate_windows_by_flag']
    k = dict(kwargs)
    if 'ai_columns' not in k and not (len(args) >= 1 and isinstance(args[0], (list, tuple))):
        k['ai_columns'] = _p3c_default_ai_columns()
    return core(df, *args, **k)



def _get_target_metrics(df_h, *args, **kwargs):
    core = _P3C_CORE_FUNCS['_get_target_metrics']
    k = dict(kwargs)
    a = list(args)
    # Legacy call style: _get_target_metrics(df_h, transition_idx, ...)
    if len(a) >= 1 and isinstance(a[0], (int, np.integer)) and 'transition_idx' not in k and 'ai_columns' not in k:
        k['transition_idx'] = int(a.pop(0))
    if 'ai_columns' not in k and not (len(a) >= 1 and isinstance(a[0], (list, tuple))):
        k['ai_columns'] = _p3c_default_ai_columns()
    return core(df_h, *a, **k)



def ensure_union_flags(df, *args, **kwargs):
    core = _P3C_CORE_FUNCS['ensure_union_flags']
    k = dict(kwargs)
    # Legacy call style: ensure_union_flags(df, base_suffix, r1_flag[, out_suffix])
    if len(args) >= 2 and isinstance(args[0], str) and isinstance(args[1], pd.Series):
        base_suffix = args[0]
        r1_flag = args[1]
        out_suffix = args[2] if len(args) >= 3 else k.get('out_suffix')
        return core(
            df,
            ai_columns=k.pop('ai_columns', _p3c_default_ai_columns()),
            base_suffix=base_suffix,
            r1_flag=r1_flag,
            out_suffix=out_suffix,
        )

    if 'ai_columns' not in k and not (len(args) >= 1 and isinstance(args[0], (list, tuple))):
        k['ai_columns'] = _p3c_default_ai_columns()
    return core(df, *args, **k)


print('Part 3C API compatibility bridge is active.')


In [ ]:
# Part 3C: Batch-based monitoring and retrospective evaluation schematic
from pathlib import Path
import importlib
import sys

helper_dir = Path.cwd()
if not (helper_dir / 'part3c_professor_alignment.py').exists():
    helper_dir = Path('code_files/active_notebooks')
if str(helper_dir) not in sys.path:
    sys.path.insert(0, str(helper_dir))

import part3c_professor_alignment as p3c_source
importlib.reload(p3c_source)

out_dir = Path('outputs/2025_12_29_counterfactual_manufacturer_swap_setup/part3c_next_iteration')
out_dir.mkdir(parents=True, exist_ok=True)
p3c_source.create_window_schematic(out_dir / 'part3c_window_schematic.png')
print('Saved:', out_dir / 'part3c_window_schematic.png')


In [ ]:
# Part 3C: Helper wiring (revised)
import numpy as np
import pandas as pd

if 'p3c' not in globals():
    raise RuntimeError('Run the embedded helper library cell before this one.')

P3C_COLS = p3c.Part3Columns(
    date_col=DATE_COL,
    manufacturer_col=MANUFACTURER_COL,
    hospital_col=HOSPITAL_COL,
    cancer_label_col=CANCER_LABEL_COL,
    ai_columns=tuple(AI_COLUMNS),
)

P3C_SETTINGS = p3c.Part3Settings(
    random_seed=RANDOM_SEED,
    screen_threshold_days=SCREEN_THRESHOLD_DAYS,
    prestart_min_cancers=PART3_PRESTART_MIN_CANCERS,
    signal_min_run=PART3_SIGNAL_MIN_RUN,
    recal_alpha=PART3_RECAL_ALPHA,
    recal_methods=tuple(PART3_RECAL_METHODS),
    recal_use_score_signals=PART3_RECAL_USE_SCORE_SIGNALS,
    ordering=PART3_ORDERING,
    default_calibration_strategy=PART3_CALIBRATION_STRATEGY,
    default_rad_mode=PART3_RAD_MODE,
    default_target_mode=PART3_TARGET_MODE,
    default_system_mode=PART3_SYSTEM_MODE,
)


def build_part3_recalibration_summary_v2(df_part3, start_date, **kwargs):
    # Compatibility wrapper kept for downstream cells.
    return p3c.build_part3_recalibration_summary_v3(
        df_part3,
        cols=P3C_COLS,
        settings=P3C_SETTINGS,
        start_date=start_date,
        **kwargs,
    )


In [ ]:
# Part 3C Stage A.1: Pilot setup (Karolinska / Hologic)
import numpy as np
import pandas as pd

PILOT_HOSPITAL = 'Karolinska University Hospital'
PILOT_MANUFACTURER = 'Hologic'
PILOT_CURRENT = 10000
PILOT_PRIOR = 10000
PILOT_STEP = 500
PILOT_GAPS = [0, 500, 1000, 2000]
PILOT_RECAL_POLICIES = [
    ('equal_current', None),
]
PILOT_CALIBRATION_BASIS = int(globals().get('P3C_CALIBRATION_BASIS_SIZE', 17000))

STAGEA_RESET = False
if STAGEA_RESET or 'STAGEA_RESULTS' not in globals():
    STAGEA_RESULTS = []

STAGEA_ANCHOR_BY_PAIR = {}
if 'df_part3' in globals():
    # Keep Stage A comparisons on the same pilot reality (shared T0 anchor).
    STAGEA_ANCHOR_BY_PAIR = p3c.compute_transition_anchor_by_pair(
        df_part3,
        cols=P3C_COLS,
        settings=P3C_SETTINGS,
        start_date=part3_start,
        ref_size=PILOT_PRIOR,
        gap_size=max(PILOT_GAPS),
        curr_size=PILOT_CURRENT,
        step_size=PILOT_STEP,
        pilot_hospital=PILOT_HOSPITAL,
        pilot_manufacturer=PILOT_MANUFACTURER,
    )

print(f'Cached Stage A partial runs: {len(STAGEA_RESULTS)}')
print('Stage A anchor pairs:', len(STAGEA_ANCHOR_BY_PAIR))
print('Pilot calibration basis size:', PILOT_CALIBRATION_BASIS)


In [ ]:
# Part 3C Stage A.2A: Build run plan (no simulation executed here)
RUN_GAPS = PILOT_GAPS
RUN_POLICIES = PILOT_RECAL_POLICIES
RUN_METRICS = ['flagging_rate', 'relative_flagging_rate', 'p_ai_given_rad', 'p_rad_given_ai']
RUN_SCENARIO_LABEL_CONTAINS = ['single', 'double']

if 'df_part3' not in globals():
    print('df_part3 not found.')
elif 'build_part3_recalibration_summary_v2' not in globals():
    print('build_part3_recalibration_summary_v2 not found. Run Stage A helper cell first.')
else:
    scenarios_to_run = []
    for s in PART3_SCENARIOS:
        lab = str(s.get('label', '')).lower()
        if any(tag in lab for tag in RUN_SCENARIO_LABEL_CONTAINS):
            scenarios_to_run.append(s)

    STAGEA_RUN_PLAN = []
    for gap in RUN_GAPS:
        for policy_name, policy_value in RUN_POLICIES:
            for scenario in scenarios_to_run:
                STAGEA_RUN_PLAN.append({
                    'gap_size': int(gap),
                    'policy_name': str(policy_name),
                    'policy_value': policy_value,
                    'scenario_label': str(scenario.get('label')),
                    'rad_mode': scenario.get('rad_mode'),
                    'target_mode': scenario.get('target_mode'),
                    'system_mode': scenario.get('system_mode'),
                })

    if len(STAGEA_RUN_PLAN) == 0:
        print('No Stage A run combinations found.')
    else:
        run_plan_df = pd.DataFrame(STAGEA_RUN_PLAN).reset_index().rename(columns={'index': 'run_idx'})
        display(run_plan_df)
        print(f'Total Stage A planned runs: {len(STAGEA_RUN_PLAN)}')


# Helper: execute exactly one plan item and cache result chunk

def stagea_run_one(run_idx: int):
    if 'STAGEA_RUN_PLAN' not in globals() or len(STAGEA_RUN_PLAN) == 0:
        print('Run Stage A.2A first to build STAGEA_RUN_PLAN.')
        return

    if run_idx < 0 or run_idx >= len(STAGEA_RUN_PLAN):
        print(f'Invalid run_idx={run_idx}. Valid range: 0..{len(STAGEA_RUN_PLAN)-1}')
        return

    item = STAGEA_RUN_PLAN[run_idx]
    transition_map = globals().get('STAGEA_ANCHOR_BY_PAIR', None)
    if transition_map is not None and len(transition_map) == 0:
        transition_map = None

    print(f"Running Stage A run_idx={run_idx}: gap={item['gap_size']}, policy={item['policy_name']}, scenario={item['scenario_label']}")

    df_res = build_part3_recalibration_summary_v2(
        df_part3,
        part3_start,
        calibration_strategy=PART3_CALIBRATION_STRATEGY,
        metrics=RUN_METRICS,
        rad_mode=item['rad_mode'],
        calib_target_mode=item['target_mode'],
        system_mode=item['system_mode'],
        scenario_label=item['scenario_label'],
        prior_size=PILOT_PRIOR,
        current_size=PILOT_CURRENT,
        gap_size=item['gap_size'],
        step_size=PILOT_STEP,
        recal_window_policy=item['policy_name'],
        recal_window_value=item['policy_value'],
        eval_reference_size=20000,
        eval_horizon_months=36,
        pilot_hospital=PILOT_HOSPITAL,
        pilot_manufacturer=PILOT_MANUFACTURER,
        calibration_basis_size=PILOT_CALIBRATION_BASIS,
        transition_idx_by_pair=transition_map,
    )

    if df_res is None or len(df_res) == 0:
        print(f'No rows produced for run_idx={run_idx}.')
        return

    d = df_res.copy()
    d['pilot_gap_size'] = int(item['gap_size'])
    d['pilot_step_size'] = int(PILOT_STEP)
    d['run_metrics'] = ','.join(RUN_METRICS)
    d['run_idx'] = int(run_idx)

    STAGEA_RESULTS.append(d.reset_index(drop=True))
    print(f'Completed run_idx={run_idx}. Added rows: {len(d)}. Cached chunks: {len(STAGEA_RESULTS)}')


In [ ]:
# Part 3C Stage A.2B: Run first simulation only (default for Run All)
if 'STAGEA_RUN_PLAN' not in globals() or len(STAGEA_RUN_PLAN) == 0:
    print('Run Stage A.2A first.')
else:
    stagea_run_one(0)


In [ ]:
# Part 3C Stage A.2C: Optional single additional simulation (OFF by default)
ENABLE_STAGEA_OPTIONAL_SINGLE = False
OPTIONAL_STAGEA_RUN_IDX = 1

if not ENABLE_STAGEA_OPTIONAL_SINGLE:
    print('Skipped optional single run (set ENABLE_STAGEA_OPTIONAL_SINGLE=True to run).')
else:
    stagea_run_one(int(OPTIONAL_STAGEA_RUN_IDX))


In [ ]:
# Part 3C Stage A.2D: Optional batch additional simulations (ON for non-zero gaps)
ENABLE_STAGEA_OPTIONAL_BATCH = True
OPTIONAL_STAGEA_RUN_INDICES = []  # auto-filled below from STAGEA_RUN_PLAN

if not ENABLE_STAGEA_OPTIONAL_BATCH:
    print('Skipped optional batch runs (set ENABLE_STAGEA_OPTIONAL_BATCH=True to run).')
else:
    if 'STAGEA_RUN_PLAN' not in globals() or len(STAGEA_RUN_PLAN) == 0:
        print('Run Stage A.2A first.')
    else:
        # Run all non-zero gap combinations so Stage A compares 0/500/1000/2000
        OPTIONAL_STAGEA_RUN_INDICES = [
            i for i, r in enumerate(STAGEA_RUN_PLAN)
            if int(r.get('gap_size', 0)) in [500, 1000, 2000]
        ]
        print('Running Stage A non-zero gap indices:', OPTIONAL_STAGEA_RUN_INDICES)
        for idx in OPTIONAL_STAGEA_RUN_INDICES:
            stagea_run_one(int(idx))


In [ ]:
# Part 3C Stage A.3: Combine cached chunks + check coverage
if 'STAGEA_RESULTS' not in globals() or len(STAGEA_RESULTS) == 0:
    print('No Stage A chunks cached yet. Run Stage A.2 first.')
else:
    stageA_pilot = pd.concat(STAGEA_RESULTS, ignore_index=True)

    key_cols = [
        'scenario', 'hospital', 'manufacturer', 'ai_system', 'signal_metric',
        'recal_method', 'recal_window_policy', 'pilot_gap_size', 'pilot_step_size',
        'prior_window_size', 'current_window_size', 'eval_reference_size'
    ]
    key_cols = [c for c in key_cols if c in stageA_pilot.columns]
    if len(key_cols) > 0:
        stageA_pilot = stageA_pilot.drop_duplicates(subset=key_cols, keep='last').reset_index(drop=True)

    coverage = (
        stageA_pilot
        .groupby(['pilot_gap_size', 'recal_window_policy', 'scenario', 'signal_metric'], as_index=False)
        .size()
        .rename(columns={'size': 'n_rows'})
        .sort_values(['pilot_gap_size', 'recal_window_policy', 'scenario', 'signal_metric'])
    )
    display(coverage)
    print(f'Total Stage A rows after dedup: {len(stageA_pilot)}')


In [ ]:
# Part 3C Stage A.4: Summary + gap selection (revised metrics)
if 'stageA_pilot' not in globals() or stageA_pilot is None or len(stageA_pilot) == 0:
    print('stageA_pilot is empty. Run Stage A.2 and Stage A.3 first.')
else:
    stageA_gap_summary, stageA_gap_choice, SELECTED_GAP_SIZE = p3c.summarize_stage_a_gap(stageA_pilot)
    SELECTED_STEP_SIZE = PILOT_STEP

    display(stageA_gap_summary)
    display(stageA_gap_choice)
    print(f'Stage A selected gap: {SELECTED_GAP_SIZE}, step: {SELECTED_STEP_SIZE}')


In [ ]:
# Part 3C Stage B.0: Methods comparison setup (run once)
import numpy as np
import pandas as pd

if 'df_part3' not in globals():
    print('df_part3 not found.')
else:
    STAGEB_RESET = True
    if STAGEB_RESET or 'STAGEB_METHOD_RESULTS' not in globals():
        STAGEB_METHOD_RESULTS = {}

    metrics = globals().get('PART3_RECAL_METRICS', ['flagging_rate', 'relative_flagging_rate', 'p_ai_given_rad', 'p_rad_given_ai'])
    gap_use = int(globals().get('SELECTED_GAP_SIZE', 0))
    step_use = int(globals().get('SELECTED_STEP_SIZE', 500))
    stageb_policy = 'equal_current'

    calibration_basis_size = int(globals().get('P3C_CALIBRATION_BASIS_SIZE', 17000))
    practical_fixed_windows = globals().get('P3C_PRACTICAL_FIXED_WINDOWS', [10000, 17000])
    practical_fixed_windows = sorted({int(x) for x in practical_fixed_windows if pd.notna(x) and int(x) > 0})

    knee_df_local = globals().get('knee_df', None)
    knee_size_map = p3c.derive_knee_size_map(knee_df_local, metrics)

    df_tmp = df_part3.copy()
    df_tmp[DATE_COL] = pd.to_datetime(df_tmp[DATE_COL], format='mixed', errors='coerce')

    df_tmp['month_key'] = df_tmp[DATE_COL].dt.to_period('M').astype(str)
    three_month_by_hospital = {}
    for hospital, d in df_tmp.groupby(HOSPITAL_COL):
        monthly_counts = d.groupby('month_key').size()
        if len(monthly_counts) == 0:
            continue
        three_month_by_hospital[hospital] = int(round(float(monthly_counts.mean() * 3)))

    df_tmp['week_key'] = df_tmp[DATE_COL].dt.to_period('W').astype(str)
    one_week_by_hospital = {}
    for hospital, d in df_tmp.groupby(HOSPITAL_COL):
        weekly_counts = d.groupby('week_key').size()
        if len(weekly_counts) == 0:
            continue
        one_week_by_hospital[hospital] = float(weekly_counts.mean())

    window_candidates = [20000]
    window_candidates.extend([int(v) for v in knee_size_map.values() if pd.notna(v) and int(v) > 0])
    window_candidates.extend([int(v) for v in three_month_by_hospital.values() if pd.notna(v) and int(v) > 0])
    window_candidates.extend([int(round(float(v))) for v in one_week_by_hospital.values() if pd.notna(v) and float(v) > 0])
    window_candidates.extend(practical_fixed_windows)
    anchor_window = int(max(window_candidates)) if len(window_candidates) > 0 else 20000

    transition_idx_by_pair = p3c.compute_transition_anchor_by_pair(
        df_part3,
        cols=P3C_COLS,
        settings=P3C_SETTINGS,
        start_date=part3_start,
        ref_size=anchor_window,
        gap_size=gap_use,
        curr_size=anchor_window,
        step_size=step_use,
    )

    STAGEB_CONTEXT = {
        'metrics': metrics,
        'gap_use': gap_use,
        'step_use': step_use,
        'stageb_policy': stageb_policy,
        'knee_size_map': knee_size_map,
        'three_month_by_hospital': three_month_by_hospital,
        'one_week_by_hospital': one_week_by_hospital,
        'calibration_basis_size': calibration_basis_size,
        'practical_fixed_windows': practical_fixed_windows,
        'anchor_window': anchor_window,
        'transition_idx_by_pair': transition_idx_by_pair,
    }

    print('Stage B context ready.')
    print('Metrics:', metrics)
    print('Gap/step:', gap_use, step_use)
    print('Calibration basis size:', calibration_basis_size)
    print('Policy:', stageb_policy)
    print('Anchor window size:', anchor_window)
    print('Shared anchor pairs:', len(transition_idx_by_pair))
    print('Knee sizes:', knee_size_map)
    print('Practical fixed windows:', practical_fixed_windows)


    # Optional lightweight subgroup profiler (hospital/manufacturer level)
    def p3c_enable_subgroup_timing():
        global _P3C_ORIG_RUN_PIPELINE
        global _P3C_SUBGROUP_TIMING_LOG
        global _P3C_PIPELINE_PROFILER_ACTIVE
        global run_part3_hospital_pipeline_v2

        if '_P3C_SUBGROUP_TIMING_LOG' not in globals():
            _P3C_SUBGROUP_TIMING_LOG = []
        if globals().get('_P3C_PIPELINE_PROFILER_ACTIVE', False):
            return

        _P3C_ORIG_RUN_PIPELINE = run_part3_hospital_pipeline_v2

        def _timed_run_part3_hospital_pipeline_v2(df_subset, hospital_name, *args, **kwargs):
            import time

            t0 = time.perf_counter()
            out = _P3C_ORIG_RUN_PIPELINE(df_subset, hospital_name, *args, **kwargs)
            dt = float(time.perf_counter() - t0)

            manufacturer = 'unknown'
            if isinstance(df_subset, pd.DataFrame) and MANUFACTURER_COL in df_subset.columns:
                vals = [v for v in df_subset[MANUFACTURER_COL].dropna().unique().tolist()]
                if len(vals) == 1:
                    manufacturer = str(vals[0])
                elif len(vals) > 1:
                    manufacturer = 'mixed'

            n_exams = int(len(df_subset)) if hasattr(df_subset, '__len__') else None
            transition_idx = out.get('transition_index') if isinstance(out, dict) else None

            _P3C_SUBGROUP_TIMING_LOG.append({
                'hospital': str(hospital_name),
                'manufacturer': manufacturer,
                'seconds': dt,
                'n_exams': n_exams,
                'transition_index': transition_idx,
            })
            return out

        run_part3_hospital_pipeline_v2 = _timed_run_part3_hospital_pipeline_v2
        p3c.run_part3_hospital_pipeline_v2 = _timed_run_part3_hospital_pipeline_v2
        _P3C_PIPELINE_PROFILER_ACTIVE = True

    def p3c_get_subgroup_timing_since(start_idx=0):
        if '_P3C_SUBGROUP_TIMING_LOG' not in globals():
            return pd.DataFrame()
        rows = _P3C_SUBGROUP_TIMING_LOG[int(start_idx):]
        if len(rows) == 0:
            return pd.DataFrame()
        return pd.DataFrame(rows)


In [ ]:
# Part 3C Stage B.1: Run method = knee (chunked + cached)
import time

if 'df_part3' not in globals() or 'STAGEB_CONTEXT' not in globals():
    print('Run Stage B.0 setup first.')
else:
    metrics = STAGEB_CONTEXT['metrics']
    gap_use = STAGEB_CONTEXT['gap_use']
    step_use = STAGEB_CONTEXT['step_use']
    stageb_policy = STAGEB_CONTEXT['stageb_policy']
    calibration_basis_size = STAGEB_CONTEXT['calibration_basis_size']
    transition_idx_by_pair = STAGEB_CONTEXT['transition_idx_by_pair']
    knee_size_map = STAGEB_CONTEXT['knee_size_map']

    # -----------------------------
    # Run controls (safe defaults)
    # -----------------------------


    PROFILE_SUBGROUP_TIMING = True
    PROFILE_TOP_N = 8
    _timing_start_idx = 0
    if PROFILE_SUBGROUP_TIMING:
        if 'p3c_enable_subgroup_timing' in globals():
            p3c_enable_subgroup_timing()
            _timing_start_idx = len(globals().get('_P3C_SUBGROUP_TIMING_LOG', []))
        else:
            print('Timing helper not found. Run Stage B.0 setup cell again.')
    KNEE_RUN_FIRST_ONLY = False   # when True, only run plan index 0
    KNEE_ENABLE_SELECTED = False # set True to run only indices below
    KNEE_SELECTED_INDICES = [1]  # used only if KNEE_ENABLE_SELECTED=True
    KNEE_ENABLE_RUN_ALL = True  # set True to run all combinations
    KNEE_RESET_CACHE = False     # set True to clear previous completed runs

    # Build run plan: one simulation per (scenario, metric)
    knee_plan = []
    for scenario in PART3_SCENARIOS:
        for metric_name in metrics:
            size_val = int(knee_size_map.get(metric_name, 20000))
            knee_plan.append({
                'scenario_label': scenario.get('label'),
                'rad_mode': scenario.get('rad_mode'),
                'target_mode': scenario.get('target_mode'),
                'system_mode': scenario.get('system_mode'),
                'metric_name': metric_name,
                'size_val': size_val,
            })

    if len(knee_plan) == 0:
        print('No knee runs planned.')
    else:
        display(pd.DataFrame(knee_plan).reset_index().rename(columns={'index': 'knee_run_idx'}))

        if KNEE_RUN_FIRST_ONLY:
            run_indices = [0]
        elif KNEE_ENABLE_SELECTED:
            run_indices = [int(i) for i in KNEE_SELECTED_INDICES if 0 <= int(i) < len(knee_plan)]
        elif KNEE_ENABLE_RUN_ALL:
            run_indices = list(range(len(knee_plan)))
        else:
            run_indices = []

        if 'STAGEB_KNEE_CACHE' not in globals() or KNEE_RESET_CACHE:
            STAGEB_KNEE_CACHE = {}

        print(f'Planned knee combinations: {len(knee_plan)} | This run will execute: {run_indices}')

        for i in run_indices:
            item = knee_plan[i]
            cache_key = (
                str(item['scenario_label']),
                str(item['metric_name']),
                int(item['size_val']),
                int(gap_use),
                int(step_use),
                str(stageb_policy),
                int(calibration_basis_size),
            )

            if cache_key in STAGEB_KNEE_CACHE:
                print(f'[skip cached] knee_run_idx={i} scenario={item["scenario_label"]} metric={item["metric_name"]}')
                continue

            t0 = time.perf_counter()
            print(f'[start] knee_run_idx={i} scenario={item["scenario_label"]} metric={item["metric_name"]} size={item["size_val"]}')

            df_res = build_part3_recalibration_summary_v2(
                df_part3,
                part3_start,
                calibration_strategy=PART3_CALIBRATION_STRATEGY,
                metrics=[item['metric_name']],
                rad_mode=item['rad_mode'],
                calib_target_mode=item['target_mode'],
                system_mode=item['system_mode'],
                scenario_label=item['scenario_label'],
                prior_size=item['size_val'],
                current_size=item['size_val'],
                gap_size=gap_use,
                step_size=step_use,
                recal_window_policy=stageb_policy,
                eval_reference_size=20000,
                eval_horizon_months=36,
                calibration_basis_size=calibration_basis_size,
                transition_idx_by_pair=transition_idx_by_pair,
            )

            if df_res is None or len(df_res) == 0:
                print(f'[done] knee_run_idx={i} -> no rows | {time.perf_counter()-t0:0.1f}s')
                continue

            d = df_res.copy()
            d['window_method'] = 'knee'
            d['metric_window_size'] = float(item['size_val'])
            d['knee_run_idx'] = int(i)
            STAGEB_KNEE_CACHE[cache_key] = d

            print(f'[done] knee_run_idx={i} -> rows={len(d)} | {time.perf_counter()-t0:0.1f}s')

        stageB_knee = (
            pd.concat(list(STAGEB_KNEE_CACHE.values()), ignore_index=True)
            if len(STAGEB_KNEE_CACHE) > 0 else pd.DataFrame()
        )
        STAGEB_METHOD_RESULTS['knee'] = stageB_knee
        print('knee rows (cached total):', len(stageB_knee))
        display(stageB_knee.head(20))


        if PROFILE_SUBGROUP_TIMING and 'p3c_get_subgroup_timing_since' in globals():
            timing_df = p3c_get_subgroup_timing_since(_timing_start_idx)
            if len(timing_df) > 0:
                print('Top subgroup runtimes (this cell run):')
                display(timing_df.sort_values('seconds', ascending=False).head(PROFILE_TOP_N))

                agg_df = (
                    timing_df.groupby(['hospital', 'manufacturer'], as_index=False)
                    .agg(total_seconds=('seconds', 'sum'), calls=('seconds', 'size'), mean_seconds=('seconds', 'mean'))
                    .sort_values('total_seconds', ascending=False)
                    .head(PROFILE_TOP_N)
                )
                print('Aggregated subgroup totals (this cell run):')
                display(agg_df)


In [ ]:
# Part 3C Stage B.2: Run method = 3_months (chunked + cached)
import time

if 'df_part3' not in globals() or 'STAGEB_CONTEXT' not in globals():
    print('Run Stage B.0 setup first.')
else:
    metrics = STAGEB_CONTEXT['metrics']
    gap_use = STAGEB_CONTEXT['gap_use']
    step_use = STAGEB_CONTEXT['step_use']
    stageb_policy = STAGEB_CONTEXT['stageb_policy']
    calibration_basis_size = STAGEB_CONTEXT['calibration_basis_size']
    transition_idx_by_pair = STAGEB_CONTEXT['transition_idx_by_pair']
    three_month_by_hospital = STAGEB_CONTEXT['three_month_by_hospital']

    # -----------------------------
    # Run controls (safe defaults)
    # -----------------------------


    PROFILE_SUBGROUP_TIMING = True
    PROFILE_TOP_N = 8
    _timing_start_idx = 0
    if PROFILE_SUBGROUP_TIMING:
        if 'p3c_enable_subgroup_timing' in globals():
            p3c_enable_subgroup_timing()
            _timing_start_idx = len(globals().get('_P3C_SUBGROUP_TIMING_LOG', []))
        else:
            print('Timing helper not found. Run Stage B.0 setup cell again.')
    M3_RUN_FIRST_ONLY = False   # when True, only run plan index 0
    M3_ENABLE_SELECTED = False # set True to run only indices below
    M3_SELECTED_INDICES = [1]  # used only if M3_ENABLE_SELECTED=True
    M3_ENABLE_RUN_ALL = True  # set True to run all combinations
    M3_RESET_CACHE = False     # set True to clear previous completed runs

    # Build run plan: one simulation per scenario (metrics are evaluated together)
    m3_plan = []
    for scenario in PART3_SCENARIOS:
        m3_plan.append({
            'scenario_label': scenario.get('label'),
            'rad_mode': scenario.get('rad_mode'),
            'target_mode': scenario.get('target_mode'),
            'system_mode': scenario.get('system_mode'),
        })

    if len(m3_plan) == 0:
        print('No 3_months runs planned.')
    else:
        display(pd.DataFrame(m3_plan).reset_index().rename(columns={'index': 'm3_run_idx'}))

        if M3_RUN_FIRST_ONLY:
            run_indices = [0]
        elif M3_ENABLE_SELECTED:
            run_indices = [int(i) for i in M3_SELECTED_INDICES if 0 <= int(i) < len(m3_plan)]
        elif M3_ENABLE_RUN_ALL:
            run_indices = list(range(len(m3_plan)))
        else:
            run_indices = []

        if 'STAGEB_3M_CACHE' not in globals() or M3_RESET_CACHE:
            STAGEB_3M_CACHE = {}

        print(f'Planned 3_months combinations: {len(m3_plan)} | This run will execute: {run_indices}')

        for i in run_indices:
            item = m3_plan[i]
            cache_key = (
                str(item['scenario_label']),
                int(gap_use),
                int(step_use),
                str(stageb_policy),
                int(calibration_basis_size),
                tuple(metrics),
            )

            if cache_key in STAGEB_3M_CACHE:
                print(f'[skip cached] m3_run_idx={i} scenario={item["scenario_label"]}')
                continue

            t0 = time.perf_counter()
            print(f'[start] m3_run_idx={i} scenario={item["scenario_label"]}')

            df_res = build_part3_recalibration_summary_v2(
                df_part3,
                part3_start,
                calibration_strategy=PART3_CALIBRATION_STRATEGY,
                metrics=metrics,
                rad_mode=item['rad_mode'],
                calib_target_mode=item['target_mode'],
                system_mode=item['system_mode'],
                scenario_label=item['scenario_label'],
                curr_size_by_hospital=three_month_by_hospital,
                prior_size_by_hospital=three_month_by_hospital,
                prior_size=None,
                current_size=20000,
                gap_size=gap_use,
                step_size=step_use,
                recal_window_policy=stageb_policy,
                eval_reference_size=20000,
                eval_horizon_months=36,
                calibration_basis_size=calibration_basis_size,
                transition_idx_by_pair=transition_idx_by_pair,
            )

            if df_res is None or len(df_res) == 0:
                print(f'[done] m3_run_idx={i} -> no rows | {time.perf_counter()-t0:0.1f}s')
                continue

            d = df_res.copy()
            d['window_method'] = '3_months'
            d['metric_window_size'] = d['hospital'].map(three_month_by_hospital).astype(float)
            d['m3_run_idx'] = int(i)
            STAGEB_3M_CACHE[cache_key] = d

            print(f'[done] m3_run_idx={i} -> rows={len(d)} | {time.perf_counter()-t0:0.1f}s')

        stageB_3months = (
            pd.concat(list(STAGEB_3M_CACHE.values()), ignore_index=True)
            if len(STAGEB_3M_CACHE) > 0 else pd.DataFrame()
        )
        STAGEB_METHOD_RESULTS['3_months'] = stageB_3months
        print('3_months rows (cached total):', len(stageB_3months))
        display(stageB_3months.head(20))


        if PROFILE_SUBGROUP_TIMING and 'p3c_get_subgroup_timing_since' in globals():
            timing_df = p3c_get_subgroup_timing_since(_timing_start_idx)
            if len(timing_df) > 0:
                print('Top subgroup runtimes (this cell run):')
                display(timing_df.sort_values('seconds', ascending=False).head(PROFILE_TOP_N))

                agg_df = (
                    timing_df.groupby(['hospital', 'manufacturer'], as_index=False)
                    .agg(total_seconds=('seconds', 'sum'), calls=('seconds', 'size'), mean_seconds=('seconds', 'mean'))
                    .sort_values('total_seconds', ascending=False)
                    .head(PROFILE_TOP_N)
                )
                print('Aggregated subgroup totals (this cell run):')
                display(agg_df)


In [ ]:
# Part 3C Stage B.3: Run method = 1_week (chunked + cached)
import time

if 'df_part3' not in globals() or 'STAGEB_CONTEXT' not in globals():
    print('Run Stage B.0 setup first.')
else:
    metrics = STAGEB_CONTEXT['metrics']
    gap_use = STAGEB_CONTEXT['gap_use']
    step_use = STAGEB_CONTEXT['step_use']
    stageb_policy = STAGEB_CONTEXT['stageb_policy']
    calibration_basis_size = STAGEB_CONTEXT['calibration_basis_size']
    transition_idx_by_pair = STAGEB_CONTEXT['transition_idx_by_pair']
    one_week_by_hospital = STAGEB_CONTEXT['one_week_by_hospital']

    # -----------------------------
    # Run controls (safe defaults)
    # -----------------------------


    PROFILE_SUBGROUP_TIMING = True
    PROFILE_TOP_N = 8
    _timing_start_idx = 0
    if PROFILE_SUBGROUP_TIMING:
        if 'p3c_enable_subgroup_timing' in globals():
            p3c_enable_subgroup_timing()
            _timing_start_idx = len(globals().get('_P3C_SUBGROUP_TIMING_LOG', []))
        else:
            print('Timing helper not found. Run Stage B.0 setup cell again.')
    W1_RUN_FIRST_ONLY = False   # when True, only run plan index 0
    W1_ENABLE_SELECTED = False # set True to run only indices below
    W1_SELECTED_INDICES = [1]  # used only if W1_ENABLE_SELECTED=True
    W1_ENABLE_RUN_ALL = True  # set True to run all combinations
    W1_RESET_CACHE = False     # set True to clear previous completed runs

    # Build run plan: one simulation per scenario (metrics are evaluated together)
    w1_plan = []
    for scenario in PART3_SCENARIOS:
        w1_plan.append({
            'scenario_label': scenario.get('label'),
            'rad_mode': scenario.get('rad_mode'),
            'target_mode': scenario.get('target_mode'),
            'system_mode': scenario.get('system_mode'),
        })

    if len(w1_plan) == 0:
        print('No 1_week runs planned.')
    else:
        display(pd.DataFrame(w1_plan).reset_index().rename(columns={'index': 'w1_run_idx'}))

        if W1_RUN_FIRST_ONLY:
            run_indices = [0]
        elif W1_ENABLE_SELECTED:
            run_indices = [int(i) for i in W1_SELECTED_INDICES if 0 <= int(i) < len(w1_plan)]
        elif W1_ENABLE_RUN_ALL:
            run_indices = list(range(len(w1_plan)))
        else:
            run_indices = []

        if 'STAGEB_1W_CACHE' not in globals() or W1_RESET_CACHE:
            STAGEB_1W_CACHE = {}

        print(f'Planned 1_week combinations: {len(w1_plan)} | This run will execute: {run_indices}')

        for i in run_indices:
            item = w1_plan[i]
            cache_key = (
                str(item['scenario_label']),
                int(gap_use),
                int(step_use),
                str(stageb_policy),
                int(calibration_basis_size),
                tuple(metrics),
            )

            if cache_key in STAGEB_1W_CACHE:
                print(f'[skip cached] w1_run_idx={i} scenario={item["scenario_label"]}')
                continue

            t0 = time.perf_counter()
            print(f'[start] w1_run_idx={i} scenario={item["scenario_label"]}')

            df_res = build_part3_recalibration_summary_v2(
                df_part3,
                part3_start,
                calibration_strategy=PART3_CALIBRATION_STRATEGY,
                metrics=metrics,
                rad_mode=item['rad_mode'],
                calib_target_mode=item['target_mode'],
                system_mode=item['system_mode'],
                scenario_label=item['scenario_label'],
                curr_size_by_hospital=one_week_by_hospital,
                prior_size_by_hospital=one_week_by_hospital,
                prior_size=None,
                current_size=20000,
                gap_size=gap_use,
                step_size=step_use,
                recal_window_policy=stageb_policy,
                eval_reference_size=20000,
                eval_horizon_months=36,
                calibration_basis_size=calibration_basis_size,
                transition_idx_by_pair=transition_idx_by_pair,
            )

            if df_res is None or len(df_res) == 0:
                print(f'[done] w1_run_idx={i} -> no rows | {time.perf_counter()-t0:0.1f}s')
                continue

            d = df_res.copy()
            d['window_method'] = '1_week'
            d['metric_window_size'] = d['hospital'].map(one_week_by_hospital).astype(float)
            d['w1_run_idx'] = int(i)
            STAGEB_1W_CACHE[cache_key] = d

            print(f'[done] w1_run_idx={i} -> rows={len(d)} | {time.perf_counter()-t0:0.1f}s')

        stageB_1week = (
            pd.concat(list(STAGEB_1W_CACHE.values()), ignore_index=True)
            if len(STAGEB_1W_CACHE) > 0 else pd.DataFrame()
        )
        STAGEB_METHOD_RESULTS['1_week'] = stageB_1week
        print('1_week rows (cached total):', len(stageB_1week))
        display(stageB_1week.head(20))


        if PROFILE_SUBGROUP_TIMING and 'p3c_get_subgroup_timing_since' in globals():
            timing_df = p3c_get_subgroup_timing_since(_timing_start_idx)
            if len(timing_df) > 0:
                print('Top subgroup runtimes (this cell run):')
                display(timing_df.sort_values('seconds', ascending=False).head(PROFILE_TOP_N))

                agg_df = (
                    timing_df.groupby(['hospital', 'manufacturer'], as_index=False)
                    .agg(total_seconds=('seconds', 'sum'), calls=('seconds', 'size'), mean_seconds=('seconds', 'mean'))
                    .sort_values('total_seconds', ascending=False)
                    .head(PROFILE_TOP_N)
                )
                print('Aggregated subgroup totals (this cell run):')
                display(agg_df)


In [ ]:
# Part 3C Stage B.4: Run method = fixed_20_000 (chunked + cached)
import time

if 'df_part3' not in globals() or 'STAGEB_CONTEXT' not in globals():
    print('Run Stage B.0 setup first.')
else:
    metrics = STAGEB_CONTEXT['metrics']
    gap_use = STAGEB_CONTEXT['gap_use']
    step_use = STAGEB_CONTEXT['step_use']
    stageb_policy = STAGEB_CONTEXT['stageb_policy']
    calibration_basis_size = STAGEB_CONTEXT['calibration_basis_size']
    transition_idx_by_pair = STAGEB_CONTEXT['transition_idx_by_pair']

    # -----------------------------
    # Run controls (safe defaults)
    # -----------------------------


    PROFILE_SUBGROUP_TIMING = True
    PROFILE_TOP_N = 8
    _timing_start_idx = 0
    if PROFILE_SUBGROUP_TIMING:
        if 'p3c_enable_subgroup_timing' in globals():
            p3c_enable_subgroup_timing()
            _timing_start_idx = len(globals().get('_P3C_SUBGROUP_TIMING_LOG', []))
        else:
            print('Timing helper not found. Run Stage B.0 setup cell again.')
    F20_RUN_FIRST_ONLY = False   # when True, only run plan index 0
    F20_ENABLE_SELECTED = False # set True to run only indices below
    F20_SELECTED_INDICES = [1]  # used only if F20_ENABLE_SELECTED=True
    F20_ENABLE_RUN_ALL = True  # set True to run all combinations
    F20_RESET_CACHE = False     # set True to clear previous completed runs

    # Build run plan: one simulation per scenario (metrics are evaluated together)
    f20_plan = []
    for scenario in PART3_SCENARIOS:
        f20_plan.append({
            'scenario_label': scenario.get('label'),
            'rad_mode': scenario.get('rad_mode'),
            'target_mode': scenario.get('target_mode'),
            'system_mode': scenario.get('system_mode'),
        })

    if len(f20_plan) == 0:
        print('No fixed_20_000 runs planned.')
    else:
        display(pd.DataFrame(f20_plan).reset_index().rename(columns={'index': 'f20_run_idx'}))

        if F20_RUN_FIRST_ONLY:
            run_indices = [0]
        elif F20_ENABLE_SELECTED:
            run_indices = [int(i) for i in F20_SELECTED_INDICES if 0 <= int(i) < len(f20_plan)]
        elif F20_ENABLE_RUN_ALL:
            run_indices = list(range(len(f20_plan)))
        else:
            run_indices = []

        if 'STAGEB_F20_CACHE' not in globals() or F20_RESET_CACHE:
            STAGEB_F20_CACHE = {}

        print(f'Planned fixed_20_000 combinations: {len(f20_plan)} | This run will execute: {run_indices}')

        for i in run_indices:
            item = f20_plan[i]
            cache_key = (
                str(item['scenario_label']),
                int(gap_use),
                int(step_use),
                str(stageb_policy),
                int(calibration_basis_size),
                tuple(metrics),
            )

            if cache_key in STAGEB_F20_CACHE:
                print(f'[skip cached] f20_run_idx={i} scenario={item["scenario_label"]}')
                continue

            t0 = time.perf_counter()
            print(f'[start] f20_run_idx={i} scenario={item["scenario_label"]}')

            df_res = build_part3_recalibration_summary_v2(
                df_part3,
                part3_start,
                calibration_strategy=PART3_CALIBRATION_STRATEGY,
                metrics=metrics,
                rad_mode=item['rad_mode'],
                calib_target_mode=item['target_mode'],
                system_mode=item['system_mode'],
                scenario_label=item['scenario_label'],
                prior_size=20000,
                current_size=20000,
                gap_size=gap_use,
                step_size=step_use,
                recal_window_policy=stageb_policy,
                eval_reference_size=20000,
                eval_horizon_months=36,
                calibration_basis_size=calibration_basis_size,
                transition_idx_by_pair=transition_idx_by_pair,
            )

            if df_res is None or len(df_res) == 0:
                print(f'[done] f20_run_idx={i} -> no rows | {time.perf_counter()-t0:0.1f}s')
                continue

            d = df_res.copy()
            d['window_method'] = 'fixed_20_000'
            d['metric_window_size'] = 20000.0
            d['f20_run_idx'] = int(i)
            STAGEB_F20_CACHE[cache_key] = d

            print(f'[done] f20_run_idx={i} -> rows={len(d)} | {time.perf_counter()-t0:0.1f}s')

        stageB_fixed20k = (
            pd.concat(list(STAGEB_F20_CACHE.values()), ignore_index=True)
            if len(STAGEB_F20_CACHE) > 0 else pd.DataFrame()
        )
        STAGEB_METHOD_RESULTS['fixed_20_000'] = stageB_fixed20k
        print('fixed_20_000 rows (cached total):', len(stageB_fixed20k))
        display(stageB_fixed20k.head(20))


        if PROFILE_SUBGROUP_TIMING and 'p3c_get_subgroup_timing_since' in globals():
            timing_df = p3c_get_subgroup_timing_since(_timing_start_idx)
            if len(timing_df) > 0:
                print('Top subgroup runtimes (this cell run):')
                display(timing_df.sort_values('seconds', ascending=False).head(PROFILE_TOP_N))

                agg_df = (
                    timing_df.groupby(['hospital', 'manufacturer'], as_index=False)
                    .agg(total_seconds=('seconds', 'sum'), calls=('seconds', 'size'), mean_seconds=('seconds', 'mean'))
                    .sort_values('total_seconds', ascending=False)
                    .head(PROFILE_TOP_N)
                )
                print('Aggregated subgroup totals (this cell run):')
                display(agg_df)


In [ ]:
# Part 3C Stage B.4b: Run practical fixed detection windows (e.g., 10k / 17k) (chunked + cached)
import time

if 'df_part3' not in globals() or 'STAGEB_CONTEXT' not in globals():
    print('Run Stage B.0 setup first.')
else:
    metrics = STAGEB_CONTEXT['metrics']
    gap_use = STAGEB_CONTEXT['gap_use']
    step_use = STAGEB_CONTEXT['step_use']
    stageb_policy = STAGEB_CONTEXT['stageb_policy']
    calibration_basis_size = STAGEB_CONTEXT['calibration_basis_size']
    transition_idx_by_pair = STAGEB_CONTEXT['transition_idx_by_pair']
    practical_windows = STAGEB_CONTEXT.get('practical_fixed_windows', [10000, 17000])

    # -----------------------------
    # Run controls (safe defaults)
    # -----------------------------


    PROFILE_SUBGROUP_TIMING = True
    PROFILE_TOP_N = 8
    _timing_start_idx = 0
    if PROFILE_SUBGROUP_TIMING:
        if 'p3c_enable_subgroup_timing' in globals():
            p3c_enable_subgroup_timing()
            _timing_start_idx = len(globals().get('_P3C_SUBGROUP_TIMING_LOG', []))
        else:
            print('Timing helper not found. Run Stage B.0 setup cell again.')
    PF_RUN_FIRST_ONLY = False   # when True, only run plan index 0
    PF_ENABLE_SELECTED = False # set True to run only indices below
    PF_SELECTED_INDICES = [1]  # used only if PF_ENABLE_SELECTED=True
    PF_ENABLE_RUN_ALL = True  # set True to run all combinations
    PF_RESET_CACHE = False     # set True to clear previous completed runs

    # Build run plan: one simulation per (window_size, scenario)
    pf_plan = []
    for win_size in practical_windows:
        win_size = int(win_size)
        if win_size <= 0:
            continue
        for scenario in PART3_SCENARIOS:
            pf_plan.append({
                'window_size': win_size,
                'scenario_label': scenario.get('label'),
                'rad_mode': scenario.get('rad_mode'),
                'target_mode': scenario.get('target_mode'),
                'system_mode': scenario.get('system_mode'),
            })

    if len(pf_plan) == 0:
        print('No practical fixed runs planned.')
    else:
        display(pd.DataFrame(pf_plan).reset_index().rename(columns={'index': 'pf_run_idx'}))

        if PF_RUN_FIRST_ONLY:
            run_indices = [0]
        elif PF_ENABLE_SELECTED:
            run_indices = [int(i) for i in PF_SELECTED_INDICES if 0 <= int(i) < len(pf_plan)]
        elif PF_ENABLE_RUN_ALL:
            run_indices = list(range(len(pf_plan)))
        else:
            run_indices = []

        if 'STAGEB_PF_CACHE' not in globals() or PF_RESET_CACHE:
            STAGEB_PF_CACHE = {}

        print(f'Planned practical-fixed combinations: {len(pf_plan)} | This run will execute: {run_indices}')

        for i in run_indices:
            item = pf_plan[i]
            cache_key = (
                int(item['window_size']),
                str(item['scenario_label']),
                int(gap_use),
                int(step_use),
                str(stageb_policy),
                int(calibration_basis_size),
                tuple(metrics),
            )

            if cache_key in STAGEB_PF_CACHE:
                print(f'[skip cached] pf_run_idx={i} size={item["window_size"]} scenario={item["scenario_label"]}')
                continue

            t0 = time.perf_counter()
            print(f'[start] pf_run_idx={i} size={item["window_size"]} scenario={item["scenario_label"]}')

            df_res = build_part3_recalibration_summary_v2(
                df_part3,
                part3_start,
                calibration_strategy=PART3_CALIBRATION_STRATEGY,
                metrics=metrics,
                rad_mode=item['rad_mode'],
                calib_target_mode=item['target_mode'],
                system_mode=item['system_mode'],
                scenario_label=item['scenario_label'],
                prior_size=item['window_size'],
                current_size=item['window_size'],
                gap_size=gap_use,
                step_size=step_use,
                recal_window_policy=stageb_policy,
                eval_reference_size=20000,
                eval_horizon_months=36,
                calibration_basis_size=calibration_basis_size,
                transition_idx_by_pair=transition_idx_by_pair,
            )

            if df_res is None or len(df_res) == 0:
                print(f'[done] pf_run_idx={i} -> no rows | {time.perf_counter()-t0:0.1f}s')
                continue

            d = df_res.copy()
            d['window_method'] = f"fixed_{item['window_size']:,}".replace(',', '_')
            d['metric_window_size'] = float(item['window_size'])
            d['practical_detection_size'] = int(item['window_size'])
            d['pf_run_idx'] = int(i)
            STAGEB_PF_CACHE[cache_key] = d

            print(f'[done] pf_run_idx={i} -> rows={len(d)} | {time.perf_counter()-t0:0.1f}s')

        stageB_practical_fixed = (
            pd.concat(list(STAGEB_PF_CACHE.values()), ignore_index=True)
            if len(STAGEB_PF_CACHE) > 0 else pd.DataFrame()
        )
        STAGEB_METHOD_RESULTS['practical_fixed'] = stageB_practical_fixed
        print('practical fixed rows (cached total):', len(stageB_practical_fixed))
        display(stageB_practical_fixed.head(20))


        if PROFILE_SUBGROUP_TIMING and 'p3c_get_subgroup_timing_since' in globals():
            timing_df = p3c_get_subgroup_timing_since(_timing_start_idx)
            if len(timing_df) > 0:
                print('Top subgroup runtimes (this cell run):')
                display(timing_df.sort_values('seconds', ascending=False).head(PROFILE_TOP_N))

                agg_df = (
                    timing_df.groupby(['hospital', 'manufacturer'], as_index=False)
                    .agg(total_seconds=('seconds', 'sum'), calls=('seconds', 'size'), mean_seconds=('seconds', 'mean'))
                    .sort_values('total_seconds', ascending=False)
                    .head(PROFILE_TOP_N)
                )
                print('Aggregated subgroup totals (this cell run):')
                display(agg_df)


In [ ]:
# Part 3C Stage B.5: Combine completed methods + summarize
if 'STAGEB_METHOD_RESULTS' not in globals() or len(STAGEB_METHOD_RESULTS) == 0:
    print('No Stage B method results found. Run one or more Stage B method cells first.')
else:
    completed = [k for k, v in STAGEB_METHOD_RESULTS.items() if v is not None and len(v) > 0]
    if len(completed) == 0:
        print('No non-empty method result found yet.')
    else:
        print('Completed methods:', completed)
        all_rows = [STAGEB_METHOD_RESULTS[k] for k in completed]
        recalib_next_iter = pd.concat(all_rows, ignore_index=True)
        stageB_summary, stageB_interval_summary = p3c.build_stage_b_summary(recalib_next_iter)

        practical_windows = STAGEB_CONTEXT.get('practical_fixed_windows', []) if 'STAGEB_CONTEXT' in globals() else []
        practical_labels = {f"fixed_{int(w):,}".replace(',', '_') for w in practical_windows}
        practical_src = recalib_next_iter[recalib_next_iter['window_method'].isin(practical_labels)].copy() if 'window_method' in recalib_next_iter.columns else pd.DataFrame()
        if len(practical_src) > 0:
            stageB_practical_summary = (
                practical_src
                .groupby(['window_method', 'signal_metric'], as_index=False)
                .agg(
                    mean_true_positive=('true_positive_detected', 'mean'),
                    mean_false_negative=('false_negative_detected', 'mean'),
                    mean_pre_signal_count=('signal_count_pre', 'mean'),
                    mean_post_signal_count=('signal_count_post', 'mean'),
                    mean_delay_exams=('first_signal_delay', 'mean'),
                    mean_delta_sens=('mean_delta_sens_recal', 'mean'),
                    mean_abs_delta_sens=('mean_abs_delta_sens_recal', 'mean'),
                    mean_delta_spec=('mean_delta_spec_recal', 'mean'),
                    mean_abs_delta_spec=('mean_abs_delta_spec_recal', 'mean'),
                    max_abs_sens=('max_abs_delta_sens_recal', 'max'),
                    max_abs_spec=('max_abs_delta_spec_recal', 'max'),
                )
            )
        else:
            stageB_practical_summary = pd.DataFrame()

        hospital_rate_map = {}
        if 'df_part3' in globals():
            d_rates = df_part3[[HOSPITAL_COL, DATE_COL]].copy()
            d_rates[DATE_COL] = pd.to_datetime(d_rates[DATE_COL], format='mixed', errors='coerce')
            rate_df = (
                d_rates.dropna(subset=[DATE_COL])
                .groupby(HOSPITAL_COL, as_index=False)
                .agg(
                    n_exams=(DATE_COL, 'size'),
                    min_date=(DATE_COL, 'min'),
                    max_date=(DATE_COL, 'max'),
                )
            )
            if len(rate_df) > 0:
                span_days = (rate_df['max_date'] - rate_df['min_date']).dt.days.clip(lower=1)
                rate_df['exams_per_day_est'] = rate_df['n_exams'] / span_days
                hospital_rate_map = dict(zip(rate_df[HOSPITAL_COL], rate_df['exams_per_day_est']))

        signal_audit = recalib_next_iter.copy()
        if 'true_positive_detected' in signal_audit.columns and 'false_negative_detected' in signal_audit.columns:
            signal_audit = signal_audit[(signal_audit['true_positive_detected'] == 1) | (signal_audit['false_negative_detected'] == 1)].copy()
        if len(signal_audit) > 0:
            signal_audit['exams_per_day_est'] = signal_audit['hospital'].map(hospital_rate_map).astype(float)
            signal_audit['first_delay_days_est'] = np.where(
                signal_audit['exams_per_day_est'] > 0,
                signal_audit['first_signal_delay'] / signal_audit['exams_per_day_est'],
                np.nan,
            )
            signal_audit['false_positive_pre_proxy'] = (signal_audit['signal_count_pre'].fillna(0) > 0).astype(int)
            keep_cols = [
                'scenario', 'hospital', 'manufacturer', 'ai_system',
                'window_method', 'signal_metric', 'recal_method',
                'first_signal_center', 'first_signal_delay', 'first_delay_days_est',
                'true_positive_detected', 'false_negative_detected', 'false_positive_pre_proxy',
                'signal_count_pre', 'signal_count_post', 'recal_point_count',
                'sens_base', 'sens_recal', 'delta_sens_base', 'delta_sens_recal',
                'spec_base', 'spec_recal', 'delta_spec_base', 'delta_spec_recal',
                'mean_delta_sens_recal', 'mean_abs_delta_sens_recal',
                'worst_negative_delta_sens_recal', 'max_abs_delta_sens_recal',
                'mean_delta_spec_recal', 'mean_abs_delta_spec_recal',
                'worst_negative_delta_spec_recal', 'max_abs_delta_spec_recal',
                'prior_window_size', 'current_window_size', 'recal_window_size',
                'gap_size', 'step_size', 'eval_reference_size',
            ]
            keep_cols = [c for c in keep_cols if c in signal_audit.columns]
            signal_audit = signal_audit[keep_cols].sort_values(
                ['window_method', 'signal_metric', 'hospital', 'manufacturer', 'first_signal_center'],
                na_position='last',
            )
        else:
            signal_audit = pd.DataFrame()

        stageB_examples = pd.DataFrame()
        if len(signal_audit) > 0 and 'first_signal_delay' in signal_audit.columns:
            tp_only = signal_audit[(signal_audit['true_positive_detected'] == 1) & signal_audit['first_signal_delay'].notna()].copy()
            if len(tp_only) > 0:
                short_example = tp_only.nsmallest(1, 'first_signal_delay').copy()
                short_example.insert(0, 'interval_example', 'short_interval')
                long_example = tp_only.nlargest(1, 'first_signal_delay').copy()
                long_example.insert(0, 'interval_example', 'long_interval')
                stageB_examples = pd.concat([short_example, long_example], ignore_index=True).drop_duplicates().reset_index(drop=True)

        print('Combined rows:', len(recalib_next_iter))
        print('Signal audit rows:', len(signal_audit))
        display(recalib_next_iter.head(30))
        display(stageB_summary)
        display(stageB_interval_summary)
        if len(stageB_practical_summary) > 0:
            display(stageB_practical_summary)
        if len(stageB_examples) > 0:
            display(stageB_examples)
        if len(signal_audit) > 0:
            display(signal_audit.head(40))


In [ ]:
# Part 3C: Export final paper/supplement tables (revised)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm, colors as mcolors
from pathlib import Path

out_dir = Path('outputs/2025_12_29_counterfactual_manufacturer_swap_setup/part3c_next_iteration')
out_dir.mkdir(parents=True, exist_ok=True)


def _pretty_stageb_cols(df):
    if df is None or len(df) == 0:
        return df
    d = df.copy()
    rename_map = {
        'window_method': 'method',
        'signal_metric': 'signal',
        'metric_window_size_mean': 'window size mean',
        'sens_base_mean': 'Sens(base)',
        'sens_recal_mean': 'Sens(recal)',
        'spec_base_mean': 'Spec(base)',
        'spec_recal_mean': 'Spec(recal)',
        'mean_delta_sens_recal': 'ΔSens mean',
        'mean_delta_spec_recal': 'ΔSpec mean',
        'mean_abs_delta_sens_recal': '|ΔSens| mean',
        'mean_abs_delta_spec_recal': '|ΔSpec| mean',
        'worst_negative_sens_recal': 'ΔSens worst -',
        'worst_negative_spec_recal': 'ΔSpec worst -',
        'max_abs_sens_recal': '|ΔSens| max',
        'max_abs_spec_recal': '|ΔSpec| max',
        'mean_false_negative': 'FN rate',
        'avg_threshold_changes': 'avg threshold changes',
        'Delta Sens (recal) interval': 'ΔSens interval (recal)',
        'Delta Spec (recal) interval': 'ΔSpec interval (recal)',
    }
    keep_order = [
        'window_method', 'signal_metric', 'metric_window_size_mean',
        'sens_base_mean', 'sens_recal_mean', 'spec_base_mean', 'spec_recal_mean',
        'mean_delta_sens_recal', 'mean_delta_spec_recal',
        'mean_abs_delta_sens_recal', 'mean_abs_delta_spec_recal',
        'worst_negative_sens_recal', 'worst_negative_spec_recal',
        'max_abs_sens_recal', 'max_abs_spec_recal',
        'mean_false_negative', 'avg_threshold_changes'
    ]
    cols = [c for c in keep_order if c in d.columns]
    if len(cols) > 0:
        d = d[cols]
    return d.rename(columns={k: v for k, v in rename_map.items() if k in d.columns})


def _format_table_for_display(df_raw):
    d = df_raw.copy()
    for c in d.columns:
        if d[c].dtype == object:
            d[c] = d[c].astype(str).str.replace('_', ' ', regex=False)

    for c in d.columns:
        if pd.api.types.is_numeric_dtype(d[c]):
            cl = str(c).lower()
            if 'delta' in cl:
                d[c] = d[c].map(lambda v: '' if pd.isna(v) else f'{float(v):+0.2f}')
            elif c in ['pilot_gap_size', 'step_size', 'prior_window_size', 'current_window_size', 'eval_reference_size']:
                d[c] = d[c].map(lambda v: '' if pd.isna(v) else f'{int(round(float(v)))}')
            elif 'window size' in cl:
                d[c] = d[c].map(lambda v: '' if pd.isna(v) else f'{int(round(float(v)))}')
            else:
                d[c] = d[c].map(lambda v: '' if pd.isna(v) else f'{float(v):0.2f}')

    labels = [str(c).replace('_', ' ') for c in d.columns]
    return d, labels


def _draw_table(df_raw, title, out_path, header_colors=None, cell_colors=None, bold_cells=None):
    if df_raw is None or len(df_raw) == 0:
        return

    d, labels = _format_table_for_display(df_raw)

    fig_h = 1.6 + 0.45 * len(d)
    fig_w = max(12, 1.6 * len(d.columns))
    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    ax.axis('off')

    table = ax.table(
        cellText=d.values,
        colLabels=labels,
        cellLoc='center',
        loc='center',
        bbox=[0, 0, 1, 1],
    )
    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.scale(1.2, 1.6)

    header_default = '#4472C4'
    alt_row_color = '#E7E6E6'

    for j, col in enumerate(df_raw.columns):
        cell = table[(0, j)]
        hcol = header_default
        if header_colors and col in header_colors:
            hcol = header_colors[col]
        cell.set_facecolor(hcol)
        cell.set_text_props(weight='bold', color='white')

    for r in range(1, len(d) + 1):
        for j in range(len(d.columns)):
            base_color = alt_row_color if r % 2 == 0 else 'white'
            if cell_colors and (r - 1, j) in cell_colors:
                base_color = cell_colors[(r - 1, j)]
            cell = table[(r, j)]
            cell.set_facecolor(base_color)
            if bold_cells and (r - 1, j) in bold_cells:
                cell.set_text_props(weight='bold')

    for cell in table.get_celld().values():
        cell.set_edgecolor('black')
        cell.set_linewidth(1.1)

    plt.title(title, fontsize=14, fontweight='bold', pad=16)
    plt.tight_layout()
    out = Path(out_path)
    out.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(out, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close(fig)


def _save_stagea_gap_choice_highlight(choice_df, selected_gap, out_path):
    d = choice_df.copy().sort_values('pilot_gap_size').reset_index(drop=True)

    key_col = 'pilot_gap_size'
    penalty_col = 'worst_case_penalty'
    penalty_components = [
        'worst_negative_sens', 'worst_negative_spec',
        'max_abs_sens', 'max_abs_spec',
        'mean_abs_sens', 'mean_abs_spec'
    ]

    header_colors = {c: '#2F5597' for c in d.columns}
    for c in penalty_components:
        if c in d.columns:
            header_colors[c] = '#5B9BD5'
    if penalty_col in d.columns:
        header_colors[penalty_col] = '#C00000'

    cell_colors = {}
    bold_cells = set()

    # Highlight selected row.
    if (selected_gap is not None) and (key_col in d.columns):
        sel_rows = d.index[pd.to_numeric(d[key_col], errors='coerce') == float(selected_gap)].tolist()
    else:
        sel_rows = []

    for r in sel_rows:
        for j in range(len(d.columns)):
            cell_colors[(r, j)] = '#FFF2CC'

    # Intensity for penalty column (the deciding score).
    if penalty_col in d.columns:
        j_pen = list(d.columns).index(penalty_col)
        vals = pd.to_numeric(d[penalty_col], errors='coerce')
        vmin = vals.min(skipna=True)
        vmax = vals.max(skipna=True)
        cmap = cm.get_cmap('RdYlGn_r')  # low=green, high=red

        for r, v in enumerate(vals):
            if pd.isna(v):
                continue
            t = 0.5 if pd.isna(vmin) or pd.isna(vmax) or abs(vmax - vmin) < 1e-12 else float((v - vmin) / (vmax - vmin))
            rgba = cmap(t)
            mix = 0.35
            color = (
                (1 - mix) * rgba[0] + mix,
                (1 - mix) * rgba[1] + mix,
                (1 - mix) * rgba[2] + mix,
                1.0,
            )
            cell_colors[(r, j_pen)] = mcolors.to_hex(color)

        # Strong highlight of selected gap's deciding score cell.
        if len(sel_rows) > 0:
            r0 = sel_rows[0]
            cell_colors[(r0, j_pen)] = '#93C47D'
            bold_cells.add((r0, j_pen))

    # Bold selected gap id cell.
    if len(sel_rows) > 0 and key_col in d.columns:
        j_key = list(d.columns).index(key_col)
        bold_cells.add((sel_rows[0], j_key))

    _draw_table(
        d,
        'Stage A Pilot: Gap Selection Rule (highlighted)',
        out_path,
        header_colors=header_colors,
        cell_colors=cell_colors,
        bold_cells=bold_cells,
    )


def _save_stagea_gap_summary_highlight(summary_df, selected_gap, out_path):
    d = summary_df.copy().sort_values(['pilot_gap_size', 'signal_metric']).reset_index(drop=True)

    header_colors = {c: '#4472C4' for c in d.columns}
    if 'pilot_gap_size' in d.columns:
        header_colors['pilot_gap_size'] = '#2F5597'

    cell_colors = {}
    bold_cells = set()

    if (selected_gap is not None) and ('pilot_gap_size' in d.columns):
        sel_rows = d.index[pd.to_numeric(d['pilot_gap_size'], errors='coerce') == float(selected_gap)].tolist()
        for r in sel_rows:
            for j in range(len(d.columns)):
                cell_colors[(r, j)] = '#E2F0D9'
        j_gap = list(d.columns).index('pilot_gap_size')
        for r in sel_rows:
            bold_cells.add((r, j_gap))

    _draw_table(
        d,
        'Stage A Pilot: Gap Comparison (selected gap highlighted)',
        out_path,
        header_colors=header_colors,
        cell_colors=cell_colors,
        bold_cells=bold_cells,
    )


selected_gap_for_export = None
if 'SELECTED_GAP_SIZE' in globals():
    try:
        selected_gap_for_export = int(SELECTED_GAP_SIZE)
    except Exception:
        selected_gap_for_export = None

if 'stageA_gap_choice' in globals() and stageA_gap_choice is not None and len(stageA_gap_choice) > 0 and selected_gap_for_export is None:
    tmp = stageA_gap_choice.copy()
    if 'worst_case_penalty' in tmp.columns and 'pilot_gap_size' in tmp.columns:
        best = float(pd.to_numeric(tmp['worst_case_penalty'], errors='coerce').min())
        tied = tmp[np.isclose(pd.to_numeric(tmp['worst_case_penalty'], errors='coerce'), best, atol=1e-9)]
        if len(tied) > 0:
            selected_gap_for_export = int(pd.to_numeric(tied['pilot_gap_size'], errors='coerce').min())

if 'stageA_gap_summary' in globals() and stageA_gap_summary is not None and len(stageA_gap_summary) > 0:
    _save_stagea_gap_summary_highlight(stageA_gap_summary, selected_gap_for_export, out_dir / 'stageA_gap_comparison.png')

if 'stageA_gap_choice' in globals() and stageA_gap_choice is not None and len(stageA_gap_choice) > 0:
    _save_stagea_gap_choice_highlight(stageA_gap_choice, selected_gap_for_export, out_dir / 'stageA_gap_selection.png')

if 'stageB_summary' in globals() and stageB_summary is not None and len(stageB_summary) > 0:
    stageB_summary_pretty = _pretty_stageb_cols(stageB_summary)
    p3c.save_table_img(stageB_summary_pretty, 'Stage B Summary (Mean + Worst-case + Max abs)', out_dir / 'stageB_summary.png')

if 'stageB_interval_summary' in globals() and stageB_interval_summary is not None and len(stageB_interval_summary) > 0:
    keep_cols = [
        'window_method', 'signal_metric', 'metric_window_size_mean',
        'sens_base_mean', 'sens_recal_mean', 'spec_base_mean', 'spec_recal_mean',
        'Delta Sens (recal) interval', 'Delta Spec (recal) interval', 'avg_threshold_changes'
    ]
    keep_cols = [c for c in keep_cols if c in stageB_interval_summary.columns]
    interval_pretty = _pretty_stageb_cols(stageB_interval_summary[keep_cols])
    p3c.save_table_img(
        interval_pretty,
        'Stage B Summary (Interval deltas)',
        out_dir / 'stageB_interval_summary.png'
    )

if 'stageB_practical_summary' in globals() and stageB_practical_summary is not None and len(stageB_practical_summary) > 0:
    practical_pretty = _pretty_stageb_cols(stageB_practical_summary)
    p3c.save_table_img(
        practical_pretty,
        'Stage B Practical Fixed Detection Sizes',
        out_dir / 'stageB_practical_summary.png'
    )

if 'stageB_examples' in globals() and stageB_examples is not None and len(stageB_examples) > 0:
    p3c.save_table_img(
        stageB_examples,
        'Stage B Numeric Examples (Short vs Long delay)',
        out_dir / 'stageB_examples_short_long.png'
    )
    stageB_examples.to_csv(out_dir / 'stageB_examples_short_long.csv', index=False)

if 'signal_audit' in globals() and signal_audit is not None and len(signal_audit) > 0:
    signal_audit.to_csv(out_dir / 'stageB_signal_audit.csv', index=False)
    p3c.save_table_img(
        signal_audit.head(60),
        'Stage B Signal Audit (first 60 rows)',
        out_dir / 'stageB_signal_audit_top60.png'
    )

    # Compact audit view for easier communication
    compact_cols = [
        'scenario', 'hospital', 'manufacturer', 'ai_system', 'window_method', 'signal_metric',
        'gap_size', 'step_size',
        'first_signal_delay', 'true_positive_detected', 'false_negative_detected', 'false_positive_pre_proxy',
        'mean_abs_delta_sens_recal', 'mean_abs_delta_spec_recal',
        'worst_negative_delta_sens_recal', 'worst_negative_delta_spec_recal',
        'avg_threshold_changes'
    ]
    compact_cols = [c for c in compact_cols if c in signal_audit.columns]
    signal_audit_compact = signal_audit[compact_cols].copy()

    if 'mean_abs_delta_sens_recal' in signal_audit_compact.columns and 'mean_abs_delta_spec_recal' in signal_audit_compact.columns:
        signal_audit_compact['instability_score'] = (
            pd.to_numeric(signal_audit_compact['mean_abs_delta_sens_recal'], errors='coerce').fillna(0)
            + pd.to_numeric(signal_audit_compact['mean_abs_delta_spec_recal'], errors='coerce').fillna(0)
        )

    rename_compact = {
        'manufacturer': 'mfr',
        'ai_system': 'AI',
        'window_method': 'method',
        'signal_metric': 'signal',
        'gap_size': 'gap',
        'step_size': 'step',
        'first_signal_delay': 'delay_exams',
        'true_positive_detected': 'TP',
        'false_negative_detected': 'FN',
        'false_positive_pre_proxy': 'FP_pre',
        'mean_abs_delta_sens_recal': '|ΔSens| mean',
        'mean_abs_delta_spec_recal': '|ΔSpec| mean',
        'worst_negative_delta_sens_recal': 'ΔSens worst -',
        'worst_negative_delta_spec_recal': 'ΔSpec worst -',
        'avg_threshold_changes': 'avg recal changes',
        'instability_score': 'instability score',
    }
    signal_audit_compact = signal_audit_compact.rename(columns={k: v for k, v in rename_compact.items() if k in signal_audit_compact.columns})

    sort_cols = [c for c in ['FN', 'instability score', 'delay_exams'] if c in signal_audit_compact.columns]
    asc = [False] * len(sort_cols)
    if len(sort_cols) > 0:
        signal_audit_compact = signal_audit_compact.sort_values(sort_cols, ascending=asc).reset_index(drop=True)

    signal_audit_compact.to_csv(out_dir / 'stageB_signal_audit_compact.csv', index=False)
    p3c.save_table_img(
        signal_audit_compact.head(60),
        'Stage B Signal Audit (compact, top 60 by FN/instability/delay)',
        out_dir / 'stageB_signal_audit_compact_top60.png'
    )

    # Method-level compact summary
    group_cols = [c for c in ['method', 'signal'] if c in signal_audit_compact.columns]
    if len(group_cols) > 0:
        agg_map = {}
        if 'TP' in signal_audit_compact.columns:
            agg_map['TP rate'] = ('TP', 'mean')
        if 'FN' in signal_audit_compact.columns:
            agg_map['FN rate'] = ('FN', 'mean')
        if 'FP_pre' in signal_audit_compact.columns:
            agg_map['FP pre rate'] = ('FP_pre', 'mean')
        if 'delay_exams' in signal_audit_compact.columns:
            agg_map['delay mean'] = ('delay_exams', 'mean')
        if 'instability score' in signal_audit_compact.columns:
            agg_map['instability mean'] = ('instability score', 'mean')
        if 'avg recal changes' in signal_audit_compact.columns:
            agg_map['avg recal changes'] = ('avg recal changes', 'mean')

        if len(agg_map) > 0:
            signal_audit_method_summary = (
                signal_audit_compact.groupby(group_cols, as_index=False)
                .agg(**agg_map)
                .sort_values(group_cols)
                .reset_index(drop=True)
            )
            signal_audit_method_summary.to_csv(out_dir / 'stageB_signal_audit_method_summary.csv', index=False)
            p3c.save_table_img(
                signal_audit_method_summary,
                'Stage B Signal Audit (method-level compact summary)',
                out_dir / 'stageB_signal_audit_method_summary.png'
            )

if 'recalib_next_iter' in globals() and recalib_next_iter is not None and len(recalib_next_iter) > 0:
    recalib_next_iter.to_csv(out_dir / 'stageB_rows.csv', index=False)

if 'stageA_pilot' in globals() and stageA_pilot is not None and len(stageA_pilot) > 0:
    stageA_pilot.to_csv(out_dir / 'stageA_pilot_rows.csv', index=False)

if 'STAGEB_CONTEXT' in globals():
    pd.DataFrame([STAGEB_CONTEXT]).to_json(out_dir / 'stageB_context.json', orient='records', indent=2)

print(f'Saved outputs to: {out_dir}')


In [ ]:
# Part 3C Figure: batch-indexed monitoring and retrospective evaluation design


Batch-indexed monitoring and retrospective evaluation design used in the hospital-level simulations. At T0 (today), simulated monitoring detects a signal by comparing the last 20,000-examination batch with the second last 20,000-examination batch, separated by a 2,000-examination gap. Retrospective evaluation uses three aligned 20,000-examination batches: the second last batch, the last batch, and a future batch after T0 with 36-month cancer outcomes. Signal verification compares the retrospective second last and last batches; the future batch is used for post-T0 outcome assessment.


In [ ]:
# Part 3C Stage C.0: Two evaluation windows setup (signal truth vs impact)
import numpy as np
import pandas as pd
import time
from pathlib import Path

# -----------------------------
# Two-window configuration
# -----------------------------
TWO_EVAL_TRUTH_SIZE = 20000       # window for signal truth assessment
TWO_EVAL_IMPACT_SIZE = 20000      # window for recalibration impact assessment
TWO_EVAL_HORIZON_MONTHS = 36      # sensitivity/specificity based on 36-month follow-up

# If True, start eval windows after the detection current window (no overlap).
# If False, allow evaluation to start at anchor index.
TWO_EVAL_NO_OVERLAP_WITH_DETECTION = True

# Signal truth rule: if max(|ΔSens|, |ΔSpec|) in truth window >= threshold => true shift.
TWO_EVAL_TRUTH_ABS_DELTA_THRESHOLD = 2.0

# Runtime controls
TWO_EVAL_RUN_ALL = False
TWO_EVAL_MAX_ROWS = 120
TWO_EVAL_RESET_CACHE = False
TWO_EVAL_PROFILE_EVERY = 20


def _safe_int(v, default=None):
    try:
        if pd.isna(v):
            return default
        return int(round(float(v)))
    except Exception:
        return default


def _safe_unique_int(vals):
    out = []
    seen = set()
    for x in vals:
        try:
            xi = int(round(float(x)))
        except Exception:
            continue
        if xi not in seen:
            seen.add(xi)
            out.append(xi)
    out.sort()
    return out


def _window_sens_spec(df_h, ai_col, flag_col, start_idx, win_size):
    n = len(df_h)
    if n == 0:
        return (np.nan, np.nan, 0, 0, 0)

    s = max(0, _safe_int(start_idx, 0))
    e = min(n, s + max(1, _safe_int(win_size, 1)))
    w = df_h.iloc[s:e].copy()

    if 'has_cancer' not in w.columns:
        return (np.nan, np.nan, len(w), 0, 0)

    w['has_cancer'] = w['has_cancer'].fillna(0).astype(int)
    if flag_col not in w.columns:
        return (np.nan, np.nan, len(w), int((w['has_cancer'] == 1).sum()), int((w['has_cancer'] == 0).sum()))

    pos = w[w['has_cancer'] == 1]
    neg = w[w['has_cancer'] == 0]

    sens = np.nan
    spec = np.nan
    if len(pos) > 0:
        sens = float((pos[flag_col].fillna(0).astype(int) == 1).mean() * 100.0)
    if len(neg) > 0:
        spec = float((neg[flag_col].fillna(0).astype(int) == 0).mean() * 100.0)

    return (sens, spec, len(w), int(len(pos)), int(len(neg)))


def _scenario_map_from_globals():
    sc_map = {}
    if 'PART3_SCENARIOS' in globals():
        for s in PART3_SCENARIOS:
            lab = str(s.get('label'))
            sc_map[lab] = s
    return sc_map


# Local evaluator for one Stage B row using two separate evaluation windows.
def p3c_eval_two_windows_row(row):
    sc_map = _scenario_map_from_globals()
    scenario_label = str(row.get('scenario'))
    s_cfg = sc_map.get(scenario_label, {})

    rad_mode = s_cfg.get('rad_mode', globals().get('PART3_RAD_MODE', 'combined'))
    calib_target_mode = s_cfg.get('target_mode', globals().get('PART3_TARGET_MODE', 'combined'))
    system_mode = s_cfg.get('system_mode', globals().get('PART3_SYSTEM_MODE', 'ai'))

    hospital = row.get('hospital')
    manufacturer = row.get('manufacturer')
    ai = row.get('ai_system')
    metric_key = row.get('signal_metric')
    recal_method = row.get('recal_method')

    prior_h = _safe_int(row.get('prior_window_size'), None)
    curr_h = _safe_int(row.get('current_window_size'), None)
    gap_use = _safe_int(row.get('gap_size'), 0)
    step_use = _safe_int(row.get('step_size'), 500)
    recal_window_h = _safe_int(row.get('recal_window_size'), curr_h if curr_h is not None else 20000)

    if prior_h is None or curr_h is None:
        return None

    d0 = df_part3.copy()
    d0 = d0[(d0[HOSPITAL_COL] == hospital) & (d0[MANUFACTURER_COL] == manufacturer)].copy()
    if len(d0) == 0:
        return None

    transition_map = None
    if 'STAGEB_CONTEXT' in globals() and isinstance(STAGEB_CONTEXT, dict):
        transition_map = STAGEB_CONTEXT.get('transition_idx_by_pair', None)
    if transition_map is None and 'transition_idx_by_pair' in globals():
        transition_map = globals().get('transition_idx_by_pair')

    pair_key = (hospital, manufacturer)
    t_override = None
    if isinstance(transition_map, dict) and pair_key in transition_map:
        t_override = _safe_int(transition_map[pair_key], None)

    calib_basis = None
    if 'STAGEB_CONTEXT' in globals() and isinstance(STAGEB_CONTEXT, dict):
        calib_basis = _safe_int(STAGEB_CONTEXT.get('calibration_basis_size'), None)
    if calib_basis is None:
        calib_basis = _safe_int(globals().get('P3C_CALIBRATION_BASIS_SIZE', 17000), 17000)

    base_result = run_part3_hospital_pipeline_v2(
        d0,
        hospital_name=str(hospital),
        cols=P3C_COLS,
        settings=P3C_SETTINGS,
        start_date=part3_start,
        calibration_strategy=str(row.get('calibration_strategy', PART3_CALIBRATION_STRATEGY)),
        rad_mode=rad_mode,
        target_mode=calib_target_mode,
        ref_size=prior_h,
        gap_size=gap_use,
        curr_size=curr_h,
        step_size=step_use,
        transition_index_override=t_override,
        calibration_basis_size=calib_basis,
    )
    if base_result is None:
        return None

    d = base_result['df_h'].copy()
    t_idx = _safe_int(base_result.get('transition_index'), None)
    if t_idx is None:
        return None

    if ai not in d.columns:
        return None

    # Base flags
    base_flag_col = f'{ai}_flagged_part3'
    if base_flag_col not in d.columns:
        return None

    # Signal centers (same mechanics as Stage B)
    signals = collect_part2_signal_centers(
        d,
        ai_columns=P3C_COLS.ai_columns,
        settings=P3C_SETTINGS,
        alpha=P3C_SETTINGS.recal_alpha,
        ref_size=prior_h,
        gap_size=gap_use,
        curr_size=curr_h,
        step_size=step_use,
        use_score_signals=P3C_SETTINGS.recal_use_score_signals,
    )
    signal_key = 'flagging_rate' if str(metric_key) == 'relative_flagging_rate' else str(metric_key)
    centers = signals.get(ai, {}).get(signal_key, [])
    sig_post = _safe_unique_int([c for c in centers if c is not None and float(c) >= float(t_idx)])

    first_signal_idx = int(sig_post[0]) if len(sig_post) > 0 else None

    # Build recalibrated flags for this specific row's metric+method
    recal_flag_col = f'{ai}_flagged_two_eval_recal'
    d[recal_flag_col] = d[base_flag_col].fillna(0).astype(int)

    targets = _get_target_metrics(
        d,
        ai_columns=P3C_COLS.ai_columns,
        transition_idx=t_idx,
        ref_size=prior_h,
        gap_size=gap_use,
        curr_size=curr_h,
        step_size=step_use,
    )

    target_val = targets.get(ai, {}).get(str(metric_key), np.nan)
    threshold_metric = str(metric_key)
    if str(row.get('target_mode', 'metric')) == 'sensitivity':
        target_val = base_result['radiologist_sensitivity_pre'].get(calib_target_mode, np.nan)
        threshold_metric = 'sensitivity'

    first_recal_idx = None
    recal_applied = False
    for sig_idx in sig_post:
        start_idx = max(0, int(sig_idx) - int(recal_window_h))
        recal_window = d.iloc[start_idx:int(sig_idx)]
        ref_start_idx = max(0, int(t_idx) - int(recal_window_h))
        ref_window = d.iloc[ref_start_idx:int(t_idx)]

        thr = _compute_recal_threshold(
            ai=ai,
            metric_key=str(metric_key),
            method=str(recal_method),
            target_value=target_val,
            recal_window=recal_window,
            ref_window=ref_window,
            threshold_metric=threshold_metric,
        )
        if pd.notna(thr):
            d.loc[int(sig_idx):, recal_flag_col] = (d.loc[int(sig_idx):, ai] >= thr).astype(int)
            recal_applied = True
            if first_recal_idx is None:
                first_recal_idx = int(sig_idx)

    # If system mode is AI+R1, union flags with R1 for both base and recal
    if str(system_mode) == 'ai_plus_r1':
        r1_flag = compute_radiologist_flag(d, mode='r1').astype(bool)
        base_union_col = f'{ai}_flagged_two_eval_base_union'
        recal_union_col = f'{ai}_flagged_two_eval_recal_union'
        d[base_union_col] = ((d[base_flag_col].fillna(0).astype(int) == 1) | r1_flag).astype(int)
        d[recal_union_col] = ((d[recal_flag_col].fillna(0).astype(int) == 1) | r1_flag).astype(int)
        base_flag_use = base_union_col
        recal_flag_use = recal_union_col
    else:
        base_flag_use = base_flag_col
        recal_flag_use = recal_flag_col

    target_sens = base_result['radiologist_sensitivity_pre'].get(calib_target_mode, np.nan)
    target_spec = base_result['radiologist_specificity_pre'].get(calib_target_mode, np.nan)

    # Anchors for separate windows
    if TWO_EVAL_NO_OVERLAP_WITH_DETECTION:
        min_eval_start = int(t_idx + curr_h)
    else:
        min_eval_start = int(t_idx)

    truth_anchor = int(first_signal_idx) if first_signal_idx is not None else None
    if truth_anchor is None:
        truth_start = None
    else:
        truth_start = max(int(truth_anchor), int(min_eval_start))

    impact_anchor_raw = int(first_recal_idx) if first_recal_idx is not None else (int(first_signal_idx) if first_signal_idx is not None else None)
    if impact_anchor_raw is None:
        impact_start = None
    else:
        impact_start = max(int(impact_anchor_raw), int(min_eval_start))

    # Truth window metrics (base only is primary for truth classification)
    if truth_start is None:
        truth_sens_base = np.nan
        truth_spec_base = np.nan
        truth_n = 0
        truth_pos = 0
        truth_neg = 0
        truth_delay = np.nan
        truth_label = 'FN_no_signal'
        truth_abs_dev_base = np.nan
        truth_delta_sens_base = np.nan
        truth_delta_spec_base = np.nan
    else:
        truth_sens_base, truth_spec_base, truth_n, truth_pos, truth_neg = _window_sens_spec(
            d, ai, base_flag_use, truth_start, TWO_EVAL_TRUTH_SIZE
        )
        truth_delay = float(truth_start - t_idx)
        truth_delta_sens_base = (
            float(truth_sens_base - target_sens) if pd.notna(truth_sens_base) and pd.notna(target_sens) else np.nan
        )
        truth_delta_spec_base = (
            float(truth_spec_base - target_spec) if pd.notna(truth_spec_base) and pd.notna(target_spec) else np.nan
        )
        devs = [abs(v) for v in [truth_delta_sens_base, truth_delta_spec_base] if pd.notna(v)]
        truth_abs_dev_base = float(max(devs)) if len(devs) > 0 else np.nan

        if pd.isna(truth_abs_dev_base):
            truth_label = 'UNSURE_no_valid_truth_rate'
        elif truth_abs_dev_base >= float(TWO_EVAL_TRUTH_ABS_DELTA_THRESHOLD):
            truth_label = 'TP_true_shift'
        else:
            truth_label = 'FP_no_material_shift'

    # Impact window metrics (compare base vs recal in same window)
    if impact_start is None:
        impact_sens_base = np.nan
        impact_spec_base = np.nan
        impact_sens_recal = np.nan
        impact_spec_recal = np.nan
        impact_n = 0
        impact_pos = 0
        impact_neg = 0
        impact_delay = np.nan
    else:
        impact_sens_base, impact_spec_base, impact_n, impact_pos, impact_neg = _window_sens_spec(
            d, ai, base_flag_use, impact_start, TWO_EVAL_IMPACT_SIZE
        )
        impact_sens_recal, impact_spec_recal, _, _, _ = _window_sens_spec(
            d, ai, recal_flag_use, impact_start, TWO_EVAL_IMPACT_SIZE
        )
        impact_delay = float(impact_start - t_idx)

    impact_delta_sens_base = (
        float(impact_sens_base - target_sens) if pd.notna(impact_sens_base) and pd.notna(target_sens) else np.nan
    )
    impact_delta_sens_recal = (
        float(impact_sens_recal - target_sens) if pd.notna(impact_sens_recal) and pd.notna(target_sens) else np.nan
    )
    impact_delta_spec_base = (
        float(impact_spec_base - target_spec) if pd.notna(impact_spec_base) and pd.notna(target_spec) else np.nan
    )
    impact_delta_spec_recal = (
        float(impact_spec_recal - target_spec) if pd.notna(impact_spec_recal) and pd.notna(target_spec) else np.nan
    )

    impact_abs_sens_base = abs(impact_delta_sens_base) if pd.notna(impact_delta_sens_base) else np.nan
    impact_abs_sens_recal = abs(impact_delta_sens_recal) if pd.notna(impact_delta_sens_recal) else np.nan
    impact_abs_spec_base = abs(impact_delta_spec_base) if pd.notna(impact_delta_spec_base) else np.nan
    impact_abs_spec_recal = abs(impact_delta_spec_recal) if pd.notna(impact_delta_spec_recal) else np.nan

    impact_abs_sens_reduction = (
        float(impact_abs_sens_base - impact_abs_sens_recal)
        if pd.notna(impact_abs_sens_base) and pd.notna(impact_abs_sens_recal) else np.nan
    )
    impact_abs_spec_reduction = (
        float(impact_abs_spec_base - impact_abs_spec_recal)
        if pd.notna(impact_abs_spec_base) and pd.notna(impact_abs_spec_recal) else np.nan
    )

    base_pair = [x for x in [impact_abs_sens_base, impact_abs_spec_base] if pd.notna(x)]
    recal_pair = [x for x in [impact_abs_sens_recal, impact_abs_spec_recal] if pd.notna(x)]
    impact_abs_mean_base = float(np.mean(base_pair)) if len(base_pair) > 0 else np.nan
    impact_abs_mean_recal = float(np.mean(recal_pair)) if len(recal_pair) > 0 else np.nan
    impact_abs_mean_reduction = (
        float(impact_abs_mean_base - impact_abs_mean_recal)
        if pd.notna(impact_abs_mean_base) and pd.notna(impact_abs_mean_recal) else np.nan
    )

    return {
        'scenario': scenario_label,
        'hospital': hospital,
        'manufacturer': manufacturer,
        'ai_system': ai,
        'window_method': row.get('window_method'),
        'signal_metric': metric_key,
        'recal_method': recal_method,
        'prior_window_size': prior_h,
        'current_window_size': curr_h,
        'gap_size': gap_use,
        'step_size': step_use,
        'truth_eval_size': int(TWO_EVAL_TRUTH_SIZE),
        'impact_eval_size': int(TWO_EVAL_IMPACT_SIZE),
        'eval_horizon_months': int(TWO_EVAL_HORIZON_MONTHS),
        'truth_anchor_idx': truth_anchor,
        'truth_eval_start': truth_start,
        'impact_anchor_idx': impact_anchor_raw,
        'impact_eval_start': impact_start,
        'truth_delay_exams': truth_delay,
        'impact_delay_exams': impact_delay,
        'truth_n': int(truth_n),
        'truth_cancers': int(truth_pos),
        'truth_non_cancers': int(truth_neg),
        'impact_n': int(impact_n),
        'impact_cancers': int(impact_pos),
        'impact_non_cancers': int(impact_neg),
        'target_sensitivity': target_sens,
        'target_specificity': target_spec,
        'truth_sens_base': truth_sens_base,
        'truth_spec_base': truth_spec_base,
        'truth_delta_sens_base': truth_delta_sens_base,
        'truth_delta_spec_base': truth_delta_spec_base,
        'truth_abs_dev_base': truth_abs_dev_base,
        'truth_label': truth_label,
        'truth_tp_true_shift': int(truth_label == 'TP_true_shift'),
        'truth_fp_no_material_shift': int(truth_label == 'FP_no_material_shift'),
        'truth_fn_no_signal': int(truth_label == 'FN_no_signal'),
        'impact_sens_base': impact_sens_base,
        'impact_sens_recal': impact_sens_recal,
        'impact_spec_base': impact_spec_base,
        'impact_spec_recal': impact_spec_recal,
        'impact_delta_sens_base': impact_delta_sens_base,
        'impact_delta_sens_recal': impact_delta_sens_recal,
        'impact_delta_spec_base': impact_delta_spec_base,
        'impact_delta_spec_recal': impact_delta_spec_recal,
        'impact_abs_sens_base': impact_abs_sens_base,
        'impact_abs_sens_recal': impact_abs_sens_recal,
        'impact_abs_spec_base': impact_abs_spec_base,
        'impact_abs_spec_recal': impact_abs_spec_recal,
        'impact_abs_sens_reduction': impact_abs_sens_reduction,
        'impact_abs_spec_reduction': impact_abs_spec_reduction,
        'impact_abs_mean_base': impact_abs_mean_base,
        'impact_abs_mean_recal': impact_abs_mean_recal,
        'impact_abs_mean_reduction': impact_abs_mean_reduction,
        'recal_applied': bool(recal_applied),
        'two_eval_no_overlap': bool(TWO_EVAL_NO_OVERLAP_WITH_DETECTION),
    }

print('Two-window helpers ready.')
print('truth_size=', TWO_EVAL_TRUTH_SIZE, '| impact_size=', TWO_EVAL_IMPACT_SIZE,
      '| no_overlap=', TWO_EVAL_NO_OVERLAP_WITH_DETECTION,
      '| truth_abs_delta_threshold=', TWO_EVAL_TRUTH_ABS_DELTA_THRESHOLD)


In [ ]:
# Part 3C Stage C.1: Run two-window evaluation (chunked)
if 'df_part3' not in globals():
    print('df_part3 not found. Load the main dataset first.')
elif 'p3c_eval_two_windows_row' not in globals():
    print('Two-window helpers not found. Run Stage C.0 first.')
else:
    out_dir = Path('outputs/2025_12_29_counterfactual_manufacturer_swap_setup/part3c_next_iteration')
    stageb_rows_path = out_dir / 'stageB_rows.csv'

    source_rows = None
    source_name = None
    if 'recalib_next_iter' in globals() and recalib_next_iter is not None and len(recalib_next_iter) > 0:
        source_rows = recalib_next_iter.copy()
        source_name = 'in-memory recalib_next_iter'
    elif stageb_rows_path.exists():
        source_rows = pd.read_csv(stageb_rows_path)
        source_name = f'CSV fallback: {stageb_rows_path}'
        recalib_next_iter = source_rows.copy()
    else:
        print('No Stage B rows available. Run Stage B.5 or export stageB_rows.csv first.')

    if source_rows is not None and len(source_rows) > 0:
        print('Using Stage B source:', source_name, '| rows =', len(source_rows))

        # Build evaluation plan from Stage B rows (deduplicated at method/scenario/system level)
        key_cols = [
            'scenario', 'hospital', 'manufacturer', 'ai_system',
            'window_method', 'signal_metric', 'recal_method',
            'prior_window_size', 'current_window_size', 'gap_size', 'step_size',
            'recal_window_size', 'calibration_strategy', 'target_mode'
        ]
        key_cols = [c for c in key_cols if c in source_rows.columns]

        if len(key_cols) == 0:
            print('No valid key columns found in source rows.')
        else:
            two_eval_plan = source_rows[key_cols].drop_duplicates().reset_index(drop=True)
            two_eval_plan = two_eval_plan.sort_values(
                ['window_method', 'scenario', 'hospital', 'manufacturer', 'ai_system', 'signal_metric', 'recal_method'],
                na_position='last'
            ).reset_index(drop=True)

            display(two_eval_plan.head(25))
            print('Two-window plan rows:', len(two_eval_plan))

            if TWO_EVAL_RUN_ALL:
                run_indices = list(range(len(two_eval_plan)))
            else:
                run_indices = list(range(min(len(two_eval_plan), int(TWO_EVAL_MAX_ROWS))))

            if 'TWO_EVAL_CACHE' not in globals() or TWO_EVAL_RESET_CACHE:
                TWO_EVAL_CACHE = {}

            print('Two-window run indices:', f"0..{run_indices[-1]}" if len(run_indices) > 0 else 'none')

            t_all = time.perf_counter()
            ok_count = 0
            none_count = 0
            for k, i in enumerate(run_indices):
                row = two_eval_plan.iloc[i]
                cache_key = tuple(row[c] for c in key_cols)
                if cache_key in TWO_EVAL_CACHE:
                    continue

                out = p3c_eval_two_windows_row(row)
                if out is not None:
                    out['plan_idx'] = int(i)
                    TWO_EVAL_CACHE[cache_key] = out
                    ok_count += 1
                else:
                    none_count += 1

                if ((k + 1) % int(TWO_EVAL_PROFILE_EVERY) == 0) or (k == len(run_indices) - 1):
                    print(f"processed {k+1}/{len(run_indices)} | elapsed {time.perf_counter()-t_all:0.1f}s | cache={len(TWO_EVAL_CACHE)} | ok={ok_count} | none={none_count}")

            two_eval_rows = pd.DataFrame(list(TWO_EVAL_CACHE.values())) if len(TWO_EVAL_CACHE) > 0 else pd.DataFrame()
            print('two_eval_rows:', len(two_eval_rows), '| total elapsed:', f"{time.perf_counter()-t_all:0.1f}s")
            if len(two_eval_rows) > 0:
                display(two_eval_rows.head(30))
            else:
                print('No two-window rows produced. Check Stage C.0 settings and whether Stage B rows contain valid combinations.')


In [ ]:
# Part 3C Stage C.2: Summarize two-window results + export
out_dir = Path('outputs/2025_12_29_counterfactual_manufacturer_swap_setup/part3c_next_iteration')
out_dir.mkdir(parents=True, exist_ok=True)

if ('two_eval_rows' not in globals()) or (two_eval_rows is None) or (len(two_eval_rows) == 0):
    fallback = out_dir / 'stageC_two_eval_rows.csv'
    if fallback.exists():
        two_eval_rows = pd.read_csv(fallback)
        print('Loaded fallback two_eval_rows from:', fallback, '| rows =', len(two_eval_rows))

if 'two_eval_rows' not in globals() or two_eval_rows is None or len(two_eval_rows) == 0:
    print('two_eval_rows is empty. Run Stage C.1 first.')
else:
    two_eval_summary = (
        two_eval_rows
        .groupby(['window_method', 'signal_metric'], as_index=False)
        .agg(
            truth_tp_rate=('truth_tp_true_shift', 'mean'),
            truth_fp_rate=('truth_fp_no_material_shift', 'mean'),
            truth_fn_rate=('truth_fn_no_signal', 'mean'),
            truth_delay_exams_mean=('truth_delay_exams', 'mean'),
            truth_abs_dev_base_mean=('truth_abs_dev_base', 'mean'),
            impact_delay_exams_mean=('impact_delay_exams', 'mean'),
            impact_abs_mean_base=('impact_abs_mean_base', 'mean'),
            impact_abs_mean_recal=('impact_abs_mean_recal', 'mean'),
            impact_abs_mean_reduction=('impact_abs_mean_reduction', 'mean'),
            impact_abs_sens_reduction=('impact_abs_sens_reduction', 'mean'),
            impact_abs_spec_reduction=('impact_abs_spec_reduction', 'mean'),
            n_rows=('window_method', 'size'),
        )
        .sort_values(['window_method', 'signal_metric'])
        .reset_index(drop=True)
    )

    # Optional by-hospital view
    two_eval_hospital_summary = (
        two_eval_rows
        .groupby(['hospital', 'manufacturer', 'window_method', 'signal_metric'], as_index=False)
        .agg(
            truth_tp_rate=('truth_tp_true_shift', 'mean'),
            truth_fp_rate=('truth_fp_no_material_shift', 'mean'),
            truth_fn_rate=('truth_fn_no_signal', 'mean'),
            impact_abs_mean_reduction=('impact_abs_mean_reduction', 'mean'),
            impact_abs_sens_reduction=('impact_abs_sens_reduction', 'mean'),
            impact_abs_spec_reduction=('impact_abs_spec_reduction', 'mean'),
            n_rows=('window_method', 'size'),
        )
        .sort_values(['hospital', 'manufacturer', 'window_method', 'signal_metric'])
        .reset_index(drop=True)
    )

    display(two_eval_summary)
    display(two_eval_hospital_summary.head(40))

    # Save CSVs
    two_eval_rows.to_csv(out_dir / 'stageC_two_eval_rows.csv', index=False)
    two_eval_summary.to_csv(out_dir / 'stageC_two_eval_summary.csv', index=False)
    two_eval_hospital_summary.to_csv(out_dir / 'stageC_two_eval_hospital_summary.csv', index=False)

    # Save images (reuse table renderer)
    p3c.save_table_img(two_eval_summary, 'Stage C: Two-Window Summary (Truth vs Impact)', out_dir / 'stageC_two_eval_summary.png')
    p3c.save_table_img(two_eval_hospital_summary.head(60), 'Stage C: Two-Window Hospital Summary (top 60 rows)', out_dir / 'stageC_two_eval_hospital_summary_top60.png')

    print('Saved Stage C outputs to:', out_dir)


In [ ]:
# Part 3C Stage C.3: Diagnostics (what is missing?)
from pathlib import Path
import pandas as pd

OUT_DIR = Path('outputs/2025_12_29_counterfactual_manufacturer_swap_setup/part3c_next_iteration')
PATH_STAGEB_ROWS = OUT_DIR / 'stageB_rows.csv'
PATH_STAGEC_ROWS = OUT_DIR / 'stageC_two_eval_rows.csv'


def _var_info(name, kind='any'):
    if name not in globals():
        return {'name': name, 'status': 'MISSING', 'detail': ''}
    v = globals()[name]
    if kind == 'df':
        if v is None:
            return {'name': name, 'status': 'MISSING', 'detail': 'None'}
        if isinstance(v, pd.DataFrame):
            return {'name': name, 'status': 'OK', 'detail': f'rows={len(v)} cols={len(v.columns)}'}
        return {'name': name, 'status': 'WARN', 'detail': f'type={type(v).__name__}'}
    return {'name': name, 'status': 'OK', 'detail': f'type={type(v).__name__}'}


checks = [
    _var_info('df_part3', 'df'),
    _var_info('P3C_COLS'),
    _var_info('P3C_SETTINGS'),
    _var_info('part3_start'),
    _var_info('STAGEB_CONTEXT'),
    _var_info('recalib_next_iter', 'df'),
    _var_info('two_eval_rows', 'df'),
]

df_checks = pd.DataFrame(checks)
display(df_checks)

file_checks = pd.DataFrame([
    {
        'file': str(PATH_STAGEB_ROWS),
        'exists': PATH_STAGEB_ROWS.exists(),
        'rows_if_csv': (len(pd.read_csv(PATH_STAGEB_ROWS)) if PATH_STAGEB_ROWS.exists() else None),
    },
    {
        'file': str(PATH_STAGEC_ROWS),
        'exists': PATH_STAGEC_ROWS.exists(),
        'rows_if_csv': (len(pd.read_csv(PATH_STAGEC_ROWS)) if PATH_STAGEC_ROWS.exists() else None),
    },
])
display(file_checks)

missing = set(df_checks[df_checks['status'] == 'MISSING']['name'].tolist())

print('')
print('Recommended next steps:')
if 'df_part3' in missing:
    print('1) Run your data-load/prep cells to recreate df_part3.')
if 'P3C_COLS' in missing or 'P3C_SETTINGS' in missing:
    print('2) Run "# Part 3C: Embedded helper library (self-contained)" and then "# Part 3C: Helper wiring (revised)".')
if 'STAGEB_CONTEXT' in missing:
    print('3) Run "# Part 3C Stage B.0: Methods comparison setup (run once)".')
if ('recalib_next_iter' in missing) and (not PATH_STAGEB_ROWS.exists()):
    print('4) Run Stage B method cells + Stage B.5, or ensure stageB_rows.csv exists in outputs.')
if 'two_eval_rows' in missing and (not PATH_STAGEC_ROWS.exists()):
    print('5) Run Stage C.0 -> Stage C.1 -> Stage C.2.')
if len(missing) == 0:
    print('All key in-memory prerequisites are present.')
if ('two_eval_rows' not in missing) and ('two_eval_rows' in globals()) and isinstance(two_eval_rows, pd.DataFrame) and len(two_eval_rows) > 0:
    print('Stage C rows are available in memory; you can run Stage C.2 directly.')
elif PATH_STAGEC_ROWS.exists():
    print('Stage C rows file exists; Stage C.2 can load fallback CSV and summarize.')


In [ ]:
# Part 3C: Recommended parameters from selected Stage A/B settings (dynamic)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

REC_SCENARIO_LABEL = 'single'   # set to 'double' for double-reader view

OUT_DIR = Path('outputs/2025_12_29_counterfactual_manufacturer_swap_setup/part3c_next_iteration')
OUT_DIR.mkdir(parents=True, exist_ok=True)


def _save_table_fallback(df, title, out_path):
    d = df.copy()
    for c in d.columns:
        if pd.api.types.is_numeric_dtype(d[c]):
            if 'delay' in str(c).lower():
                d[c] = d[c].map(lambda v: '' if pd.isna(v) else f'{float(v):0.2f}')
            else:
                d[c] = d[c].map(lambda v: '' if pd.isna(v) else f'{float(v):0.2f}')
    fig_h = 1.6 + 0.45 * len(d)
    fig_w = max(12, 1.6 * len(d.columns))
    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    ax.axis('off')
    table = ax.table(
        cellText=d.values,
        colLabels=d.columns,
        cellLoc='center',
        loc='center',
        bbox=[0, 0, 1, 1],
    )
    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.scale(1.2, 1.6)
    header_color = '#4472C4'
    alt_row_color = '#E7E6E6'
    for j in range(len(d.columns)):
        cell = table[(0, j)]
        cell.set_facecolor(header_color)
        cell.set_text_props(weight='bold', color='white')
    for r in range(1, len(d) + 1):
        for j in range(len(d.columns)):
            cell = table[(r, j)]
            cell.set_facecolor(alt_row_color if r % 2 == 0 else 'white')
    for cell in table.get_celld().values():
        cell.set_edgecolor('black')
        cell.set_linewidth(1.1)
    plt.title(title, fontsize=14, fontweight='bold', pad=16)
    plt.tight_layout()
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(out_path, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close(fig)


def _save_table(df, title, out_path):
    if 'p3c' in globals() and hasattr(p3c, 'save_table_img'):
        try:
            p3c.save_table_img(df, title, out_path)
            return
        except Exception:
            pass
    _save_table_fallback(df, title, out_path)


# Source: in-memory Stage B rows if available, otherwise CSV fallback
source_rows = None
if 'recalib_next_iter' in globals() and recalib_next_iter is not None and len(recalib_next_iter) > 0:
    source_rows = recalib_next_iter.copy()
else:
    csv_fallback = OUT_DIR / 'stageB_rows.csv'
    if csv_fallback.exists():
        source_rows = pd.read_csv(csv_fallback)

if source_rows is None or len(source_rows) == 0:
    print('No Stage B rows found. Run Stage B.5 or load stageB_rows.csv first.')
else:
    d = source_rows.copy()

    # Resolve selected step/gap from Stage A (or fallback to modal values in Stage B rows)
    gap_vals = pd.to_numeric(d.get('gap_size'), errors='coerce')
    step_vals = pd.to_numeric(d.get('step_size'), errors='coerce')

    selected_gap = int(globals().get('SELECTED_GAP_SIZE', gap_vals.mode().iloc[0] if gap_vals.notna().any() else 0))
    selected_step = int(globals().get('SELECTED_STEP_SIZE', step_vals.mode().iloc[0] if step_vals.notna().any() else 500))

    d = d[(pd.to_numeric(d['gap_size'], errors='coerce') == float(selected_gap)) &
          (pd.to_numeric(d['step_size'], errors='coerce') == float(selected_step))].copy()

    if len(d) == 0:
        print('No rows after filtering by selected gap/step:', selected_gap, selected_step)
    else:
        # Scenario subset for recommendation view
        if REC_SCENARIO_LABEL:
            d = d[d['scenario'].astype(str).str.lower().str.contains(str(REC_SCENARIO_LABEL).lower(), na=False)].copy()

        if len(d) == 0:
            print('No rows for scenario filter:', REC_SCENARIO_LABEL)
        else:
            # Choose recommended method/current-size from fixed methods with transparent penalty
            fixed = d[d['window_method'].astype(str).str.startswith('fixed_')].copy()
            if len(fixed) == 0:
                fixed = d.copy()

            score_df = (
                fixed.groupby(['window_method', 'current_window_size'], as_index=False)
                .agg(
                    fn_rate=('false_negative_detected', 'mean'),
                    tp_rate=('true_positive_detected', 'mean'),
                    fp_pre_mean=('signal_count_pre', 'mean'),
                    delay_mean=('first_signal_delay', 'mean'),
                    abs_sens_mean=('mean_abs_delta_sens_recal', 'mean'),
                    abs_spec_mean=('mean_abs_delta_spec_recal', 'mean'),
                    worst_neg_sens=('worst_negative_delta_sens_recal', 'min'),
                    worst_neg_spec=('worst_negative_delta_spec_recal', 'min'),
                    n_rows=('window_method', 'size'),
                )
            )

            score_df['selection_score'] = (
                100.0 * score_df['fn_rate']
                + score_df['abs_sens_mean']
                + score_df['abs_spec_mean']
                + score_df['worst_neg_sens'].clip(upper=0).abs()
                + score_df['worst_neg_spec'].clip(upper=0).abs()
                + 0.1 * score_df['fp_pre_mean']
                + 0.0001 * score_df['delay_mean'].fillna(score_df['delay_mean'].max())
            )

            score_df = score_df.sort_values(
                ['selection_score', 'fn_rate', 'abs_sens_mean', 'abs_spec_mean', 'current_window_size'],
                ascending=[True, True, True, True, True]
            ).reset_index(drop=True)

            best = score_df.iloc[0]
            rec_method = str(best['window_method'])
            rec_curr = int(round(float(best['current_window_size'])))

            rec_rows = d[
                (d['window_method'].astype(str) == rec_method)
                & (pd.to_numeric(d['current_window_size'], errors='coerce') == float(rec_curr))
            ].copy()

            rec_display_cols = [
                'scenario', 'hospital', 'manufacturer', 'ai_system', 'signal_metric',
                'window_method', 'current_window_size', 'gap_size', 'step_size',
                'signal_count_post', 'signal_count_pre', 'first_signal_delay',
                'true_positive_detected', 'false_negative_detected',
                'mean_abs_delta_sens_recal', 'mean_abs_delta_spec_recal', 'avg_threshold_changes'
            ]
            rec_display_cols = [c for c in rec_display_cols if c in rec_rows.columns]
            rec_table = rec_rows[rec_display_cols].copy()
            rec_table = rec_table.rename(columns={
                'scenario': 'Scenario',
                'hospital': 'Hospital',
                'manufacturer': 'Manufacturer',
                'ai_system': 'AI',
                'signal_metric': 'Metric',
                'window_method': 'Method',
                'current_window_size': 'Current window',
                'gap_size': 'Gap',
                'step_size': 'Step',
                'signal_count_post': 'signals_post_start',
                'signal_count_pre': 'signals_pre_start',
                'first_signal_delay': 'First signal delay (exams)',
                'true_positive_detected': 'TP',
                'false_negative_detected': 'FN',
                'mean_abs_delta_sens_recal': '|ΔSens| mean',
                'mean_abs_delta_spec_recal': '|ΔSpec| mean',
                'avg_threshold_changes': 'avg threshold changes',
            })

            rec_table = rec_table.sort_values(
                ['Hospital', 'Manufacturer', 'AI', 'Metric'],
                na_position='last'
            ).reset_index(drop=True)

            summary = (
                rec_table.groupby(['Metric'], as_index=False)
                .agg(
                    mean_delay=('First signal delay (exams)', 'mean'),
                    mean_signals_post=('signals_post_start', 'mean'),
                    mean_signals_pre=('signals_pre_start', 'mean'),
                    tp_rate=('TP', 'mean'),
                    fn_rate=('FN', 'mean'),
                    mean_abs_delta_sens=('|ΔSens| mean', 'mean'),
                    mean_abs_delta_spec=('|ΔSpec| mean', 'mean'),
                    avg_threshold_changes=('avg threshold changes', 'mean'),
                )
                .sort_values('Metric')
                .reset_index(drop=True)
            )

            # Export CSV
            score_df.to_csv(OUT_DIR / 'recommended_params_selected_rule.csv', index=False)
            rec_table.to_csv(OUT_DIR / 'recommended_params_selected_rows.csv', index=False)
            summary.to_csv(OUT_DIR / 'recommended_params_selected_summary.csv', index=False)

            # Export images
            rule_title = (
                f'Recommended parameter selection | scenario={REC_SCENARIO_LABEL} '
                f'| selected gap={selected_gap}, step={selected_step}'
            )
            _save_table(score_df, rule_title, OUT_DIR / 'recommended_params_selected_rule.png')

            main_title = (
                f'Recommended Params (selected) | scenario={REC_SCENARIO_LABEL} '
                f'| method={rec_method}, curr={rec_curr}, gap={selected_gap}, step={selected_step}'
            )
            _save_table(rec_table, main_title, OUT_DIR / 'recommended_params_selected_rows.png')
            _save_table(summary, 'Recommended Params (selected) - Summary by metric', OUT_DIR / 'recommended_params_selected_summary.png')

            by_hospital_dir = OUT_DIR / 'recommended_params_selected_by_hospital'
            by_hospital_dir.mkdir(parents=True, exist_ok=True)
            for hospital, df_h in rec_table.groupby('Hospital'):
                safe = str(hospital).replace('/', '_').replace(' ', '_')
                _save_table(
                    df_h,
                    f'Recommended Params (selected) | {hospital}',
                    by_hospital_dir / f'recommended_params_selected_{safe}.png'
                )

            print('Selected parameters from Stage A/B:')
            print('  scenario:', REC_SCENARIO_LABEL)
            print('  gap:', selected_gap)
            print('  step:', selected_step)
            print('  method:', rec_method)
            print('  current window:', rec_curr)
            print('Saved recommended-params outputs to:', OUT_DIR)


In [ ]:
# Runtime Profiler Report: Top 10 slowest cells
import pandas as pd
import hashlib
from pathlib import Path

if '_NB_RUNTIME_LOG' not in globals() or len(_NB_RUNTIME_LOG) == 0:
    print('No runtime logs found. Run "Runtime Profiler Setup" first, then execute cells.')
else:
    log_df = pd.DataFrame(_NB_RUNTIME_LOG)

    # Build cell metadata map from notebook source
    nb_file = Path('2026_2_4_hospital_level_simulations.ipynb')
    meta_rows = []
    if nb_file.exists():
        import json
        nb_local = json.loads(nb_file.read_text())
        for idx, c in enumerate(nb_local.get('cells', [])):
            if c.get('cell_type') != 'code':
                continue
            src = ''.join(c.get('source', []))
            h = hashlib.md5(src.encode('utf-8')).hexdigest()[:12]

            # Prefer first comment line as description
            desc = ''
            for line in src.splitlines():
                s = line.strip()
                if not s:
                    continue
                if s.startswith('#'):
                    desc = s.lstrip('#').strip()
                    break
            if not desc:
                for line in src.splitlines():
                    s = line.strip()
                    if s:
                        desc = s[:160]
                        break

            meta_rows.append({
                'cell_hash': h,
                'cell_index': idx,
                'cell_description': desc,
            })

    meta_df = pd.DataFrame(meta_rows)

    agg = (
        log_df.groupby('cell_hash', as_index=False)
        .agg(
            total_sec=('elapsed_sec', 'sum'),
            mean_sec=('elapsed_sec', 'mean'),
            max_sec=('elapsed_sec', 'max'),
            runs=('elapsed_sec', 'size'),
            last_run=('timestamp', 'max'),
        )
        .sort_values('total_sec', ascending=False)
    )

    report = agg.merge(meta_df, on='cell_hash', how='left')

    def _advice(total_sec):
        if pd.isna(total_sec):
            return ''
        if total_sec >= 300:
            return 'Very expensive: rerun only if upstream data/settings changed.'
        if total_sec >= 120:
            return 'Expensive: avoid rerun unless needed.'
        if total_sec >= 30:
            return 'Moderate: rerun selectively.'
        return 'Light.'

    report['rerun_advice'] = report['total_sec'].map(_advice)

    top10 = report.head(10).copy()
    show_cols = [
        'cell_index', 'cell_description', 'runs', 'total_sec', 'mean_sec', 'max_sec', 'last_run', 'rerun_advice'
    ]
    show_cols = [c for c in show_cols if c in top10.columns]
    top10 = top10[show_cols]

    display(top10)

    out_dir = Path('outputs/2025_12_29_counterfactual_manufacturer_swap_setup/part3c_next_iteration')
    out_dir.mkdir(parents=True, exist_ok=True)
    top10.to_csv(out_dir / 'runtime_top10_cells.csv', index=False)

    if 'p3c' in globals() and hasattr(p3c, 'save_table_img') and len(top10) > 0:
        try:
            p3c.save_table_img(top10, 'Runtime Top 10 Cells', out_dir / 'runtime_top10_cells.png')
        except Exception as e:
            print('Could not save runtime table image:', e)

    print('Saved runtime report to:', out_dir / 'runtime_top10_cells.csv')


## Part 3D: Oracle Sens/Spec Recalibration Validation (Self-contained)

This section is independent from previous Stage A/B/C logic and focuses on **recalibration correctness**.

What it does:
- Uses 3 explicit windows at each decision time: **prior**, **current**, **future**.
- Uses fixed sizes by default: prior=current=future=20,000 exams.
- Uses a **threshold diary/logbook**: for each decision point, records threshold before/after, where it applies, and resulting performance.
- Reports both simple means and explicit **time-weighted means**.
- Choice 1: step/gap sweep with `gap = step`.
- Choice 2: metric-objective comparison with two scatter plots:
  - y = sensitivity, x = 1 - specificity
  - y = sensitivity, x = PPV
- Optional supplementary repeat for different window sizes.

In [ ]:
# Part 3D.0 - Setup + helpers (self-contained)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import math


# ------------------------------
# Configuration (safe defaults)
# ------------------------------
P3D_OUTPUT_DIR = Path('outputs/2025_12_29_counterfactual_manufacturer_swap_setup/part3d_oracle_recalibration')
P3D_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

P3D_HOSPITAL = 'Karolinska University Hospital'
P3D_MANUFACTURER = 'Hologic'
P3D_RAD_MODE = 'combined'  # radiologist flagged if R1 OR R2 marked DISCUSSION

# Fixed geometry requested for primary analysis.
P3D_PRIOR_SIZE = 20000
P3D_CURRENT_SIZE = 20000
P3D_FUTURE_SIZE = 20000

# Choice 1: step-size sweep with gap=step (fast default; step=1 removed).
# Fast list: 500, 1000, 2000, 4000
P3D_STEP_GAP_LIST = [500, 1000, 2000, 4000]
P3D_PRIMARY_OBJECTIVE = 'flagging_rate'  # primary signal metric for choice 1

# Threshold diary example controls (for clear logbook figure).
P3D_DIARY_EXAMPLE_AI = None   # None -> first available AI
P3D_DIARY_EXAMPLE_STEP = 2000 # diary example shown at selected practical step

# Choice 2: signal-metric comparison at fixed geometry.
P3D_OBJECTIVES_FOR_COMPARISON = ['flagging_rate', 'relative_flagging_rate', 'p_ai_given_rad', 'p_rad_given_ai']
P3D_COMPARISON_STEP = 1000
P3D_COMPARISON_GAP = 1000

# Execution controls
P3D_AI_SYSTEMS = None  # None -> infer from available columns (usually lunit/therapixel/vara)
P3D_MAX_THRESHOLD_CANDIDATES = 101
P3D_TRIGGER_TOLERANCE = 0.0      # minimum drift magnitude required for recalibration
P3D_SIGNAL_MODE = 'pvalue'         # 'drift' or 'pvalue' (default now uses p-value signal test)
P3D_SIGNAL_P_THRESHOLD = 0.05      # used when P3D_SIGNAL_MODE='pvalue'
P3D_MIN_RECALIBRATION_INTERVAL = 2000 # minimum exams between threshold changes (practical cadence default)
P3D_MAX_DECISION_POINTS = None     # e.g., 120 to speed up, None = full

# Optional trigger-tolerance sweep (fast sensitivity analysis).
P3D_RUN_TRIGGER_SWEEP = True
P3D_TRIGGER_TOLERANCE_GRID = [0.0, 0.5, 1.0, 2.0]
P3D_TRIGGER_SWEEP_STEP = 1000

# Optional supplementary (window-size repeat for point 2)
P3D_RUN_SUPPLEMENTARY = True
P3D_SUPP_WINDOW_SIZES = [20000, 10000, 5000, 1000]


# ------------------------------
# Utility helpers
# ------------------------------
def _p3d_get_columns(df):
    hospital_col = globals().get('HOSPITAL_COL', 'hospital')
    manufacturer_col = globals().get('MANUFACTURER_COL', 'manufacturer')
    date_col = globals().get('DATE_COL', 'date')

    if hospital_col not in df.columns:
        if 'hospital' in df.columns:
            hospital_col = 'hospital'
        else:
            raise ValueError('Hospital column not found.')

    if manufacturer_col not in df.columns:
        if 'manufacturer' in df.columns:
            manufacturer_col = 'manufacturer'
        else:
            manufacturer_col = None

    if date_col not in df.columns:
        for c in ['exam_date', 'date', 'study_date', 'StudyDate', 'exam_datetime']:
            if c in df.columns:
                date_col = c
                break

    ai_cols = globals().get('AI_COLUMNS', None)
    if ai_cols is None:
        ai_guess = [c for c in ['lunit', 'therapixel', 'vara'] if c in df.columns]
        if len(ai_guess) == 0:
            raise ValueError('AI score columns not found (expected lunit/therapixel/vara or AI_COLUMNS).')
        ai_cols = ai_guess
    ai_cols = [c for c in ai_cols if c in df.columns]

    cancer_col = 'has_cancer'
    if cancer_col not in df.columns:
        for c in ['has_cancer_36m', 'cancer_36m', 'cancer', 'is_cancer']:
            if c in df.columns:
                cancer_col = c
                break
    if cancer_col not in df.columns:
        raise ValueError('Cancer outcome column not found (expected has_cancer or equivalent).')

    return {
        'hospital': hospital_col,
        'manufacturer': manufacturer_col,
        'date': date_col,
        'ai_cols': ai_cols,
        'cancer': cancer_col,
    }


def _p3d_prepare_pair_df(df_part3, cols, hospital, manufacturer):
    d = df_part3.copy()
    d = d[d[cols['hospital']] == hospital].copy()
    if cols['manufacturer'] is not None and manufacturer is not None:
        d = d[d[cols['manufacturer']] == manufacturer].copy()

    if len(d) == 0:
        return d

    if cols['date'] in d.columns:
        d[cols['date']] = pd.to_datetime(d[cols['date']], format='mixed', errors='coerce')
        d = d.sort_values([cols['date']]).reset_index(drop=True)
    else:
        d = d.reset_index(drop=True)

    d[cols['cancer']] = d[cols['cancer']].fillna(0).astype(int)
    return d


def _p3d_add_rad_flag(df, rad_mode='single'):
    d = df.copy()
    if 'compute_radiologist_flag' in globals():
        try:
            d['rad_flagged'] = compute_radiologist_flag(d, mode=rad_mode).fillna(0).astype(int)
            return d
        except Exception:
            pass

    # Fallback if helper is unavailable.
    for c in ['rad_flagged', 'radiologist_flagged', 'rad_positive', 'rad_recall']:
        if c in d.columns:
            d['rad_flagged'] = d[c].fillna(0).astype(int)
            return d

    raise ValueError('Could not create radiologist flag. Ensure compute_radiologist_flag exists or a rad_flagged-like column is present.')


def _p3d_rate_stats_from_flags(flag, cancer):
    f = np.asarray(flag).astype(int)
    y = np.asarray(cancer).astype(int)
    valid = ~np.isnan(f) & ~np.isnan(y)
    f = f[valid]
    y = y[valid]

    p = int((y == 1).sum())
    n = int((y == 0).sum())
    tp = int(((f == 1) & (y == 1)).sum())
    fp = int(((f == 1) & (y == 0)).sum())
    tn = int(((f == 0) & (y == 0)).sum())
    fn = int(((f == 0) & (y == 1)).sum())

    sens = np.nan if p == 0 else 100.0 * tp / p
    spec = np.nan if n == 0 else 100.0 * tn / n
    ppv = np.nan if (tp + fp) == 0 else 100.0 * tp / (tp + fp)

    return {
        'sens': sens,
        'spec': spec,
        'ppv': ppv,
        'tp': tp,
        'fp': fp,
        'tn': tn,
        'fn': fn,
        'p': p,
        'n': n,
        'n_total': len(y),
    }


def _p3d_rate_stats_from_threshold(scores, cancer, thr):
    s = np.asarray(scores).astype(float)
    y = np.asarray(cancer).astype(int)
    valid = ~np.isnan(s) & ~np.isnan(y)
    s = s[valid]
    y = y[valid]
    if len(s) == 0:
        return {
            'sens': np.nan, 'spec': np.nan, 'ppv': np.nan,
            'tp': 0, 'fp': 0, 'tn': 0, 'fn': 0, 'p': 0, 'n': 0, 'n_total': 0
        }
    flag = (s >= float(thr)).astype(int)
    return _p3d_rate_stats_from_flags(flag, y)


def _p3d_candidate_thresholds(scores, max_candidates=101):
    s = np.asarray(scores).astype(float)
    s = s[~np.isnan(s)]
    if len(s) == 0:
        return np.array([])

    if len(np.unique(s)) <= max_candidates:
        th = np.unique(s)
    else:
        qs = np.linspace(0.0, 1.0, max_candidates)
        th = np.unique(np.quantile(s, qs))

    # Add extreme bounds for stable search.
    th = np.unique(np.concatenate(([s.min() - 1e-9], th, [s.max() + 1e-9])))
    return th


def _p3d_signal_stats_from_threshold(window_df, ai_col, thr):
    scores = pd.to_numeric(window_df[ai_col], errors='coerce').to_numpy(dtype=float)
    rad = pd.to_numeric(window_df['rad_flagged'], errors='coerce').fillna(0).to_numpy(dtype=int)
    valid = ~np.isnan(scores)
    scores = scores[valid]
    rad = rad[valid]
    if len(scores) == 0:
        return {
            'flagging_rate': np.nan,
            'relative_flagging_rate': np.nan,
            'p_ai_given_rad': np.nan,
            'p_rad_given_ai': np.nan,
            'n_total': 0,
            'ai_pos': 0,
            'rad_pos': 0,
            'both_pos': 0,
        }

    ai = (scores >= float(thr)).astype(int)
    n_total = int(len(ai))
    ai_pos = int(ai.sum())
    rad_pos = int((rad == 1).sum())
    both_pos = int(((ai == 1) & (rad == 1)).sum())

    flagging_rate = 100.0 * ai_pos / n_total if n_total > 0 else np.nan
    relative_flagging_rate = np.nan if rad_pos <= 0 else float(ai_pos) / float(rad_pos)
    p_ai_given_rad = np.nan if rad_pos <= 0 else 100.0 * both_pos / rad_pos
    p_rad_given_ai = np.nan if ai_pos <= 0 else 100.0 * both_pos / ai_pos

    return {
        'flagging_rate': float(flagging_rate) if pd.notna(flagging_rate) else np.nan,
        'relative_flagging_rate': float(relative_flagging_rate) if pd.notna(relative_flagging_rate) else np.nan,
        'p_ai_given_rad': float(p_ai_given_rad) if pd.notna(p_ai_given_rad) else np.nan,
        'p_rad_given_ai': float(p_rad_given_ai) if pd.notna(p_rad_given_ai) else np.nan,
        'n_total': n_total,
        'ai_pos': ai_pos,
        'rad_pos': rad_pos,
        'both_pos': both_pos,
    }


def _p3d_signal_pvalue(prior_sig, curr_sig, signal_metric):
    metric = str(signal_metric)
    if metric in ('flagging_rate', 'relative_flagging_rate'):
        return _p3d_two_prop_pvalue(
            curr_sig.get('ai_pos', 0), curr_sig.get('n_total', 0),
            prior_sig.get('ai_pos', 0), prior_sig.get('n_total', 0),
        )
    if metric == 'p_ai_given_rad':
        return _p3d_two_prop_pvalue(
            curr_sig.get('both_pos', 0), curr_sig.get('rad_pos', 0),
            prior_sig.get('both_pos', 0), prior_sig.get('rad_pos', 0),
        )
    if metric == 'p_rad_given_ai':
        return _p3d_two_prop_pvalue(
            curr_sig.get('both_pos', 0), curr_sig.get('ai_pos', 0),
            prior_sig.get('both_pos', 0), prior_sig.get('ai_pos', 0),
        )
    return np.nan


def _p3d_choose_threshold(window_df, ai_col, cancer_col, targets, objective='flagging_rate', signal_targets=None, max_candidates=101):
    scores = window_df[ai_col].to_numpy(dtype=float)
    cancer = window_df[cancer_col].to_numpy(dtype=int)
    valid = ~np.isnan(scores)
    scores = scores[valid]
    cancer = cancer[valid]

    if len(scores) == 0:
        return np.nan, np.nan

    thr_candidates = _p3d_candidate_thresholds(scores, max_candidates=max_candidates)
    if len(thr_candidates) == 0:
        return np.nan, np.nan

    signal_metrics = {'flagging_rate', 'relative_flagging_rate', 'p_ai_given_rad', 'p_rad_given_ai'}
    metric = str(objective).strip()
    if metric not in signal_metrics:
        return np.nan, np.nan

    if signal_targets is None:
        signal_targets = {}

    target_val = pd.to_numeric(pd.Series([signal_targets.get(metric, np.nan)]), errors='coerce').iloc[0]
    if pd.isna(target_val):
        return np.nan, np.nan

    obj = np.full(len(thr_candidates), np.nan, dtype=float)
    for j, thr in enumerate(thr_candidates):
        sig = _p3d_signal_stats_from_threshold(window_df, ai_col, thr)
        curr_val = pd.to_numeric(pd.Series([sig.get(metric, np.nan)]), errors='coerce').iloc[0]
        obj[j] = np.nan if pd.isna(curr_val) else abs(float(curr_val) - float(target_val))

    obj = np.where(np.isnan(obj), np.inf, obj)
    if np.isinf(obj).all():
        return np.nan, np.nan
    best_idx = int(np.argmin(obj))
    return float(thr_candidates[best_idx]), float(obj[best_idx])


def _p3d_choose_initial_threshold_eval_targets(window_df, ai_col, cancer_col, targets, max_candidates=101):
    """
    One-time initialization helper: choose threshold on the first prior window
    by minimizing |sens-target_sens| + |spec-target_spec|.
    This is used only to anchor the starting threshold before signal-driven
    testing/recalibration begins.
    """
    scores = window_df[ai_col].to_numpy(dtype=float)
    cancer = window_df[cancer_col].to_numpy(dtype=int)
    valid = ~np.isnan(scores)
    scores = scores[valid]
    cancer = cancer[valid]

    if len(scores) == 0:
        return np.nan, np.nan

    thr_candidates = _p3d_candidate_thresholds(scores, max_candidates=max_candidates)
    if len(thr_candidates) == 0:
        return np.nan, np.nan

    mask = scores[:, None] >= thr_candidates[None, :]
    pos = (cancer == 1)
    neg = (cancer == 0)

    tp = (mask & pos[:, None]).sum(axis=0).astype(float)
    fp = (mask & neg[:, None]).sum(axis=0).astype(float)
    p = float(pos.sum())
    n = float(neg.sum())

    sens = np.full(len(thr_candidates), np.nan)
    spec = np.full(len(thr_candidates), np.nan)
    if p > 0:
        sens = 100.0 * tp / p
    if n > 0:
        tn = n - fp
        spec = 100.0 * tn / n

    t_sens = float(targets.get('sens', np.nan))
    t_spec = float(targets.get('spec', np.nan))
    obj = np.abs(sens - t_sens) + np.abs(spec - t_spec)
    obj = np.where(np.isnan(obj), np.inf, obj)

    if np.isinf(obj).all():
        return np.nan, np.nan
    best_idx = int(np.argmin(obj))
    return float(thr_candidates[best_idx]), float(obj[best_idx])


def _p3d_decision_end_indices(n, prior_size, current_size, gap_size, step_size, future_size, max_points=None):
    min_end = prior_size + gap_size + current_size
    max_end = n - future_size
    if max_end < min_end:
        return np.array([], dtype=int)

    idx = np.arange(min_end, max_end + 1, step_size, dtype=int)
    if max_points is not None and len(idx) > int(max_points):
        take = np.linspace(0, len(idx) - 1, int(max_points)).round().astype(int)
        idx = idx[take]
        idx = np.unique(idx)
    return idx


def _p3d_time_weighted_mean(values, weights):
    v = pd.to_numeric(values, errors='coerce').to_numpy(dtype=float)
    w = pd.to_numeric(weights, errors='coerce').to_numpy(dtype=float)
    valid = (~np.isnan(v)) & (~np.isnan(w)) & (w > 0)
    if valid.sum() == 0:
        return np.nan
    return float(np.average(v[valid], weights=w[valid]))


def _p3d_two_prop_pvalue(success_a, n_a, success_b, n_b):
    """Two-sided two-proportion z-test p-value using normal approximation."""
    try:
        sa = float(success_a)
        sb = float(success_b)
        na = float(n_a)
        nb = float(n_b)
    except Exception:
        return np.nan

    if na <= 0 or nb <= 0:
        return np.nan

    pa = sa / na
    pb = sb / nb
    p_pool = (sa + sb) / (na + nb)
    var = p_pool * (1.0 - p_pool) * (1.0 / na + 1.0 / nb)
    if var <= 0:
        return 1.0 if abs(pa - pb) < 1e-12 else 0.0

    z = (pb - pa) / math.sqrt(var)
    # two-sided p-value from normal distribution, no scipy dependency
    pval = math.erfc(abs(z) / math.sqrt(2.0))
    return float(pval)


def _p3d_run_diary(
    df_pair,
    ai_col,
    cancer_col,
    date_col,
    objective,
    prior_size,
    current_size,
    future_size,
    gap_size,
    step_size,
    trigger_tolerance=0.0,
    signal_mode='drift',
    signal_p_threshold=0.05,
    min_recalibration_interval=0,
    max_threshold_candidates=101,
    max_points=None,
):
    d = df_pair.reset_index(drop=True)
    n = len(d)
    ends = _p3d_decision_end_indices(
        n=n,
        prior_size=prior_size,
        current_size=current_size,
        gap_size=gap_size,
        step_size=step_size,
        future_size=future_size,
        max_points=max_points,
    )
    if len(ends) == 0:
        return pd.DataFrame(), {}, {'reason': 'insufficient_exams'}

    signal_metrics = {'flagging_rate', 'relative_flagging_rate', 'p_ai_given_rad', 'p_rad_given_ai'}
    objective = str(objective).strip()
    if objective not in signal_metrics:
        return pd.DataFrame(), {}, {'reason': f'unsupported_signal_metric:{objective}'}

    # Targets from first prior window using radiologist performance (evaluation targets).
    t0 = int(ends[0])
    t0_current_start = t0 - current_size
    t0_prior_end = t0_current_start - gap_size
    t0_prior_start = t0_prior_end - prior_size

    prior0 = d.iloc[t0_prior_start:t0_prior_end]
    target_stats = _p3d_rate_stats_from_flags(prior0['rad_flagged'].to_numpy(), prior0[cancer_col].to_numpy())
    targets = {
        'sens': float(target_stats['sens']),
        'spec': float(target_stats['spec']),
        'ppv': float(target_stats['ppv']),
    }

    # One-time threshold initialization (before signal-driven updates begin).
    init_thr, _ = _p3d_choose_initial_threshold_eval_targets(
        prior0,
        ai_col=ai_col,
        cancer_col=cancer_col,
        targets=targets,
        max_candidates=max_threshold_candidates,
    )
    if pd.isna(init_thr):
        init_thr = float(pd.to_numeric(prior0[ai_col], errors='coerce').median())

    signal_targets = _p3d_signal_stats_from_threshold(prior0, ai_col, init_thr)

    rows = []
    current_thr = init_thr
    last_threshold_change_end_idx = None

    for i, end_idx in enumerate(ends):
        end_idx = int(end_idx)
        current_start = end_idx - current_size
        prior_end = current_start - gap_size
        prior_start = prior_end - prior_size
        future_start = end_idx
        future_end = end_idx + future_size

        prior_w = d.iloc[prior_start:prior_end]
        curr_w = d.iloc[current_start:end_idx]
        future_w = d.iloc[future_start:future_end]

        # Performance before recalibration at this decision time (evaluation metrics).
        prior_perf = _p3d_rate_stats_from_threshold(prior_w[ai_col].to_numpy(), prior_w[cancer_col].to_numpy(), current_thr)
        curr_perf = _p3d_rate_stats_from_threshold(curr_w[ai_col].to_numpy(), curr_w[cancer_col].to_numpy(), current_thr)
        prior_sig = _p3d_signal_stats_from_threshold(prior_w, ai_col, current_thr)
        curr_sig = _p3d_signal_stats_from_threshold(curr_w, ai_col, current_thr)

        ds = float(curr_perf['sens'] - targets['sens']) if pd.notna(curr_perf['sens']) and pd.notna(targets['sens']) else np.nan
        dp = float(curr_perf['spec'] - targets['spec']) if pd.notna(curr_perf['spec']) and pd.notna(targets['spec']) else np.nan
        dppv = float(curr_perf['ppv'] - targets['ppv']) if pd.notna(curr_perf['ppv']) and pd.notna(targets['ppv']) else np.nan

        t_sig = pd.to_numeric(pd.Series([signal_targets.get(objective, np.nan)]), errors='coerce').iloc[0]
        c_sig = pd.to_numeric(pd.Series([curr_sig.get(objective, np.nan)]), errors='coerce').iloc[0]
        drift = np.nan if (pd.isna(t_sig) or pd.isna(c_sig)) else abs(float(c_sig) - float(t_sig))

        # Optional statistical signal: prior vs current window two-proportion tests.
        p_signal = _p3d_signal_pvalue(prior_sig, curr_sig, objective)

        exams_since_last_change = np.nan if last_threshold_change_end_idx is None else float(end_idx - last_threshold_change_end_idx)
        cadence_ok = bool(
            last_threshold_change_end_idx is None or
            exams_since_last_change >= float(min_recalibration_interval)
        )

        mode = str(signal_mode).lower()
        alpha = float(signal_p_threshold)
        if mode == 'pvalue':
            signal_detected = bool(pd.notna(p_signal) and p_signal < alpha)
            # Practical constraint: even if significant, require minimum drift magnitude and cadence availability.
            recalibrate = bool(signal_detected and pd.notna(drift) and drift > float(trigger_tolerance) and cadence_ok)
        else:
            # Drift mode (oracle): signal is direct deviation from target.
            signal_detected = bool(pd.notna(drift) and drift > float(trigger_tolerance))
            recalibrate = bool(signal_detected and cadence_ok)

        new_thr = current_thr
        best_obj = np.nan
        if recalibrate:
            new_thr, best_obj = _p3d_choose_threshold(
                curr_w,
                ai_col=ai_col,
                cancer_col=cancer_col,
                targets=targets,
                objective=objective,
                signal_targets=signal_targets,
                max_candidates=max_threshold_candidates,
            )
            if pd.isna(new_thr):
                new_thr = current_thr

        threshold_changed = int(pd.notna(current_thr) and pd.notna(new_thr) and (abs(new_thr - current_thr) > 1e-12))
        if threshold_changed:
            last_threshold_change_end_idx = end_idx

        # Immediate future evaluation with updated threshold.
        fut_perf = _p3d_rate_stats_from_threshold(future_w[ai_col].to_numpy(), future_w[cancer_col].to_numpy(), new_thr)

        decision_date = d.iloc[end_idx][date_col] if date_col in d.columns and end_idx < len(d) else pd.NaT

        rows.append({
            'decision_idx': i,
            'decision_end_exam_idx': end_idx,
            'decision_date': decision_date,
            'objective_metric': objective,
            'signal_metric': objective,
            'ai_system': ai_col,
            'prior_size': prior_size,
            'current_size': current_size,
            'future_size': future_size,
            'gap_size': gap_size,
            'step_size': step_size,
            'prior_start_idx': prior_start,
            'prior_end_idx': prior_end,
            'current_start_idx': current_start,
            'current_end_idx': end_idx,
            'future_start_idx': future_start,
            'future_end_idx': future_end,
            'target_sens': targets['sens'],
            'target_spec': targets['spec'],
            'target_ppv': targets['ppv'],
            'threshold_before': current_thr,
            'threshold_after': new_thr,
            'threshold_changed': threshold_changed,
            'drift_value': drift,
            'signal_mode': str(signal_mode).lower(),
            'signal_p_threshold': float(signal_p_threshold),
            'min_recalibration_interval': float(min_recalibration_interval),
            'exams_since_last_change': exams_since_last_change,
            'cadence_ok': int(cadence_ok),
            'p_value_sens': np.nan,
            'p_value_spec': np.nan,
            'p_value_ppv': np.nan,
            'p_value_signal': p_signal,
            'signal_detected': int(signal_detected),
            'recalibrated': int(recalibrate),
            'best_obj_value': best_obj,
            'prior_sens_before': prior_perf['sens'],
            'prior_spec_before': prior_perf['spec'],
            'prior_ppv_before': prior_perf['ppv'],
            'current_sens_before': curr_perf['sens'],
            'current_spec_before': curr_perf['spec'],
            'current_ppv_before': curr_perf['ppv'],
            'delta_current_sens_to_target': ds,
            'delta_current_spec_to_target': dp,
            'delta_current_ppv_to_target': dppv,
            'target_signal_value': signal_targets.get(objective, np.nan),
            'prior_signal_value': prior_sig.get(objective, np.nan),
            'current_signal_value': curr_sig.get(objective, np.nan),
            'future_sens_after': fut_perf['sens'],
            'future_spec_after': fut_perf['spec'],
            'future_ppv_after': fut_perf['ppv'],
            'future_n': fut_perf['n_total'],
        })

        current_thr = new_thr

    diary = pd.DataFrame(rows)
    if len(diary) == 0:
        return diary, targets, {'reason': 'empty_diary'}

    # Build threshold-application intervals (retrospective diary evaluation).
    seg_start = diary['decision_end_exam_idx'].astype(int).to_numpy()
    seg_end = np.r_[seg_start[1:], [len(d)]]

    seg_sens, seg_spec, seg_ppv, seg_n = [], [], [], []
    for s, e, thr in zip(seg_start, seg_end, diary['threshold_after'].to_numpy(dtype=float)):
        seg = d.iloc[int(s):int(e)]
        perf = _p3d_rate_stats_from_threshold(seg[ai_col].to_numpy(), seg[cancer_col].to_numpy(), thr)
        seg_sens.append(perf['sens'])
        seg_spec.append(perf['spec'])
        seg_ppv.append(perf['ppv'])
        seg_n.append(perf['n_total'])

    diary['segment_start_idx'] = seg_start
    diary['segment_end_idx'] = seg_end
    diary['segment_n'] = np.asarray(seg_n, dtype=float)
    diary['segment_sens'] = np.asarray(seg_sens, dtype=float)
    diary['segment_spec'] = np.asarray(seg_spec, dtype=float)
    diary['segment_ppv'] = np.asarray(seg_ppv, dtype=float)
    diary['segment_abs_delta_sens'] = np.abs(diary['segment_sens'] - diary['target_sens'])
    diary['segment_abs_delta_spec'] = np.abs(diary['segment_spec'] - diary['target_spec'])
    diary['segment_abs_delta_ppv'] = np.abs(diary['segment_ppv'] - diary['target_ppv'])

    return diary, targets, {'reason': 'ok'}


def _p3d_summarize_diary(diary_df):
    if diary_df is None or len(diary_df) == 0:
        return {}

    out = {}
    out['objective_metric'] = diary_df['objective_metric'].iloc[0]
    out['signal_metric'] = diary_df['signal_metric'].iloc[0] if 'signal_metric' in diary_df.columns else diary_df['objective_metric'].iloc[0]
    out['ai_system'] = diary_df['ai_system'].iloc[0]
    out['prior_size'] = int(diary_df['prior_size'].iloc[0])
    out['current_size'] = int(diary_df['current_size'].iloc[0])
    out['future_size'] = int(diary_df['future_size'].iloc[0])
    out['gap_size'] = int(diary_df['gap_size'].iloc[0])
    out['step_size'] = int(diary_df['step_size'].iloc[0])

    out['target_sens'] = float(pd.to_numeric(diary_df['target_sens'], errors='coerce').iloc[0])
    out['target_spec'] = float(pd.to_numeric(diary_df['target_spec'], errors='coerce').iloc[0])
    out['target_ppv'] = float(pd.to_numeric(diary_df['target_ppv'], errors='coerce').iloc[0])

    out['n_decision_points'] = int(len(diary_df))
    out['n_recalibrations'] = int(pd.to_numeric(diary_df['recalibrated'], errors='coerce').fillna(0).sum())
    if 'signal_detected' in diary_df.columns:
        out['n_signal_detected'] = int(pd.to_numeric(diary_df['signal_detected'], errors='coerce').fillna(0).sum())
    out['n_threshold_changes'] = int(pd.to_numeric(diary_df['threshold_changed'], errors='coerce').fillna(0).sum())

    w = pd.to_numeric(diary_df['segment_n'], errors='coerce').fillna(0)

    out['time_weighted_mean_sens'] = _p3d_time_weighted_mean(diary_df['segment_sens'], w)
    out['time_weighted_mean_spec'] = _p3d_time_weighted_mean(diary_df['segment_spec'], w)
    out['time_weighted_mean_ppv'] = _p3d_time_weighted_mean(diary_df['segment_ppv'], w)

    out['time_weighted_mean_abs_delta_sens'] = _p3d_time_weighted_mean(diary_df['segment_abs_delta_sens'], w)
    out['time_weighted_mean_abs_delta_spec'] = _p3d_time_weighted_mean(diary_df['segment_abs_delta_spec'], w)
    out['time_weighted_mean_abs_delta_ppv'] = _p3d_time_weighted_mean(diary_df['segment_abs_delta_ppv'], w)

    # Explicitly keep simple means to diagnose suspicious invariances.
    out['simple_mean_sens'] = float(pd.to_numeric(diary_df['segment_sens'], errors='coerce').mean())
    out['simple_mean_spec'] = float(pd.to_numeric(diary_df['segment_spec'], errors='coerce').mean())
    out['simple_mean_abs_delta_sens'] = float(pd.to_numeric(diary_df['segment_abs_delta_sens'], errors='coerce').mean())
    out['simple_mean_abs_delta_spec'] = float(pd.to_numeric(diary_df['segment_abs_delta_spec'], errors='coerce').mean())

    out['weights_total_exams'] = float(w.sum())

    out['future_mean_sens_after'] = float(pd.to_numeric(diary_df['future_sens_after'], errors='coerce').mean())
    out['future_mean_spec_after'] = float(pd.to_numeric(diary_df['future_spec_after'], errors='coerce').mean())
    out['future_mean_ppv_after'] = float(pd.to_numeric(diary_df['future_ppv_after'], errors='coerce').mean())

    return out


def _p3d_run_batch(
    df_pair,
    cols,
    objectives,
    steps,
    prior_size,
    current_size,
    future_size,
    ai_systems,
    max_threshold_candidates,
    trigger_tol,
    signal_mode='drift',
    signal_p_threshold=0.05,
    min_recalibration_interval=0,
    max_points=None,
):
    diaries = {}
    summary_rows = []

    for objective in objectives:
        for step in steps:
            gap = int(step)
            for ai in ai_systems:
                diary, targets, meta = _p3d_run_diary(
                    df_pair=df_pair,
                    ai_col=ai,
                    cancer_col=cols['cancer'],
                    date_col=cols['date'],
                    objective=objective,
                    prior_size=int(prior_size),
                    current_size=int(current_size),
                    future_size=int(future_size),
                    gap_size=int(gap),
                    step_size=int(step),
                    trigger_tolerance=float(trigger_tol),
                    signal_mode=str(signal_mode),
                    signal_p_threshold=float(signal_p_threshold),
                    min_recalibration_interval=float(min_recalibration_interval),
                    max_threshold_candidates=int(max_threshold_candidates),
                    max_points=max_points,
                )
                key = (objective, int(step), ai, int(current_size))
                diaries[key] = diary

                if len(diary) == 0:
                    summary_rows.append({
                        'objective_metric': objective,
                        'signal_metric': objective,
                        'step_size': int(step),
                        'gap_size': int(gap),
                        'ai_system': ai,
                        'current_size': int(current_size),
                        'status': meta.get('reason', 'empty'),
                    })
                    continue

                s = _p3d_summarize_diary(diary)
                s['status'] = meta.get('reason', 'ok')
                summary_rows.append(s)

    summary_df = pd.DataFrame(summary_rows)

    # Overall aggregation across AI systems (time-weighted again by available exam weights).
    overall_rows = []
    if len(summary_df) > 0:
        group_metric_col = 'signal_metric' if 'signal_metric' in summary_df.columns else 'objective_metric'
        group_cols = [group_metric_col, 'step_size', 'gap_size', 'current_size']
        for keys, d in summary_df.groupby(group_cols):
            row = {
                'objective_metric': keys[0],
                'signal_metric': keys[0],
                'step_size': int(keys[1]),
                'gap_size': int(keys[2]),
                'current_size': int(keys[3]),
                'ai_system': 'ALL',
                'status': 'ok' if (d['status'] == 'ok').any() else 'no_data',
                'n_runs': int(len(d)),
            }

            w = pd.to_numeric(d.get('weights_total_exams', pd.Series(np.nan, index=d.index)), errors='coerce').fillna(0)

            for c in [
                'time_weighted_mean_sens', 'time_weighted_mean_spec', 'time_weighted_mean_ppv',
                'time_weighted_mean_abs_delta_sens', 'time_weighted_mean_abs_delta_spec', 'time_weighted_mean_abs_delta_ppv',
                'future_mean_sens_after', 'future_mean_spec_after', 'future_mean_ppv_after',
            ]:
                if c in d.columns:
                    row[c] = _p3d_time_weighted_mean(d[c], w)

            for c in ['target_sens', 'target_spec', 'target_ppv', 'n_decision_points', 'n_recalibrations', 'n_threshold_changes']:
                if c in d.columns:
                    row[c] = float(pd.to_numeric(d[c], errors='coerce').mean())

            overall_rows.append(row)

    overall_df = pd.DataFrame(overall_rows)
    return diaries, summary_df, overall_df


if 'df_part3' not in globals():
    print('df_part3 not found. Run the Part 3 data-load section first.')
else:
    cols = _p3d_get_columns(df_part3)
    ai_systems = cols['ai_cols'] if P3D_AI_SYSTEMS is None else [a for a in P3D_AI_SYSTEMS if a in cols['ai_cols']]

    pair_df = _p3d_prepare_pair_df(df_part3, cols, P3D_HOSPITAL, P3D_MANUFACTURER)
    if len(pair_df) == 0:
        print(f'No rows for hospital={P3D_HOSPITAL} manufacturer={P3D_MANUFACTURER}.')
    else:
        pair_df = _p3d_add_rad_flag(pair_df, rad_mode=P3D_RAD_MODE)
        print('Part 3D setup ready.')
        print('Pair rows:', len(pair_df))
        print('Hospital:', P3D_HOSPITAL, '| Manufacturer:', P3D_MANUFACTURER)
        print('AI systems:', ai_systems)
        print('Prior/current/future:', P3D_PRIOR_SIZE, P3D_CURRENT_SIZE, P3D_FUTURE_SIZE)
        print('Output dir:', P3D_OUTPUT_DIR)


In [ ]:
# Part 3D.1 - Choice 1: step/gap sweep with signal-metric-driven recalibration
if 'pair_df' not in globals() or len(pair_df) == 0:
    print('Run Part 3D.0 first.')
else:
    signal_metrics = [P3D_PRIMARY_OBJECTIVE]
    steps = [int(x) for x in P3D_STEP_GAP_LIST]

    p3d_choice1_diaries, p3d_choice1_summary, p3d_choice1_overall = _p3d_run_batch(
        df_pair=pair_df,
        cols=cols,
        objectives=signal_metrics,
        steps=steps,
        prior_size=P3D_PRIOR_SIZE,
        current_size=P3D_CURRENT_SIZE,
        future_size=P3D_FUTURE_SIZE,
        ai_systems=ai_systems,
        max_threshold_candidates=P3D_MAX_THRESHOLD_CANDIDATES,
        trigger_tol=P3D_TRIGGER_TOLERANCE,
        signal_mode=P3D_SIGNAL_MODE,
        signal_p_threshold=P3D_SIGNAL_P_THRESHOLD,
        min_recalibration_interval=P3D_MIN_RECALIBRATION_INTERVAL,
        max_points=P3D_MAX_DECISION_POINTS,
    )

    # Flatten diary map to one table for export/debug.
    diary_rows = []
    for (signal_metric, step, ai, curr_size), d in p3d_choice1_diaries.items():
        if d is None or len(d) == 0:
            continue
        x = d.copy()
        x['objective_metric'] = signal_metric
        x['signal_metric'] = signal_metric
        x['step_size'] = int(step)
        x['gap_size'] = int(step)
        x['ai_system'] = ai
        x['current_size'] = int(curr_size)
        x['hospital'] = P3D_HOSPITAL
        x['manufacturer'] = P3D_MANUFACTURER
        diary_rows.append(x)
    p3d_choice1_diary_df = pd.concat(diary_rows, ignore_index=True) if len(diary_rows) > 0 else pd.DataFrame()

    # Exports
    p3d_choice1_summary.to_csv(P3D_OUTPUT_DIR / 'p3d_choice1_summary_by_ai.csv', index=False)
    p3d_choice1_overall.to_csv(P3D_OUTPUT_DIR / 'p3d_choice1_summary_overall.csv', index=False)
    p3d_choice1_diary_df.to_csv(P3D_OUTPUT_DIR / 'p3d_choice1_threshold_diary.csv', index=False)

    print('Choice 1 complete.')
    print('Summary rows (by AI):', len(p3d_choice1_summary))
    print('Summary rows (overall):', len(p3d_choice1_overall))
    print('Diary rows:', len(p3d_choice1_diary_df))

    if 'step_size' in p3d_choice1_overall.columns:
        produced_steps = sorted([int(x) for x in pd.to_numeric(p3d_choice1_overall['step_size'], errors='coerce').dropna().unique().tolist()])
        expected_steps = sorted([int(x) for x in P3D_STEP_GAP_LIST])
        missing_steps = [s for s in expected_steps if s not in produced_steps]
        extra_steps = [s for s in produced_steps if s not in expected_steps]
        print('Expected step sizes:', expected_steps)
        print('Produced step sizes:', produced_steps)
        if len(missing_steps) > 0:
            print('WARNING: missing configured steps ->', missing_steps)
        if len(extra_steps) > 0:
            print('WARNING: unexpected extra steps ->', extra_steps)

    display(p3d_choice1_overall.sort_values(['step_size']).reset_index(drop=True))


In [ ]:
# Part 3D.2 - Choice 1 plots + threshold diary example + explicit time-weighted mean check
if 'p3d_choice1_summary' not in globals() or len(p3d_choice1_summary) == 0:
    print('Run Part 3D.1 first.')
else:
    d_all = p3d_choice1_summary.copy()
    d_overall = p3d_choice1_overall.copy() if 'p3d_choice1_overall' in globals() else pd.DataFrame()

    # Keep successful rows only.
    if 'status' in d_all.columns:
        d_all = d_all[d_all['status'] == 'ok'].copy()
    if 'status' in d_overall.columns:
        d_overall = d_overall[d_overall['status'] == 'ok'].copy()

    # Guard against stale runs: keep only configured steps.
    expected_steps = sorted({int(x) for x in P3D_STEP_GAP_LIST})
    if 'step_size' in d_all.columns:
        d_all['step_size'] = pd.to_numeric(d_all['step_size'], errors='coerce')
        observed_all = sorted([int(x) for x in d_all['step_size'].dropna().unique().tolist()])
        if any(x not in expected_steps for x in observed_all):
            print('WARNING: stale step values detected in p3d_choice1_summary:', observed_all)
            print('Filtering to configured steps only:', expected_steps)
        d_all = d_all[d_all['step_size'].isin(expected_steps)].copy()

    if 'step_size' in d_overall.columns:
        d_overall['step_size'] = pd.to_numeric(d_overall['step_size'], errors='coerce')
        observed_overall = sorted([int(x) for x in d_overall['step_size'].dropna().unique().tolist()])
        if any(x not in expected_steps for x in observed_overall):
            print('WARNING: stale step values detected in p3d_choice1_overall:', observed_overall)
            print('Filtering to configured steps only:', expected_steps)
        d_overall = d_overall[d_overall['step_size'].isin(expected_steps)].copy()

    if len(d_all) == 0:
        print('No valid Choice 1 rows left after status/step filtering.')
    else:
        # 4-panel plot for point #1.
        fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharex=True)
        metrics = [
            ('time_weighted_mean_sens', 'Time-weighted mean sensitivity (%)'),
            ('time_weighted_mean_spec', 'Time-weighted mean specificity (%)'),
            ('time_weighted_mean_abs_delta_sens', 'Time-weighted mean |ΔSensitivity|'),
            ('time_weighted_mean_abs_delta_spec', 'Time-weighted mean |ΔSpecificity|'),
        ]

        for ax, (mcol, title) in zip(axes.ravel(), metrics):
            # Plot by AI
            for ai, g in d_all.groupby('ai_system'):
                if mcol not in g.columns:
                    continue
                g = g.sort_values('step_size')
                ax.plot(g['step_size'], g[mcol], marker='o', linewidth=1.8, label=str(ai).upper())

            # Overall line
            if len(d_overall) > 0 and mcol in d_overall.columns:
                g0 = d_overall.sort_values('step_size')
                ax.plot(g0['step_size'], g0[mcol], marker='s', linewidth=2.6, color='black', label='ALL')

            ax.set_title(title)
            ax.set_xlabel('Step size (exams) [gap = step]')
            ax.set_xticks(expected_steps)
            ax.grid(True, alpha=0.25)

        handles, labels = axes[0, 0].get_legend_handles_labels()
        if len(handles) > 0:
            legend = fig.legend(
                handles,
                labels,
                loc='upper center',
                bbox_to_anchor=(0.5, 0.93),
                ncol=min(5, len(labels)),
                frameon=True,
                fontsize=10,
                title='AI system',
                title_fontsize=10,
                columnspacing=1.3,
                handlelength=2.2,
                borderaxespad=0.2,
            )
            legend.get_frame().set_alpha(0.92)

        fig.suptitle(
            f'Choice 1: Gap=Step sweep | oracle recalibration (signal metric={P3D_PRIMARY_OBJECTIVE})\n'
            f'{P3D_HOSPITAL} | {P3D_MANUFACTURER} | prior=current=future=20,000',
            fontsize=13,
            fontweight='bold',
            y=0.985,
        )
        footnote = (
            'Time-weighted mean = weighted average across threshold-application segments, '
            'using segment exam count (segment_n) as weights.'
        )
        fig.text(0.5, 0.012, footnote, ha='center', va='bottom', fontsize=9)
        fig.tight_layout(rect=[0, 0.04, 1, 0.82])
        out_path = P3D_OUTPUT_DIR / 'p3d_choice1_step_gap_sweep_4plots.png'
        fig.savefig(out_path, dpi=300, bbox_inches='tight', facecolor='white')
        plt.show()

        # Explicit mean-check table (simple mean vs time-weighted mean).
        check_cols = [
            'signal_metric', 'objective_metric', 'ai_system', 'step_size', 'gap_size',
            'simple_mean_sens', 'time_weighted_mean_sens',
            'simple_mean_spec', 'time_weighted_mean_spec',
            'simple_mean_abs_delta_sens', 'time_weighted_mean_abs_delta_sens',
            'simple_mean_abs_delta_spec', 'time_weighted_mean_abs_delta_spec',
            'weights_total_exams', 'n_decision_points', 'n_recalibrations', 'n_threshold_changes'
        ]
        check_cols = [c for c in check_cols if c in d_all.columns]
        p3d_choice1_mean_check = d_all[check_cols].sort_values(['ai_system', 'step_size']).reset_index(drop=True)
        p3d_choice1_mean_check.to_csv(P3D_OUTPUT_DIR / 'p3d_choice1_mean_check_simple_vs_timeweighted.csv', index=False)

        print('Saved:', out_path)
        print('Saved mean-check CSV:', P3D_OUTPUT_DIR / 'p3d_choice1_mean_check_simple_vs_timeweighted.csv')
        display(p3d_choice1_mean_check)

    # Threshold diary example (one AI and one step).
    if 'p3d_choice1_diaries' not in globals() or len(p3d_choice1_diaries) == 0:
        print('No diaries found. Run Part 3D.1 first to generate threshold logbook.')
    else:
        if P3D_DIARY_EXAMPLE_AI is not None and P3D_DIARY_EXAMPLE_AI in ai_systems:
            example_ai = str(P3D_DIARY_EXAMPLE_AI)
        else:
            example_ai = str(ai_systems[0])

        example_step = int(P3D_DIARY_EXAMPLE_STEP)
        if example_step not in expected_steps:
            example_step = int(expected_steps[0])

        key = (P3D_PRIMARY_OBJECTIVE, example_step, example_ai, int(P3D_CURRENT_SIZE))

        if key in p3d_choice1_diaries and len(p3d_choice1_diaries[key]) > 0:
            ex = p3d_choice1_diaries[key].copy().sort_values('decision_end_exam_idx').reset_index(drop=True)

            fig2, ax = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

            # Threshold diary (step line)
            x_dec = ex['decision_end_exam_idx']
            y_thr = ex['threshold_after']
            ax[0].step(x_dec, y_thr, where='post', linewidth=2, color='C0')

            changed = pd.to_numeric(ex['threshold_changed'], errors='coerce').fillna(0) > 0
            signal_detected = pd.to_numeric(ex.get('signal_detected', 0), errors='coerce').fillna(0) > 0
            cadence_ok = pd.to_numeric(ex.get('cadence_ok', 1), errors='coerce').fillna(1) > 0

            sig_no_change = signal_detected & (~changed)
            sig_blocked = signal_detected & (~cadence_ok) & (~changed)

            # All test points
            ax[0].scatter(x_dec, y_thr, s=22, color='C0', alpha=0.9, label='test point')

            # Signal detected but threshold not updated
            if sig_no_change.any():
                ax[0].scatter(
                    ex.loc[sig_no_change, 'decision_end_exam_idx'],
                    ex.loc[sig_no_change, 'threshold_after'],
                    s=36,
                    marker='x',
                    color='#ff9800',
                    linewidths=1.5,
                    label='signal detected, threshold unchanged',
                    zorder=3,
                )

            # Signal detected but blocked by cadence rule (subset of above)
            if sig_blocked.any():
                ax[0].scatter(
                    ex.loc[sig_blocked, 'decision_end_exam_idx'],
                    ex.loc[sig_blocked, 'threshold_after'],
                    s=34,
                    marker='^',
                    facecolors='none',
                    edgecolors='#6d4c41',
                    linewidths=1.3,
                    label='signal detected, cadence blocked',
                    zorder=3,
                )

            # Threshold update points
            if changed.any():
                ax[0].scatter(
                    ex.loc[changed, 'decision_end_exam_idx'],
                    ex.loc[changed, 'threshold_after'],
                    s=45,
                    color='red',
                    label='threshold changed at test point',
                    zorder=4,
                )

            mode_txt = str(ex['signal_mode'].dropna().iloc[0]) if ('signal_mode' in ex.columns and ex['signal_mode'].notna().any()) else 'na'
            p_txt = float(pd.to_numeric(ex['signal_p_threshold'], errors='coerce').dropna().iloc[0]) if ('signal_p_threshold' in ex.columns and pd.to_numeric(ex['signal_p_threshold'], errors='coerce').notna().any()) else np.nan
            minint_txt = int(round(float(pd.to_numeric(ex['min_recalibration_interval'], errors='coerce').dropna().iloc[0]))) if ('min_recalibration_interval' in ex.columns and pd.to_numeric(ex['min_recalibration_interval'], errors='coerce').notna().any()) else 0

            ax[0].set_ylabel('Threshold')
            ax[0].set_title(f'Threshold diary | AI={example_ai} | signal metric={P3D_PRIMARY_OBJECTIVE} | step=gap={example_step}')
            ax[0].grid(True, alpha=0.25)

            # Retrospective segment performance under applied thresholds (recalibrated) and
            # comparator without recalibration (fixed initial threshold).
            base_thr = float(ex['threshold_before'].iloc[0]) if 'threshold_before' in ex.columns else float(ex['threshold_after'].iloc[0])
            no_recal_sens = []
            no_recal_spec = []
            for s_idx, e_idx in zip(
                pd.to_numeric(ex['segment_start_idx'], errors='coerce').fillna(0).astype(int),
                pd.to_numeric(ex['segment_end_idx'], errors='coerce').fillna(0).astype(int),
            ):
                seg_df = pair_df.iloc[int(s_idx):int(e_idx)]
                perf_no = _p3d_rate_stats_from_threshold(
                    seg_df[example_ai].to_numpy(),
                    seg_df[cols['cancer']].to_numpy(),
                    base_thr,
                )
                no_recal_sens.append(perf_no['sens'])
                no_recal_spec.append(perf_no['spec'])

            ex['segment_sens_no_recal'] = np.asarray(no_recal_sens, dtype=float)
            ex['segment_spec_no_recal'] = np.asarray(no_recal_spec, dtype=float)

            w_seg = pd.to_numeric(ex['segment_n'], errors='coerce').fillna(0)
            tw_sens_recal = _p3d_time_weighted_mean(ex['segment_sens'], w_seg)
            tw_spec_recal = _p3d_time_weighted_mean(ex['segment_spec'], w_seg)
            tw_sens_no = _p3d_time_weighted_mean(ex['segment_sens_no_recal'], w_seg)
            tw_spec_no = _p3d_time_weighted_mean(ex['segment_spec_no_recal'], w_seg)

            # Recalibrated curves
            ax[1].plot(
                ex['decision_end_exam_idx'], ex['segment_sens'],
                marker='o', linewidth=1.8, color='C0',
                label='Sensitivity (recalibrated)'
            )
            ax[1].plot(
                ex['decision_end_exam_idx'], ex['segment_spec'],
                marker='s', linewidth=1.8, color='C1',
                label='Specificity (recalibrated)'
            )

            # No-recalibration comparator on same segments
            ax[1].plot(
                ex['decision_end_exam_idx'], ex['segment_sens_no_recal'],
                linestyle='--', linewidth=1.4, color='C0', alpha=0.75,
                label='Sensitivity (no recalibration)'
            )
            ax[1].plot(
                ex['decision_end_exam_idx'], ex['segment_spec_no_recal'],
                linestyle='--', linewidth=1.4, color='C1', alpha=0.75,
                label='Specificity (no recalibration)'
            )

            # Targets (fixed references)
            ax[1].axhline(float(ex['target_sens'].iloc[0]), linestyle=':', linewidth=1.6, color='navy', label='Target sensitivity')
            ax[1].axhline(float(ex['target_spec'].iloc[0]), linestyle=':', linewidth=1.6, color='darkorange', label='Target specificity')

            # Actual time-weighted means
            if pd.notna(tw_sens_recal):
                ax[1].axhline(float(tw_sens_recal), linestyle='-.', linewidth=1.4, color='C0', label='TW mean sens (recal)')
            if pd.notna(tw_spec_recal):
                ax[1].axhline(float(tw_spec_recal), linestyle='-.', linewidth=1.4, color='C1', label='TW mean spec (recal)')
            if pd.notna(tw_sens_no):
                ax[1].axhline(float(tw_sens_no), linestyle='-.', linewidth=1.1, color='C0', alpha=0.55, label='TW mean sens (no recal)')
            if pd.notna(tw_spec_no):
                ax[1].axhline(float(tw_spec_no), linestyle='-.', linewidth=1.1, color='C1', alpha=0.55, label='TW mean spec (no recal)')

            # Mark decision points where threshold changed (knot correspondence with top red points)
            if changed.any():
                ax[1].scatter(
                    ex.loc[changed, 'decision_end_exam_idx'],
                    ex.loc[changed, 'segment_sens'],
                    s=50, facecolors='none', edgecolors='C0', linewidths=1.5,
                    label='Sens at threshold-change tests'
                )
                ax[1].scatter(
                    ex.loc[changed, 'decision_end_exam_idx'],
                    ex.loc[changed, 'segment_spec'],
                    s=50, marker='s', facecolors='none', edgecolors='C1', linewidths=1.5,
                    label='Spec at threshold-change tests'
                )

            ax[1].set_ylabel('Rate (%)')
            ax[1].set_xlabel('Decision time (exam index)')
            ax[1].grid(True, alpha=0.25)

            # Merge legends from both panels into one figure-level legend.
            h0, l0 = ax[0].get_legend_handles_labels()
            h1, l1 = ax[1].get_legend_handles_labels()
            merged = {}
            for h, l in list(zip(h0, l0)) + list(zip(h1, l1)):
                if l not in merged:
                    merged[l] = h

            fig2.legend(
                list(merged.values()),
                list(merged.keys()),
                loc='lower center',
                bbox_to_anchor=(0.5, 0.10),
                ncol=4,
                fontsize=9.2,
                frameon=True,
            )

            # Larger, reader-friendly footnote below the figure.
            fig2.text(
                0.5,
                0.02,
                (
                    f'Signal rule: mode={mode_txt}' + (f', p<{p_txt:.2g}' if pd.notna(p_txt) else '') + f'; min recal interval={minint_txt} exams.\n'
                    'Top: each knot is a scheduled test point. Red = threshold updated; orange x = signal detected but threshold unchanged; '
                    'brown triangle = signal detected but cadence blocked.\n'
                    'Bottom: realized segment rates under applied thresholds (solid = recalibrated, dashed = no-recalibration baseline). '
                    'Dotted lines are target rates; dash-dot lines are time-weighted means.'
                ),
                ha='center',
                va='bottom',
                fontsize=12.0,
                linespacing=1.25,
            )
            fig2.tight_layout(rect=[0, 0.26, 0.98, 0.93])
            ex_path = P3D_OUTPUT_DIR / f'p3d_threshold_diary_example_{example_ai}_step{example_step}.png'
            fig2.savefig(ex_path, dpi=300, bbox_inches='tight', facecolor='white')
            plt.show()

            ex.to_csv(P3D_OUTPUT_DIR / f'p3d_threshold_diary_example_{example_ai}_step{example_step}.csv', index=False)
            print('Saved diary example plot:', ex_path)
            print('Saved diary example CSV:', P3D_OUTPUT_DIR / f'p3d_threshold_diary_example_{example_ai}_step{example_step}.csv')
            display(ex.head(30))
        else:
            print('No diary example found for key:', key)

        # Multi-step logbook figures: one figure per AI (all configured steps).
        ai_list = [str(a) for a in ai_systems]
        all_log_rows = []
        generated = []

        for ai_name in ai_list:
            available_steps = []
            for st in expected_steps:
                k = (P3D_PRIMARY_OBJECTIVE, int(st), ai_name, int(P3D_CURRENT_SIZE))
                if k in p3d_choice1_diaries and len(p3d_choice1_diaries[k]) > 0:
                    available_steps.append(int(st))

            if len(available_steps) == 0:
                print('No multi-step diary data available for AI:', ai_name)
                continue

            nrows = len(available_steps)
            fig3, axes3 = plt.subplots(nrows, 1, figsize=(15, 2.6 * nrows), sharex=False)
            if nrows == 1:
                axes3 = [axes3]

            log_rows = []
            raw_rows = []
            rule_mode = 'na'
            rule_p = np.nan
            rule_minint = 0

            for ax3, st in zip(axes3, available_steps):
                k = (P3D_PRIMARY_OBJECTIVE, int(st), ai_name, int(P3D_CURRENT_SIZE))
                dd = p3d_choice1_diaries[k].copy().sort_values('decision_end_exam_idx').reset_index(drop=True)

                x_dec = dd['decision_end_exam_idx']
                y_thr = dd['threshold_after']
                ax3.step(x_dec, y_thr, where='post', linewidth=1.8, color='tab:blue')
                ax3.scatter(x_dec, y_thr, s=12, color='tab:blue', alpha=0.55, label='test point')

                m_changed = pd.to_numeric(dd['threshold_changed'], errors='coerce').fillna(0) > 0
                m_signal = pd.to_numeric(dd.get('signal_detected', 0), errors='coerce').fillna(0) > 0
                m_cadence = pd.to_numeric(dd.get('cadence_ok', 1), errors='coerce').fillna(1) > 0
                m_sig_no_change = m_signal & (~m_changed)
                m_sig_blocked = m_signal & (~m_cadence) & (~m_changed)

                if m_sig_no_change.any():
                    ax3.scatter(
                        dd.loc[m_sig_no_change, 'decision_end_exam_idx'],
                        dd.loc[m_sig_no_change, 'threshold_after'],
                        s=28,
                        marker='x',
                        color='#ff9800',
                        linewidths=1.2,
                        label='signal detected, threshold unchanged',
                        zorder=3,
                    )

                if m_sig_blocked.any():
                    ax3.scatter(
                        dd.loc[m_sig_blocked, 'decision_end_exam_idx'],
                        dd.loc[m_sig_blocked, 'threshold_after'],
                        s=26,
                        marker='^',
                        facecolors='none',
                        edgecolors='#6d4c41',
                        linewidths=1.2,
                        label='signal detected, cadence blocked',
                        zorder=3,
                    )

                if m_changed.any():
                    ax3.scatter(
                        dd.loc[m_changed, 'decision_end_exam_idx'],
                        dd.loc[m_changed, 'threshold_after'],
                        s=36,
                        color='red',
                        alpha=0.85,
                        label='threshold changed at test point',
                        zorder=4,
                    )

                if rule_mode == 'na':
                    if 'signal_mode' in dd.columns and dd['signal_mode'].notna().any():
                        rule_mode = str(dd['signal_mode'].dropna().iloc[0])
                    if 'signal_p_threshold' in dd.columns:
                        pseries = pd.to_numeric(dd['signal_p_threshold'], errors='coerce').dropna()
                        if len(pseries) > 0:
                            rule_p = float(pseries.iloc[0])
                    if 'min_recalibration_interval' in dd.columns:
                        mseries = pd.to_numeric(dd['min_recalibration_interval'], errors='coerce').dropna()
                        if len(mseries) > 0:
                            rule_minint = int(round(float(mseries.iloc[0])))

                n_decisions = int(len(dd))
                n_recal = int(pd.to_numeric(dd['recalibrated'], errors='coerce').fillna(0).sum())
                n_changes = int(pd.to_numeric(dd['threshold_changed'], errors='coerce').fillna(0).sum())
                n_signal = int(pd.to_numeric(dd.get('signal_detected', 0), errors='coerce').fillna(0).sum())

                seg_w = pd.to_numeric(dd['segment_n'], errors='coerce').fillna(0)
                tw_sens = _p3d_time_weighted_mean(dd['segment_sens'], seg_w)
                tw_spec = _p3d_time_weighted_mean(dd['segment_spec'], seg_w)
                tw_abs_ds = _p3d_time_weighted_mean(dd['segment_abs_delta_sens'], seg_w)
                tw_abs_dp = _p3d_time_weighted_mean(dd['segment_abs_delta_spec'], seg_w)

                # No-recalibration comparator for professor-requested summary:
                # same segment boundaries, fixed threshold = initial threshold_before.
                base_thr_dd = float(dd['threshold_before'].iloc[0]) if 'threshold_before' in dd.columns else float(dd['threshold_after'].iloc[0])
                no_sens, no_spec = [], []
                for s_idx, e_idx in zip(
                    pd.to_numeric(dd['segment_start_idx'], errors='coerce').fillna(0).astype(int),
                    pd.to_numeric(dd['segment_end_idx'], errors='coerce').fillna(0).astype(int),
                ):
                    seg_df = pair_df.iloc[int(s_idx):int(e_idx)]
                    perf_no = _p3d_rate_stats_from_threshold(
                        seg_df[ai_name].to_numpy(),
                        seg_df[cols['cancer']].to_numpy(),
                        base_thr_dd,
                    )
                    no_sens.append(perf_no['sens'])
                    no_spec.append(perf_no['spec'])

                dd['_segment_sens_no_recal'] = np.asarray(no_sens, dtype=float)
                dd['_segment_spec_no_recal'] = np.asarray(no_spec, dtype=float)
                tw_sens_no = _p3d_time_weighted_mean(dd['_segment_sens_no_recal'], seg_w)
                tw_spec_no = _p3d_time_weighted_mean(dd['_segment_spec_no_recal'], seg_w)

                target_sens_dd = float(pd.to_numeric(dd['target_sens'], errors='coerce').dropna().iloc[0]) if 'target_sens' in dd.columns and pd.to_numeric(dd['target_sens'], errors='coerce').notna().any() else np.nan
                target_spec_dd = float(pd.to_numeric(dd['target_spec'], errors='coerce').dropna().iloc[0]) if 'target_spec' in dd.columns and pd.to_numeric(dd['target_spec'], errors='coerce').notna().any() else np.nan

                ax3.set_title(
                    f'Step=Gap={st:,} | decisions={n_decisions} | signals={n_signal} | recalibrations={n_recal} | threshold changes={n_changes}',
                    fontsize=10,
                    fontweight='bold',
                )
                ax3.set_ylabel('Threshold')
                ax3.grid(True, alpha=0.22)

                log_rows.append({
                    'ai_system': ai_name,
                    'objective_metric': P3D_PRIMARY_OBJECTIVE,
                    'signal_metric': P3D_PRIMARY_OBJECTIVE,
                    'step_size': int(st),
                    'gap_size': int(st),
                    'n_decisions': n_decisions,
                    'n_recalibrations': n_recal,
                    'n_threshold_changes': n_changes,
                    'target_sensitivity': target_sens_dd,
                    'target_specificity': target_spec_dd,
                    'tw_sensitivity_recal': tw_sens,
                    'tw_specificity_recal': tw_spec,
                    'tw_sensitivity_no_recal': tw_sens_no,
                    'tw_specificity_no_recal': tw_spec_no,
                    'tw_sensitivity_delta_recal_minus_no': np.nan if (pd.isna(tw_sens) or pd.isna(tw_sens_no)) else float(tw_sens - tw_sens_no),
                    'tw_specificity_delta_recal_minus_no': np.nan if (pd.isna(tw_spec) or pd.isna(tw_spec_no)) else float(tw_spec - tw_spec_no),
                    'tw_sensitivity_abs_error_to_target_recal': np.nan if (pd.isna(tw_sens) or pd.isna(target_sens_dd)) else abs(float(tw_sens - target_sens_dd)),
                    'tw_specificity_abs_error_to_target_recal': np.nan if (pd.isna(tw_spec) or pd.isna(target_spec_dd)) else abs(float(tw_spec - target_spec_dd)),
                    'tw_sensitivity_abs_error_to_target_no_recal': np.nan if (pd.isna(tw_sens_no) or pd.isna(target_sens_dd)) else abs(float(tw_sens_no - target_sens_dd)),
                    'tw_specificity_abs_error_to_target_no_recal': np.nan if (pd.isna(tw_spec_no) or pd.isna(target_spec_dd)) else abs(float(tw_spec_no - target_spec_dd)),
                    'time_weighted_mean_abs_delta_sens': tw_abs_ds,
                    'time_weighted_mean_abs_delta_spec': tw_abs_dp,
                })

                dd_export = dd.copy()
                dd_export['step_size'] = int(st)
                dd_export['gap_size'] = int(st)
                dd_export['ai_system'] = ai_name
                raw_rows.append(dd_export)

            axes3[-1].set_xlabel('Decision time (exam index)')
            fig3.suptitle(
                f'Choice 1 logbook by step | AI={ai_name} | signal metric={P3D_PRIMARY_OBJECTIVE}',
                fontsize=12,
                fontweight='bold',
                y=0.985,
            )

            # One merged legend for all subplots
            merged = {}
            for ax_tmp in axes3:
                h_tmp, l_tmp = ax_tmp.get_legend_handles_labels()
                for h, l in zip(h_tmp, l_tmp):
                    if l not in merged:
                        merged[l] = h
            if len(merged) > 0:
                fig3.legend(
                    list(merged.values()),
                    list(merged.keys()),
                    loc='lower center',
                    bbox_to_anchor=(0.5, 0.08),
                    ncol=4,
                    fontsize=9.6,
                    frameon=True,
                )

            # Larger explanatory footnote below figure
            rule_txt = f'Signal rule: mode={rule_mode}'
            if pd.notna(rule_p):
                rule_txt += f', p<{rule_p:.2g}'
            rule_txt += f'; min recal interval={rule_minint:,} exams.'

            fig3.text(
                0.5,
                0.015,
                rule_txt + '\n' +
                'Each knot is a scheduled test point. Red = threshold updated; orange x = signal detected but threshold unchanged; '
                'brown triangle = signal detected but cadence blocked.',
                ha='center',
                va='bottom',
                fontsize=12.5,
                linespacing=1.25,
            )

            fig3.tight_layout(rect=[0, 0.20, 1, 0.95])

            multi_path = P3D_OUTPUT_DIR / f'p3d_choice1_threshold_diary_all_steps_{ai_name}.png'
            fig3.savefig(multi_path, dpi=300, bbox_inches='tight', facecolor='white')
            plt.show()

            log_df = pd.DataFrame(log_rows).sort_values('step_size').reset_index(drop=True)
            log_csv = P3D_OUTPUT_DIR / f'p3d_choice1_logbook_summary_by_step_{ai_name}.csv'
            log_df.to_csv(log_csv, index=False)

            # Professor-focused compact table: TW sens/spec with and without recalibration + targets.
            prof_cols = [
                'ai_system', 'step_size',
                'target_sensitivity', 'target_specificity',
                'tw_sensitivity_recal', 'tw_sensitivity_no_recal', 'tw_sensitivity_delta_recal_minus_no',
                'tw_specificity_recal', 'tw_specificity_no_recal', 'tw_specificity_delta_recal_minus_no',
                'tw_sensitivity_abs_error_to_target_recal', 'tw_sensitivity_abs_error_to_target_no_recal',
                'tw_specificity_abs_error_to_target_recal', 'tw_specificity_abs_error_to_target_no_recal',
                'n_recalibrations', 'n_threshold_changes',
            ]
            prof_cols = [c for c in prof_cols if c in log_df.columns]
            prof_df = log_df[prof_cols].copy()
            prof_csv = P3D_OUTPUT_DIR / f'p3d_choice1_professor_summary_{ai_name}.csv'
            prof_df.to_csv(prof_csv, index=False)

            raw_df = pd.concat(raw_rows, ignore_index=True) if len(raw_rows) > 0 else pd.DataFrame()
            raw_csv = P3D_OUTPUT_DIR / f'p3d_choice1_logbook_all_rows_{ai_name}.csv'
            raw_df.to_csv(raw_csv, index=False)

            all_log_rows.append(log_df)
            generated.append({
                'ai_system': ai_name,
                'plot_file': multi_path.name,
                'summary_csv': log_csv.name,
                'raw_csv': raw_csv.name,
                'n_steps': len(available_steps),
            })

            print('Saved multi-step diary plot:', multi_path)
            print('Saved logbook summary CSV:', log_csv)
            print('Saved professor summary CSV:', prof_csv)
            print('Saved raw logbook CSV:', raw_csv)
            display(prof_df)

        if len(all_log_rows) > 0:
            combined_log = pd.concat(all_log_rows, ignore_index=True)
            combined_csv = P3D_OUTPUT_DIR / 'p3d_choice1_logbook_summary_by_step_all_ai.csv'
            combined_log.to_csv(combined_csv, index=False)

            generated_df = pd.DataFrame(generated)
            generated_csv = P3D_OUTPUT_DIR / 'p3d_choice1_logbook_generated_files.csv'
            generated_df.to_csv(generated_csv, index=False)

            print('Saved combined logbook summary CSV:', combined_csv)
            print('Saved generated-files index CSV:', generated_csv)
            display(generated_df)


In [ ]:
# Part 3D.2b - Sanity checks for suspicious invariance + mean-calculation audit
if 'p3d_choice1_summary' not in globals() or len(p3d_choice1_summary) == 0:
    print('Run Part 3D.1 first.')
else:
    d = p3d_choice1_summary.copy()
    if 'status' in d.columns:
        d = d[d['status'] == 'ok'].copy()

    if len(d) == 0:
        print('No valid rows to audit.')
    else:
        # Long format for invariance checks over step size.
        metric_cols = [
            'time_weighted_mean_sens',
            'time_weighted_mean_spec',
            'time_weighted_mean_abs_delta_sens',
            'time_weighted_mean_abs_delta_spec',
            'simple_mean_sens',
            'simple_mean_spec',
            'simple_mean_abs_delta_sens',
            'simple_mean_abs_delta_spec',
            'n_recalibrations',
            'n_threshold_changes',
            'n_decision_points',
        ]
        metric_cols = [c for c in metric_cols if c in d.columns]

        metric_key_col = 'signal_metric' if 'signal_metric' in d.columns else 'objective_metric'

        long_rows = []
        for _, r in d.iterrows():
            for m in metric_cols:
                long_rows.append({
                    'signal_metric': r.get(metric_key_col),
                    'ai_system': r.get('ai_system'),
                    'step_size': r.get('step_size'),
                    'gap_size': r.get('gap_size'),
                    'metric_name': m,
                    'value': pd.to_numeric(r.get(m), errors='coerce'),
                })
        audit_long = pd.DataFrame(long_rows)

        # Range / uniqueness per AI + metric over step sizes.
        audit_stats = (
            audit_long.groupby(['signal_metric', 'ai_system', 'metric_name'], as_index=False)
            .agg(
                n_points=('value', 'count'),
                n_unique=('value', lambda s: int(pd.Series(s).dropna().round(12).nunique())),
                v_min=('value', 'min'),
                v_max=('value', 'max'),
            )
        )
        audit_stats['v_range'] = audit_stats['v_max'] - audit_stats['v_min']
        audit_stats['suspicious_constant'] = (audit_stats['n_points'] >= 2) & (audit_stats['n_unique'] <= 1)

        # Mean-method disagreement (simple vs time-weighted) at same step.
        comp_rows = []
        for _, r in d.iterrows():
            for pair in [
                ('sens', 'simple_mean_sens', 'time_weighted_mean_sens'),
                ('spec', 'simple_mean_spec', 'time_weighted_mean_spec'),
                ('abs_delta_sens', 'simple_mean_abs_delta_sens', 'time_weighted_mean_abs_delta_sens'),
                ('abs_delta_spec', 'simple_mean_abs_delta_spec', 'time_weighted_mean_abs_delta_spec'),
            ]:
                label, c_simple, c_tw = pair
                if c_simple in d.columns and c_tw in d.columns:
                    sv = pd.to_numeric(r.get(c_simple), errors='coerce')
                    tv = pd.to_numeric(r.get(c_tw), errors='coerce')
                    comp_rows.append({
                        'signal_metric': r.get(metric_key_col),
                        'ai_system': r.get('ai_system'),
                        'step_size': r.get('step_size'),
                        'gap_size': r.get('gap_size'),
                        'metric_group': label,
                        'simple_mean': sv,
                        'time_weighted_mean': tv,
                        'abs_diff': np.nan if (pd.isna(sv) or pd.isna(tv)) else abs(float(sv) - float(tv)),
                    })
        mean_audit = pd.DataFrame(comp_rows)

        # Save audit artifacts.
        audit_long.to_csv(P3D_OUTPUT_DIR / 'p3d_choice1_invariance_long.csv', index=False)
        audit_stats.to_csv(P3D_OUTPUT_DIR / 'p3d_choice1_invariance_stats.csv', index=False)
        mean_audit.to_csv(P3D_OUTPUT_DIR / 'p3d_choice1_mean_method_audit.csv', index=False)

        flagged = audit_stats[audit_stats['suspicious_constant']].copy().sort_values(['metric_name', 'ai_system'])
        flagged.to_csv(P3D_OUTPUT_DIR / 'p3d_choice1_invariance_flagged.csv', index=False)

        print('Sanity audit complete.')
        print('Flagged constant metrics:', len(flagged))
        print('Saved:', P3D_OUTPUT_DIR / 'p3d_choice1_invariance_stats.csv')
        print('Saved:', P3D_OUTPUT_DIR / 'p3d_choice1_mean_method_audit.csv')

        if len(flagged) > 0:
            display(flagged)
        else:
            print('No suspicious constant metric found across tested step sizes.')

        # Quick visual diagnostic: threshold changes and recalibrations vs step.
        if all(c in d.columns for c in ['ai_system', 'step_size', 'n_threshold_changes', 'n_recalibrations']):
            fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharex=True)
            for ai, g in d.groupby('ai_system'):
                g = g.sort_values('step_size')
                axes[0].plot(g['step_size'], g['n_threshold_changes'], marker='o', label=str(ai).upper())
                axes[1].plot(g['step_size'], g['n_recalibrations'], marker='o', label=str(ai).upper())
            axes[0].set_title('Threshold changes vs step')
            axes[1].set_title('Recalibration events vs step')
            for ax in axes:
                ax.set_xlabel('Step size (exams) [gap=step]')
                ax.grid(True, alpha=0.25)
            axes[0].set_ylabel('count')
            axes[1].legend(loc='best')

            signal_mode_txt = str(globals().get('P3D_SIGNAL_MODE', 'drift'))
            p_thr = globals().get('P3D_SIGNAL_P_THRESHOLD', None)
            min_int = int(globals().get('P3D_MIN_RECALIBRATION_INTERVAL', 0))

            rule_line = f'Signal rule: mode={signal_mode_txt}'
            if str(signal_mode_txt).lower() == 'pvalue' and p_thr is not None:
                try:
                    rule_line += f', p<{float(p_thr):.3g}'
                except Exception:
                    rule_line += f', p<{p_thr}'
            rule_line += f'; min recalibration interval={min_int:,} exams.'

            note_line = (
                'Counts are measured over scheduled test points (x-axis step). '
                'A recalibration event may keep the same threshold, so threshold changes can be lower than recalibration events.'
            )

            fig.text(
                0.5,
                0.01,
                rule_line + '\n' + note_line,
                ha='center',
                va='bottom',
                fontsize=10.5,
                linespacing=1.25,
            )

            fig.tight_layout(rect=[0, 0.16, 1, 1])
            diag_path = P3D_OUTPUT_DIR / 'p3d_choice1_invariance_diagnostics.png'
            fig.savefig(diag_path, dpi=300, bbox_inches='tight', facecolor='white')
            plt.show()
            print('Saved diagnostics plot:', diag_path)


In [ ]:
# Part 3D.3 - Choice 2: signal-metric comparison (two scatter plots + target-distance scoring)
if 'pair_df' not in globals() or len(pair_df) == 0:
    print('Run Part 3D.0 first.')
else:
    signal_metrics = [str(x) for x in P3D_OBJECTIVES_FOR_COMPARISON]
    steps = [int(P3D_COMPARISON_STEP)]

    p3d_choice2_diaries, p3d_choice2_summary, p3d_choice2_overall = _p3d_run_batch(
        df_pair=pair_df,
        cols=cols,
        objectives=signal_metrics,
        steps=steps,
        prior_size=P3D_PRIOR_SIZE,
        current_size=P3D_CURRENT_SIZE,
        future_size=P3D_FUTURE_SIZE,
        ai_systems=ai_systems,
        max_threshold_candidates=P3D_MAX_THRESHOLD_CANDIDATES,
        trigger_tol=P3D_TRIGGER_TOLERANCE,
        signal_mode=P3D_SIGNAL_MODE,
        signal_p_threshold=P3D_SIGNAL_P_THRESHOLD,
        min_recalibration_interval=P3D_MIN_RECALIBRATION_INTERVAL,
        max_points=P3D_MAX_DECISION_POINTS,
    )

    # Export summary tables regardless of plotting status.
    p3d_choice2_summary.to_csv(P3D_OUTPUT_DIR / 'p3d_choice2_summary_by_ai.csv', index=False)
    p3d_choice2_overall.to_csv(P3D_OUTPUT_DIR / 'p3d_choice2_summary_overall.csv', index=False)

    if len(p3d_choice2_overall) == 0:
        print('No overall results for Choice 2.')
    else:
        d = p3d_choice2_overall.copy()
        d = d[d.get('status', 'ok') == 'ok'].copy()

        # Keep only configured signal metrics.
        metric_col = 'signal_metric' if 'signal_metric' in d.columns else 'objective_metric'
        d = d[d[metric_col].astype(str).isin(signal_metrics)].copy()

        num_cols = [
            'time_weighted_mean_sens', 'time_weighted_mean_spec', 'time_weighted_mean_ppv',
            'target_sens', 'target_spec', 'target_ppv', 'step_size', 'gap_size'
        ]
        for c in num_cols:
            if c in d.columns:
                d[c] = pd.to_numeric(d[c], errors='coerce')

        d = d.dropna(subset=['time_weighted_mean_sens', 'time_weighted_mean_spec', 'time_weighted_mean_ppv']).copy()
        if len(d) == 0:
            print('No valid Choice 2 rows after numeric cleaning.')
        else:
            d['one_minus_spec'] = 100.0 - d['time_weighted_mean_spec']

            # Single target operating point (same config, so averaging is safe).
            target_sens = float(pd.to_numeric(d.get('target_sens', pd.Series(np.nan, index=d.index)), errors='coerce').mean())
            target_spec = float(pd.to_numeric(d.get('target_spec', pd.Series(np.nan, index=d.index)), errors='coerce').mean())
            target_ppv = float(pd.to_numeric(d.get('target_ppv', pd.Series(np.nan, index=d.index)), errors='coerce').mean())
            target_one_minus_spec = 100.0 - target_spec if pd.notna(target_spec) else np.nan

            # Explicit distance-to-target scoring (to pick a best signal metric).
            d['distance_target_roc_like'] = np.sqrt(
                (d['time_weighted_mean_sens'] - target_sens) ** 2 +
                (d['one_minus_spec'] - target_one_minus_spec) ** 2
            )
            d['distance_target_ppv_like'] = np.sqrt(
                (d['time_weighted_mean_sens'] - target_sens) ** 2 +
                (d['time_weighted_mean_ppv'] - target_ppv) ** 2
            )
            d['distance_target_joint'] = 0.5 * (d['distance_target_roc_like'] + d['distance_target_ppv_like'])
            d['rank_joint'] = d['distance_target_joint'].rank(method='min').astype(int)

            best_roc_obj = str(d.loc[d['distance_target_roc_like'].idxmin(), metric_col])
            best_ppv_obj = str(d.loc[d['distance_target_ppv_like'].idxmin(), metric_col])
            best_joint_obj = str(d.loc[d['distance_target_joint'].idxmin(), metric_col])

            score_cols = [
                metric_col, 'step_size', 'gap_size',
                'time_weighted_mean_sens', 'time_weighted_mean_spec', 'time_weighted_mean_ppv',
                'one_minus_spec',
                'target_sens', 'target_spec', 'target_ppv',
                'distance_target_roc_like', 'distance_target_ppv_like', 'distance_target_joint', 'rank_joint'
            ]
            score_cols = [c for c in score_cols if c in d.columns]
            p3d_choice2_target_scores = d[score_cols].sort_values('distance_target_joint').reset_index(drop=True)
            p3d_choice2_target_scores.to_csv(P3D_OUTPUT_DIR / 'p3d_choice2_target_distance_scores.csv', index=False)

            palette = ['tab:blue', 'tab:orange', 'tab:green', 'tab:red', 'tab:purple', 'tab:brown']
            metric_levels = sorted(d[metric_col].astype(str).unique().tolist())
            color_map = {m: palette[i % len(palette)] for i, m in enumerate(metric_levels)}

            # Scatter 1: y=sensitivity, x=1-specificity
            fig, ax = plt.subplots(figsize=(8.8, 6.6))
            for _, r in d.iterrows():
                obj = str(r[metric_col])
                is_best = (obj == best_roc_obj)
                ax.scatter(
                    r['one_minus_spec'],
                    r['time_weighted_mean_sens'],
                    s=170 if is_best else 95,
                    color=color_map.get(obj, 'tab:gray'),
                    edgecolors='black' if is_best else 'none',
                    linewidth=1.2 if is_best else 0.8,
                    zorder=3,
                )
                label = f"{obj} (step={int(r['step_size'])})"
                ax.annotate(label, (r['one_minus_spec'], r['time_weighted_mean_sens']), xytext=(5, 5), textcoords='offset points', fontsize=9)

            if pd.notna(target_sens) and pd.notna(target_one_minus_spec):
                ax.scatter(target_one_minus_spec, target_sens, marker='*', s=260, color='black', label='Target', zorder=4)

            txt1 = (
                f"Target: sens={target_sens:.2f}, spec={target_spec:.2f}\n"
                f"Best by ROC-like distance: {best_roc_obj}"
            )
            ax.text(0.02, 0.02, txt1, transform=ax.transAxes, fontsize=9,
                    bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.85))

            ax.set_xlabel('1 - Specificity (%)')
            ax.set_ylabel('Sensitivity (%)')
            ax.set_title('Choice 2: Signal-metric comparison (Sensitivity vs 1-Specificity)')
            ax.grid(True, alpha=0.25)
            ax.legend(loc='best')
            roc_like_path = P3D_OUTPUT_DIR / 'p3d_choice2_scatter_sens_vs_1minus_spec.png'
            fig.tight_layout()
            fig.savefig(roc_like_path, dpi=300, bbox_inches='tight', facecolor='white')
            plt.show()

            # Scatter 2: y=sensitivity, x=PPV
            fig2, ax2 = plt.subplots(figsize=(8.8, 6.6))
            for _, r in d.iterrows():
                obj = str(r[metric_col])
                is_best = (obj == best_ppv_obj)
                ax2.scatter(
                    r['time_weighted_mean_ppv'],
                    r['time_weighted_mean_sens'],
                    s=170 if is_best else 95,
                    color=color_map.get(obj, 'tab:gray'),
                    edgecolors='black' if is_best else 'none',
                    linewidth=1.2 if is_best else 0.8,
                    zorder=3,
                )
                label = f"{obj} (step={int(r['step_size'])})"
                ax2.annotate(label, (r['time_weighted_mean_ppv'], r['time_weighted_mean_sens']), xytext=(5, 5), textcoords='offset points', fontsize=9)

            if pd.notna(target_sens) and pd.notna(target_ppv):
                ax2.scatter(target_ppv, target_sens, marker='*', s=260, color='black', label='Target', zorder=4)

            txt2 = (
                f"Target: sens={target_sens:.2f}, ppv={target_ppv:.2f}\n"
                f"Best by PPV-like distance: {best_ppv_obj}\n"
                f"Best overall (joint): {best_joint_obj}"
            )
            ax2.text(0.02, 0.02, txt2, transform=ax2.transAxes, fontsize=9,
                     bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.85))

            ax2.set_xlabel('PPV (%)')
            ax2.set_ylabel('Sensitivity (%)')
            ax2.set_title('Choice 2: Signal-metric comparison (Sensitivity vs PPV)')
            ax2.grid(True, alpha=0.25)
            ax2.legend(loc='best')
            pr_like_path = P3D_OUTPUT_DIR / 'p3d_choice2_scatter_sens_vs_ppv.png'
            fig2.tight_layout()
            fig2.savefig(pr_like_path, dpi=300, bbox_inches='tight', facecolor='white')
            plt.show()

            # Keep one diary table for transparency
            diary_rows = []
            for (signal_metric, step, ai, curr_size), x in p3d_choice2_diaries.items():
                if x is None or len(x) == 0:
                    continue
                y = x.copy()
                y['objective_metric'] = signal_metric
                y['signal_metric'] = signal_metric
                y['step_size'] = int(step)
                y['gap_size'] = int(P3D_COMPARISON_GAP)
                y['ai_system'] = ai
                y['current_size'] = int(curr_size)
                diary_rows.append(y)
            p3d_choice2_diary_df = pd.concat(diary_rows, ignore_index=True) if len(diary_rows) > 0 else pd.DataFrame()
            p3d_choice2_diary_df.to_csv(P3D_OUTPUT_DIR / 'p3d_choice2_threshold_diary.csv', index=False)

            print('Saved:', roc_like_path)
            print('Saved:', pr_like_path)
            print('Saved target-distance CSV:', P3D_OUTPUT_DIR / 'p3d_choice2_target_distance_scores.csv')
            display(p3d_choice2_target_scores)


In [ ]:
# Part 3D.4 - Supplementary: repeat Choice 2 for multiple window sizes
if not P3D_RUN_SUPPLEMENTARY:
    print('P3D_RUN_SUPPLEMENTARY=False -> skipped. Set True to run supplementary window-size analysis.')
elif 'pair_df' not in globals() or len(pair_df) == 0:
    print('Run Part 3D.0 first.')
else:
    supp_rows = []

    for w in [int(x) for x in P3D_SUPP_WINDOW_SIZES]:
        diaries, summary_by_ai, summary_overall = _p3d_run_batch(
            df_pair=pair_df,
            cols=cols,
            objectives=[str(x) for x in P3D_OBJECTIVES_FOR_COMPARISON],
            steps=[int(P3D_COMPARISON_STEP)],
            prior_size=w,
            current_size=w,
            future_size=w,
            ai_systems=ai_systems,
            max_threshold_candidates=P3D_MAX_THRESHOLD_CANDIDATES,
            trigger_tol=P3D_TRIGGER_TOLERANCE,
            signal_mode=P3D_SIGNAL_MODE,
            signal_p_threshold=P3D_SIGNAL_P_THRESHOLD,
            min_recalibration_interval=P3D_MIN_RECALIBRATION_INTERVAL,
            max_points=P3D_MAX_DECISION_POINTS,
        )
        if len(summary_overall) == 0:
            continue

        z = summary_overall.copy()
        z = z[z.get('status', 'ok') == 'ok'].copy()
        if len(z) == 0:
            continue

        z['window_size'] = int(w)
        z['one_minus_spec'] = 100.0 - pd.to_numeric(z['time_weighted_mean_spec'], errors='coerce')
        supp_rows.append(z)

    p3d_supp_df = pd.concat(supp_rows, ignore_index=True) if len(supp_rows) > 0 else pd.DataFrame()
    if len(p3d_supp_df) == 0:
        print('No supplementary results produced.')
    else:
        # Numeric cleanup.
        for c in [
            'time_weighted_mean_sens', 'time_weighted_mean_spec', 'time_weighted_mean_ppv',
            'target_sens', 'target_spec', 'target_ppv', 'window_size', 'one_minus_spec'
        ]:
            if c in p3d_supp_df.columns:
                p3d_supp_df[c] = pd.to_numeric(p3d_supp_df[c], errors='coerce')

        p3d_supp_df = p3d_supp_df.dropna(subset=['time_weighted_mean_sens', 'time_weighted_mean_spec', 'time_weighted_mean_ppv']).copy()

        # Distance scores per row.
        p3d_supp_df['target_one_minus_spec'] = 100.0 - p3d_supp_df['target_spec']
        p3d_supp_df['distance_target_roc_like'] = np.sqrt(
            (p3d_supp_df['time_weighted_mean_sens'] - p3d_supp_df['target_sens']) ** 2 +
            (p3d_supp_df['one_minus_spec'] - p3d_supp_df['target_one_minus_spec']) ** 2
        )
        p3d_supp_df['distance_target_ppv_like'] = np.sqrt(
            (p3d_supp_df['time_weighted_mean_sens'] - p3d_supp_df['target_sens']) ** 2 +
            (p3d_supp_df['time_weighted_mean_ppv'] - p3d_supp_df['target_ppv']) ** 2
        )
        p3d_supp_df['distance_target_joint'] = 0.5 * (
            p3d_supp_df['distance_target_roc_like'] + p3d_supp_df['distance_target_ppv_like']
        )

        # Best signal metric per window size.
        metric_col = 'signal_metric' if 'signal_metric' in p3d_supp_df.columns else 'objective_metric'
        best_rows = []
        for w, sub in p3d_supp_df.groupby('window_size'):
            if len(sub) == 0:
                continue
            r = sub.loc[sub['distance_target_joint'].idxmin()]
            best_rows.append({
                'window_size': int(w),
                'best_signal_metric_joint': str(r[metric_col]),
                'distance_target_joint': float(r['distance_target_joint']),
                'distance_target_roc_like': float(r['distance_target_roc_like']),
                'distance_target_ppv_like': float(r['distance_target_ppv_like']),
                'target_sens': float(pd.to_numeric(sub['target_sens'], errors='coerce').mean()),
                'target_spec': float(pd.to_numeric(sub['target_spec'], errors='coerce').mean()),
                'target_ppv': float(pd.to_numeric(sub['target_ppv'], errors='coerce').mean()),
            })

        p3d_supp_best_df = pd.DataFrame(best_rows).sort_values('window_size').reset_index(drop=True)

        # Save tables.
        p3d_supp_df.to_csv(P3D_OUTPUT_DIR / 'p3d_supp_window_size_comparison.csv', index=False)
        p3d_supp_best_df.to_csv(P3D_OUTPUT_DIR / 'p3d_supp_best_signal_metric_by_window.csv', index=False)

        # Plot 1: Sensitivity vs 1-Specificity by window size.
        ws = sorted([int(x) for x in p3d_supp_df['window_size'].dropna().unique().tolist()])
        fig, axes = plt.subplots(1, len(ws), figsize=(5.3 * len(ws), 5.2), squeeze=False)
        for ax, w in zip(axes[0], ws):
            d = p3d_supp_df[p3d_supp_df['window_size'] == w].copy()
            if len(d) == 0:
                continue

            t_sens = float(pd.to_numeric(d['target_sens'], errors='coerce').mean())
            t_spec = float(pd.to_numeric(d['target_spec'], errors='coerce').mean())
            t_x = 100.0 - t_spec
            best_obj = str(d.loc[d['distance_target_joint'].idxmin(), metric_col])

            for _, r in d.iterrows():
                obj = str(r[metric_col])
                is_best = (obj == best_obj)
                ax.scatter(
                    r['one_minus_spec'],
                    r['time_weighted_mean_sens'],
                    s=155 if is_best else 80,
                    edgecolors='black' if is_best else 'none',
                    linewidth=1.0,
                    alpha=0.9,
                )
                ax.annotate(obj, (r['one_minus_spec'], r['time_weighted_mean_sens']), xytext=(4, 4), textcoords='offset points', fontsize=8)

            if pd.notna(t_sens) and pd.notna(t_x):
                ax.scatter(t_x, t_sens, marker='*', s=200, color='black', zorder=4)

            ax.set_title(f'window={w:,} | best={best_obj}', fontsize=10, fontweight='bold')
            ax.set_xlabel('1 - Specificity (%)')
            ax.set_ylabel('Sensitivity (%)')
            ax.grid(True, alpha=0.25)

        fig.suptitle('Supplementary: Sensitivity vs 1-Specificity by window size', fontsize=12, fontweight='bold')
        fig.tight_layout(rect=[0, 0, 1, 0.95])
        supp_roc_path = P3D_OUTPUT_DIR / 'p3d_supp_scatter_sens_vs_1minus_spec_by_window.png'
        fig.savefig(supp_roc_path, dpi=300, bbox_inches='tight', facecolor='white')
        plt.show()

        # Plot 2: Sensitivity vs PPV by window size.
        fig2, axes2 = plt.subplots(1, len(ws), figsize=(5.3 * len(ws), 5.2), squeeze=False)
        for ax, w in zip(axes2[0], ws):
            d = p3d_supp_df[p3d_supp_df['window_size'] == w].copy()
            if len(d) == 0:
                continue

            t_sens = float(pd.to_numeric(d['target_sens'], errors='coerce').mean())
            t_ppv = float(pd.to_numeric(d['target_ppv'], errors='coerce').mean())
            best_obj = str(d.loc[d['distance_target_joint'].idxmin(), metric_col])

            for _, r in d.iterrows():
                obj = str(r[metric_col])
                is_best = (obj == best_obj)
                ax.scatter(
                    r['time_weighted_mean_ppv'],
                    r['time_weighted_mean_sens'],
                    s=155 if is_best else 80,
                    edgecolors='black' if is_best else 'none',
                    linewidth=1.0,
                    alpha=0.9,
                )
                ax.annotate(obj, (r['time_weighted_mean_ppv'], r['time_weighted_mean_sens']), xytext=(4, 4), textcoords='offset points', fontsize=8)

            if pd.notna(t_sens) and pd.notna(t_ppv):
                ax.scatter(t_ppv, t_sens, marker='*', s=200, color='black', zorder=4)

            ax.set_title(f'window={w:,} | best={best_obj}', fontsize=10, fontweight='bold')
            ax.set_xlabel('PPV (%)')
            ax.set_ylabel('Sensitivity (%)')
            ax.grid(True, alpha=0.25)

        fig2.suptitle('Supplementary: Sensitivity vs PPV by window size', fontsize=12, fontweight='bold')
        fig2.tight_layout(rect=[0, 0, 1, 0.95])
        supp_ppv_path = P3D_OUTPUT_DIR / 'p3d_supp_scatter_sens_vs_ppv_by_window.png'
        fig2.savefig(supp_ppv_path, dpi=300, bbox_inches='tight', facecolor='white')
        plt.show()

        print('Saved supplementary CSV:', P3D_OUTPUT_DIR / 'p3d_supp_window_size_comparison.csv')
        print('Saved best-signal-metric CSV:', P3D_OUTPUT_DIR / 'p3d_supp_best_signal_metric_by_window.csv')
        print('Saved supplementary plots:', supp_roc_path, '|', supp_ppv_path)
        display(p3d_supp_df.sort_values(['window_size', metric_col]).reset_index(drop=True))
        display(p3d_supp_best_df)


In [ ]:
# Part 3D.5 - Logbook example image (first 10 rows, compact)

import pandas as pd
import matplotlib.pyplot as plt
import textwrap
from pathlib import Path

out_dir = Path('outputs/2025_12_29_counterfactual_manufacturer_swap_setup/part3d_oracle_recalibration')
out_dir.mkdir(parents=True, exist_ok=True)

# 1) Prefer in-memory diary from Part 3D.1
logbook_df = None
if 'p3d_choice1_diary_df' in globals() and p3d_choice1_diary_df is not None and len(p3d_choice1_diary_df) > 0:
    logbook_df = p3d_choice1_diary_df.copy()

# 2) Fallback to saved CSV if needed
if logbook_df is None or len(logbook_df) == 0:
    csv_path = out_dir / 'p3d_choice1_threshold_diary.csv'
    if csv_path.exists():
        logbook_df = pd.read_csv(csv_path)

if logbook_df is None or len(logbook_df) == 0:
    print('No logbook found. Run Part 3D.1 first.')
else:
    # Filters for a cleaner table example.
    EXAMPLE_SIGNAL_METRIC = str(globals().get('P3D_PRIMARY_OBJECTIVE', 'flagging_rate'))  # set None to disable filter
    EXAMPLE_AI = 'lunit'            # set None to disable filter
    EXAMPLE_STEP = 500              # set None to disable filter

    d = logbook_df.copy()
    metric_col = 'signal_metric' if 'signal_metric' in d.columns else 'objective_metric'
    if EXAMPLE_SIGNAL_METRIC is not None and metric_col in d.columns:
        d = d[d[metric_col].astype(str) == str(EXAMPLE_SIGNAL_METRIC)]
    if EXAMPLE_AI is not None and 'ai_system' in d.columns:
        d = d[d['ai_system'].astype(str) == str(EXAMPLE_AI)]
    if EXAMPLE_STEP is not None and 'step_size' in d.columns:
        d = d[pd.to_numeric(d['step_size'], errors='coerce') == float(EXAMPLE_STEP)]

    if len(d) == 0:
        print('No rows after filter. Relax EXAMPLE_* filters in this cell.')
    else:
        d = d.sort_values('decision_end_exam_idx').head(10).reset_index(drop=True)

        # Keep only essential columns for readability.
        compact_cols = [
            'decision_idx', 'decision_date', 'decision_end_exam_idx',
            'threshold_before', 'threshold_after', 'threshold_changed',
            'target_sens', 'target_spec',
            'current_sens_before', 'current_spec_before',
            'future_sens_after', 'future_spec_after',
            'segment_abs_delta_sens', 'segment_abs_delta_spec'
        ]
        compact_cols = [c for c in compact_cols if c in d.columns]
        d_show = d[compact_cols].copy()

        rename_map = {
            'decision_idx': 'idx',
            'decision_date': 'date',
            'decision_end_exam_idx': 'end_exam',
            'threshold_before': 'thr_before',
            'threshold_after': 'thr_after',
            'threshold_changed': 'thr_changed',
            'target_sens': 'target_sens',
            'target_spec': 'target_spec',
            'current_sens_before': 'curr_sens',
            'current_spec_before': 'curr_spec',
            'future_sens_after': 'future_sens',
            'future_spec_after': 'future_spec',
            'segment_abs_delta_sens': 'abs_delta_sens',
            'segment_abs_delta_spec': 'abs_delta_spec',
        }
        d_show = d_show.rename(columns=rename_map)

        # Formatting
        for c in d_show.columns:
            if 'date' in c:
                d_show[c] = pd.to_datetime(d_show[c], errors='coerce').dt.strftime('%Y-%m-%d')
            elif pd.api.types.is_numeric_dtype(d_show[c]):
                if c in ['idx', 'end_exam']:
                    d_show[c] = d_show[c].map(lambda x: '' if pd.isna(x) else f'{int(round(float(x)))}')
                elif c == 'thr_changed':
                    d_show[c] = d_show[c].map(
                        lambda x: '' if pd.isna(x) else ('yes' if int(round(float(x))) == 1 else 'no')
                    )
                else:
                    d_show[c] = d_show[c].map(lambda x: '' if pd.isna(x) else f'{float(x):0.2f}')

        fig_h = 1.8 + 0.55 * len(d_show)
        fig_w = max(18, 1.35 * len(d_show.columns))
        fig, ax = plt.subplots(figsize=(fig_w, fig_h))
        ax.axis('off')

        table = ax.table(
            cellText=d_show.values,
            colLabels=[c.replace('_', ' ') for c in d_show.columns.tolist()],
            cellLoc='center',
            loc='center',
            bbox=[0, 0, 1, 1]
        )
        table.auto_set_font_size(False)
        table.set_fontsize(9.5)
        table.scale(1.0, 1.6)

        header_color = '#4472C4'
        alt_row_color = '#E7E6E6'
        for j in range(len(d_show.columns)):
            cell = table[(0, j)]
            cell.set_facecolor(header_color)
            cell.set_text_props(weight='bold', color='white')

        for r in range(1, len(d_show) + 1):
            for j in range(len(d_show.columns)):
                table[(r, j)].set_facecolor(alt_row_color if r % 2 == 0 else 'white')

        for cell in table.get_celld().values():
            cell.set_edgecolor('black')
            cell.set_linewidth(1.0)

        title = 'Part 3D Logbook Example (first 10 rows, compact)'

        mode_txt = 'na'
        p_txt = 'na'
        min_int_txt = 'na'
        if 'signal_mode' in d.columns and d['signal_mode'].notna().any():
            mode_txt = str(d['signal_mode'].dropna().iloc[0])
        if 'signal_p_threshold' in d.columns and d['signal_p_threshold'].notna().any():
            p_txt = f"{float(pd.to_numeric(d['signal_p_threshold'], errors='coerce').dropna().iloc[0]):.3g}"
        if 'min_recalibration_interval' in d.columns and d['min_recalibration_interval'].notna().any():
            min_int_txt = f"{int(round(float(pd.to_numeric(d['min_recalibration_interval'], errors='coerce').dropna().iloc[0])))}"

        footnote = (
            f'signal metric={EXAMPLE_SIGNAL_METRIC} | ai={EXAMPLE_AI} | step={EXAMPLE_STEP} | '
            f'signal mode={mode_txt} | p<thr={p_txt} | min recal interval={min_int_txt} exams | '
            'thr changed: yes = threshold updated at this decision, no = kept previous threshold'
        )

        fig.suptitle(title, fontsize=16, fontweight='bold', y=0.985)

        # Readable multiline footnote for exported images.
        footnote_wrapped = textwrap.fill(str(footnote), width=120)
        fig.text(
            0.5, 0.016,
            footnote_wrapped,
            ha='center', va='bottom',
            fontsize=14,
            linespacing=1.25,
        )
        plt.tight_layout(rect=[0, 0.15, 1, 0.95])

        out_path = out_dir / 'p3d_logbook_first10_example.png'
        plt.savefig(out_path, dpi=300, bbox_inches='tight', facecolor='white')
        plt.show()

        # Save compact rows used for image.
        csv_out = out_dir / 'p3d_logbook_first10_example.csv'
        d_show.to_csv(csv_out, index=False)

        print('Saved image:', out_path)
        print('Saved CSV:', csv_out)
        display(d_show)


In [ ]:
# Part 3D.2c - Time-weighted performance comparison figure (recal vs no recal + target)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

out_dir = globals().get('P3D_OUTPUT_DIR', Path('outputs/2025_12_29_counterfactual_manufacturer_swap_setup/part3d_oracle_recalibration'))
out_dir = Path(out_dir)
out_dir.mkdir(parents=True, exist_ok=True)

# Prefer dedicated TW comparison summaries (generated by Part 3D.2)
prof_files = sorted(out_dir.glob('p3d_choice1_professor_summary_*.csv')) + sorted(out_dir.glob('p3d_choice1_tw_comparison_summary_*.csv'))

frames = []
for f in prof_files:
    try:
        d = pd.read_csv(f)
        if 'ai_system' not in d.columns:
            # infer from filename if needed
            ai_guess = f.stem.replace('p3d_choice1_professor_summary_', '').replace('p3d_choice1_tw_comparison_summary_', '').strip()
            d['ai_system'] = ai_guess
        frames.append(d)
    except Exception:
        continue

if len(frames) == 0:
    print('No TW comparison summary CSV found. Run Part 3D.2 first.')
else:
    df = pd.concat(frames, ignore_index=True)

    required = [
        'ai_system', 'step_size',
        'target_sensitivity', 'target_specificity',
        'tw_sensitivity_recal', 'tw_sensitivity_no_recal',
        'tw_specificity_recal', 'tw_specificity_no_recal',
    ]
    missing = [c for c in required if c not in df.columns]
    if len(missing) > 0:
        print('Missing required columns:', missing)
        print('Run Part 3D.2 again to regenerate updated TW comparison summary CSVs.')
    else:
        num_cols = [c for c in required if c not in ['ai_system']]
        for c in num_cols:
            df[c] = pd.to_numeric(df[c], errors='coerce')

        df['ai_system'] = df['ai_system'].astype(str).str.lower()
        ai_order = sorted(df['ai_system'].dropna().unique().tolist())

        # Save one combined CSV for convenience
        combined_csv = out_dir / 'p3d_choice1_tw_comparison_summary_all_ai.csv'
        df.sort_values(['ai_system', 'step_size']).to_csv(combined_csv, index=False)

        n_ai = len(ai_order)
        fig, axes = plt.subplots(n_ai, 2, figsize=(16.5, 4.1 * n_ai), sharex=True)
        if n_ai == 1:
            axes = np.array([axes])

        for r, ai in enumerate(ai_order):
            g = df[df['ai_system'] == ai].sort_values('step_size').copy()
            x = g['step_size'].to_numpy(dtype=float)

            # Sensitivity panel
            ax_s = axes[r, 0]
            ax_s.plot(x, g['tw_sensitivity_recal'], '-o', color='#1f77b4', linewidth=2.0, label='TW sensitivity (recal)')
            ax_s.plot(x, g['tw_sensitivity_no_recal'], '--o', color='#7fb3e6', linewidth=1.8, label='TW sensitivity (no recal)')
            if g['target_sensitivity'].notna().any():
                t_s = float(g['target_sensitivity'].dropna().iloc[0])
                ax_s.axhline(t_s, linestyle=':', color='black', linewidth=1.6, label='Target sensitivity')
            d_s = pd.to_numeric(g['tw_sensitivity_recal'], errors='coerce') - pd.to_numeric(g['tw_sensitivity_no_recal'], errors='coerce')
            d_s_txt = f" | mean Δ(recal-no)={float(d_s.mean()):+.2f}" if d_s.notna().any() else ''
            ax_s.set_title(f"{ai.upper()} | Sensitivity{d_s_txt}", fontsize=12.0, fontweight='bold')
            ax_s.set_ylabel('Rate (%)', fontsize=11.5)
            ax_s.grid(True, alpha=0.25)
            ax_s.tick_params(axis='both', labelsize=11)

            # Specificity panel
            ax_p = axes[r, 1]
            ax_p.plot(x, g['tw_specificity_recal'], '-s', color='#ff7f0e', linewidth=2.0, label='TW specificity (recal)')
            ax_p.plot(x, g['tw_specificity_no_recal'], '--s', color='#f3b37f', linewidth=1.8, label='TW specificity (no recal)')
            if g['target_specificity'].notna().any():
                t_p = float(g['target_specificity'].dropna().iloc[0])
                ax_p.axhline(t_p, linestyle=':', color='black', linewidth=1.6, label='Target specificity')
            d_p = pd.to_numeric(g['tw_specificity_recal'], errors='coerce') - pd.to_numeric(g['tw_specificity_no_recal'], errors='coerce')
            d_p_txt = f" | mean Δ(recal-no)={float(d_p.mean()):+.2f}" if d_p.notna().any() else ''
            ax_p.set_title(f"{ai.upper()} | Specificity{d_p_txt}", fontsize=12.0, fontweight='bold')
            ax_p.grid(True, alpha=0.25)
            ax_p.tick_params(axis='both', labelsize=11)

        for ax in axes[-1, :]:
            ax.set_xlabel('Step size (exams) [gap=step]', fontsize=11.5)

        # one merged legend for the whole figure
        h_all, l_all = [], []
        for ax in axes.flatten():
            h, l = ax.get_legend_handles_labels()
            h_all.extend(h)
            l_all.extend(l)
        uniq = {}
        for h, l in zip(h_all, l_all):
            if l not in uniq:
                uniq[l] = h

        fig.legend(
            list(uniq.values()),
            list(uniq.keys()),
            loc='lower center',
            bbox_to_anchor=(0.5, 0.09),
            ncol=3,
            fontsize=11.5,
            frameon=True,
        )

        hospital_txt = str(globals().get('P3D_HOSPITAL', 'selected hospital'))
        mfr_txt = str(globals().get('P3D_MANUFACTURER', 'selected manufacturer'))
        sig_txt = str(globals().get('P3D_PRIMARY_OBJECTIVE', 'flagging_rate'))
        mode_txt = str(globals().get('P3D_SIGNAL_MODE', 'drift'))
        p_thr = globals().get('P3D_SIGNAL_P_THRESHOLD', None)
        min_int = int(globals().get('P3D_MIN_RECALIBRATION_INTERVAL', 0))
        prior = int(globals().get('P3D_PRIOR_SIZE', np.nan)) if pd.notna(globals().get('P3D_PRIOR_SIZE', np.nan)) else np.nan
        curr = int(globals().get('P3D_CURRENT_SIZE', np.nan)) if pd.notna(globals().get('P3D_CURRENT_SIZE', np.nan)) else np.nan
        fut = int(globals().get('P3D_FUTURE_SIZE', np.nan)) if pd.notna(globals().get('P3D_FUTURE_SIZE', np.nan)) else np.nan

        rule_txt = f'Signal rule: mode={mode_txt}'
        if str(mode_txt).lower() == 'pvalue' and p_thr is not None:
            try:
                rule_txt += f', p<{float(p_thr):.3g}'
            except Exception:
                rule_txt += f', p<{p_thr}'
        rule_txt += f'; min recal interval={min_int:,} exams.'

        foot = (
            f'Context: hospital={hospital_txt} | manufacturer={mfr_txt} | signal metric={sig_txt} | '
            f'prior/current/future={prior:,}/{curr:,}/{fut:,} exams.\n'
            f'{rule_txt}\n'
            'Time-weighted mean = average over threshold-application segments weighted by segment exam count. '
            'No-recalibration baseline keeps the initial threshold fixed over time. '
            'Targets are dotted horizontal lines. Positive mean Δ(recal-no) means recalibration is higher than no-recalibration.'
        )
        fig.text(0.5, 0.02, foot, ha='center', va='bottom', fontsize=14.0, linespacing=1.28)

        fig.suptitle('Time-weighted Performance Comparison: Recalibration vs No Recalibration (with Targets)',
                     fontsize=16.0, fontweight='bold', y=0.995)
        fig.tight_layout(rect=[0, 0.24, 1, 0.95])

        out_png = out_dir / 'p3d_choice1_tw_comparison_all_ai.png'
        fig.savefig(out_png, dpi=300, bbox_inches='tight', facecolor='white')
        plt.show()

        print('Saved figure:', out_png)
        print('Saved combined CSV:', combined_csv)
        display(df.sort_values(['ai_system', 'step_size']).head(30))


In [ ]:
# Part 3D: First-change effect explainer (self-contained)
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

base = Path('outputs/2025_12_29_counterfactual_manufacturer_swap_setup/part3d_oracle_recalibration')

# Input diaries (full rows by AI)
logbook_files = {
    'lunit': base / 'p3d_choice1_logbook_all_rows_lunit.csv',
    'therapixel': base / 'p3d_choice1_logbook_all_rows_therapixel.csv',
    'vara': base / 'p3d_choice1_logbook_all_rows_vara.csv',
}

missing = [str(p) for p in logbook_files.values() if not p.exists()]
if missing:
    print('Missing input logbook CSV(s):')
    for m in missing:
        print('-', m)
else:
    rows = []

    for ai, p in logbook_files.items():
        df = pd.read_csv(p)

        for step, d in df.groupby('step_size'):
            d = d.sort_values('decision_idx').reset_index(drop=True)

            n = d['segment_n'].astype(float)
            eff_s = (d['segment_sens'] - d['_segment_sens_no_recal']) * n
            eff_p = (d['segment_spec'] - d['_segment_spec_no_recal']) * n

            changed_idx = d.index[d['threshold_changed'].fillna(0).astype(int) == 1].tolist()
            first = changed_idx[0] if changed_idx else None
            second = changed_idx[1] if len(changed_idx) > 1 else None

            if first is None:
                first_epoch_mask = pd.Series(False, index=d.index)
            else:
                if second is None:
                    first_epoch_mask = (d.index >= first)
                else:
                    first_epoch_mask = (d.index >= first) & (d.index < second)

            abs_s_total = eff_s.abs().sum()
            abs_p_total = eff_p.abs().sum()

            share_abs_s = (eff_s[first_epoch_mask].abs().sum() / abs_s_total) if abs_s_total != 0 else np.nan
            share_abs_p = (eff_p[first_epoch_mask].abs().sum() / abs_p_total) if abs_p_total != 0 else np.nan

            signed_s_total = eff_s.sum()
            signed_p_total = eff_p.sum()

            share_signed_s = (eff_s[first_epoch_mask].sum() / signed_s_total) if signed_s_total != 0 else np.nan
            share_signed_p = (eff_p[first_epoch_mask].sum() / signed_p_total) if signed_p_total != 0 else np.nan

            rows.append({
                'ai': ai,
                'step': int(step),
                'changes_total': len(changed_idx),
                'first_epoch_segments': int(first_epoch_mask.sum()),
                'first_epoch_exam_share': n[first_epoch_mask].sum() / n.sum(),
                'first_epoch_abs_share_sens_effect': share_abs_s,
                'first_epoch_abs_share_spec_effect': share_abs_p,
                'first_epoch_share_of_sens_effect_signed': share_signed_s,
                'first_epoch_share_of_spec_effect_signed': share_signed_p,
            })

    plot_df = pd.DataFrame(rows).sort_values(['ai', 'step']).reset_index(drop=True)

    # Convert ratios to % for readability
    for c in [
        'first_epoch_exam_share',
        'first_epoch_abs_share_sens_effect',
        'first_epoch_abs_share_spec_effect',
        'first_epoch_share_of_sens_effect_signed',
        'first_epoch_share_of_spec_effect_signed',
    ]:
        plot_df[c] = 100.0 * plot_df[c]

    plot_df['label'] = plot_df['ai'].str.upper() + ' | step=' + plot_df['step'].astype(int).astype(str)

    # Save decomposition table
    out_csv = base / 'p3d_first_change_effect_explainer_data.csv'
    plot_df.to_csv(out_csv, index=False)

    x = np.arange(len(plot_df))
    width = 0.26

    # -----------------------------
    # Figure 1: absolute view only
    # -----------------------------
    fig1, ax1 = plt.subplots(1, 1, figsize=(16, 7), constrained_layout=False)

    ax1.bar(x - width, plot_df['first_epoch_abs_share_sens_effect'], width=width, label='First-change share of |sens effect|')
    ax1.bar(x,         plot_df['first_epoch_abs_share_spec_effect'], width=width, label='First-change share of |spec effect|')
    ax1.bar(x + width, plot_df['first_epoch_exam_share'], width=width, label='Exam-share of first-change epoch')

    ax1.set_ylabel('Percent (%)')
    ax1.set_title('How much does the first-change epoch explain? (absolute contribution view)', fontsize=14, fontweight='bold')
    ax1.set_xticks(x)
    ax1.set_xticklabels(plot_df['label'], rotation=30, ha='right')
    ax1.grid(axis='y', alpha=0.25)
    ax1.legend(ncol=3, fontsize=11, loc='upper right')

    med_abs_s = plot_df['first_epoch_abs_share_sens_effect'].median()
    med_abs_p = plot_df['first_epoch_abs_share_spec_effect'].median()
    ax1.axhline(med_abs_s, ls='--', lw=1.5, color='tab:blue', alpha=0.8)
    ax1.axhline(med_abs_p, ls='--', lw=1.5, color='tab:orange', alpha=0.8)
    ax1.text(
        0.01, 0.03,
        f'Median first-change share |sens effect| = {med_abs_s:.2f}% | |spec effect| = {med_abs_p:.2f}%',
        transform=ax1.transAxes,
        fontsize=11,
        bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='0.7')
    )

    signal_txt = str(globals().get('P3D_PRIMARY_OBJECTIVE', 'flagging_rate'))

    fig1.suptitle(
        'Decomposition of recalibration effect: first threshold-change epoch vs entire timeline'
        f'Absolute view | Karolinska University Hospital | Hologic | signal metric={signal_txt} | step in {{500,1000,2000,4000}}',
        fontsize=16,
        fontweight='bold',
        y=0.98
    )
    fig1.text(
        0.5, 0.005,
        'If the first recalibration explained most of the effect, bars would be close to 100%. '
        'Observed values are low, indicating effects come from many later recalibrations.',
        ha='center', va='bottom', fontsize=11
    )
    fig1.subplots_adjust(top=0.80, bottom=0.24)

    out_png_abs = base / 'p3d_first_change_effect_explainer_abs.png'
    fig1.savefig(out_png_abs, dpi=200, bbox_inches='tight')
    plt.show()

    # ---------------------------
    # Figure 2: signed view only
    # ---------------------------
    fig2, ax2 = plt.subplots(1, 1, figsize=(16, 7), constrained_layout=False)

    ax2.bar(x - width/2, plot_df['first_epoch_share_of_sens_effect_signed'], width=width, label='First-change share of sens effect (signed)')
    ax2.bar(x + width/2, plot_df['first_epoch_share_of_spec_effect_signed'], width=width, label='First-change share of spec effect (signed)')
    ax2.axhline(0, color='black', lw=1)

    ax2.set_ylabel('Percent (%)')
    ax2.set_title('First-change share with sign preserved', fontsize=14, fontweight='bold')
    ax2.set_xticks(x)
    ax2.set_xticklabels(plot_df['label'], rotation=30, ha='right')
    ax2.grid(axis='y', alpha=0.25)
    ax2.legend(ncol=2, fontsize=11, loc='upper right')

    med_signed_s = plot_df['first_epoch_share_of_sens_effect_signed'].median()
    med_signed_p = plot_df['first_epoch_share_of_spec_effect_signed'].median()
    ax2.text(
        0.01, 0.03,
        f'Median signed-share sens = {med_signed_s:.2f}% | spec = {med_signed_p:.2f}%',
        transform=ax2.transAxes,
        fontsize=11,
        bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='0.7')
    )

    fig2.suptitle(
        'Decomposition of recalibration effect: first threshold-change epoch vs entire timeline'
        f'Signed view | Karolinska University Hospital | Hologic | signal metric={signal_txt} | step in {{500,1000,2000,4000}}',
        fontsize=16,
        fontweight='bold',
        y=0.98
    )
    fig2.text(
        0.5, 0.005,
        'Signed view keeps effect direction; small values indicate early effects are minor relative to the full timeline.',
        ha='center', va='bottom', fontsize=11
    )
    fig2.subplots_adjust(top=0.80, bottom=0.24)

    out_png_signed = base / 'p3d_first_change_effect_explainer_signed.png'
    fig2.savefig(out_png_signed, dpi=200, bbox_inches='tight')
    plt.show()

    print('Saved image (abs):', out_png_abs)
    print('Saved image (signed):', out_png_signed)
    print('Saved data:', out_csv)


In [ ]:
# Part 3D.6A - Three-policy TW summary (overall + per-hospital)
import numpy as np
import pandas as pd
from pathlib import Path

if 'df_part3' not in globals():
    print('df_part3 not found. Run data-load cells first.')
elif '_p3d_run_diary' not in globals() or '_p3d_get_columns' not in globals() or '_p3d_add_rad_flag' not in globals():
    print('Part 3D helper functions not found. Run Part 3D.0 first.')
else:
    out_dir = globals().get('P3D_OUTPUT_DIR', Path('outputs/2025_12_29_counterfactual_manufacturer_swap_setup/part3d_oracle_recalibration'))
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    cols = _p3d_get_columns(df_part3)
    ai_systems = cols['ai_cols'] if globals().get('P3D_AI_SYSTEMS', None) is None else [a for a in globals().get('P3D_AI_SYSTEMS', []) if a in cols['ai_cols']]

    requested_metrics = globals().get(
        'P3D_POLICY_SIGNAL_METRICS',
        globals().get('P3D_OBJECTIVES_FOR_COMPARISON', ['flagging_rate', 'relative_flagging_rate', 'p_ai_given_rad', 'p_rad_given_ai'])
    )
    valid_metrics = {'flagging_rate', 'relative_flagging_rate', 'p_ai_given_rad', 'p_rad_given_ai'}
    signal_metrics = [m for m in requested_metrics if m in valid_metrics]

    if len(signal_metrics) == 0:
        signal_metrics = ['flagging_rate', 'relative_flagging_rate', 'p_ai_given_rad', 'p_rad_given_ai']
        print('No valid signal metrics requested. Falling back to:', signal_metrics)

    prior_size = int(globals().get('P3D_POLICY_PRIOR_SIZE', 20000))
    current_size = int(globals().get('P3D_POLICY_CURRENT_SIZE', 20000))
    future_size = int(globals().get('P3D_POLICY_FUTURE_SIZE', 20000))

    step_size = int(globals().get('P3D_POLICY_STEP', 2000))
    gap_size = int(globals().get('P3D_POLICY_GAP', step_size))

    rad_mode = str(globals().get('P3D_RAD_MODE', 'single'))
    max_candidates = int(globals().get('P3D_MAX_THRESHOLD_CANDIDATES', 101))
    max_points = globals().get('P3D_MAX_DECISION_POINTS', None)

    signal_p_thr = float(globals().get('P3D_SIGNAL_P_THRESHOLD', 0.05))
    min_recal_interval = int(globals().get('P3D_MIN_RECALIBRATION_INTERVAL', 2000))

    p3d_policy_configs = {
        'without_recalibration': {
            'signal_mode': 'drift',
            'trigger_tolerance': 1e9,  # effectively no signal => no recalibration after initial calibration
            'signal_p_threshold': signal_p_thr,
            'min_recalibration_interval': 0,
        },
        'recalibrate_every_step': {
            'signal_mode': 'drift',
            'trigger_tolerance': -1.0,  # always passes drift gate when defined
            'signal_p_threshold': signal_p_thr,
            'min_recalibration_interval': 0,
        },
        'recalibrate_significant_only': {
            'signal_mode': 'pvalue',
            'trigger_tolerance': 0.0,
            'signal_p_threshold': signal_p_thr,
            'min_recalibration_interval': min_recal_interval,
        },
    }

    all_hospitals = sorted(df_part3[cols['hospital']].dropna().unique())
    hospital_filter = globals().get('P3D_POLICY_HOSPITALS', None)
    if hospital_filter is not None:
        all_hospitals = [h for h in all_hospitals if h in set(hospital_filter)]

    def _tw_mean(v, w):
        vv = pd.to_numeric(v, errors='coerce')
        ww = pd.to_numeric(w, errors='coerce').fillna(0)
        mask = vv.notna() & ww.notna() & (ww > 0)
        if mask.sum() == 0:
            return np.nan
        return float(np.average(vv[mask], weights=ww[mask]))

    def _safe_wavg(values, weights):
        v = pd.to_numeric(values, errors='coerce')
        w = pd.to_numeric(weights, errors='coerce').fillna(0)
        mask = v.notna() & w.notna() & (w > 0)
        if mask.sum() == 0:
            return np.nan
        return float(np.average(v[mask], weights=w[mask]))

    rows = []

    for hospital in all_hospitals:
        d_h = df_part3[df_part3[cols['hospital']] == hospital].copy()
        if len(d_h) == 0:
            continue

        if cols['manufacturer'] is not None and cols['manufacturer'] in d_h.columns:
            manufacturers = sorted(d_h[cols['manufacturer']].dropna().unique())
            if len(manufacturers) == 0:
                manufacturers = [None]
        else:
            manufacturers = [None]

        mfr_filter = globals().get('P3D_POLICY_MANUFACTURERS', None)
        if mfr_filter is not None:
            mfset = set(mfr_filter)
            manufacturers = [m for m in manufacturers if m in mfset]

        for manufacturer in manufacturers:
            pair_df = _p3d_prepare_pair_df(df_part3, cols, hospital, manufacturer)
            if len(pair_df) == 0:
                continue

            pair_df = _p3d_add_rad_flag(pair_df, rad_mode=rad_mode)

            for ai in ai_systems:
                for signal_metric in signal_metrics:
                    for policy_name, cfg in p3d_policy_configs.items():
                        diary, targets, meta = _p3d_run_diary(
                            df_pair=pair_df,
                            ai_col=ai,
                            cancer_col=cols['cancer'],
                            date_col=cols['date'],
                            objective=signal_metric,
                            prior_size=prior_size,
                            current_size=current_size,
                            future_size=future_size,
                            gap_size=gap_size,
                            step_size=step_size,
                            trigger_tolerance=float(cfg['trigger_tolerance']),
                            signal_mode=str(cfg['signal_mode']),
                            signal_p_threshold=float(cfg['signal_p_threshold']),
                            min_recalibration_interval=int(cfg['min_recalibration_interval']),
                            max_threshold_candidates=max_candidates,
                            max_points=max_points,
                        )

                        if diary is None or len(diary) == 0:
                            rows.append({
                                'hospital': hospital,
                                'manufacturer': manufacturer if manufacturer is not None else 'ALL',
                                'ai_system': ai,
                                'signal_metric': signal_metric,
                                'policy': policy_name,
                                'status': meta.get('reason', 'no_data') if isinstance(meta, dict) else 'no_data',
                                'step_exams': step_size,
                                'gap_exams': gap_size,
                                'n_test_steps': 0,
                                'n_signal_detected': 0,
                                'n_recalibrations': 0,
                                'n_threshold_changes': 0,
                                'mean_exams_between_recal': np.nan,
                                'median_exams_between_recal': np.nan,
                                'weights_total_exams': 0.0,
                                'target_sensitivity': np.nan,
                                'target_specificity': np.nan,
                                'tw_sensitivity_recal': np.nan,
                                'tw_specificity_recal': np.nan,
                                'tw_sensitivity_no_recal': np.nan,
                                'tw_specificity_no_recal': np.nan,
                                'delta_sens_recal_minus_no': np.nan,
                                'delta_spec_recal_minus_no': np.nan,
                            })
                            continue

                        w = pd.to_numeric(diary['segment_n'], errors='coerce').fillna(0)
                        tw_sens_recal = _tw_mean(diary['segment_sens'], w)
                        tw_spec_recal = _tw_mean(diary['segment_spec'], w)
                        tw_sens_no = _tw_mean(diary['_segment_sens_no_recal'], w) if '_segment_sens_no_recal' in diary.columns else np.nan
                        tw_spec_no = _tw_mean(diary['_segment_spec_no_recal'], w) if '_segment_spec_no_recal' in diary.columns else np.nan

                        changes = pd.to_numeric(
                            diary.loc[pd.to_numeric(diary['threshold_changed'], errors='coerce').fillna(0) > 0, 'decision_end_exam_idx'],
                            errors='coerce'
                        ).dropna().to_numpy(dtype=float)

                        if len(changes) >= 2:
                            gaps_recal = np.diff(changes)
                            mean_between_recal = float(np.mean(gaps_recal))
                            median_between_recal = float(np.median(gaps_recal))
                        else:
                            mean_between_recal = np.nan
                            median_between_recal = np.nan

                        rows.append({
                            'hospital': hospital,
                            'manufacturer': manufacturer if manufacturer is not None else 'ALL',
                            'ai_system': ai,
                            'signal_metric': signal_metric,
                            'policy': policy_name,
                            'status': meta.get('reason', 'ok') if isinstance(meta, dict) else 'ok',
                            'step_exams': step_size,
                            'gap_exams': gap_size,
                            'n_test_steps': int(len(diary)),
                            'n_signal_detected': int(pd.to_numeric(diary['signal_detected'], errors='coerce').fillna(0).sum()) if 'signal_detected' in diary.columns else np.nan,
                            'n_recalibrations': int(pd.to_numeric(diary['recalibrated'], errors='coerce').fillna(0).sum()),
                            'n_threshold_changes': int(pd.to_numeric(diary['threshold_changed'], errors='coerce').fillna(0).sum()),
                            'mean_exams_between_recal': mean_between_recal,
                            'median_exams_between_recal': median_between_recal,
                            'weights_total_exams': float(w.sum()),
                            'target_sensitivity': float(pd.to_numeric(diary['target_sens'], errors='coerce').iloc[0]),
                            'target_specificity': float(pd.to_numeric(diary['target_spec'], errors='coerce').iloc[0]),
                            'tw_sensitivity_recal': tw_sens_recal,
                            'tw_specificity_recal': tw_spec_recal,
                            'tw_sensitivity_no_recal': tw_sens_no,
                            'tw_specificity_no_recal': tw_spec_no,
                            'delta_sens_recal_minus_no': (tw_sens_recal - tw_sens_no) if pd.notna(tw_sens_recal) and pd.notna(tw_sens_no) else np.nan,
                            'delta_spec_recal_minus_no': (tw_spec_recal - tw_spec_no) if pd.notna(tw_spec_recal) and pd.notna(tw_spec_no) else np.nan,
                        })

    p3d_policy3_pair_summary = pd.DataFrame(rows)

    if len(p3d_policy3_pair_summary) == 0:
        print('No rows produced.')
    else:
        metric_cols = [
            'target_sensitivity', 'target_specificity',
            'tw_sensitivity_recal', 'tw_specificity_recal',
            'tw_sensitivity_no_recal', 'tw_specificity_no_recal',
            'delta_sens_recal_minus_no', 'delta_spec_recal_minus_no',
        ]

        # 1) Per-hospital summary (aggregated across manufacturer, weighted by available exams)
        h_rows = []
        gcols_h = ['hospital', 'ai_system', 'signal_metric', 'policy', 'step_exams', 'gap_exams']
        for keys, g in p3d_policy3_pair_summary.groupby(gcols_h, dropna=False):
            row = {
                'hospital': keys[0],
                'ai_system': keys[1],
                'signal_metric': keys[2],
                'policy': keys[3],
                'step_exams': int(keys[4]),
                'gap_exams': int(keys[5]),
                'n_pairs': int(len(g)),
                'n_test_steps_total': int(pd.to_numeric(g['n_test_steps'], errors='coerce').fillna(0).sum()),
                'n_signal_detected_total': int(pd.to_numeric(g['n_signal_detected'], errors='coerce').fillna(0).sum()),
                'n_recalibrations_total': int(pd.to_numeric(g['n_recalibrations'], errors='coerce').fillna(0).sum()),
                'n_threshold_changes_total': int(pd.to_numeric(g['n_threshold_changes'], errors='coerce').fillna(0).sum()),
                'weights_total_exams': float(pd.to_numeric(g['weights_total_exams'], errors='coerce').fillna(0).sum()),
            }
            w = pd.to_numeric(g['weights_total_exams'], errors='coerce').fillna(0)
            for c in metric_cols:
                row[c] = _safe_wavg(g[c], w)

            row['mean_exams_between_recal'] = _safe_wavg(
                g['mean_exams_between_recal'],
                pd.to_numeric(g['n_threshold_changes'], errors='coerce').fillna(0)
            )
            row['median_exams_between_recal'] = _safe_wavg(
                g['median_exams_between_recal'],
                pd.to_numeric(g['n_threshold_changes'], errors='coerce').fillna(0)
            )
            h_rows.append(row)

        p3d_policy3_hospital_summary = pd.DataFrame(h_rows)

        # 2) Overall summary (aggregated across hospitals)
        o_rows = []
        gcols_o = ['ai_system', 'signal_metric', 'policy', 'step_exams', 'gap_exams']
        for keys, g in p3d_policy3_hospital_summary.groupby(gcols_o, dropna=False):
            row = {
                'ai_system': keys[0],
                'signal_metric': keys[1],
                'policy': keys[2],
                'step_exams': int(keys[3]),
                'gap_exams': int(keys[4]),
                'n_hospitals': int(g['hospital'].nunique()),
                'n_test_steps_total': int(pd.to_numeric(g['n_test_steps_total'], errors='coerce').fillna(0).sum()),
                'n_signal_detected_total': int(pd.to_numeric(g['n_signal_detected_total'], errors='coerce').fillna(0).sum()),
                'n_recalibrations_total': int(pd.to_numeric(g['n_recalibrations_total'], errors='coerce').fillna(0).sum()),
                'n_threshold_changes_total': int(pd.to_numeric(g['n_threshold_changes_total'], errors='coerce').fillna(0).sum()),
                'weights_total_exams': float(pd.to_numeric(g['weights_total_exams'], errors='coerce').fillna(0).sum()),
            }
            w = pd.to_numeric(g['weights_total_exams'], errors='coerce').fillna(0)
            for c in metric_cols:
                row[c] = _safe_wavg(g[c], w)

            row['mean_exams_between_recal'] = _safe_wavg(
                g['mean_exams_between_recal'],
                pd.to_numeric(g['n_threshold_changes_total'], errors='coerce').fillna(0)
            )
            row['median_exams_between_recal'] = _safe_wavg(
                g['median_exams_between_recal'],
                pd.to_numeric(g['n_threshold_changes_total'], errors='coerce').fillna(0)
            )
            o_rows.append(row)

        p3d_policy3_overall_summary = pd.DataFrame(o_rows)

        # Persist
        pair_csv = out_dir / 'p3d_policy3_pair_summary.csv'
        hosp_csv = out_dir / 'p3d_policy3_hospital_summary.csv'
        over_csv = out_dir / 'p3d_policy3_overall_summary.csv'

        p3d_policy3_pair_summary.sort_values(['hospital', 'manufacturer', 'ai_system', 'signal_metric', 'policy']).to_csv(pair_csv, index=False)
        p3d_policy3_hospital_summary.sort_values(['hospital', 'ai_system', 'signal_metric', 'policy']).to_csv(hosp_csv, index=False)
        p3d_policy3_overall_summary.sort_values(['ai_system', 'signal_metric', 'policy']).to_csv(over_csv, index=False)

        print('Saved:', pair_csv)
        print('Saved:', hosp_csv)
        print('Saved:', over_csv)

        print('Pair rows:', len(p3d_policy3_pair_summary), '| Hospital rows:', len(p3d_policy3_hospital_summary), '| Overall rows:', len(p3d_policy3_overall_summary))

        display(p3d_policy3_overall_summary.sort_values(['ai_system', 'signal_metric', 'policy']).reset_index(drop=True).head(30))


In [ ]:
# Part 3D.6B - Requested summary figures/tables (overall + per hospital) [readable]
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

if 'p3d_policy3_overall_summary' not in globals() or len(p3d_policy3_overall_summary) == 0:
    print('p3d_policy3_overall_summary not found/empty. Run Part 3D.6A first.')
else:
    out_dir = Path(globals().get('P3D_OUTPUT_DIR', Path('outputs/2025_12_29_counterfactual_manufacturer_swap_setup/part3d_oracle_recalibration')))
    out_dir.mkdir(parents=True, exist_ok=True)

    policy_order = ['without_recalibration', 'recalibrate_every_step', 'recalibrate_significant_only']
    policy_label = {
        'without_recalibration': 'No recalibration',
        'recalibrate_every_step': 'Recalibrate every step',
        'recalibrate_significant_only': 'Recalibrate if significant',
    }

    d = p3d_policy3_overall_summary.copy()
    d = d[d['policy'].isin(policy_order)].copy()
    if len(d) == 0:
        print('No rows after policy filter.')
    else:
        d['policy_label'] = d['policy'].map(policy_label).fillna(d['policy'])

        for col in [
            'target_sensitivity', 'target_specificity',
            'tw_sensitivity_recal', 'tw_specificity_recal',
            'n_test_steps_total', 'step_exams', 'gap_exams',
            'n_recalibrations_total', 'n_threshold_changes_total',
            'mean_exams_between_recal'
        ]:
            if col in d.columns:
                d[col] = pd.to_numeric(d[col], errors='coerce')

        d['delta_sens_vs_target'] = d['tw_sensitivity_recal'] - d['target_sensitivity']
        d['delta_spec_vs_target'] = d['tw_specificity_recal'] - d['target_specificity']

        ai_order = sorted(d['ai_system'].dropna().astype(str).unique())
        metric_order = ['flagging_rate', 'relative_flagging_rate', 'p_ai_given_rad', 'p_rad_given_ai']
        present = set(d['signal_metric'].dropna().astype(str).unique())
        metric_order = [m for m in metric_order if m in present] + [m for m in sorted(present) if m not in metric_order]
        row_order = [(ai, sm) for ai in ai_order for sm in metric_order]
        row_labels = [f'{ai.upper()} | signal={sm}' for ai, sm in row_order]

        x = np.arange(len(row_order))
        width = 0.24
        offsets = {
            'without_recalibration': -width,
            'recalibrate_every_step': 0.0,
            'recalibrate_significant_only': width,
        }
        colors = {
            'without_recalibration': '#8da0cb',
            'recalibrate_every_step': '#fc8d62',
            'recalibrate_significant_only': '#66c2a5',
        }

        def _extract_val(df, ai, sm, pol, col):
            z = df[(df['ai_system'].astype(str) == str(ai)) & (df['signal_metric'].astype(str) == str(sm)) & (df['policy'] == pol)]
            if len(z) == 0:
                return np.nan
            return float(pd.to_numeric(z[col], errors='coerce').iloc[0])

        fig, axes = plt.subplots(2, 2, figsize=(22, 14), constrained_layout=False)

        ax = axes[0, 0]
        y_s_all = []
        for pol in policy_order:
            y = [_extract_val(d, ai, sm, pol, 'tw_sensitivity_recal') for ai, sm in row_order]
            y_s_all.extend([v for v in y if pd.notna(v)])
            ax.bar(x + offsets[pol], y, width=width, label=policy_label[pol], color=colors[pol])
        tgt_s = float(pd.to_numeric(d['target_sensitivity'], errors='coerce').dropna().mean())
        ax.axhline(tgt_s, color='black', ls=':', lw=1.8, label='Target sensitivity')
        ax.set_title('TW sensitivity (%)', fontweight='bold')
        ax.set_xticks(x)
        ax.set_xticklabels(row_labels, rotation=35, ha='right')
        ax.grid(axis='y', alpha=0.25)
        ax.set_xlabel('AI system | signal metric')
        if len(y_s_all) > 0:
            ax.set_ylim(30, max(52, float(np.nanmax(y_s_all)) + 1.5))

        ax = axes[0, 1]
        y_p_all = []
        for pol in policy_order:
            y = [_extract_val(d, ai, sm, pol, 'tw_specificity_recal') for ai, sm in row_order]
            y_p_all.extend([v for v in y if pd.notna(v)])
            ax.bar(x + offsets[pol], y, width=width, label=policy_label[pol], color=colors[pol])
        tgt_p = float(pd.to_numeric(d['target_specificity'], errors='coerce').dropna().mean())
        ax.axhline(tgt_p, color='black', ls=':', lw=1.8, label='Target specificity')
        ax.set_title('TW specificity (%)', fontweight='bold')
        ax.set_xticks(x)
        ax.set_xticklabels(row_labels, rotation=35, ha='right')
        ax.grid(axis='y', alpha=0.25)
        ax.set_xlabel('AI system | signal metric')
        if len(y_p_all) > 0:
            ax.set_ylim(80, max(95, float(np.nanmax(y_p_all)) + 1.0))

        ax = axes[1, 0]
        for pol in policy_order:
            y = [_extract_val(d, ai, sm, pol, 'n_test_steps_total') for ai, sm in row_order]
            ax.bar(x + offsets[pol], y, width=width, label=policy_label[pol], color=colors[pol])
        ax.set_title('Total number of test steps', fontweight='bold')
        ax.set_xticks(x)
        ax.set_xticklabels(row_labels, rotation=35, ha='right')
        ax.grid(axis='y', alpha=0.25)
        ax.set_xlabel('AI system | signal metric')

        ax = axes[1, 1]
        for pol in policy_order:
            y = [_extract_val(d, ai, sm, pol, 'mean_exams_between_recal') for ai, sm in row_order]
            ax.bar(x + offsets[pol], y, width=width, label=policy_label[pol], color=colors[pol])
        ax.set_title('Number of exams between recalibrations', fontweight='bold')
        ax.set_xticks(x)
        ax.set_xticklabels(row_labels, rotation=35, ha='right')
        ax.grid(axis='y', alpha=0.25)
        ax.set_xlabel('AI system | signal metric')

        handles, labels = axes[0, 0].get_legend_handles_labels()
        fig.legend(
            handles, labels,
            loc='lower center',
            bbox_to_anchor=(0.5, 0.105),
            ncol=4,
            fontsize=14,
            frameon=True
        )

        step_ex = int(pd.to_numeric(d['step_exams'], errors='coerce').dropna().iloc[0]) if pd.to_numeric(d['step_exams'], errors='coerce').notna().any() else np.nan
        gap_ex = int(pd.to_numeric(d['gap_exams'], errors='coerce').dropna().iloc[0]) if pd.to_numeric(d['gap_exams'], errors='coerce').notna().any() else np.nan
        foot = (
            f'Policies: no recalibration | recalibrate every step | recalibrate if significant. '
            f'Test cadence: every {step_ex} exams (gap={gap_ex}).\n'
            f'TW mean = weighted by segment exam count. Overall = weighted average across hospitals.'
        )

        fig.suptitle(
            'Three-policy comparison (overall across hospitals): TW performance + testing/recalibration frequency\n'
            '(Tests/recalibration driven by signal metrics: flagging_rate, relative_flagging_rate, p_ai_given_rad, p_rad_given_ai)',
            fontsize=18,
            fontweight='bold',
            y=0.98
        )
        fig.text(0.5, 0.012, foot, ha='center', va='bottom', fontsize=18, linespacing=1.30)
        fig.subplots_adjust(top=0.86, bottom=0.30, hspace=0.35, wspace=0.12)

        overall_png = out_dir / 'p3d_policy3_overall_comparison.png'
        fig.savefig(overall_png, dpi=220, bbox_inches='tight')
        plt.show()
        print('Saved:', overall_png)

        # ---------- Split into separate readable figures ----------
        # For manuscript figures, order by signal metric first and anonymize AI vendors.
        ai_label_map = {ai: f'AI-{idx + 1}' for idx, ai in enumerate(ai_order)}
        ai_name_map = {ai_label_map[ai]: str(ai).title() for ai in ai_order}

        metric_label = {
            'flagging_rate': 'Flagging rate',
            'relative_flagging_rate': 'Relative flagging rate',
            'p_ai_given_rad': 'P(AI flagged | radiologist flagged)',
            'p_rad_given_ai': 'P(radiologist flagged | AI flagged)',
        }

        row_order_fig = [(ai, sm) for sm in metric_order for ai in ai_order]
        x_fig = np.arange(len(row_order_fig))
        x_tick_labels = [ai_label_map.get(ai, str(ai)) for ai, _sm in row_order_fig]
        width_fig = 0.24
        offsets_fig = {
            'without_recalibration': -width_fig,
            'recalibrate_every_step': 0.0,
            'recalibrate_significant_only': width_fig,
        }

        split_specs = [
            (
                'tw_sensitivity_recal',
                'Time-weighted sensitivity (%)',
                'Target sensitivity',
                tgt_s,
                'p3d_policy3_overall_tw_sensitivity.png',
                'Time-weighted sensitivity (%)',
                30,
                52,
            ),
            (
                'tw_specificity_recal',
                'Time-weighted specificity (%)',
                'Target specificity',
                tgt_p,
                'p3d_policy3_overall_tw_specificity.png',
                'Time-weighted specificity (%)',
                80,
                96,
            ),
            (
                'n_test_steps_total',
                'Total number of test steps',
                None,
                None,
                'p3d_policy3_overall_n_test_steps.png',
                'Number of test steps',
                None,
                None,
            ),
            (
                'mean_exams_between_recal',
                'Number of exams between recalibrations',
                None,
                None,
                'p3d_policy3_overall_exams_between_recalibrations.png',
                'Exams between recalibrations',
                None,
                None,
            ),
        ]

        from matplotlib.lines import Line2D

        def _legend_note_handles_for(col_name):
            notes = [
                Line2D([], [], color='none', label=f'Test cadence: every {step_ex:,} exams; gap = {gap_ex:,} exams'),
            ]
            if col_name in ['tw_sensitivity_recal', 'tw_specificity_recal']:
                notes.append(Line2D([], [], color='none', label='Time-weighted mean: weighted by segment exam count'))
            notes.append(Line2D([], [], color='none', label='Overall: weighted average across hospitals'))
            return notes

        for col_name, panel_title, target_label, target_val, out_name, y_label, y_min, y_floor_max in split_specs:
            fig_s, ax_s = plt.subplots(figsize=(18, 8), constrained_layout=False)

            y_all = []
            for pol in policy_order:
                y = [_extract_val(d, ai, sm, pol, col_name) for ai, sm in row_order_fig]
                y_all.extend([v for v in y if pd.notna(v)])
                ax_s.bar(
                    x_fig + offsets_fig[pol],
                    y,
                    width=width_fig,
                    label=policy_label[pol],
                    color=colors[pol],
                )

            if target_label is not None and target_val is not None and pd.notna(target_val):
                ax_s.axhline(float(target_val), color='black', ls=':', lw=2.0, label=target_label)

            ax_s.set_title(
                f'Three-policy comparison across hospitals: {panel_title}',
                fontweight='bold',
                fontsize=16,
                pad=12,
            )
            ax_s.set_ylabel(y_label, fontsize=12)
            ax_s.set_xlabel('Signal metric group and anonymized AI system', fontsize=12, labelpad=42)
            ax_s.set_xticks(x_fig)
            ax_s.set_xticklabels(x_tick_labels, fontsize=11)
            ax_s.tick_params(axis='y', labelsize=11)
            ax_s.grid(axis='y', alpha=0.25)

            for group_idx, sm in enumerate(metric_order):
                start_pos = group_idx * len(ai_order)
                end_pos = start_pos + len(ai_order) - 1
                center_pos = (start_pos + end_pos) / 2
                ax_s.text(
                    center_pos,
                    -0.105,
                    metric_label.get(sm, str(sm).replace('_', ' ').title()),
                    transform=ax_s.get_xaxis_transform(),
                    ha='center',
                    va='top',
                    fontsize=11,
                )
                if group_idx > 0:
                    ax_s.axvline(start_pos - 0.5, color='0.85', lw=1.0, zorder=0)

            if y_min is not None and y_floor_max is not None:
                vmax = max(y_all) if len(y_all) > 0 else y_floor_max
                ax_s.set_ylim(y_min, max(y_floor_max, float(vmax) + 1.0))

            handles_s, labels_s = ax_s.get_legend_handles_labels()
            legend_note_handles = _legend_note_handles_for(col_name)
            handles_s = handles_s + legend_note_handles
            labels_s = labels_s + [h.get_label() for h in legend_note_handles]
            ax_s.legend(
                handles_s,
                labels_s,
                loc='center left',
                bbox_to_anchor=(1.01, 0.5),
                ncol=1,
                fontsize=10,
                title='Policies and figure notes',
                title_fontsize=11,
                frameon=True,
            )

            fig_s.subplots_adjust(top=0.88, bottom=0.20, left=0.07, right=0.76)

            split_png = out_dir / out_name
            fig_s.savefig(split_png, dpi=220, bbox_inches='tight')
            plt.show()
            print('Saved:', split_png)

        print('Image AI label mapping:', ', '.join(f'{anon} = {name}' for anon, name in ai_name_map.items()))

        # ---------- Readable overall tables (long format) ----------
        cols_overall = [
            'ai_system', 'signal_metric', 'policy_label',
            'target_sensitivity', 'tw_sensitivity_recal', 'delta_sens_vs_target',
            'target_specificity', 'tw_specificity_recal', 'delta_spec_vs_target',
            'n_test_steps_total', 'step_exams',
            'n_recalibrations_total', 'n_threshold_changes_total',
            'mean_exams_between_recal'
        ]
        ov = d[cols_overall].copy()
        ov = ov.rename(columns={
            'ai_system': 'AI system',
            'signal_metric': 'Signal metric',
            'policy_label': 'Policy',
            'target_sensitivity': 'Target sens (%)',
            'tw_sensitivity_recal': 'TW sens with recal (%)',
            'delta_sens_vs_target': 'Δ sens vs target',
            'target_specificity': 'Target spec (%)',
            'tw_specificity_recal': 'TW spec with recal (%)',
            'delta_spec_vs_target': 'Δ spec vs target',
            'n_test_steps_total': 'N test steps',
            'step_exams': 'Exams between tests',
            'n_recalibrations_total': 'N recalibrations',
            'n_threshold_changes_total': 'N threshold changes',
            'mean_exams_between_recal': 'Mean exams between recal'
        })

        for c in ov.columns:
            if c in ['AI system', 'Signal metric', 'Policy']:
                continue
            ov[c] = pd.to_numeric(ov[c], errors='coerce').round(2)

        ov = ov.sort_values(['AI system', 'Signal metric', 'Policy']).reset_index(drop=True)

        ov_csv = out_dir / 'p3d_policy3_overall_table.csv'
        ov_png = out_dir / 'p3d_policy3_overall_table.png'
        ov.to_csv(ov_csv, index=False)

        fig_t, ax_t = plt.subplots(figsize=(24, max(9, 1.4 + 0.50 * len(ov))))
        ax_t.axis('off')
        tbl = ax_t.table(cellText=ov.values, colLabels=ov.columns, cellLoc='center', loc='center')
        tbl.auto_set_font_size(False)
        tbl.set_fontsize(11.5)
        tbl.scale(1.0, 1.50)
        ax_t.set_title('Overall table (readable): TW performance + testing/recalibration counts', fontsize=16, fontweight='bold', pad=14)
        fig_t.text(
            0.5, 0.01,
            'Each row is one (AI system, signal metric, policy). This replaces the unreadable wide pivot table.',
            ha='center', va='bottom', fontsize=14
        )
        fig_t.savefig(ov_png, dpi=220, bbox_inches='tight')
        plt.show()

        print('Saved:', ov_csv)
        print('Saved:', ov_png)

        # ---------- Per-hospital readable tables ----------
        if 'p3d_policy3_hospital_summary' in globals() and len(p3d_policy3_hospital_summary) > 0:
            h = p3d_policy3_hospital_summary.copy()
            h = h[h['policy'].isin(policy_order)].copy()
            if len(h) > 0:
                h['policy_label'] = h['policy'].map(policy_label).fillna(h['policy'])
                for col in [
                    'target_sensitivity', 'target_specificity',
                    'tw_sensitivity_recal', 'tw_specificity_recal',
                    'n_test_steps_total', 'step_exams',
                    'n_recalibrations_total', 'n_threshold_changes_total',
                    'mean_exams_between_recal'
                ]:
                    if col in h.columns:
                        h[col] = pd.to_numeric(h[col], errors='coerce')

                h['delta_sens_vs_target'] = h['tw_sensitivity_recal'] - h['target_sensitivity']
                h['delta_spec_vs_target'] = h['tw_specificity_recal'] - h['target_specificity']

                hospitals = sorted(h['hospital'].dropna().astype(str).unique())
                for hospital in hospitals:
                    hh = h[h['hospital'].astype(str) == str(hospital)].copy()
                    if len(hh) == 0:
                        continue

                    keep = [
                        'ai_system', 'signal_metric', 'policy_label',
                        'target_sensitivity', 'tw_sensitivity_recal', 'delta_sens_vs_target',
                        'target_specificity', 'tw_specificity_recal', 'delta_spec_vs_target',
                        'n_test_steps_total', 'step_exams',
                        'n_recalibrations_total', 'n_threshold_changes_total',
                        'mean_exams_between_recal'
                    ]
                    htab = hh[keep].copy()
                    htab = htab.rename(columns={
                        'ai_system': 'AI system',
                        'signal_metric': 'Signal metric',
                        'policy_label': 'Policy',
                        'target_sensitivity': 'Target sens (%)',
                        'tw_sensitivity_recal': 'TW sens with recal (%)',
                        'delta_sens_vs_target': 'Δ sens vs target',
                        'target_specificity': 'Target spec (%)',
                        'tw_specificity_recal': 'TW spec with recal (%)',
                        'delta_spec_vs_target': 'Δ spec vs target',
                        'n_test_steps_total': 'N test steps',
                        'step_exams': 'Exams between tests',
                        'n_recalibrations_total': 'N recalibrations',
                        'n_threshold_changes_total': 'N threshold changes',
                        'mean_exams_between_recal': 'Mean exams between recal'
                    })

                    for c in htab.columns:
                        if c in ['AI system', 'Signal metric', 'Policy']:
                            continue
                        htab[c] = pd.to_numeric(htab[c], errors='coerce').round(2)

                    htab = htab.sort_values(['AI system', 'Signal metric', 'Policy']).reset_index(drop=True)

                    slug = ''.join(ch.lower() if ch.isalnum() else '_' for ch in str(hospital)).strip('_')
                    h_csv = out_dir / f'p3d_policy3_hospital_table_{slug}.csv'
                    h_png = out_dir / f'p3d_policy3_hospital_table_{slug}.png'

                    htab.to_csv(h_csv, index=False)

                    fig_h, ax_h = plt.subplots(figsize=(24, max(9, 1.4 + 0.50 * len(htab))))
                    ax_h.axis('off')
                    tbl_h = ax_h.table(cellText=htab.values, colLabels=htab.columns, cellLoc='center', loc='center')
                    tbl_h.auto_set_font_size(False)
                    tbl_h.set_fontsize(11.5)
                    tbl_h.scale(1.0, 1.50)
                    ax_h.set_title(f'Hospital table (readable): {hospital}', fontsize=15, fontweight='bold', pad=14)
                    fig_h.text(0.5, 0.012, 'Rows = AI system + signal metric + policy.', ha='center', va='bottom', fontsize=14)
                    fig_h.savefig(h_png, dpi=220, bbox_inches='tight')
                    plt.close(fig_h)

                print('Saved per-hospital readable tables in:', out_dir)


In [ ]:
# Part 3D.6C - Simplified overall summary (readable tables)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

if 'p3d_policy3_overall_summary' not in globals() or len(p3d_policy3_overall_summary) == 0:
    print('p3d_policy3_overall_summary not found/empty. Run Part 3D.6A first.')
else:
    out_dir = Path(globals().get('P3D_OUTPUT_DIR', Path('outputs/2025_12_29_counterfactual_manufacturer_swap_setup/part3d_oracle_recalibration')))
    out_dir.mkdir(parents=True, exist_ok=True)

    policy_order = ['without_recalibration', 'recalibrate_every_step', 'recalibrate_significant_only']
    policy_label = {
        'without_recalibration': 'No recalibration',
        'recalibrate_every_step': 'Recalibrate every step',
        'recalibrate_significant_only': 'Recalibrate if significant',
    }
    signal_order = ['flagging_rate', 'relative_flagging_rate', 'p_ai_given_rad', 'p_rad_given_ai']

    d = p3d_policy3_overall_summary.copy()
    d = d[d['policy'].isin(policy_order)].copy()
    if len(d) == 0:
        print('No rows after policy filter.')
    else:
        d['policy_label'] = d['policy'].map(policy_label).fillna(d['policy'])
        d['signal_metric'] = d['signal_metric'].astype(str)
        d['ai_system'] = d['ai_system'].astype(str)

        for col in [
            'target_sensitivity', 'target_specificity',
            'tw_sensitivity_recal', 'tw_specificity_recal',
            'n_test_steps_total', 'step_exams', 'gap_exams',
            'n_recalibrations_total', 'n_threshold_changes_total',
            'mean_exams_between_recal'
        ]:
            if col in d.columns:
                d[col] = pd.to_numeric(d[col], errors='coerce')

        d['delta_sens_vs_target'] = d['tw_sensitivity_recal'] - d['target_sensitivity']
        d['delta_spec_vs_target'] = d['tw_specificity_recal'] - d['target_specificity']
        d['distance_to_target'] = np.sqrt(d['delta_sens_vs_target']**2 + d['delta_spec_vs_target']**2)

        # Mark best policy per (ai, signal): smallest distance to target
        d['is_best_policy'] = False
        for (ai, sm), g in d.groupby(['ai_system', 'signal_metric'], dropna=False):
            if len(g) == 0:
                continue
            idx = pd.to_numeric(g['distance_to_target'], errors='coerce').idxmin()
            if pd.notna(idx):
                d.loc[idx, 'is_best_policy'] = True

        ai_levels = sorted(d['ai_system'].dropna().unique())
        signal_levels = [s for s in signal_order if s in set(d['signal_metric'])] + [s for s in sorted(set(d['signal_metric'])) if s not in signal_order]

        d['ai_system'] = pd.Categorical(d['ai_system'], categories=ai_levels, ordered=True)
        d['signal_metric'] = pd.Categorical(d['signal_metric'], categories=signal_levels, ordered=True)
        d['policy'] = pd.Categorical(d['policy'], categories=policy_order, ordered=True)
        d = d.sort_values(['ai_system', 'signal_metric', 'policy']).reset_index(drop=True)

        perf_cols = [
            'ai_system', 'signal_metric', 'policy_label',
            'target_sensitivity', 'target_specificity',
            'tw_sensitivity_recal', 'tw_specificity_recal',
            'delta_sens_vs_target', 'delta_spec_vs_target',
            'distance_to_target', 'is_best_policy'
        ]
        perf = d[perf_cols].copy()
        perf = perf.rename(columns={
            'ai_system': 'AI system',
            'signal_metric': 'Signal metric',
            'policy_label': 'Policy',
            'target_sensitivity': 'Target sens (%)',
            'target_specificity': 'Target spec (%)',
            'tw_sensitivity_recal': 'TW sens with recal (%)',
            'tw_specificity_recal': 'TW spec with recal (%)',
            'delta_sens_vs_target': 'Δ sens vs target',
            'delta_spec_vs_target': 'Δ spec vs target',
            'distance_to_target': 'Distance to target',
            'is_best_policy': 'Best policy (min distance)'
        })

        ops_cols = [
            'ai_system', 'signal_metric', 'policy_label',
            'n_test_steps_total', 'step_exams',
            'n_recalibrations_total', 'n_threshold_changes_total',
            'mean_exams_between_recal'
        ]
        ops = d[ops_cols].copy()
        ops = ops.rename(columns={
            'ai_system': 'AI system',
            'signal_metric': 'Signal metric',
            'policy_label': 'Policy',
            'n_test_steps_total': 'N test steps',
            'step_exams': 'Exams between tests',
            'n_recalibrations_total': 'N recalibrations',
            'n_threshold_changes_total': 'N threshold changes',
            'mean_exams_between_recal': 'Mean exams between recal'
        })

        for df_ in [perf, ops]:
            for c in df_.columns:
                if c in ['AI system', 'Signal metric', 'Policy', 'Best policy (min distance)']:
                    continue
                df_[c] = pd.to_numeric(df_[c], errors='coerce').round(2)

        perf_csv = out_dir / 'p3d_policy3_overall_performance_simple.csv'
        ops_csv = out_dir / 'p3d_policy3_overall_operations_simple.csv'
        perf.to_csv(perf_csv, index=False)
        ops.to_csv(ops_csv, index=False)

        def render_table(df_show, title, png_path, footnote):
            n_rows = len(df_show)
            fig_h = max(6, 1.0 + 0.46 * n_rows)
            fig, ax = plt.subplots(figsize=(22, fig_h + 1.2))
            ax.axis('off')
            tbl = ax.table(
                cellText=df_show.values,
                colLabels=df_show.columns,
                cellLoc='center',
                loc='center'
            )
            tbl.auto_set_font_size(False)
            tbl.set_fontsize(12.5)
            tbl.scale(1.0, 1.55)
            ax.set_title(title, fontsize=16, fontweight='bold', pad=14)
            fig.text(0.5, 0.022, footnote, ha='center', va='bottom', fontsize=15, linespacing=1.25)
            fig.savefig(png_path, dpi=220, bbox_inches='tight')
            plt.show()

        perf_png = out_dir / 'p3d_policy3_overall_performance_simple.png'
        ops_png = out_dir / 'p3d_policy3_overall_operations_simple.png'

        perf_foot = (
            'Overall = weighted average across hospitals. TW = time-weighted by segment exam count. '
            'Best policy per (AI, signal) = minimum Euclidean distance to target (sens/spec).' 
        )
        ops_foot = (
            'Operational burden summary for each policy: testing cadence and recalibration frequency.'
        )

        render_table(perf, 'Simplified overall performance table (readable)', perf_png, perf_foot)
        render_table(ops, 'Simplified overall operations table (readable)', ops_png, ops_foot)

        print('Saved:', perf_csv)
        print('Saved:', ops_csv)
        print('Saved:', perf_png)
        print('Saved:', ops_png)




In [ ]:
# Part 3D.6D - Requirement coverage check + per-hospital summary (presentation readiness)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

out_dir = Path(globals().get('P3D_OUTPUT_DIR', Path('outputs/2025_12_29_counterfactual_manufacturer_swap_setup/part3d_oracle_recalibration')))
out_dir.mkdir(parents=True, exist_ok=True)

if 'p3d_policy3_hospital_summary' not in globals() or len(p3d_policy3_hospital_summary) == 0:
    print('p3d_policy3_hospital_summary not found/empty. Run Part 3D.6A first.')
else:
    h = p3d_policy3_hospital_summary.copy()

    policy_order = ['without_recalibration', 'recalibrate_every_step', 'recalibrate_significant_only']
    policy_label = {
        'without_recalibration': 'No recalibration',
        'recalibrate_every_step': 'Recalibrate every step',
        'recalibrate_significant_only': 'Recalibrate if significant',
    }

    h = h[h['policy'].isin(policy_order)].copy()
    h['policy_label'] = h['policy'].map(policy_label).fillna(h['policy'])

    for c in [
        'target_sensitivity', 'target_specificity',
        'tw_sensitivity_recal', 'tw_specificity_recal',
        'n_test_steps_total', 'step_exams',
        'n_recalibrations_total', 'n_threshold_changes_total',
        'mean_exams_between_recal', 'weights_total_exams'
    ]:
        if c in h.columns:
            h[c] = pd.to_numeric(h[c], errors='coerce')

    h['delta_sens_vs_target'] = h['tw_sensitivity_recal'] - h['target_sensitivity']
    h['delta_spec_vs_target'] = h['tw_specificity_recal'] - h['target_specificity']

    required_files = [
        'p3d_policy3_overall_comparison.png',
        'p3d_policy3_overall_table.png',
        'p3d_policy3_overall_performance_simple.png',
        'p3d_policy3_overall_operations_simple.png',
    ]
    file_status = []
    for fn in required_files:
        file_status.append({'artifact': fn, 'exists': (out_dir / fn).exists()})

    hospitals = sorted(h['hospital'].dropna().astype(str).unique())
    ai_n = h['ai_system'].dropna().astype(str).nunique()
    signal_n = h['signal_metric'].dropna().astype(str).nunique()
    policy_n = len(policy_order)
    expected_per_hospital = ai_n * signal_n * policy_n

    completeness_rows = []
    for hosp in hospitals:
        hh = h[h['hospital'].astype(str) == hosp]
        completeness_rows.append({
            'hospital': hosp,
            'rows_found': int(len(hh)),
            'rows_expected': int(expected_per_hospital),
            'complete': bool(len(hh) == expected_per_hospital),
        })

    coverage_df = pd.DataFrame(completeness_rows).sort_values('hospital').reset_index(drop=True)
    file_df = pd.DataFrame(file_status)

    hosp_rows = []
    for hosp in hospitals:
        hh = h[h['hospital'].astype(str) == hosp].copy()
        if len(hh) == 0:
            continue
        w = pd.to_numeric(hh['weights_total_exams'], errors='coerce').fillna(0)

        def wavg(col):
            x = pd.to_numeric(hh[col], errors='coerce')
            m = x.notna() & (w > 0)
            if m.sum() == 0:
                return np.nan
            return float(np.average(x[m], weights=w[m]))

        hosp_rows.append({
            'Hospital': hosp,
            'Target sens (%)': wavg('target_sensitivity'),
            'TW sens with recal (%)': wavg('tw_sensitivity_recal'),
            'Δ sens vs target': wavg('delta_sens_vs_target'),
            'Target spec (%)': wavg('target_specificity'),
            'TW spec with recal (%)': wavg('tw_specificity_recal'),
            'Δ spec vs target': wavg('delta_spec_vs_target'),
            'Total test steps': int(pd.to_numeric(hh['n_test_steps_total'], errors='coerce').fillna(0).sum()),
            'Total recalibrations': int(pd.to_numeric(hh['n_recalibrations_total'], errors='coerce').fillna(0).sum()),
            'Mean exams between recal': float(pd.to_numeric(hh['mean_exams_between_recal'], errors='coerce').dropna().mean()) if pd.to_numeric(hh['mean_exams_between_recal'], errors='coerce').dropna().size else np.nan,
            'Rows complete': bool(len(hh) == expected_per_hospital),
        })

    hosp_summary = pd.DataFrame(hosp_rows).sort_values('Hospital').reset_index(drop=True)

    for c in hosp_summary.columns:
        if c in ['Hospital', 'Rows complete']:
            continue
        hosp_summary[c] = pd.to_numeric(hosp_summary[c], errors='coerce').round(2)

    coverage_csv = out_dir / 'p3d_policy3_requirement_coverage.csv'
    file_csv = out_dir / 'p3d_policy3_artifact_status.csv'
    hosp_csv = out_dir / 'p3d_policy3_hospital_summary_compact.csv'

    coverage_df.to_csv(coverage_csv, index=False)
    file_df.to_csv(file_csv, index=False)
    hosp_summary.to_csv(hosp_csv, index=False)

    fig_h, ax_h = plt.subplots(figsize=(24, max(8, 1.35 + 0.65 * len(hosp_summary))))
    ax_h.axis('off')
    tbl = ax_h.table(cellText=hosp_summary.values, colLabels=hosp_summary.columns, cellLoc='center', loc='center')
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(12.5)
    tbl.scale(1.0, 1.60)
    ax_h.set_title('Per-hospital compact summary (TW performance + operational totals)', fontsize=16, fontweight='bold', pad=14)
    fig_h.text(
        0.5, 0.01,
        'Readiness check: each hospital should have all AI × signal metrics × 3 policies. This table is the per-hospital counterpart to the overall summary.',
        ha='center', va='bottom', fontsize=15, linespacing=1.25
    )
    hosp_png = out_dir / 'p3d_policy3_hospital_summary_compact.png'
    fig_h.savefig(hosp_png, dpi=220, bbox_inches='tight')
    plt.show()

    print('Saved:', coverage_csv)
    print('Saved:', file_csv)
    print('Saved:', hosp_csv)
    print('Saved:', hosp_png)

    print('Artifact status:')
    display(file_df)
    print('Hospital row completeness:')
    display(coverage_df)




In [ ]:
# Part 3D.6E - Reproducible theoretical screen-detected signal figures
#
# Purpose
# -------
# This cell makes the sensitivity/specificity figures that include the extra
# theoretical signal metric `screen_flagging_rate`.
#
# The standard four signal metrics are produced in Part 3D.6A/6B. This cell adds
# a fifth metric where the recalibration signal is:
#
#     screen_flagging_rate = P(AI flagged | screen-detected cancer)
#
# with screen-detected cancer defined as has_cancer == 1 and diagnosis within
# SCREEN_THRESHOLD_DAYS (90 days). This is theoretical because it uses delayed
# outcome information that would not be available at real-time monitoring.
#
# Outputs are written to a separate folder so the standard four-metric figures
# remain untouched.

import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.lines import Line2D

required_names = [
    'df_part3', '_p3d_get_columns', '_p3d_prepare_pair_df', '_p3d_add_rad_flag',
    '_p3d_decision_end_indices', '_p3d_rate_stats_from_flags', '_p3d_rate_stats_from_threshold',
    '_p3d_choose_initial_threshold_eval_targets', '_p3d_time_weighted_mean', '_p3d_two_prop_pvalue',
    '_p3d_candidate_thresholds'
]
missing = [name for name in required_names if name not in globals()]
if missing:
    raise RuntimeError('Run Part 3D.0 and Part 3D.6A before this cell. Missing: ' + ', '.join(missing))

base_out_dir = Path(globals().get(
    'P3D_OUTPUT_DIR',
    Path('outputs/2025_12_29_counterfactual_manufacturer_swap_setup/part3d_oracle_recalibration')
))
screen_out_dir = base_out_dir.parent / 'part3d_oracle_recalibration_with_screen_theoretical'
screen_out_dir.mkdir(parents=True, exist_ok=True)

cols = _p3d_get_columns(df_part3)
ai_systems = cols['ai_cols'] if globals().get('P3D_AI_SYSTEMS', None) is None else [
    a for a in globals().get('P3D_AI_SYSTEMS', []) if a in cols['ai_cols']
]

prior_size = int(globals().get('P3D_POLICY_PRIOR_SIZE', 20000))
current_size = int(globals().get('P3D_POLICY_CURRENT_SIZE', 20000))
future_size = int(globals().get('P3D_POLICY_FUTURE_SIZE', 20000))
step_size = int(globals().get('P3D_POLICY_STEP', 2000))
gap_size = int(globals().get('P3D_POLICY_GAP', step_size))
rad_mode = str(globals().get('P3D_RAD_MODE', 'single'))
max_candidates = int(globals().get('P3D_MAX_THRESHOLD_CANDIDATES', 101))
signal_p_thr = float(globals().get('P3D_SIGNAL_P_THRESHOLD', 0.05))
min_recal_interval = int(globals().get('P3D_MIN_RECALIBRATION_INTERVAL', 2000))
max_points = globals().get('P3D_MAX_DECISION_POINTS', None)

# Ensure the delayed-outcome screen-detected flag exists in df_part3.
if 'time_based_screen' not in df_part3.columns:
    if 'days_to_diagnosis_pos' not in df_part3.columns:
        if 'extract_min_positive_days' in globals() and 'days_to_diagnosis' in df_part3.columns:
            df_part3['days_to_diagnosis_pos'] = df_part3['days_to_diagnosis'].apply(extract_min_positive_days)
        else:
            raise RuntimeError('Cannot create time_based_screen: missing days_to_diagnosis/extract_min_positive_days.')
    df_part3['time_based_screen'] = (
        (df_part3[cols['cancer']].fillna(0).astype(int) == 1)
        & (df_part3['days_to_diagnosis_pos'].notna())
        & (df_part3['days_to_diagnosis_pos'] <= int(globals().get('SCREEN_THRESHOLD_DAYS', 90)))
    )


def _p3d_screen_signal_stats_from_threshold(window_df, ai_col, thr, screen_col='time_based_screen'):
    scores = pd.to_numeric(window_df[ai_col], errors='coerce').to_numpy(dtype=float)
    screen = window_df[screen_col].fillna(False).astype(bool).to_numpy()
    valid = (~np.isnan(scores)) & screen
    scores = scores[valid]

    n_screen = int(len(scores))
    if n_screen == 0 or pd.isna(thr):
        return {'screen_flagging_rate': np.nan, 'screen_n': n_screen, 'screen_ai_pos': 0}

    ai_pos = int((scores >= float(thr)).sum())
    return {
        'screen_flagging_rate': 100.0 * ai_pos / n_screen,
        'screen_n': n_screen,
        'screen_ai_pos': ai_pos,
    }


def _p3d_screen_signal_pvalue(prior_sig, curr_sig):
    return _p3d_two_prop_pvalue(
        curr_sig.get('screen_ai_pos', 0), curr_sig.get('screen_n', 0),
        prior_sig.get('screen_ai_pos', 0), prior_sig.get('screen_n', 0),
    )


def _p3d_choose_threshold_for_screen_rate(window_df, ai_col, target_rate, screen_col='time_based_screen', max_candidates=101):
    if pd.isna(target_rate):
        return np.nan, np.nan

    scores_all = pd.to_numeric(window_df[ai_col], errors='coerce').to_numpy(dtype=float)
    thresholds = _p3d_candidate_thresholds(scores_all, max_candidates=max_candidates)
    if len(thresholds) == 0:
        return np.nan, np.nan

    obj = []
    for thr in thresholds:
        sig = _p3d_screen_signal_stats_from_threshold(window_df, ai_col, thr, screen_col=screen_col)
        val = sig.get('screen_flagging_rate', np.nan)
        obj.append(np.inf if pd.isna(val) else abs(float(val) - float(target_rate)))

    obj = np.asarray(obj, dtype=float)
    if np.isinf(obj).all():
        return np.nan, np.nan
    best_idx = int(np.argmin(obj))
    return float(thresholds[best_idx]), float(obj[best_idx])


def _p3d_run_diary_screen_flagging_rate(
    df_pair,
    ai_col,
    cancer_col,
    date_col,
    prior_size,
    current_size,
    future_size,
    gap_size,
    step_size,
    policy_name,
    signal_p_threshold=0.05,
    min_recalibration_interval=2000,
    max_threshold_candidates=101,
    max_points=None,
):
    d = df_pair.reset_index(drop=True).copy()
    n = len(d)
    ends = _p3d_decision_end_indices(
        n=n,
        prior_size=prior_size,
        current_size=current_size,
        gap_size=gap_size,
        step_size=step_size,
        future_size=future_size,
        max_points=max_points,
    )
    if len(ends) == 0:
        return pd.DataFrame(), {}, {'reason': 'insufficient_exams'}

    # Evaluation targets are still radiologist sensitivity/specificity in the first prior window.
    t0 = int(ends[0])
    t0_current_start = t0 - current_size
    t0_prior_end = t0_current_start - gap_size
    t0_prior_start = t0_prior_end - prior_size
    prior0 = d.iloc[t0_prior_start:t0_prior_end]

    target_stats = _p3d_rate_stats_from_flags(prior0['rad_flagged'].to_numpy(), prior0[cancer_col].to_numpy())
    targets = {
        'sens': float(target_stats['sens']),
        'spec': float(target_stats['spec']),
        'ppv': float(target_stats['ppv']) if 'ppv' in target_stats else np.nan,
    }

    init_thr, _ = _p3d_choose_initial_threshold_eval_targets(
        prior0,
        ai_col=ai_col,
        cancer_col=cancer_col,
        targets=targets,
        max_candidates=max_threshold_candidates,
    )
    if pd.isna(init_thr):
        init_thr = float(pd.to_numeric(prior0[ai_col], errors='coerce').median())

    # The theoretical delayed-outcome signal target is defined in the same prior window.
    signal_target_stats = _p3d_screen_signal_stats_from_threshold(prior0, ai_col, init_thr)
    target_signal_value = signal_target_stats.get('screen_flagging_rate', np.nan)

    rows = []
    current_thr = init_thr
    last_threshold_change_end_idx = None

    for i, end_idx in enumerate(ends):
        end_idx = int(end_idx)
        current_start = end_idx - current_size
        prior_end = current_start - gap_size
        prior_start = prior_end - prior_size
        future_start = end_idx
        future_end = end_idx + future_size

        prior_w = d.iloc[prior_start:prior_end]
        curr_w = d.iloc[current_start:end_idx]
        future_w = d.iloc[future_start:future_end]

        prior_perf = _p3d_rate_stats_from_threshold(prior_w[ai_col].to_numpy(), prior_w[cancer_col].to_numpy(), current_thr)
        curr_perf = _p3d_rate_stats_from_threshold(curr_w[ai_col].to_numpy(), curr_w[cancer_col].to_numpy(), current_thr)
        prior_sig = _p3d_screen_signal_stats_from_threshold(prior_w, ai_col, current_thr)
        curr_sig = _p3d_screen_signal_stats_from_threshold(curr_w, ai_col, current_thr)

        curr_signal_value = curr_sig.get('screen_flagging_rate', np.nan)
        drift = np.nan if (pd.isna(target_signal_value) or pd.isna(curr_signal_value)) else abs(float(curr_signal_value) - float(target_signal_value))
        p_signal = _p3d_screen_signal_pvalue(prior_sig, curr_sig)

        exams_since_last_change = np.nan if last_threshold_change_end_idx is None else float(end_idx - last_threshold_change_end_idx)
        cadence_ok = bool(
            last_threshold_change_end_idx is None
            or exams_since_last_change >= float(min_recalibration_interval)
        )

        if policy_name == 'without_recalibration':
            signal_detected = False
            recalibrate = False
        elif policy_name == 'recalibrate_every_step':
            # Recalibrate at every scheduled decision point where the screen signal is defined.
            signal_detected = bool(pd.notna(drift))
            recalibrate = bool(signal_detected and cadence_ok)
        elif policy_name == 'recalibrate_significant_only':
            signal_detected = bool(pd.notna(p_signal) and p_signal < float(signal_p_threshold))
            recalibrate = bool(signal_detected and pd.notna(drift) and drift > 0.0 and cadence_ok)
        else:
            raise ValueError(f'Unknown policy_name: {policy_name}')

        new_thr = current_thr
        best_obj = np.nan
        if recalibrate:
            new_thr, best_obj = _p3d_choose_threshold_for_screen_rate(
                curr_w,
                ai_col=ai_col,
                target_rate=target_signal_value,
                max_candidates=max_threshold_candidates,
            )
            if pd.isna(new_thr):
                new_thr = current_thr

        threshold_changed = int(pd.notna(current_thr) and pd.notna(new_thr) and (abs(new_thr - current_thr) > 1e-12))
        if threshold_changed:
            last_threshold_change_end_idx = end_idx

        fut_perf = _p3d_rate_stats_from_threshold(future_w[ai_col].to_numpy(), future_w[cancer_col].to_numpy(), new_thr)
        decision_date = d.iloc[end_idx][date_col] if date_col in d.columns and end_idx < len(d) else pd.NaT

        rows.append({
            'decision_idx': i,
            'decision_end_exam_idx': end_idx,
            'decision_date': decision_date,
            'objective_metric': 'screen_flagging_rate',
            'signal_metric': 'screen_flagging_rate',
            'ai_system': ai_col,
            'prior_size': prior_size,
            'current_size': current_size,
            'future_size': future_size,
            'gap_size': gap_size,
            'step_size': step_size,
            'prior_start_idx': prior_start,
            'prior_end_idx': prior_end,
            'current_start_idx': current_start,
            'current_end_idx': end_idx,
            'future_start_idx': future_start,
            'future_end_idx': future_end,
            'target_sens': targets['sens'],
            'target_spec': targets['spec'],
            'target_ppv': targets['ppv'],
            'threshold_before': current_thr,
            'threshold_after': new_thr,
            'threshold_changed': threshold_changed,
            'drift_value': drift,
            'signal_mode': 'screen_theoretical',
            'signal_p_threshold': float(signal_p_threshold),
            'min_recalibration_interval': float(min_recalibration_interval),
            'exams_since_last_change': exams_since_last_change,
            'cadence_ok': int(cadence_ok),
            'p_value_signal': p_signal,
            'signal_detected': int(signal_detected),
            'recalibrated': int(recalibrate),
            'best_obj_value': best_obj,
            'prior_sens_before': prior_perf['sens'],
            'prior_spec_before': prior_perf['spec'],
            'current_sens_before': curr_perf['sens'],
            'current_spec_before': curr_perf['spec'],
            'delta_current_sens_to_target': curr_perf['sens'] - targets['sens'] if pd.notna(curr_perf['sens']) and pd.notna(targets['sens']) else np.nan,
            'delta_current_spec_to_target': curr_perf['spec'] - targets['spec'] if pd.notna(curr_perf['spec']) and pd.notna(targets['spec']) else np.nan,
            'target_signal_value': target_signal_value,
            'prior_signal_value': prior_sig.get('screen_flagging_rate', np.nan),
            'current_signal_value': curr_signal_value,
            'future_sens_after': fut_perf['sens'],
            'future_spec_after': fut_perf['spec'],
            'future_ppv_after': fut_perf['ppv'] if 'ppv' in fut_perf else np.nan,
            'future_n': fut_perf['n_total'],
        })

        current_thr = new_thr

    diary = pd.DataFrame(rows)
    if len(diary) == 0:
        return diary, targets, {'reason': 'empty_diary'}

    # Retrospective threshold-application intervals, matching Part 3D.6A.
    seg_start = diary['decision_end_exam_idx'].astype(int).to_numpy()
    seg_end = np.r_[seg_start[1:], [len(d)]]
    seg_sens, seg_spec, seg_ppv, seg_n = [], [], [], []
    for s, e, thr in zip(seg_start, seg_end, diary['threshold_after'].to_numpy(dtype=float)):
        seg = d.iloc[int(s):int(e)]
        perf = _p3d_rate_stats_from_threshold(seg[ai_col].to_numpy(), seg[cancer_col].to_numpy(), thr)
        seg_sens.append(perf['sens'])
        seg_spec.append(perf['spec'])
        seg_ppv.append(perf['ppv'] if 'ppv' in perf else np.nan)
        seg_n.append(perf['n_total'])

    diary['segment_start_idx'] = seg_start
    diary['segment_end_idx'] = seg_end
    diary['segment_n'] = np.asarray(seg_n, dtype=float)
    diary['segment_sens'] = np.asarray(seg_sens, dtype=float)
    diary['segment_spec'] = np.asarray(seg_spec, dtype=float)
    diary['segment_ppv'] = np.asarray(seg_ppv, dtype=float)
    return diary, targets, {'reason': 'ok'}


def _safe_wavg(values, weights):
    v = pd.to_numeric(values, errors='coerce')
    w = pd.to_numeric(weights, errors='coerce').fillna(0)
    mask = v.notna() & w.notna() & (w > 0)
    if mask.sum() == 0:
        return np.nan
    return float(np.average(v[mask], weights=w[mask]))


def _screen_metric_summary_from_pair_rows(pair_summary):
    metric_cols = [
        'target_sensitivity', 'target_specificity',
        'tw_sensitivity_recal', 'tw_specificity_recal',
        'tw_sensitivity_no_recal', 'tw_specificity_no_recal',
        'delta_sens_recal_minus_no', 'delta_spec_recal_minus_no',
    ]

    h_rows = []
    gcols_h = ['hospital', 'ai_system', 'signal_metric', 'policy', 'step_exams', 'gap_exams']
    for keys, g in pair_summary.groupby(gcols_h, dropna=False):
        row = {
            'hospital': keys[0],
            'ai_system': keys[1],
            'signal_metric': keys[2],
            'policy': keys[3],
            'step_exams': int(keys[4]),
            'gap_exams': int(keys[5]),
            'n_pairs': int(len(g)),
            'n_test_steps_total': int(pd.to_numeric(g['n_test_steps'], errors='coerce').fillna(0).sum()),
            'n_signal_detected_total': int(pd.to_numeric(g['n_signal_detected'], errors='coerce').fillna(0).sum()),
            'n_recalibrations_total': int(pd.to_numeric(g['n_recalibrations'], errors='coerce').fillna(0).sum()),
            'n_threshold_changes_total': int(pd.to_numeric(g['n_threshold_changes'], errors='coerce').fillna(0).sum()),
            'weights_total_exams': float(pd.to_numeric(g['weights_total_exams'], errors='coerce').fillna(0).sum()),
        }
        w = pd.to_numeric(g['weights_total_exams'], errors='coerce').fillna(0)
        for c in metric_cols:
            row[c] = _safe_wavg(g[c], w)
        row['mean_exams_between_recal'] = _safe_wavg(
            g['mean_exams_between_recal'],
            pd.to_numeric(g['n_threshold_changes'], errors='coerce').fillna(0),
        )
        row['median_exams_between_recal'] = _safe_wavg(
            g['median_exams_between_recal'],
            pd.to_numeric(g['n_threshold_changes'], errors='coerce').fillna(0),
        )
        h_rows.append(row)
    hospital_summary = pd.DataFrame(h_rows)

    o_rows = []
    gcols_o = ['ai_system', 'signal_metric', 'policy', 'step_exams', 'gap_exams']
    for keys, g in hospital_summary.groupby(gcols_o, dropna=False):
        row = {
            'ai_system': keys[0],
            'signal_metric': keys[1],
            'policy': keys[2],
            'step_exams': int(keys[3]),
            'gap_exams': int(keys[4]),
            'n_hospitals': int(g['hospital'].nunique()),
            'n_test_steps_total': int(pd.to_numeric(g['n_test_steps_total'], errors='coerce').fillna(0).sum()),
            'n_signal_detected_total': int(pd.to_numeric(g['n_signal_detected_total'], errors='coerce').fillna(0).sum()),
            'n_recalibrations_total': int(pd.to_numeric(g['n_recalibrations_total'], errors='coerce').fillna(0).sum()),
            'n_threshold_changes_total': int(pd.to_numeric(g['n_threshold_changes_total'], errors='coerce').fillna(0).sum()),
            'weights_total_exams': float(pd.to_numeric(g['weights_total_exams'], errors='coerce').fillna(0).sum()),
        }
        w = pd.to_numeric(g['weights_total_exams'], errors='coerce').fillna(0)
        for c in metric_cols:
            row[c] = _safe_wavg(g[c], w)
        row['mean_exams_between_recal'] = _safe_wavg(
            g['mean_exams_between_recal'],
            pd.to_numeric(g['n_threshold_changes_total'], errors='coerce').fillna(0),
        )
        row['median_exams_between_recal'] = _safe_wavg(
            g['median_exams_between_recal'],
            pd.to_numeric(g['n_threshold_changes_total'], errors='coerce').fillna(0),
        )
        o_rows.append(row)
    overall_summary = pd.DataFrame(o_rows)

    # Professor-preferred aggregation: unweighted mean of hospital-level metrics.
    u_rows = []
    for keys, g in hospital_summary.groupby(gcols_o, dropna=False):
        metric_mask = pd.to_numeric(g['tw_sensitivity_recal'], errors='coerce').notna()
        valid = g.loc[metric_mask].copy()
        row = {
            'ai_system': keys[0],
            'signal_metric': keys[1],
            'policy': keys[2],
            'step_exams': int(keys[3]),
            'gap_exams': int(keys[4]),
            'n_hospitals': int(valid['hospital'].nunique()),
            'n_test_steps_total': int(pd.to_numeric(valid['n_test_steps_total'], errors='coerce').fillna(0).sum()),
            'n_signal_detected_total': int(pd.to_numeric(valid['n_signal_detected_total'], errors='coerce').fillna(0).sum()),
            'n_recalibrations_total': int(pd.to_numeric(valid['n_recalibrations_total'], errors='coerce').fillna(0).sum()),
            'n_threshold_changes_total': int(pd.to_numeric(valid['n_threshold_changes_total'], errors='coerce').fillna(0).sum()),
            'weights_total_exams': float(pd.to_numeric(valid['weights_total_exams'], errors='coerce').fillna(0).sum()),
            'aggregation_method': 'unweighted_mean_of_hospital_level_metrics',
        }
        for c in ['target_sensitivity', 'target_specificity', 'tw_sensitivity_recal', 'tw_specificity_recal']:
            row[c] = float(pd.to_numeric(valid[c], errors='coerce').mean()) if len(valid) else np.nan
        row['mean_exams_between_recal'] = float(pd.to_numeric(valid['mean_exams_between_recal'], errors='coerce').mean()) if len(valid) else np.nan
        row['median_exams_between_recal'] = float(pd.to_numeric(valid['median_exams_between_recal'], errors='coerce').mean()) if len(valid) else np.nan
        u_rows.append(row)
    unweighted_summary = pd.DataFrame(u_rows)

    return hospital_summary, overall_summary, unweighted_summary


policy_order = ['without_recalibration', 'recalibrate_every_step', 'recalibrate_significant_only']
policy_label = {
    'without_recalibration': 'No recalibration',
    'recalibrate_every_step': 'Recalibrate every step',
    'recalibrate_significant_only': 'Recalibrate if significant',
}

# Run the theoretical screen signal policy comparison.
rows = []
all_hospitals = sorted(df_part3[cols['hospital']].dropna().unique())
hospital_filter = globals().get('P3D_POLICY_HOSPITALS', None)
if hospital_filter is not None:
    all_hospitals = [h for h in all_hospitals if h in set(hospital_filter)]

for hospital in all_hospitals:
    d_h = df_part3[df_part3[cols['hospital']] == hospital].copy()
    if len(d_h) == 0:
        continue

    if cols['manufacturer'] is not None and cols['manufacturer'] in d_h.columns:
        manufacturers = sorted(d_h[cols['manufacturer']].dropna().unique())
        if len(manufacturers) == 0:
            manufacturers = [None]
    else:
        manufacturers = [None]

    mfr_filter = globals().get('P3D_POLICY_MANUFACTURERS', None)
    if mfr_filter is not None:
        manufacturers = [m for m in manufacturers if m in set(mfr_filter)]

    for manufacturer in manufacturers:
        pair_df = _p3d_prepare_pair_df(df_part3, cols, hospital, manufacturer)
        if len(pair_df) == 0:
            continue
        pair_df = _p3d_add_rad_flag(pair_df, rad_mode=rad_mode)

        for ai in ai_systems:
            for policy_name in policy_order:
                diary, targets, meta = _p3d_run_diary_screen_flagging_rate(
                    df_pair=pair_df,
                    ai_col=ai,
                    cancer_col=cols['cancer'],
                    date_col=cols['date'],
                    prior_size=prior_size,
                    current_size=current_size,
                    future_size=future_size,
                    gap_size=gap_size,
                    step_size=step_size,
                    policy_name=policy_name,
                    signal_p_threshold=signal_p_thr,
                    min_recalibration_interval=min_recal_interval,
                    max_threshold_candidates=max_candidates,
                    max_points=max_points,
                )

                if diary is None or len(diary) == 0:
                    rows.append({
                        'hospital': hospital,
                        'manufacturer': manufacturer if manufacturer is not None else 'ALL',
                        'ai_system': ai,
                        'signal_metric': 'screen_flagging_rate',
                        'policy': policy_name,
                        'status': meta.get('reason', 'no_data') if isinstance(meta, dict) else 'no_data',
                        'step_exams': step_size,
                        'gap_exams': gap_size,
                        'n_test_steps': 0,
                        'n_signal_detected': 0,
                        'n_recalibrations': 0,
                        'n_threshold_changes': 0,
                        'mean_exams_between_recal': np.nan,
                        'median_exams_between_recal': np.nan,
                        'weights_total_exams': 0.0,
                        'target_sensitivity': np.nan,
                        'target_specificity': np.nan,
                        'tw_sensitivity_recal': np.nan,
                        'tw_specificity_recal': np.nan,
                        'tw_sensitivity_no_recal': np.nan,
                        'tw_specificity_no_recal': np.nan,
                        'delta_sens_recal_minus_no': np.nan,
                        'delta_spec_recal_minus_no': np.nan,
                    })
                    continue

                w = pd.to_numeric(diary['segment_n'], errors='coerce').fillna(0)
                tw_sens_recal = _p3d_time_weighted_mean(diary['segment_sens'], w)
                tw_spec_recal = _p3d_time_weighted_mean(diary['segment_spec'], w)

                changes = pd.to_numeric(
                    diary.loc[pd.to_numeric(diary['threshold_changed'], errors='coerce').fillna(0) > 0, 'decision_end_exam_idx'],
                    errors='coerce'
                ).dropna().to_numpy(dtype=float)
                if len(changes) >= 2:
                    gaps_recal = np.diff(changes)
                    mean_between_recal = float(np.mean(gaps_recal))
                    median_between_recal = float(np.median(gaps_recal))
                else:
                    mean_between_recal = np.nan
                    median_between_recal = np.nan

                rows.append({
                    'hospital': hospital,
                    'manufacturer': manufacturer if manufacturer is not None else 'ALL',
                    'ai_system': ai,
                    'signal_metric': 'screen_flagging_rate',
                    'policy': policy_name,
                    'status': meta.get('reason', 'ok') if isinstance(meta, dict) else 'ok',
                    'step_exams': step_size,
                    'gap_exams': gap_size,
                    'n_test_steps': int(len(diary)),
                    'n_signal_detected': int(pd.to_numeric(diary['signal_detected'], errors='coerce').fillna(0).sum()),
                    'n_recalibrations': int(pd.to_numeric(diary['recalibrated'], errors='coerce').fillna(0).sum()),
                    'n_threshold_changes': int(pd.to_numeric(diary['threshold_changed'], errors='coerce').fillna(0).sum()),
                    'mean_exams_between_recal': mean_between_recal,
                    'median_exams_between_recal': median_between_recal,
                    'weights_total_exams': float(w.sum()),
                    'target_sensitivity': float(pd.to_numeric(diary['target_sens'], errors='coerce').iloc[0]),
                    'target_specificity': float(pd.to_numeric(diary['target_spec'], errors='coerce').iloc[0]),
                    'tw_sensitivity_recal': tw_sens_recal,
                    'tw_specificity_recal': tw_spec_recal,
                    'tw_sensitivity_no_recal': np.nan,
                    'tw_specificity_no_recal': np.nan,
                    'delta_sens_recal_minus_no': np.nan,
                    'delta_spec_recal_minus_no': np.nan,
                })

p3d_policy3_pair_summary_screen_metric = pd.DataFrame(rows)
p3d_policy3_hospital_summary_screen_metric, p3d_policy3_overall_summary_screen_metric, p3d_policy3_overall_summary_screen_metric_unweighted = _screen_metric_summary_from_pair_rows(
    p3d_policy3_pair_summary_screen_metric
)

# Save the screen-metric summaries separately for auditability.
pair_screen_csv = screen_out_dir / 'p3d_policy3_pair_summary_screen_metric.csv'
hospital_screen_csv = screen_out_dir / 'p3d_policy3_hospital_summary_screen_metric.csv'
overall_screen_csv = screen_out_dir / 'p3d_policy3_overall_summary_screen_metric.csv'
overall_screen_unweighted_csv = screen_out_dir / 'p3d_policy3_overall_summary_screen_metric_unweighted_hospital_mean.csv'

p3d_policy3_pair_summary_screen_metric.sort_values(['hospital', 'manufacturer', 'ai_system', 'policy']).to_csv(pair_screen_csv, index=False)
p3d_policy3_hospital_summary_screen_metric.sort_values(['hospital', 'ai_system', 'policy']).to_csv(hospital_screen_csv, index=False)
p3d_policy3_overall_summary_screen_metric.sort_values(['ai_system', 'policy']).to_csv(overall_screen_csv, index=False)
p3d_policy3_overall_summary_screen_metric_unweighted.sort_values(['ai_system', 'policy']).to_csv(overall_screen_unweighted_csv, index=False)

# Load or create the standard four-metric unweighted summary, then append the theoretical screen metric.
standard_unweighted_csv = base_out_dir / 'p3d_policy3_overall_summary_unweighted_hospital_mean.csv'
if standard_unweighted_csv.exists():
    standard_unweighted = pd.read_csv(standard_unweighted_csv)
elif 'p3d_policy3_overall_summary_unweighted_hospital_mean' in globals():
    standard_unweighted = p3d_policy3_overall_summary_unweighted_hospital_mean.copy()
else:
    raise RuntimeError('Missing standard unweighted summary. Run the unweighted aggregation cell first.')

common_cols = [
    'ai_system', 'signal_metric', 'policy', 'aggregation_method', 'step_exams', 'gap_exams',
    'n_hospitals', 'n_test_steps_total', 'n_signal_detected_total', 'n_recalibrations_total',
    'n_threshold_changes_total', 'weights_total_exams', 'target_sensitivity', 'target_specificity',
    'tw_sensitivity_recal', 'tw_specificity_recal', 'mean_exams_between_recal', 'median_exams_between_recal'
]

screen_unweighted = p3d_policy3_overall_summary_screen_metric_unweighted.copy()
screen_unweighted['aggregation_method'] = 'unweighted_mean_of_hospital_level_metrics'
for c in common_cols:
    if c not in screen_unweighted.columns:
        screen_unweighted[c] = np.nan
for c in common_cols:
    if c not in standard_unweighted.columns:
        standard_unweighted[c] = np.nan

p3d_policy3_overall_summary_unweighted_hospital_mean_with_screen_theoretical = pd.concat(
    [standard_unweighted[common_cols], screen_unweighted[common_cols]],
    ignore_index=True,
)
combined_csv = screen_out_dir / 'p3d_policy3_overall_summary_unweighted_hospital_mean_with_screen_theoretical.csv'
p3d_policy3_overall_summary_unweighted_hospital_mean_with_screen_theoretical.to_csv(combined_csv, index=False)

# Also copy the standard weighted/screen weighted summaries into this folder for a complete audit trail.
if (base_out_dir / 'p3d_policy3_overall_summary.csv').exists():
    pd.read_csv(base_out_dir / 'p3d_policy3_overall_summary.csv').to_csv(screen_out_dir / 'p3d_policy3_overall_summary.csv', index=False)
p3d_policy3_overall_summary_screen_metric.to_csv(screen_out_dir / 'p3d_policy3_overall_summary_screen_metric.csv', index=False)

# Plot the manuscript-style sensitivity and specificity figures from the combined unweighted summary.
def _plot_policy3_unweighted_with_screen(d, col_name, target_col, target_label, y_label, out_name, y_min, y_floor_max):
    d = d.copy()
    policy_order = ['without_recalibration', 'recalibrate_every_step', 'recalibrate_significant_only']
    policy_label = {
        'without_recalibration': 'No recalibration',
        'recalibrate_every_step': 'Recalibrate every step',
        'recalibrate_significant_only': 'Recalibrate if significant',
    }
    colors = {
        'without_recalibration': '#8da0cb',
        'recalibrate_every_step': '#fc8d62',
        'recalibrate_significant_only': '#66c2a5',
    }
    metric_order = [
        'flagging_rate',
        'relative_flagging_rate',
        'p_ai_given_rad',
        'p_rad_given_ai',
        'screen_flagging_rate',
    ]
    present = set(d['signal_metric'].dropna().astype(str).unique())
    metric_order = [m for m in metric_order if m in present] + [m for m in sorted(present) if m not in metric_order]
    ai_order = sorted(d['ai_system'].dropna().astype(str).unique())
    ai_label_map = {ai: f'AI-{idx + 1}' for idx, ai in enumerate(ai_order)}
    metric_label = {
        'flagging_rate': 'Flagging rate',
        'relative_flagging_rate': 'Relative\nflagging rate',
        'p_ai_given_rad': 'P(AI flagged |\nradiologist flagged)',
        'p_rad_given_ai': 'P(radiologist flagged |\nAI flagged)',
        'screen_flagging_rate': 'Screen-detected cancer\nflagging rate',
    }

    row_order = [(ai, sm) for sm in metric_order for ai in ai_order]
    x = np.arange(len(row_order))
    width = 0.24
    offsets = {
        'without_recalibration': -width,
        'recalibrate_every_step': 0.0,
        'recalibrate_significant_only': width,
    }

    def _extract_val(df, ai, sm, pol, col):
        z = df[(df['ai_system'].astype(str) == str(ai)) & (df['signal_metric'].astype(str) == str(sm)) & (df['policy'] == pol)]
        if len(z) == 0:
            return np.nan
        return float(pd.to_numeric(z[col], errors='coerce').iloc[0])

    fig, ax = plt.subplots(figsize=(18, 8), constrained_layout=False)
    y_all = []
    for pol in policy_order:
        y = [_extract_val(d, ai, sm, pol, col_name) for ai, sm in row_order]
        y_all.extend([v for v in y if pd.notna(v)])
        ax.bar(x + offsets[pol], y, width=width, label=policy_label[pol], color=colors[pol])

    target_val = float(pd.to_numeric(d[target_col], errors='coerce').dropna().mean())
    ax.axhline(target_val, color='black', ls=':', lw=2.0, label=target_label)

    ax.set_title(f'Three-policy comparison across hospitals: {y_label}', fontweight='bold', fontsize=16, pad=12)
    ax.set_ylabel(y_label, fontsize=12)
    ax.set_xticks(x)
    ax.set_xticklabels([ai_label_map.get(ai, str(ai)) for ai, _ in row_order], fontsize=11)
    ax.tick_params(axis='y', labelsize=11)
    ax.grid(axis='y', alpha=0.25)

    for group_idx, sm in enumerate(metric_order):
        start_pos = group_idx * len(ai_order)
        end_pos = start_pos + len(ai_order) - 1
        center_pos = (start_pos + end_pos) / 2
        ax.text(
            center_pos,
            -0.17,
            metric_label.get(sm, str(sm).replace('_', ' ').title()),
            transform=ax.get_xaxis_transform(),
            ha='center',
            va='top',
            fontsize=11,
        )
        if group_idx > 0:
            ax.axvline(start_pos - 0.5, color='0.85', lw=1.0, zorder=0)

    ax.set_xlabel('')
    if y_min is not None and y_floor_max is not None:
        vmax = max(y_all) if len(y_all) > 0 else y_floor_max
        ax.set_ylim(y_min, max(y_floor_max, float(vmax) + 1.0))

    step_ex = int(pd.to_numeric(d['step_exams'], errors='coerce').dropna().iloc[0])
    gap_ex = int(pd.to_numeric(d['gap_exams'], errors='coerce').dropna().iloc[0])
    handles, labels = ax.get_legend_handles_labels()
    note_handles = [
        Line2D([], [], color='none', label=f'Test cadence: every {step_ex:,} exams; gap = {gap_ex:,} exams'),
        Line2D([], [], color='none', label='Overall: unweighted mean of hospital-level metrics'),
        Line2D([], [], color='none', label='Screen-detected cancer signal is theoretical'),
        Line2D([], [], color='none', label='and uses delayed outcome information'),
    ]
    handles = handles + note_handles
    labels = labels + [h.get_label() for h in note_handles]
    ax.legend(
        handles,
        labels,
        loc='center left',
        bbox_to_anchor=(1.01, 0.5),
        ncol=1,
        fontsize=10,
        title='Policies and figure notes',
        title_fontsize=11,
        frameon=True,
    )

    fig.subplots_adjust(top=0.88, bottom=0.24, left=0.07, right=0.76)
    out_path = screen_out_dir / out_name
    fig.savefig(out_path, dpi=220, bbox_inches='tight')
    plt.show()
    return out_path

sens_png = _plot_policy3_unweighted_with_screen(
    p3d_policy3_overall_summary_unweighted_hospital_mean_with_screen_theoretical,
    col_name='tw_sensitivity_recal',
    target_col='target_sensitivity',
    target_label='Target sensitivity',
    y_label='Time-weighted sensitivity (%)',
    out_name='p3d_policy3_overall_unweighted_hospital_mean_sensitivity_with_screen_theoretical.png',
    y_min=34,
    y_floor_max=50,
)
spec_png = _plot_policy3_unweighted_with_screen(
    p3d_policy3_overall_summary_unweighted_hospital_mean_with_screen_theoretical,
    col_name='tw_specificity_recal',
    target_col='target_specificity',
    target_label='Target specificity',
    y_label='Time-weighted specificity (%)',
    out_name='p3d_policy3_overall_unweighted_hospital_mean_specificity_with_screen_theoretical.png',
    y_min=88,
    y_floor_max=98,
)

print('Saved screen-metric pair summary:', pair_screen_csv)
print('Saved screen-metric hospital summary:', hospital_screen_csv)
print('Saved screen-metric overall summary:', overall_screen_csv)
print('Saved screen-metric unweighted overall summary:', overall_screen_unweighted_csv)
print('Saved combined plotting table:', combined_csv)
print('Saved figure:', sens_png)
print('Saved figure:', spec_png)
print('Image AI label mapping:', ', '.join(f'AI-{idx + 1} = {ai.title()}' for idx, ai in enumerate(sorted(ai_systems))))


In [ ]:
# Part 3D.6F - Reproducible theoretical screen-detected signal figures excluding Danderyds Hospital
#
# This cell repeats the Part 3D.6E plotting table, but recomputes the overall
# unweighted hospital-level mean after excluding Danderyds Hospital. The source
# hospital-level tables are filtered first, then averaged. This means the image
# does not contain Danderyds Hospital.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.lines import Line2D

base_out_dir = Path(globals().get(
    'P3D_OUTPUT_DIR',
    Path('outputs/2025_12_29_counterfactual_manufacturer_swap_setup/part3d_oracle_recalibration')
))
screen_base_out_dir = base_out_dir.parent / 'part3d_oracle_recalibration_with_screen_theoretical'
no_danderyd_out_dir = screen_base_out_dir / 'excluding_danderyd'
no_danderyd_out_dir.mkdir(parents=True, exist_ok=True)

DANDERYD_EXCLUSION_PATTERN = 'danderyd'
DANDERYD_EXCLUSION_LABEL = 'Danderyds Hospital'


def _load_table_from_memory_or_csv(var_name, csv_path):
    obj = globals().get(var_name)
    if isinstance(obj, pd.DataFrame) and len(obj) > 0:
        return obj.copy()
    csv_path = Path(csv_path)
    if csv_path.exists():
        return pd.read_csv(csv_path)
    raise RuntimeError(f'Missing {var_name}. Run the upstream Part 3D cells first, or create {csv_path}.')


standard_hospital_summary = _load_table_from_memory_or_csv(
    'p3d_policy3_hospital_summary',
    base_out_dir / 'p3d_policy3_hospital_summary.csv',
)
screen_hospital_summary = _load_table_from_memory_or_csv(
    'p3d_policy3_hospital_summary_screen_metric',
    screen_base_out_dir / 'p3d_policy3_hospital_summary_screen_metric.csv',
)


def _exclude_danderyd_rows(d):
    d = d.copy()
    if 'hospital' not in d.columns:
        raise RuntimeError('Expected a hospital column before excluding Danderyds Hospital.')
    mask = d['hospital'].astype(str).str.contains(DANDERYD_EXCLUSION_PATTERN, case=False, na=False)
    excluded = sorted(d.loc[mask, 'hospital'].dropna().astype(str).unique())
    included = d.loc[~mask].copy()
    return included, excluded


standard_hospital_no_danderyd, standard_excluded = _exclude_danderyd_rows(standard_hospital_summary)
screen_hospital_no_danderyd, screen_excluded = _exclude_danderyd_rows(screen_hospital_summary)
excluded_hospitals = sorted(set(standard_excluded) | set(screen_excluded))

if len(excluded_hospitals) == 0:
    print('Warning: no hospital matching Danderyd was found in the source tables.')
else:
    print('Excluded hospital(s):', ', '.join(excluded_hospitals))

print('Standard metric hospital rows retained:', len(standard_hospital_no_danderyd))
print('Screen metric hospital rows retained:', len(screen_hospital_no_danderyd))

policy_order = ['without_recalibration', 'recalibrate_every_step', 'recalibrate_significant_only']
policy_label = {
    'without_recalibration': 'No recalibration',
    'recalibrate_every_step': 'Recalibrate every step',
    'recalibrate_significant_only': 'Recalibrate if significant',
}
colors = {
    'without_recalibration': '#8da0cb',
    'recalibrate_every_step': '#fc8d62',
    'recalibrate_significant_only': '#66c2a5',
}
metric_order = [
    'flagging_rate',
    'relative_flagging_rate',
    'p_ai_given_rad',
    'p_rad_given_ai',
    'screen_flagging_rate',
]
metric_label = {
    'flagging_rate': 'Flagging rate',
    'relative_flagging_rate': 'Relative\nflagging rate',
    'p_ai_given_rad': 'P(AI flagged |\nradiologist flagged)',
    'p_rad_given_ai': 'P(radiologist flagged |\nAI flagged)',
    'screen_flagging_rate': 'Screen-detected cancer\nflagging rate',
}


def _as_int_or_nan(v):
    return np.nan if pd.isna(v) else int(v)


def _unweighted_overall_from_hospital_rows(hospital_summary, aggregation_method):
    h = hospital_summary.copy()
    h = h[h['policy'].isin(policy_order)].copy()
    if len(h) == 0:
        return pd.DataFrame()

    gcols = ['ai_system', 'signal_metric', 'policy', 'step_exams', 'gap_exams']
    metric_cols = [
        'target_sensitivity', 'target_specificity',
        'tw_sensitivity_recal', 'tw_specificity_recal',
        'mean_exams_between_recal', 'median_exams_between_recal',
    ]
    count_cols = [
        'n_test_steps_total', 'n_signal_detected_total', 'n_recalibrations_total',
        'n_threshold_changes_total', 'weights_total_exams',
    ]

    rows = []
    for keys, g in h.groupby(gcols, dropna=False):
        valid = g.loc[pd.to_numeric(g['tw_sensitivity_recal'], errors='coerce').notna()].copy()
        row = {
            'ai_system': keys[0],
            'signal_metric': keys[1],
            'policy': keys[2],
            'step_exams': _as_int_or_nan(keys[3]),
            'gap_exams': _as_int_or_nan(keys[4]),
            'n_hospitals': int(valid['hospital'].nunique()) if len(valid) else 0,
            'aggregation_method': aggregation_method,
        }
        for c in count_cols:
            if c in valid.columns:
                row[c] = float(pd.to_numeric(valid[c], errors='coerce').fillna(0).sum())
        for c in metric_cols:
            if c in valid.columns:
                row[c] = float(pd.to_numeric(valid[c], errors='coerce').mean()) if len(valid) else np.nan
        rows.append(row)

    return pd.DataFrame(rows)


standard_unweighted_no_danderyd = _unweighted_overall_from_hospital_rows(
    standard_hospital_no_danderyd,
    'unweighted_mean_of_hospital_level_metrics_excluding_danderyd',
)
screen_unweighted_no_danderyd = _unweighted_overall_from_hospital_rows(
    screen_hospital_no_danderyd,
    'unweighted_mean_of_hospital_level_metrics_excluding_danderyd',
)

common_cols = sorted(set(standard_unweighted_no_danderyd.columns) | set(screen_unweighted_no_danderyd.columns))
p3d_policy3_overall_summary_unweighted_hospital_mean_with_screen_theoretical_no_danderyd = pd.concat(
    [standard_unweighted_no_danderyd.reindex(columns=common_cols), screen_unweighted_no_danderyd.reindex(columns=common_cols)],
    ignore_index=True,
)

standard_hospital_no_danderyd_csv = no_danderyd_out_dir / 'p3d_policy3_hospital_summary_standard_metrics_no_danderyd.csv'
screen_hospital_no_danderyd_csv = no_danderyd_out_dir / 'p3d_policy3_hospital_summary_screen_metric_no_danderyd.csv'
combined_no_danderyd_csv = no_danderyd_out_dir / 'p3d_policy3_overall_summary_unweighted_hospital_mean_with_screen_theoretical_no_danderyd.csv'

standard_hospital_no_danderyd.to_csv(standard_hospital_no_danderyd_csv, index=False)
screen_hospital_no_danderyd.to_csv(screen_hospital_no_danderyd_csv, index=False)
p3d_policy3_overall_summary_unweighted_hospital_mean_with_screen_theoretical_no_danderyd.to_csv(combined_no_danderyd_csv, index=False)


def _plot_policy3_unweighted_with_screen_no_danderyd(d, col_name, target_col, target_label, y_label, out_name, preferred_y_min, preferred_y_max):
    d = d.copy()
    d = d[d['policy'].isin(policy_order)].copy()
    if len(d) == 0:
        raise RuntimeError('No rows available for plotting after excluding Danderyds Hospital.')

    present_metrics = set(d['signal_metric'].dropna().astype(str).unique())
    metrics = [m for m in metric_order if m in present_metrics] + [m for m in sorted(present_metrics) if m not in metric_order]
    ai_order = sorted(d['ai_system'].dropna().astype(str).unique())
    ai_label_map = {ai: f'AI-{idx + 1}' for idx, ai in enumerate(ai_order)}

    row_order = [(ai, sm) for sm in metrics for ai in ai_order]
    x = np.arange(len(row_order))
    width = 0.24
    offsets = {
        'without_recalibration': -width,
        'recalibrate_every_step': 0.0,
        'recalibrate_significant_only': width,
    }

    def _extract_val(df, ai, sm, pol, col):
        z = df[(df['ai_system'].astype(str) == str(ai)) & (df['signal_metric'].astype(str) == str(sm)) & (df['policy'] == pol)]
        if len(z) == 0:
            return np.nan
        return float(pd.to_numeric(z[col], errors='coerce').iloc[0])

    fig, ax = plt.subplots(figsize=(18, 8), constrained_layout=False)
    y_all = []
    for pol in policy_order:
        y = [_extract_val(d, ai, sm, pol, col_name) for ai, sm in row_order]
        y_all.extend([v for v in y if pd.notna(v)])
        ax.bar(x + offsets[pol], y, width=width, label=policy_label[pol], color=colors[pol])

    target_vals = pd.to_numeric(d[target_col], errors='coerce').dropna()
    target_val = float(target_vals.mean()) if len(target_vals) else np.nan
    if pd.notna(target_val):
        ax.axhline(target_val, color='black', ls=':', lw=2.0, label=target_label)

    ax.set_title(
        f'Three-policy comparison across hospitals excluding {DANDERYD_EXCLUSION_LABEL}: {y_label}',
        fontweight='bold', fontsize=16, pad=12,
    )
    ax.set_ylabel(y_label, fontsize=12)
    ax.set_xticks(x)
    ax.set_xticklabels([ai_label_map.get(ai, str(ai)) for ai, _ in row_order], fontsize=11)
    ax.tick_params(axis='y', labelsize=11)
    ax.grid(axis='y', alpha=0.25)

    for group_idx, sm in enumerate(metrics):
        start_pos = group_idx * len(ai_order)
        end_pos = start_pos + len(ai_order) - 1
        center_pos = (start_pos + end_pos) / 2
        ax.text(
            center_pos,
            -0.17,
            metric_label.get(sm, str(sm).replace('_', ' ').title()),
            transform=ax.get_xaxis_transform(),
            ha='center',
            va='top',
            fontsize=11,
        )
        if group_idx > 0:
            ax.axvline(start_pos - 0.5, color='0.85', lw=1.0, zorder=0)

    ax.set_xlabel('')
    finite_vals = [v for v in y_all if pd.notna(v)]
    if pd.notna(target_val):
        finite_vals.append(target_val)
    if len(finite_vals) > 0:
        lower = min(float(preferred_y_min), float(np.nanmin(finite_vals)) - 1.0)
        upper = max(float(preferred_y_max), float(np.nanmax(finite_vals)) + 1.0)
        ax.set_ylim(max(0.0, lower), min(100.0, upper))

    step_ex = int(pd.to_numeric(d['step_exams'], errors='coerce').dropna().iloc[0])
    gap_ex = int(pd.to_numeric(d['gap_exams'], errors='coerce').dropna().iloc[0])
    handles, labels = ax.get_legend_handles_labels()
    note_handles = [
        Line2D([], [], color='none', label=f'Test cadence: every {step_ex:,} exams; gap = {gap_ex:,} exams'),
        Line2D([], [], color='none', label='Overall: unweighted mean of hospital-level metrics'),
        Line2D([], [], color='none', label=f'{DANDERYD_EXCLUSION_LABEL} excluded'),
        Line2D([], [], color='none', label='Screen-detected cancer signal is theoretical'),
        Line2D([], [], color='none', label='and uses delayed outcome information'),
    ]
    handles = handles + note_handles
    labels = labels + [h.get_label() for h in note_handles]
    ax.legend(
        handles,
        labels,
        loc='center left',
        bbox_to_anchor=(1.01, 0.5),
        ncol=1,
        fontsize=10,
        title='Policies and figure notes',
        title_fontsize=11,
        frameon=True,
    )

    fig.subplots_adjust(top=0.88, bottom=0.24, left=0.07, right=0.76)
    out_path = no_danderyd_out_dir / out_name
    fig.savefig(out_path, dpi=220, bbox_inches='tight')
    plt.show()
    return out_path


sens_png_no_danderyd = _plot_policy3_unweighted_with_screen_no_danderyd(
    p3d_policy3_overall_summary_unweighted_hospital_mean_with_screen_theoretical_no_danderyd,
    col_name='tw_sensitivity_recal',
    target_col='target_sensitivity',
    target_label='Target sensitivity',
    y_label='Time-weighted sensitivity (%)',
    out_name='p3d_policy3_overall_unweighted_hospital_mean_sensitivity_with_screen_theoretical_no_danderyd.png',
    preferred_y_min=34,
    preferred_y_max=50,
)
spec_png_no_danderyd = _plot_policy3_unweighted_with_screen_no_danderyd(
    p3d_policy3_overall_summary_unweighted_hospital_mean_with_screen_theoretical_no_danderyd,
    col_name='tw_specificity_recal',
    target_col='target_specificity',
    target_label='Target specificity',
    y_label='Time-weighted specificity (%)',
    out_name='p3d_policy3_overall_unweighted_hospital_mean_specificity_with_screen_theoretical_no_danderyd.png',
    preferred_y_min=88,
    preferred_y_max=98,
)

ai_order_for_print = sorted(
    p3d_policy3_overall_summary_unweighted_hospital_mean_with_screen_theoretical_no_danderyd['ai_system'].dropna().astype(str).unique()
)
print('Image AI label mapping:', ', '.join(f'AI-{idx + 1} = {name}' for idx, name in enumerate(ai_order_for_print)))
print('Saved standard hospital table without Danderyds:', standard_hospital_no_danderyd_csv)
print('Saved screen hospital table without Danderyds:', screen_hospital_no_danderyd_csv)
print('Saved combined plotting table without Danderyds:', combined_no_danderyd_csv)
print('Saved sensitivity figure without Danderyds:', sens_png_no_danderyd)
print('Saved specificity figure without Danderyds:', spec_png_no_danderyd)

display(p3d_policy3_overall_summary_unweighted_hospital_mean_with_screen_theoretical_no_danderyd.head(20))


In [ ]:
# Part 3D.6G - Theoretical screen-detected signal figures by hospital-manufacturer pair
#
# This cell makes the same style of figures as Part 3D.6E, but without overall
# aggregation. Each output image contains one hospital-manufacturer pair and the
# title/legend state exactly which hospital and manufacturer are being plotted.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.lines import Line2D

base_out_dir = Path(globals().get(
    'P3D_OUTPUT_DIR',
    Path('outputs/2025_12_29_counterfactual_manufacturer_swap_setup/part3d_oracle_recalibration')
))
screen_base_out_dir = base_out_dir.parent / 'part3d_oracle_recalibration_with_screen_theoretical'
pair_plot_out_dir = screen_base_out_dir / 'hospital_manufacturer_pair_figures'
pair_plot_out_dir.mkdir(parents=True, exist_ok=True)


def _load_table_from_memory_or_csv(var_name, csv_path):
    obj = globals().get(var_name)
    if isinstance(obj, pd.DataFrame) and len(obj) > 0:
        return obj.copy()
    csv_path = Path(csv_path)
    if csv_path.exists():
        return pd.read_csv(csv_path)
    raise RuntimeError(f'Missing {var_name}. Run the upstream Part 3D cells first, or create {csv_path}.')


standard_pair_summary = _load_table_from_memory_or_csv(
    'p3d_policy3_pair_summary',
    base_out_dir / 'p3d_policy3_pair_summary.csv',
)
screen_pair_summary = _load_table_from_memory_or_csv(
    'p3d_policy3_pair_summary_screen_metric',
    screen_base_out_dir / 'p3d_policy3_pair_summary_screen_metric.csv',
)

for d in [standard_pair_summary, screen_pair_summary]:
    if 'manufacturer' not in d.columns:
        d['manufacturer'] = 'ALL'

common_cols = sorted(set(standard_pair_summary.columns) | set(screen_pair_summary.columns))
p3d_policy3_pair_summary_with_screen_theoretical = pd.concat(
    [standard_pair_summary.reindex(columns=common_cols), screen_pair_summary.reindex(columns=common_cols)],
    ignore_index=True,
    sort=False,
)

pair_combined_csv = pair_plot_out_dir / 'p3d_policy3_pair_summary_with_screen_theoretical.csv'
p3d_policy3_pair_summary_with_screen_theoretical.to_csv(pair_combined_csv, index=False)

# Remove stale pair PNGs from previous runs so empty plots do not remain in the folder.
for old_png in pair_plot_out_dir.glob('p3d_policy3_pair_*_with_screen_theoretical_*.png'):
    old_png.unlink()

policy_order = ['without_recalibration', 'recalibrate_every_step', 'recalibrate_significant_only']
policy_label = {
    'without_recalibration': 'No recalibration',
    'recalibrate_every_step': 'Recalibrate every step',
    'recalibrate_significant_only': 'Recalibrate if significant',
}
colors = {
    'without_recalibration': '#8da0cb',
    'recalibrate_every_step': '#fc8d62',
    'recalibrate_significant_only': '#66c2a5',
}
metric_order = [
    'flagging_rate',
    'relative_flagging_rate',
    'p_ai_given_rad',
    'p_rad_given_ai',
    'screen_flagging_rate',
]
metric_label = {
    'flagging_rate': 'Flagging rate',
    'relative_flagging_rate': 'Relative\nflagging rate',
    'p_ai_given_rad': 'P(AI flagged |\nradiologist flagged)',
    'p_rad_given_ai': 'P(radiologist flagged |\nAI flagged)',
    'screen_flagging_rate': 'Screen-detected cancer\nflagging rate',
}


def _slug(value):
    text = str(value) if pd.notna(value) else 'missing'
    slug = ''.join(ch.lower() if ch.isalnum() else '_' for ch in text).strip('_')
    while '__' in slug:
        slug = slug.replace('__', '_')
    return slug or 'missing'


def _plot_policy3_pair_with_screen(d_pair, col_name, target_col, target_label, y_label, hospital, manufacturer, out_name, preferred_y_min, preferred_y_max):
    d_pair = d_pair.copy()
    d_pair = d_pair[d_pair['policy'].isin(policy_order)].copy()
    if len(d_pair) == 0:
        return None

    finite_bar_values = pd.to_numeric(d_pair[col_name], errors='coerce').dropna()
    if len(finite_bar_values) == 0:
        print(f'[skip empty] {hospital} | {manufacturer}: no finite values for {y_label}')
        return None

    present_metrics = set(d_pair['signal_metric'].dropna().astype(str).unique())
    metrics = [m for m in metric_order if m in present_metrics] + [m for m in sorted(present_metrics) if m not in metric_order]
    ai_order = sorted(d_pair['ai_system'].dropna().astype(str).unique())
    ai_label_map = {ai: f'AI-{idx + 1}' for idx, ai in enumerate(ai_order)}

    row_order = [(ai, sm) for sm in metrics for ai in ai_order]
    x = np.arange(len(row_order))
    width = 0.24
    offsets = {
        'without_recalibration': -width,
        'recalibrate_every_step': 0.0,
        'recalibrate_significant_only': width,
    }

    def _extract_val(df, ai, sm, pol, col):
        z = df[(df['ai_system'].astype(str) == str(ai)) & (df['signal_metric'].astype(str) == str(sm)) & (df['policy'] == pol)]
        if len(z) == 0:
            return np.nan
        return float(pd.to_numeric(z[col], errors='coerce').iloc[0])

    fig, ax = plt.subplots(figsize=(18, 8), constrained_layout=False)
    y_all = []
    for pol in policy_order:
        y = [_extract_val(d_pair, ai, sm, pol, col_name) for ai, sm in row_order]
        y_all.extend([v for v in y if pd.notna(v)])
        ax.bar(x + offsets[pol], y, width=width, label=policy_label[pol], color=colors[pol])

    target_vals = pd.to_numeric(d_pair[target_col], errors='coerce').dropna()
    target_val = float(target_vals.mean()) if len(target_vals) else np.nan
    if pd.notna(target_val):
        ax.axhline(target_val, color='black', ls=':', lw=2.0, label=target_label)

    ax.set_title(
        f'{hospital} | {manufacturer}: {y_label}',
        fontweight='bold', fontsize=16, pad=12,
    )
    ax.set_ylabel(y_label, fontsize=12)
    ax.set_xticks(x)
    ax.set_xticklabels([ai_label_map.get(ai, str(ai)) for ai, _ in row_order], fontsize=11)
    ax.tick_params(axis='y', labelsize=11)
    ax.grid(axis='y', alpha=0.25)

    for group_idx, sm in enumerate(metrics):
        start_pos = group_idx * len(ai_order)
        end_pos = start_pos + len(ai_order) - 1
        center_pos = (start_pos + end_pos) / 2
        ax.text(
            center_pos,
            -0.17,
            metric_label.get(sm, str(sm).replace('_', ' ').title()),
            transform=ax.get_xaxis_transform(),
            ha='center',
            va='top',
            fontsize=11,
        )
        if group_idx > 0:
            ax.axvline(start_pos - 0.5, color='0.85', lw=1.0, zorder=0)

    ax.set_xlabel('')
    finite_vals = [v for v in y_all if pd.notna(v)]
    if pd.notna(target_val):
        finite_vals.append(target_val)
    if len(finite_vals) > 0:
        lower = min(float(preferred_y_min), float(np.nanmin(finite_vals)) - 1.0)
        upper = max(float(preferred_y_max), float(np.nanmax(finite_vals)) + 1.0)
        ax.set_ylim(max(0.0, lower), min(100.0, upper))

    step_ex = int(pd.to_numeric(d_pair['step_exams'], errors='coerce').dropna().iloc[0])
    gap_ex = int(pd.to_numeric(d_pair['gap_exams'], errors='coerce').dropna().iloc[0])
    handles, labels = ax.get_legend_handles_labels()
    note_handles = [
        Line2D([], [], color='none', label=f'Hospital: {hospital}'),
        Line2D([], [], color='none', label=f'Manufacturer: {manufacturer}'),
        Line2D([], [], color='none', label=f'Test cadence: every {step_ex:,} exams; gap = {gap_ex:,} exams'),
        Line2D([], [], color='none', label='No aggregation: one hospital-manufacturer pair'),
        Line2D([], [], color='none', label='Screen-detected cancer signal is theoretical'),
        Line2D([], [], color='none', label='and uses delayed outcome information'),
    ]
    handles = handles + note_handles
    labels = labels + [h.get_label() for h in note_handles]
    ax.legend(
        handles,
        labels,
        loc='center left',
        bbox_to_anchor=(1.01, 0.5),
        ncol=1,
        fontsize=10,
        title='Policies and figure notes',
        title_fontsize=11,
        frameon=True,
    )

    fig.subplots_adjust(top=0.88, bottom=0.24, left=0.07, right=0.76)
    out_path = pair_plot_out_dir / out_name
    fig.savefig(out_path, dpi=220, bbox_inches='tight')
    if bool(globals().get('P3D_PAIR_PLOTS_SHOW', True)):
        plt.show()
    else:
        plt.close(fig)
    return out_path


pair_df = p3d_policy3_pair_summary_with_screen_theoretical.copy()
pair_df['hospital'] = pair_df['hospital'].astype(str)
pair_df['manufacturer'] = pair_df['manufacturer'].fillna('ALL').astype(str)
pairs = pair_df[['hospital', 'manufacturer']].drop_duplicates().sort_values(['hospital', 'manufacturer']).reset_index(drop=True)

figure_rows = []
for _, pair_row in pairs.iterrows():
    hospital = pair_row['hospital']
    manufacturer = pair_row['manufacturer']
    d_pair = pair_df[(pair_df['hospital'] == hospital) & (pair_df['manufacturer'] == manufacturer)].copy()
    if len(d_pair) == 0:
        continue

    pair_slug = f'{_slug(hospital)}_{_slug(manufacturer)}'
    sens_path = _plot_policy3_pair_with_screen(
        d_pair,
        col_name='tw_sensitivity_recal',
        target_col='target_sensitivity',
        target_label='Target sensitivity',
        y_label='Time-weighted sensitivity (%)',
        hospital=hospital,
        manufacturer=manufacturer,
        out_name=f'p3d_policy3_pair_sensitivity_with_screen_theoretical_{pair_slug}.png',
        preferred_y_min=34,
        preferred_y_max=50,
    )
    spec_path = _plot_policy3_pair_with_screen(
        d_pair,
        col_name='tw_specificity_recal',
        target_col='target_specificity',
        target_label='Target specificity',
        y_label='Time-weighted specificity (%)',
        hospital=hospital,
        manufacturer=manufacturer,
        out_name=f'p3d_policy3_pair_specificity_with_screen_theoretical_{pair_slug}.png',
        preferred_y_min=88,
        preferred_y_max=98,
    )

    if sens_path is None and spec_path is None:
        continue

    figure_rows.append({
        'hospital': hospital,
        'manufacturer': manufacturer,
        'sensitivity_png': '' if sens_path is None else str(sens_path),
        'specificity_png': '' if spec_path is None else str(spec_path),
    })

p3d_policy3_pair_screen_figure_index = pd.DataFrame(figure_rows)
figure_index_csv = pair_plot_out_dir / 'p3d_policy3_pair_screen_figure_index.csv'
p3d_policy3_pair_screen_figure_index.to_csv(figure_index_csv, index=False)

ai_order_for_print = sorted(pair_df['ai_system'].dropna().astype(str).unique())
print('Image AI label mapping:', ', '.join(f'AI-{idx + 1} = {name}' for idx, name in enumerate(ai_order_for_print)))
print('Saved combined pair plotting table:', pair_combined_csv)
print('Saved pair figure index:', figure_index_csv)
print('Saved hospital-manufacturer pair figures in:', pair_plot_out_dir)

display(p3d_policy3_pair_screen_figure_index)


In [ ]:
# Part 3D.6H - Cancer detection rate recalibration figures
#
# This cell keeps the older conditional screen-detected cancer metric available
# in Part 3D.6E, but produces the active figures with the requested metric:
#
#     cancer_detection_rate = 1,000 * (# AI-flagged screen-detected cancers) / (# exams)
#
# The numerator depends on the AI threshold, so this can be used to choose a new
# threshold. The signal is still theoretical because screen-detected cancer status
# uses delayed outcome information.

import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.lines import Line2D

required_names = [
    'df_part3', '_p3d_get_columns', '_p3d_prepare_pair_df', '_p3d_add_rad_flag',
    '_p3d_decision_end_indices', '_p3d_rate_stats_from_flags', '_p3d_rate_stats_from_threshold',
    '_p3d_choose_initial_threshold_eval_targets', '_p3d_time_weighted_mean', '_p3d_two_prop_pvalue',
    '_p3d_candidate_thresholds'
]
missing = [name for name in required_names if name not in globals()]
if missing:
    raise RuntimeError('Run Part 3D.0 and Part 3D.6A before this cell. Missing: ' + ', '.join(missing))

base_out_dir = Path(globals().get(
    'P3D_OUTPUT_DIR',
    Path('outputs/2025_12_29_counterfactual_manufacturer_swap_setup/part3d_oracle_recalibration')
))
cdr_out_dir = base_out_dir.parent / 'part3d_oracle_recalibration_with_cancer_detection_rate_theoretical'
cdr_out_dir.mkdir(parents=True, exist_ok=True)

cols = _p3d_get_columns(df_part3)
ai_systems = cols['ai_cols'] if globals().get('P3D_AI_SYSTEMS', None) is None else [
    a for a in globals().get('P3D_AI_SYSTEMS', []) if a in cols['ai_cols']
]

prior_size = int(globals().get('P3D_POLICY_PRIOR_SIZE', 20000))
current_size = int(globals().get('P3D_POLICY_CURRENT_SIZE', 20000))
future_size = int(globals().get('P3D_POLICY_FUTURE_SIZE', 20000))
step_size = int(globals().get('P3D_POLICY_STEP', 2000))
gap_size = int(globals().get('P3D_POLICY_GAP', step_size))
rad_mode = str(globals().get('P3D_RAD_MODE', 'single'))
max_candidates = int(globals().get('P3D_MAX_THRESHOLD_CANDIDATES', 101))
signal_p_thr = float(globals().get('P3D_SIGNAL_P_THRESHOLD', 0.05))
min_recal_interval = int(globals().get('P3D_MIN_RECALIBRATION_INTERVAL', 2000))
max_points = globals().get('P3D_MAX_DECISION_POINTS', None)

# Ensure the delayed-outcome screen-detected flag exists in df_part3.
if 'time_based_screen' not in df_part3.columns:
    if 'days_to_diagnosis_pos' not in df_part3.columns:
        if 'extract_min_positive_days' in globals() and 'days_to_diagnosis' in df_part3.columns:
            df_part3['days_to_diagnosis_pos'] = df_part3['days_to_diagnosis'].apply(extract_min_positive_days)
        else:
            raise RuntimeError('Cannot create time_based_screen: missing days_to_diagnosis/extract_min_positive_days.')
    df_part3['time_based_screen'] = (
        (df_part3[cols['cancer']].fillna(0).astype(int) == 1)
        & (df_part3['days_to_diagnosis_pos'].notna())
        & (df_part3['days_to_diagnosis_pos'] <= int(globals().get('SCREEN_THRESHOLD_DAYS', 90)))
    )


policy_order = ['without_recalibration', 'recalibrate_every_step', 'recalibrate_significant_only']
policy_label = {
    'without_recalibration': 'No recalibration',
    'recalibrate_every_step': 'Recalibrate every step',
    'recalibrate_significant_only': 'Recalibrate if significant',
}
colors = {
    'without_recalibration': '#8da0cb',
    'recalibrate_every_step': '#fc8d62',
    'recalibrate_significant_only': '#66c2a5',
}
metric_order = [
    'flagging_rate',
    'relative_flagging_rate',
    'p_ai_given_rad',
    'p_rad_given_ai',
    'cancer_detection_rate',
]
metric_label = {
    'flagging_rate': 'Flagging rate',
    'relative_flagging_rate': 'Relative\nflagging rate',
    'p_ai_given_rad': 'P(AI flagged |\nradiologist flagged)',
    'p_rad_given_ai': 'P(radiologist flagged |\nAI flagged)',
    'cancer_detection_rate': 'Cancer detection\nrate',
}


def _p3d_cdr_signal_stats_from_threshold(window_df, ai_col, thr, screen_col='time_based_screen'):
    """AI-flagged screen-detected cancers per 1,000 exams at a threshold."""
    scores = pd.to_numeric(window_df[ai_col], errors='coerce').to_numpy(dtype=float)
    screen = window_df[screen_col].fillna(False).astype(bool).to_numpy()
    valid = ~np.isnan(scores)
    scores = scores[valid]
    screen = screen[valid]

    n_total = int(len(scores))
    if n_total == 0 or pd.isna(thr):
        return {
            'cancer_detection_rate': np.nan,
            'cdr_per_1000': np.nan,
            'cdr_n_total': n_total,
            'cdr_ai_screen_pos': 0,
            'screen_n': int(screen.sum()) if len(screen) else 0,
        }

    ai_pos = scores >= float(thr)
    ai_screen_pos = int((ai_pos & screen).sum())
    screen_n = int(screen.sum())
    cdr_per_1000 = 1000.0 * ai_screen_pos / n_total
    return {
        'cancer_detection_rate': float(cdr_per_1000),
        'cdr_per_1000': float(cdr_per_1000),
        'cdr_n_total': n_total,
        'cdr_ai_screen_pos': ai_screen_pos,
        'screen_n': screen_n,
    }


def _p3d_cdr_signal_pvalue(prior_sig, curr_sig):
    return _p3d_two_prop_pvalue(
        curr_sig.get('cdr_ai_screen_pos', 0), curr_sig.get('cdr_n_total', 0),
        prior_sig.get('cdr_ai_screen_pos', 0), prior_sig.get('cdr_n_total', 0),
    )


def _p3d_choose_threshold_for_cdr(window_df, ai_col, target_rate, screen_col='time_based_screen', max_candidates=101):
    if pd.isna(target_rate):
        return np.nan, np.nan

    scores_all = pd.to_numeric(window_df[ai_col], errors='coerce').to_numpy(dtype=float)
    thresholds = _p3d_candidate_thresholds(scores_all, max_candidates=max_candidates)
    if len(thresholds) == 0:
        return np.nan, np.nan

    obj = []
    for thr in thresholds:
        sig = _p3d_cdr_signal_stats_from_threshold(window_df, ai_col, thr, screen_col=screen_col)
        val = sig.get('cancer_detection_rate', np.nan)
        obj.append(np.inf if pd.isna(val) else abs(float(val) - float(target_rate)))

    obj = np.asarray(obj, dtype=float)
    if np.isinf(obj).all():
        return np.nan, np.nan
    best_idx = int(np.argmin(obj))
    return float(thresholds[best_idx]), float(obj[best_idx])


def _p3d_run_diary_cancer_detection_rate(
    df_pair,
    ai_col,
    cancer_col,
    date_col,
    prior_size,
    current_size,
    future_size,
    gap_size,
    step_size,
    policy_name,
    signal_p_threshold=0.05,
    min_recalibration_interval=2000,
    max_threshold_candidates=101,
    max_points=None,
):
    d = df_pair.reset_index(drop=True)
    n = len(d)
    ends = _p3d_decision_end_indices(
        n=n,
        prior_size=prior_size,
        current_size=current_size,
        gap_size=gap_size,
        step_size=step_size,
        future_size=future_size,
        max_points=max_points,
    )
    if len(ends) == 0:
        return pd.DataFrame(), {}, {'reason': 'insufficient_exams'}

    t0 = int(ends[0])
    t0_current_start = t0 - current_size
    t0_prior_end = t0_current_start - gap_size
    t0_prior_start = t0_prior_end - prior_size
    prior0 = d.iloc[t0_prior_start:t0_prior_end]

    target_stats = _p3d_rate_stats_from_flags(prior0['rad_flagged'].to_numpy(), prior0[cancer_col].to_numpy())
    targets = {
        'sens': float(target_stats['sens']),
        'spec': float(target_stats['spec']),
        'ppv': float(target_stats['ppv']),
    }

    # Start from the same initial operating point used by the standard metrics.
    init_thr, _ = _p3d_choose_initial_threshold_eval_targets(
        prior0,
        ai_col=ai_col,
        cancer_col=cancer_col,
        targets=targets,
        max_candidates=max_threshold_candidates,
    )
    if pd.isna(init_thr):
        init_thr = float(pd.to_numeric(prior0[ai_col], errors='coerce').median())

    signal_target_stats = _p3d_cdr_signal_stats_from_threshold(prior0, ai_col, init_thr)
    target_signal_value = signal_target_stats.get('cancer_detection_rate', np.nan)

    rows = []
    current_thr = init_thr
    last_threshold_change_end_idx = None

    for i, end_idx in enumerate(ends):
        end_idx = int(end_idx)
        current_start = end_idx - current_size
        prior_end = current_start - gap_size
        prior_start = prior_end - prior_size
        future_start = end_idx
        future_end = end_idx + future_size

        prior_w = d.iloc[prior_start:prior_end]
        curr_w = d.iloc[current_start:end_idx]
        future_w = d.iloc[future_start:future_end]

        prior_perf = _p3d_rate_stats_from_threshold(prior_w[ai_col].to_numpy(), prior_w[cancer_col].to_numpy(), current_thr)
        curr_perf = _p3d_rate_stats_from_threshold(curr_w[ai_col].to_numpy(), curr_w[cancer_col].to_numpy(), current_thr)
        prior_sig = _p3d_cdr_signal_stats_from_threshold(prior_w, ai_col, current_thr)
        curr_sig = _p3d_cdr_signal_stats_from_threshold(curr_w, ai_col, current_thr)

        current_signal_value = curr_sig.get('cancer_detection_rate', np.nan)
        drift = np.nan if (pd.isna(target_signal_value) or pd.isna(current_signal_value)) else abs(float(current_signal_value) - float(target_signal_value))
        p_signal = _p3d_cdr_signal_pvalue(prior_sig, curr_sig)

        exams_since_last_change = np.nan if last_threshold_change_end_idx is None else float(end_idx - last_threshold_change_end_idx)
        cadence_ok = bool(
            last_threshold_change_end_idx is None or
            exams_since_last_change >= float(min_recalibration_interval)
        )

        if policy_name == 'without_recalibration':
            signal_detected = False
            recalibrate = False
        elif policy_name == 'recalibrate_every_step':
            signal_detected = bool(pd.notna(drift))
            recalibrate = bool(signal_detected and cadence_ok)
        elif policy_name == 'recalibrate_significant_only':
            signal_detected = bool(pd.notna(p_signal) and p_signal < float(signal_p_threshold))
            recalibrate = bool(signal_detected and pd.notna(drift) and drift > 0.0 and cadence_ok)
        else:
            raise ValueError(f'Unsupported policy_name: {policy_name}')

        new_thr = current_thr
        best_obj = np.nan
        if recalibrate:
            new_thr, best_obj = _p3d_choose_threshold_for_cdr(
                curr_w,
                ai_col=ai_col,
                target_rate=target_signal_value,
                max_candidates=max_threshold_candidates,
            )
            if pd.isna(new_thr):
                new_thr = current_thr

        threshold_changed = int(pd.notna(current_thr) and pd.notna(new_thr) and (abs(new_thr - current_thr) > 1e-12))
        if threshold_changed:
            last_threshold_change_end_idx = end_idx

        fut_perf = _p3d_rate_stats_from_threshold(future_w[ai_col].to_numpy(), future_w[cancer_col].to_numpy(), new_thr)
        decision_date = d.iloc[end_idx][date_col] if date_col in d.columns and end_idx < len(d) else pd.NaT

        rows.append({
            'decision_idx': i,
            'decision_end_exam_idx': end_idx,
            'decision_date': decision_date,
            'objective_metric': 'cancer_detection_rate',
            'signal_metric': 'cancer_detection_rate',
            'ai_system': ai_col,
            'prior_size': prior_size,
            'current_size': current_size,
            'future_size': future_size,
            'gap_size': gap_size,
            'step_size': step_size,
            'prior_start_idx': prior_start,
            'prior_end_idx': prior_end,
            'current_start_idx': current_start,
            'current_end_idx': end_idx,
            'future_start_idx': future_start,
            'future_end_idx': future_end,
            'target_sens': targets['sens'],
            'target_spec': targets['spec'],
            'target_ppv': targets['ppv'],
            'threshold_before': current_thr,
            'threshold_after': new_thr,
            'threshold_changed': threshold_changed,
            'drift_value': drift,
            'signal_mode': policy_name,
            'signal_p_threshold': float(signal_p_threshold),
            'min_recalibration_interval': float(min_recalibration_interval),
            'exams_since_last_change': exams_since_last_change,
            'cadence_ok': int(cadence_ok),
            'p_value_signal': p_signal,
            'signal_detected': int(signal_detected),
            'recalibrated': int(recalibrate),
            'best_obj_value': best_obj,
            'prior_sens_before': prior_perf['sens'],
            'prior_spec_before': prior_perf['spec'],
            'prior_ppv_before': prior_perf['ppv'],
            'current_sens_before': curr_perf['sens'],
            'current_spec_before': curr_perf['spec'],
            'current_ppv_before': curr_perf['ppv'],
            'target_signal_value': target_signal_value,
            'prior_signal_value': prior_sig.get('cancer_detection_rate', np.nan),
            'current_signal_value': current_signal_value,
            'future_sens_after': fut_perf['sens'],
            'future_spec_after': fut_perf['spec'],
            'future_ppv_after': fut_perf['ppv'],
            'future_n': fut_perf['n_total'],
            'prior_cdr_ai_screen_pos': prior_sig.get('cdr_ai_screen_pos', 0),
            'prior_cdr_n_total': prior_sig.get('cdr_n_total', 0),
            'current_cdr_ai_screen_pos': curr_sig.get('cdr_ai_screen_pos', 0),
            'current_cdr_n_total': curr_sig.get('cdr_n_total', 0),
        })

        current_thr = new_thr

    diary = pd.DataFrame(rows)
    if len(diary) == 0:
        return diary, targets, {'reason': 'empty_diary'}

    seg_start = diary['decision_end_exam_idx'].astype(int).to_numpy()
    seg_end = np.r_[seg_start[1:], [len(d)]]
    seg_sens, seg_spec, seg_ppv, seg_n = [], [], [], []
    for s, e, thr in zip(seg_start, seg_end, diary['threshold_after'].to_numpy(dtype=float)):
        seg = d.iloc[int(s):int(e)]
        perf = _p3d_rate_stats_from_threshold(seg[ai_col].to_numpy(), seg[cancer_col].to_numpy(), thr)
        seg_sens.append(perf['sens'])
        seg_spec.append(perf['spec'])
        seg_ppv.append(perf['ppv'])
        seg_n.append(perf['n_total'])

    diary['segment_start_idx'] = seg_start
    diary['segment_end_idx'] = seg_end
    diary['segment_n'] = np.asarray(seg_n, dtype=float)
    diary['segment_sens'] = np.asarray(seg_sens, dtype=float)
    diary['segment_spec'] = np.asarray(seg_spec, dtype=float)
    diary['segment_ppv'] = np.asarray(seg_ppv, dtype=float)
    diary['segment_abs_delta_sens'] = np.abs(diary['segment_sens'] - diary['target_sens'])
    diary['segment_abs_delta_spec'] = np.abs(diary['segment_spec'] - diary['target_spec'])
    diary['segment_abs_delta_ppv'] = np.abs(diary['segment_ppv'] - diary['target_ppv'])

    return diary, targets, {'reason': 'ok'}


def _safe_wavg(values, weights):
    v = pd.to_numeric(values, errors='coerce')
    w = pd.to_numeric(weights, errors='coerce').fillna(0)
    mask = v.notna() & w.notna() & (w > 0)
    if mask.sum() == 0:
        return np.nan
    return float(np.average(v[mask], weights=w[mask]))


def _p3d_metric_summary_from_pair_rows(pair_summary, aggregation_label):
    metric_cols = [
        'target_sensitivity', 'target_specificity',
        'tw_sensitivity_recal', 'tw_specificity_recal',
        'tw_sensitivity_no_recal', 'tw_specificity_no_recal',
        'delta_sens_recal_minus_no', 'delta_spec_recal_minus_no',
    ]

    h_rows = []
    gcols_h = ['hospital', 'ai_system', 'signal_metric', 'policy', 'step_exams', 'gap_exams']
    for keys, g in pair_summary.groupby(gcols_h, dropna=False):
        row = {
            'hospital': keys[0],
            'ai_system': keys[1],
            'signal_metric': keys[2],
            'policy': keys[3],
            'step_exams': int(keys[4]),
            'gap_exams': int(keys[5]),
            'n_pairs': int(len(g)),
            'n_test_steps_total': int(pd.to_numeric(g['n_test_steps'], errors='coerce').fillna(0).sum()),
            'n_signal_detected_total': int(pd.to_numeric(g['n_signal_detected'], errors='coerce').fillna(0).sum()),
            'n_recalibrations_total': int(pd.to_numeric(g['n_recalibrations'], errors='coerce').fillna(0).sum()),
            'n_threshold_changes_total': int(pd.to_numeric(g['n_threshold_changes'], errors='coerce').fillna(0).sum()),
            'weights_total_exams': float(pd.to_numeric(g['weights_total_exams'], errors='coerce').fillna(0).sum()),
        }
        w = pd.to_numeric(g['weights_total_exams'], errors='coerce').fillna(0)
        for c in metric_cols:
            row[c] = _safe_wavg(g[c], w)
        row['mean_exams_between_recal'] = _safe_wavg(
            g['mean_exams_between_recal'],
            pd.to_numeric(g['n_threshold_changes'], errors='coerce').fillna(0),
        )
        row['median_exams_between_recal'] = _safe_wavg(
            g['median_exams_between_recal'],
            pd.to_numeric(g['n_threshold_changes'], errors='coerce').fillna(0),
        )
        h_rows.append(row)
    hospital_summary = pd.DataFrame(h_rows)

    o_rows = []
    gcols_o = ['ai_system', 'signal_metric', 'policy', 'step_exams', 'gap_exams']
    for keys, g in hospital_summary.groupby(gcols_o, dropna=False):
        row = {
            'ai_system': keys[0],
            'signal_metric': keys[1],
            'policy': keys[2],
            'step_exams': int(keys[3]),
            'gap_exams': int(keys[4]),
            'n_hospitals': int(g['hospital'].nunique()),
            'n_test_steps_total': int(pd.to_numeric(g['n_test_steps_total'], errors='coerce').fillna(0).sum()),
            'n_signal_detected_total': int(pd.to_numeric(g['n_signal_detected_total'], errors='coerce').fillna(0).sum()),
            'n_recalibrations_total': int(pd.to_numeric(g['n_recalibrations_total'], errors='coerce').fillna(0).sum()),
            'n_threshold_changes_total': int(pd.to_numeric(g['n_threshold_changes_total'], errors='coerce').fillna(0).sum()),
            'weights_total_exams': float(pd.to_numeric(g['weights_total_exams'], errors='coerce').fillna(0).sum()),
        }
        w = pd.to_numeric(g['weights_total_exams'], errors='coerce').fillna(0)
        for c in metric_cols:
            row[c] = _safe_wavg(g[c], w)
        row['mean_exams_between_recal'] = _safe_wavg(
            g['mean_exams_between_recal'],
            pd.to_numeric(g['n_threshold_changes_total'], errors='coerce').fillna(0),
        )
        row['median_exams_between_recal'] = _safe_wavg(
            g['median_exams_between_recal'],
            pd.to_numeric(g['n_threshold_changes_total'], errors='coerce').fillna(0),
        )
        o_rows.append(row)
    overall_summary = pd.DataFrame(o_rows)

    u_rows = []
    for keys, g in hospital_summary.groupby(gcols_o, dropna=False):
        metric_mask = pd.to_numeric(g['tw_sensitivity_recal'], errors='coerce').notna()
        valid = g.loc[metric_mask].copy()
        row = {
            'ai_system': keys[0],
            'signal_metric': keys[1],
            'policy': keys[2],
            'step_exams': int(keys[3]),
            'gap_exams': int(keys[4]),
            'n_hospitals': int(valid['hospital'].nunique()),
            'n_test_steps_total': int(pd.to_numeric(valid['n_test_steps_total'], errors='coerce').fillna(0).sum()),
            'n_signal_detected_total': int(pd.to_numeric(valid['n_signal_detected_total'], errors='coerce').fillna(0).sum()),
            'n_recalibrations_total': int(pd.to_numeric(valid['n_recalibrations_total'], errors='coerce').fillna(0).sum()),
            'n_threshold_changes_total': int(pd.to_numeric(valid['n_threshold_changes_total'], errors='coerce').fillna(0).sum()),
            'weights_total_exams': float(pd.to_numeric(valid['weights_total_exams'], errors='coerce').fillna(0).sum()),
            'aggregation_method': aggregation_label,
        }
        for c in ['target_sensitivity', 'target_specificity', 'tw_sensitivity_recal', 'tw_specificity_recal']:
            row[c] = float(pd.to_numeric(valid[c], errors='coerce').mean()) if len(valid) else np.nan
        row['mean_exams_between_recal'] = float(pd.to_numeric(valid['mean_exams_between_recal'], errors='coerce').mean()) if len(valid) else np.nan
        row['median_exams_between_recal'] = float(pd.to_numeric(valid['median_exams_between_recal'], errors='coerce').mean()) if len(valid) else np.nan
        u_rows.append(row)
    unweighted_summary = pd.DataFrame(u_rows)

    return hospital_summary, overall_summary, unweighted_summary


def _load_table_from_memory_or_csv(var_name, csv_path):
    obj = globals().get(var_name)
    if isinstance(obj, pd.DataFrame) and len(obj) > 0:
        return obj.copy()
    csv_path = Path(csv_path)
    if csv_path.exists():
        return pd.read_csv(csv_path)
    raise RuntimeError(f'Missing {var_name}. Run the upstream Part 3D cells first, or create {csv_path}.')


def _combine_tables(a, b):
    cols_all = sorted(set(a.columns) | set(b.columns))
    return pd.concat([a.reindex(columns=cols_all), b.reindex(columns=cols_all)], ignore_index=True, sort=False)


def _slug(value):
    text = str(value) if pd.notna(value) else 'missing'
    slug = ''.join(ch.lower() if ch.isalnum() else '_' for ch in text).strip('_')
    while '__' in slug:
        slug = slug.replace('__', '_')
    return slug or 'missing'


# ------------------------------------------------------------------
# Run cancer detection rate recalibration across hospital pairs.
# ------------------------------------------------------------------
rows = []
all_hospitals = sorted(df_part3[cols['hospital']].dropna().unique())
hospital_filter = globals().get('P3D_POLICY_HOSPITALS', None)
if hospital_filter is not None:
    all_hospitals = [h for h in all_hospitals if h in set(hospital_filter)]

for hospital in all_hospitals:
    d_h = df_part3[df_part3[cols['hospital']] == hospital].copy()
    if len(d_h) == 0:
        continue

    if cols['manufacturer'] is not None and cols['manufacturer'] in d_h.columns:
        manufacturers = sorted(d_h[cols['manufacturer']].dropna().unique())
        if len(manufacturers) == 0:
            manufacturers = [None]
    else:
        manufacturers = [None]

    mfr_filter = globals().get('P3D_POLICY_MANUFACTURERS', None)
    if mfr_filter is not None:
        manufacturers = [m for m in manufacturers if m in set(mfr_filter)]

    for manufacturer in manufacturers:
        pair_df = _p3d_prepare_pair_df(df_part3, cols, hospital, manufacturer)
        if len(pair_df) == 0:
            continue
        pair_df = _p3d_add_rad_flag(pair_df, rad_mode=rad_mode)

        for ai in ai_systems:
            for policy_name in policy_order:
                diary, targets, meta = _p3d_run_diary_cancer_detection_rate(
                    df_pair=pair_df,
                    ai_col=ai,
                    cancer_col=cols['cancer'],
                    date_col=cols['date'],
                    prior_size=prior_size,
                    current_size=current_size,
                    future_size=future_size,
                    gap_size=gap_size,
                    step_size=step_size,
                    policy_name=policy_name,
                    signal_p_threshold=signal_p_thr,
                    min_recalibration_interval=min_recal_interval,
                    max_threshold_candidates=max_candidates,
                    max_points=max_points,
                )

                if diary is None or len(diary) == 0:
                    rows.append({
                        'hospital': hospital,
                        'manufacturer': manufacturer if manufacturer is not None else 'ALL',
                        'ai_system': ai,
                        'signal_metric': 'cancer_detection_rate',
                        'policy': policy_name,
                        'status': meta.get('reason', 'no_data') if isinstance(meta, dict) else 'no_data',
                        'step_exams': step_size,
                        'gap_exams': gap_size,
                        'n_test_steps': 0,
                        'n_signal_detected': 0,
                        'n_recalibrations': 0,
                        'n_threshold_changes': 0,
                        'mean_exams_between_recal': np.nan,
                        'median_exams_between_recal': np.nan,
                        'weights_total_exams': 0.0,
                        'target_sensitivity': np.nan,
                        'target_specificity': np.nan,
                        'tw_sensitivity_recal': np.nan,
                        'tw_specificity_recal': np.nan,
                        'tw_sensitivity_no_recal': np.nan,
                        'tw_specificity_no_recal': np.nan,
                        'delta_sens_recal_minus_no': np.nan,
                        'delta_spec_recal_minus_no': np.nan,
                        'target_cancer_detection_rate_per_1000': np.nan,
                    })
                    continue

                w = pd.to_numeric(diary['segment_n'], errors='coerce').fillna(0)
                tw_sens_recal = _p3d_time_weighted_mean(diary['segment_sens'], w)
                tw_spec_recal = _p3d_time_weighted_mean(diary['segment_spec'], w)

                changes = pd.to_numeric(
                    diary.loc[pd.to_numeric(diary['threshold_changed'], errors='coerce').fillna(0) > 0, 'decision_end_exam_idx'],
                    errors='coerce'
                ).dropna().to_numpy(dtype=float)
                if len(changes) >= 2:
                    gaps_recal = np.diff(changes)
                    mean_between_recal = float(np.mean(gaps_recal))
                    median_between_recal = float(np.median(gaps_recal))
                else:
                    mean_between_recal = np.nan
                    median_between_recal = np.nan

                rows.append({
                    'hospital': hospital,
                    'manufacturer': manufacturer if manufacturer is not None else 'ALL',
                    'ai_system': ai,
                    'signal_metric': 'cancer_detection_rate',
                    'policy': policy_name,
                    'status': meta.get('reason', 'ok') if isinstance(meta, dict) else 'ok',
                    'step_exams': step_size,
                    'gap_exams': gap_size,
                    'n_test_steps': int(len(diary)),
                    'n_signal_detected': int(pd.to_numeric(diary['signal_detected'], errors='coerce').fillna(0).sum()),
                    'n_recalibrations': int(pd.to_numeric(diary['recalibrated'], errors='coerce').fillna(0).sum()),
                    'n_threshold_changes': int(pd.to_numeric(diary['threshold_changed'], errors='coerce').fillna(0).sum()),
                    'mean_exams_between_recal': mean_between_recal,
                    'median_exams_between_recal': median_between_recal,
                    'weights_total_exams': float(w.sum()),
                    'target_sensitivity': float(pd.to_numeric(diary['target_sens'], errors='coerce').iloc[0]),
                    'target_specificity': float(pd.to_numeric(diary['target_spec'], errors='coerce').iloc[0]),
                    'tw_sensitivity_recal': tw_sens_recal,
                    'tw_specificity_recal': tw_spec_recal,
                    'tw_sensitivity_no_recal': np.nan,
                    'tw_specificity_no_recal': np.nan,
                    'delta_sens_recal_minus_no': np.nan,
                    'delta_spec_recal_minus_no': np.nan,
                    'target_cancer_detection_rate_per_1000': float(pd.to_numeric(diary['target_signal_value'], errors='coerce').dropna().iloc[0]) if pd.to_numeric(diary['target_signal_value'], errors='coerce').notna().any() else np.nan,
                })

p3d_policy3_pair_summary_cancer_detection_rate = pd.DataFrame(rows)
p3d_policy3_hospital_summary_cancer_detection_rate, p3d_policy3_overall_summary_cancer_detection_rate, p3d_policy3_overall_summary_cancer_detection_rate_unweighted = _p3d_metric_summary_from_pair_rows(
    p3d_policy3_pair_summary_cancer_detection_rate,
    aggregation_label='unweighted_mean_of_hospital_level_metrics',
)

pair_cdr_csv = cdr_out_dir / 'p3d_policy3_pair_summary_cancer_detection_rate.csv'
hospital_cdr_csv = cdr_out_dir / 'p3d_policy3_hospital_summary_cancer_detection_rate.csv'
overall_cdr_csv = cdr_out_dir / 'p3d_policy3_overall_summary_cancer_detection_rate.csv'
overall_cdr_unweighted_csv = cdr_out_dir / 'p3d_policy3_overall_summary_cancer_detection_rate_unweighted_hospital_mean.csv'
p3d_policy3_pair_summary_cancer_detection_rate.sort_values(['hospital', 'manufacturer', 'ai_system', 'policy']).to_csv(pair_cdr_csv, index=False)
p3d_policy3_hospital_summary_cancer_detection_rate.sort_values(['hospital', 'ai_system', 'policy']).to_csv(hospital_cdr_csv, index=False)
p3d_policy3_overall_summary_cancer_detection_rate.sort_values(['ai_system', 'policy']).to_csv(overall_cdr_csv, index=False)
p3d_policy3_overall_summary_cancer_detection_rate_unweighted.sort_values(['ai_system', 'policy']).to_csv(overall_cdr_unweighted_csv, index=False)

# Combine standard four signal metrics with Fredrik's cancer detection rate metric.
standard_unweighted = _load_table_from_memory_or_csv(
    'p3d_policy3_overall_summary_unweighted_hospital_mean',
    base_out_dir / 'p3d_policy3_overall_summary_unweighted_hospital_mean.csv',
)
if 'aggregation_method' not in standard_unweighted.columns:
    standard_unweighted['aggregation_method'] = 'unweighted_mean_of_hospital_level_metrics'

p3d_policy3_overall_summary_unweighted_hospital_mean_with_cancer_detection_rate_theoretical = _combine_tables(
    standard_unweighted,
    p3d_policy3_overall_summary_cancer_detection_rate_unweighted,
)
combined_overall_cdr_csv = cdr_out_dir / 'p3d_policy3_overall_summary_unweighted_hospital_mean_with_cancer_detection_rate_theoretical.csv'
p3d_policy3_overall_summary_unweighted_hospital_mean_with_cancer_detection_rate_theoretical.to_csv(combined_overall_cdr_csv, index=False)

standard_pair_summary = _load_table_from_memory_or_csv(
    'p3d_policy3_pair_summary',
    base_out_dir / 'p3d_policy3_pair_summary.csv',
)
if 'manufacturer' not in standard_pair_summary.columns:
    standard_pair_summary['manufacturer'] = 'ALL'

p3d_policy3_pair_summary_with_cancer_detection_rate_theoretical = _combine_tables(
    standard_pair_summary,
    p3d_policy3_pair_summary_cancer_detection_rate,
)
combined_pair_cdr_csv = cdr_out_dir / 'p3d_policy3_pair_summary_with_cancer_detection_rate_theoretical.csv'
p3d_policy3_pair_summary_with_cancer_detection_rate_theoretical.to_csv(combined_pair_cdr_csv, index=False)

standard_hospital_summary = _load_table_from_memory_or_csv(
    'p3d_policy3_hospital_summary',
    base_out_dir / 'p3d_policy3_hospital_summary.csv',
)
p3d_policy3_hospital_summary_with_cancer_detection_rate_theoretical = _combine_tables(
    standard_hospital_summary,
    p3d_policy3_hospital_summary_cancer_detection_rate,
)
combined_hospital_cdr_csv = cdr_out_dir / 'p3d_policy3_hospital_summary_with_cancer_detection_rate_theoretical.csv'
p3d_policy3_hospital_summary_with_cancer_detection_rate_theoretical.to_csv(combined_hospital_cdr_csv, index=False)


# ------------------------------------------------------------------
# Plotting helpers.
# ------------------------------------------------------------------
def _extract_val(df, ai, sm, pol, col):
    z = df[(df['ai_system'].astype(str) == str(ai)) & (df['signal_metric'].astype(str) == str(sm)) & (df['policy'] == pol)]
    if len(z) == 0:
        return np.nan
    return float(pd.to_numeric(z[col], errors='coerce').iloc[0])


def _plot_policy3_cdr(d, col_name, target_col, target_label, y_label, out_dir, out_name, title_prefix, note_lines, preferred_y_min, preferred_y_max):
    d = d.copy()
    d = d[d['policy'].isin(policy_order)].copy()
    if len(d) == 0:
        raise RuntimeError('No rows available for plotting.')

    present = set(d['signal_metric'].dropna().astype(str).unique())
    metrics = [m for m in metric_order if m in present] + [m for m in sorted(present) if m not in metric_order]
    ai_order = sorted(d['ai_system'].dropna().astype(str).unique())
    ai_label_map = {ai: f'AI-{idx + 1}' for idx, ai in enumerate(ai_order)}

    row_order = [(ai, sm) for sm in metrics for ai in ai_order]
    x = np.arange(len(row_order))
    width = 0.24
    offsets = {
        'without_recalibration': -width,
        'recalibrate_every_step': 0.0,
        'recalibrate_significant_only': width,
    }

    fig, ax = plt.subplots(figsize=(18, 8), constrained_layout=False)
    y_all = []
    for pol in policy_order:
        y = [_extract_val(d, ai, sm, pol, col_name) for ai, sm in row_order]
        y_all.extend([v for v in y if pd.notna(v)])
        ax.bar(x + offsets[pol], y, width=width, label=policy_label[pol], color=colors[pol])

    target_vals = pd.to_numeric(d[target_col], errors='coerce').dropna()
    target_val = float(target_vals.mean()) if len(target_vals) else np.nan
    if pd.notna(target_val):
        ax.axhline(target_val, color='black', ls=':', lw=2.0, label=target_label)

    ax.set_title(f'{title_prefix}: {y_label}', fontweight='bold', fontsize=16, pad=12)
    ax.set_ylabel(y_label, fontsize=12)
    ax.set_xticks(x)
    ax.set_xticklabels([ai_label_map.get(ai, str(ai)) for ai, _ in row_order], fontsize=11)
    ax.tick_params(axis='y', labelsize=11)
    ax.grid(axis='y', alpha=0.25)

    for group_idx, sm in enumerate(metrics):
        start_pos = group_idx * len(ai_order)
        end_pos = start_pos + len(ai_order) - 1
        center_pos = (start_pos + end_pos) / 2
        ax.text(
            center_pos,
            -0.17,
            metric_label.get(sm, str(sm).replace('_', ' ').title()),
            transform=ax.get_xaxis_transform(),
            ha='center',
            va='top',
            fontsize=11,
        )
        if group_idx > 0:
            ax.axvline(start_pos - 0.5, color='0.85', lw=1.0, zorder=0)

    ax.set_xlabel('')
    finite_vals = [v for v in y_all if pd.notna(v)]
    if pd.notna(target_val):
        finite_vals.append(target_val)
    if len(finite_vals) > 0:
        lower = min(float(preferred_y_min), float(np.nanmin(finite_vals)) - 1.0)
        upper = max(float(preferred_y_max), float(np.nanmax(finite_vals)) + 1.0)
        ax.set_ylim(max(0.0, lower), min(100.0, upper))

    step_ex = int(pd.to_numeric(d['step_exams'], errors='coerce').dropna().iloc[0])
    gap_ex = int(pd.to_numeric(d['gap_exams'], errors='coerce').dropna().iloc[0])
    handles, labels = ax.get_legend_handles_labels()
    note_handles = [
        Line2D([], [], color='none', label=f'Test cadence: every {step_ex:,} exams; gap = {gap_ex:,} exams'),
        Line2D([], [], color='none', label='Cancer detection rate = AI-flagged screen-detected'),
        Line2D([], [], color='none', label='cancers per 1,000 exams'),
        Line2D([], [], color='none', label='Cancer detection rate signal is theoretical'),
        Line2D([], [], color='none', label='and uses delayed outcome information'),
    ]
    note_handles += [Line2D([], [], color='none', label=line) for line in note_lines]
    handles = handles + note_handles
    labels = labels + [h.get_label() for h in note_handles]
    ax.legend(
        handles,
        labels,
        loc='center left',
        bbox_to_anchor=(1.01, 0.5),
        ncol=1,
        fontsize=10,
        title='Policies and figure notes',
        title_fontsize=11,
        frameon=True,
    )

    fig.subplots_adjust(top=0.88, bottom=0.24, left=0.07, right=0.76)
    out_path = Path(out_dir) / out_name
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_path, dpi=220, bbox_inches='tight')
    if bool(globals().get('P3D_CDR_PLOTS_SHOW', True)):
        plt.show()
    else:
        plt.close(fig)
    return out_path


# Overall plots across hospitals.
sens_png = _plot_policy3_cdr(
    p3d_policy3_overall_summary_unweighted_hospital_mean_with_cancer_detection_rate_theoretical,
    col_name='tw_sensitivity_recal',
    target_col='target_sensitivity',
    target_label='Target sensitivity',
    y_label='Time-weighted sensitivity (%)',
    out_dir=cdr_out_dir,
    out_name='p3d_policy3_overall_unweighted_hospital_mean_sensitivity_with_cancer_detection_rate_theoretical.png',
    title_prefix='Three-policy comparison across hospitals',
    note_lines=['Overall: unweighted mean of hospital-level metrics'],
    preferred_y_min=34,
    preferred_y_max=50,
)
spec_png = _plot_policy3_cdr(
    p3d_policy3_overall_summary_unweighted_hospital_mean_with_cancer_detection_rate_theoretical,
    col_name='tw_specificity_recal',
    target_col='target_specificity',
    target_label='Target specificity',
    y_label='Time-weighted specificity (%)',
    out_dir=cdr_out_dir,
    out_name='p3d_policy3_overall_unweighted_hospital_mean_specificity_with_cancer_detection_rate_theoretical.png',
    title_prefix='Three-policy comparison across hospitals',
    note_lines=['Overall: unweighted mean of hospital-level metrics'],
    preferred_y_min=88,
    preferred_y_max=98,
)

# Danderyds-excluded plots.
no_danderyd_dir = cdr_out_dir / 'excluding_danderyd'
DANDERYD_EXCLUSION_PATTERN = 'danderyd'
DANDERYD_EXCLUSION_LABEL = 'Danderyds Hospital'

hospital_no_danderyd = p3d_policy3_hospital_summary_with_cancer_detection_rate_theoretical[
    ~p3d_policy3_hospital_summary_with_cancer_detection_rate_theoretical['hospital'].astype(str).str.contains(DANDERYD_EXCLUSION_PATTERN, case=False, na=False)
].copy()

# Recompute unweighted no-Danderyd directly from hospital-level rows.
def _unweighted_from_hospital_rows(h):
    h = h[h['policy'].isin(policy_order)].copy()
    rows_local = []
    gcols = ['ai_system', 'signal_metric', 'policy', 'step_exams', 'gap_exams']
    for keys, g in h.groupby(gcols, dropna=False):
        valid = g.loc[pd.to_numeric(g['tw_sensitivity_recal'], errors='coerce').notna()].copy()
        row = {
            'ai_system': keys[0],
            'signal_metric': keys[1],
            'policy': keys[2],
            'step_exams': int(keys[3]),
            'gap_exams': int(keys[4]),
            'n_hospitals': int(valid['hospital'].nunique()) if len(valid) else 0,
            'aggregation_method': 'unweighted_mean_of_hospital_level_metrics_excluding_danderyd',
        }
        for c in ['n_test_steps_total', 'n_signal_detected_total', 'n_recalibrations_total', 'n_threshold_changes_total', 'weights_total_exams']:
            if c in valid.columns:
                row[c] = float(pd.to_numeric(valid[c], errors='coerce').fillna(0).sum())
        for c in ['target_sensitivity', 'target_specificity', 'tw_sensitivity_recal', 'tw_specificity_recal', 'mean_exams_between_recal', 'median_exams_between_recal']:
            if c in valid.columns:
                row[c] = float(pd.to_numeric(valid[c], errors='coerce').mean()) if len(valid) else np.nan
        rows_local.append(row)
    return pd.DataFrame(rows_local)

p3d_policy3_overall_summary_unweighted_hospital_mean_with_cancer_detection_rate_theoretical_no_danderyd = _unweighted_from_hospital_rows(hospital_no_danderyd)
no_danderyd_csv = no_danderyd_dir / 'p3d_policy3_overall_summary_unweighted_hospital_mean_with_cancer_detection_rate_theoretical_no_danderyd.csv'
hospital_no_danderyd_csv = no_danderyd_dir / 'p3d_policy3_hospital_summary_with_cancer_detection_rate_theoretical_no_danderyd.csv'
no_danderyd_dir.mkdir(parents=True, exist_ok=True)
p3d_policy3_overall_summary_unweighted_hospital_mean_with_cancer_detection_rate_theoretical_no_danderyd.to_csv(no_danderyd_csv, index=False)
hospital_no_danderyd.to_csv(hospital_no_danderyd_csv, index=False)

sens_png_no_danderyd = _plot_policy3_cdr(
    p3d_policy3_overall_summary_unweighted_hospital_mean_with_cancer_detection_rate_theoretical_no_danderyd,
    col_name='tw_sensitivity_recal',
    target_col='target_sensitivity',
    target_label='Target sensitivity',
    y_label='Time-weighted sensitivity (%)',
    out_dir=no_danderyd_dir,
    out_name='p3d_policy3_overall_unweighted_hospital_mean_sensitivity_with_cancer_detection_rate_theoretical_no_danderyd.png',
    title_prefix=f'Three-policy comparison across hospitals excluding {DANDERYD_EXCLUSION_LABEL}',
    note_lines=['Overall: unweighted mean of hospital-level metrics', f'{DANDERYD_EXCLUSION_LABEL} excluded'],
    preferred_y_min=34,
    preferred_y_max=50,
)
spec_png_no_danderyd = _plot_policy3_cdr(
    p3d_policy3_overall_summary_unweighted_hospital_mean_with_cancer_detection_rate_theoretical_no_danderyd,
    col_name='tw_specificity_recal',
    target_col='target_specificity',
    target_label='Target specificity',
    y_label='Time-weighted specificity (%)',
    out_dir=no_danderyd_dir,
    out_name='p3d_policy3_overall_unweighted_hospital_mean_specificity_with_cancer_detection_rate_theoretical_no_danderyd.png',
    title_prefix=f'Three-policy comparison across hospitals excluding {DANDERYD_EXCLUSION_LABEL}',
    note_lines=['Overall: unweighted mean of hospital-level metrics', f'{DANDERYD_EXCLUSION_LABEL} excluded'],
    preferred_y_min=88,
    preferred_y_max=98,
)

# Hospital-manufacturer pair plots.
pair_plot_out_dir = cdr_out_dir / 'hospital_manufacturer_pair_figures'
pair_plot_out_dir.mkdir(parents=True, exist_ok=True)
for old_png in pair_plot_out_dir.glob('p3d_policy3_pair_*_with_cancer_detection_rate_theoretical_*.png'):
    old_png.unlink()

pair_df = p3d_policy3_pair_summary_with_cancer_detection_rate_theoretical.copy()
pair_df['hospital'] = pair_df['hospital'].astype(str)
pair_df['manufacturer'] = pair_df['manufacturer'].fillna('ALL').astype(str)
pairs = pair_df[['hospital', 'manufacturer']].drop_duplicates().sort_values(['hospital', 'manufacturer']).reset_index(drop=True)

figure_rows = []
for _, pair_row in pairs.iterrows():
    hospital = pair_row['hospital']
    manufacturer = pair_row['manufacturer']
    d_pair = pair_df[(pair_df['hospital'] == hospital) & (pair_df['manufacturer'] == manufacturer)].copy()
    if len(d_pair) == 0:
        continue
    pair_slug = f'{_slug(hospital)}_{_slug(manufacturer)}'

    has_sens = pd.to_numeric(d_pair['tw_sensitivity_recal'], errors='coerce').notna().any()
    has_spec = pd.to_numeric(d_pair['tw_specificity_recal'], errors='coerce').notna().any()
    if not has_sens and not has_spec:
        print(f'[skip empty] {hospital} | {manufacturer}: no finite sensitivity/specificity values')
        continue

    sens_pair_path = _plot_policy3_cdr(
        d_pair,
        col_name='tw_sensitivity_recal',
        target_col='target_sensitivity',
        target_label='Target sensitivity',
        y_label='Time-weighted sensitivity (%)',
        out_dir=pair_plot_out_dir,
        out_name=f'p3d_policy3_pair_sensitivity_with_cancer_detection_rate_theoretical_{pair_slug}.png',
        title_prefix=f'{hospital} | {manufacturer}',
        note_lines=[f'Hospital: {hospital}', f'Manufacturer: {manufacturer}', 'No aggregation: one hospital-manufacturer pair'],
        preferred_y_min=34,
        preferred_y_max=50,
    ) if has_sens else None
    spec_pair_path = _plot_policy3_cdr(
        d_pair,
        col_name='tw_specificity_recal',
        target_col='target_specificity',
        target_label='Target specificity',
        y_label='Time-weighted specificity (%)',
        out_dir=pair_plot_out_dir,
        out_name=f'p3d_policy3_pair_specificity_with_cancer_detection_rate_theoretical_{pair_slug}.png',
        title_prefix=f'{hospital} | {manufacturer}',
        note_lines=[f'Hospital: {hospital}', f'Manufacturer: {manufacturer}', 'No aggregation: one hospital-manufacturer pair'],
        preferred_y_min=88,
        preferred_y_max=98,
    ) if has_spec else None

    figure_rows.append({
        'hospital': hospital,
        'manufacturer': manufacturer,
        'sensitivity_png': '' if sens_pair_path is None else str(sens_pair_path),
        'specificity_png': '' if spec_pair_path is None else str(spec_pair_path),
    })

p3d_policy3_pair_cancer_detection_rate_figure_index = pd.DataFrame(figure_rows)
figure_index_csv = pair_plot_out_dir / 'p3d_policy3_pair_cancer_detection_rate_figure_index.csv'
p3d_policy3_pair_cancer_detection_rate_figure_index.to_csv(figure_index_csv, index=False)

ai_order_for_print = sorted(p3d_policy3_overall_summary_unweighted_hospital_mean_with_cancer_detection_rate_theoretical['ai_system'].dropna().astype(str).unique())
print('Image AI label mapping:', ', '.join(f'AI-{idx + 1} = {name}' for idx, name in enumerate(ai_order_for_print)))
print('Saved cancer detection rate pair summary:', pair_cdr_csv)
print('Saved cancer detection rate hospital summary:', hospital_cdr_csv)
print('Saved cancer detection rate overall summary:', overall_cdr_csv)
print('Saved combined overall plotting table:', combined_overall_cdr_csv)
print('Saved combined hospital plotting table:', combined_hospital_cdr_csv)
print('Saved combined pair plotting table:', combined_pair_cdr_csv)
print('Saved overall sensitivity figure:', sens_png)
print('Saved overall specificity figure:', spec_png)
print('Saved no-Danderyd sensitivity figure:', sens_png_no_danderyd)
print('Saved no-Danderyd specificity figure:', spec_png_no_danderyd)
print('Saved pair figure index:', figure_index_csv)
print('Saved hospital-manufacturer pair figures in:', pair_plot_out_dir)

display(p3d_policy3_overall_summary_unweighted_hospital_mean_with_cancer_detection_rate_theoretical.head(20))
display(p3d_policy3_pair_cancer_detection_rate_figure_index)


In [ ]:
# Part 3D.7 - Underlying hospital data diagnostics without AI (20,000-exam moving windows)
#
# Professor-requested diagnostic plots. These plots do not use AI scores,
# AI thresholds, or recalibration. They describe the underlying hospital data
# over time with moving 20,000-exam windows.
#
# Graph 1 per hospital:
#   - Radiologist sensitivity (%) on the left y-axis
#   - Radiologist specificity (%) on the right y-axis
#
# Graph 2 per hospital:
#   - Screen-detected cancer detection rate per 1,000 exams on the left y-axis
#   - Radiologist flagging rate per 1,000 exams on the right y-axis
#
# By default this cell uses the balanced Part 3 simulation input so the curves
# correspond to the same hospital-level data used in the recalibration analyses.
# Set UNDERLYING_DATA_MODE = 'raw_source' before running this cell if you instead
# want the raw source rows before the Part 3 monthly 1:100 control sampling.

import ast
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.lines import Line2D

# -----------------------------
# User-facing controls
# -----------------------------
UNDERLYING_WINDOW_EXAMS = int(globals().get('UNDERLYING_WINDOW_EXAMS', 20000))
UNDERLYING_STEP_EXAMS = int(globals().get('UNDERLYING_STEP_EXAMS', 2000))
UNDERLYING_MIN_EXAMS = int(globals().get('UNDERLYING_MIN_EXAMS', UNDERLYING_WINDOW_EXAMS))
if 'UNDERLYING_DATA_MODE' in globals():
    UNDERLYING_DATA_MODE = str(globals().get('UNDERLYING_DATA_MODE'))
elif 'UNDERLYING_USE_BALANCED_PART3_DATA' in globals():
    UNDERLYING_DATA_MODE = 'balanced_part3' if bool(globals().get('UNDERLYING_USE_BALANCED_PART3_DATA')) else 'raw_source'
else:
    UNDERLYING_DATA_MODE = 'balanced_part3'
UNDERLYING_SHOW_PLOTS = bool(globals().get('UNDERLYING_SHOW_PLOTS', True))
UNDERLYING_RAD_MODE = str(globals().get('UNDERLYING_RAD_MODE', 'combined'))  # 'r1', 'r2', 'combined', or 'existing'
UNDERLYING_SCREEN_THRESHOLD_DAYS = int(globals().get('SCREEN_THRESHOLD_DAYS', 90))

UNDERLYING_OUTPUT_DIR = Path(globals().get(
    'UNDERLYING_OUTPUT_DIR',
    Path('outputs/2025_12_29_counterfactual_manufacturer_swap_setup/part3d_underlying_hospital_data_diagnostics')
)) / UNDERLYING_DATA_MODE
UNDERLYING_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# -----------------------------
# Self-contained helpers
# -----------------------------
def _underlying_slug(value):
    text = str(value) if pd.notna(value) else 'missing'
    slug = ''.join(ch.lower() if ch.isalnum() else '_' for ch in text).strip('_')
    while '__' in slug:
        slug = slug.replace('__', '_')
    return slug or 'missing'


def _underlying_extract_min_positive_days(val):
    if pd.isna(val):
        return np.nan
    if isinstance(val, (int, float, np.integer, np.floating)):
        return float(val) if float(val) > 0 else np.nan
    s = str(val).strip()
    if s in ('[]', ''):
        return np.nan
    try:
        parsed = ast.literal_eval(s)
    except Exception:
        return np.nan
    if isinstance(parsed, (list, tuple, set)):
        nums = []
        for x in parsed:
            try:
                if pd.notna(x):
                    nums.append(float(x))
            except Exception:
                pass
    else:
        try:
            nums = [float(parsed)]
        except Exception:
            nums = []
    pos = [x for x in nums if x > 0]
    return min(pos) if pos else np.nan


def _underlying_pick_col(df, preferred, candidates, label):
    if preferred is not None and preferred in df.columns:
        return preferred
    for c in candidates:
        if c in df.columns:
            return c
    raise RuntimeError(f'Could not find {label} column. Tried: {candidates}')


def _underlying_compute_rad_flag(df, mode='combined'):
    mode = str(mode).lower()
    if mode in ('existing', 'precomputed') and 'rad_flagged' in df.columns:
        return df['rad_flagged'].fillna(0).astype(int)
    if mode in ('r1', 'reader1', 'reader_1'):
        return (df['r1'].astype(str) == 'DISCUSSION').astype(int)
    if mode in ('r2', 'reader2', 'reader_2'):
        return (df['r2'].astype(str) == 'DISCUSSION').astype(int)
    if mode in ('combined', 'double', 'any', 'r1_or_r2'):
        return ((df['r1'].astype(str) == 'DISCUSSION') | (df['r2'].astype(str) == 'DISCUSSION')).astype(int)
    raise RuntimeError("UNDERLYING_RAD_MODE must be one of: 'r1', 'r2', 'combined', or 'existing'.")


def _underlying_load_raw_from_path():
    data_path_value = globals().get('DATA_PATH', '../../data_vaib/project_starting_dataset.csv')
    data_path = Path(data_path_value)
    if not data_path.is_absolute():
        data_path = Path.cwd() / data_path
    if not data_path.exists():
        raise RuntimeError(
            'Could not find DATA_PATH. Run the configuration cell that defines DATA_PATH, '
            'or set DATA_PATH before running this cell.'
        )
    return pd.read_csv(data_path, low_memory=False)


def _underlying_balance_controls_to_ratio_monthly(df_raw, ratio=100, random_state=44):
    d = df_raw.copy()
    hospital_col = _underlying_pick_col(
        d,
        globals().get('HOSPITAL_COL', None),
        ['imp_performing_clinic', 'hospital', 'institution_name', 'performing_clinic'],
        'hospital',
    )
    manufacturer_col = _underlying_pick_col(
        d,
        globals().get('MANUFACTURER_COL', None),
        ['manufacturer', 'vendor', 'manufacturer_name'],
        'manufacturer',
    )
    date_col = _underlying_pick_col(
        d,
        globals().get('DATE_COL', None),
        ['study_date', 'date', 'exam_date', 'examination_date', 'RISRecord_examination_date'],
        'date',
    )
    cancer_col = _underlying_pick_col(
        d,
        'has_cancer' if 'has_cancer' in d.columns else globals().get('CANCER_LABEL_COL', None),
        ['has_cancer', 'CANCER_36_months_within_exam', 'cancer_36_months', 'cancer_case'],
        'cancer',
    )

    d[date_col] = pd.to_datetime(d[date_col], format='mixed', errors='coerce')
    d = d[d[date_col].notna()].copy()
    d['has_cancer'] = pd.to_numeric(d[cancer_col], errors='coerce').fillna(0).astype(int)
    d['month_key'] = d[date_col].dt.to_period('M').astype(str)

    balanced = []
    for _, group in d.groupby([hospital_col, manufacturer_col, 'month_key']):
        cases = group[group['has_cancer'] == 1]
        controls = group[group['has_cancer'] == 0]
        n_cases = len(cases)
        if n_cases == 0:
            continue
        target_controls = int(n_cases * ratio)
        if len(controls) == 0:
            balanced.append(cases)
            continue
        replace = len(controls) < target_controls
        controls_sample = controls.sample(n=target_controls, replace=replace, random_state=random_state)
        balanced.append(pd.concat([cases, controls_sample], ignore_index=True))

    if len(balanced) == 0:
        return d.iloc[0:0].copy()
    return pd.concat(balanced, ignore_index=True)


def _underlying_load_source_data():
    mode = str(UNDERLYING_DATA_MODE).lower().strip()
    if mode in ('balanced_part3', 'simulation_input', 'balanced'):
        if 'df_part3' in globals():
            return df_part3.copy(), 'balanced_part3_simulation_input'
        raw = _underlying_load_raw_from_path()
        balanced = _underlying_balance_controls_to_ratio_monthly(
            raw,
            ratio=int(globals().get('UNDERLYING_BALANCE_CONTROLS_PER_CANCER', 100)),
            random_state=int(globals().get('RANDOM_SEED', 44)),
        )
        return balanced, 'balanced_part3_recreated_from_DATA_PATH'

    if mode in ('raw_source', 'raw', 'source'):
        if 'df_part3_raw' in globals():
            return df_part3_raw.copy(), 'raw_part3_source_before_case_control_sampling'
        return _underlying_load_raw_from_path(), 'raw_source_loaded_from_DATA_PATH'

    raise RuntimeError("UNDERLYING_DATA_MODE must be 'balanced_part3' or 'raw_source'.")

def _underlying_prepare_data(df_source):
    d = df_source.copy()
    hospital_col = _underlying_pick_col(
        d,
        globals().get('HOSPITAL_COL', None),
        ['imp_performing_clinic', 'hospital', 'institution_name', 'performing_clinic'],
        'hospital',
    )
    date_col = _underlying_pick_col(
        d,
        globals().get('DATE_COL', None),
        ['study_date', 'date', 'exam_date', 'examination_date', 'RISRecord_examination_date'],
        'date',
    )
    cancer_col = _underlying_pick_col(
        d,
        'has_cancer' if 'has_cancer' in d.columns else globals().get('CANCER_LABEL_COL', None),
        ['has_cancer', 'CANCER_36_months_within_exam', 'cancer_36_months', 'cancer_case'],
        'cancer',
    )

    d[date_col] = pd.to_datetime(d[date_col], format='mixed', errors='coerce')
    d = d[d[date_col].notna()].copy()
    d = d[d[hospital_col].notna()].copy()
    d['has_cancer_underlying'] = pd.to_numeric(d[cancer_col], errors='coerce').fillna(0).astype(int)

    if 'time_based_screen' in d.columns:
        d['screen_detected_underlying'] = d['time_based_screen'].fillna(False).astype(bool)
        screen_source = 'existing time_based_screen column'
    else:
        if 'days_to_diagnosis_pos' not in d.columns:
            if 'days_to_diagnosis' in d.columns:
                d['days_to_diagnosis_pos'] = d['days_to_diagnosis'].apply(_underlying_extract_min_positive_days)
            else:
                d['days_to_diagnosis_pos'] = np.nan
        if d['days_to_diagnosis_pos'].notna().any():
            d['screen_detected_underlying'] = (
                (d['has_cancer_underlying'] == 1)
                & (d['days_to_diagnosis_pos'].notna())
                & (d['days_to_diagnosis_pos'] <= UNDERLYING_SCREEN_THRESHOLD_DAYS)
            )
            screen_source = f'has_cancer and days_to_diagnosis <= {UNDERLYING_SCREEN_THRESHOLD_DAYS}'
        else:
            d['screen_detected_underlying'] = d['has_cancer_underlying'] == 1
            screen_source = 'has_cancer fallback; days_to_diagnosis unavailable'

    d['rad_flagged_underlying'] = _underlying_compute_rad_flag(d, mode=UNDERLYING_RAD_MODE)
    d = d.sort_values([hospital_col, date_col], kind='mergesort').reset_index(drop=True)
    return d, {
        'hospital_col': hospital_col,
        'date_col': date_col,
        'cancer_col': cancer_col,
        'screen_source': screen_source,
    }


def _underlying_moving_windows_for_hospital(df_h, date_col):
    d = df_h.sort_values(date_col, kind='mergesort').reset_index(drop=True)
    n = len(d)
    if n < UNDERLYING_WINDOW_EXAMS:
        return pd.DataFrame()

    starts = list(range(0, n - UNDERLYING_WINDOW_EXAMS + 1, UNDERLYING_STEP_EXAMS))
    last_start = n - UNDERLYING_WINDOW_EXAMS
    if starts[-1] != last_start:
        starts.append(last_start)

    cancer = d['has_cancer_underlying'].to_numpy(dtype=int)
    screen = d['screen_detected_underlying'].to_numpy(dtype=bool)
    rad = d['rad_flagged_underlying'].to_numpy(dtype=int)
    dates = d[date_col].reset_index(drop=True)

    rows = []
    for window_id, s in enumerate(starts):
        e = int(s + UNDERLYING_WINDOW_EXAMS)
        c = cancer[s:e] == 1
        non_c = ~c
        r = rad[s:e] == 1
        sc = screen[s:e]
        n_window = int(e - s)
        n_cancers = int(c.sum())
        n_non_cancers = int(non_c.sum())
        n_screen = int(sc.sum())
        n_flagged = int(r.sum())

        sens = 100.0 * int((r & c).sum()) / n_cancers if n_cancers > 0 else np.nan
        spec = 100.0 * int(((~r) & non_c).sum()) / n_non_cancers if n_non_cancers > 0 else np.nan
        cdr_per_1000 = 1000.0 * n_screen / n_window if n_window > 0 else np.nan
        flag_per_1000 = 1000.0 * n_flagged / n_window if n_window > 0 else np.nan

        rows.append({
            'window_id': int(window_id),
            'start_exam_index': int(s),
            'end_exam_index_exclusive': int(e),
            'center_exam_index': float((s + e - 1) / 2),
            'window_start_date': dates.iloc[s],
            'window_center_date': dates.iloc[int((s + e - 1) // 2)],
            'window_end_date': dates.iloc[e - 1],
            'n_exams': n_window,
            'n_cancers': n_cancers,
            'n_screen_detected_cancers': n_screen,
            'n_radiologist_flagged': n_flagged,
            'radiologist_sensitivity_percent': sens,
            'radiologist_specificity_percent': spec,
            'cancer_detection_rate_per_1000': cdr_per_1000,
            'radiologist_flagging_rate_per_1000': flag_per_1000,
        })

    return pd.DataFrame(rows)


def _underlying_format_date_range(windows):
    if len(windows) == 0:
        return ''
    start = pd.to_datetime(windows['window_start_date'].iloc[0]).strftime('%Y-%m-%d')
    end = pd.to_datetime(windows['window_end_date'].iloc[0]).strftime('%Y-%m-%d')
    return f'{start} to {end}'


def _underlying_plot_sens_spec(windows, hospital, data_source_label, out_path):
    fig, ax_sens = plt.subplots(figsize=(14, 5.4), constrained_layout=False)
    ax_spec = ax_sens.twinx()
    x = pd.to_datetime(windows['window_center_date'])

    line_sens, = ax_sens.plot(
        x,
        windows['radiologist_sensitivity_percent'],
        color='#1f77b4',
        lw=2.0,
        marker='o',
        ms=3.5,
        label='Radiologist sensitivity',
    )
    line_spec, = ax_spec.plot(
        x,
        windows['radiologist_specificity_percent'],
        color='#d62728',
        lw=2.0,
        marker='s',
        ms=3.5,
        label='Radiologist specificity',
    )

    first = windows.iloc[0]
    ax_sens.axvspan(first['window_start_date'], first['window_end_date'], color='0.85', alpha=0.45, label='Initial 20,000-exam target window')
    if pd.notna(first['radiologist_sensitivity_percent']):
        ax_sens.axhline(first['radiologist_sensitivity_percent'], color='#1f77b4', ls=':', lw=1.5, alpha=0.9, label='Initial sensitivity')
    if pd.notna(first['radiologist_specificity_percent']):
        ax_spec.axhline(first['radiologist_specificity_percent'], color='#d62728', ls=':', lw=1.5, alpha=0.9, label='Initial specificity')

    ax_sens.set_title(f'{hospital}: radiologist sensitivity and specificity over time', fontsize=15, fontweight='bold', pad=12)
    ax_sens.set_xlabel('Time (20,000-exam moving window center)', fontsize=11)
    ax_sens.set_ylabel('Sensitivity (%)', color='#1f77b4', fontsize=11)
    ax_spec.set_ylabel('Specificity (%)', color='#d62728', fontsize=11)
    ax_sens.tick_params(axis='y', labelcolor='#1f77b4')
    ax_spec.tick_params(axis='y', labelcolor='#d62728')
    ax_sens.grid(axis='both', alpha=0.25)

    sens_vals = pd.to_numeric(windows['radiologist_sensitivity_percent'], errors='coerce').dropna()
    spec_vals = pd.to_numeric(windows['radiologist_specificity_percent'], errors='coerce').dropna()
    if len(sens_vals):
        ax_sens.set_ylim(max(0, sens_vals.min() - 10), min(100, sens_vals.max() + 10))
    if len(spec_vals):
        ax_spec.set_ylim(max(0, spec_vals.min() - 3), min(100, spec_vals.max() + 3))

    note_handles = [
        Line2D([], [], color='none', label=f'Window: {UNDERLYING_WINDOW_EXAMS:,} exams; step: {UNDERLYING_STEP_EXAMS:,} exams'),
        Line2D([], [], color='none', label=f'Initial target window: {_underlying_format_date_range(windows)}'),
        Line2D([], [], color='none', label=f'Radiologist mode: {UNDERLYING_RAD_MODE}'),
        Line2D([], [], color='none', label='No AI scores or AI thresholds used'),
        Line2D([], [], color='none', label=f'Data source: {data_source_label}'),
    ]
    handles1, labels1 = ax_sens.get_legend_handles_labels()
    handles2, labels2 = ax_spec.get_legend_handles_labels()
    handles = handles1 + handles2 + note_handles
    labels = labels1 + labels2 + [h.get_label() for h in note_handles]
    ax_sens.legend(handles, labels, loc='center left', bbox_to_anchor=(1.08, 0.5), fontsize=9, title='Figure notes', title_fontsize=10, frameon=True)
    fig.autofmt_xdate(rotation=30, ha='right')
    fig.subplots_adjust(left=0.08, right=0.74, bottom=0.18, top=0.88)
    fig.savefig(out_path, dpi=220, bbox_inches='tight', facecolor='white')
    if UNDERLYING_SHOW_PLOTS:
        plt.show()
    else:
        plt.close(fig)


def _underlying_plot_cdr_flagging(windows, hospital, data_source_label, screen_source, out_path):
    fig, ax_cdr = plt.subplots(figsize=(14, 5.4), constrained_layout=False)
    ax_flag = ax_cdr.twinx()
    x = pd.to_datetime(windows['window_center_date'])

    line_cdr, = ax_cdr.plot(
        x,
        windows['cancer_detection_rate_per_1000'],
        color='#2ca02c',
        lw=2.0,
        marker='o',
        ms=3.5,
        label='Cancer detection rate',
    )
    line_flag, = ax_flag.plot(
        x,
        windows['radiologist_flagging_rate_per_1000'],
        color='#9467bd',
        lw=2.0,
        marker='s',
        ms=3.5,
        label='Radiologist flagging rate',
    )

    first = windows.iloc[0]
    ax_cdr.axvspan(first['window_start_date'], first['window_end_date'], color='0.85', alpha=0.45, label='Initial 20,000-exam target window')
    if pd.notna(first['cancer_detection_rate_per_1000']):
        ax_cdr.axhline(first['cancer_detection_rate_per_1000'], color='#2ca02c', ls=':', lw=1.5, alpha=0.9, label='Initial cancer detection rate')
    if pd.notna(first['radiologist_flagging_rate_per_1000']):
        ax_flag.axhline(first['radiologist_flagging_rate_per_1000'], color='#9467bd', ls=':', lw=1.5, alpha=0.9, label='Initial flagging rate')

    ax_cdr.set_title(f'{hospital}: cancer detection and radiologist flagging rates over time', fontsize=15, fontweight='bold', pad=12)
    ax_cdr.set_xlabel('Time (20,000-exam moving window center)', fontsize=11)
    ax_cdr.set_ylabel('Cancer detection rate per 1,000 exams', color='#2ca02c', fontsize=11)
    ax_flag.set_ylabel('Radiologist flagging rate per 1,000 exams', color='#9467bd', fontsize=11)
    ax_cdr.tick_params(axis='y', labelcolor='#2ca02c')
    ax_flag.tick_params(axis='y', labelcolor='#9467bd')
    ax_cdr.grid(axis='both', alpha=0.25)

    cdr_vals = pd.to_numeric(windows['cancer_detection_rate_per_1000'], errors='coerce').dropna()
    flag_vals = pd.to_numeric(windows['radiologist_flagging_rate_per_1000'], errors='coerce').dropna()
    if len(cdr_vals):
        ax_cdr.set_ylim(0, max(1.0, cdr_vals.max() * 1.25))
    if len(flag_vals):
        ax_flag.set_ylim(0, max(1.0, flag_vals.max() * 1.15))

    note_handles = [
        Line2D([], [], color='none', label=f'Window: {UNDERLYING_WINDOW_EXAMS:,} exams; step: {UNDERLYING_STEP_EXAMS:,} exams'),
        Line2D([], [], color='none', label=f'Initial target window: {_underlying_format_date_range(windows)}'),
        Line2D([], [], color='none', label=f'Cancer detection source: {screen_source}'),
        Line2D([], [], color='none', label=f'Radiologist mode: {UNDERLYING_RAD_MODE}'),
        Line2D([], [], color='none', label='No AI scores or AI thresholds used'),
        Line2D([], [], color='none', label=f'Data source: {data_source_label}'),
    ]
    handles1, labels1 = ax_cdr.get_legend_handles_labels()
    handles2, labels2 = ax_flag.get_legend_handles_labels()
    handles = handles1 + handles2 + note_handles
    labels = labels1 + labels2 + [h.get_label() for h in note_handles]
    ax_cdr.legend(handles, labels, loc='center left', bbox_to_anchor=(1.08, 0.5), fontsize=9, title='Figure notes', title_fontsize=10, frameon=True)
    fig.autofmt_xdate(rotation=30, ha='right')
    fig.subplots_adjust(left=0.08, right=0.74, bottom=0.18, top=0.88)
    fig.savefig(out_path, dpi=220, bbox_inches='tight', facecolor='white')
    if UNDERLYING_SHOW_PLOTS:
        plt.show()
    else:
        plt.close(fig)


# -----------------------------
# Build all hospital diagnostics
# -----------------------------
source_df, data_source_label = _underlying_load_source_data()
underlying_df, underlying_cols = _underlying_prepare_data(source_df)
hospital_col = underlying_cols['hospital_col']
date_col = underlying_cols['date_col']
screen_source = underlying_cols['screen_source']

hospital_filter = globals().get('UNDERLYING_HOSPITALS', None)
if hospital_filter is None:
    hospitals = sorted(underlying_df[hospital_col].dropna().astype(str).unique())
else:
    hospitals = [str(h) for h in hospital_filter]

all_window_rows = []
figure_rows = []
skipped_rows = []

for hospital in hospitals:
    df_h = underlying_df[underlying_df[hospital_col].astype(str) == str(hospital)].copy()
    n_h = len(df_h)
    if n_h < UNDERLYING_MIN_EXAMS:
        skipped_rows.append({'hospital': hospital, 'n_exams': n_h, 'reason': f'fewer than {UNDERLYING_MIN_EXAMS:,} exams'})
        continue

    windows = _underlying_moving_windows_for_hospital(df_h, date_col=date_col)
    if len(windows) == 0:
        skipped_rows.append({'hospital': hospital, 'n_exams': n_h, 'reason': 'no complete moving window'})
        continue

    windows.insert(0, 'hospital', hospital)
    windows.insert(1, 'data_source', data_source_label)
    all_window_rows.append(windows)

    slug = _underlying_slug(hospital)
    graph1_png = UNDERLYING_OUTPUT_DIR / f'underlying_{slug}_graph1_sensitivity_specificity_moving_{UNDERLYING_WINDOW_EXAMS}.png'
    graph2_png = UNDERLYING_OUTPUT_DIR / f'underlying_{slug}_graph2_cancer_detection_flagging_moving_{UNDERLYING_WINDOW_EXAMS}.png'

    _underlying_plot_sens_spec(windows, hospital, data_source_label, graph1_png)
    _underlying_plot_cdr_flagging(windows, hospital, data_source_label, screen_source, graph2_png)

    figure_rows.append({
        'hospital': hospital,
        'n_exams': n_h,
        'n_windows': len(windows),
        'initial_window_start': windows['window_start_date'].iloc[0],
        'initial_window_end': windows['window_end_date'].iloc[0],
        'initial_sensitivity_percent': windows['radiologist_sensitivity_percent'].iloc[0],
        'initial_specificity_percent': windows['radiologist_specificity_percent'].iloc[0],
        'initial_cancer_detection_rate_per_1000': windows['cancer_detection_rate_per_1000'].iloc[0],
        'initial_radiologist_flagging_rate_per_1000': windows['radiologist_flagging_rate_per_1000'].iloc[0],
        'graph1_sensitivity_specificity_png': str(graph1_png),
        'graph2_cancer_detection_flagging_png': str(graph2_png),
    })

underlying_moving_window_summary = pd.concat(all_window_rows, ignore_index=True) if all_window_rows else pd.DataFrame()
underlying_figure_index = pd.DataFrame(figure_rows)
underlying_skipped_hospitals = pd.DataFrame(skipped_rows)

summary_csv = UNDERLYING_OUTPUT_DIR / f'underlying_hospital_moving_{UNDERLYING_WINDOW_EXAMS}_window_summary.csv'
figure_index_csv = UNDERLYING_OUTPUT_DIR / 'underlying_hospital_figure_index.csv'
skipped_csv = UNDERLYING_OUTPUT_DIR / 'underlying_hospital_skipped.csv'

underlying_moving_window_summary.to_csv(summary_csv, index=False)
underlying_figure_index.to_csv(figure_index_csv, index=False)
underlying_skipped_hospitals.to_csv(skipped_csv, index=False)

print('Underlying hospital diagnostics complete.')
print('Data mode:', UNDERLYING_DATA_MODE)
print('Data source:', data_source_label)
print('Hospital column:', hospital_col)
print('Date column:', date_col)
print('Radiologist mode:', UNDERLYING_RAD_MODE)
print('Screen/cancer detection source:', screen_source)
print('Window exams:', UNDERLYING_WINDOW_EXAMS, '| Step exams:', UNDERLYING_STEP_EXAMS)
print('Hospitals plotted:', len(underlying_figure_index))
print('Hospitals skipped:', len(underlying_skipped_hospitals))
print('Saved moving-window summary:', summary_csv)
print('Saved figure index:', figure_index_csv)
print('Saved skipped-hospital table:', skipped_csv)
print('Saved PNGs in:', UNDERLYING_OUTPUT_DIR)

try:
    display(underlying_figure_index)
    if len(underlying_skipped_hospitals) > 0:
        display(underlying_skipped_hospitals)
except Exception:
    pass


In [ ]:
# Part 3D.6I - Cancer detection rate + relative cancer detection rate figures
#
# This cell adds Fredrik's relative screen-detection metric while keeping the
# existing cancer detection rate metric:
#
#   cancer_detection_rate = 1,000 * (# AI-flagged screen-detected cancers) / (# exams)
#
#   relative_cancer_detection_rate =
#       (# AI-flagged screen-detected cancers)
#       / (# radiologist-flagged screen-detected cancers)
#
# The second metric follows the same logic as relative flagging rate, but it is
# restricted to screen-detected cancers. Both metrics are theoretical because
# they require delayed cancer outcome information.

import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.lines import Line2D

required_names = [
    'df_part3', '_p3d_get_columns', '_p3d_prepare_pair_df', '_p3d_add_rad_flag',
    '_p3d_decision_end_indices', '_p3d_rate_stats_from_flags', '_p3d_rate_stats_from_threshold',
    '_p3d_choose_initial_threshold_eval_targets', '_p3d_time_weighted_mean', '_p3d_two_prop_pvalue',
    '_p3d_candidate_thresholds'
]
missing = [name for name in required_names if name not in globals()]
if missing:
    raise RuntimeError(
        'Run Part 3 data setup, Part 3 helper functions, and Part 3D.0 before this cell. Missing: '
        + ', '.join(missing)
    )

base_out_dir = Path(globals().get(
    'P3D_OUTPUT_DIR',
    Path('outputs/2025_12_29_counterfactual_manufacturer_swap_setup/part3d_oracle_recalibration')
))
cdr_source_dir = base_out_dir.parent / 'part3d_oracle_recalibration_with_cancer_detection_rate_theoretical'
combined_out_dir = base_out_dir.parent / 'part3d_oracle_recalibration_with_cancer_detection_and_relative_screen_theoretical'
combined_out_dir.mkdir(parents=True, exist_ok=True)

cols = _p3d_get_columns(df_part3)
ai_systems = cols['ai_cols'] if globals().get('P3D_AI_SYSTEMS', None) is None else [
    a for a in globals().get('P3D_AI_SYSTEMS', []) if a in cols['ai_cols']
]

prior_size = int(globals().get('P3D_POLICY_PRIOR_SIZE', 20000))
current_size = int(globals().get('P3D_POLICY_CURRENT_SIZE', 20000))
future_size = int(globals().get('P3D_POLICY_FUTURE_SIZE', 20000))
step_size = int(globals().get('P3D_POLICY_STEP', 2000))
gap_size = int(globals().get('P3D_POLICY_GAP', step_size))
rad_mode = str(globals().get('P3D_RAD_MODE', 'single'))
max_candidates = int(globals().get('P3D_MAX_THRESHOLD_CANDIDATES', 101))
signal_p_thr = float(globals().get('P3D_SIGNAL_P_THRESHOLD', 0.05))
min_recal_interval = int(globals().get('P3D_MIN_RECALIBRATION_INTERVAL', 2000))
max_points = globals().get('P3D_MAX_DECISION_POINTS', None)

if 'time_based_screen' not in df_part3.columns:
    if 'days_to_diagnosis_pos' not in df_part3.columns:
        if 'extract_min_positive_days' in globals() and 'days_to_diagnosis' in df_part3.columns:
            df_part3['days_to_diagnosis_pos'] = df_part3['days_to_diagnosis'].apply(extract_min_positive_days)
        else:
            raise RuntimeError('Cannot create time_based_screen: missing days_to_diagnosis/extract_min_positive_days.')
    df_part3['time_based_screen'] = (
        (df_part3[cols['cancer']].fillna(0).astype(int) == 1)
        & (df_part3['days_to_diagnosis_pos'].notna())
        & (df_part3['days_to_diagnosis_pos'] <= int(globals().get('SCREEN_THRESHOLD_DAYS', 90)))
    )

policy_order = ['without_recalibration', 'recalibrate_every_step', 'recalibrate_significant_only']
policy_label = {
    'without_recalibration': 'No recalibration',
    'recalibrate_every_step': 'Recalibrate every step',
    'recalibrate_significant_only': 'Recalibrate if significant',
}
colors = {
    'without_recalibration': '#8da0cb',
    'recalibrate_every_step': '#fc8d62',
    'recalibrate_significant_only': '#66c2a5',
}
metric_order = [
    'flagging_rate',
    'relative_flagging_rate',
    'p_ai_given_rad',
    'p_rad_given_ai',
    'cancer_detection_rate',
    'relative_cancer_detection_rate',
]
metric_label = {
    'flagging_rate': 'Flagging rate',
    'relative_flagging_rate': 'Relative\nflagging rate',
    'p_ai_given_rad': 'P(AI flagged |\nradiologist flagged)',
    'p_rad_given_ai': 'P(radiologist flagged |\nAI flagged)',
    'cancer_detection_rate': 'Cancer detection\nrate',
    'relative_cancer_detection_rate': 'Relative cancer\ndetection rate',
}


def _p3d_relative_cdr_signal_stats_from_threshold(window_df, ai_col, thr, screen_col='time_based_screen'):
    scores = pd.to_numeric(window_df[ai_col], errors='coerce').to_numpy(dtype=float)
    screen = window_df[screen_col].fillna(False).astype(bool).to_numpy()
    rad = window_df['rad_flagged'].fillna(0).astype(int).to_numpy() == 1
    valid = ~np.isnan(scores)
    scores = scores[valid]
    screen = screen[valid]
    rad = rad[valid]

    n_total = int(len(scores))
    screen_n = int(screen.sum())
    rad_screen_pos = int((rad & screen).sum())
    if n_total == 0 or pd.isna(thr):
        return {
            'relative_cancer_detection_rate': np.nan,
            'relative_screen_detection_rate': np.nan,
            'rel_screen_n_total': n_total,
            'rel_screen_n': screen_n,
            'rel_ai_screen_pos': 0,
            'rel_rad_screen_pos': rad_screen_pos,
        }

    ai = scores >= float(thr)
    ai_screen_pos = int((ai & screen).sum())
    relative_rate = np.nan if rad_screen_pos <= 0 else float(ai_screen_pos) / float(rad_screen_pos)
    return {
        'relative_cancer_detection_rate': float(relative_rate) if pd.notna(relative_rate) else np.nan,
        'relative_screen_detection_rate': float(relative_rate) if pd.notna(relative_rate) else np.nan,
        'rel_screen_n_total': n_total,
        'rel_screen_n': screen_n,
        'rel_ai_screen_pos': ai_screen_pos,
        'rel_rad_screen_pos': rad_screen_pos,
    }


def _p3d_relative_cdr_signal_pvalue(prior_sig, curr_sig):
    # For significance gating, compare the AI-flagged fraction among screen-detected cancers.
    # This avoids using the count-ratio denominator as a binomial denominator.
    return _p3d_two_prop_pvalue(
        curr_sig.get('rel_ai_screen_pos', 0), curr_sig.get('rel_screen_n', 0),
        prior_sig.get('rel_ai_screen_pos', 0), prior_sig.get('rel_screen_n', 0),
    )


def _p3d_choose_threshold_for_relative_cdr(window_df, ai_col, target_rate, screen_col='time_based_screen', max_candidates=101):
    if pd.isna(target_rate):
        return np.nan, np.nan

    scores_all = pd.to_numeric(window_df[ai_col], errors='coerce').to_numpy(dtype=float)
    thresholds = _p3d_candidate_thresholds(scores_all, max_candidates=max_candidates)
    if len(thresholds) == 0:
        return np.nan, np.nan

    obj = []
    for thr in thresholds:
        sig = _p3d_relative_cdr_signal_stats_from_threshold(window_df, ai_col, thr, screen_col=screen_col)
        val = sig.get('relative_cancer_detection_rate', np.nan)
        obj.append(np.inf if pd.isna(val) else abs(float(val) - float(target_rate)))

    obj = np.asarray(obj, dtype=float)
    if np.isinf(obj).all():
        return np.nan, np.nan
    best_idx = int(np.argmin(obj))
    return float(thresholds[best_idx]), float(obj[best_idx])


def _p3d_run_diary_relative_cancer_detection_rate(
    df_pair,
    ai_col,
    cancer_col,
    date_col,
    prior_size,
    current_size,
    future_size,
    gap_size,
    step_size,
    policy_name,
    signal_p_threshold=0.05,
    min_recalibration_interval=2000,
    max_threshold_candidates=101,
    max_points=None,
):
    d = df_pair.reset_index(drop=True)
    n = len(d)
    ends = _p3d_decision_end_indices(
        n=n,
        prior_size=prior_size,
        current_size=current_size,
        gap_size=gap_size,
        step_size=step_size,
        future_size=future_size,
        max_points=max_points,
    )
    if len(ends) == 0:
        return pd.DataFrame(), {}, {'reason': 'insufficient_exams'}

    t0 = int(ends[0])
    t0_current_start = t0 - current_size
    t0_prior_end = t0_current_start - gap_size
    t0_prior_start = t0_prior_end - prior_size
    prior0 = d.iloc[t0_prior_start:t0_prior_end]

    target_stats = _p3d_rate_stats_from_flags(prior0['rad_flagged'].to_numpy(), prior0[cancer_col].to_numpy())
    targets = {
        'sens': float(target_stats['sens']),
        'spec': float(target_stats['spec']),
        'ppv': float(target_stats['ppv']),
    }

    init_thr, _ = _p3d_choose_initial_threshold_eval_targets(
        prior0,
        ai_col=ai_col,
        cancer_col=cancer_col,
        targets=targets,
        max_candidates=max_threshold_candidates,
    )
    if pd.isna(init_thr):
        init_thr = float(pd.to_numeric(prior0[ai_col], errors='coerce').median())

    signal_target_stats = _p3d_relative_cdr_signal_stats_from_threshold(prior0, ai_col, init_thr)
    target_signal_value = signal_target_stats.get('relative_cancer_detection_rate', np.nan)

    rows = []
    current_thr = init_thr
    last_threshold_change_end_idx = None

    for i, end_idx in enumerate(ends):
        end_idx = int(end_idx)
        current_start = end_idx - current_size
        prior_end = current_start - gap_size
        prior_start = prior_end - prior_size
        future_start = end_idx
        future_end = end_idx + future_size

        prior_w = d.iloc[prior_start:prior_end]
        curr_w = d.iloc[current_start:end_idx]
        future_w = d.iloc[future_start:future_end]

        prior_perf = _p3d_rate_stats_from_threshold(prior_w[ai_col].to_numpy(), prior_w[cancer_col].to_numpy(), current_thr)
        curr_perf = _p3d_rate_stats_from_threshold(curr_w[ai_col].to_numpy(), curr_w[cancer_col].to_numpy(), current_thr)
        prior_sig = _p3d_relative_cdr_signal_stats_from_threshold(prior_w, ai_col, current_thr)
        curr_sig = _p3d_relative_cdr_signal_stats_from_threshold(curr_w, ai_col, current_thr)

        current_signal_value = curr_sig.get('relative_cancer_detection_rate', np.nan)
        drift = np.nan if (pd.isna(target_signal_value) or pd.isna(current_signal_value)) else abs(float(current_signal_value) - float(target_signal_value))
        p_signal = _p3d_relative_cdr_signal_pvalue(prior_sig, curr_sig)

        exams_since_last_change = np.nan if last_threshold_change_end_idx is None else float(end_idx - last_threshold_change_end_idx)
        cadence_ok = bool(
            last_threshold_change_end_idx is None or
            exams_since_last_change >= float(min_recalibration_interval)
        )

        if policy_name == 'without_recalibration':
            signal_detected = False
            recalibrate = False
        elif policy_name == 'recalibrate_every_step':
            signal_detected = bool(pd.notna(drift))
            recalibrate = bool(signal_detected and cadence_ok)
        elif policy_name == 'recalibrate_significant_only':
            signal_detected = bool(pd.notna(p_signal) and p_signal < float(signal_p_threshold))
            recalibrate = bool(signal_detected and pd.notna(drift) and drift > 0.0 and cadence_ok)
        else:
            raise ValueError(f'Unsupported policy_name: {policy_name}')

        new_thr = current_thr
        best_obj = np.nan
        if recalibrate:
            new_thr, best_obj = _p3d_choose_threshold_for_relative_cdr(
                curr_w,
                ai_col=ai_col,
                target_rate=target_signal_value,
                max_candidates=max_threshold_candidates,
            )
            if pd.isna(new_thr):
                new_thr = current_thr

        threshold_changed = int(pd.notna(current_thr) and pd.notna(new_thr) and (abs(new_thr - current_thr) > 1e-12))
        if threshold_changed:
            last_threshold_change_end_idx = end_idx

        fut_perf = _p3d_rate_stats_from_threshold(future_w[ai_col].to_numpy(), future_w[cancer_col].to_numpy(), new_thr)
        decision_date = d.iloc[end_idx][date_col] if date_col in d.columns and end_idx < len(d) else pd.NaT

        rows.append({
            'decision_idx': i,
            'decision_end_exam_idx': end_idx,
            'decision_date': decision_date,
            'objective_metric': 'relative_cancer_detection_rate',
            'signal_metric': 'relative_cancer_detection_rate',
            'ai_system': ai_col,
            'prior_size': prior_size,
            'current_size': current_size,
            'future_size': future_size,
            'gap_size': gap_size,
            'step_size': step_size,
            'prior_start_idx': prior_start,
            'prior_end_idx': prior_end,
            'current_start_idx': current_start,
            'current_end_idx': end_idx,
            'future_start_idx': future_start,
            'future_end_idx': future_end,
            'target_sens': targets['sens'],
            'target_spec': targets['spec'],
            'target_ppv': targets['ppv'],
            'threshold_before': current_thr,
            'threshold_after': new_thr,
            'threshold_changed': threshold_changed,
            'drift_value': drift,
            'signal_mode': policy_name,
            'signal_p_threshold': float(signal_p_threshold),
            'min_recalibration_interval': float(min_recalibration_interval),
            'exams_since_last_change': exams_since_last_change,
            'cadence_ok': int(cadence_ok),
            'p_value_signal': p_signal,
            'signal_detected': int(signal_detected),
            'recalibrated': int(recalibrate),
            'best_obj_value': best_obj,
            'prior_sens_before': prior_perf['sens'],
            'prior_spec_before': prior_perf['spec'],
            'prior_ppv_before': prior_perf['ppv'],
            'current_sens_before': curr_perf['sens'],
            'current_spec_before': curr_perf['spec'],
            'current_ppv_before': curr_perf['ppv'],
            'target_signal_value': target_signal_value,
            'prior_signal_value': prior_sig.get('relative_cancer_detection_rate', np.nan),
            'current_signal_value': current_signal_value,
            'future_sens_after': fut_perf['sens'],
            'future_spec_after': fut_perf['spec'],
            'future_ppv_after': fut_perf['ppv'],
            'future_n': fut_perf['n_total'],
            'prior_ai_screen_pos': prior_sig.get('rel_ai_screen_pos', 0),
            'prior_rad_screen_pos': prior_sig.get('rel_rad_screen_pos', 0),
            'current_ai_screen_pos': curr_sig.get('rel_ai_screen_pos', 0),
            'current_rad_screen_pos': curr_sig.get('rel_rad_screen_pos', 0),
        })

        current_thr = new_thr

    diary = pd.DataFrame(rows)
    if len(diary) == 0:
        return diary, targets, {'reason': 'empty_diary'}

    seg_start = diary['decision_end_exam_idx'].astype(int).to_numpy()
    seg_end = np.r_[seg_start[1:], [len(d)]]
    seg_sens, seg_spec, seg_ppv, seg_n = [], [], [], []
    for s, e, thr in zip(seg_start, seg_end, diary['threshold_after'].to_numpy(dtype=float)):
        seg = d.iloc[int(s):int(e)]
        perf = _p3d_rate_stats_from_threshold(seg[ai_col].to_numpy(), seg[cancer_col].to_numpy(), thr)
        seg_sens.append(perf['sens'])
        seg_spec.append(perf['spec'])
        seg_ppv.append(perf['ppv'])
        seg_n.append(perf['n_total'])

    diary['segment_start_idx'] = seg_start
    diary['segment_end_idx'] = seg_end
    diary['segment_n'] = np.asarray(seg_n, dtype=float)
    diary['segment_sens'] = np.asarray(seg_sens, dtype=float)
    diary['segment_spec'] = np.asarray(seg_spec, dtype=float)
    diary['segment_ppv'] = np.asarray(seg_ppv, dtype=float)
    diary['segment_abs_delta_sens'] = np.abs(diary['segment_sens'] - diary['target_sens'])
    diary['segment_abs_delta_spec'] = np.abs(diary['segment_spec'] - diary['target_spec'])
    diary['segment_abs_delta_ppv'] = np.abs(diary['segment_ppv'] - diary['target_ppv'])

    return diary, targets, {'reason': 'ok'}


def _safe_wavg(values, weights):
    v = pd.to_numeric(values, errors='coerce')
    w = pd.to_numeric(weights, errors='coerce').fillna(0)
    mask = v.notna() & w.notna() & (w > 0)
    if mask.sum() == 0:
        return np.nan
    return float(np.average(v[mask], weights=w[mask]))


def _metric_summary_from_pair_rows(pair_summary, aggregation_label):
    metric_cols = [
        'target_sensitivity', 'target_specificity',
        'tw_sensitivity_recal', 'tw_specificity_recal',
        'tw_sensitivity_no_recal', 'tw_specificity_no_recal',
        'delta_sens_recal_minus_no', 'delta_spec_recal_minus_no',
    ]
    h_rows = []
    gcols_h = ['hospital', 'ai_system', 'signal_metric', 'policy', 'step_exams', 'gap_exams']
    for keys, g in pair_summary.groupby(gcols_h, dropna=False):
        row = {
            'hospital': keys[0],
            'ai_system': keys[1],
            'signal_metric': keys[2],
            'policy': keys[3],
            'step_exams': int(keys[4]),
            'gap_exams': int(keys[5]),
            'n_pairs': int(len(g)),
            'n_test_steps_total': int(pd.to_numeric(g['n_test_steps'], errors='coerce').fillna(0).sum()),
            'n_signal_detected_total': int(pd.to_numeric(g['n_signal_detected'], errors='coerce').fillna(0).sum()),
            'n_recalibrations_total': int(pd.to_numeric(g['n_recalibrations'], errors='coerce').fillna(0).sum()),
            'n_threshold_changes_total': int(pd.to_numeric(g['n_threshold_changes'], errors='coerce').fillna(0).sum()),
            'weights_total_exams': float(pd.to_numeric(g['weights_total_exams'], errors='coerce').fillna(0).sum()),
        }
        w = pd.to_numeric(g['weights_total_exams'], errors='coerce').fillna(0)
        for c in metric_cols:
            row[c] = _safe_wavg(g[c], w)
        row['mean_exams_between_recal'] = _safe_wavg(
            g['mean_exams_between_recal'],
            pd.to_numeric(g['n_threshold_changes'], errors='coerce').fillna(0),
        )
        row['median_exams_between_recal'] = _safe_wavg(
            g['median_exams_between_recal'],
            pd.to_numeric(g['n_threshold_changes'], errors='coerce').fillna(0),
        )
        h_rows.append(row)
    hospital_summary = pd.DataFrame(h_rows)

    o_rows = []
    gcols_o = ['ai_system', 'signal_metric', 'policy', 'step_exams', 'gap_exams']
    for keys, g in hospital_summary.groupby(gcols_o, dropna=False):
        valid = g.loc[pd.to_numeric(g['tw_sensitivity_recal'], errors='coerce').notna()].copy()
        row = {
            'ai_system': keys[0],
            'signal_metric': keys[1],
            'policy': keys[2],
            'step_exams': int(keys[3]),
            'gap_exams': int(keys[4]),
            'n_hospitals': int(valid['hospital'].nunique()) if len(valid) else 0,
            'n_test_steps_total': int(pd.to_numeric(valid['n_test_steps_total'], errors='coerce').fillna(0).sum()),
            'n_signal_detected_total': int(pd.to_numeric(valid['n_signal_detected_total'], errors='coerce').fillna(0).sum()),
            'n_recalibrations_total': int(pd.to_numeric(valid['n_recalibrations_total'], errors='coerce').fillna(0).sum()),
            'n_threshold_changes_total': int(pd.to_numeric(valid['n_threshold_changes_total'], errors='coerce').fillna(0).sum()),
            'weights_total_exams': float(pd.to_numeric(valid['weights_total_exams'], errors='coerce').fillna(0).sum()),
            'aggregation_method': aggregation_label,
        }
        for c in ['target_sensitivity', 'target_specificity', 'tw_sensitivity_recal', 'tw_specificity_recal']:
            row[c] = float(pd.to_numeric(valid[c], errors='coerce').mean()) if len(valid) else np.nan
        row['mean_exams_between_recal'] = float(pd.to_numeric(valid['mean_exams_between_recal'], errors='coerce').mean()) if len(valid) else np.nan
        row['median_exams_between_recal'] = float(pd.to_numeric(valid['median_exams_between_recal'], errors='coerce').mean()) if len(valid) else np.nan
        o_rows.append(row)
    unweighted_summary = pd.DataFrame(o_rows)
    return hospital_summary, unweighted_summary


def _load_table_from_memory_or_csv(var_name, csv_path, required=True):
    obj = globals().get(var_name)
    if isinstance(obj, pd.DataFrame) and len(obj) > 0:
        return obj.copy()
    csv_path = Path(csv_path)
    if csv_path.exists():
        return pd.read_csv(csv_path)
    if required:
        raise RuntimeError(f'Missing {var_name} and {csv_path}. Run the upstream Part 3D cells first.')
    return pd.DataFrame()


def _combine_tables(*tables):
    nonempty = [t.copy() for t in tables if isinstance(t, pd.DataFrame) and len(t) > 0]
    if len(nonempty) == 0:
        return pd.DataFrame()
    cols_all = sorted(set().union(*[set(t.columns) for t in nonempty]))
    return pd.concat([t.reindex(columns=cols_all) for t in nonempty], ignore_index=True, sort=False)


def _slug(value):
    text = str(value) if pd.notna(value) else 'missing'
    slug = ''.join(ch.lower() if ch.isalnum() else '_' for ch in text).strip('_')
    while '__' in slug:
        slug = slug.replace('__', '_')
    return slug or 'missing'


# ------------------------------------------------------------------
# Run relative cancer detection rate recalibration across hospital pairs.
# ------------------------------------------------------------------
rows = []
all_hospitals = sorted(df_part3[cols['hospital']].dropna().unique())
hospital_filter = globals().get('P3D_POLICY_HOSPITALS', None)
if hospital_filter is not None:
    all_hospitals = [h for h in all_hospitals if h in set(hospital_filter)]

for hospital in all_hospitals:
    d_h = df_part3[df_part3[cols['hospital']] == hospital].copy()
    if len(d_h) == 0:
        continue

    if cols['manufacturer'] is not None and cols['manufacturer'] in d_h.columns:
        manufacturers = sorted(d_h[cols['manufacturer']].dropna().unique())
        if len(manufacturers) == 0:
            manufacturers = [None]
    else:
        manufacturers = [None]

    mfr_filter = globals().get('P3D_POLICY_MANUFACTURERS', None)
    if mfr_filter is not None:
        manufacturers = [m for m in manufacturers if m in set(mfr_filter)]

    for manufacturer in manufacturers:
        pair_df = _p3d_prepare_pair_df(df_part3, cols, hospital, manufacturer)
        if len(pair_df) == 0:
            continue
        pair_df = _p3d_add_rad_flag(pair_df, rad_mode=rad_mode)

        for ai in ai_systems:
            for policy_name in policy_order:
                diary, targets, meta = _p3d_run_diary_relative_cancer_detection_rate(
                    df_pair=pair_df,
                    ai_col=ai,
                    cancer_col=cols['cancer'],
                    date_col=cols['date'],
                    prior_size=prior_size,
                    current_size=current_size,
                    future_size=future_size,
                    gap_size=gap_size,
                    step_size=step_size,
                    policy_name=policy_name,
                    signal_p_threshold=signal_p_thr,
                    min_recalibration_interval=min_recal_interval,
                    max_threshold_candidates=max_candidates,
                    max_points=max_points,
                )

                if diary is None or len(diary) == 0:
                    rows.append({
                        'hospital': hospital,
                        'manufacturer': manufacturer if manufacturer is not None else 'ALL',
                        'ai_system': ai,
                        'signal_metric': 'relative_cancer_detection_rate',
                        'policy': policy_name,
                        'status': meta.get('reason', 'no_data') if isinstance(meta, dict) else 'no_data',
                        'step_exams': step_size,
                        'gap_exams': gap_size,
                        'n_test_steps': 0,
                        'n_signal_detected': 0,
                        'n_recalibrations': 0,
                        'n_threshold_changes': 0,
                        'mean_exams_between_recal': np.nan,
                        'median_exams_between_recal': np.nan,
                        'weights_total_exams': 0.0,
                        'target_sensitivity': np.nan,
                        'target_specificity': np.nan,
                        'tw_sensitivity_recal': np.nan,
                        'tw_specificity_recal': np.nan,
                        'tw_sensitivity_no_recal': np.nan,
                        'tw_specificity_no_recal': np.nan,
                        'delta_sens_recal_minus_no': np.nan,
                        'delta_spec_recal_minus_no': np.nan,
                        'target_relative_cancer_detection_rate': np.nan,
                    })
                    continue

                w = pd.to_numeric(diary['segment_n'], errors='coerce').fillna(0)
                tw_sens_recal = _p3d_time_weighted_mean(diary['segment_sens'], w)
                tw_spec_recal = _p3d_time_weighted_mean(diary['segment_spec'], w)

                changes = pd.to_numeric(
                    diary.loc[pd.to_numeric(diary['threshold_changed'], errors='coerce').fillna(0) > 0, 'decision_end_exam_idx'],
                    errors='coerce'
                ).dropna().to_numpy(dtype=float)
                if len(changes) >= 2:
                    gaps_recal = np.diff(changes)
                    mean_between_recal = float(np.mean(gaps_recal))
                    median_between_recal = float(np.median(gaps_recal))
                else:
                    mean_between_recal = np.nan
                    median_between_recal = np.nan

                target_signal_series = pd.to_numeric(diary['target_signal_value'], errors='coerce').dropna()
                rows.append({
                    'hospital': hospital,
                    'manufacturer': manufacturer if manufacturer is not None else 'ALL',
                    'ai_system': ai,
                    'signal_metric': 'relative_cancer_detection_rate',
                    'policy': policy_name,
                    'status': meta.get('reason', 'ok') if isinstance(meta, dict) else 'ok',
                    'step_exams': step_size,
                    'gap_exams': gap_size,
                    'n_test_steps': int(len(diary)),
                    'n_signal_detected': int(pd.to_numeric(diary['signal_detected'], errors='coerce').fillna(0).sum()),
                    'n_recalibrations': int(pd.to_numeric(diary['recalibrated'], errors='coerce').fillna(0).sum()),
                    'n_threshold_changes': int(pd.to_numeric(diary['threshold_changed'], errors='coerce').fillna(0).sum()),
                    'mean_exams_between_recal': mean_between_recal,
                    'median_exams_between_recal': median_between_recal,
                    'weights_total_exams': float(w.sum()),
                    'target_sensitivity': float(pd.to_numeric(diary['target_sens'], errors='coerce').iloc[0]),
                    'target_specificity': float(pd.to_numeric(diary['target_spec'], errors='coerce').iloc[0]),
                    'tw_sensitivity_recal': tw_sens_recal,
                    'tw_specificity_recal': tw_spec_recal,
                    'tw_sensitivity_no_recal': np.nan,
                    'tw_specificity_no_recal': np.nan,
                    'delta_sens_recal_minus_no': np.nan,
                    'delta_spec_recal_minus_no': np.nan,
                    'target_relative_cancer_detection_rate': float(target_signal_series.iloc[0]) if len(target_signal_series) else np.nan,
                })

p3d_policy3_pair_summary_relative_cancer_detection_rate = pd.DataFrame(rows)
p3d_policy3_hospital_summary_relative_cancer_detection_rate, p3d_policy3_overall_summary_relative_cancer_detection_rate_unweighted = _metric_summary_from_pair_rows(
    p3d_policy3_pair_summary_relative_cancer_detection_rate,
    aggregation_label='unweighted_mean_of_hospital_level_metrics',
)

pair_rel_csv = combined_out_dir / 'p3d_policy3_pair_summary_relative_cancer_detection_rate.csv'
hospital_rel_csv = combined_out_dir / 'p3d_policy3_hospital_summary_relative_cancer_detection_rate.csv'
overall_rel_unweighted_csv = combined_out_dir / 'p3d_policy3_overall_summary_relative_cancer_detection_rate_unweighted_hospital_mean.csv'
p3d_policy3_pair_summary_relative_cancer_detection_rate.sort_values(['hospital', 'manufacturer', 'ai_system', 'policy']).to_csv(pair_rel_csv, index=False)
p3d_policy3_hospital_summary_relative_cancer_detection_rate.sort_values(['hospital', 'ai_system', 'policy']).to_csv(hospital_rel_csv, index=False)
p3d_policy3_overall_summary_relative_cancer_detection_rate_unweighted.sort_values(['ai_system', 'policy']).to_csv(overall_rel_unweighted_csv, index=False)

# Load standard metrics and existing cancer detection rate metric.
standard_overall = _load_table_from_memory_or_csv(
    'p3d_policy3_overall_summary_unweighted_hospital_mean',
    base_out_dir / 'p3d_policy3_overall_summary_unweighted_hospital_mean.csv',
)
if 'aggregation_method' not in standard_overall.columns:
    standard_overall['aggregation_method'] = 'unweighted_mean_of_hospital_level_metrics'

standard_pair = _load_table_from_memory_or_csv(
    'p3d_policy3_pair_summary',
    base_out_dir / 'p3d_policy3_pair_summary.csv',
)
if 'manufacturer' not in standard_pair.columns:
    standard_pair['manufacturer'] = 'ALL'
standard_hospital = _load_table_from_memory_or_csv(
    'p3d_policy3_hospital_summary',
    base_out_dir / 'p3d_policy3_hospital_summary.csv',
)

cdr_overall = _load_table_from_memory_or_csv(
    'p3d_policy3_overall_summary_cancer_detection_rate_unweighted',
    cdr_source_dir / 'p3d_policy3_overall_summary_cancer_detection_rate_unweighted_hospital_mean.csv',
)
cdr_pair = _load_table_from_memory_or_csv(
    'p3d_policy3_pair_summary_cancer_detection_rate',
    cdr_source_dir / 'p3d_policy3_pair_summary_cancer_detection_rate.csv',
)
cdr_hospital = _load_table_from_memory_or_csv(
    'p3d_policy3_hospital_summary_cancer_detection_rate',
    cdr_source_dir / 'p3d_policy3_hospital_summary_cancer_detection_rate.csv',
)

p3d_policy3_overall_summary_unweighted_hospital_mean_with_cancer_detection_and_relative_theoretical = _combine_tables(
    standard_overall,
    cdr_overall,
    p3d_policy3_overall_summary_relative_cancer_detection_rate_unweighted,
)
p3d_policy3_pair_summary_with_cancer_detection_and_relative_theoretical = _combine_tables(
    standard_pair,
    cdr_pair,
    p3d_policy3_pair_summary_relative_cancer_detection_rate,
)
p3d_policy3_hospital_summary_with_cancer_detection_and_relative_theoretical = _combine_tables(
    standard_hospital,
    cdr_hospital,
    p3d_policy3_hospital_summary_relative_cancer_detection_rate,
)

combined_overall_csv = combined_out_dir / 'p3d_policy3_overall_summary_unweighted_hospital_mean_with_cancer_detection_and_relative_theoretical.csv'
combined_pair_csv = combined_out_dir / 'p3d_policy3_pair_summary_with_cancer_detection_and_relative_theoretical.csv'
combined_hospital_csv = combined_out_dir / 'p3d_policy3_hospital_summary_with_cancer_detection_and_relative_theoretical.csv'
p3d_policy3_overall_summary_unweighted_hospital_mean_with_cancer_detection_and_relative_theoretical.to_csv(combined_overall_csv, index=False)
p3d_policy3_pair_summary_with_cancer_detection_and_relative_theoretical.to_csv(combined_pair_csv, index=False)
p3d_policy3_hospital_summary_with_cancer_detection_and_relative_theoretical.to_csv(combined_hospital_csv, index=False)


# ------------------------------------------------------------------
# Plot helpers.
# ------------------------------------------------------------------
def _extract_val(df, ai, sm, pol, col):
    z = df[(df['ai_system'].astype(str) == str(ai)) & (df['signal_metric'].astype(str) == str(sm)) & (df['policy'] == pol)]
    if len(z) == 0:
        return np.nan
    return float(pd.to_numeric(z[col], errors='coerce').iloc[0])


def _plot_policy3_combined_screen_metrics(d, col_name, target_col, target_label, y_label, out_dir, out_name, title_prefix, note_lines, preferred_y_min, preferred_y_max):
    d = d.copy()
    d = d[d['policy'].isin(policy_order)].copy()
    if len(d) == 0:
        raise RuntimeError('No rows available for plotting.')

    present = set(d['signal_metric'].dropna().astype(str).unique())
    metrics = [m for m in metric_order if m in present] + [m for m in sorted(present) if m not in metric_order]
    ai_order = sorted(d['ai_system'].dropna().astype(str).unique())
    ai_label_map = {ai: f'AI-{idx + 1}' for idx, ai in enumerate(ai_order)}

    row_order = [(ai, sm) for sm in metrics for ai in ai_order]
    x = np.arange(len(row_order))
    width = 0.24
    offsets = {
        'without_recalibration': -width,
        'recalibrate_every_step': 0.0,
        'recalibrate_significant_only': width,
    }

    fig, ax = plt.subplots(figsize=(21, 8), constrained_layout=False)
    y_all = []
    for pol in policy_order:
        y = [_extract_val(d, ai, sm, pol, col_name) for ai, sm in row_order]
        y_all.extend([v for v in y if pd.notna(v)])
        ax.bar(x + offsets[pol], y, width=width, label=policy_label[pol], color=colors[pol])

    target_vals = pd.to_numeric(d[target_col], errors='coerce').dropna()
    target_val = float(target_vals.mean()) if len(target_vals) else np.nan
    if pd.notna(target_val):
        ax.axhline(target_val, color='black', ls=':', lw=2.0, label=target_label)

    ax.set_title(f'{title_prefix}: {y_label}', fontweight='bold', fontsize=16, pad=12)
    ax.set_ylabel(y_label, fontsize=12)
    ax.set_xticks(x)
    ax.set_xticklabels([ai_label_map.get(ai, str(ai)) for ai, _ in row_order], fontsize=11)
    ax.tick_params(axis='y', labelsize=11)
    ax.grid(axis='y', alpha=0.25)

    for group_idx, sm in enumerate(metrics):
        start_pos = group_idx * len(ai_order)
        end_pos = start_pos + len(ai_order) - 1
        center_pos = (start_pos + end_pos) / 2
        ax.text(
            center_pos,
            -0.17,
            metric_label.get(sm, str(sm).replace('_', ' ').title()),
            transform=ax.get_xaxis_transform(),
            ha='center',
            va='top',
            fontsize=10.5,
        )
        if group_idx > 0:
            ax.axvline(start_pos - 0.5, color='0.85', lw=1.0, zorder=0)

    ax.set_xlabel('')
    finite_vals = [v for v in y_all if pd.notna(v)]
    if pd.notna(target_val):
        finite_vals.append(target_val)
    if len(finite_vals) > 0:
        lower = min(float(preferred_y_min), float(np.nanmin(finite_vals)) - 1.0)
        upper = max(float(preferred_y_max), float(np.nanmax(finite_vals)) + 1.0)
        ax.set_ylim(max(0.0, lower), min(100.0, upper))

    step_ex = int(pd.to_numeric(d['step_exams'], errors='coerce').dropna().iloc[0])
    gap_ex = int(pd.to_numeric(d['gap_exams'], errors='coerce').dropna().iloc[0])
    handles, labels = ax.get_legend_handles_labels()
    note_handles = [
        Line2D([], [], color='none', label=f'Test cadence: every {step_ex:,} exams; gap = {gap_ex:,} exams'),
        Line2D([], [], color='none', label='Cancer detection rate = AI-flagged screen-detected'),
        Line2D([], [], color='none', label='cancers per 1,000 exams'),
        Line2D([], [], color='none', label='Relative cancer detection rate = AI-flagged'),
        Line2D([], [], color='none', label='screen-detected cancers / radiologist-flagged'),
        Line2D([], [], color='none', label='screen-detected cancers'),
        Line2D([], [], color='none', label='Cancer outcome signals are theoretical'),
        Line2D([], [], color='none', label='and use delayed outcome information'),
    ]
    note_handles += [Line2D([], [], color='none', label=line) for line in note_lines]
    handles = handles + note_handles
    labels = labels + [h.get_label() for h in note_handles]
    ax.legend(
        handles,
        labels,
        loc='center left',
        bbox_to_anchor=(1.01, 0.5),
        ncol=1,
        fontsize=9.5,
        title='Policies and figure notes',
        title_fontsize=11,
        frameon=True,
    )

    fig.subplots_adjust(top=0.88, bottom=0.24, left=0.06, right=0.75)
    out_path = Path(out_dir) / out_name
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_path, dpi=220, bbox_inches='tight')
    if bool(globals().get('P3D_RELATIVE_CDR_PLOTS_SHOW', True)):
        plt.show()
    else:
        plt.close(fig)
    return out_path


# Overall plots across hospitals.
sens_png = _plot_policy3_combined_screen_metrics(
    p3d_policy3_overall_summary_unweighted_hospital_mean_with_cancer_detection_and_relative_theoretical,
    col_name='tw_sensitivity_recal',
    target_col='target_sensitivity',
    target_label='Target sensitivity',
    y_label='Time-weighted sensitivity (%)',
    out_dir=combined_out_dir,
    out_name='p3d_policy3_overall_unweighted_hospital_mean_sensitivity_with_cancer_detection_and_relative_theoretical.png',
    title_prefix='Three-policy comparison across hospitals',
    note_lines=['Overall: unweighted mean of hospital-level metrics'],
    preferred_y_min=34,
    preferred_y_max=50,
)
spec_png = _plot_policy3_combined_screen_metrics(
    p3d_policy3_overall_summary_unweighted_hospital_mean_with_cancer_detection_and_relative_theoretical,
    col_name='tw_specificity_recal',
    target_col='target_specificity',
    target_label='Target specificity',
    y_label='Time-weighted specificity (%)',
    out_dir=combined_out_dir,
    out_name='p3d_policy3_overall_unweighted_hospital_mean_specificity_with_cancer_detection_and_relative_theoretical.png',
    title_prefix='Three-policy comparison across hospitals',
    note_lines=['Overall: unweighted mean of hospital-level metrics'],
    preferred_y_min=88,
    preferred_y_max=98,
)

# Danderyds-excluded plots.
no_danderyd_dir = combined_out_dir / 'excluding_danderyd'
DANDERYD_EXCLUSION_PATTERN = 'danderyd'
DANDERYD_EXCLUSION_LABEL = 'Danderyds Hospital'
hospital_no_danderyd = p3d_policy3_hospital_summary_with_cancer_detection_and_relative_theoretical[
    ~p3d_policy3_hospital_summary_with_cancer_detection_and_relative_theoretical['hospital'].astype(str).str.contains(DANDERYD_EXCLUSION_PATTERN, case=False, na=False)
].copy()

# Recompute unweighted no-Danderyd directly from hospital-level rows.
no_danderyd_rows = []
gcols = ['ai_system', 'signal_metric', 'policy', 'step_exams', 'gap_exams']
for keys, g in hospital_no_danderyd.groupby(gcols, dropna=False):
    valid = g.loc[pd.to_numeric(g['tw_sensitivity_recal'], errors='coerce').notna()].copy()
    row = {
        'ai_system': keys[0],
        'signal_metric': keys[1],
        'policy': keys[2],
        'step_exams': int(keys[3]),
        'gap_exams': int(keys[4]),
        'n_hospitals': int(valid['hospital'].nunique()) if len(valid) else 0,
        'aggregation_method': 'unweighted_mean_of_hospital_level_metrics_excluding_danderyd',
    }
    for c in ['n_test_steps_total', 'n_signal_detected_total', 'n_recalibrations_total', 'n_threshold_changes_total', 'weights_total_exams']:
        if c in valid.columns:
            row[c] = float(pd.to_numeric(valid[c], errors='coerce').fillna(0).sum())
    for c in ['target_sensitivity', 'target_specificity', 'tw_sensitivity_recal', 'tw_specificity_recal', 'mean_exams_between_recal', 'median_exams_between_recal']:
        if c in valid.columns:
            row[c] = float(pd.to_numeric(valid[c], errors='coerce').mean()) if len(valid) else np.nan
    no_danderyd_rows.append(row)
p3d_policy3_overall_summary_unweighted_hospital_mean_with_cancer_detection_and_relative_theoretical_no_danderyd = pd.DataFrame(no_danderyd_rows)

no_danderyd_csv = no_danderyd_dir / 'p3d_policy3_overall_summary_unweighted_hospital_mean_with_cancer_detection_and_relative_theoretical_no_danderyd.csv'
hospital_no_danderyd_csv = no_danderyd_dir / 'p3d_policy3_hospital_summary_with_cancer_detection_and_relative_theoretical_no_danderyd.csv'
no_danderyd_dir.mkdir(parents=True, exist_ok=True)
p3d_policy3_overall_summary_unweighted_hospital_mean_with_cancer_detection_and_relative_theoretical_no_danderyd.to_csv(no_danderyd_csv, index=False)
hospital_no_danderyd.to_csv(hospital_no_danderyd_csv, index=False)

sens_png_no_danderyd = _plot_policy3_combined_screen_metrics(
    p3d_policy3_overall_summary_unweighted_hospital_mean_with_cancer_detection_and_relative_theoretical_no_danderyd,
    col_name='tw_sensitivity_recal',
    target_col='target_sensitivity',
    target_label='Target sensitivity',
    y_label='Time-weighted sensitivity (%)',
    out_dir=no_danderyd_dir,
    out_name='p3d_policy3_overall_unweighted_hospital_mean_sensitivity_with_cancer_detection_and_relative_theoretical_no_danderyd.png',
    title_prefix=f'Three-policy comparison across hospitals excluding {DANDERYD_EXCLUSION_LABEL}',
    note_lines=['Overall: unweighted mean of hospital-level metrics', f'{DANDERYD_EXCLUSION_LABEL} excluded'],
    preferred_y_min=34,
    preferred_y_max=50,
)
spec_png_no_danderyd = _plot_policy3_combined_screen_metrics(
    p3d_policy3_overall_summary_unweighted_hospital_mean_with_cancer_detection_and_relative_theoretical_no_danderyd,
    col_name='tw_specificity_recal',
    target_col='target_specificity',
    target_label='Target specificity',
    y_label='Time-weighted specificity (%)',
    out_dir=no_danderyd_dir,
    out_name='p3d_policy3_overall_unweighted_hospital_mean_specificity_with_cancer_detection_and_relative_theoretical_no_danderyd.png',
    title_prefix=f'Three-policy comparison across hospitals excluding {DANDERYD_EXCLUSION_LABEL}',
    note_lines=['Overall: unweighted mean of hospital-level metrics', f'{DANDERYD_EXCLUSION_LABEL} excluded'],
    preferred_y_min=88,
    preferred_y_max=98,
)

# Hospital-manufacturer pair plots.
pair_plot_out_dir = combined_out_dir / 'hospital_manufacturer_pair_figures'
pair_plot_out_dir.mkdir(parents=True, exist_ok=True)
for old_png in pair_plot_out_dir.glob('p3d_policy3_pair_*_with_cancer_detection_and_relative_theoretical_*.png'):
    old_png.unlink()

pair_df = p3d_policy3_pair_summary_with_cancer_detection_and_relative_theoretical.copy()
pair_df['hospital'] = pair_df['hospital'].astype(str)
pair_df['manufacturer'] = pair_df['manufacturer'].fillna('ALL').astype(str)
pairs = pair_df[['hospital', 'manufacturer']].drop_duplicates().sort_values(['hospital', 'manufacturer']).reset_index(drop=True)

figure_rows = []
for _, pair_row in pairs.iterrows():
    hospital = pair_row['hospital']
    manufacturer = pair_row['manufacturer']
    d_pair = pair_df[(pair_df['hospital'] == hospital) & (pair_df['manufacturer'] == manufacturer)].copy()
    if len(d_pair) == 0:
        continue
    pair_slug = f'{_slug(hospital)}_{_slug(manufacturer)}'

    has_sens = pd.to_numeric(d_pair['tw_sensitivity_recal'], errors='coerce').notna().any()
    has_spec = pd.to_numeric(d_pair['tw_specificity_recal'], errors='coerce').notna().any()
    if not has_sens and not has_spec:
        print(f'[skip empty] {hospital} | {manufacturer}: no finite sensitivity/specificity values')
        continue

    sens_pair_path = _plot_policy3_combined_screen_metrics(
        d_pair,
        col_name='tw_sensitivity_recal',
        target_col='target_sensitivity',
        target_label='Target sensitivity',
        y_label='Time-weighted sensitivity (%)',
        out_dir=pair_plot_out_dir,
        out_name=f'p3d_policy3_pair_sensitivity_with_cancer_detection_and_relative_theoretical_{pair_slug}.png',
        title_prefix=f'{hospital} | {manufacturer}',
        note_lines=[f'Hospital: {hospital}', f'Manufacturer: {manufacturer}', 'No aggregation: one hospital-manufacturer pair'],
        preferred_y_min=34,
        preferred_y_max=50,
    ) if has_sens else None
    spec_pair_path = _plot_policy3_combined_screen_metrics(
        d_pair,
        col_name='tw_specificity_recal',
        target_col='target_specificity',
        target_label='Target specificity',
        y_label='Time-weighted specificity (%)',
        out_dir=pair_plot_out_dir,
        out_name=f'p3d_policy3_pair_specificity_with_cancer_detection_and_relative_theoretical_{pair_slug}.png',
        title_prefix=f'{hospital} | {manufacturer}',
        note_lines=[f'Hospital: {hospital}', f'Manufacturer: {manufacturer}', 'No aggregation: one hospital-manufacturer pair'],
        preferred_y_min=88,
        preferred_y_max=98,
    ) if has_spec else None

    figure_rows.append({
        'hospital': hospital,
        'manufacturer': manufacturer,
        'sensitivity_png': '' if sens_pair_path is None else str(sens_pair_path),
        'specificity_png': '' if spec_pair_path is None else str(spec_pair_path),
    })

p3d_policy3_pair_cancer_detection_and_relative_figure_index = pd.DataFrame(figure_rows)
figure_index_csv = pair_plot_out_dir / 'p3d_policy3_pair_cancer_detection_and_relative_figure_index.csv'
p3d_policy3_pair_cancer_detection_and_relative_figure_index.to_csv(figure_index_csv, index=False)

ai_order_for_print = sorted(p3d_policy3_overall_summary_unweighted_hospital_mean_with_cancer_detection_and_relative_theoretical['ai_system'].dropna().astype(str).unique())
print('Image AI label mapping:', ', '.join(f'AI-{idx + 1} = {name}' for idx, name in enumerate(ai_order_for_print)))
print('Saved relative cancer detection rate pair summary:', pair_rel_csv)
print('Saved relative cancer detection rate hospital summary:', hospital_rel_csv)
print('Saved relative cancer detection rate overall summary:', overall_rel_unweighted_csv)
print('Saved combined overall plotting table:', combined_overall_csv)
print('Saved combined hospital plotting table:', combined_hospital_csv)
print('Saved combined pair plotting table:', combined_pair_csv)
print('Saved overall sensitivity figure:', sens_png)
print('Saved overall specificity figure:', spec_png)
print('Saved no-Danderyd sensitivity figure:', sens_png_no_danderyd)
print('Saved no-Danderyd specificity figure:', spec_png_no_danderyd)
print('Saved pair figure index:', figure_index_csv)
print('Saved hospital-manufacturer pair figures in:', pair_plot_out_dir)

try:
    display(p3d_policy3_overall_summary_unweighted_hospital_mean_with_cancer_detection_and_relative_theoretical.head(24))
    display(p3d_policy3_pair_cancer_detection_and_relative_figure_index)
except Exception:
    pass
